# X-Ray Hologram V13 — Tabs 1–8

Tabs 1–7 retain their existing implementations. Tab 8 is rebuilt as a clean optimized implementation.


In [ ]:
def _tab6_physical_geometry(self):
    """Return (physical_width_nm, physical_height_nm, output_width, output_height)."""
    arr = self._proc_image_display.get_array()
    if arr is None:
        return None
    h, w = np.asarray(arr).shape[:2]
    base = self._proc_get_tab5_base_pixel_size()
    if base is None or not np.isfinite(base) or base <= 0:
        return None
    ow = int(getattr(self, 'original_roi_w', 0) or w)
    oh = int(getattr(self, 'original_roi_h', 0) or h)
    if ow <= 0 or oh <= 0 or w <= 0 or (h <= 0):
        return None
    return (float(ow) * float(base), float(oh) * float(base), int(w), int(h))

def _tab6_nm_to_output_px(self, x_nm, y_nm):
    """Convert physical nm coordinates from the displayed axes to output pixels."""
    geom = self._tab6_physical_geometry()
    if geom is None:
        return (float(x_nm), float(y_nm))
    pw, ph, w, h = geom
    px = float(x_nm) * w / pw
    py = float(y_nm) * h / ph
    return (px, py)

def _tab6_output_px_to_nm_rect(self, xmin_px, xmax_px, ymin_px, ymax_px):
    """Convert an internal pixel ROI rectangle to displayed physical nm bounds."""
    geom = self._tab6_physical_geometry()
    if geom is None:
        return (float(xmin_px), float(xmax_px), float(ymin_px), float(ymax_px))
    pw, ph, w, h = geom
    return (float(xmin_px) * pw / w, float(xmax_px + 1) * pw / w, float(ymin_px) * ph / h, float(ymax_px + 1) * ph / h)

def _tab6_secondary_status_nm(self):
    """Return a human-readable Secondary ROI status in nm."""
    c = getattr(self, 'roi2_coords', None)
    if c is None:
        return 'Secondary ROI: not selected'
    x0, x1, y0, y1 = self._tab6_output_px_to_nm_rect(int(c['xmin']), int(c['xmax']), int(c['ymin']), int(c['ymax']))
    return f'Secondary ROI: X={x0:.3g}–{x1:.3g} nm, Y={y0:.3g}–{y1:.3g} nm | Width={x1 - x0:.3g} nm, Height={y1 - y0:.3g} nm'

def _proc_secondary_on_release_nm(self, event):
    """Select Secondary ROI on the physical-nm display while storing output pixels internally."""
    if not getattr(self, '_proc_secondary_roi_selecting', False) or self._proc_secondary_press_xy is None:
        return
    if event.inaxes is not self._proc_ax or event.xdata is None or event.ydata is None:
        self._proc_secondary_press_xy = None
        return
    x0_nm, y0_nm = self._proc_secondary_press_xy
    x1_nm, y1_nm = (float(event.xdata), float(event.ydata))
    geom = self._tab6_physical_geometry()
    if geom is None:
        self.proc_secondary_status_var.set('Primary ROI has no valid Tab-5 final nm/pixel calibration.')
        self._proc_secondary_press_xy = None
        return
    physical_w_nm = float(geom[0])
    physical_h_nm = float(geom[1])
    w = int(geom[2])
    h = int(geom[3])
    x0_nm = max(0.0, min(x0_nm, physical_w_nm))
    x1_nm = max(0.0, min(x1_nm, physical_w_nm))
    y0_nm = max(0.0, min(y0_nm, physical_h_nm))
    y1_nm = max(0.0, min(y1_nm, physical_h_nm))
    xmin_nm, xmax_nm = sorted((x0_nm, x1_nm))
    ymin_nm, ymax_nm = sorted((y0_nm, y1_nm))
    xmin = int(np.floor(xmin_nm * w / physical_w_nm))
    xmax = int(np.ceil(xmax_nm * w / physical_w_nm) - 1)
    ymin = int(np.floor(ymin_nm * h / physical_h_nm))
    ymax = int(np.ceil(ymax_nm * h / physical_h_nm) - 1)
    xmin = max(0, min(xmin, w - 1))
    xmax = max(0, min(xmax, w - 1))
    ymin = max(0, min(ymin, h - 1))
    ymax = max(0, min(ymax, h - 1))
    self._proc_secondary_press_xy = None
    if xmax <= xmin or ymax <= ymin:
        self.proc_secondary_status_var.set('Secondary ROI too small. Drag a larger rectangle in nm coordinates.')
        return
    self.roi2_coords = {'xmin': xmin, 'xmax': xmax, 'ymin': ymin, 'ymax': ymax}
    if hasattr(self, 'roi2_idx_var'):
        self.roi2_idx_var.set(max(0, min(int(self.proc_idx_var.get()), len(self.recons) - 1)))
    self.proc_secondary_status_var.set(self._tab6_secondary_status_nm() + ' | Click CONFIRM SECONDARY ROI.')
    self._proc_secondary_roi_selecting = False
    try:
        self.proc_secondary_select_button.configure(text='SELECT SECONDARY ROI')
    except Exception:
        pass
    self._proc_draw_secondary_roi_patch(confirmed=False)

def _proc_secondary_on_move_nm(self, event):
    """Draw the Secondary ROI preview in physical nm coordinates."""
    if not getattr(self, '_proc_secondary_roi_selecting', False) or self._proc_secondary_press_xy is None:
        return
    if event.inaxes is not self._proc_ax or event.xdata is None or event.ydata is None:
        return
    x0, y0 = self._proc_secondary_press_xy
    x1, y1 = (float(event.xdata), float(event.ydata))
    xmin, xmax = sorted((x0, x1))
    ymin, ymax = sorted((y0, y1))
    geom = self._tab6_physical_geometry()
    if geom is not None:
        pw = float(geom[0])
        ph = float(geom[1])
        xmin = max(0.0, min(xmin, pw))
        xmax = max(0.0, min(xmax, pw))
        ymin = max(0.0, min(ymin, ph))
        ymax = max(0.0, min(ymax, ph))
    if self._proc_secondary_patch is None:
        self._proc_secondary_patch = plt.Rectangle((xmin, ymin), max(1e-12, xmax - xmin), max(1e-12, ymax - ymin), fill=False, edgecolor='red', linewidth=2.5, zorder=20)
        self._proc_ax.add_patch(self._proc_secondary_patch)
    else:
        self._proc_secondary_patch.set_xy((xmin, ymin))
        self._proc_secondary_patch.set_width(max(1e-12, xmax - xmin))
        self._proc_secondary_patch.set_height(max(1e-12, ymax - ymin))
    self._proc_canvas.draw_idle()

def _proc_draw_secondary_roi_patch_nm(self, confirmed=False, redraw=True):
    """Draw stored Secondary ROI as a physical-nm rectangle."""
    if self._proc_secondary_patch is not None:
        try:
            self._proc_secondary_patch.remove()
        except Exception:
            pass
        self._proc_secondary_patch = None
    c = getattr(self, 'roi2_coords', None)
    if c is None:
        if redraw:
            self._proc_canvas.draw_idle()
        return
    x0, x1, y0, y1 = self._tab6_output_px_to_nm_rect(int(c['xmin']), int(c['xmax']), int(c['ymin']), int(c['ymax']))
    edge = 'lime' if confirmed else 'red'
    self._proc_secondary_patch = plt.Rectangle((x0, y0), max(1e-12, x1 - x0), max(1e-12, y1 - y0), fill=False, edgecolor=edge, linewidth=2.5, zorder=20)
    self._proc_ax.add_patch(self._proc_secondary_patch)
    if redraw:
        self._proc_canvas.draw_idle()

def _proc_secondary_geometry_values(self):
    geom = self._tab6_physical_geometry()
    if geom is None:
        return None
    if len(geom) >= 4:
        return (float(geom[0]), float(geom[1]), int(geom[2]), int(geom[3]))
    raise RuntimeError(f'Unexpected Tab 6 geometry tuple length: {len(geom)}')

def _proc_secondary_on_move_nm_safe(self, event):
    if not getattr(self, '_proc_secondary_roi_selecting', False) or self._proc_secondary_press_xy is None:
        return
    if event.inaxes is not self._proc_ax or event.xdata is None or event.ydata is None:
        return
    x0, y0 = self._proc_secondary_press_xy
    x1, y1 = (float(event.xdata), float(event.ydata))
    xmin, xmax = sorted((x0, x1))
    ymin, ymax = sorted((y0, y1))
    dims = _proc_secondary_geometry_values(self)
    if dims is None:
        return
    pw, ph, _, _ = dims
    xmin = max(0.0, min(xmin, pw))
    xmax = max(0.0, min(xmax, pw))
    ymin = max(0.0, min(ymin, ph))
    ymax = max(0.0, min(ymax, ph))
    if self._proc_secondary_patch is None:
        self._proc_secondary_patch = plt.Rectangle((xmin, ymin), max(1e-12, xmax - xmin), max(1e-12, ymax - ymin), fill=False, edgecolor='red', linewidth=2.5, zorder=20)
        self._proc_ax.add_patch(self._proc_secondary_patch)
    else:
        self._proc_secondary_patch.set_xy((xmin, ymin))
        self._proc_secondary_patch.set_width(max(1e-12, xmax - xmin))
        self._proc_secondary_patch.set_height(max(1e-12, ymax - ymin))
    self._proc_canvas.draw_idle()

def _proc_secondary_on_release_nm_safe(self, event):
    if not getattr(self, '_proc_secondary_roi_selecting', False) or self._proc_secondary_press_xy is None:
        return
    if event.inaxes is not self._proc_ax or event.xdata is None or event.ydata is None:
        self._proc_secondary_press_xy = None
        return
    x0_nm, y0_nm = self._proc_secondary_press_xy
    x1_nm, y1_nm = (float(event.xdata), float(event.ydata))
    dims = _proc_secondary_geometry_values(self)
    if dims is None:
        self._proc_secondary_press_xy = None
        return
    physical_w_nm, physical_h_nm, w, h = dims
    x0_nm = max(0.0, min(x0_nm, physical_w_nm))
    x1_nm = max(0.0, min(x1_nm, physical_w_nm))
    y0_nm = max(0.0, min(y0_nm, physical_h_nm))
    y1_nm = max(0.0, min(y1_nm, physical_h_nm))
    xmin_nm, xmax_nm = sorted((x0_nm, x1_nm))
    ymin_nm, ymax_nm = sorted((y0_nm, y1_nm))
    xmin = int(np.floor(xmin_nm * w / physical_w_nm))
    xmax = int(np.ceil(xmax_nm * w / physical_w_nm) - 1)
    ymin = int(np.floor(ymin_nm * h / physical_h_nm))
    ymax = int(np.ceil(ymax_nm * h / physical_h_nm) - 1)
    xmin = max(0, min(xmin, w - 1))
    xmax = max(0, min(xmax, w - 1))
    ymin = max(0, min(ymin, h - 1))
    ymax = max(0, min(ymax, h - 1))
    self._proc_secondary_press_xy = None
    if xmax <= xmin or ymax <= ymin:
        try:
            self.proc_secondary_status_var.set('Secondary ROI too small. Drag a larger rectangle in nm coordinates.')
        except Exception:
            pass
        return
    self.roi2_coords = {'xmin': xmin, 'xmax': xmax, 'ymin': ymin, 'ymax': ymax}
    if hasattr(self, 'roi2_idx_var'):
        try:
            self.roi2_idx_var.set(max(0, min(int(self.proc_idx_var.get()), len(self.recons) - 1)))
        except Exception:
            pass
    try:
        self.proc_secondary_status_var.set(self._tab6_secondary_status_nm() + ' | Click CONFIRM SECONDARY ROI.')
    except Exception:
        pass
    self._proc_secondary_roi_selecting = False
    try:
        self.proc_secondary_select_button.configure(text='SELECT SECONDARY ROI')
    except Exception:
        pass
    try:
        self._proc_draw_secondary_roi_patch(confirmed=False)
    except Exception:
        pass
import os, re, json, traceback, importlib.util, types, sys
from pathlib import Path
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import numpy as np
import h5py
import pandas as pd
from PIL import Image
from scipy.interpolate import interp1d
from scipy.ndimage import zoom, gaussian_filter, map_coordinates
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline
from skimage import exposure, filters, measure, morphology
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.widgets import RectangleSelector

class CDIWorkflowApp:

    def __init__(self, root):
        self.root = root
        self.root.title('CDI Reconstruction / ROI / Image Processing — Single Tkinter Window')
        self.root.geometry('1500x950')
        self.root.minsize(1200, 800)
        self.data_dir = None
        self.hdf5_files = []
        self.ramp_direction_var = tk.StringVar(value='Ramp Up (- to +)')
        self.ramp_direction_status_var = tk.StringVar(value='Dataset direction: --')
        self.field_calib_file = None
        self.frc_file = None
        self.mask_lib = None
        self.mask_lib_path = None
        self.field_interpolator = None
        self.calib_current = None
        self.calib_field_mT = None
        self.recons = None
        self.complex_recons = None
        self.real_recons = None
        self.imag_recons = None
        self.amplitude_recons = None
        self.phase_recons = None
        self.supportmask = None
        self.supportmask2 = None
        self.raw_fields = None
        self.real_fields_mT = None
        self.roi_mask = None
        self.roi_coords = None
        self.roi_stack = None
        self.roi_mean = None
        self.roi_std = None
        self.roi_crop = None
        self.confirmed_roi_frame = None
        self.confirmed_roi_frame_index = None
        self.roi_width_var = tk.StringVar(value='0')
        self.roi_height_var = tk.StringVar(value='0')
        self.proc_roi_info_var = tk.StringVar(value='ROI = --')
        self.proc_status_var = tk.StringVar(value='Ready')
        self.proc_idx_var = tk.IntVar(value=0)
        self.roi_resolution = None
        self.roi_resolution_applied = False
        self.sem_image = None
        self.sem_image_path = None
        self.sem_scale_line = None
        self.sem_scale_pixels = None
        self.sem_known_length_nm_var = tk.DoubleVar(value=1000.0)
        self.sem_nm_per_pixel = None
        self.sem_calibration_confirmed = False
        self.sem_verification_line = None
        self.sem_verification_pixels = None
        self.sem_verification_length_nm = None
        self.sem_verification_confirmed = False
        self.primary_roi_scale_line = None
        self.primary_roi_scale_pixels = None
        self.primary_roi_final_nm_per_pixel = None
        self.pixel_cal_line_color_var = tk.StringVar(value='red')
        self.pixel_cal_line_style_var = tk.StringVar(value='-')
        self.pixel_cal_line_width_var = tk.DoubleVar(value=2.5)
        self._pixel_cal_zoom_limits = {'sem': None, 'roi': None}
        self._pixel_cal_roi_display_shape = None
        self._pixel_cal_roi_display_support = None
        self._pixel_cal_roi_origin_full = None
        self._pixel_cal_zoom_factor = 1.25
        self.pixel_cal_status_var = tk.StringVar(value='Upload an SEM image to begin pixel calibration.')
        self.pixel_cal_sem_result_var = tk.StringVar(value='SEM nm/pixel: --')
        self.pixel_cal_verification_result_var = tk.StringVar(value='Verification: --')
        self.pixel_cal_roi_pixels_var = tk.StringVar(value='Primary ROI line: -- pixels')
        self.pixel_cal_roi_length_var = tk.StringVar(value='Transferred physical length: -- nm')
        self.pixel_cal_final_var = tk.StringVar(value='FINAL Primary ROI nm/pixel: --')
        self.pixel_cal_direct_nm_per_pixel_var = tk.StringVar(value='')
        self.pixel_cal_active_nm_per_pixel = None
        self.roi2_coords = None
        self.roi2_confirmed_coords = None
        self.roi2_mask = None
        self.roi2_stack = None
        self.roi2_idx_var = tk.IntVar(value=0)
        self.roi2_status_var = tk.StringVar(value='')
        self.roi2_field_var = tk.StringVar(value='Frame 1/0 | Field = --')
        self.multi_fit_frame_var = tk.IntVar(value=0)
        self.multi_fit_roi_var = tk.StringVar(value='ROI 1')
        self.multi_fit_contrast_fraction_var = tk.DoubleVar(value=0.2)
        self.multi_fit_range_start_var = tk.IntVar(value=0)
        self.multi_fit_range_end_var = tk.IntVar(value=0)
        self.multi_fit_field_var = tk.StringVar(value='Frame 1/0 | Field = --')
        self._roi2_patch = None
        self._roi2_press_xy = None
        self._roi2_play_job = None
        self._roi2_playing = False
        self.current_index = 0
        self.current_display = None
        self.current_processed = None
        self._slider_redraw_job = None
        self.recon_dynamic_zoom_var = tk.BooleanVar(value=False)
        self._recon_zoom_popup = None
        self._recon_zoom_canvas = None
        self._recon_zoom_axes = None
        self._recon_zoom_fig = None
        self._recon_zoom_img_artist = None
        self._recon_zoom_active_ax = None
        self._recon_zoom_bindings = []
        self.last_blobs = None
        self.gauss2d_results = None
        self.line_source_stack = None
        self.line_source_coords = None
        self.line_idx_var = None
        self.line_source_status_var = None
        self.line_status_var = None
        self.line_fit_status_var = None
        self.line_contrast_fraction_var = None
        self.line_center_x_var = None
        self.line_center_y_var = None
        self.line_center_fixed = False
        self.line_profile_0 = None
        self.line_profile_90 = None
        self.line_profile_avg = None
        self.line_fit_result = None
        self.line_fit_resid_ax = None
        self.line_ref_mode_var = tk.StringVar(value='2')
        self.line_ref_angles = [0.0, 90.0]
        self.line_ref_profiles = {}
        self.line_ref_average = None
        self.line_range_start_var = tk.IntVar(value=0)
        self.line_range_end_var = tk.IntVar(value=0)
        self.line_range_status_var = tk.StringVar(value='No FWHM range tracking performed.')
        self.line_fwhm_results = None
        self.line_fwhm_frame_ax = None
        self.line_ax_amp = None
        self.gauss2d_source_stack = None
        self.gauss2d_source_coords = None
        self.gauss2d_source_status_var = None
        self.gauss2d_idx_var = None
        self.gauss2d_status_var = None
        self.gauss2d_fit_status_var = None
        self.gauss2d_auto_guess_var = None
        self.gauss2d_bg_var = None
        self.gauss2d_amp_var = None
        self.gauss2d_x0_var = None
        self.gauss2d_y0_var = None
        self.gauss2d_sx_var = None
        self.gauss2d_sigma_min_var = None
        self.gauss2d_sigma_max_var = None
        self.gauss2d_center_margin_var = None
        self.gauss2d_results = None
        self.gauss2d_source_stack = None
        self.gauss2d_source_coords = None
        self.gauss2d_source_status_var = None
        self.gauss2d_idx_var = None
        self.gauss2d_status_var = None
        self.gauss2d_fit_status_var = None
        self.gauss2d_auto_guess_var = None
        self.gauss2d_bg_var = None
        self.gauss2d_amp_var = None
        self.gauss2d_x0_var = None
        self.gauss2d_y0_var = None
        self.gauss2d_sx_var = None
        self.gauss2d_sy_var = None
        self.gauss2d_theta_var = None
        self.gauss2d_sigma_min_var = None
        self.gauss2d_sigma_max_var = None
        self.gauss2d_center_margin_var = None
        self._roi_selector = None
        self._figs = {}
        self.multi_rois = []
        self.multi_roi_patches = []
        self.multi_roi_labels = []
        self.multi_roi_temp_patch = None
        self._multi_roi_selecting = False
        self._multi_roi_press_xy = None
        self.multi_roi_frame_var = tk.IntVar(value=0)
        self.multi_roi_status_var = tk.StringVar(value='No ROIs selected.')
        self.multi_roi_fit_status_var = tk.StringVar(value='Select ROIs in Tab 9, then fit them in Tab 9.')
        self.multi_fit_frame_var = tk.IntVar(value=0)
        self.multi_fit_start_var = tk.IntVar(value=0)
        self.multi_fit_end_var = tk.IntVar(value=0)
        self.multi_fit_range_start_var = tk.IntVar(value=0)
        self.multi_fit_range_end_var = tk.IntVar(value=0)
        self.multi_roi_fit_results = None
        self._log_messages = []
        self.log_text = None
        if not hasattr(self, 'proc_idx_var'):
            self.proc_idx_var = tk.IntVar(value=0)
        if not hasattr(self, 'brightness_var'):
            self.brightness_var = tk.DoubleVar(value=0.0)
        if not hasattr(self, 'contrast_var'):
            self.contrast_var = tk.DoubleVar(value=1.0)
        if not hasattr(self, 'filter_var'):
            self.filter_var = tk.StringVar(value='Gaussian')
        if not hasattr(self, 'strength_var'):
            self.strength_var = tk.DoubleVar(value=3.0)
        if not hasattr(self, 'colormap_var'):
            self.colormap_var = tk.StringVar(value='RdBu_r')
        if not hasattr(self, 'view_var'):
            self.view_var = tk.StringVar(value='IFFT')
        if not hasattr(self, 'fft_mask_var'):
            self.fft_mask_var = tk.DoubleVar(value=0.6)
        if not hasattr(self, 'fps_var'):
            self.fps_var = tk.IntVar(value=10)
        try:
            self._build_ui()
        except Exception as exc:
            self.log(f'UI initialization warning: {type(exc).__name__}: {exc}')
            if self.log_text is None:
                self._build_log_tab()



    def _install_tab_reset_buttons(self):
        """Put an orange TAB RESET control inside the top-right of every tab."""
        try:
            tabs = list(self.nb.tabs())
        except Exception:
            return

        for tab_id in tabs:
            try:
                frame = self.root.nametowidget(tab_id)
                # Reuse/replace only this tab's reset button.
                old = getattr(frame, "_tab_reset_button", None)
                if old is not None:
                    try:
                        old.destroy()
                    except Exception:
                        pass

                btn = tk.Button(
                    frame,
                    text="TAB RESET",
                    command=self._reset_current_tab,
                    bg="orange",
                    activebackground="darkorange",
                    fg="black",
                    font=("Arial", 8, "bold"),
                    relief="raised",
                    bd=2,
                    padx=7,
                    pady=2,
                    cursor="hand2"
                )
                frame._tab_reset_button = btn

                # Float over the tab content at the extreme top-right.
                # This does not re-grid/re-pack existing tab controls.
                btn.place(
                    relx=0.995,
                    x=-5,
                    y=4,
                    anchor="ne"
                )
                try:
                    btn.lift()
                except Exception:
                    pass
            except Exception:
                pass



    def _clear_existing_tab_children_for_reset(self, frame):
        """
        Remove all widgets inside an existing notebook tab before rebuilding it.
        The notebook tab frame itself is preserved. This prevents duplicate
        figures/images/widgets when TAB RESET is pressed repeatedly.
        """
        if frame is None:
            return
        try:
            # Destroy every direct child. Tkinter recursively destroys its
            # descendants, including embedded Matplotlib canvases and scrollbars.
            for child in list(frame.winfo_children()):
                try:
                    child.destroy()
                except Exception:
                    try:
                        child.pack_forget()
                    except Exception:
                        pass
        except Exception:
            pass

    def _reset_current_tab(self):
        """Safely rebuild only the active tab."""
        selected = self.nb.select()
        if not selected:
            return

        tab_index = self.nb.index(selected)
        tab_title = self.nb.tab(selected, "text")

        try:
            old_frame = self.root.nametowidget(selected)
        except Exception:
            old_frame = self.nb.nametowidget(selected)

        specs = {
            0: ("tab_data", self._build_data_tab, False),
            1: ("tab_recon", self._build_recon_tab, False),
            2: ("tab_support", self._build_support_tab, False),
            3: ("tab_roi", self._build_roi_tab, False),
            4: ("tab_pixel_calibration", self._build_pixel_calibration_tab, False),
            5: ("tab_proc", self._build_proc_tab, False),
            6: ("tab_line_profile_analysis", _build_line_profile_analysis_tab, False),
            7: ("tab_particle_density", _tab8_v6_build, False),
            8: ("tab_primary_roi9", _tab9_roi4_build, False),
            9: ("tab_single_roi10", _tab10_build, True),
            10: ("tab_multi_roi11", _tab11_build, True),
            11: ("tab_hdf5_12", _tab12_build, True),
            12: ("tab_frc_13", _tab13_build, True),
            13: ("tab14_log", _tab14_build, True),
        }

        if tab_index not in specs:
            self.status_var.set(
                f"TAB RESET unsupported: Tab {tab_index + 1}"
            )
            return

        attr_name, builder, creates_tab = specs[tab_index]
        old_value = getattr(self, attr_name, None)

        # Tab 8: the builder intentionally works on an already-created
        # tab_particle_density frame. Do not delete that frame attribute.
        # Tab 9-14 create/register their own fresh frame.
        try:
            if tab_index == 8:
                stopper = getattr(self, "_tab9_roi4_stop", None)
                if callable(stopper):
                    stopper()
        except Exception:
            pass

        try:
            if tab_index == 9:
                # _tab10_build binds this attribute during construction.
                # Make it available before the build begins.
                self._tab10_tab_changed = (
                    lambda event=None: _tab10_tab_changed(self, event)
                )

                # Remove any old Tab-10 notebook-change binding before rebuild
                # to prevent duplicate callbacks after repeated resets.
                try:
                    self.nb.unbind(
                        "<<NotebookTabChanged>>",
                        self._tab10_tab_bind_id
                    )
                except Exception:
                    pass

            if creates_tab:
                # Remove the old notebook entry only after the new tab builds
                # successfully. This avoids blanking the UI on build failure.
                before = set(self.nb.tabs())

                # Force the builder's "if hasattr(...)" guard to create fresh
                # frame for Tabs 9-14.
                try:
                    delattr(self, attr_name)
                except Exception:
                    pass

                if tab_index == 13:
                    self._tab14_built = False

                # Build the new tab.
                if getattr(builder, "__self__", None) is self:
                    builder()
                else:
                    builder(self)

                after = list(self.nb.tabs())
                added = [tid for tid in after if tid not in before]

                if not added:
                    raise RuntimeError(
                        f"Tab {tab_index + 1} builder did not create a notebook tab."
                    )

                new_id = added[-1]
                new_frame = self.root.nametowidget(new_id)

                # Position new tab at original location and keep original title.
                self.nb.forget(new_id)
                self.nb.insert(
                    tab_index,
                    new_frame,
                    text=tab_title
                )

                # Now remove old tab/frame.
                try:
                    self.nb.forget(old_frame)
                except Exception:
                    pass
                try:
                    old_frame.destroy()
                except Exception:
                    pass

            else:
                # Tabs 1-7: rebuild into the existing frame.
                # Tab 8 is especially important: creating another Notebook
                # entry here was the source of repeated '8. Blob Analysis' tabs.
                if tab_index == 7:
                    # Tab 8 is already fully built. Reset only its internal
                    # state and redraw in-place; never rebuild the Notebook
                    # tab/canvas tree.
                    self.tab_particle_density = old_frame
                    try:
                        _tab8_v6_reset_tab_state(self)
                    except Exception:
                        if getattr(builder, "__self__", None) is self:
                            builder()
                        else:
                            builder(self)
                    new_frame = old_frame

                else:
                    # Existing tabs keep their original notebook frame.
                    # Clear the contents before rebuilding any tab whose builder
                    # creates widgets/canvases.  Tab 7 was previously excluded
                    # here, so every TAB RESET appended a second copy of its
                    # Line Profile controls/Matplotlib canvas into the same tab.
                    if 0 <= tab_index <= 6:
                        self._clear_existing_tab_children_for_reset(old_frame)

                    if getattr(builder, "__self__", None) is self:
                        builder()
                    else:
                        builder(self)
                    new_frame = old_frame

            # Tab 9 uses an existing notebook frame. Re-fetch its processed
            # Primary ROI1 after rebuilding so the reset returns to a usable state.
            if tab_index == 8:
                try:
                    _tab9_roi4_fetch_from_tab6(self, preserve_frame=True)
                except Exception:
                    pass

            # Tabs 2-5 are fully rebuilt in-place after their old children
            # were destroyed. Force a geometry refresh so embedded figures
            # immediately occupy the available tab area.
            if 0 <= tab_index <= 4:
                try:
                    self.root.update_idletasks()
                except Exception:
                    pass

            # Tab-specific post-build stabilization.
            try:
                if tab_index == 7:
                    fn = getattr(self, "_tab8_v6_restore_default_controls", None)
                    if callable(fn):
                        fn()
            except Exception:
                pass

            # Tab 10 must have its tab-change callback bound exactly once.
            if tab_index == 9:
                try:
                    self._tab10_tab_changed = (
                        lambda event=None: _tab10_tab_changed(self, event)
                    )
                    self._tab10_tab_bind_id = self.nb.bind(
                        "<<NotebookTabChanged>>",
                        self._tab10_tab_changed,
                        add="+"
                    )
                except Exception:
                    pass

            try:
                self.nb.select(tab_index)
            except Exception:
                pass

            if tab_index == 7:
                try:
                    _tab8_v6_cleanup_duplicate_tabs(self)
                except Exception:
                    pass
            try:
                self._install_tab_reset_buttons()
            except Exception:
                pass

            self.status_var.set(
                f"TAB RESET complete — {tab_title}"
            )
            try:
                self.log(f"TAB RESET: {tab_title}")
            except Exception:
                pass

        except Exception as exc:
            # Roll back object reference where possible. The old tab was not
            # destroyed until the new builder completed.
            try:
                if old_value is not None:
                    setattr(self, attr_name, old_value)
            except Exception:
                pass

            try:
                # If a newly created tab exists at the expected position and is
                # different from the old tab, remove it.
                tabs_now = list(self.nb.tabs())
                for tid in tabs_now:
                    try:
                        fr = self.root.nametowidget(tid)
                    except Exception:
                        continue
                    if fr is not old_frame and tid != str(old_frame):
                        if tab_index == self.nb.index(tid):
                            try:
                                self.nb.forget(tid)
                                fr.destroy()
                            except Exception:
                                pass
                            break
            except Exception:
                pass

            try:
                self.nb.select(old_frame)
            except Exception:
                try:
                    self.nb.select(tab_index)
                except Exception:
                    pass

            try:
                self._install_tab_reset_buttons()
            except Exception:
                pass

            try:
                self.status_var.set(
                    f"TAB RESET failed — {tab_title}: "
                    f"{type(exc).__name__}: {exc}"
                )
            except Exception:
                pass

            try:
                messagebox.showerror(
                    "TAB RESET",
                    f"Could not reset {tab_title}.\n\n"
                    f"{type(exc).__name__}: {exc}",
                    parent=self.root
                )
            except Exception:
                pass


    def _build_ui(self):
        style = ttk.Style()
        try:
            style.theme_use('clam')
        except Exception:
            pass
        top = ttk.Frame(self.root, padding=8)
        top.pack(fill='x')
        top.grid_columnconfigure(0, weight=0)
        top.grid_columnconfigure(1, weight=1)
        top.grid_columnconfigure(2, weight=0)

        ttk.Label(
            top,
            text='CDI / Reconstruction / ROI Workflow',
            font=('Arial', 18, 'bold')
        ).grid(row=0, column=0, sticky='w')

        self.status_var = tk.StringVar(value='Ready.')
        ttk.Label(
            top,
            textvariable=self.status_var,
            foreground='darkgreen',
            anchor='e'
        ).grid(row=0, column=1, sticky='e', padx=(10, 8))

        # Global tab reset control: always visible at the top-right.
        # It operates ONLY on the currently selected tab.
        self.tab_reset_button = tk.Button(
            top,
            text='TAB RESET',
            command=self._reset_current_tab,
            bg='orange',
            activebackground='darkorange',
            fg='black',
            font=('Arial', 9, 'bold'),
            relief='raised',
            bd=2,
            padx=10,
            pady=3
        )
        self.tab_reset_button.grid(
            row=0, column=2, sticky='e'
        )
        self.nb = ttk.Notebook(self.root)
        self.nb.pack(fill='both', expand=True, padx=8, pady=(0, 8))
        self.tab_data = ttk.Frame(self.nb)
        self.tab_recon = ttk.Frame(self.nb)
        self.tab_support = ttk.Frame(self.nb)
        self.tab_roi = ttk.Frame(self.nb)
        self.tab_pixel_calibration = ttk.Frame(self.nb)
        self.tab_proc = ttk.Frame(self.nb)
        self.tab_line_profile = ttk.Frame(self.nb)
        self.tab_particle_density = ttk.Frame(self.nb)
        tab_defs = [
            (self.tab_data, '1. Data / Calibration'),
            (self.tab_recon, '2. Reconstruction'),
            (self.tab_support, '3. Support Mask Optimization'),
            (self.tab_roi, '4. ROI Selection'),
            (self.tab_pixel_calibration, '5. Pixel Calibration'),
            (self.tab_proc, '6. Processing'),
            (self.tab_line_profile, '7. Line Profile'),
            (self.tab_particle_density, '8. Blob Analysis'),
        ]
        for tab, name in tab_defs:
            self.nb.add(tab, text=name)
        self._build_log_tab()
        self._build_data_tab()
        self._build_recon_tab()
        self._build_support_tab()
        self._build_roi_tab()
        self._build_pixel_calibration_tab()
        self._build_proc_tab()
        self._build_line_profile_tab()


    def _build_data_tab(self):
        main = ttk.Panedwindow(self.tab_data, orient='horizontal')
        main.pack(fill='both', expand=True)
        left_host = ttk.Frame(main)
        right = ttk.Frame(main, padding=10)
        main.add(left_host, weight=0)
        main.add(right, weight=1)
        self.data_control_canvas = tk.Canvas(left_host, highlightthickness=0, borderwidth=0, width=445)
        self.data_control_scrollbar = ttk.Scrollbar(left_host, orient='vertical', command=self.data_control_canvas.yview)
        self.data_control_canvas.configure(yscrollcommand=self.data_control_scrollbar.set)
        self.data_control_canvas.pack(side='left', fill='both', expand=True)
        self.data_control_scrollbar.pack(side='right', fill='y')
        left = ttk.Frame(self.data_control_canvas, padding=10)
        self.data_control_window = self.data_control_canvas.create_window((0, 0), window=left, anchor='nw')

        def _data_controls_configure(event=None):
            self.data_control_canvas.configure(scrollregion=self.data_control_canvas.bbox('all'))
            try:
                self.data_control_canvas.itemconfigure(self.data_control_window, width=self.data_control_canvas.winfo_width())
            except Exception:
                pass
        left.bind('<Configure>', _data_controls_configure)
        self.data_control_canvas.bind('<Configure>', _data_controls_configure)

        def _data_control_wheel(event):
            if getattr(event, 'num', None) == 4:
                self.data_control_canvas.yview_scroll(-3, 'units')
            elif getattr(event, 'num', None) == 5:
                self.data_control_canvas.yview_scroll(3, 'units')
            elif getattr(event, 'delta', 0):
                self.data_control_canvas.yview_scroll(int(-event.delta / 120), 'units')
        self.data_control_canvas.bind('<MouseWheel>', _data_control_wheel)
        left.bind('<MouseWheel>', _data_control_wheel)
        self.field_plot_frame = ttk.LabelFrame(right, text='Field Calibration / Raw vs Calibrated', padding=6)
        self.field_plot_frame.pack(fill='both', expand=True, pady=(0, 8))
        self.field_plot_fig = None
        self.field_plot_canvas = None
        ttk.Label(left, text='HDF5 data folder', font=('Arial', 11, 'bold')).pack(anchor='w')
        self.data_dir_var = tk.StringVar()
        ttk.Entry(left, textvariable=self.data_dir_var, width=55).pack(fill='x', pady=4)
        ttk.Button(left, text='Browse data folder', command=self.browse_data_folder).pack(fill='x', pady=4)
        ttk.Label(left, text='Field calibration JSON', font=('Arial', 11, 'bold')).pack(anchor='w', pady=(15, 0))
        self.calib_var = tk.StringVar()
        ttk.Entry(left, textvariable=self.calib_var, width=55).pack(fill='x', pady=4)
        ttk.Button(left, text='Browse calibration JSON', command=self.browse_calibration).pack(fill='x', pady=4)
        ttk.Button(left, text='Load calibration', command=self.load_calibration).pack(fill='x', pady=4)
        ttk.Label(left, text='Mask library (optional)', font=('Arial', 11, 'bold')).pack(anchor='w', pady=(15, 0))
        self.mask_lib_var = tk.StringVar()
        ttk.Entry(left, textvariable=self.mask_lib_var, width=55).pack(fill='x', pady=4)
        mask_row = ttk.Frame(left)
        mask_row.pack(fill='x', pady=4)
        ttk.Button(mask_row, text='Browse mask_lib.py', command=self.browse_mask_library).pack(side='left', expand=True, fill='x', padx=(0, 3))
        ttk.Button(mask_row, text='Load mask_lib', command=self.load_mask_library).pack(side='left', expand=True, fill='x', padx=(3, 0))
        self.mask_status_var = tk.StringVar(value='Mask library: optional — reconstruction does not require it')
        ttk.Label(left, textvariable=self.mask_status_var, wraplength=360).pack(anchor='w', pady=3)
        ttk.Separator(left).pack(fill='x', pady=12)
        ramp_frame = ttk.LabelFrame(left, text='Reconstruction Ramp Direction', padding=6)
        ramp_frame.pack(fill='x', pady=(0, 6))
        ttk.Label(ramp_frame, text='Order:').pack(side='left', padx=(0, 6))
        self.ramp_direction_combo = ttk.Combobox(ramp_frame, textvariable=self.ramp_direction_var, values=['Ramp Down (+ to -)', 'Ramp Up (- to +)'], state='readonly', width=22)
        self.ramp_direction_combo.pack(side='left', fill='x', expand=True)
        self.ramp_direction_combo.bind('<<ComboboxSelected>>', self._ramp_direction_changed)
        ttk.Label(ramp_frame, textvariable=self.ramp_direction_status_var, wraplength=360).pack(anchor='w', pady=(5, 0))
        ttk.Button(left, text='Load ALL reconstructions', command=self.load_all_reconstructions).pack(fill='x', pady=5)
        ttk.Button(left, text='Test HDF5 reconstruction', command=self.test_single_hdf5).pack(fill='x', pady=5)
        ttk.Button(left, text='Load / refresh calibration + reconstructions', command=self.full_load).pack(fill='x', pady=5)
        self.file_count_var = tk.StringVar(value='0 HDF5 files')
        ttk.Label(left, textvariable=self.file_count_var).pack(anchor='w', pady=5)
        self.info_var = tk.StringVar(value='No reconstruction loaded.')
        ttk.Label(left, textvariable=self.info_var, wraplength=360, justify='left').pack(anchor='w', pady=5)
        self.file_list = tk.Listbox(left, height=20, width=58)
        self.file_list.pack(fill='both', expand=False, pady=6)
        self.log_box_data = tk.Text(right, height=35, wrap='word')
        self.log_box_data.pack(fill='both', expand=True)

    def _build_recon_tab(self):
        ctrl = ttk.Frame(self.tab_recon, padding=8)
        ctrl.pack(fill='x')
        ttk.Button(ctrl, text='Show current reconstruction', command=self.show_current_recon).pack(side='left', padx=4)
        ttk.Button(ctrl, text='Show support mask', command=self.show_support).pack(side='left', padx=4)
        ttk.Button(ctrl, text='Optimize support mask', command=lambda: self.nb.select(self.tab_support)).pack(side='left', padx=4)
        ttk.Button(ctrl, text='Field slider', command=self.setup_field_slider).pack(side='left', padx=4)
        ttk.Button(ctrl, text='Sort by field', command=self.sort_by_field).pack(side='left', padx=4)
        ttk.Button(ctrl, text='Reset', command=self.reset_recon_tab).pack(side='left', padx=4)
        ttk.Checkbutton(ctrl, text='Enable Dynamic Zoom', variable=self.recon_dynamic_zoom_var, command=self._toggle_recon_dynamic_zoom).pack(side='left', padx=(12, 4))
        self.field_index_var = tk.IntVar(value=0)
        self.field_slider_var = tk.DoubleVar(value=0)
        ttk.Label(ctrl, text='Index:').pack(side='left', padx=(20, 2))
        self.field_slider = ttk.Scale(ctrl, from_=0, to=100, variable=self.field_slider_var, command=self._slider_index_update, length=350)
        self.field_slider.pack(side='left')
        self.field_label_var = tk.StringVar(value='Field: --')
        ttk.Label(ctrl, textvariable=self.field_label_var).pack(side='left', padx=8)
        recon_outer = ttk.Frame(self.tab_recon)
        recon_outer.pack(fill='both', expand=True, padx=8, pady=6)
        self.recon_view_canvas = tk.Canvas(recon_outer, highlightthickness=0, borderwidth=0)
        self.recon_vscroll = ttk.Scrollbar(recon_outer, orient='vertical', command=self.recon_view_canvas.yview)
        self.recon_hscroll = ttk.Scrollbar(recon_outer, orient='horizontal', command=self.recon_view_canvas.xview)
        self.recon_scroll_inner = ttk.Frame(self.recon_view_canvas)
        self.recon_view_window = self.recon_view_canvas.create_window((0, 0), window=self.recon_scroll_inner, anchor='nw')
        self.recon_view_canvas.configure(yscrollcommand=self.recon_vscroll.set, xscrollcommand=self.recon_hscroll.set)
        self.recon_view_canvas.grid(row=0, column=0, sticky='nsew')
        self.recon_vscroll.grid(row=0, column=1, sticky='ns')
        self.recon_hscroll.grid(row=1, column=0, sticky='ew')
        recon_outer.rowconfigure(0, weight=1)
        recon_outer.columnconfigure(0, weight=1)

        def _recon_update_scrollregion(event=None):
            self.recon_scroll_inner.update_idletasks()
            viewport_w = self.recon_view_canvas.winfo_width()
            viewport_h = self.recon_view_canvas.winfo_height()
            inner_w = self.recon_scroll_inner.winfo_reqwidth()
            inner_h = self.recon_scroll_inner.winfo_reqheight()
            canvas_w = max(viewport_w, inner_w)
            canvas_h = max(viewport_h, inner_h)
            x = max(0, (viewport_w - inner_w) // 2)
            y = max(0, (viewport_h - inner_h) // 2)
            self.recon_view_canvas.coords(self.recon_view_window, x, y)
            self.recon_view_canvas.configure(scrollregion=(0, 0, canvas_w, canvas_h))
        self.recon_scroll_inner.bind('<Configure>', _recon_update_scrollregion)
        self.recon_view_canvas.bind('<Configure>', _recon_update_scrollregion)

        def _recon_mousewheel(event):
            if event.delta:
                self.recon_view_canvas.yview_scroll(int(-event.delta / 120), 'units')
            elif getattr(event, 'num', None) == 4:
                self.recon_view_canvas.yview_scroll(-3, 'units')
            elif getattr(event, 'num', None) == 5:
                self.recon_view_canvas.yview_scroll(3, 'units')
        self.recon_view_canvas.bind('<MouseWheel>', _recon_mousewheel)
        self.recon_scroll_inner.bind('<MouseWheel>', _recon_mousewheel)
        self.recon_canvas_frame = self.recon_scroll_inner

    def _build_roi_tab(self):
        top = ttk.Frame(self.tab_roi, padding=8)
        top.pack(fill='x')
        ttk.Label(top, text='ROI frame index:').pack(side='left')
        self.roi_index_var = tk.IntVar(value=50)
        ttk.Spinbox(top, from_=0, to=0, textvariable=self.roi_index_var, width=8, command=self.refresh_roi_image).pack(side='left', padx=4)
        ttk.Button(top, text='Show ROI', command=self.refresh_roi_image).pack(side='left', padx=4)
        ttk.Button(top, text='Confirm ROI', command=self.confirm_roi).pack(side='left', padx=4)
        ttk.Button(top, text='Clear ROI', command=self.clear_roi).pack(side='left', padx=4)
        self.roi_status_var = tk.StringVar(value='Drag a rectangle over the image. After confirmation, Tab 5 opens automatically.')
        ttk.Label(top, textvariable=self.roi_status_var).pack(side='left', padx=12)
        self.roi_canvas_frame = ttk.Frame(self.tab_roi)
        self.roi_canvas_frame.pack(fill='both', expand=True)

    def _build_pixel_calibration_tab(self):
        """Tab 5 — SEM calibration, verification, and independent ROI calibration.

        Tab 5 owns the calibration state. It does not read Tab 6 output resolution
        or its resampled ROI stack. The only value exported to Tab 6 is
        ``primary_roi_final_nm_per_pixel``.

        Layout: 1 x 3
          1. Scrollable controls
          2. SEM image
          3. Confirmed Primary ROI image from Tab 4
        """
        outer = ttk.Frame(self.tab_pixel_calibration, padding=6)
        outer.pack(fill='both', expand=True)
        outer.grid_rowconfigure(0, weight=1)
        outer.grid_columnconfigure(0, weight=0, minsize=380)
        outer.grid_columnconfigure(1, weight=1, minsize=300)
        outer.grid_columnconfigure(2, weight=1, minsize=300)
        control_box = ttk.LabelFrame(outer, text='PIXEL CALIBRATION — CONTROLS', padding=4)
        control_box.grid(row=0, column=0, sticky='nsew', padx=(0, 5))
        control_box.configure(width=380)
        control_box.grid_propagate(False)
        control_canvas = tk.Canvas(control_box, highlightthickness=0, borderwidth=0)
        control_scroll = ttk.Scrollbar(control_box, orient='vertical', command=control_canvas.yview)
        control_inner = ttk.Frame(control_canvas, padding=6)
        control_window = control_canvas.create_window((0, 0), window=control_inner, anchor='nw')
        control_canvas.configure(yscrollcommand=control_scroll.set)
        control_scroll.pack(side='right', fill='y')
        control_canvas.pack(side='left', fill='both', expand=True)

        def _control_configure(event=None):
            try:
                control_canvas.configure(scrollregion=control_canvas.bbox('all'))
                control_canvas.itemconfigure(control_window, width=max(1, control_canvas.winfo_width()))
            except Exception:
                pass

        def _control_wheel(event):
            try:
                if getattr(event, 'delta', 0):
                    control_canvas.yview_scroll(-3 if event.delta > 0 else 3, 'units')
                elif getattr(event, 'num', None) == 4:
                    control_canvas.yview_scroll(-3, 'units')
                elif getattr(event, 'num', None) == 5:
                    control_canvas.yview_scroll(3, 'units')
            except Exception:
                pass
        control_inner.bind('<Configure>', _control_configure)
        control_canvas.bind('<Configure>', _control_configure)
        for _w in (control_canvas, control_inner):
            _w.bind('<MouseWheel>', _control_wheel, add='+')
            _w.bind('<Button-4>', _control_wheel, add='+')
            _w.bind('<Button-5>', _control_wheel, add='+')

        def section(title):
            lf = ttk.LabelFrame(control_inner, text=title, padding=6)
            lf.pack(fill='x', pady=(0, 8))
            return lf
        sec1 = section('1. SEM IMAGE / KNOWN SCALE')
        ttk.Button(sec1, text='BROWSE / UPLOAD SEM IMAGE', command=self._pixel_cal_browse_sem).pack(fill='x', pady=2)
        ttk.Label(sec1, text='Known scale-bar length (nm):').pack(anchor='w', pady=(7, 1))
        ttk.Entry(sec1, textvariable=self.sem_known_length_nm_var).pack(fill='x', pady=2)
        sec2 = section('DRAWING LINE APPEARANCE')
        ttk.Label(sec2, text='Line width:').pack(anchor='w')
        _lw_combo = ttk.Combobox(sec2, textvariable=self.pixel_cal_line_width_var, values=(1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0), state='readonly')
        _lw_combo.pack(fill='x', pady=(1, 5))
        _lw_combo.bind('<<ComboboxSelected>>', lambda e: self._pixel_cal_update_line_style())
        ttk.Label(sec2, text='Line style:').pack(anchor='w')
        _ls_combo = ttk.Combobox(sec2, textvariable=self.pixel_cal_line_style_var, values=('-', '--', '-.', ':'), state='readonly')
        _ls_combo.pack(fill='x', pady=(1, 5))
        _ls_combo.bind('<<ComboboxSelected>>', lambda e: self._pixel_cal_update_line_style())
        ttk.Label(sec2, text='Line color:').pack(anchor='w')
        _lc_combo = ttk.Combobox(sec2, textvariable=self.pixel_cal_line_color_var, values=('red', 'blue', 'green', 'yellow'), state='readonly')
        _lc_combo.pack(fill='x', pady=(1, 2))
        _lc_combo.bind('<<ComboboxSelected>>', lambda e: self._pixel_cal_update_line_style())
        sec3 = section('2–6. SEM CALIBRATION + VERIFICATION — CLICK 1 + CLICK 2')
        ttk.Button(sec3, text='2. DRAW SEM SCALE LINE', command=self._pixel_cal_start_sem_line).pack(fill='x', pady=2)
        ttk.Button(sec3, text='3. CALCULATE SEM nm/pixel', command=self._pixel_cal_calculate_sem).pack(fill='x', pady=2)
        ttk.Label(sec3, textvariable=self.pixel_cal_sem_result_var, wraplength=330).pack(anchor='w', pady=(5, 3))
        ttk.Button(sec3, text='4. VERIFICATION — DRAW PROFILE LINE', command=self._pixel_cal_start_sem_verification_line).pack(fill='x', pady=2)
        ttk.Label(sec3, textvariable=self.pixel_cal_verification_result_var, wraplength=330).pack(anchor='w', pady=(4, 2))
        ttk.Button(sec3, text='5. CONFIRM VERIFICATION', command=self._pixel_cal_confirm_verification).pack(fill='x', pady=2)
        ttk.Button(sec3, text='6. CONFIRM SEM PIXEL SIZE', command=self._pixel_cal_confirm_sem).pack(fill='x', pady=2)
        sec4 = section('IMAGE ZOOM')
        ttk.Label(sec4, text='SEM image').pack(anchor='w')
        ttk.Button(sec4, text='SEM  RESET ZOOM', command=lambda: self._pixel_cal_reset_zoom('sem')).pack(fill='x', pady=(1, 5))
        ttk.Label(sec4, text='Mouse wheel over SEM: scroll up/down to zoom.\nZoom is centered on the mouse cursor.').pack(anchor='w', pady=(0, 6))
        ttk.Label(sec4, text='Primary ROI image').pack(anchor='w')
        ttk.Button(sec4, text='ROI  RESET ZOOM', command=lambda: self._pixel_cal_reset_zoom('roi')).pack(fill='x', pady=1)
        ttk.Label(sec4, text='Mouse wheel over ROI: scroll up/down to zoom.\nZoom is centered on the mouse cursor.').pack(anchor='w', pady=(3, 0))
        sec5 = section('7–9. PRIMARY ROI CALIBRATION')
        ttk.Button(sec5, text='7. REFRESH PRIMARY ROI FROM TAB 4', command=self._pixel_cal_refresh_primary_roi).pack(fill='x', pady=2)
        ttk.Button(sec5, text='8. DRAW LINE OVER PRIMARY ROI', command=self._pixel_cal_start_roi_line).pack(fill='x', pady=2)
        ttk.Button(sec5, text='9. CALCULATE / STORE FINAL ROI nm/pixel', command=self._pixel_cal_calculate_final).pack(fill='x', pady=2)
        ttk.Label(sec5, textvariable=self.pixel_cal_roi_pixels_var, wraplength=330).pack(anchor='w', pady=(5, 1))
        ttk.Label(sec5, textvariable=self.pixel_cal_roi_length_var, wraplength=330).pack(anchor='w', pady=1)
        ttk.Label(sec5, textvariable=self.pixel_cal_final_var, wraplength=330, font=('Arial', 10, 'bold')).pack(anchor='w', pady=(1, 2))
        direct_row = ttk.Frame(sec5)
        direct_row.pack(fill='x', pady=(1, 3))
        ttk.Label(direct_row, text='Pixel size =').pack(side='left')
        self.pixel_cal_direct_entry = ttk.Entry(direct_row, textvariable=self.pixel_cal_direct_nm_per_pixel_var, width=14)
        self.pixel_cal_direct_entry.pack(side='left', padx=4)
        ttk.Label(direct_row, text='nm/px').pack(side='left')
        ttk.Button(direct_row, text='USE DIRECT', command=self._pixel_cal_apply_direct).pack(side='left', padx=(6, 0))
        ttk.Label(sec5, text='Direct entry overrides calibration and is used for further processing.', wraplength=330).pack(anchor='w', pady=(0, 3))
        ttk.Button(sec5, text='CLEAR CALIBRATION', command=self._pixel_cal_clear).pack(fill='x', pady=2)
        sec6 = section('STATUS')
        ttk.Label(sec6, textvariable=self.pixel_cal_status_var, wraplength=330, justify='left').pack(anchor='w')
        sem_frame = ttk.LabelFrame(outer, text='SEM IMAGE — SCALE / VERIFICATION', padding=3)
        sem_frame.grid(row=0, column=1, sticky='nsew', padx=3)
        self.pixel_cal_sem_canvas_frame = sem_frame
        self.pixel_cal_sem_fig, self.pixel_cal_sem_ax = plt.subplots(figsize=(7, 7), dpi=100)
        self.pixel_cal_sem_fig.subplots_adjust(left=0.08, right=0.98, bottom=0.08, top=0.93)
        self.pixel_cal_sem_canvas = FigureCanvasTkAgg(self.pixel_cal_sem_fig, master=sem_frame)
        self.pixel_cal_sem_canvas.get_tk_widget().pack(fill='both', expand=True)
        self._pixel_cal_sem_click_cid = self.pixel_cal_sem_canvas.mpl_connect('button_press_event', self._pixel_cal_mouse_press)
        self._pixel_cal_sem_motion_cid = self.pixel_cal_sem_canvas.mpl_connect('motion_notify_event', self._pixel_cal_mouse_motion)
        self.pixel_cal_sem_canvas.mpl_connect('scroll_event', lambda e: self._pixel_cal_zoom_scroll('sem', e))
        roi_frame = ttk.LabelFrame(outer, text='PRIMARY ROI FROM TAB 4 — ROI CALIBRATION', padding=3)
        roi_frame.grid(row=0, column=2, sticky='nsew', padx=(3, 0))
        self.pixel_cal_roi_canvas_frame = roi_frame
        self.pixel_cal_roi_fig, self.pixel_cal_roi_ax = plt.subplots(figsize=(7, 7), dpi=100)
        self.pixel_cal_roi_fig.subplots_adjust(left=0.08, right=0.98, bottom=0.08, top=0.93)
        self.pixel_cal_roi_canvas = FigureCanvasTkAgg(self.pixel_cal_roi_fig, master=roi_frame)
        self.pixel_cal_roi_canvas.get_tk_widget().pack(fill='both', expand=True)
        self._pixel_cal_roi_click_cid = self.pixel_cal_roi_canvas.mpl_connect('button_press_event', self._pixel_cal_mouse_press)
        self._pixel_cal_roi_motion_cid = self.pixel_cal_roi_canvas.mpl_connect('motion_notify_event', self._pixel_cal_mouse_motion)
        self.pixel_cal_roi_canvas.mpl_connect('scroll_event', lambda e: self._pixel_cal_zoom_scroll('roi', e))
        self._pixel_cal_sem_line_artist = None
        self._pixel_cal_sem_verification_line_artist = None
        self._pixel_cal_roi_line_artist = None
        self._pixel_cal_mode = None
        self._pixel_cal_press_xy = None
        self._pixel_cal_line_stage = 0
        self._pixel_cal_second_point = None
        self._pixel_cal_refresh_primary_roi()
        self.pixel_cal_status_var.set('Upload an SEM image to begin. After scale-bar calibration, verify a second profile length before confirming SEM pixel size.')

    def _pixel_cal_update_line_style(self):
        color = self.pixel_cal_line_color_var.get()
        style = self.pixel_cal_line_style_var.get()
        width = float(self.pixel_cal_line_width_var.get())
        for artist in (self._pixel_cal_sem_line_artist, self._pixel_cal_sem_verification_line_artist, self._pixel_cal_roi_line_artist):
            if artist is None:
                continue
            try:
                artist.set_color(color)
                artist.set_linestyle(style)
                artist.set_linewidth(width)
            except Exception:
                pass
        try:
            self.pixel_cal_sem_canvas.draw_idle()
        except Exception:
            pass
        try:
            self.pixel_cal_roi_canvas.draw_idle()
        except Exception:
            pass

    def _pixel_cal_tab_changed(self, event=None):
        try:
            if event is not None and event.widget.select() != str(self.tab_pixel_calibration):
                return
            self._pixel_cal_refresh_primary_roi()
        except Exception as exc:
            self.log(f'Pixel Calibration refresh warning: {exc}')

    def _pixel_cal_browse_sem(self):
        fp = filedialog.askopenfilename(parent=self.root, title='Select SEM image', filetypes=[('Image files', '*.png *.jpg *.jpeg *.tif *.tiff *.bmp *.webp'), ('PNG', '*.png'), ('JPEG', '*.jpg *.jpeg'), ('TIFF', '*.tif *.tiff'), ('All files', '*.*')])
        if not fp:
            return
        try:
            img = Image.open(fp)
            if img.mode not in ('L', 'I', 'F'):
                img = img.convert('L')
            arr = np.asarray(img, dtype=float)
            if arr.ndim != 2:
                raise ValueError(f'Expected a 2-D SEM image; got shape {arr.shape}.')
            self.sem_image = arr
            self.sem_image_path = fp
            self.sem_scale_line = None
            self.sem_scale_pixels = None
            self.sem_nm_per_pixel = None
            self.sem_calibration_confirmed = False
            self.sem_verification_line = None
            self.sem_verification_pixels = None
            self.sem_verification_length_nm = None
            self.sem_verification_confirmed = False
            self._pixel_cal_zoom_limits['sem'] = None
            self._pixel_cal_mode = None
            self._pixel_cal_press_xy = None
            self._pixel_cal_line_stage = 0
            self._pixel_cal_second_point = None
            self._pixel_cal_sem_line_artist = None
            self._pixel_cal_sem_verification_line_artist = None
            self.pixel_cal_sem_result_var.set('SEM nm/pixel: --')
            self.pixel_cal_verification_result_var.set('Verification: --')
            self.pixel_cal_final_var.set('FINAL Primary ROI nm/pixel: --')
            self.pixel_cal_status_var.set(f'SEM loaded: {Path(fp).name} | image size = {arr.shape[1]} × {arr.shape[0]} pixels. Draw the known scale-bar line.')
            self._pixel_cal_show_sem_image()
            self.log(f'Pixel Calibration: loaded SEM image {fp} ({arr.shape[1]} x {arr.shape[0]} pixels).')
        except Exception as exc:
            messagebox.showerror('SEM Image', f'Could not load the SEM image.\n\n{exc}', parent=self.root)

    def _pixel_cal_show_sem_image(self):
        """Render the SEM image in native orientation; redraw scale + verification lines."""
        if self.sem_image is None:
            return
        ax = self.pixel_cal_sem_ax
        ax.clear()
        ax.imshow(self.sem_image, cmap='gray', origin='upper', interpolation='nearest', aspect='equal')
        ax.set_title('SEM image — scale bar + verification')
        ax.set_xlabel('X pixel')
        ax.set_ylabel('Y pixel')
        if self._pixel_cal_zoom_limits['sem'] is None:
            ax.set_xlim(-0.5, self.sem_image.shape[1] - 0.5)
            ax.set_ylim(self.sem_image.shape[0] - 0.5, -0.5)
        else:
            (x0, x1), (y0, y1) = self._pixel_cal_zoom_limits['sem']
            ax.set_xlim(x0, x1)
            ax.set_ylim(y0, y1)
        self._pixel_cal_sem_line_artist = None
        self._pixel_cal_sem_verification_line_artist = None
        self._pixel_cal_redraw_sem_line()
        self._pixel_cal_redraw_sem_verification_line()
        self.pixel_cal_sem_fig.tight_layout()
        self.pixel_cal_sem_canvas.draw_idle()

    def _pixel_cal_refresh_primary_roi(self):
        """Render the actual confirmed/support-limited ROI from Tab 4.

        Tab 4 owns the ROI selection.  The red rectangle is only the user's
        selection window; the confirmed ROI image shown here is the supported
        region inside that window, cropped to the supported-region bounding box.
        This keeps Tab 5 independent of Tab 6 output-resolution changes.
        """
        ax = self.pixel_cal_roi_ax
        ax.clear()
        self._pixel_cal_roi_line_artist = None
        frame = getattr(self, 'confirmed_roi_frame', None)
        frame_idx = getattr(self, 'confirmed_roi_frame_index', None)
        if self.recons is None or frame is None or frame_idx is None:
            ax.set_title('Waiting for confirmed Primary ROI from Tab 4')
            ax.text(0.5, 0.5, 'Confirm Primary ROI in Tab 4', ha='center', va='center', transform=ax.transAxes)
            ax.set_xlabel('ROI X pixel')
            ax.set_ylabel('ROI Y pixel')
            self._pixel_cal_roi_display_shape = None
            self._pixel_cal_roi_display_support = None
            self.pixel_cal_roi_canvas.draw_idle()
            return
        try:
            idx = max(0, min(int(frame_idx), len(self.recons) - 1))
            x0 = int(frame['xmin'])
            x1 = int(frame['xmax'])
            y0 = int(frame['ymin'])
            y1 = int(frame['ymax'])
            if x1 < x0 or y1 < y0:
                raise RuntimeError('Invalid confirmed ROI frame.')
            selected = np.asarray(self.recons[idx, y0:y1 + 1, x0:x1 + 1], dtype=float)
            if getattr(self, 'roi_mask', None) is not None:
                support_full = np.asarray(self.roi_mask, dtype=bool)
                support = support_full[y0:y1 + 1, x0:x1 + 1]
            elif getattr(self, 'supportmask2', None) is not None:
                support_full = np.asarray(self.supportmask2, dtype=bool)
                support = support_full[y0:y1 + 1, x0:x1 + 1]
            else:
                raise RuntimeError('No confirmed support mask is available from Tab 4.')
            if support.shape != selected.shape:
                raise RuntimeError(f'Support mask shape {support.shape} does not match ROI frame {selected.shape}.')
            yy, xx = np.where(support)
            if yy.size == 0 or xx.size == 0:
                raise RuntimeError('The selected ROI contains no supported pixels.')
            sy0, sy1 = (int(yy.min()), int(yy.max()))
            sx0, sx1 = (int(xx.min()), int(xx.max()))
            img = selected[sy0:sy1 + 1, sx0:sx1 + 1].copy()
            local_support = support[sy0:sy1 + 1, sx0:sx1 + 1]
            finite = np.isfinite(img) & local_support
            if not np.any(finite):
                raise RuntimeError('Confirmed ROI has no finite pixels inside the supported region.')
            vals = img[finite]
            lo, hi = np.percentile(vals, (1, 99))
            if hi <= lo:
                hi = lo + 1e-12
            disp = np.nan_to_num((img - lo) / (hi - lo), nan=0.0, posinf=1.0, neginf=0.0)
            disp = np.clip(disp, 0.0, 1.0)
            disp[~local_support] = 0.0
            h, w = disp.shape
            ax.imshow(disp, cmap='gray', origin='upper', interpolation='nearest', aspect='equal', vmin=0, vmax=1)
            ax.contour(local_support.astype(float), levels=[0.5], colors='cyan', linewidths=0.9, zorder=15)
            field_text = ''
            if self.real_fields_mT is not None and idx < len(self.real_fields_mT):
                field_text = f' | {self.real_fields_mT[idx]:+.2f} mT'
            ax.set_title(f'PRIMARY ROI FROM TAB 4 | frame {idx + 1}{field_text}\nROI = {w} × {h} px | cyan = supported region')
            ax.set_xlabel('ROI X pixel')
            ax.set_ylabel('ROI Y pixel')
            self._pixel_cal_roi_display_shape = (h, w)
            self._pixel_cal_roi_display_support = local_support.copy()
            self._pixel_cal_roi_origin_full = {'xmin': x0 + sx0, 'xmax': x0 + sx1, 'ymin': y0 + sy0, 'ymax': y0 + sy1, 'confirmed_xmin': x0, 'confirmed_xmax': x1, 'confirmed_ymin': y0, 'confirmed_ymax': y1, 'frame_index': idx}
            self._pixel_cal_zoom_limits['roi'] = None
            ax.set_xlim(-0.5, w - 0.5)
            ax.set_ylim(h - 0.5, -0.5)
            self._pixel_cal_redraw_roi_line()
            self.pixel_cal_roi_fig.tight_layout()
            self.pixel_cal_roi_canvas.draw_idle()
            self.pixel_cal_status_var.set(f'Primary ROI loaded from Tab 4: supported-region crop {w} × {h} px. Tab 5 uses the confirmed Tab-4 ROI and is independent of Tab 6 output resolution.')
        except Exception as exc:
            self._pixel_cal_roi_display_shape = None
            self._pixel_cal_roi_display_support = None
            ax.set_title('Primary ROI could not be displayed')
            ax.text(0.5, 0.5, str(exc), ha='center', va='center', transform=ax.transAxes)
            self.pixel_cal_roi_canvas.draw_idle()

    def _pixel_cal_get_display_image(self, which):
        """Return the actual ndarray displayed in a Pixel Calibration pane."""
        if which == 'sem':
            return self.sem_image
        if which == 'roi':
            if getattr(self, 'roi_mask', None) is None or getattr(self, 'roi_stack', None) is None:
                return None
            try:
                idx = int(getattr(self, 'current_index', 0))
                idx = max(0, min(idx, len(self.recons) - 1))
                return self._get_roi_image(idx)
            except Exception:
                return None
        return None

    def _pixel_cal_zoom_button(self, which, direction):
        ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
        image = self._pixel_cal_get_display_image(which)
        if image is None:
            return
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        cx = 0.5 * (xlim[0] + xlim[1])
        cy = 0.5 * (ylim[0] + ylim[1])
        factor = self._pixel_cal_zoom_factor if direction > 0 else 1.0 / self._pixel_cal_zoom_factor
        hx = 0.5 * abs(xlim[1] - xlim[0]) / factor
        hy = 0.5 * abs(ylim[1] - ylim[0]) / factor
        self._pixel_cal_apply_zoom_limits(which, cx, cy, hx, hy, image.shape[1], image.shape[0])

    def _pixel_cal_zoom_scroll(self, which, event):
        """Cursor-centred wheel zoom without changing the SEM/ROI orientation.

        The Y axis is intentionally descending for image display (native image
        orientation: row 0 at the top).  Zooming therefore has to preserve the
        signed Y span instead of using abs(y1-y0), otherwise the view flips
        vertically after a wheel event.
        """
        if event.inaxes is None:
            return
        ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
        if event.inaxes is not ax or event.xdata is None or event.ydata is None:
            return
        step = getattr(event, 'step', 0)
        if step == 0:
            button = getattr(event, 'button', None)
            if button in ('up', 4):
                step = 1
            elif button in ('down', 5):
                step = -1
        if step == 0:
            return
        image = self._pixel_cal_get_display_image(which)
        if image is None:
            return
        factor = self._pixel_cal_zoom_factor if step > 0 else 1.0 / self._pixel_cal_zoom_factor
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        span_x = x1 - x0
        span_y = y1 - y0
        if span_x == 0 or span_y == 0:
            return
        rel_x = (event.xdata - x0) / span_x
        rel_y = (event.ydata - y0) / span_y
        new_span_x = span_x / factor
        new_span_y = span_y / factor
        nx0 = event.xdata - rel_x * new_span_x
        nx1 = nx0 + new_span_x
        ny0 = event.ydata - rel_y * new_span_y
        ny1 = ny0 + new_span_y
        self._pixel_cal_set_zoom_limits(which, nx0, nx1, ny0, ny1, image.shape[1], image.shape[0])

    def _pixel_cal_set_zoom_limits(self, which, x0, x1, y0, y1, width, height):
        """Clamp zoom limits while preserving each axis direction exactly."""
        full_x0, full_x1 = (-0.5, width - 0.5)
        full_y_top = -0.5
        full_y_bottom = height - 0.5
        x_desc = x0 > x1
        xlo_req, xhi_req = sorted((float(x0), float(x1)))
        full_w = full_x1 - full_x0
        req_w = xhi_req - xlo_req
        if req_w >= full_w:
            xlo, xhi = (full_x0, full_x1)
        else:
            xlo, xhi = (xlo_req, xhi_req)
            if xlo < full_x0:
                shift = full_x0 - xlo
                xlo += shift
                xhi += shift
            if xhi > full_x1:
                shift = xhi - full_x1
                xlo -= shift
                xhi -= shift
            xlo = max(full_x0, xlo)
            xhi = min(full_x1, xhi)
        xlims = (xhi, xlo) if x_desc else (xlo, xhi)
        y_desc = y0 > y1
        ylo_req, yhi_req = sorted((float(y0), float(y1)))
        full_h = full_y_bottom - full_y_top
        req_h = yhi_req - ylo_req
        if req_h >= full_h:
            ylo, yhi = (full_y_top, full_y_bottom)
        else:
            ylo, yhi = (ylo_req, yhi_req)
            if ylo < full_y_top:
                shift = full_y_top - ylo
                ylo += shift
                yhi += shift
            if yhi > full_y_bottom:
                shift = yhi - full_y_bottom
                ylo -= shift
                yhi -= shift
            ylo = max(full_y_top, ylo)
            yhi = min(full_y_bottom, yhi)
        ylims = (yhi, ylo) if y_desc else (ylo, yhi)
        self._pixel_cal_zoom_limits[which] = (xlims, ylims)
        ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
        ax.set_xlim(*xlims)
        ax.set_ylim(*ylims)
        canvas = self.pixel_cal_sem_canvas if which == 'sem' else self.pixel_cal_roi_canvas
        canvas.draw_idle()

    def _pixel_cal_apply_zoom_limits(self, which, cx, cy, hx, hy, width, height):
        ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
        descending = ax.get_ylim()[0] > ax.get_ylim()[1]
        self._pixel_cal_set_zoom_limits(which, cx - hx, cx + hx, cy + hy if descending else cy - hy, cy - hy if descending else cy + hy, width, height)

    def _pixel_cal_reset_zoom(self, which):
        image = self._pixel_cal_get_display_image(which)
        if image is None:
            return
        self._pixel_cal_zoom_limits[which] = None
        ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
        ax.set_xlim(-0.5, image.shape[1] - 0.5)
        ax.set_ylim(image.shape[0] - 0.5, -0.5)
        if which == 'sem':
            self._pixel_cal_redraw_sem_line()
        else:
            self._pixel_cal_redraw_roi_line()
        (self.pixel_cal_sem_canvas if which == 'sem' else self.pixel_cal_roi_canvas).draw_idle()

    def _pixel_cal_redraw_sem_line(self):
        if self.sem_scale_line is None:
            return
        ax = self.pixel_cal_sem_ax
        p0, p1 = self.sem_scale_line
        if self._pixel_cal_sem_line_artist is None or self._pixel_cal_sem_line_artist.axes is not ax:
            self._pixel_cal_sem_line_artist, = ax.plot([p0[0], p1[0]], [p0[1], p1[1]], linewidth=float(self.pixel_cal_line_width_var.get()), linestyle=self.pixel_cal_line_style_var.get(), color=self.pixel_cal_line_color_var.get(), marker='o', markersize=4.5, markerfacecolor=self.pixel_cal_line_color_var.get(), markeredgecolor=self.pixel_cal_line_color_var.get(), zorder=50)
        else:
            self._pixel_cal_sem_line_artist.set_data([p0[0], p1[0]], [p0[1], p1[1]])
            self._pixel_cal_sem_line_artist.set_linewidth(float(self.pixel_cal_line_width_var.get()))
            self._pixel_cal_sem_line_artist.set_linestyle(self.pixel_cal_line_style_var.get())
            self._pixel_cal_sem_line_artist.set_color(self.pixel_cal_line_color_var.get())

    def _pixel_cal_redraw_sem_verification_line(self):
        if self.sem_verification_line is None:
            return
        ax = self.pixel_cal_sem_ax
        p0, p1 = self.sem_verification_line
        if self._pixel_cal_sem_verification_line_artist is None or self._pixel_cal_sem_verification_line_artist.axes is not ax:
            self._pixel_cal_sem_verification_line_artist, = ax.plot([p0[0], p1[0]], [p0[1], p1[1]], linewidth=float(self.pixel_cal_line_width_var.get()), linestyle=self.pixel_cal_line_style_var.get(), color=self.pixel_cal_line_color_var.get(), marker='o', markersize=4.5, markerfacecolor=self.pixel_cal_line_color_var.get(), markeredgecolor=self.pixel_cal_line_color_var.get(), zorder=51)
        else:
            self._pixel_cal_sem_verification_line_artist.set_data([p0[0], p1[0]], [p0[1], p1[1]])
            self._pixel_cal_sem_verification_line_artist.set_linewidth(float(self.pixel_cal_line_width_var.get()))
            self._pixel_cal_sem_verification_line_artist.set_linestyle(self.pixel_cal_line_style_var.get())
            self._pixel_cal_sem_verification_line_artist.set_color(self.pixel_cal_line_color_var.get())

    def _pixel_cal_redraw_roi_line(self):
        if self.primary_roi_scale_line is None:
            return
        ax = self.pixel_cal_roi_ax
        p0, p1 = self.primary_roi_scale_line
        if self._pixel_cal_roi_line_artist is None or self._pixel_cal_roi_line_artist.axes is not ax:
            self._pixel_cal_roi_line_artist, = ax.plot([p0[0], p1[0]], [p0[1], p1[1]], linewidth=float(self.pixel_cal_line_width_var.get()), linestyle=self.pixel_cal_line_style_var.get(), color=self.pixel_cal_line_color_var.get(), marker='o', markersize=4.5, markerfacecolor=self.pixel_cal_line_color_var.get(), markeredgecolor=self.pixel_cal_line_color_var.get(), zorder=50)
        else:
            self._pixel_cal_roi_line_artist.set_data([p0[0], p1[0]], [p0[1], p1[1]])
            self._pixel_cal_roi_line_artist.set_linewidth(float(self.pixel_cal_line_width_var.get()))
            self._pixel_cal_roi_line_artist.set_linestyle(self.pixel_cal_line_style_var.get())
            self._pixel_cal_roi_line_artist.set_color(self.pixel_cal_line_color_var.get())

    def _pixel_cal_start_sem_line(self):
        if self.sem_image is None:
            messagebox.showwarning('Pixel Calibration', 'Upload an SEM image first.', parent=self.root)
            return
        self._pixel_cal_mode = 'sem_scale'
        self._pixel_cal_press_xy = None
        self._pixel_cal_endpoint1 = None
        self._pixel_cal_line_stage = 0
        self._pixel_cal_second_point = None
        self.pixel_cal_status_var.set('SEM scale-bar mode ACTIVE: LEFT-CLICK once to FIX endpoint 1, then LEFT-CLICK a second time to FIX endpoint 2.')

    def _pixel_cal_start_sem_verification_line(self):
        if self.sem_image is None:
            messagebox.showwarning('Pixel Calibration', 'Upload an SEM image first.', parent=self.root)
            return
        if self.sem_nm_per_pixel is None or not np.isfinite(self.sem_nm_per_pixel):
            messagebox.showwarning('Pixel Calibration', 'Calculate the preliminary SEM nm/pixel first.', parent=self.root)
            return
        if self.sem_verification_confirmed:
            messagebox.showinfo('Pixel Calibration', 'SEM verification is already confirmed. Recalculate calibration to start a new verification.', parent=self.root)
            return
        self._pixel_cal_mode = 'sem_verify'
        self._pixel_cal_press_xy = None
        self._pixel_cal_endpoint1 = None
        self._pixel_cal_line_stage = 0
        self._pixel_cal_second_point = None
        self.pixel_cal_status_var.set('VERIFICATION mode ACTIVE: LEFT-CLICK once for endpoint 1, then LEFT-CLICK a second time for endpoint 2.')

    def _pixel_cal_start_roi_line(self):
        if not self.sem_calibration_confirmed or self.sem_nm_per_pixel is None:
            messagebox.showwarning('Pixel Calibration', 'Complete verification and confirm SEM pixel size first.', parent=self.root)
            return
        if getattr(self, 'confirmed_roi_frame', None) is None or self.recons is None:
            messagebox.showwarning('Pixel Calibration', 'First confirm the Primary ROI in Tab 4.', parent=self.root)
            return
        self._pixel_cal_mode = 'roi'
        self._pixel_cal_press_xy = None
        self._pixel_cal_endpoint1 = None
        self._pixel_cal_line_stage = 0
        self._pixel_cal_second_point = None
        self.pixel_cal_status_var.set('PRIMARY ROI mode ACTIVE: LEFT-CLICK once for endpoint 1, then LEFT-CLICK a second time for endpoint 2.')

    @staticmethod
    def _pixel_cal_distance(p0, p1):
        return float(np.hypot(float(p1[0]) - float(p0[0]), float(p1[1]) - float(p0[1])))

    def _pixel_cal_mouse_press(self, event):
        """Strict CLICK-CLICK measurement for Tab 5.

        Interaction is deliberately simple and does not use drag or release:
          1) first LEFT CLICK fixes endpoint 1;
          2) second LEFT CLICK fixes endpoint 2;
          3) completed line is frozen until the DRAW button is pressed again.

        The state is stored in ``_pixel_cal_endpoint1`` so a second click cannot
        be mistaken for a drag/release event.
        """
        if self._pixel_cal_mode is None:
            return
        if getattr(event, 'button', None) != 1:
            return
        if event.xdata is None or event.ydata is None:
            return
        target_ax = self.pixel_cal_sem_ax if self._pixel_cal_mode in ('sem_scale', 'sem_verify') else self.pixel_cal_roi_ax
        if event.inaxes is not target_ax:
            return
        p = (float(event.xdata), float(event.ydata))
        canvas = self.pixel_cal_sem_canvas if target_ax is self.pixel_cal_sem_ax else self.pixel_cal_roi_canvas
        color = self.pixel_cal_line_color_var.get()
        style = self.pixel_cal_line_style_var.get()
        width = float(self.pixel_cal_line_width_var.get())
        if self._pixel_cal_endpoint1 is None:
            self._pixel_cal_endpoint1 = p
            self._pixel_cal_press_xy = p
            self._pixel_cal_line_stage = 1
            old_artist = {'sem_scale': getattr(self, '_pixel_cal_sem_line_artist', None), 'sem_verify': getattr(self, '_pixel_cal_sem_verification_line_artist', None), 'roi': getattr(self, '_pixel_cal_roi_line_artist', None)}.get(self._pixel_cal_mode)
            if old_artist is not None:
                try:
                    old_artist.remove()
                except Exception:
                    pass
            marker_artist, = target_ax.plot([p[0], p[0]], [p[1], p[1]], linestyle=style, linewidth=width, marker='o', markersize=7, markerfacecolor=color, markeredgecolor=color, color=color, zorder=100)
            if self._pixel_cal_mode == 'sem_scale':
                self._pixel_cal_sem_line_artist = marker_artist
                msg = 'ENDPOINT 1 FIXED ✓ — now LEFT-CLICK once at ENDPOINT 2 on the SEM scale bar.'
            elif self._pixel_cal_mode == 'sem_verify':
                self._pixel_cal_sem_verification_line_artist = marker_artist
                msg = 'VERIFICATION endpoint 1 FIXED ✓ — move to endpoint 2; the line follows the mouse. LEFT-CLICK to FIX endpoint 2.'
            else:
                self._pixel_cal_roi_line_artist = marker_artist
                msg = 'ROI endpoint 1 FIXED ✓ — move to endpoint 2; the line follows the mouse. LEFT-CLICK to FIX endpoint 2.'
            self.pixel_cal_status_var.set(msg)
            canvas.draw_idle()
            return
        p0 = self._pixel_cal_endpoint1
        p1 = p
        mode = self._pixel_cal_mode
        pixels = self._pixel_cal_distance(p0, p1)
        old_artist = {'sem_scale': getattr(self, '_pixel_cal_sem_line_artist', None), 'sem_verify': getattr(self, '_pixel_cal_sem_verification_line_artist', None), 'roi': getattr(self, '_pixel_cal_roi_line_artist', None)}.get(mode)
        if old_artist is not None:
            try:
                old_artist.remove()
            except Exception:
                pass
        final_artist, = target_ax.plot([p0[0], p1[0]], [p0[1], p1[1]], linewidth=width, linestyle=style, color=color, zorder=100, marker='o', markersize=7, markerfacecolor=color, markeredgecolor=color)
        if mode == 'sem_scale':
            self._pixel_cal_sem_line_artist = final_artist
        elif mode == 'sem_verify':
            self._pixel_cal_sem_verification_line_artist = final_artist
        else:
            self._pixel_cal_roi_line_artist = final_artist
        self._pixel_cal_second_point = p1
        self._pixel_cal_press_xy = None
        self._pixel_cal_endpoint1 = None
        self._pixel_cal_line_stage = 0
        self._pixel_cal_mode = None
        if pixels <= 0.5:
            self.pixel_cal_status_var.set('Second endpoint is too close to endpoint 1. Press the DRAW button and select two distinct points.')
            canvas.draw_idle()
            return
        if mode == 'sem_scale':
            self.sem_scale_line = (p0, p1)
            self.sem_scale_pixels = pixels
            self.pixel_cal_status_var.set(f'SEM SCALE LINE FIXED ✓ | {pixels:.3f} pixels. Both endpoints locked. Click CALCULATE SEM nm/pixel.')
        elif mode == 'sem_verify':
            self.sem_verification_line = (p0, p1)
            self.sem_verification_pixels = pixels
            if self.sem_nm_per_pixel is not None:
                self.sem_verification_length_nm = float(self.sem_nm_per_pixel) * pixels
                self.pixel_cal_verification_result_var.set(f'Verification profile = {self.sem_verification_length_nm:.6g} nm ({pixels:.3f} px)')
            self.sem_verification_confirmed = False
            self.pixel_cal_status_var.set('SEM VERIFICATION LINE FIXED ✓ — inspect both endpoint markers, then click CONFIRM VERIFICATION.')
        else:
            self.primary_roi_scale_line = (p0, p1)
            self.primary_roi_scale_pixels = pixels
            self.pixel_cal_roi_pixels_var.set(f'Primary ROI line = {pixels:.3f} pixels')
            ref_nm = self.sem_verification_length_nm
            if ref_nm is not None:
                self.pixel_cal_roi_length_var.set(f'Reference physical length = {ref_nm:.6g} nm')
            self.pixel_cal_status_var.set('PRIMARY ROI LINE FIXED ✓ — click CALCULATE / STORE FINAL ROI nm/pixel.')
        canvas.draw_idle()

    def _pixel_cal_mouse_motion(self, event):
        """Dynamic preview: endpoint 1 stays fixed while endpoint 2 follows the mouse."""
        if self._pixel_cal_mode is None or self._pixel_cal_endpoint1 is None:
            return
        if event.xdata is None or event.ydata is None:
            return
        target_ax = self.pixel_cal_sem_ax if self._pixel_cal_mode in ('sem_scale', 'sem_verify') else self.pixel_cal_roi_ax
        if event.inaxes is not target_ax:
            return
        p0 = self._pixel_cal_endpoint1
        p1 = (float(event.xdata), float(event.ydata))
        artist = {'sem_scale': getattr(self, '_pixel_cal_sem_line_artist', None), 'sem_verify': getattr(self, '_pixel_cal_sem_verification_line_artist', None), 'roi': getattr(self, '_pixel_cal_roi_line_artist', None)}.get(self._pixel_cal_mode)
        if artist is None:
            return
        artist.set_data([p0[0], p1[0]], [p0[1], p1[1]])
        canvas = self.pixel_cal_sem_canvas if target_ax is self.pixel_cal_sem_ax else self.pixel_cal_roi_canvas
        canvas.draw_idle()

    def _pixel_cal_mouse_release(self, event):
        return

    def _pixel_cal_calculate_sem(self):
        if self.sem_image is None:
            messagebox.showwarning('Pixel Calibration', 'Upload an SEM image first.', parent=self.root)
            return
        if self.sem_scale_pixels is None or self.sem_scale_pixels <= 0:
            messagebox.showwarning('Pixel Calibration', 'Draw the line over the SEM scale bar first.', parent=self.root)
            return
        try:
            known_nm = float(self.sem_known_length_nm_var.get())
        except Exception:
            messagebox.showerror('Pixel Calibration', 'Known scale length must be a valid number in nm.', parent=self.root)
            return
        if not np.isfinite(known_nm) or known_nm <= 0:
            messagebox.showerror('Pixel Calibration', 'Known scale length must be > 0 nm.', parent=self.root)
            return
        self.sem_nm_per_pixel = known_nm / self.sem_scale_pixels
        self.sem_calibration_confirmed = False
        self.sem_verification_confirmed = False
        self.sem_verification_length_nm = None
        self.pixel_cal_verification_result_var.set('Verification: --')
        self.pixel_cal_sem_result_var.set(f'PRELIMINARY SEM nm/pixel = {self.sem_nm_per_pixel:.9g}')
        self.pixel_cal_status_var.set(f'Preliminary SEM calibration = {known_nm:.6g} nm / {self.sem_scale_pixels:.3f} px = {self.sem_nm_per_pixel:.9g} nm/pixel. Now perform Verification.')

    def _pixel_cal_confirm_verification(self):
        if self.sem_verification_line is None or self.sem_verification_pixels is None or self.sem_verification_length_nm is None:
            messagebox.showwarning('Pixel Calibration', 'Draw the Verification line and obtain its dimension first.', parent=self.root)
            return
        if self.sem_nm_per_pixel is None:
            messagebox.showwarning('Pixel Calibration', 'Calculate the preliminary SEM nm/pixel first.', parent=self.root)
            return
        self.sem_verification_confirmed = True
        self.pixel_cal_verification_result_var.set(f'VERIFIED profile = {self.sem_verification_length_nm:.6g} nm ({self.sem_verification_pixels:.3f} px) ✓')
        self.pixel_cal_status_var.set(f'Verification CONFIRMED: profile dimension = {self.sem_verification_length_nm:.6g} nm. Now click CONFIRM SEM PIXEL SIZE.')

    def _pixel_cal_confirm_sem(self):
        if self.sem_nm_per_pixel is None or not np.isfinite(self.sem_nm_per_pixel):
            messagebox.showwarning('Pixel Calibration', 'Calculate SEM nm/pixel first.', parent=self.root)
            return
        if not self.sem_verification_confirmed or self.sem_verification_length_nm is None:
            messagebox.showwarning('Pixel Calibration', 'Confirm the Verification measurement before confirming SEM pixel size.', parent=self.root)
            return
        self.sem_calibration_confirmed = True
        self.pixel_cal_status_var.set(f'SEM PIXEL SIZE CONFIRMED = {self.sem_nm_per_pixel:.9g} nm/pixel. Now calibrate the Primary ROI using the corresponding profile.')

    def _pixel_cal_calculate_final(self):
        if not self.sem_calibration_confirmed or self.sem_nm_per_pixel is None:
            messagebox.showwarning('Pixel Calibration', 'Confirm the SEM pixel size after Verification first.', parent=self.root)
            return
        if self.primary_roi_scale_pixels is None or self.primary_roi_scale_pixels <= 0:
            messagebox.showwarning('Pixel Calibration', 'Draw a line over the Primary ROI first.', parent=self.root)
            return
        if self.sem_verification_length_nm is None or self.sem_verification_length_nm <= 0:
            messagebox.showwarning('Pixel Calibration', 'A confirmed Verification physical length is required before ROI calibration.', parent=self.root)
            return
        final_value = float(self.sem_verification_length_nm) / float(self.primary_roi_scale_pixels)
        self.primary_roi_final_nm_per_pixel = final_value
        self.pixel_cal_active_nm_per_pixel = final_value
        self.pixel_cal_roi_length_var.set(f'Reference physical length = {self.sem_verification_length_nm:.6g} nm')
        self.pixel_cal_final_var.set(f'FINAL Primary ROI nm/pixel = {final_value:.9g}')
        self.pixel_cal_status_var.set(f'FINAL ROI CALIBRATION STORED = {final_value:.9g} nm/pixel. This value is the only calibration value read by Tab 6.')
        self.log(f'Pixel Calibration: SEM={self.sem_nm_per_pixel:.9g} nm/pixel; verified SEM profile={self.sem_verification_length_nm:.6g} nm; ROI profile={self.primary_roi_scale_pixels:.3f} px; Final ROI={final_value:.9g} nm/pixel.')

    def _pixel_cal_apply_direct(self):
        try:
            value = float(str(self.pixel_cal_direct_nm_per_pixel_var.get()).strip())
        except Exception:
            messagebox.showerror('Pixel Calibration', 'Enter a valid positive Pixel size in nm/px.', parent=self.root)
            return
        if not np.isfinite(value) or value <= 0:
            messagebox.showerror('Pixel Calibration', 'Pixel size must be a positive finite value in nm/px.', parent=self.root)
            return
        self.pixel_cal_active_nm_per_pixel = value
        self.primary_roi_final_nm_per_pixel = value
        self.pixel_cal_final_var.set(f'FINAL Primary ROI nm/pixel = {value:.9g}  [DIRECT ENTRY]')
        self.pixel_cal_status_var.set(f'DIRECT PIXEL SIZE ACTIVE = {value:.9g} nm/pixel. This value is used for further processing.')
        try:
            self._tab6_update_physical_axes_from_calibration()
        except Exception:
            pass
        try:
            self._line_profile_update_image(preserve_lines=True)
        except Exception:
            pass
        try:
            self._line_profile_redraw_profiles()
        except Exception:
            pass

    def _pixel_cal_clear(self):
        """Clear calibration/measurement values and lines, but preserve selected images/ROI."""
        self.pixel_cal_direct_nm_per_pixel_var.set('')
        self.pixel_cal_active_nm_per_pixel = None
        self.sem_scale_line = None
        self.sem_scale_pixels = None
        self.sem_nm_per_pixel = None
        self.sem_calibration_confirmed = False
        self.sem_verification_line = None
        self.sem_verification_pixels = None
        self.sem_verification_length_nm = None
        self.sem_verification_confirmed = False
        self.primary_roi_scale_line = None
        self.primary_roi_scale_pixels = None
        self.primary_roi_final_nm_per_pixel = None
        self._pixel_cal_mode = None
        self._pixel_cal_press_xy = None
        self._pixel_cal_endpoint1 = None
        self._pixel_cal_line_stage = 0
        self._pixel_cal_second_point = None
        self._pixel_cal_endpoint1 = None
        for attr in ('_pixel_cal_sem_line_artist', '_pixel_cal_sem_verification_line_artist', '_pixel_cal_roi_line_artist'):
            artist = getattr(self, attr, None)
            if artist is not None:
                try:
                    artist.remove()
                except Exception:
                    pass
                setattr(self, attr, None)
        self.pixel_cal_sem_result_var.set('SEM nm/pixel: --')
        self.pixel_cal_verification_result_var.set('Verification: --')
        self.pixel_cal_roi_pixels_var.set('Primary ROI line: -- pixels')
        self.pixel_cal_roi_length_var.set('Reference physical length: -- nm')
        self.pixel_cal_final_var.set('FINAL Primary ROI nm/pixel: --')
        self.pixel_cal_status_var.set('Calibration values and measurement lines cleared. SEM image and Primary ROI are retained.')
        if self.sem_image is not None:
            self.pixel_cal_sem_canvas.draw_idle()
        if getattr(self, 'confirmed_roi_frame', None) is not None:
            self.pixel_cal_roi_canvas.draw_idle()

    def _build_proc_tab(self):
        self._add_export_folder_controls(self.tab_proc)
        ctrl = ttk.Frame(self.tab_proc, padding=8)
        ctrl.pack(fill='x')
        self.proc_idx_var = tk.DoubleVar(value=0)
        ttk.Label(ctrl, text='Frame No. / Equivalent Field').pack(side='left')
        self.proc_slider = ttk.Scale(ctrl, from_=0, to=100, variable=self.proc_idx_var, command=self._proc_slider_update, length=300)
        self.proc_slider.pack(side='left', padx=5)
        self.proc_field_var = tk.StringVar(value='--')
        ttk.Label(ctrl, textvariable=self.proc_field_var).pack(side='left', padx=5)
        ttk.Label(ctrl, text='Brightness').pack(side='left', padx=(20, 2))
        self.brightness_var = tk.DoubleVar(value=0.0)
        ttk.Scale(ctrl, from_=-1, to=1, variable=self.brightness_var, command=lambda _: self.update_processing_view(), length=130).pack(side='left')
        ttk.Label(ctrl, text='Contrast').pack(side='left', padx=(15, 2))
        self.contrast_var = tk.DoubleVar(value=1.0)
        ttk.Scale(ctrl, from_=0.2, to=3, variable=self.contrast_var, command=lambda _: self.update_processing_view(), length=130).pack(side='left')
        ttk.Label(ctrl, text='Filter').pack(side='left', padx=(15, 2))
        self.filter_var = tk.StringVar(value='None')
        cb = ttk.Combobox(ctrl, textvariable=self.filter_var, values=['None', 'Gaussian', 'Low-pass', 'High-pass', 'Band-pass'], width=12, state='readonly')
        cb.pack(side='left')
        cb.bind('<<ComboboxSelected>>', lambda e: self.update_processing_view())
        ttk.Button(ctrl, text='FFT', command=self.show_fft).pack(side='left', padx=5)
        ttk.Button(ctrl, text='Blob analysis', command=self.blob_analysis).pack(side='left', padx=5)
        ttk.Button(ctrl, text='Reset', command=self.reset_processing).pack(side='left', padx=5)
        ttk.Button(ctrl, text='Export PNG', command=lambda: self.export_processed('png')).pack(side='left', padx=5)
        self.proc_canvas_frame = ttk.Frame(self.tab_proc)
        self.proc_canvas_frame.pack(fill='both', expand=True)

    def _build_log_tab(self):
        if self.log_text is not None:
            try:
                self.log_text.destroy()
            except Exception:
                pass
        self.log_text = tk.Text(self.tab_log, wrap='word')
        self.log_text.pack(fill='both', expand=True, padx=8, pady=8)

        def clear_log():
            self._log_messages.clear()
            self.log_text.delete('1.0', 'end')
        ttk.Button(self.tab_log, text='Clear log', command=clear_log).pack(pady=5)
        if self._log_messages:
            self.log_text.insert('end', '\n'.join(self._log_messages) + '\n')
            self.log_text.see('end')

    def log(self, msg):
        text = str(msg)
        print(text)
        if not hasattr(self, '_log_messages'):
            self._log_messages = []
        self._log_messages.append(text)
        widget = getattr(self, 'log_text', None)
        if widget is not None:
            try:
                widget.insert('end', text + '\n')
                widget.see('end')
            except tk.TclError:
                self.log_text = None
        status_var = getattr(self, 'status_var', None)
        if status_var is not None:
            try:
                status_var.set(text[:180])
            except tk.TclError:
                pass
        try:
            self.root.update_idletasks()
        except Exception:
            pass

    def clear_canvas_frame(self, frame):
        for w in frame.winfo_children():
            w.destroy()
        for k, fig in list(self._figs.items()):
            if fig is not None:
                plt.close(fig)
        self._figs.clear()

    def embed_figure(self, frame, fig, key='main'):
        self.clear_canvas_frame(frame)
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True)
        self._figs[key] = fig
        return canvas

    def browse_data_folder(self):
        folder = filedialog.askdirectory(parent=self.root, title='Select HDF5 Data Folder')
        if not folder:
            return
        self.data_dir = Path(folder)
        self.data_dir_var.set(str(self.data_dir))
        self.hdf5_files = sorted(list(self.data_dir.glob('*.h5')) + list(self.data_dir.glob('*.hdf5')))
        self.reconstruction_files = []
        self.reconstruction_paths = []
        self.reconstruction_raw_fields = np.asarray([], dtype=float)
        self._discover_reconstruction_files()
        self._update_ramp_direction_status(select_default=True)
        self.file_list.delete(0, 'end')
        recon_set = {str(Path(fp).resolve()) for fp in self.reconstruction_paths}
        for i, f in enumerate(self.hdf5_files):
            tag = ' [RECON]' if str(Path(f).resolve()) in recon_set else ' [RAW/OTHER]'
            self.file_list.insert('end', f'{i}: {f.name}{tag}')
        self.file_count_var.set(f'{len(self.hdf5_files)} HDF5 files | {len(self.reconstruction_paths)} reconstruction files')
        self.log(f'Found {len(self.hdf5_files)} HDF5 files; identified {len(self.reconstruction_paths)} CDI reconstruction files.')
        if self.reconstruction_paths:
            self.roi_index_var.set(0)
            self.proc_slider.configure(to=max(0, len(self.reconstruction_paths) - 1))
            self.field_slider_var.set(0)
            self.info_var.set(f'{len(self.reconstruction_paths)} reconstruction files found. Click LOAD ALL RECONSTRUCTIONS.')
        else:
            self.info_var.set('No CDI reconstruction files found. Expected HDF5 datasets: p_pc and n_pc.')

    @staticmethod
    def _find_hdf5_dataset(h5, name):
        """Return a dataset named *name* whether it is top-level or nested."""
        if name in h5:
            return h5[name]
        found = []

        def visitor(path, obj):
            if isinstance(obj, h5py.Dataset) and path.split('/')[-1] == name:
                found.append(path)
        h5.visititems(visitor)
        if not found:
            return None
        return h5[found[0]]

    @staticmethod
    def _normalize_reconstruction_array(array, dataset_name='reconstruction'):
        arr = np.asarray(array)
        arr = np.real(arr).astype(np.float64, copy=False)
        arr = np.squeeze(arr)
        if arr.ndim != 2:
            raise ValueError(f'{dataset_name} must resolve to a 2-D image after squeezing singleton axes; got {arr.shape}')
        return arr

    @staticmethod
    def _normalize_complex_reconstruction_array(array, dataset_name='complex reconstruction'):
        arr = np.asarray(array, dtype=np.complex128)
        arr = np.squeeze(arr)
        if arr.ndim != 2:
            raise ValueError(f'{dataset_name} must resolve to a 2-D complex image after squeezing singleton axes; got {arr.shape}')
        return arr

    def _discover_reconstruction_files(self):
        """Find CDI reconstruction HDF5 files robustly by dataset content."""
        if self.data_dir is None:
            return []
        items = []
        for fp in self.hdf5_files:
            name = fp.name
            if name.startswith('Data_ImId_'):
                continue
            try:
                with h5py.File(fp, 'r') as f:
                    p_ds = self._find_hdf5_dataset(f, 'p_pc')
                    n_ds = self._find_hdf5_dataset(f, 'n_pc')
                    if p_ds is None or n_ds is None:
                        continue
                    if p_ds.shape != n_ds.shape:
                        self.log(f'Skipping {name}: p_pc shape {p_ds.shape} != n_pc shape {n_ds.shape}')
                        continue
                    squeezed_shape = tuple((dim for dim in p_ds.shape if int(dim) != 1))
                    if len(squeezed_shape) != 2:
                        self.log(f'Skipping {name}: p_pc/n_pc resolve to shape {p_ds.shape}; expected one 2-D image (singleton axes are allowed).')
                        continue
            except Exception as exc:
                self.log(f'Skipping unreadable HDF5 {name}: {type(exc).__name__}: {exc}')
                continue
            field = None
            patterns = ['cdi_field[=:_-]([-+]?\\d*\\.?\\d+(?:[eE][-+]?\\d+)?)', 'field[=:_-]([-+]?\\d*\\.?\\d+(?:[eE][-+]?\\d+)?)']
            for pat in patterns:
                m = re.search(pat, name, flags=re.IGNORECASE)
                if m:
                    try:
                        field = float(m.group(1))
                        break
                    except ValueError:
                        pass
            if field is None:
                field = float(len(items))
                self.log(f'No field value found in {name}; using sequential raw field index {field:g}.')
            items.append((field, fp))
        items.sort(key=lambda item: item[0])
        self.reconstruction_files = items
        self.reconstruction_paths = [fp for _, fp in items]
        self.reconstruction_raw_fields = np.asarray([field for field, _ in items], dtype=float)
        return items

    def browse_calibration(self):
        f = filedialog.askopenfilename(parent=self.root, title='Select Field Calibration JSON', filetypes=[('JSON', '*.json'), ('All files', '*.*')])
        if f:
            self.field_calib_file = Path(f)
            self.calib_var.set(str(f))

    def load_calibration(self):
        if not self.field_calib_file:
            messagebox.showwarning('Calibration', 'Select the calibration JSON first.')
            return False
        with open(self.field_calib_file, 'r', encoding='utf-8') as fh:
            d = json.load(fh)
        cur = np.asarray(d['current'], dtype=float)
        fld = np.asarray(d['field'], dtype=float)
        valid = np.isfinite(cur) & np.isfinite(fld)
        cur, fld = (cur[valid], fld[valid])
        order = np.argsort(cur)
        cur, fld = (cur[order], fld[order])
        cur, unique_idx = np.unique(cur, return_index=True)
        fld = fld[unique_idx]
        self.calib_current, self.calib_field_mT = (cur, fld)
        self.field_interpolator = interp1d(cur, fld, kind='linear', bounds_error=False, fill_value='extrapolate')
        self.log(f'Calibration loaded: {len(cur)} points; field {fld.min():.2f} to {fld.max():.2f} mT')
        self._plot_field_calibration()
        return True

    def _ordered_frame_field_axes(self):
        """Return reconstruction frame numbers and fields in the active ramp order.

        X1 = frame number in acquisition/reconstruction order.
        X2 = calibrated field mapped onto the same X1 coordinate.
        This is intentionally derived from the already ordered reconstruction
        stack so Tabs 2 onward inherit the selected ramp without re-sorting their
        analysis data independently.
        """
        if self.real_fields_mT is None:
            return (np.array([], dtype=float), np.array([], dtype=float))
        x1 = np.arange(1, len(self.real_fields_mT) + 1, dtype=float)
        x2 = np.asarray(self.real_fields_mT, dtype=float)
        return (x1, x2)

    def _plot_field_calibration(self):
        if self.field_plot_canvas is not None:
            try:
                self.field_plot_canvas.get_tk_widget().destroy()
            except Exception:
                pass
        if self.field_plot_fig is not None:
            try:
                plt.close(self.field_plot_fig)
            except Exception:
                pass
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), dpi=100)
        self.field_plot_fig = fig
        if self.calib_current is not None and self.calib_field_mT is not None:
            axes[0].plot(self.calib_current, self.calib_field_mT, 'o-', markersize=4, linewidth=1.5, label='Calibration data')
            xx = np.linspace(float(np.min(self.calib_current)), float(np.max(self.calib_current)), 500)
            if self.field_interpolator is not None:
                axes[0].plot(xx, self.field_interpolator(xx), '-', linewidth=2, label='Interpolation')
            axes[0].set_title('Raw magnet calibration')
            axes[0].set_xlabel('Current (A)')
            axes[0].set_ylabel('Field (mT)')
            axes[0].grid(True, linestyle=':', alpha=0.7)
            axes[0].legend(fontsize=8)
        else:
            axes[0].set_title('Raw magnet calibration')
            axes[0].text(0.5, 0.5, 'Load calibration JSON', ha='center', va='center', transform=axes[0].transAxes)
            axes[0].set_axis_off()
        if self.raw_fields is not None and self.real_fields_mT is not None:
            idx, _ = self._ordered_frame_field_axes()
            idx0 = idx - 1.0
            axes[1].plot(idx0, self.raw_fields, 'o-', ms=3, lw=1.0, label='Raw cdi_field')
            axes[1].plot(idx0, self.real_fields_mT, 'o-', ms=3, lw=1.0, label='Calibrated field')
            axes[1].set_title('Raw vs calibrated reconstruction field')
            axes[1].set_xlabel('Reconstruction index')
            axes[1].set_ylabel('Field (mT)')
            axes[1].grid(True, linestyle=':', alpha=0.7)
            axes[1].legend(fontsize=8)
        else:
            axes[1].set_title('Raw vs calibrated reconstruction field')
            axes[1].text(0.5, 0.5, 'Load reconstructions', ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_axis_off()
        fig.tight_layout()
        self.field_plot_canvas = FigureCanvasTkAgg(fig, master=self.field_plot_frame)
        self.field_plot_canvas.draw()
        self.field_plot_canvas.get_tk_widget().pack(fill='both', expand=True)

    def browse_mask_library(self):
        filename = filedialog.askopenfilename(parent=self.root, title='Select mask_lib.py', filetypes=[('Python files', '*.py'), ('All files', '*.*')])
        if not filename:
            return
        self.mask_lib_path = Path(filename)
        self.mask_lib_var.set(str(self.mask_lib_path))
        self.mask_status_var.set('Mask library selected — click Load mask_lib')
        self.log(f'Mask library selected: {self.mask_lib_path}')

    def _fallback_scale_mask(self, mask, target_shape):
        """Exact fallback equivalent to the scale_mask() found in the original notebook."""
        mask = np.asarray(mask, dtype=float)
        target_shape = tuple(np.asarray(target_shape, dtype=int))
        scale_factor = np.array(target_shape) / np.array(mask.shape)
        if scale_factor[0] != scale_factor[1]:
            self.log('Warning: mask scaling is different for both axes!')
        mask_rescaled = zoom(mask, scale_factor)
        mask_rescaled[mask_rescaled <= 0.5] = 0
        mask_rescaled[mask_rescaled > 0.5] = 1
        return mask_rescaled.astype(bool)

    def _install_compatible_mask_lib(self, reason=None):
        """Create a local mask_lib-compatible object exposing the required scale_mask()."""
        self.mask_lib = types.SimpleNamespace(scale_mask=self._fallback_scale_mask)
        if reason:
            self.log(f'Using embedded mask_lib.scale_mask() compatibility mode: {reason}')
        else:
            self.log('Using embedded mask_lib.scale_mask() compatibility mode.')
        self.mask_status_var.set('✓ mask_lib compatible scale_mask loaded')
        return True

    def load_mask_library(self):
        if self.mask_lib_path is None:
            messagebox.showwarning('Mask Library', 'Please browse and select mask_lib.py first.')
            return False
        if not self.mask_lib_path.exists():
            messagebox.showerror('Mask Library', f'File does not exist:\n{self.mask_lib_path}')
            return False
        try:
            library_dir = str(self.mask_lib_path.parent)
            if library_dir not in sys.path:
                sys.path.insert(0, library_dir)
            spec = importlib.util.spec_from_file_location('mask_lib', str(self.mask_lib_path))
            if spec is None or spec.loader is None:
                raise ImportError(f'Could not create import specification for {self.mask_lib_path}')
            module = importlib.util.module_from_spec(spec)
            sys.modules['mask_lib'] = module
            spec.loader.exec_module(module)
            if not hasattr(module, 'scale_mask'):
                module.scale_mask = self._fallback_scale_mask
                self.log('mask_lib.py loaded, but scale_mask() was missing; embedded scale_mask() attached.')
            self.mask_lib = module
            self.mask_status_var.set('✓ full mask_lib loaded')
            self.log(f'Loaded mask library: {self.mask_lib_path}')
            self.log(f'scale_mask: {self.mask_lib.scale_mask}')
            return True
        except ModuleNotFoundError as exc:
            missing = getattr(exc, 'name', '') or str(exc)
            if missing == 'CCI_core':
                return self._install_compatible_mask_lib('CCI_core is unavailable')
            return self._install_compatible_mask_lib(f'missing dependency: {missing}')
        except Exception as exc:
            self.log(f'Full mask_lib load failed: {exc}')
            return self._install_compatible_mask_lib(type(exc).__name__)

    def _scale_support_mask(self, mask, target_shape):
        if self.mask_lib is None:
            self.mask_lib = types.SimpleNamespace(scale_mask=self._fallback_scale_mask)
            self.mask_status_var.set('Fallback scale_mask active')
        try:
            scaled = np.asarray(self.mask_lib.scale_mask(mask, target_shape), dtype=bool)
            if scaled.shape != tuple(target_shape):
                raise ValueError(f'scale_mask returned {scaled.shape}, expected {tuple(target_shape)}')
            return scaled
        except Exception as exc:
            self.log(f'mask_lib.scale_mask failed: {exc}; using fallback.')
            return self._fallback_scale_mask(mask, target_shape)

    def test_single_hdf5(self):
        """Diagnostic reconstruction from one HDF5 using only p_pc and n_pc.
        mask_lib/supportmask are optional for this test.
        """
        fp = filedialog.askopenfilename(parent=self.root, title='Select ONE CDI reconstruction HDF5', filetypes=[('HDF5 files', '*.h5 *.hdf5'), ('All files', '*.*')])
        if not fp:
            return
        fp = Path(fp)
        try:
            with h5py.File(fp, 'r') as f:
                if 'p_pc' not in f or 'n_pc' not in f:
                    raise KeyError('The selected file must contain p_pc and n_pc.')
                p_pc = np.asarray(f['p_pc'][:], dtype=np.complex128)
                n_pc = np.asarray(f['n_pc'][:], dtype=np.complex128)
            if p_pc.shape != n_pc.shape:
                raise ValueError(f'p_pc shape {p_pc.shape} != n_pc shape {n_pc.shape}')
            complex_reconstruction = p_pc - n_pc
            reconstruction = np.real(complex_reconstruction)
            imaginary = np.imag(complex_reconstruction)
            image = reconstruction[0] if reconstruction.ndim == 3 else reconstruction
            imag_image = imaginary[0] if imaginary.ndim == 3 else imaginary
            amplitude = np.abs(image)
            phase = np.angle(complex_reconstruction[0] if complex_reconstruction.ndim == 3 else complex_reconstruction)
            finite = np.isfinite(image)
            if np.any(finite):
                lo, hi = np.percentile(image[finite], (1, 99))
                if hi <= lo:
                    hi = lo + 1e-12
            else:
                lo, hi = (0, 1)
            fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=100)
            axes[0, 0].imshow(np.real(p_pc[0] if p_pc.ndim == 3 else p_pc), cmap='gray', origin='lower')
            axes[0, 0].set_title('p_pc (Real)')
            axes[0, 1].imshow(np.real(n_pc[0] if n_pc.ndim == 3 else n_pc), cmap='gray', origin='lower')
            axes[0, 1].set_title('n_pc (Real)')
            axes[0, 2].imshow(image, cmap='gray', origin='lower', vmin=lo, vmax=hi)
            axes[0, 2].set_title('Real Reconstruction')
            axes[1, 0].imshow(imag_image, cmap='gray', origin='lower')
            axes[1, 0].set_title('Imaginary Reconstruction')
            axes[1, 1].imshow(amplitude, cmap='gray', origin='lower')
            axes[1, 1].set_title('Amplitude')
            vals = image[finite]
            axes[1, 2].hist(vals.ravel(), bins=100)
            axes[1, 2].set_title('Real reconstruction histogram')
            axes[1, 2].set_xlabel('Intensity')
            for ax in axes.flat:
                ax.axis('image' if ax is not axes[1, 2] else 'auto')
            fig.suptitle(f'Single HDF5 reconstruction test — {fp.name}', fontsize=13)
            fig.tight_layout()
            self.embed_figure(self.recon_canvas_frame, fig, key='single_test')
            self.nb.select(self.tab_recon)
            self.log(f'Single HDF5 test: {fp.name} | p_pc={p_pc.shape}, n_pc={n_pc.shape}, reconstruction={reconstruction.shape}')
        except Exception as exc:
            messagebox.showerror('Single HDF5 reconstruction test', str(exc), parent=self.root)
            self.log(f'Single HDF5 test failed: {exc}')

    def full_load(self):
        if self.field_calib_file:
            try:
                self.load_calibration()
            except Exception as exc:
                self.log(f'Calibration load failed: {type(exc).__name__}: {exc}; continuing with raw reconstruction field values.')
        return self.load_all_reconstructions()

    def _parse_field(self, filename):
        m = re.search('cdi_field=([-+]?\\d*\\.?\\d+)', Path(filename).name)
        if m is None:
            raise ValueError(f"Could not find 'cdi_field=' in {filename}")
        return float(m.group(1))

    def _ramp_direction_changed(self, event=None):
        """Update the displayed dataset-order status without loading data."""
        self._update_ramp_direction_status(select_default=False)

    @staticmethod
    def _natural_path_key(path_obj):
        """Natural filename sort so frame/index numbers are ordered numerically."""
        name = Path(path_obj).name.lower()
        return [int(part) if part.isdigit() else part for part in re.split('(\\d+)', name)]

    def _infer_uploaded_dataset_direction(self):
        """Infer acquisition direction from the uploaded data-file sequence.

        This deliberately uses the original uploaded HDF5 file ordering rather
        than the field-sorted reconstruction list, so the inferred direction
        reflects the dataset sequence before this application applies any
        user-selected ramp ordering.
        """
        recon_set = {str(Path(p).resolve()) for p in getattr(self, 'reconstruction_paths', []) or []}
        paths = [Path(p) for p in getattr(self, 'hdf5_files', []) or [] if str(Path(p).resolve()) in recon_set]
        paths = sorted(paths, key=self._natural_path_key)
        pairs = []
        for fp in paths:
            name = fp.name
            field = None
            for pat in ('cdi_field[=:_-]([-+]?\\d*\\.?\\d+(?:[eE][-+]?\\d+)?)', 'field[=:_-]([-+]?\\d*\\.?\\d+(?:[eE][-+]?\\d+)?)'):
                m = re.search(pat, name, flags=re.IGNORECASE)
                if m:
                    try:
                        field = float(m.group(1))
                        break
                    except ValueError:
                        pass
            if field is not None and np.isfinite(field):
                pairs.append(field)
        if len(pairs) < 2:
            return None
        diffs = np.diff(np.asarray(pairs, dtype=float))
        diffs = diffs[np.isfinite(diffs) & (np.abs(diffs) > 1e-12)]
        if diffs.size == 0:
            return None
        pos = int(np.count_nonzero(diffs > 0))
        neg = int(np.count_nonzero(diffs < 0))
        if pos > neg:
            return 'Ramp Up (- to +)'
        if neg > pos:
            return 'Ramp Down (+ to -)'
        return None

    def _update_ramp_direction_status(self, select_default=False):
        """Describe detected field coverage/order and optionally choose a dataset-based default."""
        raw = np.asarray(getattr(self, 'reconstruction_raw_fields', np.asarray([], dtype=float)), dtype=float)
        raw = raw[np.isfinite(raw)]
        if raw.size == 0:
            self.ramp_direction_status_var.set('Dataset direction: no field values detected yet.')
            return
        fmin = float(np.min(raw))
        fmax = float(np.max(raw))
        inferred = self._infer_uploaded_dataset_direction()
        if select_default and inferred is not None:
            self.ramp_direction_var.set(inferred)
        detected_text = inferred if inferred is not None else 'unable to determine'
        self.ramp_direction_status_var.set(f'Field range: {fmin:+.3f} to {fmax:+.3f} mT | Dataset sequence: {detected_text} | Load order: {self.ramp_direction_var.get()}')

    def _apply_ramp_direction_order(self):
        """Sort discovered reconstruction files in the selected physical field direction."""
        items = list(getattr(self, 'reconstruction_files', []) or [])
        if not items:
            return
        direction = self.ramp_direction_var.get().strip()
        if direction not in ('Ramp Down (+ to -)', 'Ramp Up (- to +)'):
            direction = 'Ramp Up (- to +)'
            self.ramp_direction_var.set(direction)
        descending = direction == 'Ramp Down (+ to -)'
        items.sort(key=lambda item: float(item[0]), reverse=descending)
        self.reconstruction_files = items
        self.reconstruction_paths = [fp for _, fp in items]
        self.reconstruction_raw_fields = np.asarray([float(field) for field, _ in items], dtype=float)
        self._update_ramp_direction_status(select_default=False)

    def load_all_reconstructions(self):
        """Load all CDI reconstructions found in Tab 1."""
        if self.data_dir is None:
            messagebox.showwarning('Data', 'Select an HDF5 data folder first.', parent=self.root)
            return False
        self._discover_reconstruction_files()
        if not self.reconstruction_paths:
            messagebox.showerror('No CDI reconstructions', 'No readable HDF5 files containing both p_pc and n_pc were found.\n\nCheck the selected folder and the HDF5 dataset names.', parent=self.root)
            self.log('ERROR: no readable CDI reconstruction HDF5 files found.')
            return False
        self._apply_ramp_direction_order()
        self.log(f'Loading {len(self.reconstruction_paths)} CDI reconstructions in {self.ramp_direction_var.get()} order.')
        self.status_var.set('Loading reconstructions...')
        self.root.update_idletasks()
        recons = []
        complex_recons = []
        support = None
        raw_fields = []
        failures = []
        for i, fp in enumerate(self.reconstruction_paths):
            try:
                with h5py.File(fp, 'r') as f:
                    p_ds = self._find_hdf5_dataset(f, 'p_pc')
                    n_ds = self._find_hdf5_dataset(f, 'n_pc')
                    if p_ds is None or n_ds is None:
                        raise KeyError('p_pc and/or n_pc dataset not found')
                    if p_ds.shape != n_ds.shape:
                        raise ValueError(f'p_pc shape {p_ds.shape} != n_pc shape {n_ds.shape}')
                    complex_reconstruction = self._normalize_complex_reconstruction_array(p_ds[()] - n_ds[()], 'p_pc - n_pc')
                    reconstruction = np.real(complex_reconstruction).astype(np.float64, copy=False)
                    recons.append(reconstruction)
                    complex_recons.append(complex_reconstruction)
                    support_ds = self._find_hdf5_dataset(f, 'supportmask')
                    if support is None and support_ds is not None:
                        support = np.asarray(support_ds[...], dtype=bool)
                raw_field = float(self.reconstruction_raw_fields[i])
                raw_fields.append(raw_field)
                self.log(f'Loaded {i + 1}/{len(self.reconstruction_paths)}: {fp.name} | raw field={raw_field:+.6f} | shape={reconstruction.shape}')
            except Exception as exc:
                failures.append((fp.name, f'{type(exc).__name__}: {exc}'))
                self.log(f'FAILED {fp.name}: {type(exc).__name__}: {exc}')
        if not recons:
            messagebox.showerror('Reconstruction Load Error', 'All detected reconstruction files failed to load.\n\n' + '\n'.join((f'{name}: {err}' for name, err in failures[:8])), parent=self.root)
            return False
        shapes = [r.shape for r in recons]
        if len(set(shapes)) != 1:
            messagebox.showerror('Reconstruction Shape Error', 'Loaded reconstruction files do not have identical shapes.\n\n' + '\n'.join((f'{i + 1}: {sh}' for i, sh in enumerate(shapes[:10]))), parent=self.root)
            self.log(f'Reconstruction shape mismatch: {shapes}')
            return False
        self.recons = np.stack(recons, axis=0)
        self.complex_recons = np.stack(complex_recons, axis=0)
        self.real_recons = np.real(self.complex_recons).astype(np.float64, copy=False)
        self.imag_recons = np.imag(self.complex_recons).astype(np.float64, copy=False)
        self.amplitude_recons = np.abs(self.complex_recons).astype(np.float64, copy=False)
        self.phase_recons = np.angle(self.complex_recons).astype(np.float64, copy=False)
        self.supportmask = support
        target_shape = self.recons.shape[1:3]
        if support is None:
            self.supportmask2 = np.ones(target_shape, dtype=bool)
            self.mask_status_var.set('Support mask not present — using full image support')
            self.log('No supportmask found; reconstruction loaded without mask_lib.')
        else:
            try:
                self.supportmask2 = self._scale_support_mask(support, target_shape)
            except Exception as exc:
                self.log(f'Support-mask scaling failed: {type(exc).__name__}: {exc}; using full image support.')
                self.supportmask2 = np.ones(target_shape, dtype=bool)
        self.raw_fields = np.asarray(raw_fields, dtype=float)
        if self.field_interpolator is not None:
            try:
                self.real_fields_mT = np.asarray(self.field_interpolator(self.raw_fields), dtype=float)
            except Exception as exc:
                self.log(f'Field calibration failed: {type(exc).__name__}: {exc}; using raw field values.')
                self.real_fields_mT = self.raw_fields.copy()
        else:
            self.real_fields_mT = self.raw_fields.copy()
        self._plot_field_calibration()
        support_label = 'full image' if support is None else str(self.supportmask2.shape)
        self.info_var.set(f'Reconstructions: {self.recons.shape}; Support: {support_label}; Fields: {len(self.real_fields_mT)}')
        self.roi_index_var.set(min(50, len(self.recons) - 1))
        self.proc_slider.configure(to=max(0, len(self.recons) - 1))
        if hasattr(self, 'roi2_slider'):
            self.roi2_slider.configure(from_=0, to=max(0, len(self.recons) - 1), state='disabled')
            self.roi2_idx_var.set(0)
        self.roi_status_var.set('Ready for ROI selection.')
        self.current_index = 0
        self.field_slider_var.set(0)
        try:
            self.field_slider.configure(to=max(0, len(self.recons) - 1))
        except Exception:
            pass
        if self.real_fields_mT is not None and len(self.real_fields_mT):
            self.field_label_var.set(f'Field: {self.real_fields_mT[0]:+.2f} mT')
        elif self.raw_fields is not None and len(self.raw_fields):
            self.field_label_var.set(f'Field: {self.raw_fields[0]:+.2f}')
        self.log(f'✓ Loaded reconstruction stack: {self.recons.shape} | success={len(recons)} | failed={len(failures)}')
        if hasattr(self, '_roi2_sync_controls'):
            self._roi2_sync_controls()
        if hasattr(self, '_multi_roi_sync_controls'):
            self._multi_roi_sync_controls()
        if hasattr(self, '_multi_fit_sync_controls'):
            self._multi_fit_sync_controls()
        self.nb.select(self.tab_recon)
        self.show_current_recon()
        return True

    def sort_by_field(self):
        if self.recons is None:
            return
        direction = getattr(self, 'ramp_direction_var', None)
        direction = direction.get() if direction is not None else 'Ramp Up (- to +)'
        descending = direction == 'Ramp Down (+ to -)'
        order = np.argsort(self.real_fields_mT)
        if descending:
            order = order[::-1]
        self.recons = self.recons[order]
        if self.complex_recons is not None:
            self.complex_recons = self.complex_recons[order]
        if self.real_recons is not None:
            self.real_recons = self.real_recons[order]
        if self.imag_recons is not None:
            self.imag_recons = self.imag_recons[order]
        if self.amplitude_recons is not None:
            self.amplitude_recons = self.amplitude_recons[order]
        if self.phase_recons is not None:
            self.phase_recons = self.phase_recons[order]
        self.raw_fields = self.raw_fields[order]
        self.real_fields_mT = self.real_fields_mT[order]
        self.log('Reconstruction stack sorted by calibrated field.')
        self.refresh_roi_image()
        self.show_current_recon()

    def _slider_index_update(self, val):
        """Update selected field/frame without causing recursive redraws."""
        if self.recons is None:
            return
        try:
            idx = int(round(float(val)))
        except (TypeError, ValueError):
            idx = int(round(float(self.field_slider_var.get())))
        idx = max(0, min(idx, len(self.recons) - 1))
        self.current_index = idx
        if self.real_fields_mT is not None and idx < len(self.real_fields_mT):
            self.field_label_var.set(f'Field: {self.real_fields_mT[idx]:+.2f} mT')
        elif self.raw_fields is not None and idx < len(self.raw_fields):
            self.field_label_var.set(f'Field: {self.raw_fields[idx]:+.2f}')
        if self._slider_redraw_job is not None:
            try:
                self.root.after_cancel(self._slider_redraw_job)
            except Exception:
                pass
            self._slider_redraw_job = None
        if hasattr(self, 'recon_view_canvas'):
            self._slider_redraw_job = self.root.after(80, self._redraw_recon_from_slider)

    def _redraw_recon_from_slider(self):
        """Deferred redraw for the currently selected Tab-2 frame."""
        self._slider_redraw_job = None
        if self.recons is None:
            return
        self.show_current_recon()

    def show_current_recon(self):
        """Display Real / Imaginary / Amplitude / Phase in a centered
        2x2 figure fixed at figsize=(10, 10), inside a scrollable area.
        """
        try:
            if self.recons is None:
                self.field_label_var.set('Field: -- | Load reconstructions first')
                self.log('Tab 2: no reconstruction stack loaded.')
                return
            n_frames = len(self.recons)
            idx = max(0, min(int(self.current_index), n_frames - 1))
            self.current_index = idx
            complex_img = None
            if getattr(self, 'complex_recons', None) is not None and len(self.complex_recons) == n_frames and (idx < len(self.complex_recons)):
                complex_img = np.asarray(self.complex_recons[idx], dtype=np.complex128)
            if complex_img is None:
                paths = getattr(self, 'reconstruction_paths', [])
                if paths and idx < len(paths):
                    fp = Path(paths[idx])
                    with h5py.File(fp, 'r') as f:
                        p_ds = self._find_hdf5_dataset(f, 'p_pc')
                        n_ds = self._find_hdf5_dataset(f, 'n_pc')
                        if p_ds is None or n_ds is None:
                            raise KeyError(f'Selected HDF5 does not contain p_pc and n_pc:\n{fp}')
                        p_arr = np.asarray(p_ds[()], dtype=np.complex128)
                        n_arr = np.asarray(n_ds[()], dtype=np.complex128)
                    if p_arr.shape != n_arr.shape:
                        raise ValueError(f'p_pc shape {p_arr.shape} != n_pc shape {n_arr.shape}')
                    complex_img = np.squeeze(p_arr - n_arr)
            if complex_img is None:
                complex_img = np.asarray(self.recons[idx], dtype=np.float64).astype(np.complex128)
            complex_img = np.squeeze(complex_img)
            if complex_img.ndim != 2:
                raise ValueError(f'Selected reconstruction must be 2-D after squeezing; got {complex_img.shape}')
            real_img = np.real(complex_img)
            imag_img = np.imag(complex_img)
            amplitude_img = np.abs(complex_img)
            phase_img = np.angle(complex_img)

            def robust_limits(image, symmetric=False):
                finite = image[np.isfinite(image)]
                if finite.size == 0:
                    return (-1.0, 1.0)
                if symmetric:
                    vmax = float(np.percentile(np.abs(finite), 99))
                    if not np.isfinite(vmax) or vmax <= 0:
                        vmax = float(np.max(np.abs(finite)))
                    if not np.isfinite(vmax) or vmax <= 0:
                        vmax = 1.0
                    return (-vmax, vmax)
                vmin, vmax = np.percentile(finite, [1, 99])
                if not np.isfinite(vmin) or not np.isfinite(vmax):
                    vmin = float(np.min(finite))
                    vmax = float(np.max(finite))
                if vmax <= vmin:
                    vmax = vmin + 1e-12
                return (float(vmin), float(vmax))
            real_vmin, real_vmax = robust_limits(real_img, symmetric=True)
            imag_vmin, imag_vmax = robust_limits(imag_img, symmetric=True)
            amp_vmin, amp_vmax = robust_limits(amplitude_img, symmetric=False)
            fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=100)
            im0 = axes[0, 0].imshow(real_img, cmap='gray', origin='lower', vmin=real_vmin, vmax=real_vmax, interpolation='nearest', aspect='equal')
            axes[0, 0].set_title('1. Real Reconstruction')
            axes[0, 0].set_xlabel('X pixel')
            axes[0, 0].set_ylabel('Y pixel')
            fig.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04, label='Real part')
            im1 = axes[0, 1].imshow(imag_img, cmap='gray', origin='lower', vmin=imag_vmin, vmax=imag_vmax, interpolation='nearest', aspect='equal')
            axes[0, 1].set_title('2. Imaginary Reconstruction')
            axes[0, 1].set_xlabel('X pixel')
            axes[0, 1].set_ylabel('Y pixel')
            fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04, label='Imaginary part')
            im2 = axes[1, 0].imshow(amplitude_img, cmap='gray', origin='lower', vmin=amp_vmin, vmax=amp_vmax, interpolation='nearest', aspect='equal')
            axes[1, 0].set_title('3. Amplitude $|p_{pc}-n_{pc}|$')
            axes[1, 0].set_xlabel('X pixel')
            axes[1, 0].set_ylabel('Y pixel')
            fig.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04, label='Amplitude')
            im3 = axes[1, 1].imshow(phase_img, cmap='twilight', origin='lower', vmin=-np.pi, vmax=np.pi, interpolation='nearest', aspect='equal')
            axes[1, 1].set_title('4. Phase $\\arg(p_{pc}-n_{pc})$')
            axes[1, 1].set_xlabel('X pixel')
            axes[1, 1].set_ylabel('Y pixel')
            phase_cbar = fig.colorbar(im3, ax=axes[1, 1], fraction=0.046, pad=0.04)
            phase_cbar.set_label('Phase (rad)')
            if self.supportmask2 is not None:
                try:
                    support = np.asarray(self.supportmask2, dtype=float)
                    if support.shape == real_img.shape:
                        for ax in axes.ravel():
                            ax.contour(support, levels=[0.5], colors='red', linewidths=0.8)
                except Exception as exc:
                    self.log(f'Tab 2 support contour skipped: {exc}')
            field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None and idx < len(self.real_fields_mT) else float(self.raw_fields[idx]) if self.raw_fields is not None and idx < len(self.raw_fields) else float(idx)
            self.field_label_var.set(f'Field: {field:+.2f} mT')
            fig.suptitle(f'Reconstruction {idx + 1}/{n_frames} | Field = {field:+.2f} mT', fontsize=13)
            fig.subplots_adjust(left=0.07, right=0.94, bottom=0.07, top=0.9, wspace=0.24, hspace=0.24)
            self._recon_axes = axes
            self._recon_panel_images = [(axes[0, 0], real_img, '1. Real Reconstruction'), (axes[0, 1], imag_img, '2. Imaginary Reconstruction'), (axes[1, 0], amplitude_img, '3. Amplitude'), (axes[1, 1], phase_img, '4. Phase')]
            old_holder = getattr(self, '_recon_figure_holder', None)
            if old_holder is not None:
                try:
                    old_holder.destroy()
                except Exception:
                    pass
            old_fig = self._figs.pop('recon', None)
            if old_fig is not None:
                try:
                    plt.close(old_fig)
                except Exception:
                    pass
            self._recon_figure_holder = ttk.Frame(self.recon_canvas_frame)
            self._recon_figure_holder.pack(anchor='center', padx=12, pady=12)
            canvas = FigureCanvasTkAgg(fig, master=self._recon_figure_holder)
            canvas.draw()
            canvas.get_tk_widget().pack(fill='none', expand=False)
            self._recon_canvas = canvas
            self._figs['recon'] = fig
            if self.recon_dynamic_zoom_var.get():
                self._bind_recon_dynamic_zoom()
            self.recon_scroll_inner.update_idletasks()
            self.recon_view_canvas.update_idletasks()
            viewport_w = self.recon_view_canvas.winfo_width()
            viewport_h = self.recon_view_canvas.winfo_height()
            inner_w = self.recon_scroll_inner.winfo_reqwidth()
            inner_h = self.recon_scroll_inner.winfo_reqheight()
            canvas_w = max(viewport_w, inner_w)
            canvas_h = max(viewport_h, inner_h)
            x = max(0, (viewport_w - inner_w) // 2)
            y = max(0, (viewport_h - inner_h) // 2)
            self.recon_view_canvas.coords(self.recon_view_window, x, y)
            self.recon_view_canvas.configure(scrollregion=(0, 0, canvas_w, canvas_h))
            if inner_w > viewport_w:
                self.recon_view_canvas.xview_moveto(0.0)
            if inner_h > viewport_h:
                self.recon_view_canvas.yview_moveto(0.0)
            self.log(f'Tab 2 displayed frame {idx + 1}/{n_frames} | field={field:+.2f} mT | shape={real_img.shape}')
        except Exception as exc:
            self.field_label_var.set('Field: -- | ERROR')
            self.log(f'Tab 2 display failed: {type(exc).__name__}: {exc}')
            messagebox.showerror('Tab 2 Reconstruction Display', f'Could not display the selected reconstruction.\\n\\n{type(exc).__name__}: {exc}\\n\\nCheck Tab 1 and the selected HDF5 files.', parent=self.root)

    def reset_recon_tab(self):
        """Reset Tab-2 viewing state without reloading or modifying the data."""
        if self.recons is None or len(self.recons) == 0:
            self.current_index = 0
            self.field_slider_var.set(0)
            self.field_label_var.set('Field: --')
            return
        if getattr(self, '_slider_redraw_job', None) is not None:
            try:
                self.root.after_cancel(self._slider_redraw_job)
            except Exception:
                pass
            self._slider_redraw_job = None
        self.current_index = 0
        self.field_slider_var.set(0)
        if self.real_fields_mT is not None and len(self.real_fields_mT):
            self.field_label_var.set(f'Field: {self.real_fields_mT[0]:+.2f} mT')
        elif self.raw_fields is not None and len(self.raw_fields):
            self.field_label_var.set(f'Field: {self.raw_fields[0]:+.2f}')
        else:
            self.field_label_var.set('Field: --')
        self.nb.select(self.tab_recon)
        self.show_current_recon()
        self.log('Tab 2 reset: returned to first reconstruction frame.')

    def _toggle_recon_dynamic_zoom(self):
        """Enable/disable the live mouse-position zoom preview for Tab 2."""
        if self.recon_dynamic_zoom_var.get():
            self._bind_recon_dynamic_zoom()
            self.log('Tab 2 Dynamic Zoom: ENABLED.')
        else:
            self._unbind_recon_dynamic_zoom()
            self._hide_recon_zoom_popup()
            self.log('Tab 2 Dynamic Zoom: DISABLED.')

    def _unbind_recon_dynamic_zoom(self):
        """Disconnect previously registered Matplotlib mouse callbacks."""
        canvas = getattr(self, '_recon_canvas', None)
        if canvas is None:
            self._recon_zoom_bindings = []
            return
        for cid in getattr(self, '_recon_zoom_bindings', []):
            try:
                canvas.mpl_disconnect(cid)
            except Exception:
                pass
        self._recon_zoom_bindings = []

    def _bind_recon_dynamic_zoom(self):
        """Attach hover callbacks to all four Tab-2 image axes."""
        self._unbind_recon_dynamic_zoom()
        canvas = getattr(self, '_recon_canvas', None)
        axes = getattr(self, '_recon_axes', None)
        if canvas is None or axes is None:
            return
        cid = canvas.mpl_connect('motion_notify_event', self._recon_dynamic_zoom_motion)
        self._recon_zoom_bindings = [cid]

    def _hide_recon_zoom_popup(self):
        popup = getattr(self, '_recon_zoom_popup', None)
        if popup is not None:
            try:
                popup.withdraw()
            except Exception:
                pass
        self._recon_zoom_active_ax = None

    def _show_recon_zoom_popup(self, image, x, y, title, source_ax=None):
        """Show/update a small zoomed preview centered on (x, y)."""
        if image is None or x is None or y is None:
            return
        h, w = image.shape[:2]
        xi = int(round(x))
        yi = int(round(y))
        half = min(60, max(5, int(round(min(h, w) * 0.08))))
        x0 = max(0, xi - half)
        x1 = min(w, xi + half + 1)
        y0 = max(0, yi - half)
        y1 = min(h, yi + half + 1)
        patch = np.asarray(image[y0:y1, x0:x1])
        popup = getattr(self, '_recon_zoom_popup', None)
        if popup is None or not popup.winfo_exists():
            popup = tk.Toplevel(self.root)
            popup.title('Dynamic Zoom Preview')
            popup.geometry('420x420')
            popup.resizable(False, False)
            popup.attributes('-topmost', True)
            top = ttk.Frame(popup, padding=(6, 4))
            top.pack(fill='x')
            self._recon_zoom_title_var = tk.StringVar(value='Dynamic Zoom Preview')
            ttk.Label(top, textvariable=self._recon_zoom_title_var, font=('Arial', 10, 'bold')).pack(side='left')
            ttk.Button(top, text='Close', command=self._hide_recon_zoom_popup).pack(side='right')
            fig, ax = plt.subplots(figsize=(4.0, 4.0), dpi=100)
            fig.subplots_adjust(left=0.08, right=0.98, bottom=0.07, top=0.9)
            self._recon_zoom_fig = fig
            self._recon_zoom_axes = ax
            self._recon_zoom_img_artist = ax.imshow(patch, cmap='gray', origin='lower', interpolation='nearest', aspect='equal')
            ax.set_xlabel('X pixel')
            ax.set_ylabel('Y pixel')
            ax.set_title(title)
            self._recon_zoom_canvas = FigureCanvasTkAgg(fig, master=popup)
            self._recon_zoom_canvas.draw()
            self._recon_zoom_canvas.get_tk_widget().pack(fill='both', expand=True, padx=4, pady=(0, 4))
            self._recon_zoom_popup = popup
        else:
            ax = self._recon_zoom_axes
            self._recon_zoom_img_artist = ax.images[0]
            try:
                popup.deiconify()
                popup.attributes('-topmost', True)
                popup.lift()
            except Exception:
                pass
        if source_ax is not None and source_ax.images:
            source_im = source_ax.images[0]
            try:
                vmin, vmax = source_im.get_clim()
                self._recon_zoom_img_artist.set_clim(vmin, vmax)
                self._recon_zoom_img_artist.set_cmap(source_im.get_cmap())
            except Exception:
                pass
        self._recon_zoom_img_artist.set_data(patch)
        self._recon_zoom_img_artist.set_extent((x0 - 0.5, x1 - 0.5, y0 - 0.5, y1 - 0.5))
        ax = self._recon_zoom_axes
        ax.set_xlim(x0 - 0.5, x1 - 0.5)
        ax.set_ylim(y0 - 0.5, y1 - 0.5)
        ax.set_title(f'{title}\nCenter: X={xi}, Y={yi}')
        if hasattr(self, '_recon_zoom_title_var'):
            self._recon_zoom_title_var.set(f'Dynamic Zoom — {title}')
        self._recon_zoom_active_ax = source_ax
        self._recon_zoom_canvas.draw_idle()
        try:
            px = self.root.winfo_pointerx()
            py = self.root.winfo_pointery()
            popup = self._recon_zoom_popup
            popup.update_idletasks()
            popup_w = popup.winfo_width()
            popup_h = popup.winfo_height()
            screen_w = popup.winfo_screenwidth()
            screen_h = popup.winfo_screenheight()
            offset = 22
            left = px + offset
            top = py + offset
            if left + popup_w > screen_w:
                left = px - popup_w - offset
            if top + popup_h > screen_h:
                top = py - popup_h - offset
            left = max(0, left)
            top = max(0, top)
            popup.geometry(f'+{int(left)}+{int(top)}')
            popup.deiconify()
        except Exception:
            try:
                self._recon_zoom_popup.deiconify()
            except Exception:
                pass

    def _recon_dynamic_zoom_motion(self, event):
        """Update zoom preview while hovering over any Tab-2 image."""
        if not self.recon_dynamic_zoom_var.get():
            return
        axes = getattr(self, '_recon_axes', None)
        panel_images = getattr(self, '_recon_panel_images', None)
        if axes is None or panel_images is None:
            return
        active_ax = None
        active_image = None
        active_title = None
        for ax, image, title in panel_images:
            if event.inaxes is ax:
                active_ax = ax
                active_image = image
                active_title = title
                break
        if active_ax is None or event.xdata is None or event.ydata is None:
            self._hide_recon_zoom_popup()
            return
        self._show_recon_zoom_popup(active_image, event.xdata, event.ydata, active_title, source_ax=active_ax)

    def show_support(self):
        """Open original and scaled support masks in a separate window."""
        if self.supportmask is None and self.supportmask2 is None:
            messagebox.showwarning('Support Mask', 'No support mask is available.', parent=self.root)
            return
        existing = getattr(self, '_support_popup', None)
        if existing is not None:
            try:
                if existing.winfo_exists():
                    existing.deiconify()
                    existing.lift()
                    existing.focus_force()
                    return
            except Exception:
                pass
        popup = tk.Toplevel(self.root)
        popup.title('Support Masks — Original vs Scaled')
        popup.geometry('1500x850')
        popup.minsize(1000, 650)
        top = ttk.Frame(popup, padding=8)
        top.pack(fill='x')
        ttk.Label(top, text='Support Mask Comparison', font=('Arial', 13, 'bold')).pack(side='left')
        original = np.asarray(self.supportmask, dtype=bool) if self.supportmask is not None else None
        scaled = np.asarray(self.supportmask2, dtype=bool) if self.supportmask2 is not None else None
        fig, axes = plt.subplots(1, 2, figsize=(14, 7), dpi=100)
        if original is not None:
            ax0 = axes[0]
            ax0.imshow(original, cmap='gray', origin='lower', interpolation='nearest', aspect='equal')
            ax0.contour(original.astype(float), levels=[0.5], colors='red', linewidths=1.0)
            ax0.set_title(f'Original Support Mask\n{original.shape[1]} × {original.shape[0]} px | {int(original.sum()):,} pixels')
            ax0.set_xlabel('X pixel')
            ax0.set_ylabel('Y pixel')
        else:
            axes[0].text(0.5, 0.5, 'Original support unavailable', ha='center', va='center', transform=axes[0].transAxes)
            axes[0].axis('off')
        if scaled is not None:
            ax1 = axes[1]
            ax1.imshow(scaled, cmap='gray', origin='lower', interpolation='nearest', aspect='equal')
            ax1.contour(scaled.astype(float), levels=[0.5], colors='red', linewidths=1.0)
            ax1.set_title(f'Scaled Support Mask\n{scaled.shape[1]} × {scaled.shape[0]} px | {int(scaled.sum()):,} pixels')
            ax1.set_xlabel('X pixel')
            ax1.set_ylabel('Y pixel')
        else:
            axes[1].text(0.5, 0.5, 'Scaled support unavailable', ha='center', va='center', transform=axes[1].transAxes)
            axes[1].axis('off')
        fig.suptitle('Original Support Mask vs Scaled Support Mask', fontsize=14)
        fig.subplots_adjust(left=0.035, right=0.975, bottom=0.075, top=0.87, wspace=0.22)
        canvas = FigureCanvasTkAgg(fig, master=popup)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True, padx=8, pady=(0, 8))
        bottom = ttk.Frame(popup, padding=(8, 0, 8, 8))
        bottom.pack(fill='x')
        original_info = f'Original: {original.shape[1]}×{original.shape[0]} ({int(original.sum()):,} pixels)' if original is not None else 'Original: unavailable'
        scaled_info = f'Scaled: {scaled.shape[1]}×{scaled.shape[0]} ({int(scaled.sum()):,} pixels)' if scaled is not None else 'Scaled: unavailable'
        ttk.Label(bottom, text=f'{original_info}    |    {scaled_info}').pack(side='left')
        ttk.Button(bottom, text='Close', command=self._close_support_popup).pack(side='right')
        self._support_popup = popup
        self._support_popup_fig = fig
        self._support_popup_canvas = canvas
        popup.protocol('WM_DELETE_WINDOW', self._close_support_popup)
        popup.lift()
        popup.focus_force()

    def _close_support_popup(self):
        """Close the separate support-mask window."""
        popup = getattr(self, '_support_popup', None)
        try:
            if getattr(self, '_support_popup_fig', None) is not None:
                plt.close(self._support_popup_fig)
        except Exception:
            pass
        try:
            if popup is not None and popup.winfo_exists():
                popup.destroy()
        except Exception:
            pass
        self._support_popup = None
        self._support_popup_fig = None
        self._support_popup_canvas = None

    def setup_field_slider(self):
        self.nb.select(self.tab_recon)
        self.show_current_recon()

    def _build_support_tab(self):
        """Build support-mask optimization with a fully scrollable control panel."""
        self.support_threshold_var = tk.DoubleVar(value=0.12)
        self.support_consensus_var = tk.DoubleVar(value=0.5)
        self.support_min_size_var = tk.IntVar(value=500)
        self.support_close_var = tk.IntVar(value=3)
        self.support_dilate_var = tk.IntVar(value=3)
        self.support_sigma_var = tk.DoubleVar(value=1.0)
        self.support_mode_var = tk.StringVar(value='Median')
        self.support_seed_var = tk.StringVar(value='Original support')
        self.support_start_var = tk.IntVar(value=0)
        self.support_end_var = tk.IntVar(value=100)
        self.support_source_var = tk.StringVar(value='Generate from reconstruction')
        self._uploaded_support_mask = None
        self._uploaded_support_path = None
        self.support_status_var = tk.StringVar(value='Load the 101 reconstructions first.')
        self.support_original_pixels_var = tk.StringVar(value='Original: --')
        self.support_candidate_pixels_var = tk.StringVar(value='Candidate: --')
        main = ttk.Frame(self.tab_support, padding=8)
        main.pack(fill='both', expand=True)
        left = ttk.Frame(main)
        left.pack(side='left', fill='both', expand=True, padx=(0, 8))
        self.support_fig, self.support_axes = plt.subplots(2, 2, figsize=(10, 8), dpi=100)
        self.support_canvas = FigureCanvasTkAgg(self.support_fig, master=left)
        self.support_canvas.draw()
        self.support_canvas.get_tk_widget().pack(fill='both', expand=True)
        panel = ttk.Frame(main, width=440)
        panel.pack(side='right', fill='y')
        panel.pack_propagate(False)
        self.support_ctrl_canvas = tk.Canvas(panel, highlightthickness=0, borderwidth=0)
        self.support_ctrl_scrollbar = ttk.Scrollbar(panel, orient='vertical', command=self.support_ctrl_canvas.yview)
        self.support_ctrl_canvas.configure(yscrollcommand=self.support_ctrl_scrollbar.set)
        self.support_ctrl_scrollbar.pack(side='right', fill='y')
        self.support_ctrl_canvas.pack(side='left', fill='both', expand=True)
        self.support_ctrl_inner = ttk.Frame(self.support_ctrl_canvas, padding=(6, 4, 10, 12))
        self.support_ctrl_window = self.support_ctrl_canvas.create_window((0, 0), window=self.support_ctrl_inner, anchor='nw')

        def _support_update_scrollregion(event=None):
            self.support_ctrl_canvas.configure(scrollregion=self.support_ctrl_canvas.bbox('all'))

        def _support_resize_inner(event):
            self.support_ctrl_canvas.itemconfigure(self.support_ctrl_window, width=event.width)
        self.support_ctrl_inner.bind('<Configure>', _support_update_scrollregion)
        self.support_ctrl_canvas.bind('<Configure>', _support_resize_inner)

        def _support_mousewheel(event):
            if event.delta:
                self.support_ctrl_canvas.yview_scroll(int(-event.delta / 120), 'units')
            elif getattr(event, 'num', None) == 4:
                self.support_ctrl_canvas.yview_scroll(-3, 'units')
            elif getattr(event, 'num', None) == 5:
                self.support_ctrl_canvas.yview_scroll(3, 'units')
        self.support_ctrl_canvas.bind('<MouseWheel>', _support_mousewheel)
        self.support_ctrl_inner.bind('<MouseWheel>', _support_mousewheel)
        ttk.Label(self.support_ctrl_inner, text='CONSENSUS SUPPORT GENERATOR', font=('Arial', 12, 'bold')).pack(anchor='w', pady=(2, 4))
        ttk.Label(self.support_ctrl_inner, text='Full controls are available by scrolling.\\nPipeline: normalize → smooth → threshold → clean → consensus → final morphology.', wraplength=390, justify='left').pack(anchor='w', pady=(0, 10))
        source_box = ttk.LabelFrame(self.support_ctrl_inner, text='0. MASK SOURCE — choose one', padding=8)
        source_box.pack(fill='x', pady=5)
        ttk.Radiobutton(source_box, text='Generate support from reconstruction stack', variable=self.support_source_var, value='Generate from reconstruction', command=self._support_source_changed).pack(anchor='w', pady=2)
        ttk.Radiobutton(source_box, text='Use uploaded .npy mask', variable=self.support_source_var, value='Uploaded .npy mask', command=self._support_source_changed).pack(anchor='w', pady=2)
        source_buttons = ttk.Frame(source_box)
        source_buttons.pack(fill='x', pady=(6, 2))
        ttk.Button(source_buttons, text='UPLOAD .NPY MASK', command=self.upload_support_mask).pack(side='left', expand=True, fill='x', padx=(0, 3))
        ttk.Button(source_buttons, text='GENERATE MASK', command=self.generate_support_mask).pack(side='left', expand=True, fill='x', padx=(3, 0))
        self.support_source_file_var = tk.StringVar(value='No .npy mask loaded.')
        ttk.Label(source_box, textvariable=self.support_source_file_var, wraplength=390, justify='left').pack(anchor='w', pady=(4, 0))
        ttk.Label(source_box, text='Uploaded masks are validated as 2-D arrays and resized to the reconstruction size when necessary. Upload does not replace the active support until APPLY AS SUPPORT MASK is pressed.', wraplength=390, justify='left').pack(anchor='w', pady=(5, 0))
        box = ttk.LabelFrame(self.support_ctrl_inner, text='1. Detection / Consensus', padding=8)
        box.pack(fill='x', pady=5)
        ttk.Label(box, text='Threshold (relative amplitude)').pack(anchor='w')
        tk.Scale(box, from_=0.01, to=0.5, resolution=0.01, orient='horizontal', variable=self.support_threshold_var, length=360, showvalue=True, command=lambda _: self.update_support_preview()).pack(fill='x')
        ttk.Label(box, text='Consensus fraction').pack(anchor='w', pady=(8, 0))
        tk.Scale(box, from_=0.05, to=0.95, resolution=0.05, orient='horizontal', variable=self.support_consensus_var, length=360, showvalue=True, command=lambda _: self.update_support_preview()).pack(fill='x')
        morph = ttk.LabelFrame(self.support_ctrl_inner, text='2. Morphological Cleanup', padding=8)
        morph.pack(fill='x', pady=5)
        ttk.Label(morph, text='Minimum object size (pixels)').pack(anchor='w')
        tk.Scale(morph, from_=0, to=5000, resolution=25, orient='horizontal', variable=self.support_min_size_var, length=360, showvalue=True, command=lambda _: self.update_support_preview()).pack(fill='x')
        ttk.Label(morph, text='Closing radius').pack(anchor='w', pady=(8, 0))
        tk.Scale(morph, from_=0, to=15, resolution=1, orient='horizontal', variable=self.support_close_var, length=360, showvalue=True, command=lambda _: self.update_support_preview()).pack(fill='x')
        ttk.Label(morph, text='Dilation radius / support margin').pack(anchor='w', pady=(8, 0))
        tk.Scale(morph, from_=0, to=20, resolution=1, orient='horizontal', variable=self.support_dilate_var, length=360, showvalue=True, command=lambda _: self.update_support_preview()).pack(fill='x')
        ttk.Label(morph, text='Gaussian pre-smoothing σ').pack(anchor='w', pady=(8, 0))
        tk.Scale(morph, from_=0.0, to=5.0, resolution=0.25, orient='horizontal', variable=self.support_sigma_var, length=360, showvalue=True, command=lambda _: self.update_support_preview()).pack(fill='x')
        stackbox = ttk.LabelFrame(self.support_ctrl_inner, text='3. Reconstruction Stack', padding=8)
        stackbox.pack(fill='x', pady=5)
        ttk.Label(stackbox, text='Stack statistic').pack(anchor='w')
        ttk.Combobox(stackbox, textvariable=self.support_mode_var, values=['Median', 'Mean', 'Maximum'], state='readonly', width=30).pack(fill='x', pady=(2, 6))
        ttk.Label(stackbox, text='Seed constraint').pack(anchor='w')
        ttk.Combobox(stackbox, textvariable=self.support_seed_var, values=['Original support', 'No constraint'], state='readonly', width=30).pack(fill='x', pady=(2, 2))
        ttk.Label(stackbox, text='Recommended: Original support. This prevents isolated reconstruction noise outside the known object.', wraplength=390, justify='left').pack(anchor='w', pady=(5, 0))
        range_box = ttk.LabelFrame(self.support_ctrl_inner, text='4. Frames Used for Consensus', padding=8)
        range_box.pack(fill='x', pady=5)
        ttk.Label(range_box, text='Start index').grid(row=0, column=0, sticky='w', padx=4, pady=3)
        self.support_start_spin = tk.Spinbox(range_box, from_=0, to=10000, width=10, textvariable=self.support_start_var, command=self.update_support_preview)
        self.support_start_spin.grid(row=0, column=1, sticky='ew', padx=4, pady=3)
        ttk.Label(range_box, text='End index').grid(row=1, column=0, sticky='w', padx=4, pady=3)
        self.support_end_spin = tk.Spinbox(range_box, from_=0, to=10000, width=10, textvariable=self.support_end_var, command=self.update_support_preview)
        self.support_end_spin.grid(row=1, column=1, sticky='ew', padx=4, pady=3)
        ttk.Button(range_box, text='USE ALL FRAMES', command=self.support_use_all_frames).grid(row=2, column=0, columnspan=2, sticky='ew', pady=(7, 3))
        range_box.columnconfigure(1, weight=1)
        stats = ttk.LabelFrame(self.support_ctrl_inner, text='5. Mask Statistics', padding=8)
        stats.pack(fill='x', pady=5)
        ttk.Label(stats, textvariable=self.support_original_pixels_var).pack(anchor='w')
        ttk.Label(stats, textvariable=self.support_candidate_pixels_var).pack(anchor='w')
        self.support_removed_pixels_var = tk.StringVar(value='Changed: --')
        ttk.Label(stats, textvariable=self.support_removed_pixels_var).pack(anchor='w')
        actions = ttk.LabelFrame(self.support_ctrl_inner, text='6. Actions', padding=8)
        actions.pack(fill='x', pady=5)
        ttk.Button(actions, text='GENERATE / REFRESH', command=self.generate_support_mask).pack(fill='x', pady=3)
        ttk.Button(actions, text='✓ APPLY AS SUPPORT MASK', command=self.apply_optimized_support).pack(fill='x', pady=3)
        ttk.Button(actions, text='SAVE MASK', command=self.save_optimized_support).pack(fill='x', pady=3)
        ttk.Button(actions, text='RESET TO ORIGINAL', command=self.reset_support_to_original).pack(fill='x', pady=3)
        ttk.Button(actions, text='SCROLL TO TOP', command=lambda: self.support_ctrl_canvas.yview_moveto(0)).pack(fill='x', pady=(8, 3))
        status = ttk.LabelFrame(self.support_ctrl_inner, text='Status / Recommended Starting Point', padding=8)
        status.pack(fill='x', pady=5)
        ttk.Label(status, textvariable=self.support_status_var, wraplength=390, justify='left').pack(fill='x')
        ttk.Label(status, text='Suggested starting values:\\nThreshold 0.10–0.15\\nConsensus 0.50–0.70\\nMin object 500–1500 px\\nClosing 2–4 px\\nDilation 2–5 px\\nGaussian σ 1–2\\nMedian + Original support + all frames', wraplength=390, justify='left').pack(anchor='w', pady=(8, 0))
        self.support_ctrl_inner.update_idletasks()
        self.support_ctrl_canvas.configure(scrollregion=self.support_ctrl_canvas.bbox('all'))
        self.support_ctrl_canvas.yview_moveto(0.0)
        self._support_candidate = None
        self._support_vote = None
        self._support_score = None

    def _support_source_changed(self):
        """Switch between generated support and an uploaded .npy support."""
        source = self.support_source_var.get()
        if source == 'Uploaded .npy mask':
            if self._uploaded_support_mask is None:
                self.support_status_var.set('Upload a .npy support mask first.')
                return
            self._show_uploaded_support_preview()
        else:
            self.support_status_var.set('Generate mode selected. Adjust controls and click GENERATE MASK.')
            if self.recons is not None:
                self.update_support_preview()

    def upload_support_mask(self):
        """Load a binary/array .npy support mask for use instead of generation."""
        if self.recons is None:
            messagebox.showwarning('Support Mask', 'Load the CDI reconstruction stack first so the mask size can be validated.', parent=self.root)
            return
        fp = filedialog.askopenfilename(parent=self.root, title='Select support mask (.npy)', filetypes=[('NumPy mask', '*.npy'), ('All files', '*.*')])
        if not fp:
            return
        try:
            raw = np.load(fp, allow_pickle=False)
            arr = np.asarray(raw)
            if arr.ndim != 2:
                raise ValueError(f'Uploaded mask must be 2-D; received shape {arr.shape}.')
            if arr.size == 0:
                raise ValueError('Uploaded mask is empty.')
            if not np.all(np.isfinite(arr.astype(np.float64, copy=False))):
                raise ValueError('Uploaded mask contains NaN or infinite values.')
            mask = arr > 0
            if not np.any(mask):
                raise ValueError('Uploaded mask contains no positive/support pixels.')
            target_shape = tuple(self.recons.shape[1:3])
            original_shape = mask.shape
            if original_shape != target_shape:
                mask = self._scale_support_mask(mask, target_shape)
            if mask.shape != target_shape:
                raise ValueError(f'Final mask shape {mask.shape} does not match reconstruction shape {target_shape}.')
            self._uploaded_support_mask = mask.astype(bool)
            self._uploaded_support_path = str(fp)
            self._support_candidate = self._uploaded_support_mask.copy()
            self._support_vote = self._uploaded_support_mask.astype(float)
            self._support_score = np.abs(np.asarray(self.recons[0], dtype=float))
            self.support_source_var.set('Uploaded .npy mask')
            self.support_source_file_var.set(f'Loaded: {Path(fp).name}\nOriginal shape: {original_shape} → Applied shape: {mask.shape}\nSupport pixels: {int(mask.sum()):,}')
            self.support_original_pixels_var.set(f'Original active support: {int(np.count_nonzero(self.supportmask2)):,} pixels' if self.supportmask2 is not None else 'Original active support: --')
            self.support_candidate_pixels_var.set(f'Uploaded mask: {int(mask.sum()):,} pixels ({100.0 * np.mean(mask):.2f}% of image)')
            if self.supportmask2 is not None:
                old_mask = np.asarray(self.supportmask2, dtype=bool)
                added = int(np.count_nonzero(mask & ~old_mask))
                removed = int(np.count_nonzero(old_mask & ~mask))
                self.support_removed_pixels_var.set(f'Changed vs active support: +{added:,} added / -{removed:,} removed')
            else:
                self.support_removed_pixels_var.set('Changed vs active support: --')
            self._show_uploaded_support_preview()
            self.support_status_var.set(f'Uploaded support ready. Press APPLY AS SUPPORT MASK to activate it.')
            self.log(f'Uploaded support mask: {fp}; original_shape={original_shape}, final_shape={mask.shape}, pixels={int(mask.sum())}')
        except Exception as exc:
            messagebox.showerror('Support Mask Upload', f'Could not load the .npy support mask.\n\n{exc}', parent=self.root)
            self.support_status_var.set(f'Upload failed: {exc}')

    def _show_uploaded_support_preview(self):
        """Display the uploaded mask using the same support visualization."""
        mask = self._uploaded_support_mask
        if mask is None:
            return
        if self.recons is not None and len(self.recons):
            img = np.abs(np.asarray(self.recons[0], dtype=float))
            finite = np.isfinite(img)
            if np.any(finite):
                lo, hi = np.percentile(img[finite], (2, 98))
                if hi <= lo:
                    hi = lo + 1.0
                representative = np.clip((img - lo) / (hi - lo), 0, 1)
            else:
                representative = np.zeros_like(img, dtype=float)
        else:
            representative = mask.astype(float)
        self._support_candidate = mask.copy()
        self._support_vote = mask.astype(float)
        self._support_score = representative
        self._draw_support_preview(representative, mask, mask.astype(float))

    def support_use_all_frames(self):
        n = len(self.recons) if self.recons is not None else 101
        self.support_start_var.set(0)
        self.support_end_var.set(max(0, n - 1))
        self.update_support_preview()

    def _support_prepare_stack(self, indices):
        """Return normalized amplitude/contrast stack for support generation."""
        stack = np.asarray(self.recons[indices], dtype=np.float32)
        out = np.zeros_like(stack, dtype=np.float32)
        for k in range(stack.shape[0]):
            img = np.abs(stack[k])
            finite = np.isfinite(img)
            if not np.any(finite):
                continue
            vals = img[finite]
            lo = float(np.percentile(vals, 1.0))
            hi = float(np.percentile(vals, 99.5))
            if hi <= lo:
                hi = lo + 1e-12
            img = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
            sigma = float(self.support_sigma_var.get())
            if sigma > 0:
                img = gaussian_filter(img, sigma=sigma)
            out[k] = img
        return out

    def _support_cleanup(self, mask):
        """Clean candidate support using connected components and morphology."""
        mask = np.asarray(mask, dtype=bool)
        close_r = int(self.support_close_var.get())
        if close_r > 0:
            mask = morphology.closing(mask, morphology.disk(close_r))
        mask = morphology.remove_small_holes(mask, area_threshold=max(16, int(self.support_min_size_var.get())))
        min_size = int(self.support_min_size_var.get())
        if min_size > 0:
            mask = morphology.remove_small_objects(mask, max_size=max(0, int(min_size)-1))
        dilate_r = int(self.support_dilate_var.get())
        if dilate_r > 0:
            mask = morphology.binary_dilation(mask, morphology.disk(dilate_r))
        mask = morphology.remove_small_holes(mask, area_threshold=max(16, int(self.support_min_size_var.get())))
        return mask.astype(bool)

    def _support_seed(self):
        if self.recons is None:
            return None
        if self.support_seed_var.get() == 'Original support':
            if self.supportmask2 is not None:
                return np.asarray(self.supportmask2, dtype=bool)
        return np.ones(self.recons.shape[1:3], dtype=bool)

    def generate_support_mask(self):
        self.support_source_var.set('Generate from reconstruction')
        if self.recons is None:
            messagebox.showwarning('Support Mask', 'Load the 101 CDI reconstructions first.', parent=self.root)
            return None
        n = len(self.recons)
        try:
            start = max(0, min(int(self.support_start_var.get()), n - 1))
            end = max(0, min(int(self.support_end_var.get()), n - 1))
        except Exception:
            start, end = (0, n - 1)
        if end < start:
            start, end = (end, start)
        self.support_start_var.set(start)
        self.support_end_var.set(end)
        indices = list(range(start, end + 1))
        stack = self._support_prepare_stack(indices)
        threshold = float(self.support_threshold_var.get())
        seed = self._support_seed()
        candidates = stack >= threshold
        if seed is not None:
            candidates &= seed[None, :, :]
        cleaned = np.zeros_like(candidates, dtype=bool)
        for k in range(candidates.shape[0]):
            m = candidates[k]
            if self.support_min_size_var.get() > 0:
                m = morphology.remove_small_objects(m, max_size=max(7, int(self.support_min_size_var.get() // 4)-1))
            cleaned[k] = m
        vote = np.mean(cleaned.astype(np.float32), axis=0)
        consensus = float(self.support_consensus_var.get())
        candidate = vote >= consensus
        candidate = self._support_cleanup(candidate)
        candidate &= np.ones_like(candidate, dtype=bool)
        self._support_candidate = candidate
        self._support_vote = vote
        mode = self.support_mode_var.get()
        if mode == 'Mean':
            representative = np.mean(stack, axis=0)
        elif mode == 'Maximum':
            representative = np.max(stack, axis=0)
        else:
            representative = np.median(stack, axis=0)
        self._support_score = representative
        self.support_original_pixels_var.set(f'Original: {int(np.count_nonzero(self.supportmask2)):,} pixels' if self.supportmask2 is not None else 'Original: full image')
        self.support_candidate_pixels_var.set(f'Candidate: {int(np.count_nonzero(candidate)):,} pixels ({100.0 * np.mean(candidate):.2f}% of image)')
        if self.supportmask2 is not None:
            old_mask = np.asarray(self.supportmask2, dtype=bool)
            added = int(np.count_nonzero(candidate & ~old_mask))
            removed = int(np.count_nonzero(old_mask & ~candidate))
            self.support_removed_pixels_var.set(f'Changed: +{added:,} added / -{removed:,} removed')
        else:
            self.support_removed_pixels_var.set('Changed: --')
        self._draw_support_preview(representative, candidate, vote)
        self.support_status_var.set(f'Generated support from frames {start}–{end} ({len(indices)} frames). Threshold={threshold:.2f}, consensus={consensus:.2f}.')
        self.log(f'Support optimization: frames={start}:{end}, threshold={threshold:.2f}, consensus={consensus:.2f}, pixels={int(candidate.sum())}')
        return candidate

    def _draw_support_preview(self, representative, candidate, vote):
        axes = self.support_axes
        old_cbar = getattr(self, '_support_colorbar', None)
        if old_cbar is not None:
            try:
                old_cbar.remove()
            except Exception:
                pass
            self._support_colorbar = None
        old_cax = getattr(self, '_support_colorbar_ax', None)
        if old_cax is not None:
            try:
                old_cax.remove()
            except Exception:
                pass
            self._support_colorbar_ax = None
        for ax in axes.flat:
            ax.clear()
        original = np.asarray(self.supportmask2, dtype=bool) if self.supportmask2 is not None else np.ones_like(candidate, dtype=bool)
        axes[0, 0].imshow(representative, cmap='gray', origin='lower', interpolation='nearest')
        axes[0, 0].contour(original.astype(float), levels=[0.5], colors='red', linewidths=0.8)
        axes[0, 0].set_title('Representative reconstruction + ORIGINAL support')
        axes[0, 1].imshow(representative, cmap='gray', origin='lower', interpolation='nearest')
        axes[0, 1].contour(candidate.astype(float), levels=[0.5], colors='lime', linewidths=1.0)
        axes[0, 1].set_title('Representative + OPTIMIZED support')
        axes[1, 0].imshow(candidate, cmap='gray', origin='lower', interpolation='nearest')
        axes[1, 0].set_title(f'Optimized support | {int(candidate.sum()):,} px')
        im = axes[1, 1].imshow(vote, cmap='viridis', origin='lower', vmin=0, vmax=1, interpolation='nearest')
        axes[1, 1].contour(candidate.astype(float), levels=[0.5], colors='white', linewidths=0.8)
        axes[1, 1].set_title('Consensus / vote fraction')
        for ax in axes.flat:
            ax.set_aspect('equal')
            ax.set_xlabel('X pixel')
            ax.set_ylabel('Y pixel')
        self.support_fig.suptitle('SUPPORT MASK OPTIMIZATION — compare original vs consensus support', fontsize=13)
        panel_positions = {(0, 0): [0.065, 0.535, 0.385, 0.335], (0, 1): [0.515, 0.535, 0.385, 0.335], (1, 0): [0.065, 0.085, 0.385, 0.355], (1, 1): [0.515, 0.085, 0.36, 0.355]}
        for (r, c), pos in panel_positions.items():
            axes[r, c].set_position(pos)
        cax = self.support_fig.add_axes([0.888, 0.085, 0.018, 0.355])
        self._support_colorbar_ax = cax
        self._support_colorbar = self.support_fig.colorbar(im, cax=cax)
        self._support_colorbar.set_label('Consensus fraction')
        self.support_canvas.draw_idle()
        try:
            self.support_scroll_inner.update_idletasks()
            self.support_view_canvas.update_idletasks()
            self.support_view_canvas.configure(scrollregion=self.support_view_canvas.bbox('all'))
        except Exception:
            pass

    def update_support_preview(self):
        if self.support_source_var.get() == 'Uploaded .npy mask':
            if self._uploaded_support_mask is not None:
                self._show_uploaded_support_preview()
            return
        if self.recons is None:
            return
        try:
            self.generate_support_mask()
        except Exception as exc:
            self.support_status_var.set(f'Preview error: {exc}')
            self.log(f'Support preview error: {exc}')

    def apply_optimized_support(self):
        if self._support_candidate is None:
            self.generate_support_mask()
        if self._support_candidate is None:
            return
        self.supportmask2 = np.asarray(self._support_candidate, dtype=bool).copy()
        self.supportmask = self.supportmask2.copy()
        self.roi_mask = None
        self.roi_coords = None
        self.roi_stack = None
        self.roi_crop = None
        self.confirmed_roi_frame = None
        self.confirmed_roi_frame_index = None
        self.current_processed = None
        source_name = 'uploaded .npy mask' if self.support_source_var.get() == 'Uploaded .npy mask' else 'generated optimized support'
        self.support_status_var.set(f'✓ {source_name.capitalize()} applied: {int(self.supportmask2.sum()):,} pixels.')
        self.log(f'Optimized support APPLIED as supportmask2: {self.supportmask2.shape}, pixels={int(self.supportmask2.sum())}')
        self.show_current_recon()

    def reset_support_to_original(self):
        if self.supportmask is None:
            self.support_status_var.set('No original support mask is available.')
            return
        self.supportmask2 = self._scale_support_mask(self.supportmask, self.recons.shape[1:3])
        self.support_source_var.set('Generate from reconstruction')
        self.roi_mask = None
        self.roi_coords = None
        self.roi_stack = None
        self.roi_crop = None
        self.current_processed = None
        self.support_status_var.set(f'Original support restored: {int(self.supportmask2.sum()):,} pixels.')
        self.log('Original support mask restored.')
        self.show_current_recon()

    def save_optimized_support(self):
        if self._support_candidate is None:
            messagebox.showwarning('Save Support', 'Generate an optimized support first.', parent=self.root)
            return
        fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Save optimized support mask', defaultextension='.npy', filetypes=[('NumPy mask', '*.npy'), ('PNG mask', '*.png'), ('All files', '*.*')])
        if not fp:
            return
        mask = np.asarray(self._support_candidate, dtype=bool)
        ext = Path(fp).suffix.lower()
        if ext == '.png':
            Image.fromarray(mask.astype(np.uint8) * 255).save(fp)
        else:
            np.save(fp, mask)
        self.support_status_var.set(f'Saved optimized support: {fp}')
        self.log(f'Saved optimized support mask: {fp}')

    def refresh_roi_image(self):
        """Show the ROI frame and install a robust manual mouse selector."""
        if self.recons is None:
            self.roi_status_var.set('Load reconstructions first.')
            return
        idx = max(0, min(int(self.roi_index_var.get()), len(self.recons) - 1))
        self.roi_index_var.set(idx)
        img = np.asarray(self.recons[idx], dtype=float)
        support = np.asarray(self.supportmask2, dtype=bool)
        valid = support & np.isfinite(img)
        if np.any(valid):
            vmin, vmax = np.percentile(img[valid], (2, 98))
            if vmax <= vmin:
                vmax = vmin + 1.0
        else:
            vmin, vmax = (0.0, 1.0)
        disp = np.clip((img - vmin) / (vmax - vmin), 0.0, 1.0)
        disp[~support] = np.nan
        self.roi_coords = None
        self.roi_mask = None
        fig, ax = plt.subplots(figsize=(8, 7), dpi=100)
        ax.imshow(disp, cmap='gray', origin='lower', vmin=0, vmax=1, interpolation='nearest')
        ax.contour(support.astype(float), levels=[0.5], colors='cyan', linewidths=0.8)
        ax.set_title(f'DRAG LEFT MOUSE TO SELECT ROI | Index {idx} | Field {self.real_fields_mT[idx]:+.2f} mT')
        ax.set_xlabel('X pixel')
        ax.set_ylabel('Y pixel')
        ax.axis('image')
        ax.set_xlim(0, img.shape[1] - 1)
        ax.set_ylim(0, img.shape[0] - 1)
        self.clear_canvas_frame(self.roi_canvas_frame)
        canvas = FigureCanvasTkAgg(fig, master=self.roi_canvas_frame)
        canvas_widget = canvas.get_tk_widget()
        canvas_widget.pack(fill='both', expand=True)
        canvas.draw()
        self._figs['roi'] = fig
        self._roi_canvas = canvas
        self._roi_ax = ax
        self._roi_patch = None
        self._roi_press_xy = None
        self._roi_move_cid = None
        self._roi_press_cid = None
        self._roi_release_cid = None

        def clamp_xy(event):
            if event.xdata is None or event.ydata is None:
                return None
            x = float(np.clip(event.xdata, 0, img.shape[1] - 1))
            y = float(np.clip(event.ydata, 0, img.shape[0] - 1))
            return (x, y)

        def on_press(event):
            if event.inaxes is not ax or event.button != 1:
                return
            xy = clamp_xy(event)
            if xy is None:
                return
            self._roi_press_xy = xy
            if self._roi_patch is not None:
                self._roi_patch.remove()
                self._roi_patch = None
            self.roi_coords = None
            self.roi_mask = None
            self.roi_status_var.set(f'Start ROI: X={xy[0]:.1f}, Y={xy[1]:.1f} — drag to size ROI.')
            canvas.draw_idle()

        def on_move(event):
            if self._roi_press_xy is None or event.inaxes is not ax:
                return
            xy = clamp_xy(event)
            if xy is None:
                return
            x0, y0 = self._roi_press_xy
            x1, y1 = xy
            xmin, xmax = sorted((x0, x1))
            ymin, ymax = sorted((y0, y1))
            width = max(1.0, xmax - xmin)
            height = max(1.0, ymax - ymin)
            if self._roi_patch is None:
                self._roi_patch = plt.Rectangle((xmin, ymin), width, height, fill=False, edgecolor='red', linewidth=2.5, zorder=10)
                ax.add_patch(self._roi_patch)
            else:
                self._roi_patch.set_xy((xmin, ymin))
                self._roi_patch.set_width(width)
                self._roi_patch.set_height(height)
            self.roi_status_var.set(f'Selecting ROI: X={xmin:.0f}:{xmax:.0f}, Y={ymin:.0f}:{ymax:.0f}')
            canvas.draw_idle()

        def on_release(event):
            if self._roi_press_xy is None or event.button != 1:
                return
            xy = clamp_xy(event)
            if xy is None:
                self._roi_press_xy = None
                return
            x0, y0 = self._roi_press_xy
            x1, y1 = xy
            xmin = int(round(min(x0, x1)))
            xmax = int(round(max(x0, x1)))
            ymin = int(round(min(y0, y1)))
            ymax = int(round(max(y0, y1)))
            self._roi_press_xy = None
            if xmax <= xmin or ymax <= ymin:
                self.roi_coords = None
                self.roi_status_var.set('ROI too small. Drag a larger rectangle.')
                return
            self.roi_coords = {'xmin': xmin, 'xmax': xmax, 'ymin': ymin, 'ymax': ymax}
            if self._roi_patch is None:
                self._roi_patch = plt.Rectangle((xmin, ymin), xmax - xmin + 1, ymax - ymin + 1, fill=False, edgecolor='red', linewidth=2.5, zorder=10)
                ax.add_patch(self._roi_patch)
            else:
                self._roi_patch.set_xy((xmin, ymin))
                self._roi_patch.set_width(xmax - xmin + 1)
                self._roi_patch.set_height(ymax - ymin + 1)
            self.roi_status_var.set(f'ROI selected: X={xmin}:{xmax}, Y={ymin}:{ymax}. Pixels={(xmax - xmin + 1) * (ymax - ymin + 1)}. Click CONFIRM ROI.')
            canvas.draw_idle()
        self._roi_press_cid = canvas.mpl_connect('button_press_event', on_press)
        self._roi_move_cid = canvas.mpl_connect('motion_notify_event', on_move)
        self._roi_release_cid = canvas.mpl_connect('button_release_event', on_release)
        self.roi_status_var.set('ROI READY: LEFT-CLICK and DRAG a rectangle, then click CONFIRM ROI.')
        self.nb.select(self.tab_roi)

    def confirm_roi(self):
        if self.recons is None:
            messagebox.showwarning('ROI', 'Load reconstructions first.')
            return
        if self.roi_coords is None:
            messagebox.showwarning('ROI', 'No ROI selected. Drag a rectangle with the LEFT mouse button first.')
            return
        x0 = self.roi_coords['xmin']
        x1 = self.roi_coords['xmax']
        y0 = self.roi_coords['ymin']
        y1 = self.roi_coords['ymax']
        idx = max(0, min(int(self.roi_index_var.get()), len(self.recons) - 1))
        mask = np.zeros(self.recons.shape[1:], dtype=bool)
        mask[y0:y1 + 1, x0:x1 + 1] = True
        mask &= np.asarray(self.supportmask2, dtype=bool)
        valid = mask & np.isfinite(self.recons[idx])
        n_valid = int(np.count_nonzero(valid))
        if n_valid == 0:
            messagebox.showerror('ROI', 'The selected ROI has no valid pixels inside the support mask.')
            return
        self.roi_mask = mask
        self.confirmed_roi_frame = {'xmin': x0, 'xmax': x1, 'ymin': y0, 'ymax': y1}
        self.confirmed_roi_frame_index = idx
        self.current_index = idx
        self.build_roi_stack()
        try:
            if hasattr(self, '_pixel_cal_refresh_primary_roi'):
                self._pixel_cal_refresh_primary_roi()
        except Exception:
            pass
        self.roi_status_var.set(f'ROI CONFIRMED ✓ | X={x0}:{x1} | Y={y0}:{y1} | ROI pixels={int(mask.sum())} | valid={n_valid}')
        self.update_processing_view()
        try:
            self._pixel_cal_refresh_primary_roi()
        except Exception as _exc:
            self.log(f'Pixel Calibration ROI refresh warning: {_exc}')
        self.nb.select(self.tab_pixel_calibration)
        self.log(f'ROI confirmed: X={x0}:{x1}, Y={y0}:{y1}, pixels={int(mask.sum())}, valid={n_valid}')

    def clear_roi(self):
        self.roi_mask = None
        self.roi_coords = None
        self.roi_stack = None
        self.roi_mean = None
        self.roi_std = None
        self.roi_crop = None
        self.current_processed = None
        self._roi_press_xy = None
        self._roi_patch = None
        self.roi_status_var.set('ROI cleared. Drag a new rectangle.')
        if self.recons is not None:
            self.refresh_roi_image()
        else:
            self.clear_canvas_frame(self.roi_canvas_frame)

    def build_roi_stack(self):
        mask = np.asarray(self.roi_mask, dtype=bool)
        yy, xx = np.where(mask)
        if len(xx) == 0:
            raise RuntimeError('Confirmed ROI contains no pixels.')
        x0, x1 = (int(xx.min()), int(xx.max()))
        y0, y1 = (int(yy.min()), int(yy.max()))
        self.roi_crop = {'xmin': x0, 'xmax': x1, 'ymin': y0, 'ymax': y1}
        local_mask = mask[y0:y1 + 1, x0:x1 + 1]
        local_data = self.recons[:, y0:y1 + 1, x0:x1 + 1]
        self.roi_stack = np.where(local_mask[None, :, :], local_data, np.nan).astype(np.float64)
        vals = self.roi_stack
        with np.errstate(all='ignore'):
            self.roi_mean = np.nanmean(vals, axis=(1, 2))
            self.roi_std = np.nanstd(vals, axis=(1, 2))
        self.proc_slider.configure(to=max(0, len(self.recons) - 1))
        self.proc_idx_var.set(float(self.roi_index_var.get()))
        self.log(f'ROI applied to all {len(self.roi_stack)} images. ROI bounding box: X={x0}:{x1}, Y={y0}:{y1}. Cropped ROI stack: {self.roi_stack.shape}')

    def _get_roi_image(self, idx):
        if self.roi_mask is None or self.roi_stack is None:
            raise RuntimeError('Confirm an ROI first.')
        img = self.roi_stack[idx].copy()
        finite = np.isfinite(img)
        if not np.any(finite):
            return np.zeros_like(img, dtype=float)
        vals = img[finite]
        lo, hi = np.percentile(vals, (1, 99))
        if hi <= lo:
            hi = lo + 1e-12
        out = np.nan_to_num((img - lo) / (hi - lo), nan=0.0, posinf=1.0, neginf=0.0)
        return np.clip(out, 0, 1)

    def _apply_filter(self, img):
        kind = self.filter_var.get()
        if kind == 'None':
            return img
        if kind == 'Gaussian':
            return gaussian_filter(img, sigma=1.5)
        fy = filters.gaussian(img, sigma=1.5)
        if kind == 'Low-pass':
            return fy
        if kind == 'High-pass':
            return img - fy
        if kind == 'Band-pass':
            return filters.gaussian(img, sigma=1.0) - filters.gaussian(img, sigma=4.0)
        return img

    def update_processing_view(self):
        if self.roi_mask is None:
            return
        idx = max(0, min(int(round(self.proc_idx_var.get())), len(self.recons) - 1))
        self.proc_idx_var.set(idx)
        self.current_index = idx
        try:
            nproc = int(len(self.recons))
            self.proc_mp4_start_spin.configure(from_=1, to=max(1, nproc), increment=1, state='normal')
            self.proc_mp4_end_spin.configure(from_=1, to=max(1, nproc), increment=1, state='normal')
            st = max(1, min(int(self.proc_mp4_start_var.get()), nproc))
            en = max(st, min(int(self.proc_mp4_end_var.get()), nproc))
            self.proc_mp4_start_var.set(st)
            self.proc_mp4_end_var.set(en)
        except Exception:
            pass
        img = self._get_roi_image(idx)
        img = self._apply_filter(img)
        c = float(self.contrast_var.get())
        b = float(self.brightness_var.get())
        img = (img - 0.5) * c + 0.5 + b
        img = np.clip(img, 0, 1)
        self.current_processed = img
        self.proc_field_var.set(f'Frame No. {idx + 1}/{len(self.recons)} | Equivalent Field: {self.real_fields_mT[idx]:+.2f} mT')
        fig, ax = plt.subplots(figsize=(8, 7))
        ax.imshow(img, cmap='gray', origin='lower', vmin=0, vmax=1)
        ax.contour(self._get_roi_mask_local().astype(float), levels=[0.5], colors='red', linewidths=0.7)
        ax.set_title(f'ROI processed | index {idx} | field {self.real_fields_mT[idx]:+.2f} mT')
        ax.axis('image')
        fig.tight_layout()
        self.embed_figure(self.proc_canvas_frame, fig, key='proc')

def _proc_mp4_range_spin_changed(self, which=None):
    """Robust TAB-6 MP4 START/END Spinbox handler.

    Tk Spinbox arrows update the linked IntVar first and then invoke
    this callback.  Re-read the value, clamp it to the loaded frame
    count, and keep START <= END.
    """
    try:
        n = int(len(self.recons)) if self.recons is not None else 0
    except Exception:
        n = 0
    if n <= 0:
        return
    try:
        start = int(self.proc_mp4_start_var.get())
    except Exception:
        start = 1
    try:
        end = int(self.proc_mp4_end_var.get())
    except Exception:
        end = n

    start = max(1, min(start, n))
    end = max(1, min(end, n))

    if start > end:
        if which == 'start':
            end = start
        elif which == 'end':
            start = end
        else:
            start, end = sorted((start, end))

    self.proc_mp4_start_var.set(start)
    self.proc_mp4_end_var.set(end)

    try:
        self.proc_mp4_start_spin.configure(from_=1, to=n, increment=1, state='normal')
        self.proc_mp4_end_spin.configure(from_=1, to=n, increment=1, state='normal')
    except Exception:
        pass

    try:
        self.proc_status_var.set(f'MP4 export range: frames {start}–{end}.')
    except Exception:
        pass


def _proc_slider_update(self, _=None):
    self.update_processing_view()

def _get_roi_mask_local(self):
    if self.roi_mask is None:
        return np.zeros((10, 10), dtype=bool)
    if self.roi_crop is not None:
        y0 = int(self.roi_crop['ymin'])
        y1 = int(self.roi_crop['ymax'])
        x0 = int(self.roi_crop['xmin'])
        x1 = int(self.roi_crop['xmax'])
    else:
        yy, xx = np.where(np.asarray(self.roi_mask, dtype=bool))
        if yy.size == 0 or xx.size == 0:
            return np.zeros((10, 10), dtype=bool)
        y0, y1 = (int(yy.min()), int(yy.max()))
        x0, x1 = (int(xx.min()), int(xx.max()))
    mask = np.asarray(self.roi_mask, dtype=bool)
    y0 = max(0, min(y0, mask.shape[0] - 1))
    y1 = max(y0, min(y1, mask.shape[0] - 1))
    x0 = max(0, min(x0, mask.shape[1] - 1))
    x1 = max(x0, min(x1, mask.shape[1] - 1))
    return np.asarray(mask[y0:y1 + 1, x0:x1 + 1], dtype=bool)

    def _get_roi_mask_local(self):
        if self.roi_mask is None:
            return np.zeros((10, 10), dtype=bool)
        if self.roi_crop is not None:
            y0 = self.roi_crop['ymin']
            y1 = self.roi_crop['ymax']
            x0 = self.roi_crop['xmin']
            x1 = self.roi_crop['xmax']
        else:
            yy, xx = np.where(self.roi_mask)
            if len(xx) == 0:
                return np.zeros((10, 10), dtype=bool)
            y0, y1 = (yy.min(), yy.max())
            x0, x1 = (xx.min(), xx.max())
        return np.asarray(self.roi_mask[y0:y1 + 1, x0:x1 + 1], dtype=bool)

    def show_fft(self):
        if self.current_processed is None:
            self.update_processing_view()
        img = self.current_processed
        F = np.fft.fftshift(np.fft.fft2(img))
        amp = np.log1p(np.abs(F))
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
        ax1.imshow(img, cmap='gray', origin='lower')
        ax1.axis('image')
        ax1.set_title('ROI')
        ax2.imshow(amp, cmap='magma', origin='lower')
        ax2.axis('image')
        ax2.set_title('FFT log amplitude')
        fig.tight_layout()
        self.embed_figure(self.proc_canvas_frame, fig, key='fft')

    def blob_analysis(self):
        if self.current_processed is None:
            self.update_processing_view()
        img = self.current_processed
        threshold = filters.threshold_otsu(img) if np.any(img > 0) else 0.5
        binary = img > threshold
        lab = measure.label(binary)
        props = measure.regionprops(lab)
        self.last_blobs = props
        fig, ax = plt.subplots(figsize=(8, 7))
        ax.imshow(img, cmap='gray', origin='lower', vmin=0, vmax=1)
        for p in props:
            if p.area < 5:
                continue
            y, x = p.centroid
            ax.plot(x, y, 'r+')
            ax.text(x, y, str(p.label), color='yellow', fontsize=8)
        ax.set_title(f'Blob analysis | {len(props)} regions | threshold={threshold:.3f}')
        ax.axis('image')
        fig.tight_layout()
        self.embed_figure(self.proc_canvas_frame, fig, key='blobs')
        self.log(f'Blob analysis: {len(props)} connected regions found.')

    def reset_processing(self):
        self.brightness_var.set(0.0)
        self.contrast_var.set(1.0)
        self.filter_var.set('None')
        self.update_processing_view()

    def export_processed(self, ext='png'):
        if self.current_processed is None:
            self.update_processing_view()
        fp = filedialog.asksaveasfilename(parent=self.root, initialdir=self._export_initialdir(), defaultextension='.' + ext, filetypes=[(ext.upper(), '*.' + ext)])
        if not fp:
            return
        Image.fromarray(np.uint8(np.clip(self.current_processed, 0, 1) * 255)).save(fp)
        self.log(f'Exported {fp}')

    def browse_frc(self):
        f = filedialog.askopenfilename(parent=self.root, title='Select FRC HDF5', filetypes=[('HDF5', '*.h5 *.hdf5'), ('All', '*.*')])
        if f:
            self.frc_file = Path(f)
            self.frc_var.set(str(f))

    def load_frc(self):
        if not self.frc_file:
            return
        rows = []
        curves = {}
        with h5py.File(self.frc_file, 'r') as f:
            for key in f.keys():
                g = f[key]
                if not isinstance(g, h5py.Group):
                    continue
                rows.append({'entry': key, 'magnet_B': float(g.attrs.get('magnet_B', np.nan)), 'resolution': float(g.attrs.get('resolution', np.nan)), 'q_cut': float(g.attrs.get('q_cut', np.nan)), 'FFT_pixel_cut': float(g.attrs.get('FFT_pixel_cut', np.nan))})
                curves[key] = g['frc_smooth'][:] if 'frc_smooth' in g else g['frc_raw'][:] if 'frc_raw' in g else None
        df = pd.DataFrame(rows).sort_values('magnet_B').reset_index(drop=True)
        self.frc_info_var.set(f'{len(df)} entries')
        fig, ax = plt.subplots(figsize=(9, 6))
        x = df['magnet_B'].to_numpy()
        y = df['resolution'].to_numpy()
        ax.plot(x, y, 'o-')
        ax.set_xlabel('magnet_B')
        ax.set_ylabel('resolution')
        ax.grid(True, ls=':')
        fig.tight_layout()
        self.embed_figure(self.frc_canvas_frame, fig, key='frc')
        self.log(f'Loaded FRC: {len(df)} entries.')

def _proc_build(self):
    for w in self.tab_proc.winfo_children():
        w.destroy()
    # User-adjustable horizontal divider between the Tab 6 image and controls.
    # The previous fixed left/right packing is replaced only at the Tab 6
    # layout level; all image/processing callbacks remain unchanged.
    main = ttk.Panedwindow(self.tab_proc, orient='horizontal')
    main.pack(fill='both', expand=True, padx=2, pady=2)
    image_frame = ttk.Frame(main)
    control_outer = ttk.Frame(main, width=430)
    control_outer.pack_propagate(False)
    main.add(image_frame, weight=1)
    main.add(control_outer, weight=0)
    control_canvas = tk.Canvas(control_outer, highlightthickness=0)
    control_scroll = ttk.Scrollbar(control_outer, orient='vertical', command=control_canvas.yview)
    control_frame = ttk.Frame(control_canvas)
    control_frame.bind(
        '<Configure>',
        lambda e: control_canvas.configure(scrollregion=control_canvas.bbox('all'))
    )
    self._proc_control_window_id = control_canvas.create_window(
        (0, 0), window=control_frame, anchor='nw', width=410
    )
    def _resize_tab6_control_canvas(event=None):
        try:
            width = max(250, control_canvas.winfo_width() - control_scroll.winfo_reqwidth())
            control_canvas.itemconfigure(
                self._proc_control_window_id, width=width
            )
        except Exception:
            pass
    control_canvas.bind('<Configure>', _resize_tab6_control_canvas, add='+')
    control_canvas.configure(yscrollcommand=control_scroll.set)
    control_canvas.pack(side='left', fill='both', expand=True)
    control_scroll.pack(side='right', fill='y')
    self._proc_control_canvas = control_canvas
    self._proc_control_frame = control_frame

    def mousewheel(event):
        control_canvas.yview_scroll(int(-1 * (event.delta / 120)), 'units')
    control_canvas.bind('<Enter>', lambda e: control_canvas.bind_all('<MouseWheel>', mousewheel))
    control_canvas.bind('<Leave>', lambda e: control_canvas.unbind_all('<MouseWheel>'))
    self._proc_fig, self._proc_ax = plt.subplots(figsize=(8, 7), dpi=100)
    self._proc_fig.subplots_adjust(left=0, right=1, bottom=0, top=0.93)
    self._proc_image_display = self._proc_ax.imshow(np.zeros((10, 10), dtype=float), cmap='gray', origin='lower', vmin=0, vmax=1, interpolation='nearest', extent=(-0.5, 9.5, -0.5, 9.5), aspect='equal')
    self._proc_ax.axis('off')
    self._proc_title = self._proc_ax.set_title('', fontsize=12)
    self._proc_canvas = FigureCanvasTkAgg(self._proc_fig, master=image_frame)
    self._proc_canvas.draw()
    self._proc_canvas.get_tk_widget().pack(fill='both', expand=True)
    self._proc_display_shape = (10, 10)
    self._proc_zoomed = False
    self._proc_zoom_limits = None
    self._proc_zoomed = False
    self._proc_zoom_limits = None
    self._proc_canvas.mpl_connect('scroll_event', self._proc_zoom_scroll)
    self._proc_canvas.mpl_connect('button_press_event', self._proc_secondary_on_press)
    self._proc_canvas.mpl_connect('motion_notify_event', self._proc_secondary_on_move)
    self._proc_canvas.mpl_connect('button_release_event', self._proc_secondary_on_release)
    self._proc_canvas.mpl_connect('button_press_event', self._proc_profile_on_press)
    self._proc_canvas.mpl_connect('motion_notify_event', self._proc_profile_on_move)
    self._proc_canvas.mpl_connect('button_release_event', self._proc_profile_on_release)
    self.proc_idx_var = tk.IntVar(value=0)
    self.brightness_var = tk.DoubleVar(value=0.0)
    self.contrast_var = tk.DoubleVar(value=1.0)
    self.filter_var = tk.StringVar(value='Gaussian')
    self.strength_var = tk.DoubleVar(value=3.0)
    self.colormap_var = tk.StringVar(value='RdBu_r')
    self.view_var = tk.StringVar(value='IFFT')
    self.fft_mask_var = tk.DoubleVar(value=0.60)
    self.fps_var = tk.IntVar(value=10)
    self._roi_play_job = None
    self._roi_playing = False
    self.roi_width_var = tk.StringVar(value='0')
    self.roi_height_var = tk.StringVar(value='0')
    self.output_var = tk.StringVar(value=getattr(self, 'output_dir', os.path.abspath('ROI_exports')))
    self.proc_field_var = tk.StringVar(value='Field = --')
    self.proc_roi_info_var = tk.StringVar(value='ROI = --')
    self.proc_status_var = tk.StringVar(value='Ready')
    self.proc_export_dpi_var = tk.IntVar(value=300)
    self.proc_mp4_start_var = tk.IntVar(value=1)
    self.proc_mp4_end_var = tk.IntVar(value=1)
    self._proc_secondary_roi_selecting = False
    self._proc_secondary_press_xy = None
    self._proc_secondary_patch = None
    self.proc_secondary_status_var = tk.StringVar(value='Secondary ROI: not selected')
    self.proc_pixel_size_var = tk.StringVar(value='Pixel Size = -- nm/px')
    self.proc_effective_pixel_size_var = tk.StringVar(value='Effective Pixel Size = -- nm/px')
    self.proc_profile_pixels_var = tk.StringVar(value='Profile length = -- px')
    self.proc_profile_nm_var = tk.StringVar(value='Profile length = -- nm')
    self.proc_profile_mode = False
    self.proc_profile_press_xy = None
    self._proc_profile_line_artist = None
    self.proc_profile_line_color_var = tk.StringVar(value='red')
    self.proc_profile_line_style_var = tk.StringVar(value='-')
    self.proc_profile_line_width_var = tk.DoubleVar(value=2.5)
    self.proc_profile_measure_nm_per_px = None
    self._proc_colorbar = None
    self._proc_colorbar_mappable = None
    rf = ttk.LabelFrame(control_frame, text='SAME CONFIRMED ROI — OUTPUT RESOLUTION', padding=7)
    rf.pack(fill='x', padx=6, pady=6)
    ttk.Label(rf, text='ROI Width (px):').grid(row=0, column=0, sticky='w', padx=4, pady=3)
    self.roi_width_entry = ttk.Entry(rf, textvariable=self.roi_width_var, width=12)
    self.roi_width_entry.grid(row=0, column=1, sticky='ew', padx=4, pady=3)
    self.roi_width_entry.bind('<KeyRelease>', lambda e: self.update_roi_height_from_width())
    ttk.Label(rf, text='ROI Height (px):').grid(row=1, column=0, sticky='w', padx=4, pady=3)
    self.roi_height_entry = ttk.Entry(rf, textvariable=self.roi_height_var, width=12, state='readonly')
    self.roi_height_entry.grid(row=1, column=1, sticky='ew', padx=4, pady=3)
    rf.columnconfigure(1, weight=1)
    ttk.Button(rf, text='256 × 256', command=lambda: self.set_roi_preset(256, 256)).grid(row=2, column=0, sticky='ew', padx=3, pady=3)
    ttk.Button(rf, text='512 × 512', command=lambda: self.set_roi_preset(512, 512)).grid(row=2, column=1, sticky='ew', padx=3, pady=3)
    ttk.Button(rf, text='1024 × 1024', command=lambda: self.set_roi_preset(1024, 1024)).grid(row=3, column=0, sticky='ew', padx=3, pady=3)
    ttk.Button(rf, text='2048 × 2048', command=lambda: self.set_roi_preset(2048, 2048)).grid(row=3, column=1, sticky='ew', padx=3, pady=3)
    ttk.Button(rf, text='Original ROI', command=self.set_original_roi_preset).grid(row=4, column=0, sticky='ew', padx=3, pady=3)
    ttk.Button(rf, text='✓ APPLY RESOLUTION', command=self.apply_roi_resolution_from_gui).grid(row=4, column=1, columnspan=2, sticky='ew', padx=3, pady=6)
    ttk.Label(rf, text='Confirmed ROI stays fixed. Only output sampling resolution changes.', justify='center', wraplength=370).grid(row=6, column=0, columnspan=2, padx=3, pady=(0, 2))
    scale_box = ttk.LabelFrame(control_frame, text='PHYSICAL PIXEL SIZE', padding=7)
    scale_box.pack(fill='x', padx=6, pady=(2, 6))
    scale_row = ttk.Frame(scale_box)
    scale_row.pack(fill='x')
    ttk.Label(scale_row, textvariable=self.proc_pixel_size_var, font=('Arial', 10, 'bold')).pack(side='left', padx=(2, 8))
    ttk.Button(scale_row, text='↻ UPDATE', command=self.update_processing_pixel_size).pack(side='right')
    ttk.Label(scale_box, textvariable=self.proc_effective_pixel_size_var, font=('Arial', 10, 'bold')).pack(anchor='w', pady=(4, 0))
    ttk.Label(scale_box, text='Base value = final Primary ROI nm/pixel from Tab 5.\nEffective value is recalculated for the applied output resolution.', justify='left', wraplength=370).pack(anchor='w', pady=(3, 0))
    profile_box = ttk.LabelFrame(control_frame, text='ROI PROFILE DIMENSION — LINE MEASUREMENT', padding=7)
    profile_box.pack(fill='x', padx=6, pady=(2, 8))
    ttk.Label(profile_box, text='Draw a straight I-type line on the ROI to measure a feature dimension in nm.', wraplength=370, justify='left').pack(anchor='w', pady=(0, 5))
    pbtns = ttk.Frame(profile_box)
    pbtns.pack(fill='x')
    ttk.Button(pbtns, text='DRAW LINE', command=self.start_processing_profile_measure).pack(side='left', expand=True, fill='x', padx=(0, 3))
    ttk.Button(pbtns, text='CLEAR', command=self.clear_processing_profile_measure).pack(side='left', expand=True, fill='x', padx=(3, 0))
    ttk.Label(profile_box, text='Line width:').pack(anchor='w', pady=(6, 0))
    ttk.Combobox(profile_box, textvariable=self.proc_profile_line_width_var, values=(1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0), state='readonly', width=12).pack(fill='x')
    ttk.Label(profile_box, text='Line style:').pack(anchor='w', pady=(4, 0))
    ttk.Combobox(profile_box, textvariable=self.proc_profile_line_style_var, values=('-', '--', '-.', ':'), state='readonly', width=12).pack(fill='x')
    ttk.Label(profile_box, text='Line color:').pack(anchor='w', pady=(4, 0))
    ttk.Combobox(profile_box, textvariable=self.proc_profile_line_color_var, values=('red', 'blue', 'green', 'yellow'), state='readonly', width=12).pack(fill='x')
    ttk.Label(profile_box, textvariable=self.proc_profile_pixels_var).pack(anchor='w', pady=(7, 0))
    ttk.Label(profile_box, textvariable=self.proc_profile_nm_var, font=('Arial', 10, 'bold')).pack(anchor='w', pady=(2, 0))
    ttk.Label(control_frame, textvariable=self.proc_field_var, font=('Arial', 11, 'bold')).pack(pady=(5, 2))
    ttk.Label(control_frame, text='FIELD / IMAGE').pack(anchor='w', padx=10)
    self.proc_slider = tk.Scale(control_frame, from_=0, to=0, orient='horizontal', variable=self.proc_idx_var, resolution=1, length=340, command=lambda _: self.update_processing_view())
    self.proc_slider.pack(padx=10, pady=2)
    ttk.Label(control_frame, textvariable=self.proc_roi_info_var).pack(anchor='w', padx=10)
    zoom_box = ttk.LabelFrame(control_frame, text='PRIMARY ROI — MOUSE ZOOM', padding=7)
    zoom_box.pack(fill='x', padx=6, pady=(5, 6))
    zoom_btns = ttk.Frame(zoom_box)
    zoom_btns.pack(fill='x')
    ttk.Button(zoom_btns, text='RESET ZOOM', command=self._proc_reset_zoom).pack(fill='x')
    ttk.Label(zoom_box, text='Move the cursor over the primary ROI and use the mouse wheel: UP = zoom in, DOWN = zoom out. Zoom stays fixed while changing frames.', wraplength=370, justify='left').pack(anchor='w', pady=(4, 0))
    sroi_box = ttk.LabelFrame(control_frame, text='SECONDARY ROI — SELECT FROM PROCESSED DATA', padding=7)
    sroi_box.pack(fill='x', padx=6, pady=(10, 5))
    ttk.Label(sroi_box, text='Legacy Secondary ROI controls for existing ROI2 processing.', wraplength=370, justify='left').pack(anchor='w', pady=(0, 6))
    sroi_btns = ttk.Frame(sroi_box)
    sroi_btns.pack(fill='x')
    self.proc_secondary_select_button = ttk.Button(sroi_btns, text='SELECT SECONDARY ROI', command=self.start_secondary_roi_selection)
    self.proc_secondary_select_button.pack(side='left', expand=True, fill='x', padx=(0, 3))
    ttk.Button(sroi_btns, text='✓ CONFIRM SECONDARY ROI', command=self.confirm_secondary_roi_from_processing).pack(side='left', expand=True, fill='x', padx=(3, 0))
    sroi_btns2 = ttk.Frame(sroi_box)
    sroi_btns2.pack(fill='x', pady=(5, 0))
    ttk.Button(sroi_btns2, text='CLEAR SECONDARY ROI', command=self.clear_secondary_roi_from_processing).pack(side='left', expand=True, fill='x', padx=(0, 3))
    ttk.Label(sroi_box, textvariable=self.proc_secondary_status_var, wraplength=370, justify='left').pack(anchor='w', pady=(6, 0))
    ttk.Label(control_frame, text='BRIGHTNESS').pack(anchor='w', padx=10, pady=(10, 0))
    tk.Scale(control_frame, from_=-1, to=1, resolution=0.01, orient='horizontal', variable=self.brightness_var, length=340, command=lambda _: self.update_processing_view()).pack(padx=10)
    ttk.Label(control_frame, text='CONTRAST').pack(anchor='w', padx=10, pady=(10, 0))
    tk.Scale(control_frame, from_=0.1, to=3.0, resolution=0.05, orient='horizontal', variable=self.contrast_var, length=340, command=lambda _: self.update_processing_view()).pack(padx=10)
    ttk.Label(control_frame, text='COLOR MAP').pack(anchor='w', padx=10, pady=(10, 0))
    color_combo = ttk.Combobox(control_frame, textvariable=self.colormap_var, values=['gray', 'viridis', 'plasma', 'inferno', 'magma', 'cividis', 'turbo', 'jet', 'seismic', 'RdBu_r'], state='readonly', width=28)
    color_combo.pack(padx=10)
    color_combo.bind('<<ComboboxSelected>>', lambda e: self.update_processing_view())
    ttk.Label(control_frame, text='VIEW').pack(anchor='w', padx=10, pady=(10, 0))
    view_combo = ttk.Combobox(control_frame, textvariable=self.view_var, values=['Image', 'FFT', 'IFFT'], state='readonly', width=28)
    self.view_var.set('Image')
    view_combo.pack(padx=10)
    view_combo.bind('<<ComboboxSelected>>', lambda e: self.update_processing_view())
    ttk.Label(control_frame, text='FILTER').pack(anchor='w', padx=10, pady=(10, 0))
    filt = ttk.Combobox(control_frame, textvariable=self.filter_var, values=['None', 'Gaussian', 'Low-pass', 'High-pass', 'Band-pass'], state='readonly', width=28)
    filt.pack(padx=10)
    filt.bind('<<ComboboxSelected>>', lambda e: self.update_processing_view())
    ttk.Label(control_frame, text='FILTER STRENGTH').pack(anchor='w', padx=10, pady=(10, 0))
    tk.Scale(control_frame, from_=0.01, to=20.0, resolution=0.01, orient='horizontal', variable=self.strength_var, length=340, command=lambda _: self.update_processing_view()).pack(padx=10)
    ttk.Label(control_frame, text='FFT MASK RADIUS (0–1, step = 0.01)').pack(anchor='w', padx=10, pady=(10, 0))
    self.fft_mask_scale = tk.Scale(control_frame, from_=0.0, to=1.0, resolution=0.01, orient='horizontal', variable=self.fft_mask_var, length=340, command=lambda _: self.update_processing_view())
    self.fft_mask_scale.pack(padx=10)
    bf = ttk.Frame(control_frame)
    bf.pack(fill='x', padx=10, pady=12)
    ttk.Button(bf, text='RESET', command=self.reset_processing).pack(side='left', expand=True, fill='x', padx=2)
    ttk.Label(control_frame, text='EXPORT FOLDER').pack(anchor='w', padx=10, pady=(15, 0))
    ttk.Entry(control_frame, textvariable=self.output_var, width=34).pack(padx=10, pady=2, fill='x')
    ttk.Button(control_frame, text='SELECT FOLDER', command=self.choose_output_folder).pack(padx=10, pady=5, fill='x')
    dpi_box = ttk.LabelFrame(control_frame, text='EXPORT RESOLUTION — SELECT DPI BEFORE EXPORT', padding=7)
    dpi_box.pack(fill='x', padx=6, pady=(15, 5))
    ttk.Label(dpi_box, text='Select the DPI for export:').pack(side='left', padx=(2, 8))
    self.proc_dpi_combo = ttk.Combobox(dpi_box, textvariable=self.proc_export_dpi_var, values=(150, 300, 600, 900), state='readonly', width=8)
    self.proc_dpi_combo.pack(side='left')
    ttk.Label(dpi_box, text='  150 / 300 / 600 / 900 DPI').pack(side='left', padx=(8, 0))
    ttk.Label(control_frame, text='SINGLE IMAGE').pack(anchor='w', padx=10, pady=(15, 3))
    sf = ttk.Frame(control_frame)
    sf.pack(padx=10, fill='x')
    for label, ext, fmt in [('PNG', '.png', 'PNG'), ('JPG', '.jpg', 'JPEG'), ('TIFF', '.tiff', 'TIFF')]:
        ttk.Button(sf, text=label, command=lambda ext=ext, fmt=fmt: self.save_single(ext, fmt)).pack(side='left', expand=True, fill='x', padx=2)
    ttk.Label(control_frame, text='BATCH IMAGES').pack(anchor='w', padx=10, pady=(15, 3))
    bf2 = ttk.Frame(control_frame)
    bf2.pack(padx=10, fill='x')
    for label, ext, fmt in [('PNG', '.png', 'PNG'), ('JPG', '.jpg', 'JPEG'), ('TIFF', '.tiff', 'TIFF')]:
        ttk.Button(bf2, text=label, command=lambda ext=ext, fmt=fmt: self.batch_export(ext, fmt)).pack(side='left', expand=True, fill='x', padx=2)
    playback_box = ttk.LabelFrame(control_frame, text='ROI FRAME PLAYBACK', padding=7)
    playback_box.pack(fill='x', padx=6, pady=(15, 5))
    ttk.Label(playback_box, text='Frame rate (frames / second)').pack(anchor='w')
    tk.Scale(playback_box, from_=1, to=60, resolution=1, orient='horizontal', variable=self.fps_var, length=340, showvalue=True, command=lambda _: self._roi_playback_fps_changed()).pack(padx=2, fill='x')
    playback_buttons = ttk.Frame(playback_box)
    playback_buttons.pack(fill='x', pady=(5, 2))
    self.roi_play_button = ttk.Button(playback_buttons, text='▶ PLAY', command=self.play_roi_frames)
    self.roi_play_button.pack(side='left', expand=True, fill='x', padx=(0, 3))
    self.roi_play_button.bind('<Double-Button-1>', self._roi_play_button_double_click)
    ttk.Button(playback_buttons, text='↩ FIRST FRAME', command=self.go_to_first_roi_frame).pack(side='left', expand=True, fill='x', padx=(3, 0))
    self.roi_play_status_var = tk.StringVar(value='Playback: stopped')
    ttk.Label(playback_box, textvariable=self.roi_play_status_var).pack(anchor='w', pady=(4, 0))
    ttk.Label(playback_box, text='PLAY starts from the current frame. Click again to pause; click PLAY to resume. Change FPS to control playback speed.', wraplength=370, justify='left').pack(anchor='w', pady=(3, 0))
    ttk.Label(control_frame, text='MP4 EXPORT').pack(anchor='w', padx=10, pady=(10, 0))
    mp4_range = ttk.Frame(control_frame)
    mp4_range.pack(padx=10, fill='x', pady=(3, 2))
    ttk.Label(mp4_range, text='START FRAME').pack(side='left')
    self.proc_mp4_start_spin = tk.Spinbox(
        mp4_range, from_=1, to=1, width=7,
        textvariable=self.proc_mp4_start_var, increment=1,
        command=lambda: _proc_mp4_range_spin_changed(self, 'start')
    )
    self.proc_mp4_start_spin.pack(side='left', padx=(4, 10))
    self.proc_mp4_start_spin.bind(
        '<Return>', lambda e: _proc_mp4_range_spin_changed(self, 'start')
    )
    self.proc_mp4_start_spin.bind(
        '<FocusOut>', lambda e: _proc_mp4_range_spin_changed(self, 'start')
    )
    ttk.Label(mp4_range, text='END FRAME').pack(side='left')
    self.proc_mp4_end_spin = tk.Spinbox(
        mp4_range, from_=1, to=1, width=7,
        textvariable=self.proc_mp4_end_var, increment=1,
        command=lambda: _proc_mp4_range_spin_changed(self, 'end')
    )
    self.proc_mp4_end_spin.pack(side='left', padx=(4, 10))
    self.proc_mp4_end_spin.bind(
        '<Return>', lambda e: _proc_mp4_range_spin_changed(self, 'end')
    )
    self.proc_mp4_end_spin.bind(
        '<FocusOut>', lambda e: _proc_mp4_range_spin_changed(self, 'end')
    )
    ttk.Label(mp4_range, text='FPS = FPS control above').pack(side='left')
    vf = ttk.Frame(control_frame)
    vf.pack(padx=10, fill='x', pady=4)
    ttk.Button(vf, text='SINGLE MP4', command=self.export_single_mp4).pack(side='left', expand=True, fill='x', padx=2)
    ttk.Button(vf, text='BATCH MP4', command=self.export_batch_mp4).pack(side='left', expand=True, fill='x', padx=2)
    ttk.Label(control_frame, textvariable=self.proc_status_var, wraplength=380).pack(padx=10, pady=20)

def _proc_get_tab5_base_pixel_size(self):
    """Return the final Primary ROI nm/pixel stored by Tab 5."""
    try:
        value = getattr(self, 'primary_roi_final_nm_per_pixel', None)
        if value is None:
            return None
        value = float(value)
        if not np.isfinite(value) or value <= 0:
            return None
        return value
    except Exception:
        return None

def update_processing_pixel_size(self):
    """Read ONLY the final Primary ROI nm/pixel from Tab 5.

    Tab 6 output resolution is independent. The Tab-5 calibration value is never
    modified by this function.  For line measurements on a resampled Tab-6 image,
    convert output pixels back to the original confirmed-ROI pixel coordinate using
    the local sampling ratio, while keeping Tab 5's final nm/pixel authoritative.
    """
    base = self._proc_get_tab5_base_pixel_size()
    if base is None or not np.isfinite(base) or base <= 0:
        self.proc_pixel_size_var.set('Pixel Size = -- nm/px')
        self.proc_effective_pixel_size_var.set('Tab 5 final ROI pixel size unavailable')
        self.proc_profile_measure_nm_per_px = None
        return
    self.proc_pixel_size_var.set(f'Pixel Size = {base:.9g} nm/px')
    try:
        ow = int(getattr(self, 'original_roi_w', 0) or 0)
        oh = int(getattr(self, 'original_roi_h', 0) or 0)
        rw, rh = getattr(self, 'roi_resolution', (0, 0))
        rw, rh = (int(rw), int(rh))
        if ow <= 0 or oh <= 0 or rw <= 0 or (rh <= 0):
            raise ValueError
        out_px_nm_x = float(base) * float(ow) / float(rw)
        out_px_nm_y = float(base) * float(oh) / float(rh)
        self.proc_profile_measure_nm_per_px = float(out_px_nm_x)
        self.proc_effective_pixel_size_var.set(f'Tab 6 measurement scale = {out_px_nm_x:.9g} nm/output-px (output {rw} × {rh}; Tab 5 remains {base:.9g} nm/px)')
    except Exception:
        self.proc_profile_measure_nm_per_px = None
        self.proc_effective_pixel_size_var.set('Tab 6 output sampling scale = --')

def _proc_profile_line_kwargs(self):
    return dict(color=self.proc_profile_line_color_var.get(), linestyle=self.proc_profile_line_style_var.get(), linewidth=float(self.proc_profile_line_width_var.get()), solid_capstyle='round')

def start_processing_profile_measure(self):
    """Enable physical-dimension line measurement on the displayed ROI.

    The ROI axes in Tab 6 are in nm, so measurements are read directly in nm.
    Tab 5 supplies only the authoritative final ROI nm/pixel value used to define
    the physical axes. Tab 6 output sampling never writes back to Tab 5.
    """
    if self.roi_mask is None or self.recons is None:
        messagebox.showwarning('ROI Measurement', 'Confirm the Primary ROI in Tab 4 first.', parent=self.root)
        return
    base = self._proc_get_tab5_base_pixel_size()
    if base is None or not np.isfinite(base) or base <= 0:
        messagebox.showwarning('ROI Measurement', 'No valid Final Primary ROI nm/pixel was found in Tab 5. Confirm Tab-5 calibration first.', parent=self.root)
        return
    self.update_processing_pixel_size()
    self.proc_profile_mode = True
    self._proc_secondary_roi_selecting = False
    self.proc_profile_press_xy = None
    self.proc_status_var.set('PROFILE LINE MODE: click on Endpoint 1, move to Endpoint 2, then click again. Length is reported in nm.')

def _proc_profile_on_press(self, event):
    if not getattr(self, 'proc_profile_mode', False):
        return
    if event.inaxes is not self._proc_ax or event.xdata is None or event.ydata is None:
        return
    self.proc_profile_press_xy = (float(event.xdata), float(event.ydata))
    if self._proc_profile_line_artist is not None:
        try:
            self._proc_profile_line_artist.remove()
        except Exception:
            pass
        self._proc_profile_line_artist = None
    self._proc_profile_line_artist, = self._proc_ax.plot([event.xdata, event.xdata], [event.ydata, event.ydata], zorder=20, **self._proc_profile_line_kwargs())
    self._proc_canvas.draw_idle()

def _proc_profile_on_move(self, event):
    if not getattr(self, 'proc_profile_mode', False):
        return
    if self.proc_profile_press_xy is None or event.inaxes is not self._proc_ax:
        return
    if event.xdata is None or event.ydata is None or self._proc_profile_line_artist is None:
        return
    x0, y0 = self.proc_profile_press_xy
    self._proc_profile_line_artist.set_data([x0, float(event.xdata)], [y0, float(event.ydata)])
    self._proc_canvas.draw_idle()

def _proc_profile_on_release(self, event):
    if not getattr(self, 'proc_profile_mode', False):
        return
    if self.proc_profile_press_xy is None or event.inaxes is not self._proc_ax:
        return
    if event.xdata is None or event.ydata is None:
        self.proc_profile_press_xy = None
        return
    x0, y0 = self.proc_profile_press_xy
    x1, y1 = (float(event.xdata), float(event.ydata))
    length_nm = float(np.hypot(x1 - x0, y1 - y0))
    output_px = np.nan
    try:
        rw, rh = getattr(self, 'roi_resolution', (0, 0))
        rw, rh = (int(rw), int(rh))
        base = self._proc_get_tab5_base_pixel_size()
        ow = float(getattr(self, 'original_roi_w', 0) or 0)
        oh = float(getattr(self, 'original_roi_h', 0) or 0)
        if base is not None and base > 0 and (rw > 0) and (rh > 0) and (ow > 0) and (oh > 0):
            physical_w_nm = ow * float(base)
            physical_h_nm = oh * float(base)
            px_x = (x1 - x0) * rw / physical_w_nm if physical_w_nm > 0 else 0.0
            px_y = (y1 - y0) * rh / physical_h_nm if physical_h_nm > 0 else 0.0
            output_px = float(np.hypot(px_x, px_y))
    except Exception:
        pass
    if length_nm > 0:
        if np.isfinite(output_px):
            self.proc_profile_pixels_var.set(f'Profile length = {output_px:.3f} output-px')
        else:
            self.proc_profile_pixels_var.set('Profile length = -- output-px')
        self.proc_profile_nm_var.set(f'Profile length = {length_nm:.6g} nm')
        self.proc_status_var.set(f'Profile measured: {length_nm:.6g} nm.')
        try:
            self._proc_profile_line_artist.set_data([x0, x1], [y0, y1])
        except Exception:
            pass
    self.proc_profile_press_xy = None
    self.proc_profile_mode = False
    self._proc_canvas.draw_idle()

def clear_processing_profile_measure(self):
    """Clear only the profile line and measurement values; retain the ROI image."""
    self.proc_profile_mode = False
    self.proc_profile_press_xy = None
    if self._proc_profile_line_artist is not None:
        try:
            self._proc_profile_line_artist.remove()
        except Exception:
            pass
        self._proc_profile_line_artist = None
    self.proc_profile_pixels_var.set('Profile length = -- px')
    self.proc_profile_nm_var.set('Profile length = -- nm')
    self.proc_status_var.set('Profile measurement cleared; ROI image retained.')
    self._proc_canvas.draw_idle()

def _proc_update_physical_axes_and_colorbar(self, display):
    """Update Tab 6 ROI display using physical nm coordinates from Tab 5.

    Tab 5 remains authoritative for the base ROI nm/pixel.  Tab 6 output
    resolution changes only the sampling density; the physical ROI extent is
    kept fixed.  Zoom limits are stored in the same physical nm coordinate
    system, so changing resolution or frame does not corrupt the viewport.
    """
    arr = np.asarray(display)
    h, w = arr.shape[:2]
    base = self._proc_get_tab5_base_pixel_size()
    if base is not None and np.isfinite(base) and (base > 0):
        try:
            ow = int(getattr(self, 'original_roi_w', w) or w)
            oh = int(getattr(self, 'original_roi_h', h) or h)
            physical_w_nm = float(ow) * float(base)
            physical_h_nm = float(oh) * float(base)
            self._proc_image_display.set_extent((0.0, physical_w_nm, 0.0, physical_h_nm))
            ax = self._proc_ax
            ax.set_xlabel('X (nm)')
            ax.set_ylabel('Y (nm)')
            ax.set_aspect('equal', adjustable='box')
            ax.set_anchor('C')
            ax.set_autoscale_on(False)
            self._proc_full_physical_extent = (0.0, physical_w_nm, 0.0, physical_h_nm)
            limits = getattr(self, '_proc_zoom_limits', None)
            if limits is None:
                ax.set_xlim(0.0, physical_w_nm)
                ax.set_ylim(0.0, physical_h_nm)
            else:
                self._proc_restore_zoom_limits(w, h)
        except Exception:
            self._proc_ax.set_xlabel('X (nm)')
            self._proc_ax.set_ylabel('Y (nm)')
    else:
        ax = self._proc_ax
        ax.set_xlabel('X (pixel)')
        ax.set_ylabel('Y (pixel)')
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
        self._proc_full_physical_extent = (-0.5, float(w) - 0.5, -0.5, float(h) - 0.5)
    try:
        if getattr(self, '_proc_colorbar', None) is None:
            cax = self._proc_fig.add_axes([0.9, 0.07, 0.025, 0.86])
            self._proc_colorbar = self._proc_fig.colorbar(self._proc_image_display, cax=cax)
            self._proc_colorbar.set_label('Intensity')
            self._proc_colorbar_mappable = self._proc_image_display
        else:
            self._proc_colorbar.update_normal(self._proc_image_display)
            try:
                self._proc_colorbar.ax.set_position([0.9, 0.07, 0.025, 0.86])
            except Exception:
                pass
        self._proc_image_display.set_clim(0.0, 1.0)
    except Exception:
        pass

def _roi_play_button_double_click(self, event=None):
    """Double-click on Play/Resume pauses playback."""
    self.pause_roi_frames()
    return 'break'

def go_to_first_roi_frame(self):
    """Stop playback and return the ROI view to the first frame."""
    self.pause_roi_frames()
    if self.recons is None or len(self.recons) == 0:
        self.roi_play_status_var.set('Playback: no frames loaded')
        return
    self.proc_idx_var.set(0)
    self.current_index = 0
    self.update_processing_view()
    self.roi_play_status_var.set(f'Playback: ready | frame 1/{len(self.recons)}')

def _roi_playback_fps_changed(self):
    """Reschedule playback immediately when FPS is changed."""
    if not getattr(self, '_roi_playing', False):
        return
    if getattr(self, '_roi_play_job', None) is not None:
        try:
            self.root.after_cancel(self._roi_play_job)
        except Exception:
            pass
        self._roi_play_job = None
    fps = max(1, int(self.fps_var.get()))
    self.roi_play_status_var.set(f'Playback: playing/resuming at {fps} FPS | frame {int(self.proc_idx_var.get()) + 1}/{len(self.recons)}')
    self._roi_play_job = self.root.after(max(1, int(round(1000.0 / fps))), self._roi_play_next)

def play_roi_frames(self):
    """Play the ROI reconstruction stack at the selected FPS."""
    if self.recons is None or self.roi_mask is None:
        messagebox.showwarning('ROI Playback', 'Load the reconstructions and confirm an ROI first.', parent=self.root)
        return
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Playback', 'Apply the confirmed ROI resolution first.', parent=self.root)
        return
    nframes = len(self.recons)
    if nframes == 0:
        return
    if getattr(self, '_roi_playing', False):
        return
    idx = max(0, min(int(round(self.proc_idx_var.get())), nframes - 1))
    self.proc_idx_var.set(idx)
    self.current_index = idx
    self._roi_playing = True
    fps = max(1, int(self.fps_var.get()))
    self.roi_play_status_var.set(f'Playback: playing at {fps} FPS | frame {idx + 1}/{nframes}')
    self.update_processing_view()
    self._roi_play_job = self.root.after(max(1, int(round(1000.0 / fps))), self._roi_play_next)

def _roi_play_next(self):
    """Advance one frame and schedule the next frame."""
    self._roi_play_job = None
    if not getattr(self, '_roi_playing', False):
        return
    if self.recons is None or len(self.recons) == 0:
        self.pause_roi_frames()
        return
    nframes = len(self.recons)
    idx = int(round(self.proc_idx_var.get()))
    if idx >= nframes - 1:
        self.proc_idx_var.set(nframes - 1)
        self.update_processing_view()
        self._roi_playing = False
        self.roi_play_status_var.set(f'Playback: finished | frame {nframes}/{nframes}')
        return
    idx += 1
    self.proc_idx_var.set(idx)
    self.current_index = idx
    self.update_processing_view()
    fps = max(1, int(self.fps_var.get()))
    self.roi_play_status_var.set(f'Playback: playing at {fps} FPS | frame {idx + 1}/{nframes}')
    self._roi_play_job = self.root.after(max(1, int(round(1000.0 / fps))), self._roi_play_next)

def pause_roi_frames(self):
    """Pause playback and keep the current frame."""
    self._roi_playing = False
    job = getattr(self, '_roi_play_job', None)
    if job is not None:
        try:
            self.root.after_cancel(job)
        except Exception:
            pass
        self._roi_play_job = None
    if hasattr(self, 'roi_play_status_var'):
        nframes = len(self.recons) if self.recons is not None else 0
        idx = int(round(self.proc_idx_var.get())) if hasattr(self, 'proc_idx_var') else 0
        if nframes:
            self.roi_play_status_var.set(f'Playback: paused | frame {idx + 1}/{nframes}')
        else:
            self.roi_play_status_var.set('Playback: stopped')

def batch_export_roi2_mp4(self):
    """Export Secondary ROI range to MP4 with X/Y axes, field and frame number."""
    if self.roi2_stack is None:
        messagebox.showwarning('Secondary ROI MP4 Export', 'Confirm the Secondary ROI in Tab 5 first.', parent=self.root)
        return
    try:
        start = int(self.roi2_start_frame_var.get())
        end = int(self.roi2_end_frame_var.get())
        fps = max(1, min(120, int(self.fps_var.get())))
        dpi = int(self.roi2_export_dpi_var.get())
    except Exception:
        messagebox.showerror('Secondary ROI MP4 Export', 'Start frame, end frame, FPS and DPI must be valid values.', parent=self.root)
        return
    nframes = len(self.roi2_stack)
    if not 1 <= start <= end <= nframes:
        messagebox.showwarning('Secondary ROI MP4 Export', f'Frame range must satisfy 1 <= Start <= End <= {nframes}.', parent=self.root)
        return
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Secondary ROI MP4 Export', 'Select DPI: 150, 300, 600 or 900.', parent=self.root)
        return
    imageio = _proc_imageio(self)
    if imageio is None:
        return
    self._roi2_stop_playback()
    folder = self.output_var.get()
    os.makedirs(folder, exist_ok=True)
    indices = list(range(start - 1, end))
    path = os.path.join(folder, f'SecondaryROI_frames_{start:04d}-{end:04d}_{fps}fps_{dpi}dpi_annotated.mp4')
    fig = plt.figure(figsize=(8.0, 6.5), dpi=dpi, facecolor='white')
    ax = fig.add_axes([0.1, 0.12, 0.84, 0.8])
    writer = None
    try:
        codec_attempts = ('libx264', 'mpeg4', None)
        errors = []
        for codec in codec_attempts:
            try:
                kwargs = {'fps': fps, 'format': 'ffmpeg', 'macro_block_size': None}
                if codec is not None:
                    kwargs['codec'] = codec
                writer = imageio.get_writer(path, **kwargs)
                break
            except Exception as exc:
                errors.append(f"{codec or 'default'}: {exc}")
        if writer is None:
            raise RuntimeError('Could not initialize an MP4 encoder.\n\n' + '\n'.join(errors))
        for count, idx in enumerate(indices, start=1):
            image = np.asarray(self.roi2_stack[idx], dtype=float)
            ax.clear()
            ax.set_aspect('equal', adjustable='box')
            finite = np.isfinite(image)
            if np.any(finite):
                vmin = float(np.nanmin(image[finite]))
                vmax = float(np.nanmax(image[finite]))
                if vmax <= vmin:
                    vmax = vmin + 1e-12
            else:
                vmin, vmax = (0.0, 1.0)
            ax.imshow(image, cmap=self.colormap_var.get(), origin='lower', interpolation='nearest', vmin=vmin, vmax=vmax)
            field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None and idx < len(self.real_fields_mT) else float(idx)
            ax.set_xlabel('X pixel', fontsize=11)
            ax.set_ylabel('Y pixel', fontsize=11)
            ax.set_title(f'Secondary ROI | Frame {idx + 1}/{nframes} | Field = {field:+.2f} mT', fontsize=13, pad=10)
            ax.tick_params(labelsize=9)
            ax.grid(False)
            fig.canvas.draw()
            frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            h, w = frame.shape[:2]
            if h % 2 or w % 2:
                frame = np.pad(frame, ((0, h % 2), (0, w % 2), (0, 0)), mode='edge')
            writer.append_data(frame)
            self.roi2_export_status_var.set(f'Exporting annotated MP4: {count}/{len(indices)} | frames {start}–{end} | {fps} FPS | {dpi} DPI')
            self.root.update_idletasks()
    finally:
        try:
            if writer is not None:
                writer.close()
        except Exception:
            pass
        plt.close(fig)
    self.roi2_export_status_var.set(f'✓ MP4 saved | frames {start}–{end} | {fps} FPS | {dpi} DPI | X/Y scale + Frame + Field included')
    self.log(f'Secondary ROI annotated MP4 saved: {path} | frames={start}-{end} | fps={fps} | dpi={dpi} | includes X/Y scale + field + frame number.')
    messagebox.showinfo('Secondary ROI MP4 Export', f'MP4 export complete.\n\nFrames: {start}–{end}\nFPS: {fps}\nDPI: {dpi}\n\nEvery video frame contains X/Y scale, frame number and field (mT).', parent=self.root)

def _roi2_update_fps_display(self):
    try:
        fps = max(1, min(120, int(self.fps_var.get())))
        self.fps_var.set(fps)
        self._roi2_fps_display_var.set(f'{fps} FPS')
    except Exception:
        self.fps_var.set(10)
        self._roi2_fps_display_var.set('10 FPS')

def _roi2_use_all_frames(self):
    if self.roi2_stack is None:
        return
    n = len(self.roi2_stack)
    self.roi2_start_frame_var.set(1)
    self.roi2_end_frame_var.set(n)
    self._roi2_update_fps_display()
    self.roi2_export_status_var.set(f'Export range set to frames 1–{n}.')

def _roi2_sync_controls(self):
    """Synchronize Secondary ROI slider, FPS and export-range controls."""
    if not hasattr(self, 'roi2_slider'):
        return
    self._roi2_update_fps_display()
    if self.roi2_stack is None:
        self.roi2_slider.configure(from_=0, to=0, state='disabled')
        self.roi2_idx_var.set(0)
        self.roi2_start_frame_var.set(1)
        self.roi2_end_frame_var.set(1)
        if hasattr(self, 'roi2_play_button'):
            self.roi2_play_button.configure(state='disabled', text='▶ PLAY')
        if hasattr(self, 'roi2_first_button'):
            self.roi2_first_button.configure(state='disabled')
        if hasattr(self, 'roi2_export_status_var'):
            self.roi2_export_status_var.set('No confirmed Secondary ROI available for export.')
        return
    n = int(len(self.roi2_stack))
    self.roi2_slider.configure(from_=0, to=max(0, n - 1), state='normal')
    if hasattr(self, 'roi2_play_button'):
        self.roi2_play_button.configure(state='normal', text='⏸ PAUSE' if self._roi2_playing else '▶ PLAY')
    idx = max(0, min(int(self.roi2_idx_var.get()), n - 1))
    self.roi2_idx_var.set(idx)
    try:
        start = int(self.roi2_start_frame_var.get())
    except Exception:
        start = 1
    try:
        end = int(self.roi2_end_frame_var.get())
    except Exception:
        end = n
    start = max(1, min(start, n))
    end = max(start, min(end, n))
    self.roi2_start_frame_var.set(start)
    self.roi2_end_frame_var.set(end)
    if hasattr(self, 'roi2_play_button'):
        self.roi2_play_button.configure(state='normal')
    if hasattr(self, 'roi2_first_button'):
        self.roi2_first_button.configure(state='normal')

def batch_export_roi2_images(self):
    """Export annotated Secondary ROI images for the selected frame range.

    JPEG is encoded ONLY through PIL after Matplotlib has rendered the figure
    into its in-memory RGBA buffer. This completely avoids FigureCanvasAgg
    PRINT_JPG_DISABLED()/quality keyword compatibility issues.
    """
    if self.roi2_stack is None:
        messagebox.showwarning('Secondary ROI Image Export', 'Confirm the Secondary ROI in Tab 5 first.', parent=self.root)
        return
    try:
        start = int(self.roi2_start_frame_var.get())
        end = int(self.roi2_end_frame_var.get())
        dpi = int(self.roi2_export_dpi_var.get())
        fmt = str(self.roi2_image_format_var.get()).strip().upper()
    except Exception:
        messagebox.showerror('Secondary ROI Image Export', 'Start frame, end frame and DPI must be valid values.', parent=self.root)
        return
    nframes = len(self.roi2_stack)
    if not 1 <= start <= end <= nframes:
        messagebox.showwarning('Secondary ROI Image Export', f'Frame range must satisfy 1 <= Start <= End <= {nframes}.', parent=self.root)
        return
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Secondary ROI Image Export', 'Select DPI: 150, 300, 600 or 900.', parent=self.root)
        return
    if fmt not in ('PNG', 'JPEG', 'TIFF'):
        messagebox.showwarning('Secondary ROI Image Export', 'Select PNG, JPEG or TIFF.', parent=self.root)
        return
    self._roi2_stop_playback()
    folder = self.output_var.get()
    os.makedirs(folder, exist_ok=True)
    ext = {'PNG': 'png', 'JPEG': 'jpg', 'TIFF': 'tif'}[fmt]
    total = end - start + 1
    fig = plt.figure(figsize=(8.0, 6.5), dpi=dpi)
    ax = fig.add_axes([0.11, 0.12, 0.84, 0.8])
    ax.set_aspect('equal', adjustable='box')
    try:
        for count, idx in enumerate(range(start - 1, end), start=1):
            image = np.asarray(self.roi2_stack[idx], dtype=float)
            ax.clear()
            finite = np.isfinite(image)
            if np.any(finite):
                vmin = float(np.nanmin(image[finite]))
                vmax = float(np.nanmax(image[finite]))
                if vmax <= vmin:
                    vmax = vmin + 1e-12
            else:
                vmin, vmax = (0.0, 1.0)
            ax.imshow(image, cmap=self.colormap_var.get(), origin='lower', interpolation='nearest', vmin=vmin, vmax=vmax)
            field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None and idx < len(self.real_fields_mT) else float(idx)
            ax.set_xlabel('X pixel', fontsize=11)
            ax.set_ylabel('Y pixel', fontsize=11)
            ax.set_title(f'Secondary ROI | Frame {idx + 1}/{nframes} | Field = {field:+.2f} mT', fontsize=13, pad=10)
            ax.tick_params(labelsize=9)
            ax.grid(False)
            path = os.path.join(folder, f'SecondaryROI_frame_{idx + 1:04d}_{field:+.2f}mT_{dpi}dpi.{ext}')
            fig.canvas.draw()
            rgba = np.asarray(fig.canvas.buffer_rgba())
            rgb = rgba[:, :, :3].copy()
            pil_img = Image.fromarray(rgb, mode='RGB')
            if fmt == 'JPEG':
                pil_img.save(path, format='JPEG', dpi=(dpi, dpi), subsampling=0)
            elif fmt == 'PNG':
                pil_img.save(path, format='PNG', dpi=(dpi, dpi))
            else:
                pil_img.save(path, format='TIFF', dpi=(dpi, dpi), compression='tiff_lzw')
            self.roi2_export_status_var.set(f'Exporting {fmt}: {count}/{total} | frames {start}–{end} | {dpi} DPI')
            self.root.update_idletasks()
        self.roi2_export_status_var.set(f'✓ {total} {fmt} image(s) saved | frames {start}–{end} | {dpi} DPI')
        self.log(f'Secondary ROI {fmt} export: frames={start}-{end} | count={total} | dpi={dpi}')
        messagebox.showinfo('Secondary ROI Image Export', f'Export complete.\n\nFormat: {fmt}\nResolution: {dpi} DPI\nFrames: {start}–{end}\nImages: {total}', parent=self.root)
    except Exception as exc:
        self.roi2_export_status_var.set(f'✗ Image export failed: {exc}')
        self.log(f'Secondary ROI image export failed: {exc}')
        messagebox.showerror('Secondary ROI Image Export', str(exc), parent=self.root)
    finally:
        try:
            plt.close(fig)
        except Exception:
            pass

def _roi2_build(self):
    """Secondary ROI viewer/playback with selectable export DPI."""
    self.roi2_idx_var = tk.IntVar(value=0)
    self.roi2_start_frame_var = tk.IntVar(value=1)
    self.roi2_end_frame_var = tk.IntVar(value=1)
    self.roi2_image_format_var = tk.StringVar(value='PNG')
    self.roi2_export_dpi_var = tk.IntVar(value=300)
    self._roi2_fps_display_var = tk.StringVar(value='10 FPS')
    self.roi2_field_var = tk.StringVar(value='Frame = -- | Field = --')
    self.roi2_status_var = tk.StringVar(value='Confirmed in 5. ROI Processing. This tab shows only the selected secondary ROI.')
    self.roi2_export_status_var = tk.StringVar(value='')
    top = ttk.Frame(self.tab_roi2, padding=8)
    top.pack(fill='x')
    ttk.Label(top, text='SECONDARY ROI — ZOOMED VIEW / PLAYBACK', font=('Arial', 12, 'bold')).pack(side='left')
    ttk.Label(top, textvariable=self.roi2_field_var).pack(side='left', padx=18)
    ttk.Label(top, textvariable=self.roi2_status_var, wraplength=700).pack(side='left', padx=12)
    controls = ttk.Frame(self.tab_roi2, padding=(8, 0, 8, 8))
    controls.pack(fill='x')
    ttk.Label(controls, text='FRAME SLIDER').pack(side='left')
    self.roi2_slider = tk.Scale(controls, from_=0, to=0, orient='horizontal', variable=self.roi2_idx_var, resolution=1, length=300, showvalue=True, command=lambda _: self._roi2_slider_changed())
    self.roi2_slider.pack(side='left', padx=6)
    ttk.Label(controls, text='FPS').pack(side='left', padx=(10, 3))
    self.roi2_fps_spin = tk.Spinbox(controls, from_=1, to=120, textvariable=self.fps_var, width=6, increment=1)
    self.roi2_fps_spin.pack(side='left', padx=(0, 4))
    self.roi2_fps_spin.bind('<Return>', lambda e: self._roi2_update_fps_display())
    self.roi2_fps_spin.bind('<FocusOut>', lambda e: self._roi2_update_fps_display())
    self.roi2_play_button = ttk.Button(controls, text='▶ PLAY', command=self.toggle_roi2_playback, state='disabled')
    self.roi2_play_button.pack(side='left', padx=6)
    self.roi2_first_button = ttk.Button(controls, text='↩ FIRST FRAME', command=self.go_to_first_roi2_frame, state='disabled')
    self.roi2_first_button.pack(side='left', padx=3)
    range_box = ttk.LabelFrame(self.tab_roi2, text='SECONDARY ROI — BATCH EXPORT', padding=7)
    range_box.pack(fill='x', padx=8, pady=(0, 6))
    ttk.Label(range_box, text='Start frame no.').grid(row=0, column=0, padx=(4, 3), pady=3, sticky='w')
    self.roi2_start_frame_spin = tk.Spinbox(
        range_box, from_=1, to=1, textvariable=self.roi2_start_frame_var,
        width=8, increment=1
    )
    self.roi2_start_frame_spin.grid(row=0, column=1, padx=3, pady=3, sticky='w')
    ttk.Label(range_box, text='End frame no.').grid(row=0, column=2, padx=(14, 3), pady=3, sticky='w')
    self.roi2_end_frame_spin = tk.Spinbox(
        range_box, from_=1, to=1, textvariable=self.roi2_end_frame_var,
        width=8, increment=1
    )
    self.roi2_end_frame_spin.grid(row=0, column=3, padx=3, pady=3, sticky='w')
    ttk.Label(range_box, text='Select DPI before export').grid(row=0, column=4, padx=(15, 3), pady=3, sticky='w')
    self.roi2_dpi_combo = ttk.Combobox(range_box, textvariable=self.roi2_export_dpi_var, values=(150, 300, 600, 900), state='readonly', width=7)
    self.roi2_dpi_combo.grid(row=0, column=5, padx=3, pady=3, sticky='w')
    ttk.Button(range_box, text='BATCH EXPORT MP4', command=self.batch_export_roi2_mp4).grid(row=0, column=6, padx=(12, 5), pady=3, sticky='ew')
    ttk.Label(range_box, text='Image format').grid(row=0, column=7, padx=(10, 3), pady=3, sticky='w')
    self.roi2_format_combo = ttk.Combobox(range_box, textvariable=self.roi2_image_format_var, values=('PNG', 'JPEG', 'TIFF'), state='readonly', width=8)
    self.roi2_format_combo.grid(row=0, column=8, padx=3, pady=3, sticky='w')
    ttk.Button(range_box, text='BATCH EXPORT IMAGES', command=self.batch_export_roi2_images).grid(row=0, column=9, padx=(8, 5), pady=3, sticky='ew')
    ttk.Button(range_box, text='USE ALL FRAMES', command=self._roi2_use_all_frames).grid(row=0, column=10, padx=5, pady=3, sticky='ew')
    ttk.Label(range_box, text='Select DPI before export: 150 / 300 / 600 / 900. The selected DPI is applied to both exports; images/video include X/Y scale, frame number and field (mT).').grid(row=1, column=0, columnspan=11, padx=4, pady=(2, 0), sticky='w')
    ttk.Label(range_box, textvariable=self.roi2_export_status_var, wraplength=1200).grid(row=2, column=0, columnspan=11, padx=4, pady=(2, 0), sticky='w')
    image_frame = ttk.Frame(self.tab_roi2)
    image_frame.pack(fill='both', expand=True, padx=8, pady=(0, 8))
    self.roi2_fig, self.roi2_ax = plt.subplots(figsize=(9, 7), dpi=100)
    self.roi2_image = self.roi2_ax.imshow(np.zeros((10, 10), dtype=float), cmap=self.colormap_var.get(), origin='lower', vmin=0, vmax=1, interpolation='nearest')
    self.roi2_colorbar = self.roi2_fig.colorbar(self.roi2_image, ax=self.roi2_ax, fraction=0.046, pad=0.05)
    self.roi2_colorbar.set_label('Selected ROI contrast', rotation=90)
    self.roi2_ax.set_title('SECONDARY ROI — ZOOMED VIEW')
    self.roi2_ax.set_xlabel('X pixel')
    self.roi2_ax.set_ylabel('Y pixel')
    self.roi2_ax.axis('image')
    self.roi2_canvas = FigureCanvasTkAgg(self.roi2_fig, master=image_frame)
    self.roi2_canvas.draw()
    self.roi2_canvas.get_tk_widget().pack(fill='both', expand=True)
    self._roi2_sync_controls()
    if self.recons is not None:
        self.roi2_slider.configure(from_=0, to=max(0, len(self.recons) - 1), state='disabled' if self.roi2_stack is None else 'normal')
    self.nb.bind('<<NotebookTabChanged>>', lambda event: self.update_roi2_view() if event.widget.select() == str(self.tab_roi2) else None)

def _roi2_source_image(self, idx):
    """Return exactly the processed image used in ROI Processing."""
    if self.recons is None:
        raise RuntimeError('Load reconstructions first.')
    if getattr(self, 'roi_resolution_applied', False):
        return np.asarray(_proc_image(self, idx), dtype=float)
    raise RuntimeError('Confirm the primary ROI and apply the ROI Processing resolution first.')

def _roi2_slider_changed(self):
    """Handle the Secondary ROI frame slider."""
    if self.roi2_stack is None:
        return
    if self._roi2_playing:
        self._roi2_stop_playback(update_status=False)
    n = len(self.roi2_stack)
    idx = max(0, min(int(round(float(self.roi2_idx_var.get()))), n - 1))
    self.roi2_idx_var.set(idx)
    self.current_index = idx
    self.update_roi2_view()

def _roi2_update_fps_display(self):
    try:
        fps = max(1, min(120, int(self.fps_var.get())))
        self.fps_var.set(fps)
        self._roi2_fps_display_var.set(f'{fps} FPS')
    except Exception:
        self.fps_var.set(10)
        self._roi2_fps_display_var.set('10 FPS')

def _roi2_use_all_frames(self):
    if self.roi2_stack is None:
        return
    n = len(self.roi2_stack)
    self.roi2_start_frame_var.set(1)
    self.roi2_end_frame_var.set(n)
    self._roi2_update_fps_display()
    self.roi2_export_status_var.set(f'Export range set to frames 1–{n}.')

def _roi2_sync_controls(self):
    """Synchronize Secondary ROI slider, FPS and export-range controls."""
    if not hasattr(self, 'roi2_slider'):
        return
    self._roi2_update_fps_display()
    if self.roi2_stack is None:
        self.roi2_slider.configure(from_=0, to=0, state='disabled')
        self.roi2_idx_var.set(0)
        self.roi2_start_frame_var.set(1)
        self.roi2_end_frame_var.set(1)
        if hasattr(self, 'roi2_play_button'):
            self.roi2_play_button.configure(state='disabled', text='▶ PLAY')
        if hasattr(self, 'roi2_first_button'):
            self.roi2_first_button.configure(state='disabled')
        if hasattr(self, 'roi2_export_status_var'):
            self.roi2_export_status_var.set('No confirmed Secondary ROI available for export.')
        return
    n = int(len(self.roi2_stack))
    self.roi2_slider.configure(from_=0, to=max(0, n - 1), state='normal')
    _set_frame_slider_midpoint_once(self,self.roi2_slider,n,self.roi2_idx_var,zero_based=True,token='roi2')
    if hasattr(self, 'roi2_play_button'):
        self.roi2_play_button.configure(state='normal', text='⏸ PAUSE' if self._roi2_playing else '▶ PLAY')
    idx = max(0, min(int(self.roi2_idx_var.get()), n - 1))
    self.roi2_idx_var.set(idx)
    try:
        start = int(self.roi2_start_frame_var.get())
    except Exception:
        start = 1
    try:
        end = int(self.roi2_end_frame_var.get())
    except Exception:
        end = n
    start = max(1, min(start, n))
    end = max(start, min(end, n))
    self.roi2_start_frame_var.set(start)
    self.roi2_end_frame_var.set(end)
    if hasattr(self, 'roi2_play_button'):
        self.roi2_play_button.configure(state='normal')
    if hasattr(self, 'roi2_first_button'):
        self.roi2_first_button.configure(state='normal')

def update_roi2_view(self):
    """Display ONLY the confirmed secondary ROI crop."""
    if self.recons is None:
        return
    self._roi2_sync_controls()
    if self.roi2_stack is None:
        self.roi2_status_var.set('No confirmed secondary ROI. Select/confirm it in 5. ROI Processing.')
        if self.roi2_ax is not None:
            self.roi2_ax.set_title('SECONDARY ROI — waiting for confirmation')
            self.roi2_ax.set_xlabel('Secondary ROI X pixel')
            self.roi2_ax.set_ylabel('Secondary ROI Y pixel')
        if self.roi2_canvas is not None:
            self.roi2_canvas.draw_idle()
        return
    nframes = len(self.roi2_stack)
    idx = max(0, min(int(self.roi2_idx_var.get()), nframes - 1))
    self.roi2_idx_var.set(idx)
    self.current_index = idx
    image = np.asarray(self.roi2_stack[idx], dtype=np.float32)
    h, w = image.shape[:2]
    cmap_name = self.colormap_var.get()
    if getattr(self, 'roi2_image', None) is None or getattr(self.roi2_image, 'axes', None) is None or getattr(self.roi2_image, 'figure', None) is None:
        self.roi2_image = self.roi2_ax.imshow(image, cmap=cmap_name, origin='lower', vmin=0, vmax=1, interpolation='nearest', extent=(-0.5, w - 0.5, -0.5, h - 0.5))
    else:
        self.roi2_image.set_cmap(cmap_name)
        self.roi2_image.set_data(image)
        self.roi2_image.set_extent((-0.5, w - 0.5, -0.5, h - 0.5))
        finite = np.isfinite(image)
        if np.any(finite):
            data_min = float(np.nanmin(image[finite]))
            data_max = float(np.nanmax(image[finite]))
            if np.isclose(data_min, data_max):
                pad = max(abs(data_min) * 0.05, 1e-06)
                vmin, vmax = (data_min - pad, data_max + pad)
            elif data_min < 0.0 < data_max:
                lim = max(abs(data_min), abs(data_max))
                vmin, vmax = (-lim, lim)
            else:
                vmin, vmax = (data_min, data_max)
        else:
            vmin, vmax = (0.0, 1.0)
        self.roi2_image.set_clim(vmin, vmax)
        if hasattr(self, 'roi2_colorbar') and self.roi2_colorbar is not None:
            self.roi2_colorbar.update_normal(self.roi2_image)
            self.roi2_colorbar.set_label(f'Selected ROI contrast [{vmin:.4g}, {vmax:.4g}]', rotation=90)
    self.roi2_ax.set_xlim(-0.5, w - 0.5)
    self.roi2_ax.set_ylim(-0.5, h - 0.5)
    self.roi2_ax.set_aspect('equal', adjustable='box')
    if self._roi2_patch is not None:
        try:
            self._roi2_patch.remove()
        except Exception:
            pass
        self._roi2_patch = None
    if self.roi2_confirmed_coords is not None:
        c = self.roi2_confirmed_coords
        self.roi2_ax.set_title(f"SECONDARY ROI — ZOOMED VIEW | {w} × {h} px | Parent X={int(c['xmin'])}:{int(c['xmax'])}, Y={int(c['ymin'])}:{int(c['ymax'])}", fontsize=12)
    else:
        self.roi2_ax.set_title(f'SECONDARY ROI — ZOOMED VIEW | {w} × {h} px', fontsize=12)
    self.roi2_ax.set_xlabel('Secondary ROI X pixel')
    self.roi2_ax.set_ylabel('Secondary ROI Y pixel')
    field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    self.roi2_field_var.set(f'Frame {idx + 1}/{nframes} | Field = {field:+.2f} mT | Color map = {cmap_name} | Scale = {vmin:.4g} to {vmax:.4g}')
    self._roi2_fps_display_var.set(f'{max(1, int(self.fps_var.get()))} FPS')
    self.roi2_status_var.set(f'✓ Confirmed secondary ROI | {w} × {h} px | frame {idx + 1}/{nframes} | {cmap_name}')
    if hasattr(self, 'roi2_slider'):
        self.roi2_slider.set(idx)
    self.roi2_canvas.draw_idle()

def _roi2_on_press(self, event):
    return

def _roi2_on_move(self, event):
    return

def _roi2_on_release(self, event):
    return

def confirm_roi2(self):
    if self.recons is None:
        messagebox.showwarning('Secondary ROI', 'Load reconstructions first.', parent=self.root)
        return
    if self.roi2_coords is None:
        messagebox.showwarning('Secondary ROI', 'Drag a rectangle on the processed image first.', parent=self.root)
        return
    self._roi2_stop_playback()
    idx = max(0, min(int(self.roi2_idx_var.get()), len(self.recons) - 1))
    try:
        reference = self._roi2_source_image(idx)
    except Exception as exc:
        messagebox.showerror('Secondary ROI', str(exc), parent=self.root)
        return
    x0 = self.roi2_coords['xmin']
    x1 = self.roi2_coords['xmax']
    y0 = self.roi2_coords['ymin']
    y1 = self.roi2_coords['ymax']
    h, w = reference.shape[:2]
    x0, x1 = (max(0, min(x0, w - 1)), max(0, min(x1, w - 1)))
    y0, y1 = (max(0, min(y0, h - 1)), max(0, min(y1, h - 1)))
    mask = np.zeros((h, w), dtype=bool)
    mask[y0:y1 + 1, x0:x1 + 1] = True
    if getattr(self, 'local_mask', None) is not None:
        lm = np.asarray(self.local_mask, dtype=bool)
        if lm.shape == mask.shape:
            mask &= lm
    if not np.any(mask):
        messagebox.showerror('Secondary ROI', 'Selected ROI contains no valid processed pixels.', parent=self.root)
        return
    self.roi2_mask = mask
    self.roi2_coords = {'xmin': x0, 'xmax': x1, 'ymin': y0, 'ymax': y1}
    self.roi2_confirmed_coords = dict(self.roi2_coords)
    self.roi2_stack = np.zeros((len(self.recons), y1 - y0 + 1, x1 - x0 + 1), dtype=np.float32)
    local_mask = mask[y0:y1 + 1, x0:x1 + 1]
    for i in range(len(self.recons)):
        img = self._roi2_source_image(i)
        crop = np.asarray(img[y0:y1 + 1, x0:x1 + 1], dtype=np.float32)
        crop[~local_mask] = 0.0
        self.roi2_stack[i] = crop
    self.roi2_status_var.set(f'✓ SECONDARY ROI CONFIRMED | X={x0}:{x1} | Y={y0}:{y1} | {x1 - x0 + 1} × {y1 - y0 + 1} px | {len(self.recons)} frames')
    self.log(f'Secondary ROI confirmed: X={x0}:{x1}, Y={y0}:{y1}, size={x1 - x0 + 1}x{y1 - y0 + 1}, frames={len(self.recons)}')
    self._roi2_sync_controls()
    self.roi2_idx_var.set(max(0, min(int(self.proc_idx_var.get()), len(self.roi2_stack) - 1)) if hasattr(self, 'proc_idx_var') else 0)
    self.update_roi2_view()
    try:
        sync = getattr(self, '_pd_sync_secondary_roi', None)
        if callable(sync):
            sync()
    except Exception:
        pass

def clear_roi2(self):
    self._roi2_stop_playback()
    self.roi2_coords = None
    self.roi2_confirmed_coords = None
    self.roi2_mask = None
    self.roi2_stack = None
    self._roi2_press_xy = None
    if self._roi2_patch is not None:
        try:
            self._roi2_patch.remove()
        except Exception:
            pass
        self._roi2_patch = None
    self.roi2_status_var.set('Secondary ROI cleared. Drag a new rectangle and confirm it.')
    if self.recons is not None:
        self.update_roi2_view()

def _roi2_stop_playback(self, update_status=True):
    """Stop scheduled Tab-6 playback safely."""
    self._roi2_playing = False
    job = getattr(self, '_roi2_play_job', None)
    if job is not None:
        try:
            self.root.after_cancel(job)
        except Exception:
            pass
        self._roi2_play_job = None
    if hasattr(self, 'roi2_play_button'):
        try:
            self.roi2_play_button.configure(text='▶ PLAY')
        except Exception:
            pass
    if update_status and self.roi2_stack is not None:
        idx = max(0, min(int(self.roi2_idx_var.get()), len(self.roi2_stack) - 1))
        self.roi2_status_var.set(f'Playback paused | frame {idx + 1}/{len(self.roi2_stack)}')

def _roi2_play_button_double_click(self, event=None):
    """Legacy compatibility: double-click now pauses without changing the frame."""
    self._roi2_stop_playback(update_status=True)
    return 'break'

def toggle_roi2_playback(self):
    """Single-click toggle: PLAY -> PAUSE -> RESUME."""
    if self.roi2_stack is None or len(self.roi2_stack) == 0:
        messagebox.showwarning('Secondary ROI', 'Confirm the secondary ROI in 5. ROI Processing first.', parent=self.root)
        return
    if self._roi2_playing:
        self._roi2_stop_playback(update_status=True)
        return
    self.play_roi2_frames()

def play_roi2_frames(self):
    """Start or resume confirmed Secondary ROI playback from the current frame."""
    if self.roi2_stack is None or len(self.roi2_stack) == 0:
        messagebox.showwarning('Secondary ROI', 'Confirm the secondary ROI in 5. ROI Processing first.', parent=self.root)
        return
    self._roi2_sync_controls()
    n = len(self.roi2_stack)
    idx = max(0, min(int(round(float(self.roi2_idx_var.get()))), n - 1))
    if idx >= n - 1:
        idx = 0
    self.roi2_idx_var.set(idx)
    self.current_index = idx
    self._roi2_playing = True
    if hasattr(self, 'roi2_play_button'):
        self.roi2_play_button.configure(text='⏸ PAUSE')
    fps = max(1, min(120, int(self.fps_var.get())))
    self.roi2_status_var.set(f'▶ Playing at {fps} FPS | frame {idx + 1}/{n}')
    if self._roi2_play_job is not None:
        try:
            self.root.after_cancel(self._roi2_play_job)
        except Exception:
            pass
        self._roi2_play_job = None
    self.update_roi2_view()
    self._roi2_play_job = self.root.after(max(1, int(round(1000.0 / fps))), self._roi2_play_next)

def _roi2_play_next(self):
    """Advance exactly one frame and schedule the next callback."""
    self._roi2_play_job = None
    if not self._roi2_playing or self.roi2_stack is None:
        return
    n = len(self.roi2_stack)
    if n == 0:
        self._roi2_stop_playback(update_status=False)
        return
    idx = max(0, min(int(round(float(self.roi2_idx_var.get()))), n - 1))
    if idx >= n - 1:
        self.roi2_idx_var.set(n - 1)
        self.update_roi2_view()
        self._roi2_playing = False
        self.roi2_play_button.configure(text='▶ PLAY')
        self.roi2_status_var.set(f'Playback finished | frame {n}/{n}')
        return
    idx += 1
    self.roi2_idx_var.set(idx)
    self.current_index = idx
    self.update_roi2_view()
    fps = max(1, min(120, int(self.fps_var.get())))
    self.roi2_status_var.set(f'▶ Playing at {fps} FPS | frame {idx + 1}/{n}')
    self._roi2_play_job = self.root.after(max(1, int(round(1000.0 / fps))), self._roi2_play_next)

def go_to_first_roi2_frame(self):
    """Stop playback and jump to secondary-ROI frame 1."""
    self._roi2_stop_playback()
    if self.roi2_stack is None:
        self.roi2_idx_var.set(0)
        self.roi2_status_var.set('No confirmed secondary ROI. Confirm it in 5. ROI Processing.')
        return
    self.roi2_idx_var.set(0)
    self.current_index = 0
    self._roi2_sync_controls()
    self.update_roi2_view()
    if hasattr(self, 'roi2_play_button'):
        self.roi2_play_button.configure(text='▶ PLAY', state='normal')
    self.roi2_status_var.set(f'Playback ready | frame 1/{len(self.roi2_stack)}')

def start_secondary_roi_selection(self):
    """Enable mouse-drag selection of a secondary ROI on the processed image."""
    if self.recons is None or self.roi_mask is None:
        messagebox.showwarning('Secondary ROI', 'Confirm the primary ROI and load the processed data first.', parent=self.root)
        return
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('Secondary ROI', 'Apply the confirmed ROI resolution first.', parent=self.root)
        return
    if hasattr(self, '_roi2_stop_playback'):
        try:
            self._roi2_stop_playback()
        except Exception:
            pass
    self._proc_secondary_roi_selecting = True
    self._proc_secondary_press_xy = None
    try:
        self.proc_secondary_select_button.configure(text='DRAWING — DRAG ON IMAGE')
    except Exception:
        pass
    self.proc_secondary_status_var.set('DRAWING MODE ON: left-click and drag a rectangle on the processed image.')

def _proc_secondary_on_press(self, event):
    if not getattr(self, '_proc_secondary_roi_selecting', False):
        return
    if event.inaxes != self._proc_ax:
        return
    if event.button != 1 or event.xdata is None or event.ydata is None:
        return
    self._proc_secondary_press_xy = (float(event.xdata), float(event.ydata))
    if self._proc_secondary_patch is not None:
        try:
            self._proc_secondary_patch.remove()
        except Exception:
            pass
        self._proc_secondary_patch = None
    self.proc_secondary_status_var.set('Dragging secondary ROI — release the mouse to finish.')
    self._proc_canvas.draw_idle()

def _proc_secondary_on_move(self, event):
    if not getattr(self, '_proc_secondary_roi_selecting', False) or self._proc_secondary_press_xy is None:
        return
    if event.inaxes != self._proc_ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    x0, y0 = self._proc_secondary_press_xy
    x1, y1 = (float(event.xdata), float(event.ydata))
    xmin, xmax = sorted((x0, x1))
    ymin, ymax = sorted((y0, y1))
    if self._proc_secondary_patch is None:
        self._proc_secondary_patch = plt.Rectangle((xmin, ymin), max(1e-06, xmax - xmin), max(1e-06, ymax - ymin), fill=False, edgecolor='red', linewidth=2.5, zorder=20)
        self._proc_ax.add_patch(self._proc_secondary_patch)
    else:
        self._proc_secondary_patch.set_xy((xmin, ymin))
        self._proc_secondary_patch.set_width(max(1e-06, xmax - xmin))
        self._proc_secondary_patch.set_height(max(1e-06, ymax - ymin))
    self._proc_canvas.draw_idle()

def _proc_secondary_on_release(self, event):
    if not getattr(self, '_proc_secondary_roi_selecting', False) or self._proc_secondary_press_xy is None:
        return
    if event.inaxes != self._proc_ax or event.xdata is None or event.ydata is None:
        self._proc_secondary_press_xy = None
        return
    x0, y0 = self._proc_secondary_press_xy
    x1, y1 = (float(event.xdata), float(event.ydata))
    self._proc_secondary_press_xy = None
    display = self._proc_image_display.get_array()
    if display is None:
        return
    h, w = np.asarray(display).shape[:2]
    xmin = max(0, min(int(np.floor(min(x0, x1))), w - 1))
    xmax = max(0, min(int(np.ceil(max(x0, x1))), w - 1))
    ymin = max(0, min(int(np.floor(min(y0, y1))), h - 1))
    ymax = max(0, min(int(np.ceil(max(y0, y1))), h - 1))
    if xmax <= xmin or ymax <= ymin:
        self.proc_secondary_status_var.set('Secondary ROI too small. Drag a larger rectangle.')
        return
    self.roi2_coords = {'xmin': xmin, 'xmax': xmax, 'ymin': ymin, 'ymax': ymax}
    if hasattr(self, 'roi2_idx_var'):
        self.roi2_idx_var.set(max(0, min(int(self.proc_idx_var.get()), len(self.recons) - 1)))
    self.proc_secondary_status_var.set(f'Secondary ROI selected: X={xmin}:{xmax}, Y={ymin}:{ymax}. Click CONFIRM SECONDARY ROI.')
    self._proc_secondary_roi_selecting = False
    try:
        self.proc_secondary_select_button.configure(text='SELECT SECONDARY ROI')
    except Exception:
        pass
    self._proc_draw_secondary_roi_patch(confirmed=False)

def _proc_draw_secondary_roi_patch(self, confirmed=False, redraw=True):
    """Draw the current tentative/confirmed secondary ROI on Processing."""
    if self._proc_secondary_patch is not None:
        try:
            self._proc_secondary_patch.remove()
        except Exception:
            pass
        self._proc_secondary_patch = None
    if getattr(self, 'roi2_coords', None) is None:
        if redraw:
            self._proc_canvas.draw_idle()
        return
    c = self.roi2_coords
    edge = 'lime' if confirmed else 'red'
    self._proc_secondary_patch = plt.Rectangle((c['xmin'], c['ymin']), c['xmax'] - c['xmin'] + 1, c['ymax'] - c['ymin'] + 1, fill=False, edgecolor=edge, linewidth=2.5, zorder=20)
    self._proc_ax.add_patch(self._proc_secondary_patch)
    if redraw:
        self._proc_canvas.draw_idle()

def confirm_secondary_roi_from_processing(self):
    """Confirm the ROI selected directly on ROI Processing."""
    if getattr(self, 'roi2_coords', None) is None:
        messagebox.showwarning('Secondary ROI', 'First click SELECT SECONDARY ROI and drag a rectangle on the processed image.', parent=self.root)
        return
    if self.recons is None:
        return
    if hasattr(self, 'roi2_idx_var'):
        self.roi2_idx_var.set(max(0, min(int(self.proc_idx_var.get()), len(self.recons) - 1)))
    try:
        self.confirm_roi2()
        if self.roi2_coords is not None:
            self.roi2_confirmed_coords = dict(self.roi2_coords)
    except Exception as exc:
        messagebox.showerror('Secondary ROI', f'Failed to confirm secondary ROI:\n{type(exc).__name__}: {exc}', parent=self.root)
        self.log(f'Secondary ROI confirmation failed: {type(exc).__name__}: {exc}')
        return
    self._proc_secondary_roi_selecting = False
    self._proc_draw_secondary_roi_patch(confirmed=True)
    self.proc_secondary_status_var.set('✓ SECONDARY ROI CONFIRMED for legacy ROI2 processing.')
    try:
        self.proc_secondary_select_button.configure(text='SELECT SECONDARY ROI')
    except Exception:
        pass

def clear_secondary_roi_from_processing(self):
    """Clear the secondary ROI from both Processing and Secondary ROI tabs."""
    self._proc_secondary_roi_selecting = False
    self._proc_secondary_press_xy = None
    if hasattr(self, '_roi2_stop_playback'):
        try:
            self._roi2_stop_playback()
        except Exception:
            pass
    if hasattr(self, 'clear_roi2'):
        try:
            self.clear_roi2()
        except Exception:
            pass
    if self._proc_secondary_patch is not None:
        try:
            self._proc_secondary_patch.remove()
        except Exception:
            pass
        self._proc_secondary_patch = None
    self.proc_secondary_status_var.set('Secondary ROI cleared.')
    try:
        self.proc_secondary_select_button.configure(text='SELECT SECONDARY ROI')
    except Exception:
        pass
    self._proc_canvas.draw_idle()

def _proc_set_roi_preset(self, width, height):
    ow = int(getattr(self, 'original_roi_w', 0) or 0)
    oh = int(getattr(self, 'original_roi_h', 0) or 0)
    if ow < 1 or oh < 1:
        self.roi_width_var.set(str(int(width)))
        self.roi_height_var.set(str(int(height)))
        return
    scale = min(float(width) / ow, float(height) / oh)
    out_w = max(1, int(round(ow * scale)))
    out_h = max(1, int(round(oh * scale)))
    self.roi_width_var.set(str(out_w))
    self.roi_height_var.set(str(out_h))
    if hasattr(self, 'proc_status_var'):
        self.proc_status_var.set(f'Resolution selected: {out_w} × {out_h} (aspect ratio preserved) — press APPLY SAME CONFIRMED ROI')

def _proc_update_roi_height_from_width(self):
    try:
        width = int(self.roi_width_var.get())
        ow = int(getattr(self, 'original_roi_w', 0) or 0)
        oh = int(getattr(self, 'original_roi_h', 0) or 0)
        if width < 1 or ow < 1 or oh < 1:
            return
        height = max(1, int(round(width * oh / ow)))
        self.roi_height_var.set(str(height))
    except Exception:
        pass

def _proc_set_original_roi_preset(self):
    if getattr(self, 'original_roi_w', None) and getattr(self, 'original_roi_h', None):
        _proc_set_roi_preset(self, self.original_roi_w, self.original_roi_h)

def _proc_resize_image(self, image, width, height):
    image = np.asarray(image, dtype=np.float32)
    image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
    image = np.clip(image, 0.0, 1.0)
    pil = Image.fromarray(np.uint8(image * 255.0), mode='L')
    return np.asarray(pil.resize((int(width), int(height)), Image.Resampling.BICUBIC), dtype=np.float32) / 255.0

def _proc_resize_mask(self, mask, width, height):
    pil = Image.fromarray(np.asarray(mask, dtype=np.uint8) * 255, mode='L')
    return np.asarray(pil.resize((int(width), int(height)), Image.Resampling.NEAREST)) > 127

def _proc_build_roi_stack(self):
    if self.roi_mask is None or self.recons is None:
        raise RuntimeError('Confirm ROI first.')
    mask = np.asarray(self.roi_mask, dtype=bool)
    yy, xx = np.where(mask)
    if len(xx) == 0:
        raise RuntimeError('Confirmed ROI contains no pixels.')
    x0, x1 = (int(xx.min()), int(xx.max()))
    y0, y1 = (int(yy.min()), int(yy.max()))
    self.roi_crop = {'xmin': x0, 'xmax': x1, 'ymin': y0, 'ymax': y1}
    local = mask[y0:y1 + 1, x0:x1 + 1]
    self.original_roi_h, self.original_roi_w = local.shape
    self.roi_aspect_ratio = float(self.original_roi_w) / float(self.original_roi_h)
    if hasattr(self, 'roi_width_var'):
        self.roi_width_var.set(str(self.original_roi_w))
    if hasattr(self, 'roi_height_var'):
        self.roi_height_var.set(str(self.original_roi_h))
    _proc_apply_resolution(self, self.original_roi_w, self.original_roi_h)

def _proc_apply_resolution(self, width, height):
    if self.roi_mask is None or self.recons is None:
        raise RuntimeError('Confirm ROI first.')
    width, height = (int(width), int(height))
    if width < 1:
        raise ValueError('ROI width must be greater than zero.')
    ow = int(getattr(self, 'original_roi_w', 0) or 0)
    oh = int(getattr(self, 'original_roi_h', 0) or 0)
    if ow > 0 and oh > 0:
        height = max(1, int(round(width * oh / ow)))
    yy, xx = np.where(np.asarray(self.roi_mask, dtype=bool))
    if len(xx) == 0:
        raise ValueError('ROI contains no pixels.')
    x0, x1 = (int(xx.min()), int(xx.max()))
    y0, y1 = (int(yy.min()), int(yy.max()))
    local_orig = np.asarray(self.roi_mask, dtype=bool)[y0:y1 + 1, x0:x1 + 1]
    resized_mask = _proc_resize_mask(self, local_orig, width, height)
    new_stack = np.zeros((len(self.recons), height, width), dtype=np.float32)
    for i in range(len(self.recons)):
        image = np.asarray(self.recons[i, y0:y1 + 1, x0:x1 + 1], dtype=float).copy()
        valid = local_orig & np.isfinite(image)
        if np.any(valid):
            values = image[valid]
            fill = float(np.median(values))
            vmin = float(np.percentile(values, 1))
            vmax = float(np.percentile(values, 99))
        else:
            fill, vmin, vmax = (0.0, 0.0, 1.0)
        image[~valid] = fill
        if vmax <= vmin:
            vmax = vmin + 1e-12
        image = np.clip((image - vmin) / (vmax - vmin), 0, 1)
        image[~local_orig] = 0.0
        r = _proc_resize_image(self, image, width, height)
        r[~resized_mask] = 0.0
        new_stack[i] = r
    self.roi_crop = {'xmin': x0, 'xmax': x1, 'ymin': y0, 'ymax': y1}
    self.roi_stack = new_stack
    self.local_mask = resized_mask
    self.original_roi_h, self.original_roi_w = local_orig.shape
    self.roi_resolution = (width, height)
    self.roi_resolution_applied = True
    self.roi_mean = np.mean(np.where(resized_mask[None, :, :], new_stack, np.nan), axis=(1, 2))
    self.roi_std = np.std(np.where(resized_mask[None, :, :], new_stack, np.nan), axis=(1, 2))
    if hasattr(self, 'proc_slider'):
        self.proc_slider.configure(to=max(0, len(self.recons) - 1))
        if hasattr(self, 'proc_idx_var'):
            self.proc_idx_var.set(max(0, min(int(self.roi_index_var.get()), len(self.recons) - 1)))
    if hasattr(self, 'proc_roi_info_var'):
        self.proc_roi_info_var.set(f'ROI = {width} × {height} px | pixels={int(resized_mask.sum())}')
    if hasattr(self, 'proc_status_var'):
        self.proc_status_var.set(f'SAME ROI APPLIED ✓  {width} × {height}  |  {len(self.recons)} images')
    try:
        self.update_processing_pixel_size()
    except Exception:
        pass
    if hasattr(self, 'clear_multi_rois'):
        try:
            self.clear_multi_rois()
        except Exception:
            pass
    self.log(f'SAME CONFIRMED ROI — resolution applied: original {(self.original_roi_w, self.original_roi_h)}, output {(width, height)}, stack={self.roi_stack.shape}')

def _proc_apply_from_gui(self):
    try:
        width = int(self.roi_width_var.get())
        height = int(self.roi_height_var.get())
        _proc_apply_resolution(self, width, height)
        self.update_processing_view()
    except Exception as e:
        messagebox.showerror('ROI Resolution Error', str(e), parent=self.root)

def _proc_image(self, index):
    if not getattr(self, 'roi_resolution_applied', False):
        raise RuntimeError('Apply SAME CONFIRMED ROI resolution first.')
    image = np.asarray(self.roi_stack[index], dtype=float)
    valid = self.local_mask & np.isfinite(image)
    if not np.any(valid):
        return np.zeros_like(image)
    vals = image[valid]
    clean = image.copy()
    clean[~valid] = float(np.median(vals))
    vmin = float(np.percentile(vals, 1))
    vmax = float(np.percentile(vals, 99))
    if vmax <= vmin:
        vmax = vmin + 1e-12
    image_norm = np.clip((clean - vmin) / (vmax - vmin), 0, 1)
    kind = self.filter_var.get()
    strength = max(float(self.strength_var.get()), 0.01)
    if kind in ('Gaussian', 'Low-pass'):
        processed = gaussian_filter(image_norm, sigma=strength)
    elif kind == 'High-pass':
        processed = image_norm - gaussian_filter(image_norm, sigma=strength)
        mn, mx = (float(processed.min()), float(processed.max()))
        processed = (processed - mn) / (mx - mn + 1e-12)
    elif kind == 'Band-pass':
        processed = gaussian_filter(image_norm, sigma=strength) - gaussian_filter(image_norm, sigma=strength * 2)
        mn, mx = (float(processed.min()), float(processed.max()))
        processed = (processed - mn) / (mx - mn + 1e-12)
    else:
        processed = image_norm.copy()
    processed = np.where(self.local_mask, processed, 0.0)
    processed = np.clip((processed - 0.5) * float(self.contrast_var.get()) + 0.5 + float(self.brightness_var.get()), 0, 1)
    return processed.astype(np.float32)

def _proc_fft(self, image):
    F = np.fft.fftshift(np.fft.fft2(image))
    mag = np.log1p(np.abs(F))
    mn, mx = (float(mag.min()), float(mag.max()))
    mag = (mag - mn) / (mx - mn + 1e-12)
    return (F, mag)

def _proc_fft_mask(self, F, radius):
    h, w = F.shape
    cy, cx = (h // 2, w // 2)
    yy, xx = np.ogrid[:h, :w]
    d = np.sqrt(((xx - cx) / max(cx, 1)) ** 2 + ((yy - cy) / max(cy, 1)) ** 2)
    m = d <= float(radius)
    return (F * m, m)

def _proc_update(self):
    """Render one Tab-6 frame while preserving physical axes and zoom."""
    if self.roi_mask is None or self.recons is None:
        return
    if not getattr(self, 'roi_resolution_applied', False):
        try:
            _proc_apply_resolution(self, int(self.roi_width_var.get()), int(self.roi_height_var.get()))
        except Exception:
            return
    idx = max(0, min(int(round(self.proc_idx_var.get())), len(self.recons) - 1))
    self.proc_idx_var.set(idx)
    self.current_index = idx
    image = _proc_image(self, idx)
    view = self.view_var.get()
    if view == 'Image':
        display = image
    elif view == 'FFT':
        _, display = _proc_fft(self, image)
    else:
        F, _ = _proc_fft(self, image)
        Fm, _ = _proc_fft_mask(self, F, self.fft_mask_var.get())
        r = np.abs(np.fft.ifft2(np.fft.ifftshift(Fm)))
        mn, mx = (float(r.min()), float(r.max()))
        display = (r - mn) / (mx - mn + 1e-12)
        display = np.where(self.local_mask, display, 0.0)
    self.current_processed = image
    self._proc_image_display.set_data(display)
    self._proc_image_display.set_cmap(self.colormap_var.get())
    self._proc_image_display.set_clim(0, 1)
    self._proc_display_shape = tuple(np.asarray(display).shape[:2])
    self._proc_update_physical_axes_and_colorbar(display)
    field = float(self.real_fields_mT[idx])
    self._proc_title.set_text(f'{view} | Field = {field:+.2f} mT | ROI = {self.roi_resolution[0]} × {self.roi_resolution[1]}')
    self.proc_field_var.set(f'Frame No. {idx + 1}/{len(self.recons)} | Equivalent Field: {field:+.2f} mT')
    self.proc_roi_info_var.set(f'ROI = {self.roi_resolution[0]} × {self.roi_resolution[1]} px | pixels={int(self.local_mask.sum())}')
    self._proc_draw_secondary_roi_patch(confirmed=getattr(self, 'roi2_confirmed_coords', None) is not None or getattr(self, 'roi2_stack', None) is not None, redraw=False)
    self.update_processing_pixel_size()
    self._proc_ax.set_anchor('C')
    self._proc_canvas.draw_idle()

def _proc_show_fft(self):
    if self.current_processed is None:
        self.update_processing_view()
    if self.current_processed is None:
        return
    F, mag = _proc_fft(self, self.current_processed)
    fig, (a1, a2) = plt.subplots(1, 2, igsize=(11, 5), dpi=100)
    a1.imshow(self.current_processed, cmap='gray', origin='lower', vmin=0, vmax=1)
    a1.axis('image')
    a1.set_title('ROI')
    a2.imshow(mag, cmap='magma', origin='lower', vmin=0, vmax=1)
    a2.axis('image')
    a2.set_title('FFT log amplitude')
    fig.tight_layout()
    self._show_proc_toplevel(fig, 'ROI FFT', 1100, 650)

def _proc_blob(self):
    if self.current_processed is None:
        self.update_processing_view()
    if self.current_processed is None:
        return
    img = self.current_processed
    vals = img[self.local_mask]
    threshold = filters.threshold_otsu(vals) if np.any(vals > 0) else 0.5
    binary = (img > threshold) & self.local_mask
    lab = measure.label(binary)
    props = measure.regionprops(lab)
    self.last_blobs = props
    fig, ax = plt.subplots(figsize=(8, 7), dpi=100)
    ax.imshow(img, cmap='gray', origin='lower', vmin=0, vmax=1)
    for p in props:
        if p.area < 5:
            continue
        y, x = p.centroid
        ax.plot(x, y, 'r+')
        ax.text(x, y, str(p.label), color='yellow', fontsize=10)
    ax.set_title(f'Blob analysis | {len(props)} regions | threshold={threshold:.3f}')
    ax.axis('image')
    fig.tight_layout()
    self._show_proc_toplevel(fig, 'ROI Blob Analysis', 900, 700)
    self.log(f'Blob analysis: {len(props)} connected regions found.')

def _proc_show_toplevel(self, fig, title, width, height):
    try:
        plt.close(fig)
    except Exception:
        pass
    try:
        self.log(f'{title}: separate popup disabled; use the main workflow window.')
    except Exception:
        pass
    return None

def _proc_reset(self):
    self.proc_idx_var.set(max(0, min(self.current_index, len(self.recons) - 1)) if self.recons is not None else 0)
    self.brightness_var.set(0.0)
    self.contrast_var.set(1.0)
    self.filter_var.set('Gaussian')
    self.strength_var.set(3.0)
    self.colormap_var.set('RdBu_r')
    self.view_var.set('IFFT')
    self.fft_mask_var.set(0.60)
    try:
        self.clear_processing_profile_measure()
    except Exception:
        pass
    self.update_processing_view()

def _proc_choose_folder(self):
    folder = filedialog.askdirectory(parent=self.root, title='Select ROI export folder')
    if folder:
        self.output_var.set(folder)
        self.output_dir = folder

def _proc_rgb(self, image):
    return (plt.get_cmap(self.colormap_var.get())(np.clip(image, 0, 1))[..., :3] * 255).astype(np.uint8)

def _proc_export_rgb(self, image):
    rgb = _proc_rgb(self, image)
    return np.flipud(rgb).copy()

def _proc_save_single(self, extension, fmt):
    """Export one currently selected Tab-6 ROI frame safely."""
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Resolution Required', 'Apply SAME CONFIRMED ROI resolution first.', parent=self.root)
        return
    try:
        dpi = int(self.proc_export_dpi_var.get())
    except Exception:
        dpi = 300
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Export DPI', 'Select 150, 300, 600 or 900 DPI.', parent=self.root)
        return
    try:
        folder = os.path.abspath(os.path.expanduser(str(self.output_var.get()).strip() or 'ROI_exports'))
        os.makedirs(folder, exist_ok=True)
        i = max(0, min(int(round(float(self.proc_idx_var.get()))), len(self.recons) - 1))
        field = float(self.real_fields_mT[i])
        path = os.path.join(folder, f'ROI_{i:03d}_{field:+.2f}mT{extension}')
        rgb = _proc_export_rgb(self, _proc_image(self, i))
        img = Image.fromarray(rgb, mode='RGB')
        save_kwargs = {'format': fmt, 'dpi': (dpi, dpi)}
        if fmt == 'JPEG':
            save_kwargs['subsampling'] = 0
        if fmt == 'TIFF':
            save_kwargs['compression'] = 'tiff_lzw'
        img.save(path, **save_kwargs)
        self.proc_status_var.set(f'Saved: {path} | {dpi} DPI')
        self.log(f'Tab 6 export complete: {path} | format={fmt} | dpi={dpi}')
    except Exception as exc:
        self.proc_status_var.set('Single-image export failed.')
        self.log(f'Tab 6 single export failed: {exc}')
        messagebox.showerror('Export Error', str(exc), parent=self.root)

def _proc_batch(self, extension, fmt):
    """Export all Tab-6 ROI frames with consistent orientation and DPI."""
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Resolution Required', 'Apply SAME CONFIRMED ROI resolution first.', parent=self.root)
        return
    try:
        dpi = int(self.proc_export_dpi_var.get())
    except Exception:
        dpi = 300
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Export DPI', 'Select 150, 300, 600 or 900 DPI.', parent=self.root)
        return
    try:
        folder = os.path.abspath(os.path.expanduser(str(self.output_var.get()).strip() or 'ROI_exports'))
        os.makedirs(folder, exist_ok=True)
        total = len(self.recons)
        for i in range(total):
            field = float(self.real_fields_mT[i])
            path = os.path.join(folder, f'ROI_{i:03d}_{field:+.2f}mT{extension}')
            rgb = _proc_export_rgb(self, _proc_image(self, i))
            img = Image.fromarray(rgb, mode='RGB')
            save_kwargs = {'format': fmt, 'dpi': (dpi, dpi)}
            if fmt == 'JPEG':
                save_kwargs['subsampling'] = 0
            if fmt == 'TIFF':
                save_kwargs['compression'] = 'tiff_lzw'
            img.save(path, **save_kwargs)
            self.proc_status_var.set(f'Exporting {fmt}: {i + 1}/{total} | {dpi} DPI')
            self.root.update_idletasks()
        self.proc_status_var.set(f'Batch {fmt} export complete | {total} images | {dpi} DPI')
        self.log(f'Tab 6 batch export complete: format={fmt} | count={total} | dpi={dpi} | folder={folder}')
    except Exception as exc:
        self.proc_status_var.set('Batch export failed.')
        self.log(f'Tab 6 batch export failed: {exc}')
        messagebox.showerror('Batch Export Error', str(exc), parent=self.root)

def _proc_imageio(self):
    try:
        import imageio.v2 as imageio
        return imageio
    except ImportError:
        messagebox.showerror('Missing package', 'imageio is required.\n\nInstall:\npip install imageio imageio-ffmpeg', parent=self.root)
        return None

def _proc_mp4(self, indices, filename):
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Resolution Required', 'Apply SAME CONFIRMED ROI first.', parent=self.root)
        return
    imageio = _proc_imageio(self)
    if imageio is None:
        return
    try:
        dpi = int(self.proc_export_dpi_var.get())
    except Exception:
        dpi = 300
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Export DPI', 'Select 150, 300, 600 or 900 DPI.', parent=self.root)
        return
    folder = self.output_var.get()
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, filename)
    indices = list(indices)
    if not indices:
        messagebox.showwarning('MP4 Export', 'No frames selected.', parent=self.root)
        return
    fps = max(1, int(self.fps_var.get()))
    fig = plt.figure(figsize=(8, 6), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()
    writer = None
    try:
        codec_attempts = ['libx264', 'mpeg4', None]
        errors = []
        for codec in codec_attempts:
            try:
                kwargs = {'fps': fps, 'format': 'ffmpeg', 'macro_block_size': None}
                if codec is not None:
                    kwargs['codec'] = codec
                writer = imageio.get_writer(path, **kwargs)
                break
            except Exception as exc:
                errors.append(f"{codec or 'default'}: {exc}")
        if writer is None:
            raise RuntimeError('Could not initialize an MP4 encoder.\n\n' + '\n'.join(errors))
        total = len(indices)
        for count, i in enumerate(indices, start=1):
            image = _proc_image(self, i)
            rgb = _proc_export_rgb(self, image)
            ax.clear()
            ax.set_axis_off()
            ax.imshow(rgb, origin='upper', interpolation='nearest')
            fig.canvas.draw()
            frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            h, w = frame.shape[:2]
            if h % 2 or w % 2:
                frame = np.pad(frame, ((0, h % 2), (0, w % 2), (0, 0)), mode='edge')
            writer.append_data(frame)
            self.proc_status_var.set(f'MP4: {count}/{total} | {fps} FPS | {dpi} DPI')
            self.root.update_idletasks()
    except Exception as exc:
        self.proc_status_var.set('MP4 export failed.')
        self.log(f'MP4 export failed: {path} | {exc}')
        messagebox.showerror('MP4 Error', str(exc), parent=self.root)
        return
    finally:
        try:
            if writer is not None:
                writer.close()
        except Exception:
            pass
        try:
            plt.close(fig)
        except Exception:
            pass
    self.proc_status_var.set(f'MP4 saved: {filename} | {fps} FPS | {dpi} DPI')
    self.log(f'MP4 saved: {path} | frames={len(indices)} | fps={fps} | dpi={dpi}')
    messagebox.showinfo('MP4 Export', f'MP4 successfully saved:\n\n{path}\n\nFrames: {len(indices)}\nFPS: {fps}\nRaster DPI: {dpi}', parent=self.root)

def _proc_single_mp4(self):
    i = int(self.proc_idx_var.get())
    frames = max(1, int(self.fps_var.get()) * 2)
    _proc_mp4(self, [i] * frames, f'ROI_single_{i:03d}_{self.real_fields_mT[i]:+.2f}mT.mp4')

def _proc_get_batch_mp4_indices(self):
    """Return exactly the user-selected 1-based START..END frames for batch MP4."""
    n = int(len(self.recons)) if self.recons is not None else 0
    if n <= 0:
        raise RuntimeError('No reconstruction frames are loaded.')
    try:
        start = int(self.proc_mp4_start_var.get())
        end = int(self.proc_mp4_end_var.get())
        fps = int(self.fps_var.get())
    except Exception:
        raise ValueError('START frame, END frame and FPS must be valid numbers.')
    if not (1 <= start <= end <= n):
        raise ValueError(f'MP4 frame range must satisfy 1 <= START <= END <= {n}.')
    if fps < 1:
        raise ValueError('FPS must be at least 1.')
    return list(range(start - 1, end)), start, end, fps


def _proc_batch_mp4(self):
    try:
        indices, start, end, fps = _proc_get_batch_mp4_indices(self)
        self.fps_var.set(fps)
        _proc_mp4(self, indices, f'ROI_frames_{start:04d}-{end:04d}_{fps}fps.mp4')
    except Exception as exc:
        messagebox.showerror('Batch MP4 Export', str(exc), parent=self.root)

def _proc_make_toplevel(self, fig, title, width, height):
    return _proc_show_toplevel(self, fig, title, width, height)

def _line_contrast_center(self, image):
    """Determine the center from the strongest signed image contrast."""
    image = np.asarray(image, dtype=float)
    finite = np.isfinite(image)
    if not np.any(finite):
        raise ValueError('No finite pixels in Secondary ROI.')
    z = image.copy()
    z[~finite] = np.nanmedian(z[finite])
    h, w = z.shape
    b = max(1, int(round(0.1 * min(h, w))))
    border = np.concatenate([z[:b, :].ravel(), z[-b:, :].ravel(), z[:, :b].ravel(), z[:, -b:].ravel()])
    background = float(np.nanmedian(border))
    contrast = z - background
    max_pos = float(np.nanmax(contrast))
    max_neg = float(np.nanmin(contrast))
    if abs(max_pos) >= abs(max_neg):
        signed = contrast
        sign = 1.0
        peak = max_pos
    else:
        signed = -contrast
        sign = -1.0
        peak = -max_neg
    peak = max(peak, 1e-12)
    frac = float(np.clip(self.line_contrast_fraction_var.get(), 0.0, 0.95))
    threshold = frac * peak
    weights = np.clip(signed - threshold, 0.0, None)
    if np.sum(weights) <= 0:
        weights = np.clip(signed, 0.0, None)
    yy, xx = np.indices(z.shape, dtype=float)
    ws = float(np.sum(weights))
    if ws <= 0:
        y0, x0 = np.unravel_index(np.argmax(np.abs(contrast)), z.shape)
        x0, y0 = (float(x0), float(y0))
    else:
        x0 = float(np.sum(xx * weights) / ws)
        y0 = float(np.sum(yy * weights) / ws)
    return {'x0': x0, 'y0': y0, 'background': background, 'sign': sign, 'peak': peak, 'threshold': threshold}

def _get_line_reference_angles(self):
    return {'1': [0.0], '2': [0.0, 90.0], '4': [0.0, 45.0, 90.0, 135.0], '6': [0.0, 30.0, 60.0, 90.0, 120.0, 150.0]}.get(str(self.line_ref_mode_var.get()), [0.0, 90.0])

def _extract_multi_line_profiles(self, image, x0, y0, angles_deg=None, nsamples=800):
    image = np.asarray(image, dtype=float)
    h, w = image.shape
    angles_deg = self._get_line_reference_angles() if angles_deg is None else list(angles_deg)
    rmax = 2.0 * max(1.0, min(float(x0), float(w - 1 - x0), float(y0), float(h - 1 - y0)))
    t = np.linspace(-rmax, rmax, max(200, int(nsamples)))
    profiles = {}
    for a in angles_deg:
        th = np.radians(float(a))
        profiles[float(a)] = map_coordinates(image, [y0 + t * np.sin(th), x0 + t * np.cos(th)], order=1, mode='nearest')
    avg = np.nanmean(np.vstack([profiles[float(a)] for a in angles_deg]), axis=0)
    return (t, profiles, avg)

def _extract_line_profiles(self, image, x0, y0, samples=800):
    t, p, avg = self._extract_multi_line_profiles(image, x0, y0, [0.0, 90.0], samples)
    return (t, p[0.0], p[90.0], avg, p[0.0] - p[90.0])

def _fit_fixed_center_1d_gaussian(self, t, avg):
    """Fit B + A exp[-t²/(2 sigma²)] with center fixed at t=0."""
    t = np.asarray(t, dtype=float)
    y = np.asarray(avg, dtype=float)
    good = np.isfinite(t) & np.isfinite(y)
    if np.count_nonzero(good) < 20:
        raise ValueError('Too few valid line-profile points.')
    tf = t[good]
    yf = y[good]
    nedge = max(5, int(0.1 * len(yf)))
    b0 = float(np.median(np.r_[yf[:nedge], yf[-nedge:]]))
    cidx = int(np.argmin(np.abs(tf)))
    a0 = float(yf[cidx] - b0)
    signed = np.sign(a0 if abs(a0) > 1e-12 else 1.0) * (yf - b0)
    wt = np.clip(signed, 0.0, None)
    if np.sum(wt) > 0:
        sigma0 = float(np.sqrt(np.sum(wt * tf ** 2) / np.sum(wt)))
    else:
        sigma0 = max(2.0, np.ptp(tf) / 6.0)
    sigma0 = max(sigma0, 0.5)
    span = max(float(np.nanmax(yf) - np.nanmin(yf)), 1e-12)

    def model(x, background, amplitude, sigma):
        return background + amplitude * np.exp(-0.5 * (x / np.maximum(sigma, 1e-12)) ** 2)
    p0 = np.clip([b0, a0, sigma0], [float(np.nanmin(yf) - span), -10.0 * span, 0.25], [float(np.nanmax(yf) + span), 10.0 * span, max(2.0, np.ptp(tf))])
    popt, pcov = curve_fit(model, tf, yf, p0=p0, bounds=([float(np.nanmin(yf) - span), -10.0 * span, 0.25], [float(np.nanmax(yf) + span), 10.0 * span, max(2.0, np.ptp(tf))]), maxfev=20000)
    fit = model(t, *popt)
    residual = y - fit
    ss_res = float(np.sum(residual[good] ** 2))
    ss_tot = float(np.sum((yf - np.mean(yf)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse = float(np.sqrt(np.mean(residual[good] ** 2)))
    sigma = abs(float(popt[2]))
    fwhm_factor = 2.0 * np.sqrt(2.0 * np.log(2.0))
    fwhm = fwhm_factor * sigma
    try:
        perr = np.sqrt(np.diag(pcov))
        amp_err = float(perr[1]) if np.isfinite(perr[1]) else np.nan
        sigma_err = float(perr[2]) if np.isfinite(perr[2]) else np.nan
        fwhm_err = fwhm_factor * sigma_err if np.isfinite(sigma_err) else np.nan
    except Exception:
        amp_err = np.nan
        sigma_err = np.nan
        fwhm_err = np.nan
    return {'background': float(popt[0]), 'amplitude': float(popt[1]), 'amplitude_err': amp_err, 'sigma': sigma, 'sigma_err': sigma_err, 'fwhm': float(fwhm), 'fwhm_err': float(fwhm_err), 'r2': r2, 'rmse': rmse, 'fit_profile': fit, 'residual_profile': residual, 'covariance': pcov}

def load_line_from_tab6(self):
    if getattr(self, 'roi2_stack', None) is None:
        messagebox.showwarning('Line Profile', 'Confirm the Secondary ROI in 5. ROI Processing and verify it in 6. Secondary ROI.', parent=self.root)
        return False
    stack = np.asarray(self.roi2_stack, dtype=np.float64)
    if stack.ndim != 3 or stack.shape[0] == 0:
        messagebox.showerror('Line Profile', f'Invalid Secondary ROI stack shape: {stack.shape}', parent=self.root)
        return False
    self.line_source_stack = stack.copy()
    self.line_source_coords = dict(self.roi2_confirmed_coords) if getattr(self, 'roi2_confirmed_coords', None) is not None else None
    n, h, w = stack.shape
    self.line_slider.configure(from_=0, to=max(0, n - 1))
    self.line_range_start_spin.configure(from_=0, to=max(0, n - 1))
    self.line_range_end_spin.configure(from_=0, to=max(0, n - 1))
    self.line_range_start_var.set(0)
    self.line_range_end_var.set(max(0, n - 1))
    self.line_range_status_var.set(f'Range ready: frames 0–{max(0, n - 1)}')
    current = int(self.roi2_idx_var.get()) if getattr(self, 'roi2_idx_var', None) is not None else 0
    self.line_idx_var.set(max(0, min(current, n - 1)))
    if self.line_source_coords is not None:
        c = self.line_source_coords
        self.line_source_status_var.set(f"✓ Source = 6. Secondary ROI | confirmed X={int(c['xmin'])}:{int(c['xmax'])}, Y={int(c['ymin'])}:{int(c['ymax'])} | crop={w} × {h} px | frames={n}")
    else:
        self.line_source_status_var.set(f'✓ Source = 6. Secondary ROI | crop={w} × {h} px | frames={n}')
    self._line_frame_changed()
    return True

def line_recalculate_center(self):
    if self.line_source_stack is None:
        self.line_status_var.set('Load confirmed Secondary ROI from tab 6 first.')
        return
    idx = max(0, min(int(self.line_idx_var.get()), len(self.line_source_stack) - 1))
    image = self.line_source_stack[idx]
    c = self._line_contrast_center(image)
    self.line_center_x_var.set(float(c['x0']))
    self.line_center_y_var.set(float(c['y0']))
    self.line_center_fixed = True
    self.line_center_label_var.set(f"FIXED center = ({c['x0']:.3f}, {c['y0']:.3f}) px | background={c['background']:.5g} | threshold={c['threshold']:.5g}")
    self._line_refresh_display(image, c['x0'], c['y0'])

def _line_frame_changed(self):
    if self.line_fit_resid_ax is not None:
        try:
            self.line_fit_resid_ax.remove()
        except Exception:
            pass
        self.line_fit_resid_ax = None
    if self.line_source_stack is None:
        return
    n = len(self.line_source_stack)
    idx = max(0, min(int(self.line_idx_var.get()), n - 1))
    self.line_idx_var.set(idx)
    image = np.asarray(self.line_source_stack[idx], dtype=float)
    c = self._line_contrast_center(image)
    self.line_center_x_var.set(float(c['x0']))
    self.line_center_y_var.set(float(c['y0']))
    self.line_center_fixed = True
    self.line_center_label_var.set(f"FIXED center = ({c['x0']:.3f}, {c['y0']:.3f}) px | threshold={c['threshold']:.5g}")
    self._line_refresh_display(image, c['x0'], c['y0'])
    field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    self.line_field_var.set(f'Frame {idx + 1}/{n} | Field = {field:+.2f} mT')
    self.line_fit_status_var.set('No average-profile fit for current frame.')
    for var in self.line_param_vars.values():
        var.set('--')

def line_reference_mode_changed(self):
    desc = {'1': '1 line: 0°', '2': '2 lines: 0° / 90°', '4': '4 lines: 0° / 45° / 90° / 135°', '6': '6 lines: 0° / 30° / 60° / 90° / 120° / 150°'}
    self.line_ref_description_var.set(desc.get(str(self.line_ref_mode_var.get()), '2 lines: 0° / 90°'))
    self.line_ref_angles = self._get_line_reference_angles()
    if self.line_source_stack is not None:
        self._line_frame_changed()
    self.line_status_var.set('Reference-line configuration updated. Run FIT AVERAGE FULL PROFILE.')

def _line_refresh_display(self, image, x0, y0):
    cmap = self.colormap_var.get()
    self.line_image.set_cmap(cmap)
    self.line_image.set_data(image)
    extent = (-0.5, image.shape[1] - 0.5, -0.5, image.shape[0] - 0.5)
    self.line_image.set_extent(extent)
    finite = np.isfinite(image)
    if np.any(finite):
        vmin = float(np.nanmin(image[finite]))
        vmax = float(np.nanmax(image[finite]))
        if vmax <= vmin:
            vmax = vmin + 1e-12
        self.line_image.set_clim(vmin, vmax)
    self.line_ax_image.set_xlim(-0.5, image.shape[1] - 0.5)
    self.line_ax_image.set_ylim(-0.5, image.shape[0] - 0.5)
    self.line_ax_image.set_aspect('equal', adjustable='box')
    self.line_ax_image.clear()
    self.line_image = self.line_ax_image.imshow(image, cmap=cmap, origin='lower', interpolation='nearest', extent=extent)
    finite = np.isfinite(image)
    if np.any(finite):
        vmin = float(np.nanmin(image[finite]))
        vmax = float(np.nanmax(image[finite]))
        if vmax <= vmin:
            vmax = vmin + 1e-12
        self.line_image.set_clim(vmin, vmax)
    if hasattr(self, 'line_roi_cbar'):
        self.line_roi_cbar.update_normal(self.line_image)
    self.line_ax_image.plot([x0], [y0], marker='+', markersize=15, markeredgewidth=2.4, linestyle='None', color='yellow', zorder=50)
    ref_angles = self._get_line_reference_angles()
    line_colors = ['yellow', 'cyan', 'lime', 'magenta', 'orange', 'white']
    rmax = 2.0 * max(1.0, min(float(x0), float(image.shape[1] - 1 - x0), float(y0), float(image.shape[0] - 1 - y0)))
    for j, a in enumerate(ref_angles):
        th = np.radians(float(a))
        dx = rmax * np.cos(th)
        dy = rmax * np.sin(th)
        self.line_ax_image.plot([x0 - dx, x0 + dx], [y0 - dy, y0 + dy], linestyle='--', linewidth=1.6, color=line_colors[j % len(line_colors)], zorder=49 - j)
    angle_text = ' / '.join((f'{(int(a) if float(a).is_integer() else a):g}°' for a in ref_angles))
    self.line_ax_image.text(0.02, 0.98, f'FIXED CONTRAST CENTER ({x0:.2f}, {y0:.2f}) px\nReference lines: {angle_text}', transform=self.line_ax_image.transAxes, ha='left', va='top', fontsize=9, color='white', bbox=dict(facecolor='black', alpha=0.6, edgecolor='white', pad=3), zorder=60)
    self.line_ax_image.set_title('Tab-6 Secondary ROI — fixed contrast center')
    self.line_ax_image.set_xlabel('X pixel')
    self.line_ax_image.set_ylabel('Y pixel')
    self.line_ax_image.set_aspect('equal', adjustable='box')
    self.line_ax_profiles.clear()
    self.line_ax_profiles.set_title('0° + 90° full line profiles + average')
    self.line_ax_profiles.set_xlabel('Position from fixed center (px)')
    self.line_ax_profiles.set_ylabel('Intensity')
    self.line_ax_profiles.grid(alpha=0.2)
    self.line_ax_fit.clear()
    self.line_ax_fit.set_title('Average Full Profile - 1D Gaussian Fit')
    self.line_ax_fit.set_xlabel('Position From Fixed Center (px)')
    self.line_ax_fit.set_ylabel('Intensity')
    self.line_ax_fit.grid(alpha=0.2)
    self.line_canvas.draw_idle()

def line_set_all_range(self):
    if self.line_source_stack is None:
        self.line_status_var.set('Load confirmed Secondary ROI from tab 6 first.')
        return
    n = len(self.line_source_stack)
    self.line_range_start_var.set(0)
    self.line_range_end_var.set(max(0, n - 1))
    self.line_range_status_var.set(f'Range = frames 0–{max(0, n - 1)} ({n} fields)')

def track_line_fwhm_range(self):
    if self.line_source_stack is None:
        if not self.load_line_from_tab6():
            return None
    n = len(self.line_source_stack)
    if n <= 0:
        self.line_range_status_var.set('No frames available for FWHM tracking.')
        return None
    try:
        a = int(self.line_range_start_var.get())
        b = int(self.line_range_end_var.get())
    except Exception as exc:
        self.line_range_status_var.set(f'Invalid frame range: {exc}')
        return None
    a = max(0, min(a, n - 1))
    b = max(0, min(b, n - 1))
    if a > b:
        a, b = (b, a)
    ref_angles = self._get_line_reference_angles()
    rows = []
    total = b - a + 1
    for count, idx in enumerate(range(a, b + 1), 1):
        image = np.asarray(self.line_source_stack[idx], dtype=float)
        field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
        try:
            c = self._line_contrast_center(image)
            x0 = float(c['x0'])
            y0 = float(c['y0'])
            t, profiles, avg = self._extract_multi_line_profiles(image, x0, y0, ref_angles)
            fit = self._fit_fixed_center_1d_gaussian(t, avg)
            stack = np.vstack([profiles[float(angle)] for angle in ref_angles])
            per = np.sqrt(np.nanmean((stack - avg[None, :]) ** 2, axis=1))
            line_rmse = float(np.nanmean(per))
            line_cv = line_rmse / max(float(np.nanmax(np.abs(avg))), 1e-12)
            rows.append({'frame': idx, 'field_mT': field, 'reference_line_count': len(ref_angles), 'reference_angles_deg': ','.join((f'{x:g}' for x in ref_angles)), 'center_x_px': x0, 'center_y_px': y0, 'sigma_px': float(fit['sigma']), 'sigma_err_px': float(fit['sigma_err']), 'amplitude': float(fit['amplitude']), 'amplitude_err': float(fit['amplitude_err']), 'avg_fwhm_px': float(fit['fwhm']), 'fwhm_err_px': float(fit['fwhm_err']), 'r2': float(fit['r2']), 'rmse': float(fit['rmse']), 'line_to_average_rmse': line_rmse, 'line_to_average_cv': line_cv, 'success': True, 'error': ''})
        except Exception as exc:
            rows.append({'frame': idx, 'field_mT': field, 'reference_line_count': len(ref_angles), 'reference_angles_deg': ','.join((f'{x:g}' for x in ref_angles)), 'center_x_px': np.nan, 'center_y_px': np.nan, 'sigma_px': np.nan, 'sigma_err_px': np.nan, 'amplitude': np.nan, 'amplitude_err': np.nan, 'avg_fwhm_px': np.nan, 'fwhm_err_px': np.nan, 'r2': np.nan, 'rmse': np.nan, 'line_to_average_rmse': np.nan, 'line_to_average_cv': np.nan, 'success': False, 'error': f'{type(exc).__name__}: {exc}'})
        self.line_range_status_var.set(f'Tracking {len(ref_angles)}-line Avg. FWHM: {count}/{total} fields...')
        self.root.update_idletasks()
    df = pd.DataFrame(rows)
    good = df['success'].astype(bool).to_numpy() & np.isfinite(df['avg_fwhm_px'].to_numpy(float))
    self.line_fwhm_results = {'table': df, 'reference_line_count': len(ref_angles), 'reference_angles_deg': ref_angles}
    self.line_ax_fwhm.clear()
    if getattr(self, 'line_fwhm_frame_ax', None) is not None:
        try:
            self.line_fwhm_frame_ax.remove()
        except Exception:
            pass
        self.line_fwhm_frame_ax = None
    if np.any(good):
        x_frame = df.loc[good, 'frame'].to_numpy(float)
        x_field = df.loc[good, 'field_mT'].to_numpy(float)
        y_fwhm = df.loc[good, 'avg_fwhm_px'].to_numpy(float)
        e_fwhm = df.loc[good, 'fwhm_err_px'].to_numpy(float)
        y_amp = df.loc[good, 'amplitude'].to_numpy(float)
        e_amp = df.loc[good, 'amplitude_err'].to_numpy(float)
        if getattr(self, 'line_ax_amp', None) is not None:
            try:
                self.line_ax_amp.remove()
            except Exception:
                pass
            self.line_ax_amp = None
        fwhm_container = self.line_ax_fwhm.errorbar(x_frame, y_fwhm, yerr=e_fwhm, fmt='o-', markersize=4, linewidth=1.6, elinewidth=1.0, capsize=3, label=f'Avg. FWHM ({len(ref_angles)} lines)')
        self.line_ax_amp = self.line_ax_fwhm.twinx()
        amp_container = self.line_ax_amp.errorbar(x_frame, y_amp, yerr=e_amp, fmt='s--', markersize=4, linewidth=1.4, elinewidth=1.0, capsize=3, label='Amplitude')
        self.line_ax_amp.set_ylabel('Gaussian Amplitude')
        from matplotlib.ticker import FixedLocator, FixedFormatter
        self.line_ax_fwhm.xaxis.set_major_locator(FixedLocator(x_frame))
        self.line_ax_fwhm.xaxis.set_major_formatter(FixedFormatter([str(int(round(frame))) for frame in x_frame]))
        self.line_ax_fwhm.set_xlabel('Frame number')
        self.line_fwhm_frame_ax = self.line_ax_fwhm.twiny()
        xmin = float(np.nanmin(x_frame))
        xmax = float(np.nanmax(x_frame))
        span = xmax - xmin
        margin = max(0.5, 0.05 * span) if span > 0 else 0.5
        self.line_ax_fwhm.set_xlim(xmin - margin, xmax + margin)
        self.line_fwhm_frame_ax.set_xlim(xmin - margin, xmax + margin)
        self.line_fwhm_frame_ax.xaxis.set_major_locator(FixedLocator(x_frame))
        field_labels = [f'{int(round(field)):+d}' for field in x_field]
        self.line_fwhm_frame_ax.xaxis.set_major_formatter(FixedFormatter(field_labels))
        self.line_fwhm_frame_ax.tick_params(axis='x', which='major', labelrotation=90, pad=4)
        for label in self.line_fwhm_frame_ax.get_xticklabels():
            label.set_rotation(90)
            label.set_horizontalalignment('center')
            label.set_verticalalignment('bottom')
        self.line_fwhm_frame_ax.set_xlabel('Equivalent field (mT)', labelpad=14)
        self.line_fwhm_frame_ax.xaxis.set_minor_locator(FixedLocator([]))
    self.line_ax_fwhm.set_title(f'Avg. FWHM vs Frame — {len(ref_angles)} reference lines')
    self.line_ax_fwhm.set_xlabel('Frame number')
    self.line_ax_fwhm.set_ylabel('Avg. FWHM (px)')
    if self.line_ax_amp is not None:
        self.line_ax_amp.set_ylabel('Gaussian Amplitude')
    self.line_ax_fwhm.grid(alpha=0.2)
    if np.any(good):
        handles1, labels1 = self.line_ax_fwhm.get_legend_handles_labels()
        handles2, labels2 = self.line_ax_amp.get_legend_handles_labels() if self.line_ax_amp is not None else ([], [])
        self.line_ax_fwhm.legend(handles1 + handles2, labels1 + labels2, loc='best', fontsize=8)
    self.line_range_status_var.set(f'✓ {len(ref_angles)}-line Avg. FWHM tracking complete | frames {a}–{b} | {int(np.count_nonzero(good))}/{total} successful')
    self.line_status_var.set('Reference lines updated; center contrast-fixed per field.')
    self.line_canvas.draw_idle()
    return self.line_fwhm_results

def export_line_fwhm_range_csv(self):
    if self.line_fwhm_results is None:
        messagebox.showwarning('Avg. FWHM Tracking', 'Run TRACK AVG. FWHM vs FRAME first.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Save Avg. FWHM vs Frame', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    self.line_fwhm_results['table'].to_csv(fp, index=False)
    self.line_range_status_var.set(f'Saved Avg. FWHM range results: {fp}')

def fit_line_average_current(self):
    if self.line_source_stack is None:
        if not self.load_line_from_tab6():
            return None
    idx = max(0, min(int(self.line_idx_var.get()), len(self.line_source_stack) - 1))
    image = np.asarray(self.line_source_stack[idx], dtype=float)
    c = self._line_contrast_center(image)
    x0, y0 = (float(c['x0']), float(c['y0']))
    self.line_center_x_var.set(x0)
    self.line_center_y_var.set(y0)
    self.line_center_fixed = True
    ref_angles = self._get_line_reference_angles()
    self.line_ref_angles = ref_angles
    angle_text = ' / '.join((f'{(int(a) if float(a).is_integer() else a):g}°' for a in ref_angles))
    t, profiles, avg = self._extract_multi_line_profiles(image, x0, y0, ref_angles)
    result = self._fit_fixed_center_1d_gaussian(t, avg)
    stack = np.vstack([profiles[float(a)] for a in ref_angles])
    per = np.sqrt(np.nanmean((stack - avg[None, :]) ** 2, axis=1))
    line_rmse = float(np.nanmean(per))
    line_cv = line_rmse / max(float(np.nanmax(np.abs(avg))), 1e-12)
    field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    result.update({'frame': idx, 'field_mT': field, 'center_x': x0, 'center_y': y0, 'reference_angles_deg': ref_angles, 'reference_line_rmse': per, 'reference_line_average_rmse': line_rmse, 'reference_line_variation': line_cv})
    self.line_ref_profiles = profiles
    self.line_ref_average = avg
    self.line_fit_result = result
    self.line_profile_avg = (t, avg)
    self.line_profile_0 = (t, profiles[0.0]) if 0.0 in profiles else None
    self.line_profile_90 = (t, profiles[90.0]) if 90.0 in profiles else None
    self.line_ax_profiles.clear()
    colors = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5']
    for j, a in enumerate(ref_angles):
        self.line_ax_profiles.plot(t, profiles[float(a)], linewidth=1.0, color=colors[j % 6], label=f'{(int(a) if float(a).is_integer() else a):g}°')
    self.line_ax_profiles.plot(t, avg, linewidth=2.7, color='black', label='Average')
    self.line_ax_profiles.axvline(0.0, linestyle='--', linewidth=1.0)
    self.line_ax_profiles.set_title(f'Selected full line profiles + average ({len(ref_angles)} lines)')
    self.line_ax_profiles.set_xlabel('Position from fixed center (px)')
    self.line_ax_profiles.set_ylabel('Intensity')
    self.line_ax_profiles.grid(alpha=0.2)
    self.line_ax_profiles.legend(loc='best', fontsize=8)
    self.line_ax_fit.clear()
    if getattr(self, 'line_fit_resid_ax', None) is not None:
        try:
            self.line_fit_resid_ax.remove()
        except Exception:
            pass
        self.line_fit_resid_ax = None
    self.line_ax_fit.plot(t, avg, linewidth=1.8, label='Average full profile')
    self.line_ax_fit.plot(t, result['fit_profile'], color='blue',linewidth=2.6, label='Fixed-center Gaussian fit')
    self.line_ax_fit.axvline(0.0, linestyle=':', linewidth=1.0)
    self.line_ax_fit.set_title(f'Average of {len(ref_angles)} lines → Gaussian fit')
    self.line_ax_fit.set_xlabel('Position from fixed center (px)')
    self.line_ax_fit.set_ylabel('Intensity')
    self.line_ax_fit.grid(alpha=0.2)
    self.line_fit_resid_ax = self.line_ax_fit.twinx()
    self.line_fit_resid_ax.plot(t, result['residual_profile'], linestyle='--', linewidth=1.3, label='Residual')
    self.line_fit_resid_ax.axhline(0.0, linestyle=':', linewidth=0.9)
    self.line_fit_resid_ax.set_ylabel('Residual')
    h1, l1 = self.line_ax_fit.get_legend_handles_labels()
    h2, l2 = self.line_fit_resid_ax.get_legend_handles_labels()
    self.line_ax_fit.legend(h1 + h2, l1 + l2, loc='best', fontsize=8)
    for key, var in self.line_param_vars.items():
        value = line_rmse if key == 'orthogonal_rmse' else line_cv if key == 'orthogonal_cv' else result.get(key, np.nan)
        var.set(f'{float(value):.6g}' if np.isfinite(value) else '--')
    self.line_fit_status_var.set(f"✓ {len(ref_angles)}-line average fit | FWHM={result['fwhm']:.4g} px | R²={result['r2']:.5f} | Line-to-average RMSE={line_rmse:.5g}")
    self.line_status_var.set(f'Center fixed from contrast. Processed: {angle_text}.')
    self.line_canvas.draw_idle()
    return result

def export_line_profile_csv(self):
    if self.line_fit_result is None or self.line_profile_avg is None:
        messagebox.showwarning('Line Profile', 'Fit the average full profile first.', parent=self.root)
        return
    t, avg = self.line_profile_avg
    data = {'position_from_center_px': t}
    for a in self._get_line_reference_angles():
        data[f'profile_{(int(a) if float(a).is_integer() else a):g}deg'] = self.line_ref_profiles[float(a)]
    data['profile_average'] = avg
    data['gaussian_fit'] = self.line_fit_result['fit_profile']
    data['residual'] = self.line_fit_result['residual_profile']
    fp = filedialog.asksaveasfilename(parent=self.root, initialdir=self._export_initialdir(), title='Save current multi-line profile fit', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    pd.DataFrame(data).to_csv(fp, index=False)
    self.line_status_var.set(f'Saved current {len(self._get_line_reference_angles())}-line profile fit: {fp}')

def update_multi_roi_view(self):
    """Render one frame without resetting an existing zoom."""
    if not hasattr(self, 'multi_roi_ax') or self.multi_roi_ax is None:
        return
    if self.recons is None:
        return
    zoom_limits = getattr(self, '_multi_roi_zoom_limits', None)
    zoom_active = zoom_limits is not None
    try:
        image = self._multi_roi_get_image(int(self.multi_roi_frame_var.get()))
    except Exception as exc:
        self.multi_roi_status_var.set(str(exc))
        return
    self.multi_roi_image.set_cmap(self.colormap_var.get())
    self.multi_roi_image.set_data(image)
    finite = np.isfinite(image)
    if np.any(finite):
        vmin = float(np.nanmin(image[finite]))
        vmax = float(np.nanmax(image[finite]))
        if vmax <= vmin:
            vmax = vmin + 1e-12
        self.multi_roi_image.set_clim(vmin, vmax)
    new_shape = tuple(image.shape[:2])
    old_shape = getattr(self, '_multi_roi_display_shape', None)
    shape_changed = old_shape != new_shape
    if shape_changed:
        self.multi_roi_image.set_extent((-0.5, new_shape[1] - 0.5, -0.5, new_shape[0] - 0.5))
        self._multi_roi_display_shape = new_shape
    if zoom_active:
        xmin, xmax, ymin, ymax = [float(v) for v in zoom_limits]
        self._multi_roi_zoomed = True
        self.multi_roi_ax.set_autoscale_on(False)
        self.multi_roi_ax.set_xlim(xmin, xmax)
        self.multi_roi_ax.set_ylim(ymin, ymax)
    elif shape_changed:
        self.multi_roi_ax.set_autoscale_on(False)
        self.multi_roi_ax.set_xlim(-0.5, new_shape[1] - 0.5)
        self.multi_roi_ax.set_ylim(-0.5, new_shape[0] - 0.5)
        self.multi_roi_ax.set_aspect('equal', adjustable='box')
    for artist in self.multi_roi_patches + self.multi_roi_labels:
        try:
            artist.remove()
        except Exception:
            pass
    self.multi_roi_patches = []
    self.multi_roi_labels = []
    for number, roi in enumerate(self.multi_rois, start=1):
        rect = plt.Rectangle((roi['xmin'], roi['ymin']), roi['xmax'] - roi['xmin'] + 1, roi['ymax'] - roi['ymin'] + 1, fill=False, linewidth=2.2, edgecolor='yellow', zorder=20)
        self.multi_roi_ax.add_patch(rect)
        self.multi_roi_patches.append(rect)
        label = self.multi_roi_ax.text((roi['xmin'] + roi['xmax']) / 2.0, (roi['ymin'] + roi['ymax']) / 2.0, str(number), color='white', fontsize=12, fontweight='bold', ha='center', va='center', bbox=dict(facecolor='black', alpha=0.65, edgecolor='white', pad=2), zorder=25)
        self.multi_roi_labels.append(label)
    field = float(self.real_fields_mT[int(self.multi_roi_frame_var.get())]) if self.real_fields_mT is not None else float(self.multi_roi_frame_var.get())
    self.multi_roi_ax.set_title(f'MULTI-ROI SELECTION | Frame {int(self.multi_roi_frame_var.get()) + 1}/{len(self.recons)} | Field = {field:+.2f} mT | ROIs = {len(self.multi_rois)}')
    self.multi_roi_status_var.set(f'{len(self.multi_rois)} ROI(s) selected. ROIs are numbered in selection order.')
    if zoom_active:
        self.multi_roi_ax.set_autoscale_on(False)
        self.multi_roi_ax.set_xlim(float(zoom_limits[0]), float(zoom_limits[1]))
        self.multi_roi_ax.set_ylim(float(zoom_limits[2]), float(zoom_limits[3]))
    self.multi_roi_canvas.draw_idle()

def start_multi_roi_selection(self):
    if self.recons is None:
        messagebox.showwarning('Multi-ROI Selection', 'Load reconstructions first.', parent=self.root)
        return
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('Multi-ROI Selection', 'First apply the ROI Processing resolution in Tab 5.', parent=self.root)
        return
    self._multi_roi_selecting = True
    self._multi_roi_press_xy = None
    self.multi_roi_status_var.set('DRAWING MODE ON — drag one rectangle at a time. They will be numbered ROI 1, ROI 2, ROI 3, ...')

def finish_multi_roi_selection(self):
    self._multi_roi_selecting = False
    self._multi_roi_press_xy = None
    if self.multi_roi_temp_patch is not None:
        try:
            self.multi_roi_temp_patch.remove()
        except Exception:
            pass
        self.multi_roi_temp_patch = None
    self.multi_roi_canvas.draw_idle()
    if self.multi_rois:
        self.multi_roi_status_var.set(f'✓ Finished: {len(self.multi_rois)} ROI(s). Go to Tab 10 for Gaussian/FWHM fitting.')
    else:
        self.multi_roi_status_var.set('No ROIs selected.')

def clear_multi_rois(self):
    self._multi_roi_selecting = False
    self._multi_roi_press_xy = None
    self.multi_rois = []
    self.multi_roi_fit_results = None
    if self.multi_roi_temp_patch is not None:
        try:
            self.multi_roi_temp_patch.remove()
        except Exception:
            pass
        self.multi_roi_temp_patch = None
    self.multi_roi_status_var.set('All multi-ROIs cleared.')
    if hasattr(self, 'multi_roi_canvas'):
        self.update_multi_roi_view()

def _get_multi_reference_angles(self):
    try:
        return list(self._get_line_reference_angles())
    except Exception:
        mode = str(self.line_ref_mode_var.get())
        presets = {'1': [0.0], '2': [0.0, 90.0], '4': [0.0, 45.0, 90.0, 135.0], '6': [0.0, 30.0, 60.0, 90.0, 120.0, 150.0]}
        return presets.get(mode, [0.0, 90.0])

def _extract_multi_profiles(self, crop, cx, cy, angles):
    try:
        return self._extract_line_profiles(crop, cx, cy, angles)
    except Exception:
        h, w = crop.shape
        max_r = max(2.0, min(cx, cy, w - 1 - cx, h - 1 - cy))
        t = np.linspace(-max_r, max_r, 401)
        profiles = {}
        for angle in angles:
            rad = np.deg2rad(float(angle))
            xx = cx + t * np.cos(rad)
            yy = cy + t * np.sin(rad)
            profile = scipy.ndimage.map_coordinates(crop, [yy, xx], order=1, mode='nearest')
            profiles[float(angle)] = profile
        avg = np.nanmean(np.vstack(list(profiles.values())), axis=0)
        return (t, profiles, avg)

def _fit_one_multi_roi(self, roi, frame_idx, ref_angles):
    image = np.asarray(self._multi_roi_get_image(frame_idx), dtype=float)
    x0 = int(roi['xmin'])
    x1 = int(roi['xmax'])
    y0 = int(roi['ymin'])
    y1 = int(roi['ymax'])
    crop = image[y0:y1 + 1, x0:x1 + 1].copy()
    if crop.size == 0:
        raise ValueError('Empty ROI.')
    center = self._line_contrast_center(crop)
    cx = float(center['x0'])
    cy = float(center['y0'])
    t, profiles, avg = self._extract_multi_profiles(crop, cx, cy, ref_angles)
    fit = self._fit_fixed_center_1d_gaussian(t, avg)
    field = float(self.real_fields_mT[frame_idx]) if self.real_fields_mT is not None else float(frame_idx)
    return {'roi_index': self.multi_rois.index(roi) + 1, 'frame': frame_idx, 'field_mT': field, 'global_xmin': x0, 'global_xmax': x1, 'global_ymin': y0, 'global_ymax': y1, 'center_x_px': x0 + cx, 'center_y_px': y0 + cy, 'fwhm': float(fit['fwhm']), 'fwhm_err': float(fit.get('fwhm_err', np.nan)), 'sigma': float(fit['sigma']), 'sigma_err': float(fit.get('sigma_err', np.nan)), 'amplitude': float(fit['amplitude']), 'background': float(fit['background']), 'r2': float(fit.get('r2', np.nan)), 'rmse': float(fit.get('rmse', np.nan)), 't': t, 'profiles': profiles, 'average_profile': avg, 'fit_profile': fit['fit_profile'], 'residual_profile': fit['residual_profile']}

def fit_all_multi_rois_current_frame(self):
    if self.recons is None:
        return
    if not self.multi_rois:
        messagebox.showwarning('Multi-ROI Gaussian Fit', 'Select at least one ROI in Tab 9 first.', parent=self.root)
        return
    frame_idx = max(0, min(int(self.multi_fit_frame_var.get()), len(self.recons) - 1))
    ref_angles = self._get_multi_reference_angles()
    results = []
    errors = []
    for roi in self.multi_rois:
        try:
            results.append(self._fit_one_multi_roi(roi, frame_idx, ref_angles))
        except Exception as exc:
            errors.append(f'ROI {self.multi_rois.index(roi) + 1}: {type(exc).__name__}: {exc}')
    self.multi_roi_fit_results = {'frame': frame_idx, 'field_mT': float(self.real_fields_mT[frame_idx]) if self.real_fields_mT is not None else float(frame_idx), 'results': results, 'errors': errors}
    self._plot_multi_fit_results(self.multi_roi_fit_results)
    self.multi_roi_fit_status_var.set(f'✓ Fitted {len(results)}/{len(self.multi_rois)} ROI(s) | Frame {frame_idx + 1}/{len(self.recons)}')

def _plot_multi_fit_results(self, pack):
    for ax in (self.multi_fit_ax_image, self.multi_fit_ax_profiles, self.multi_fit_ax_fwhm):
        ax.clear()
    frame_idx = int(pack['frame'])
    try:
        image = self._multi_roi_get_image(frame_idx)
    except Exception:
        image = np.zeros((10, 10), dtype=float)
    self.multi_fit_ax_image.imshow(image, cmap=self.colormap_var.get(), origin='lower', interpolation='nearest')
    results = pack['results']
    for r in results:
        rect = plt.Rectangle((r['global_xmin'], r['global_ymin']), r['global_xmax'] - r['global_xmin'] + 1, r['global_ymax'] - r['global_ymin'] + 1, fill=False, linewidth=2, edgecolor='yellow')
        self.multi_fit_ax_image.add_patch(rect)
        self.multi_fit_ax_image.text((r['global_xmin'] + r['global_xmax']) / 2, (r['global_ymin'] + r['global_ymax']) / 2, str(r['roi_index']), color='white', fontsize=11, fontweight='bold', ha='center', va='center', bbox=dict(facecolor='black', alpha=0.65, edgecolor='white', pad=2))
        self.multi_fit_ax_profiles.plot(r['t'], r['average_profile'], linewidth=1.3, label=f"ROI {r['roi_index']} data")
        self.multi_fit_ax_profiles.plot(r['t'], r['fit_profile'], linewidth=2, linestyle='--', label=f"ROI {r['roi_index']} Gaussian")
    self.multi_fit_ax_image.set_title(f'Frame {frame_idx + 1} | {len(results)} ROI(s)')
    self.multi_fit_ax_image.set_xlabel('X pixel')
    self.multi_fit_ax_image.set_ylabel('Y pixel')
    self.multi_fit_ax_image.axis('image')
    self.multi_fit_ax_profiles.set_title('Selected ROI average profiles + Gaussian fits')
    self.multi_fit_ax_profiles.set_xlabel('Position from maximum-contrast center (px)')
    self.multi_fit_ax_profiles.set_ylabel('Intensity')
    if results:
        self.multi_fit_ax_profiles.legend(fontsize=7, loc='best')
        roi_ids = [r['roi_index'] for r in results]
        fwhm = [r['fwhm'] for r in results]
        self.multi_fit_ax_fwhm.plot(roi_ids, fwhm, 'o-', linewidth=1.8)
        self.multi_fit_ax_fwhm.set_xticks(roi_ids)
    self.multi_fit_ax_fwhm.set_title('FWHM for selected ROIs')
    self.multi_fit_ax_fwhm.set_xlabel('ROI number')
    self.multi_fit_ax_fwhm.set_ylabel('FWHM (px)')
    self.multi_fit_ax_field.clear()
    self.multi_fit_ax_field.set_title('FWHM vs Field — use TRACK ALL ROI FWHM vs FIELD')
    self.multi_fit_ax_field.set_xlabel('Field (mT)')
    self.multi_fit_ax_field.set_ylabel('FWHM (px)')
    for ax in (self.multi_fit_ax_image, self.multi_fit_ax_profiles, self.multi_fit_ax_fwhm, self.multi_fit_ax_field):
        ax.grid(alpha=0.2)
    for item in self.multi_fit_tree.get_children():
        self.multi_fit_tree.delete(item)
    for r in results:
        self.multi_fit_tree.insert('', 'end', values=(r['roi_index'], f"{r['fwhm']:.5g}", f"{r['sigma']:.5g}", f"{r['r2']:.5f}", f"{r['center_x_px']:.3f}", f"{r['center_y_px']:.3f}"))
    self.multi_fit_canvas.draw_idle()

def set_all_multi_fit_range(self):
    if self.recons is None:
        return
    self.multi_fit_start_var.set(0)
    self.multi_fit_end_var.set(max(0, len(self.recons) - 1))

def _tab9_particle_core_detected(self, crop, roi):
    """
    Strict particle-presence test used ONLY by TAB 9 TRACK ALL ROIs.

    A candidate must survive background removal, multi-scale local-extremum
    detection, SNR, local-prominence, compactness, and fixed-ROI-center checks.
    The configured contrast fraction (for example 0.60) is used as an
    additional core threshold, but is not used by itself.
    """
    try:
        a = np.asarray(crop, dtype=float)
        if a.ndim != 2 or a.size < 49:
            return False

        finite = np.isfinite(a)
        if not np.any(finite):
            return False
        med = float(np.nanmedian(a[finite]))
        if not np.isfinite(med):
            return False
        a = np.where(finite, a, med)

        h, w = a.shape
        if min(h, w) < 7:
            return False

        # Robust perimeter background/noise.
        b = max(2, int(round(0.10 * min(h, w))))
        border = np.concatenate((
            a[:b, :].ravel(), a[-b:, :].ravel(),
            a[:, :b].ravel(), a[:, -b:].ravel()
        ))
        border = border[np.isfinite(border)]
        if border.size < 10:
            return False

        bg = float(np.median(border))
        mad = float(np.median(np.abs(border - bg)))
        noise = 1.4826 * mad
        if not np.isfinite(noise) or noise <= 0:
            noise = float(np.std(border))
        if not np.isfinite(noise) or noise <= 0:
            noise = max(np.finfo(float).eps, abs(bg) * 1e-12)

        raw_contrast = a - bg
        pos_peak = float(np.max(raw_contrast))
        neg_peak = float(np.min(raw_contrast))

        # Keep the existing bright/dark contrast convention.
        if abs(pos_peak) >= abs(neg_peak):
            signed_raw = raw_contrast
            sign = 1.0
            raw_peak = pos_peak
        else:
            signed_raw = -raw_contrast
            sign = -1.0
            raw_peak = -neg_peak

        if not np.isfinite(raw_peak) or raw_peak <= 0:
            return False

        # Remove slow background/gradient before looking for a particle.
        smooth_scales = (
            max(0.8, min(h, w) / 60.0),
            max(1.2, min(h, w) / 35.0),
            max(1.8, min(h, w) / 22.0),
        )

        best = None

        # Use the existing configured contrast threshold.
        try:
            frac = float(np.clip(
                self.multi_fit_contrast_fraction_var.get(), 0.0, 0.95
            ))
        except Exception:
            frac = 0.60

        # A relative contrast threshold alone can select noise.  Require
        # substantial absolute SNR as well.
        min_snr = 5.0

        # Fixed center = geometric center of the selected ROI.
        roi_cx = 0.5 * (float(roi['xmin']) + float(roi['xmax']))
        roi_cy = 0.5 * (float(roi['ymin']) + float(roi['ymax']))
        center_tol = 0.45 * min(h, w)

        for sig in smooth_scales:
            sm = gaussian_filter(a, sigma=sig)
            broad_sigma = max(3.0 * sig, 0.22 * min(h, w))
            bg_slow = gaussian_filter(sm, sigma=broad_sigma)
            residual = sm - bg_slow

            # Candidate peak in the background-corrected image.
            if sign > 0:
                signed = residual
                extrema = ndi.maximum_filter(
                    signed, size=max(5, int(round(0.12 * min(h, w))) | 1),
                    mode='nearest'
                ) == signed
            else:
                signed = -residual
                extrema = ndi.minimum_filter(
                    signed, size=max(5, int(round(0.12 * min(h, w))) | 1),
                    mode='nearest'
                ) == signed

            peak_abs = float(np.max(signed))
            if not np.isfinite(peak_abs) or peak_abs <= 0:
                continue

            # Robust residual noise estimate from perimeter.
            rb = np.concatenate((
                residual[:b, :].ravel(), residual[-b:, :].ravel(),
                residual[:, :b].ravel(), residual[:, -b:].ravel()
            ))
            rb = rb[np.isfinite(rb)]
            rmad = float(np.median(np.abs(rb - np.median(rb))))
            rnoise = 1.4826 * rmad
            if not np.isfinite(rnoise) or rnoise <= 0:
                rnoise = float(np.std(rb))
            if not np.isfinite(rnoise) or rnoise <= 0:
                rnoise = noise

            threshold = frac * peak_abs
            candidate_mask = (signed >= threshold) & extrema
            yy, xx = np.nonzero(candidate_mask)
            if xx.size == 0:
                continue

            # Test the strongest local extrema first.
            order = np.argsort(signed[yy, xx])[::-1]
            for kk in order[:min(12, len(order))]:
                cx, cy = int(xx[kk]), int(yy[kk])
                peak = float(signed[cy, cx])

                if peak / max(rnoise, np.finfo(float).eps) < min_snr:
                    continue

                # Fixed-center constraint: do not chase a random edge/noise peak.
                if np.hypot(cx - (w - 1) / 2.0,
                            cy - (h - 1) / 2.0) > center_tol:
                    continue

                # Local prominence against a surrounding annulus.
                rr = max(4, int(round(0.12 * min(h, w))))
                yy0, yy1 = max(0, cy - rr), min(h, cy + rr + 1)
                xx0, xx1 = max(0, cx - rr), min(w, cx + rr + 1)
                patch = signed[yy0:yy1, xx0:xx1]
                gy, gx = np.indices(patch.shape)
                pcx, pcy = cx - xx0, cy - yy0
                d2 = (gx - pcx) ** 2 + (gy - pcy) ** 2

                inner_r = max(1.5, 0.35 * rr)
                outer_r = max(inner_r + 1, 0.85 * rr)
                inner = patch[d2 <= inner_r ** 2]
                annulus = patch[
                    (d2 >= inner_r ** 2) & (d2 <= outer_r ** 2)
                ]

                if inner.size < 3 or annulus.size < 5:
                    continue

                prominence = float(
                    np.median(inner) - np.median(annulus)
                )
                if prominence < max(3.0 * rnoise, 0.12 * peak):
                    continue

                # Compactness: the 0.60-threshold core should be a localized
                # component, not a large gradient/background region.
                core = signed >= threshold
                labels, nlab = ndi.label(core)
                lab = int(labels[cy, cx])
                if lab <= 0:
                    continue
                comp = labels == lab
                area = int(comp.sum())
                min_area = max(3, int(round(0.0015 * h * w)))
                max_area = max(min_area + 1, int(round(0.25 * h * w)))
                if area < min_area or area > max_area:
                    continue

                ys, xs = np.nonzero(comp)
                if xs.size == 0:
                    continue

                # Reject a structure clipped by the ROI edge.
                if (xs.min() <= 0 or ys.min() <= 0 or
                    xs.max() >= w - 1 or ys.max() >= h - 1):
                    continue

                # Score candidates by SNR + prominence and use the best scale.
                score = (
                    peak / max(rnoise, np.finfo(float).eps)
                    + prominence / max(rnoise, np.finfo(float).eps)
                )
                if best is None or score > best:
                    best = score

        return best is not None

    except Exception:
        return False


def track_all_multi_rois(self):
    if self.recons is None:
        return
    if not self.multi_rois:
        messagebox.showwarning('Multi-ROI FWHM', 'Select at least one ROI in Tab 9 first.', parent=self.root)
        return
    start = max(0, min(int(self.multi_fit_start_var.get()), len(self.recons) - 1))
    end = max(0, min(int(self.multi_fit_end_var.get()), len(self.recons) - 1))
    if start > end:
        start, end = (end, start)
    ref_angles = self._get_multi_reference_angles()
    rows = []
    for frame_idx in range(start, end + 1):
        for roi in self.multi_rois:
            try:
                # IMPORTANT: validate particle presence BEFORE any Gaussian fit.
                # If this ROI contains only background/noise, do not fit it.
                image = np.asarray(self._multi_roi_get_image(frame_idx), dtype=float)
                x0 = int(roi['xmin'])
                x1 = int(roi['xmax'])
                y0 = int(roi['ymin'])
                y1 = int(roi['ymax'])
                crop = image[y0:y1 + 1, x0:x1 + 1].copy()

                if not _tab9_particle_core_detected(self, crop, roi):
                    field = (
                        float(self.real_fields_mT[frame_idx])
                        if self.real_fields_mT is not None
                        else float(frame_idx)
                    )
                    rows.append({
                        'roi_index': self.multi_rois.index(roi) + 1,
                        'frame': frame_idx,
                        'field_mT': field,
                        'fwhm': 0.0,
                        'fwhm_err': 0.0,
                        'sigma': 0.0,
                        'sigma_err': 0.0,
                        'amplitude': 0.0,
                        'amplitude_err': 0.0,
                        'r2': 0.0,
                        'rmse': 0.0,
                        'center_x_px': np.nan,
                        'center_y_px': np.nan,
                        'error': 'Particle core not detected; noise/background rejected.'
                    })
                else:
                    r = self._fit_one_multi_roi(roi, frame_idx, ref_angles)
                    rows.append({
                        k: v for k, v in r.items()
                        if k not in (
                            't', 'profiles', 'average_profile',
                            'fit_profile', 'residual_profile'
                        )
                    })
            except Exception as exc:
                # A failed detection/fit is represented by zero measurement,
                # never by a noise-derived point.
                field = (
                    float(self.real_fields_mT[frame_idx])
                    if self.real_fields_mT is not None
                    else float(frame_idx)
                )
                rows.append({
                    'roi_index': self.multi_rois.index(roi) + 1,
                    'frame': frame_idx,
                    'field_mT': field,
                    'fwhm': 0.0,
                    'fwhm_err': 0.0,
                    'sigma': 0.0,
                    'sigma_err': 0.0,
                    'amplitude': 0.0,
                    'amplitude_err': 0.0,
                    'r2': 0.0,
                    'rmse': 0.0,
                    'center_x_px': np.nan,
                    'center_y_px': np.nan,
                    'error': f'Core detection/fit rejected: {type(exc).__name__}: {exc}'
                })
            self.root.update_idletasks()
    df = pd.DataFrame(rows)
    self.multi_roi_fit_results = {'range_table': df, 'range_start': start, 'range_end': end}
    self.multi_fit_ax_field.clear()
    if not df.empty and 'fwhm' in df.columns:
        for roi_id in sorted(df['roi_index'].dropna().astype(int).unique()):
            sub = df[df['roi_index'].astype(int) == roi_id].sort_values('field_mT')
            self.multi_fit_ax_field.plot(sub['field_mT'].to_numpy(float), sub['fwhm'].to_numpy(float), 'o-', linewidth=1.5, markersize=4, label=f'ROI {roi_id}')
        self.multi_fit_ax_field.legend(fontsize=8, loc='best')
    self.multi_fit_ax_field.set_title(f'Multi-ROI FWHM vs Field | frames {start}–{end}')
    self.multi_fit_ax_field.set_xlabel('Field (mT)')
    self.multi_fit_ax_field.set_ylabel('FWHM (px)')
    self.multi_fit_ax_field.grid(alpha=0.2)
    self.multi_fit_canvas.draw_idle()
    self.multi_roi_fit_status_var.set(f'✓ Multi-ROI tracking complete | {len(self.multi_rois)} ROI(s) | {len(df)} fits')

def export_multi_roi_csv(self):
    """
    Figure-4 batch export after fitting.

    Each selected ROI is exported as a separate CSV file.

    Automatic filenames:
        FWHM_Amp_ROI1.CSV
        FWHM_Amp_ROI2.CSV
        ...
        FWHM_Amp_ROIn.CSV

    The user selects only the output folder.
    """
    if not isinstance(self.multi_roi_fit_results, dict):
        messagebox.showwarning('Figure-4 Export', 'Run TRACK ALL ROI FWHM vs FIELD after fitting first.', parent=self.root)
        return
    if not getattr(self, 'multi_rois', None):
        messagebox.showwarning('Figure-4 Export', 'No ROIs are selected in Tab 9.', parent=self.root)
        return
    table = self.multi_roi_fit_results.get('range_table')
    if table is None or table.empty:
        messagebox.showwarning('Figure-4 Export', 'No Figure-4 tracking results are available.', parent=self.root)
        return
    if 'roi_index' not in table.columns:
        messagebox.showerror('Figure-4 Export', 'The result table does not contain roi_index.', parent=self.root)
        return
    selected_roi_ids = list(range(1, len(self.multi_rois) + 1))
    available_roi_ids = set(table['roi_index'].dropna().astype(int).tolist())
    export_roi_ids = [roi_id for roi_id in selected_roi_ids if roi_id in available_roi_ids]
    if not export_roi_ids:
        messagebox.showwarning('Figure-4 Export', 'No selected ROI has fitted Figure-4 data.', parent=self.root)
        return
    output_dir = filedialog.askdirectory(parent=self.root, title='Select folder for FWHM_Amp_ROIx.CSV files')
    if not output_dir:
        return
    output_dir = Path(output_dir)
    preferred_columns = ['roi_index', 'frame', 'field_mT', 'fwhm', 'fwhm_err', 'amplitude', 'amplitude_err', 'sigma', 'sigma_err', 'background', 'r2', 'rmse', 'orthogonal_rmse', 'orthogonal_cv', 'contrast_background', 'contrast_threshold', 'center_x_px', 'center_y_px', 'local_center_x_px', 'local_center_y_px', 'global_xmin', 'global_xmax', 'global_ymin', 'global_ymax']
    export_columns = [col for col in preferred_columns if col in table.columns]
    export_columns += [col for col in table.columns if col not in export_columns and col not in ('t', 'profiles', 'average_profile', 'fit_profile', 'residual_profile')]
    created_files = []
    failed_files = []
    for roi_id in export_roi_ids:
        try:
            roi_table = table[table['roi_index'].astype(int) == int(roi_id)].copy()
            if roi_table.empty:
                continue
            sort_columns = ['frame']
            if 'field_mT' in roi_table.columns:
                sort_columns.append('field_mT')
            roi_table = roi_table.sort_values(sort_columns, kind='stable')
            roi_table = roi_table[export_columns]
            file_path = output_dir / f'FWHM_Amp_ROI{int(roi_id)}.CSV'
            roi_table.to_csv(file_path, index=False)
            created_files.append(file_path)
        except Exception as exc:
            failed_files.append((roi_id, f'{type(exc).__name__}: {exc}'))
    if created_files:
        names = '\n'.join((p.name for p in created_files))
        self.multi_roi_fit_status_var.set(f'✓ Exported {len(created_files)} ROI CSV file(s)')
        self.log(f'Figure-4 batch CSV export complete: {output_dir}\n' + names)
        message = f'Figure-4 batch export complete.\n\nCreated: {len(created_files)} file(s)\n\n{names}'
        if failed_files:
            message += '\n\nFailed:\n' + '\n'.join((f'ROI {roi_id}: {err}' for roi_id, err in failed_files))
        messagebox.showinfo('Figure-4 Export Complete', message, parent=self.root)
    else:
        messagebox.showerror('Figure-4 Export', 'No ROI CSV files were created.', parent=self.root)
    if failed_files:
        self.log('Figure-4 CSV export failures: ' + '; '.join((f'ROI {roi_id}: {err}' for roi_id, err in failed_files)))

def _proc_reset_zoom(self):
    """Reset Tab 6 ROI view to the full image in physical coordinates."""
    try:
        ax = self._proc_ax
        base = self._proc_get_tab5_base_pixel_size()
        arr = self._proc_image_display.get_array()
        if arr is None:
            return
        h, w = np.asarray(arr).shape[:2]
        if base is not None and np.isfinite(base) and (base > 0):
            ow = int(getattr(self, 'original_roi_w', w) or w)
            oh = int(getattr(self, 'original_roi_h', h) or h)
            full = (0.0, float(ow) * float(base), 0.0, float(oh) * float(base))
        else:
            full = (-0.5, float(w) - 0.5, -0.5, float(h) - 0.5)
        self._proc_full_physical_extent = full
        self._proc_zoom_limits = None
        self._proc_zoomed = False
        ax.set_autoscale_on(False)
        ax.set_xlim(full[0], full[1])
        ax.set_ylim(full[2], full[3])
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
        self._proc_canvas.draw_idle()
    except Exception as exc:
        try:
            self.log(f'Tab 6 RESET ZOOM failed: {exc}')
        except Exception:
            pass

def _proc_zoom_button(self, direction):
    """Zoom around the current view center in physical nm coordinates."""
    try:
        ax = self._proc_ax
        arr = self._proc_image_display.get_array()
        if arr is None:
            return
        base = self._proc_get_tab5_base_pixel_size()
        h, w = np.asarray(arr).shape[:2]
        if base is not None and np.isfinite(base) and (base > 0):
            ow = int(getattr(self, 'original_roi_w', w) or w)
            oh = int(getattr(self, 'original_roi_h', h) or h)
            full = (0.0, float(ow) * float(base), 0.0, float(oh) * float(base))
        else:
            full = (-0.5, float(w) - 0.5, -0.5, float(h) - 0.5)
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        if not self._proc_zoomed or self._proc_zoom_limits is None:
            x0, x1, y0, y1 = full
        cx, cy = (0.5 * (x0 + x1), 0.5 * (y0 + y1))
        factor = 1.25 if direction > 0 else 1.0 / 1.25
        hx = 0.5 * abs(x1 - x0) * factor
        hy = 0.5 * abs(y1 - y0) * factor
        nx0, nx1 = (cx - hx, cx + hx)
        ny0, ny1 = (cy - hy, cy + hy)
        if nx1 - nx0 >= full[1] - full[0] - 1e-12 and ny1 - ny0 >= full[3] - full[2] - 1e-12:
            return self._proc_reset_zoom()
        if nx0 < full[0]:
            nx1 += full[0] - nx0
            nx0 = full[0]
        if nx1 > full[1]:
            nx0 -= nx1 - full[1]
            nx1 = full[1]
        if ny0 < full[2]:
            ny1 += full[2] - ny0
            ny0 = full[2]
        if ny1 > full[3]:
            ny0 -= ny1 - full[3]
            ny1 = full[3]
        self._proc_zoom_limits = (nx0, nx1, ny0, ny1)
        self._proc_zoomed = True
        ax.set_autoscale_on(False)
        ax.set_xlim(nx0, nx1)
        ax.set_ylim(ny0, ny1)
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
        self._proc_canvas.draw_idle()
    except Exception as exc:
        try:
            self.log(f'Tab 6 zoom button failed: {exc}')
        except Exception:
            pass

def _proc_restore_zoom_limits(self, width, height):
    """Restore stored Tab 6 zoom limits in physical nm coordinates."""
    limits = getattr(self, '_proc_zoom_limits', None)
    if not limits:
        return
    base = self._proc_get_tab5_base_pixel_size()
    if base is not None and np.isfinite(base) and (base > 0):
        ow = int(getattr(self, 'original_roi_w', width) or width)
        oh = int(getattr(self, 'original_roi_h', height) or height)
        full = (0.0, float(ow) * float(base), 0.0, float(oh) * float(base))
    else:
        full = (-0.5, float(width) - 0.5, -0.5, float(height) - 0.5)
    x0, x1, y0, y1 = [float(v) for v in limits]
    x0 = max(full[0], min(x0, full[1]))
    x1 = max(full[0], min(x1, full[1]))
    y0 = max(full[2], min(y0, full[3]))
    y1 = max(full[2], min(y1, full[3]))
    if x1 <= x0:
        x0, x1 = (full[0], full[1])
    if y1 <= y0:
        y0, y1 = (full[2], full[3])
    self._proc_zoom_limits = (x0, x1, y0, y1)
    self._proc_zoomed = True
    self._proc_ax.set_autoscale_on(False)
    self._proc_ax.set_xlim(x0, x1)
    self._proc_ax.set_ylim(y0, y1)
    self._proc_ax.set_aspect('equal', adjustable='box')
    self._proc_ax.set_anchor('C')

def _proc_zoom_scroll(self, event):
    """Cursor-centred Tab 6 zoom in the same coordinate system shown on the axes."""
    if getattr(event, 'inaxes', None) is not self._proc_ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    step = getattr(event, 'step', 0)
    if not step:
        return
    arr = self._proc_image_display.get_array()
    if arr is None:
        return
    base = self._proc_get_tab5_base_pixel_size()
    h, w = np.asarray(arr).shape[:2]
    if base is not None and np.isfinite(base) and (base > 0):
        ow = int(getattr(self, 'original_roi_w', w) or w)
        oh = int(getattr(self, 'original_roi_h', h) or h)
        full = (0.0, float(ow) * float(base), 0.0, float(oh) * float(base))
    else:
        full = (-0.5, float(w) - 0.5, -0.5, float(h) - 0.5)
    ax = self._proc_ax
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    scale = 1.2 ** (-float(step))
    cx, cy = (float(event.xdata), float(event.ydata))
    nx0 = cx - (cx - x0) * scale
    nx1 = cx + (x1 - cx) * scale
    ny0 = cy - (cy - y0) * scale
    ny1 = cy + (y1 - cy) * scale
    if nx0 < full[0]:
        nx0 = full[0]
    if nx1 > full[1]:
        nx1 = full[1]
    if ny0 < full[2]:
        ny0 = full[2]
    if ny1 > full[3]:
        ny1 = full[3]
    if nx1 - nx0 >= 0.9999 * (full[1] - full[0]) and ny1 - ny0 >= 0.9999 * (full[3] - full[2]):
        return self._proc_reset_zoom()
    self._proc_zoom_limits = (nx0, nx1, ny0, ny1)
    self._proc_zoomed = True
    ax.set_autoscale_on(False)
    ax.set_xlim(nx0, nx1)
    ax.set_ylim(ny0, ny1)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    self._proc_canvas.draw_idle()

def _proc_restore_zoom_limits(self, width, height):
    """Restore/clamp stored Tab-5 zoom limits to the current image."""
    limits = getattr(self, '_proc_zoom_limits', None)
    if not limits:
        return
    xmin, xmax, ymin, ymax = [float(v) for v in limits]
    full_xmin, full_xmax = (-0.5, float(width) - 0.5)
    full_ymin, full_ymax = (-0.5, float(height) - 0.5)
    xmin = max(full_xmin, min(xmin, full_xmax))
    xmax = max(full_xmin, min(xmax, full_xmax))
    ymin = max(full_ymin, min(ymin, full_ymax))
    ymax = max(full_ymin, min(ymax, full_ymax))
    if xmax <= xmin:
        xmin, xmax = (full_xmin, full_xmax)
    if ymax <= ymin:
        ymin, ymax = (full_ymin, full_ymax)
    self._proc_zoom_limits = (xmin, xmax, ymin, ymax)
    self._proc_ax.set_xlim(xmin, xmax)
    self._proc_ax.set_ylim(ymin, ymax)

def _proc_zoom_scroll(self, event):
    """Zoom Tab-5 primary ROI around the exact mouse cursor position."""
    if getattr(event, 'inaxes', None) is not self._proc_ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    step = getattr(event, 'step', 0)
    if not step:
        return
    base_scale = 1.2
    scale = base_scale ** (-float(step))
    ax = self._proc_ax
    ax.set_autoscale_on(False)
    if getattr(self, '_proc_zoomed', False) and self._proc_zoom_limits is not None:
        xmin, xmax, ymin, ymax = self._proc_zoom_limits
    else:
        image = self._proc_image_display.get_array()
        if image is None:
            return
        h, w = np.asarray(image).shape[:2]
        xmin, xmax = (-0.5, w - 0.5)
        ymin, ymax = (-0.5, h - 0.5)
    cx, cy = (float(event.xdata), float(event.ydata))
    new_xmin = cx - (cx - xmin) * scale
    new_xmax = cx + (xmax - cx) * scale
    new_ymin = cy - (cy - ymin) * scale
    new_ymax = cy + (ymax - cy) * scale
    image = self._proc_image_display.get_array()
    if image is None:
        return
    h, w = np.asarray(image).shape[:2]
    full_xmin, full_xmax = (-0.5, float(w) - 0.5)
    full_ymin, full_ymax = (-0.5, float(h) - 0.5)
    new_xmin = max(full_xmin, new_xmin)
    new_xmax = min(full_xmax, new_xmax)
    new_ymin = max(full_ymin, new_ymin)
    new_ymax = min(full_ymax, new_ymax)
    if new_xmax - new_xmin >= (full_xmax - full_xmin) * 0.999:
        self._proc_reset_zoom()
        return
    self._proc_zoomed = True
    self._proc_zoom_limits = (new_xmin, new_xmax, new_ymin, new_ymax)
    ax.set_xlim(new_xmin, new_xmax)
    ax.set_ylim(new_ymin, new_ymax)
    ax.set_aspect('equal', adjustable='box')
    self._proc_canvas.draw_idle()
CDIWorkflowApp.update_multi_roi_view = update_multi_roi_view
CDIWorkflowApp.start_multi_roi_selection = start_multi_roi_selection
CDIWorkflowApp.finish_multi_roi_selection = finish_multi_roi_selection
CDIWorkflowApp.clear_multi_rois = clear_multi_rois
CDIWorkflowApp._get_multi_reference_angles = _get_multi_reference_angles
CDIWorkflowApp._extract_multi_profiles = _extract_multi_profiles
CDIWorkflowApp._fit_one_multi_roi = _fit_one_multi_roi
CDIWorkflowApp.fit_all_multi_rois_current_frame = fit_all_multi_rois_current_frame
CDIWorkflowApp._plot_multi_fit_results = _plot_multi_fit_results
CDIWorkflowApp.set_all_multi_fit_range = set_all_multi_fit_range
CDIWorkflowApp.track_all_multi_rois = track_all_multi_rois
CDIWorkflowApp.export_multi_roi_csv = export_multi_roi_csv
CDIWorkflowApp._get_line_reference_angles = _get_line_reference_angles
CDIWorkflowApp._extract_multi_line_profiles = _extract_multi_line_profiles
CDIWorkflowApp.line_reference_mode_changed = line_reference_mode_changed
CDIWorkflowApp._line_contrast_center = _line_contrast_center
CDIWorkflowApp._extract_line_profiles = _extract_line_profiles
CDIWorkflowApp._fit_fixed_center_1d_gaussian = _fit_fixed_center_1d_gaussian
CDIWorkflowApp.load_line_from_tab6 = load_line_from_tab6
CDIWorkflowApp.line_recalculate_center = line_recalculate_center
CDIWorkflowApp._line_frame_changed = _line_frame_changed
CDIWorkflowApp._line_refresh_display = _line_refresh_display
CDIWorkflowApp.fit_line_average_current = fit_line_average_current
CDIWorkflowApp.export_line_profile_csv = export_line_profile_csv
CDIWorkflowApp.line_set_all_range = line_set_all_range
CDIWorkflowApp.track_line_fwhm_range = track_line_fwhm_range
CDIWorkflowApp.export_line_fwhm_range_csv = export_line_fwhm_range_csv
CDIWorkflowApp._build_proc_tab = _proc_build
CDIWorkflowApp.set_roi_preset = _proc_set_roi_preset
CDIWorkflowApp.update_roi_height_from_width = _proc_update_roi_height_from_width
CDIWorkflowApp.set_original_roi_preset = _proc_set_original_roi_preset
CDIWorkflowApp.build_roi_stack = _proc_build_roi_stack
CDIWorkflowApp.apply_same_confirmed_roi_resolution = _proc_apply_resolution
CDIWorkflowApp.apply_roi_resolution_from_gui = _proc_apply_from_gui
CDIWorkflowApp.process_image = _proc_image
CDIWorkflowApp.fft_image = _proc_fft
CDIWorkflowApp.apply_fft_mask = _proc_fft_mask
CDIWorkflowApp.update_processing_view = _proc_update
CDIWorkflowApp._proc_slider_update = _proc_slider_update
CDIWorkflowApp._get_roi_mask_local = _get_roi_mask_local
CDIWorkflowApp._show_proc_toplevel = _proc_show_toplevel
CDIWorkflowApp.show_fft = _proc_show_fft
CDIWorkflowApp.blob_analysis = _proc_blob
CDIWorkflowApp.reset_processing = _proc_reset
CDIWorkflowApp.choose_output_folder = _proc_choose_folder
CDIWorkflowApp.image_to_rgb = _proc_rgb
CDIWorkflowApp.save_single = _proc_save_single
CDIWorkflowApp.batch_export = _proc_batch
CDIWorkflowApp._get_imageio = _proc_imageio
CDIWorkflowApp.create_mp4 = _proc_mp4
CDIWorkflowApp.export_single_mp4 = _proc_single_mp4
CDIWorkflowApp.export_batch_mp4 = _proc_batch_mp4
CDIWorkflowApp.start_secondary_roi_selection = start_secondary_roi_selection
CDIWorkflowApp._proc_secondary_on_press = _proc_secondary_on_press
CDIWorkflowApp._proc_secondary_on_move = _proc_secondary_on_move
CDIWorkflowApp._proc_secondary_on_release = _proc_secondary_on_release
CDIWorkflowApp._proc_draw_secondary_roi_patch = _proc_draw_secondary_roi_patch
CDIWorkflowApp.confirm_secondary_roi_from_processing = confirm_secondary_roi_from_processing
CDIWorkflowApp.clear_secondary_roi_from_processing = clear_secondary_roi_from_processing
CDIWorkflowApp.update_roi2_view = update_roi2_view
CDIWorkflowApp._roi2_source_image = _roi2_source_image
CDIWorkflowApp._roi2_on_press = _roi2_on_press
CDIWorkflowApp._roi2_on_move = _roi2_on_move
CDIWorkflowApp._roi2_on_release = _roi2_on_release
CDIWorkflowApp.confirm_roi2 = confirm_roi2
CDIWorkflowApp.clear_roi2 = clear_roi2
CDIWorkflowApp._roi2_stop_playback = _roi2_stop_playback
CDIWorkflowApp._roi2_play_button_double_click = _roi2_play_button_double_click
CDIWorkflowApp._roi2_sync_controls = _roi2_sync_controls
CDIWorkflowApp._roi2_slider_changed = _roi2_slider_changed
CDIWorkflowApp.play_roi2_frames = play_roi2_frames
CDIWorkflowApp.toggle_roi2_playback = toggle_roi2_playback
CDIWorkflowApp._roi2_play_next = _roi2_play_next
CDIWorkflowApp.go_to_first_roi2_frame = go_to_first_roi2_frame
CDIWorkflowApp.play_roi_frames = play_roi_frames
CDIWorkflowApp._roi_play_next = _roi_play_next
CDIWorkflowApp.pause_roi_frames = pause_roi_frames
CDIWorkflowApp._roi_playback_fps_changed = _roi_playback_fps_changed
CDIWorkflowApp._roi_play_button_double_click = _roi_play_button_double_click
CDIWorkflowApp.go_to_first_roi_frame = go_to_first_roi_frame
_old_clear_roi = CDIWorkflowApp.clear_roi

def _clear_roi_new(self):
    if getattr(self, '_roi_playing', False):
        try:
            self.pause_roi_frames()
        except Exception:
            pass
    if hasattr(self, 'clear_roi2'):
        try:
            self.clear_roi2()
        except Exception:
            pass
    if hasattr(self, 'clear_secondary_roi_from_processing'):
        try:
            self.clear_secondary_roi_from_processing()
        except Exception:
            pass
    _old_clear_roi(self)
    self.local_mask = None
    self.original_roi_h = None
    self.original_roi_w = None
    self.roi_resolution = (0, 0)
    self.roi_resolution_applied = False
    if hasattr(self, 'proc_roi_info_var'):
        self.proc_roi_info_var.set('ROI = --')
        self.proc_status_var.set('ROI cleared.')
CDIWorkflowApp.clear_roi = _clear_roi_new
CDIWorkflowApp._proc_get_tab5_base_pixel_size = _proc_get_tab5_base_pixel_size
CDIWorkflowApp.update_processing_pixel_size = update_processing_pixel_size
CDIWorkflowApp._proc_profile_line_kwargs = _proc_profile_line_kwargs
CDIWorkflowApp.start_processing_profile_measure = start_processing_profile_measure
CDIWorkflowApp._proc_profile_on_press = _proc_profile_on_press
CDIWorkflowApp._proc_profile_on_move = _proc_profile_on_move
CDIWorkflowApp._proc_profile_on_release = _proc_profile_on_release
CDIWorkflowApp.clear_processing_profile_measure = clear_processing_profile_measure
CDIWorkflowApp._proc_update_physical_axes_and_colorbar = _proc_update_physical_axes_and_colorbar
CDIWorkflowApp._proc_reset_zoom = _proc_reset_zoom
CDIWorkflowApp._proc_restore_zoom_limits = _proc_restore_zoom_limits
CDIWorkflowApp._proc_zoom_scroll = _proc_zoom_scroll
CDIWorkflowApp._roi2_use_all_frames = _roi2_use_all_frames
CDIWorkflowApp._roi2_update_fps_display = _roi2_update_fps_display
CDIWorkflowApp.batch_export_roi2_mp4 = batch_export_roi2_mp4
CDIWorkflowApp.batch_export_roi2_images = batch_export_roi2_images
CDIWorkflowApp._proc_save_single = _proc_save_single
CDIWorkflowApp._proc_batch = _proc_batch
CDIWorkflowApp._proc_mp4 = _proc_mp4

def _line_profile_on_tab_selected(self, event=None):
    try:
        if event is not None and event.widget.select() != str(self.tab_line_profile):
            return
        self._line_profile_sync_from_secondary_roi()
    except Exception as exc:
        if hasattr(self, 'line_profile_status_var'):
            self.line_profile_status_var.set(f'Line Profile: {exc}')

def _line_profile_sync_from_secondary_roi(self):
    """Use the currently confirmed Secondary ROI stack and its own frame index."""
    if not hasattr(self, 'line_profile_image'):
        return
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        self.line_profile_frame_label.config(text='--')
        self.line_profile_status_var.set('Waiting for a confirmed Secondary ROI. Select/confirm it in ROI Processing / Tab 7.')
        self.line_profile_ax_image.set_title('Secondary ROI — waiting for confirmation')
        self.line_profile_canvas_image.draw_idle()
        return
    try:
        idx = int(round(float(self.roi2_idx_var.get())))
    except Exception:
        idx = 0
    idx = max(0, min(idx, len(stack) - 1))
    self._line_profile_update_image(idx=idx, preserve_lines=True)

def _line_profile_frame_changed(self, *args):
    """Compatibility callback: Secondary ROI frame changed."""
    return self._line_profile_sync_from_secondary_roi()

def _line_profile_update_image(self, idx=None, preserve_lines=True):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        self._line_profile_sync_from_secondary_roi()
        return
    if idx is None:
        try:
            idx = int(round(float(self.roi2_idx_var.get())))
        except Exception:
            idx = 0
    idx = max(0, min(int(idx), len(stack) - 1))
    try:
        self.roi2_idx_var.set(idx)
    except Exception:
        pass
    image = np.asarray(stack[idx], dtype=float)
    h, w = image.shape[:2]
    cmap_name = getattr(self, 'colormap_var', tk.StringVar(value='gray')).get()
    self.line_profile_image.set_data(image)
    self.line_profile_image.set_cmap(cmap_name)
    self.line_profile_image.set_clim(0, 1)
    self.line_profile_image.set_extent((-0.5, w - 0.5, -0.5, h - 0.5))
    self.line_profile_ax_image.set_xlim(-0.5, w - 0.5)
    self.line_profile_ax_image.set_ylim(-0.5, h - 0.5)
    self.line_profile_ax_image.set_aspect('equal', adjustable='box')
    field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    self.line_profile_ax_image.set_title(f'Secondary ROI — Frame {idx + 1}/{len(stack)} | Field = {field:+.2f} mT')
    self.line_profile_colorbar.update_normal(self.line_profile_image)
    self.line_profile_frame_label.config(text=f'{idx + 1}/{len(stack)} | {field:+.2f} mT')
    if preserve_lines and self._line_profile_lines:
        self._line_profile_redraw_profiles(idx)
    else:
        self.line_profile_canvas_graph.draw_idle()
    self.line_profile_canvas_image.draw_idle()

def _line_profile_start_drawing(self):
    try:
        target = max(1, min(20, int(self.line_profile_sets_var.get())))
    except Exception:
        target = 1
    self.line_profile_sets_var.set(target)
    if getattr(self, 'roi2_stack', None) is None:
        self.line_profile_status_var.set('Confirm a Secondary ROI first.')
        return
    if len(self._line_profile_lines) >= target:
        self.line_profile_status_var.set(f'{target} sets already drawn. Increase the number of sets or clear lines.')
        return
    self._line_profile_active = True
    self._line_profile_press_xy = None
    self.line_profile_status_var.set(f'DRAWING MODE: draw set {len(self._line_profile_lines) + 1}/{target} on the Secondary ROI.')

def _line_profile_set_count_changed(self):
    try:
        target = max(1, min(20, int(self.line_profile_sets_var.get())))
    except Exception:
        target = 1
    self.line_profile_sets_var.set(target)
    if len(self._line_profile_lines) > target:
        self._line_profile_lines = self._line_profile_lines[:target]
        self._line_profile_redraw_profiles()
    self.line_profile_status_var.set(f'Number of line sets = {target}. Press DRAW LINE SETS to add the remaining sets.')

def _line_profile_image_press(self, event):
    if event.inaxes is not self.line_profile_ax_image or event.xdata is None or event.ydata is None:
        return
    if not self._line_profile_active:
        return
    self._line_profile_press_xy = (float(event.xdata), float(event.ydata))
    if self._line_profile_temp_artist is not None:
        try:
            self._line_profile_temp_artist.remove()
        except Exception:
            pass
    self._line_profile_temp_artist, = self.line_profile_ax_image.plot([event.xdata, event.xdata], [event.ydata, event.ydata], '--', lw=1.8, color=self._line_profile_line_color(len(self._line_profile_lines)))
    self.line_profile_canvas_image.draw_idle()

def _line_profile_image_motion(self, event):
    if self._line_profile_press_xy is None or event.inaxes is not self.line_profile_ax_image:
        return
    if event.xdata is None or event.ydata is None:
        return
    x0, y0 = self._line_profile_press_xy
    self._line_profile_temp_artist.set_data([x0, float(event.xdata)], [y0, float(event.ydata)])
    self.line_profile_canvas_image.draw_idle()

def _line_profile_image_release(self, event):
    if self._line_profile_press_xy is None:
        return
    p0 = self._line_profile_press_xy
    self._line_profile_press_xy = None
    if self._line_profile_temp_artist is not None:
        try:
            self._line_profile_temp_artist.remove()
        except Exception:
            pass
        self._line_profile_temp_artist = None
    if event.inaxes is not self.line_profile_ax_image or event.xdata is None or event.ydata is None:
        self.line_profile_canvas_image.draw_idle()
        return
    p1 = (float(event.xdata), float(event.ydata))
    if np.hypot(p1[0] - p0[0], p1[1] - p0[1]) < 1.0:
        self.line_profile_status_var.set('Line too short. Draw a longer line.')
        return
    target = max(1, min(20, int(self.line_profile_sets_var.get())))
    if len(self._line_profile_lines) >= target:
        self._line_profile_active = False
        self.line_profile_status_var.set(f'All {target} requested line sets are already drawn.')
        return
    self._line_profile_lines.append({'p0': p0, 'p1': p1, 'set': len(self._line_profile_lines) + 1})
    self._line_profile_active = len(self._line_profile_lines) < target
    self._line_profile_redraw_profiles()
    self.line_profile_status_var.set(f'Line set {len(self._line_profile_lines)}/{target} added. ' + ('Draw the next line.' if self._line_profile_active else 'All requested sets are complete.'))

def _line_profile_line_color(self, i):
    cmap_name = getattr(self, 'colormap_var', tk.StringVar(value='viridis')).get()
    cmap = plt.get_cmap(cmap_name)
    n = max(1, int(getattr(self, 'line_profile_sets_var', tk.IntVar(value=1)).get()))
    value = 0.18 + 0.6 * (i % n / max(1, n - 1)) if n > 1 else 0.5
    rgba = np.asarray(cmap(value), dtype=float)
    if float(np.dot(rgba[:3], [0.299, 0.587, 0.114])) > 0.88:
        rgba[:3] *= 0.72
    return tuple(rgba)

def _line_profile_color(self, i=0):
    return self._line_profile_line_color(i)

def _line_profile_extract(self, image, p0, p1):
    x0, y0 = p0
    x1, y1 = p1
    length = float(np.hypot(x1 - x0, y1 - y0))
    if length <= 0:
        return (np.array([0.0]), np.array([float(image[int(round(y0)), int(round(x0))])]))
    n = max(2, int(np.ceil(length * 2.0)) + 1)
    dist = np.linspace(0, length, n)
    xs = x0 + (x1 - x0) * dist / length
    ys = y0 + (y1 - y0) * dist / length
    profile = map_coordinates(image, [ys, xs], order=1, mode='nearest')
    return (dist, profile)

def _line_profile_extract_and_plot(self, *args, **kwargs):
    return self._line_profile_redraw_profiles()

def _line_profile_extrema_stats(self, dist, profile):
    from scipy.signal import find_peaks
    try:
        md = max(1, int(round(float(self.line_profile_min_distance_var.get()))))
    except Exception:
        md = 3
    try:
        prom = max(0.0, min(1.0, float(self.line_profile_prominence_var.get())))
    except Exception:
        prom = 0.03
    max_idx, _ = find_peaks(profile, distance=md, prominence=prom)
    min_idx, _ = find_peaks(-profile, distance=md, prominence=prom)
    max_x = dist[max_idx]
    min_x = dist[min_idx]

    def diffs(a):
        return np.abs(np.diff(a)) if len(a) >= 2 else np.array([], dtype=float)
    mm = diffs(min_x)
    xx = diffs(max_x)
    all_idx = np.sort(np.concatenate([min_idx, max_idx]))
    all_d = diffs(dist[all_idx])
    mixed = []
    for a, b in zip(all_idx[:-1], all_idx[1:]):
        if a in min_idx and b in max_idx or (a in max_idx and b in min_idx):
            mixed.append(abs(dist[b] - dist[a]))
    mixed = np.asarray(mixed, dtype=float)

    def stat(a):
        if len(a) == 0:
            return (np.nan, np.nan, 0)
        return (float(np.mean(a)), float(np.std(a, ddof=1)) if len(a) > 1 else 0.0, int(len(a)))
    return {'min_x': min_x, 'max_x': max_x, 'minmin': stat(mm), 'maxmax': stat(xx), 'minmax': stat(mixed), 'all': stat(all_d)}

def _line_profile_redraw_image_lines(self):
    for artist in getattr(self, '_line_profile_image_line_artists', []):
        try:
            artist.remove()
        except Exception:
            pass
    self._line_profile_image_line_artists = []
    for i, line in enumerate(getattr(self, '_line_profile_lines', [])):
        c = self._line_profile_line_color(i)
        a, = self.line_profile_ax_image.plot([line['p0'][0], line['p1'][0]], [line['p0'][1], line['p1'][1]], '-', lw=2.2, color=c, label=f'Set {i + 1}')
        self._line_profile_image_line_artists.append(a)
    for a in getattr(self, '_line_profile_measure_image_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_image_artists = []
    for k, pt in enumerate(getattr(self, '_line_profile_measure_points', [])):
        try:
            x, _y, setno = pt
            ri = int(setno) - 1
            if 0 <= ri < len(getattr(self, '_line_profile_results', [])):
                r = self._line_profile_results[ri]
                L = float(r.get('length', 0.0))
                if L > 0:
                    frac = np.clip(float(x) / L, 0.0, 1.0)
                    p0 = np.asarray(r['p0'], dtype=float)
                    p1 = np.asarray(r['p1'], dtype=float)
                    xy = p0 + frac * (p1 - p0)
                    mc = self._line_profile_line_color(ri)
                    hnd = self.line_profile_ax_image.scatter([xy[0]], [xy[1]], s=72, facecolor=mc, edgecolor='black', linewidth=1.4, zorder=20, marker='o')
                    self._line_profile_measure_image_artists.append(hnd)
                    tag = self.line_profile_ax_image.text(xy[0] + 3, xy[1] + 3, f'M{k + 1}', color='black', fontsize=9, weight='bold', zorder=21, bbox=dict(boxstyle='round,pad=0.15', facecolor='0.85', edgecolor='black', alpha=0.85))
                    self._line_profile_measure_image_artists.append(tag)
        except Exception:
            pass
    self.line_profile_canvas_image.draw_idle()

def _line_profile_redraw_profiles(self, idx=None):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        self._line_profile_results = []
        self._line_profile_redraw_image_lines()
        self._line_profile_update_table()
        return
    if idx is None:
        try:
            idx = int(round(float(self.roi2_idx_var.get())))
        except Exception:
            idx = 0
    idx = max(0, min(int(idx), len(stack) - 1))
    image = np.asarray(stack[idx], dtype=float)
    self.line_profile_ax_graph.clear()
    self.line_profile_ax_graph.set_xlabel('Length (pixels)')
    self.line_profile_ax_graph.set_ylabel('Intensity')
    self.line_profile_ax_graph.set_title(f'Secondary ROI line profiles — Frame {idx + 1}/{len(stack)}')
    self.line_profile_ax_graph.grid(True, alpha=0.22)
    self._line_profile_results = []
    self._line_profile_graph_artists = []
    for i, line in enumerate(self._line_profile_lines):
        dist, profile = self._line_profile_extract(image, line['p0'], line['p1'])
        stats = self._line_profile_extrema_stats(dist, profile)
        result = {'set': i + 1, 'p0': line['p0'], 'p1': line['p1'], 'length': float(dist[-1]) if len(dist) else 0.0, 'dist': dist, 'profile': profile, 'stats': stats}
        self._line_profile_results.append(result)
        ln, = self.line_profile_ax_graph.plot(dist, profile, lw=2.0, color=self._line_profile_line_color(i), label=f'Set {i + 1}', picker=14, pickradius=14, zorder=4)
        ln._line_profile_setno = i + 1
        self._line_profile_graph_artists.append(ln)
    if self._line_profile_results:
        self.line_profile_ax_graph.legend(loc='best')
    self._line_profile_current_profile = self._line_profile_results[-1] if self._line_profile_results else None
    self._line_profile_redraw_image_lines()
    self._line_profile_update_table()
    self._line_profile_draw_measurement()
    self.line_profile_canvas_graph.draw_idle()

def _line_profile_update_table(self):
    if not hasattr(self, 'line_profile_tree'):
        return
    for item in self.line_profile_tree.get_children():
        self.line_profile_tree.delete(item)

    def fmt(stat):
        if stat is None or np.isnan(stat[0]):
            return '—'
        return f'{stat[0]:.2f} ± {stat[1]:.2f}'
    for r in getattr(self, '_line_profile_results', []):
        st = r['stats']
        self.line_profile_tree.insert('', 'end', values=(r['set'], f"{r['length']:.2f}", fmt(st['minmin']), fmt(st['maxmax']), fmt(st['minmax']), fmt(st['all']), len(st['min_x']), len(st['max_x'])))

def _line_profile_recalculate_all(self):
    if self._line_profile_lines:
        self._line_profile_redraw_profiles()
        self.line_profile_status_var.set('Extrema recalculated for all line sets.')
    else:
        self.line_profile_status_var.set('Draw at least one line first.')

def _line_profile_redo_last(self):
    if not self._line_profile_lines:
        self.line_profile_status_var.set('No line to redo.')
        return
    self._line_profile_lines.pop()
    self._line_profile_active = True
    self._line_profile_redraw_profiles()
    self.line_profile_status_var.set(f'Redraw set {len(self._line_profile_lines) + 1}/{self.line_profile_sets_var.get()}.')

def _line_profile_clear_line(self, index=None):
    if not self._line_profile_lines:
        return
    if index is None:
        index = len(self._line_profile_lines) - 1
    try:
        index = int(index)
    except Exception:
        index = len(self._line_profile_lines) - 1
    if 0 <= index < len(self._line_profile_lines):
        self._line_profile_lines.pop(index)
    for i, line in enumerate(self._line_profile_lines):
        line['set'] = i + 1
    self._line_profile_active = len(self._line_profile_lines) < int(self.line_profile_sets_var.get())
    self._line_profile_redraw_profiles()

def _line_profile_clear_all_lines(self):
    self._line_profile_active = False
    self._line_profile_press_xy = None
    self._line_profile_lines = []
    self._line_profile_results = []
    if getattr(self, '_line_profile_temp_artist', None) is not None:
        try:
            self._line_profile_temp_artist.remove()
        except Exception:
            pass
        self._line_profile_temp_artist = None
    self._line_profile_clear_measurement(redraw=False)
    self._line_profile_redraw_profiles()
    self.line_profile_status_var.set('All line sets cleared. Set the number of sets and press DRAW LINE SETS.')

def _line_profile_start_measurement(self):
    if not getattr(self, '_line_profile_results', []):
        self.line_profile_measure_var.set('Draw at least one line first, then use the dynamic measuring scale.')
        return
    self._line_profile_clear_measurement(redraw=False)
    self._line_profile_measure_mode = True
    self._line_profile_measure_drag_index = None
    self.line_profile_measure_var.set('DYNAMIC SCALE: click the first point and second point on a profile. Then drag either M1/M2 handle to adjust the scale; the matching point moves on the image line.')

def _line_profile_find_profile_point(self, x, y=None, set_hint=None):
    """Find the nearest sampled profile point in DISPLAY space.

    Using both x and y makes point selection reliable when several profiles
    overlap in x.  A set_hint from a picked curve is preferred.
    """
    candidates = getattr(self, '_line_profile_results', [])
    if set_hint is not None:
        candidates = [r for r in candidates if int(r.get('set', -1)) == int(set_hint)] or candidates
    if not candidates or x is None:
        return None
    best = None
    ax = self.line_profile_ax_graph
    target_disp = ax.transData.transform((float(x), float(y))) if y is not None else None
    for r in candidates:
        dist = np.asarray(r.get('dist', []), dtype=float)
        prof = np.asarray(r.get('profile', []), dtype=float)
        if dist.size == 0 or prof.size != dist.size:
            continue
        if target_disp is None:
            j = int(np.argmin(np.abs(dist - float(x))))
            dpx = abs(float(dist[j]) - float(x))
        else:
            pts = ax.transData.transform(np.column_stack((dist, prof)))
            dd = np.hypot(pts[:, 0] - target_disp[0], pts[:, 1] - target_disp[1])
            j = int(np.argmin(dd))
            dpx = float(dd[j])
        xx = float(dist[j])
        yy = float(prof[j])
        if best is None or dpx < best[0]:
            best = (dpx, xx, yy, int(r['set']))
    return best

def _line_profile_graph_click(self, event):
    """Place the next dynamic measurement endpoint on the nearest profile."""
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.xdata is None or event.ydata is None or (not self._line_profile_measure_mode):
        return
    best = self._line_profile_find_profile_point(event.xdata, event.ydata)
    if best is None:
        self.line_profile_measure_var.set('Click directly on a profile curve. No profile point was found.')
        return
    if best[0] > 35.0:
        self.line_profile_measure_var.set('Click closer to the plotted intensity curve.')
        return
    _, x, y, setno = best
    self._line_profile_measure_points.append((x, y, setno))
    if len(self._line_profile_measure_points) >= 2:
        self._line_profile_measure_points = self._line_profile_measure_points[:2]
        self._line_profile_measure_mode = False
    self._line_profile_update_measurement_status()
    self._line_profile_draw_measurement()

def _line_profile_graph_pick(self, event):
    """Pick handler: clicking a plotted line selects its nearest data point."""
    try:
        artist = event.artist
        setno = getattr(artist, '_line_profile_setno', None)
        me = event.mouseevent
        if me is None or me.inaxes is not self.line_profile_ax_graph:
            return
        if not self._line_profile_measure_mode:
            return
        x = me.xdata
        y = me.ydata
        if x is None or y is None:
            return
        best = self._line_profile_find_profile_point(x, y, set_hint=setno)
        if best is None or best[0] > 35.0:
            return
        _, xx, yy, sn = best
        self._line_profile_measure_points.append((xx, yy, sn))
        if len(self._line_profile_measure_points) >= 2:
            self._line_profile_measure_points = self._line_profile_measure_points[:2]
            self._line_profile_measure_mode = False
        self._line_profile_update_measurement_status()
        self._line_profile_draw_measurement()
    except Exception:
        pass

def _line_profile_graph_press(self, event):
    """Robust graph interaction for selecting and dragging M1/M2."""
    if event.inaxes is not self.line_profile_ax_graph:
        return
    if event.button not in (None, 1):
        return
    if len(self._line_profile_measure_points) == 2 and event.x is not None and (event.y is not None):
        best_i, best_d = (None, float('inf'))
        for i, (px, py, _setno) in enumerate(self._line_profile_measure_points):
            disp = self.line_profile_ax_graph.transData.transform((float(px), float(py)))
            d = float(np.hypot(float(event.x) - disp[0], float(event.y) - disp[1]))
            if d < best_d:
                best_d, best_i = (d, i)
        if best_i is not None and best_d <= 28.0:
            self._line_profile_measure_drag_index = best_i
            self._line_profile_measure_mode = False
            self.line_profile_measure_var.set(f'Dragging M{best_i + 1}: move horizontally along the profile; the image marker follows.')
            return
    if self._line_profile_measure_mode:
        self._line_profile_graph_click(event)

def _line_profile_graph_motion(self, event):
    """Drag M1/M2 horizontally and snap to the corresponding profile."""
    i = self._line_profile_measure_drag_index
    if i is None or event.inaxes is not self.line_profile_ax_graph or event.xdata is None:
        return
    pts = self._line_profile_measure_points
    if not pts or i >= len(pts):
        return
    target_set = int(pts[i][2])
    candidates = [r for r in self._line_profile_results if int(r.get('set', -1)) == target_set]
    if not candidates:
        return
    r = candidates[0]
    dist = np.asarray(r['dist'], dtype=float)
    prof = np.asarray(r['profile'], dtype=float)
    if dist.size == 0:
        return
    j = int(np.argmin(np.abs(dist - float(event.xdata))))
    self._line_profile_measure_points[i] = (float(dist[j]), float(prof[j]), target_set)
    self._line_profile_update_measurement_status()
    self._line_profile_draw_measurement()

def _line_profile_graph_release(self, event):
    if self._line_profile_measure_drag_index is not None:
        self._line_profile_update_measurement_status()
    self._line_profile_measure_drag_index = None

def _line_profile_update_measurement_status(self):
    pts = self._line_profile_measure_points
    if len(pts) == 0:
        self.line_profile_measure_var.set('Measurement cleared. Click DYNAMIC MEASURING SCALE, then click directly on the profile curve.')
    elif len(pts) == 1:
        x, y, setno = pts[0]
        self.line_profile_measure_var.set(f'M1: Set {setno}, position = {x:.2f} px, intensity = {y:.4f}. Click the second point on a profile.')
    else:
        p0, p1 = pts[:2]
        delta = abs(float(p1[0]) - float(p0[0]))
        self.line_profile_measure_var.set(f'Dynamic scale ΔL = {delta:.2f} pixels | M1: Set {p0[2]}, {p0[0]:.2f} px, I={p0[1]:.4f} | M2: Set {p1[2]}, {p1[0]:.2f} px, I={p1[1]:.4f}. Drag either marker to adjust.')

def _line_profile_draw_measurement(self):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    pts = self._line_profile_measure_points
    if not pts:
        self._line_profile_redraw_image_lines()
        if hasattr(self, 'line_profile_canvas_graph'):
            self.line_profile_canvas_graph.draw_idle()
        return
    c = self._line_profile_line_color(0)
    for k, (x, y, setno) in enumerate(pts[:2]):
        self._line_profile_measure_artists += [self.line_profile_ax_graph.axvline(x, color=c, ls='--', lw=1.6, alpha=0.9), self.line_profile_ax_graph.scatter([x], [y], s=58, color=c, edgecolors='black', linewidths=1.2, zorder=8), self.line_profile_ax_graph.text(x, y, f' M{k + 1} ', color='black', fontsize=8, weight='bold', ha='center', va='bottom', zorder=9, bbox=dict(boxstyle='round,pad=0.12', facecolor='0.85', edgecolor='black', alpha=0.85))]
    if len(pts) == 2:
        x0, y0, _ = pts[0]
        x1, y1, _ = pts[1]
        ymin, ymax = self.line_profile_ax_graph.get_ylim()
        ybar = ymax - 0.1 * max(ymax - ymin, 1e-09)
        self._line_profile_measure_artists.append(self.line_profile_ax_graph.annotate('', xy=(x0, ybar), xytext=(x1, ybar), arrowprops=dict(arrowstyle='<->', color=c, lw=2)))
        self._line_profile_measure_artists.append(self.line_profile_ax_graph.text((x0 + x1) / 2, ybar, f' ΔL = {abs(x1 - x0):.2f} px ', ha='center', va='bottom', color='black', bbox=dict(boxstyle='round,pad=0.2', facecolor='0.85', alpha=0.9, edgecolor='black')))
    self._line_profile_redraw_image_lines()
    self.line_profile_canvas_graph.draw_idle()

def _line_profile_clear_measurement(self, redraw=True):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    for a in getattr(self, '_line_profile_measure_image_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    self._line_profile_measure_image_artists = []
    self._line_profile_measure_points = []
    self._line_profile_measure_mode = False
    self._line_profile_measure_drag_index = None
    if hasattr(self, 'line_profile_measure_var'):
        self.line_profile_measure_var.set('Measurement cleared. Click DYNAMIC MEASURING SCALE to select two points.')
    if redraw:
        self._line_profile_redraw_image_lines()
        if hasattr(self, 'line_profile_canvas_graph'):
            self.line_profile_canvas_graph.draw_idle()

def _line_profile_colormap_refresh(self):
    try:
        self._line_profile_update_image(preserve_lines=True)
    except Exception:
        pass

def _line_profile_export_profiles(self):
    """Export every plotted line-profile sample (x, intensity) to CSV."""
    if not getattr(self, '_line_profile_results', []):
        messagebox.showwarning('Line Profile', 'Draw at least one line first.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Export line-profile graph data', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    try:
        idx = int(round(float(self.roi2_idx_var.get())))
    except Exception:
        idx = 0
    rows = []
    for r in self._line_profile_results:
        for x, y in zip(np.asarray(r['dist'], dtype=float), np.asarray(r['profile'], dtype=float)):
            rows.append({'set': int(r['set']), 'frame': idx + 1, 'field_mT': float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else '', 'length_px': float(x), 'intensity': float(y)})
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} profile points: {fp}')

def _line_profile_export_results(self):
    if not getattr(self, '_line_profile_results', []):
        messagebox.showwarning('Line Profile', 'Draw at least one line first.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Export line-profile results table', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    try:
        idx = int(round(float(self.roi2_idx_var.get())))
    except Exception:
        idx = 0
    rows = []
    for r in self._line_profile_results:
        st = r['stats']

        def v(a, i):
            return '' if np.isnan(a[i]) else a[i]
        rows.append({'set': r['set'], 'frame': idx + 1, 'field_mT': float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else '', 'line_length_px': r['length'], 'min_min_avg_px': v(st['minmin'], 0), 'min_min_error_px': v(st['minmin'], 1), 'max_max_avg_px': v(st['maxmax'], 0), 'max_max_error_px': v(st['maxmax'], 1), 'min_max_avg_px': v(st['minmax'], 0), 'min_max_error_px': v(st['minmax'], 1), 'all_successive_avg_px': v(st['all'], 0), 'all_successive_error_px': v(st['all'], 1), 'min_count': len(st['min_x']), 'max_count': len(st['max_x'])})
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} line-set results: {fp}')
_old_roi2_slider_changed_for_line_profile = CDIWorkflowApp._roi2_slider_changed

def _roi2_slider_changed_with_line_profile(self):
    result = _old_roi2_slider_changed_for_line_profile(self)
    try:
        self._line_profile_sync_from_secondary_roi()
    except Exception:
        pass
    return result
CDIWorkflowApp._roi2_slider_changed = _roi2_slider_changed_with_line_profile
_old_roi2_update_view_for_line_profile = CDIWorkflowApp.update_roi2_view

def _roi2_update_view_with_line_profile(self):
    result = _old_roi2_update_view_for_line_profile(self)
    try:
        self._line_profile_sync_from_secondary_roi()
    except Exception:
        pass
    return result
CDIWorkflowApp.update_roi2_view = _roi2_update_view_with_line_profile
CDIWorkflowApp._line_profile_on_tab_selected = _line_profile_on_tab_selected
CDIWorkflowApp._line_profile_sync_from_secondary_roi = _line_profile_sync_from_secondary_roi
CDIWorkflowApp._line_profile_sync_from_tab5 = _line_profile_sync_from_secondary_roi
CDIWorkflowApp._line_profile_frame_changed = _line_profile_frame_changed
CDIWorkflowApp._line_profile_update_image = _line_profile_update_image
CDIWorkflowApp._line_profile_image_press = _line_profile_image_press
CDIWorkflowApp._line_profile_image_motion = _line_profile_image_motion
CDIWorkflowApp._line_profile_image_release = _line_profile_image_release
CDIWorkflowApp._line_profile_graph_press = _line_profile_graph_press
CDIWorkflowApp._line_profile_graph_pick = _line_profile_graph_pick
CDIWorkflowApp._line_profile_graph_motion = _line_profile_graph_motion
CDIWorkflowApp._line_profile_graph_release = _line_profile_graph_release
CDIWorkflowApp._line_profile_graph_click = _line_profile_graph_click
CDIWorkflowApp._line_profile_extract_and_plot = _line_profile_extract_and_plot
CDIWorkflowApp._line_profile_color = _line_profile_color
CDIWorkflowApp._line_profile_start_measurement = _line_profile_start_measurement
CDIWorkflowApp._line_profile_graph_click = _line_profile_graph_click
CDIWorkflowApp._line_profile_draw_measurement = _line_profile_draw_measurement
CDIWorkflowApp._line_profile_clear_line = _line_profile_clear_line
CDIWorkflowApp._line_profile_clear_measurement = _line_profile_clear_measurement
CDIWorkflowApp._line_profile_update_measurement_status = _line_profile_update_measurement_status
CDIWorkflowApp._line_profile_colormap_refresh = _line_profile_colormap_refresh
CDIWorkflowApp._line_profile_start_drawing = _line_profile_start_drawing
CDIWorkflowApp._line_profile_set_count_changed = _line_profile_set_count_changed
CDIWorkflowApp._line_profile_line_color = _line_profile_line_color
CDIWorkflowApp._line_profile_extract = _line_profile_extract
CDIWorkflowApp._line_profile_extrema_stats = _line_profile_extrema_stats
CDIWorkflowApp._line_profile_redraw_image_lines = _line_profile_redraw_image_lines
CDIWorkflowApp._line_profile_redraw_profiles = _line_profile_redraw_profiles
CDIWorkflowApp._line_profile_update_table = _line_profile_update_table
CDIWorkflowApp._line_profile_recalculate_all = _line_profile_recalculate_all
CDIWorkflowApp._line_profile_redo_last = _line_profile_redo_last
CDIWorkflowApp._line_profile_clear_all_lines = _line_profile_clear_all_lines
CDIWorkflowApp._line_profile_export_results = _line_profile_export_results
CDIWorkflowApp._line_profile_export_profiles = _line_profile_export_profiles

def _bind_mousewheel_to_scale(self, scale_widget, step=None):
    if scale_widget is None or getattr(scale_widget, '_wheel_slider_bound', False):
        return
    try:
        if isinstance(scale_widget, tk.Scale):
            resolution = float(scale_widget.cget('resolution'))
            if resolution <= 0:
                resolution = 1.0
        else:
            lo = float(scale_widget.cget('from'))
            hi = float(scale_widget.cget('to'))
            resolution = abs(hi - lo) / 100.0
            if resolution <= 0:
                resolution = 1.0
        if step is not None:
            resolution = abs(float(step))
            if resolution <= 0:
                resolution = 1.0
    except Exception:
        resolution = 1.0

    def move(direction):
        try:
            value = float(scale_widget.get())
            lo = float(scale_widget.cget('from'))
            hi = float(scale_widget.cget('to'))
            lower, upper = (min(lo, hi), max(lo, hi))
            value = max(lower, min(upper, value + direction * resolution))
            if isinstance(scale_widget, tk.Scale):
                try:
                    base = float(scale_widget.cget('from'))
                    value = base + round((value - base) / resolution) * resolution
                    value = max(lower, min(upper, value))
                except Exception:
                    pass
            scale_widget.set(value)
            command = scale_widget.cget('command')
            if command:
                try:
                    scale_widget.tk.call(command, str(value))
                except Exception:
                    pass
        except Exception:
            pass

    def wheel(event):
        move(1 if event.delta > 0 else -1)
        return 'break'
    scale_widget.bind('<MouseWheel>', wheel, add='+')
    scale_widget.bind('<Button-4>', lambda event: (move(1), 'break')[1], add='+')
    scale_widget.bind('<Button-5>', lambda event: (move(-1), 'break')[1], add='+')
    scale_widget._wheel_slider_bound = True

def _bind_all_sliders_mousewheel(self):

    def walk(widget):
        yield widget
        try:
            for child in widget.winfo_children():
                yield from walk(child)
        except Exception:
            pass
    for widget in walk(self.root):
        if isinstance(widget, (tk.Scale, ttk.Scale)):
            self._bind_mousewheel_to_scale(widget)
CDIWorkflowApp._bind_mousewheel_to_scale = _bind_mousewheel_to_scale
CDIWorkflowApp._bind_all_sliders_mousewheel = _bind_all_sliders_mousewheel

def _bind_mousewheel_to_scale(self, scale_widget, step=None):
    if getattr(scale_widget, '_wheel_slider_bound', False):
        return
    try:
        resolution = float(scale_widget.cget('resolution')) if isinstance(scale_widget, tk.Scale) else abs(float(scale_widget.cget('to')) - float(scale_widget.cget('from'))) / 100.0
        if resolution <= 0:
            resolution = 1.0
        if step is not None:
            resolution = abs(float(step)) or 1.0
    except Exception:
        resolution = 1.0

    def move(d):
        try:
            v = float(scale_widget.get())
            lo = float(scale_widget.cget('from'))
            hi = float(scale_widget.cget('to'))
            v = max(min(v + d * resolution, max(lo, hi)), min(lo, hi))
            scale_widget.set(v)
            cmd = scale_widget.cget('command')
            if cmd:
                try:
                    scale_widget.tk.call(cmd, str(v))
                except Exception:
                    pass
        except Exception:
            pass

    def wheel(e):
        move(1 if e.delta > 0 else -1)
        return 'break'
    scale_widget.bind('<MouseWheel>', wheel, add='+')
    scale_widget.bind('<Button-4>', lambda e: (move(1), 'break')[1], add='+')
    scale_widget.bind('<Button-5>', lambda e: (move(-1), 'break')[1], add='+')
    scale_widget._wheel_slider_bound = True

def _bind_all_sliders_mousewheel(self):

    def walk(w):
        yield w
        try:
            for c in w.winfo_children():
                yield from walk(c)
        except Exception:
            pass
    for w in walk(self.root):
        if isinstance(w, (tk.Scale, ttk.Scale)):
            self._bind_mousewheel_to_scale(w)
CDIWorkflowApp._bind_mousewheel_to_scale = _bind_mousewheel_to_scale
CDIWorkflowApp._bind_all_sliders_mousewheel = _bind_all_sliders_mousewheel

def _line_profile_start_measurement_robust(self):
    """Activate two-point measurement with generous snapping and a clear UI state."""
    if not getattr(self, '_line_profile_results', None):
        try:
            if getattr(self, '_line_profile_lines', None) and getattr(self, 'roi2_stack', None) is not None:
                self._line_profile_redraw_profiles()
        except Exception:
            pass
    if not getattr(self, '_line_profile_results', None):
        self._line_profile_measure_mode = False
        self.line_profile_measure_var.set('No profile data available. Draw at least one line and make sure the profile is visible.')
        return
    self._line_profile_clear_measurement(redraw=False)
    self._line_profile_measure_mode = True
    self._line_profile_measure_drag_index = None
    self.line_profile_measure_var.set('DYNAMIC SCALE ACTIVE: click two points directly on any profile curve. The nearest curve point will be selected automatically. Then drag M1 or M2.')
    try:
        self.line_profile_canvas_graph.get_tk_widget().configure(cursor='crosshair')
    except Exception:
        pass

def _line_profile_find_profile_point_robust(self, x, y=None, set_hint=None, max_px=None):
    """Nearest sampled profile point, optionally restricted to a selected set."""
    candidates = list(getattr(self, '_line_profile_results', []) or [])
    if set_hint is not None:
        hinted = [r for r in candidates if int(r.get('set', -1)) == int(set_hint)]
        if hinted:
            candidates = hinted
    if not candidates or x is None:
        return None
    ax = self.line_profile_ax_graph
    try:
        target = ax.transData.transform((float(x), float(y if y is not None else 0.0)))
    except Exception:
        target = None
    best = None
    for r in candidates:
        dist = np.asarray(r.get('dist', []), dtype=float)
        prof = np.asarray(r.get('profile', []), dtype=float)
        if dist.size == 0 or prof.size != dist.size:
            continue
        if target is None or y is None:
            j = int(np.argmin(np.abs(dist - float(x))))
            d = abs(float(dist[j]) - float(x))
        else:
            pts = ax.transData.transform(np.column_stack((dist, prof)))
            dd = np.hypot(pts[:, 0] - target[0], pts[:, 1] - target[1])
            j = int(np.argmin(dd))
            d = float(dd[j])
        candidate = (d, float(dist[j]), float(prof[j]), int(r.get('set', 1)))
        if best is None or candidate[0] < best[0]:
            best = candidate
    if best is not None and max_px is not None and (best[0] > float(max_px)):
        return None
    return best

def _line_profile_graph_press_robust(self, event):
    """Single, reliable graph mouse handler for both selection and dragging."""
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.button not in (None, 1):
        return
    pts = getattr(self, '_line_profile_measure_points', [])
    if len(pts) == 2 and event.x is not None and (event.y is not None):
        best_i, best_d = (None, float('inf'))
        for i, (px, py, _setno) in enumerate(pts):
            try:
                sx, sy = self.line_profile_ax_graph.transData.transform((float(px), float(py)))
                d = float(np.hypot(float(event.x) - sx, float(event.y) - sy))
            except Exception:
                continue
            if d < best_d:
                best_d, best_i = (d, i)
        if best_i is not None and best_d <= 40.0:
            self._line_profile_measure_drag_index = best_i
            self._line_profile_measure_mode = False
            self.line_profile_measure_var.set(f'Dragging M{best_i + 1}: move horizontally along its profile. Release to keep the point.')
            return
    if not getattr(self, '_line_profile_measure_mode', False):
        return
    if event.xdata is None or event.ydata is None:
        return
    best = self._line_profile_find_profile_point(event.xdata, event.ydata, max_px=80.0)
    if best is None:
        self.line_profile_measure_var.set('No profile close enough. Click on/near a plotted profile curve.')
        return
    _, x, y, setno = best
    self._line_profile_measure_points.append((x, y, setno))
    if len(self._line_profile_measure_points) >= 2:
        self._line_profile_measure_points = self._line_profile_measure_points[:2]
        self._line_profile_measure_mode = False
        try:
            self.line_profile_canvas_graph.get_tk_widget().configure(cursor='')
        except Exception:
            pass
    self._line_profile_update_measurement_status()
    self._line_profile_draw_measurement()

def _line_profile_graph_motion_robust(self, event):
    i = getattr(self, '_line_profile_measure_drag_index', None)
    if i is None or event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    pts = getattr(self, '_line_profile_measure_points', [])
    if not pts or i >= len(pts) or event.xdata is None:
        return
    target_set = int(pts[i][2])
    candidates = [r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == target_set]
    if not candidates:
        return
    r = candidates[0]
    dist = np.asarray(r.get('dist', []), dtype=float)
    prof = np.asarray(r.get('profile', []), dtype=float)
    if dist.size == 0 or prof.size != dist.size:
        return
    j = int(np.argmin(np.abs(dist - float(event.xdata))))
    self._line_profile_measure_points[i] = (float(dist[j]), float(prof[j]), target_set)
    self._line_profile_update_measurement_status()
    self._line_profile_draw_measurement()

def _line_profile_graph_release_robust(self, event):
    if getattr(self, '_line_profile_measure_drag_index', None) is not None:
        self._line_profile_update_measurement_status()
    self._line_profile_measure_drag_index = None
CDIWorkflowApp._line_profile_start_measurement = _line_profile_start_measurement_robust
CDIWorkflowApp._line_profile_find_profile_point = _line_profile_find_profile_point_robust
CDIWorkflowApp._line_profile_graph_press = _line_profile_graph_press_robust
CDIWorkflowApp._line_profile_graph_motion = _line_profile_graph_motion_robust
CDIWorkflowApp._line_profile_graph_release = _line_profile_graph_release_robust

def _line_profile_extract_optimized(self, image, p0, p1):
    x0, y0 = map(float, p0)
    x1, y1 = map(float, p1)
    length = float(np.hypot(x1 - x0, y1 - y0))
    if length <= 0.0:
        ix = int(np.clip(round(x0), 0, image.shape[1] - 1))
        iy = int(np.clip(round(y0), 0, image.shape[0] - 1))
        return (np.array([0.0], dtype=float), np.array([float(image[iy, ix])], dtype=float))
    n = max(2, int(np.ceil(length * 4.0)) + 1)
    dist = np.linspace(0.0, length, n, dtype=float)
    frac = dist / length
    xs = x0 + (x1 - x0) * frac
    ys = y0 + (y1 - y0) * frac
    profile = map_coordinates(np.asarray(image, dtype=float), [ys, xs], order=1, mode='nearest')
    return (dist, np.asarray(profile, dtype=float))

def _line_profile_extrema_stats_optimized(self, dist, profile):
    from scipy.signal import find_peaks
    dist = np.asarray(dist, dtype=float)
    profile = np.asarray(profile, dtype=float)
    if dist.size < 3 or profile.size != dist.size:
        return {'min_x': np.array([], dtype=float), 'max_x': np.array([], dtype=float), 'minmin': (np.nan, np.nan, 0), 'maxmax': (np.nan, np.nan, 0), 'minmax': (np.nan, np.nan, 0), 'all': (np.nan, np.nan, 0)}
    try:
        min_distance_px = max(1.0, float(self.line_profile_min_distance_var.get()))
    except Exception:
        min_distance_px = 3.0
    try:
        prom = max(0.0, min(1.0, float(self.line_profile_prominence_var.get())))
    except Exception:
        prom = 0.03
    steps = np.diff(dist)
    step_px = float(np.median(steps[steps > 0])) if np.any(steps > 0) else 1.0
    sample_distance = max(1, int(np.ceil(min_distance_px / max(step_px, 1e-12))))
    max_idx, _ = find_peaks(profile, distance=sample_distance, prominence=prom)
    min_idx, _ = find_peaks(-profile, distance=sample_distance, prominence=prom)

    def refine(idx):
        refined = []
        for j in np.asarray(idx, dtype=int):
            if j <= 0 or j >= len(profile) - 1:
                refined.append(float(dist[j]))
                continue
            y1, y2, y3 = (float(profile[j - 1]), float(profile[j]), float(profile[j + 1]))
            den = y1 - 2.0 * y2 + y3
            if abs(den) < 1e-14:
                delta = 0.0
            else:
                delta = 0.5 * (y1 - y3) / den
                delta = float(np.clip(delta, -1.0, 1.0))
            refined.append(float(dist[j] + delta * step_px))
        return np.asarray(refined, dtype=float)
    min_x = refine(min_idx)
    max_x = refine(max_idx)

    def diffs(a):
        return np.abs(np.diff(np.asarray(a, dtype=float))) if len(a) >= 2 else np.array([], dtype=float)
    mm = diffs(min_x)
    xx = diffs(max_x)
    events = [(float(x), 'min') for x in min_x] + [(float(x), 'max') for x in max_x]
    events.sort(key=lambda q: q[0])
    mixed = []
    all_d = []
    for a, b in zip(events[:-1], events[1:]):
        d = abs(b[0] - a[0])
        all_d.append(d)
        if a[1] != b[1]:
            mixed.append(d)
    all_d = np.asarray(all_d, dtype=float)
    mixed = np.asarray(mixed, dtype=float)

    def stat(a):
        a = np.asarray(a, dtype=float)
        if a.size == 0:
            return (np.nan, np.nan, 0)
        err = float(np.std(a, ddof=1)) if a.size > 1 else 0.0
        return (float(np.mean(a)), err, int(a.size))
    return {'min_x': min_x, 'max_x': max_x, 'minmin': stat(mm), 'maxmax': stat(xx), 'minmax': stat(mixed), 'all': stat(all_d)}

def _line_profile_refresh_measurement_values(self):
    """Keep selected measurement x values, but recompute y on the current frame."""
    pts = list(getattr(self, '_line_profile_measure_points', []) or [])
    if not pts:
        return
    refreshed = []
    results = list(getattr(self, '_line_profile_results', []) or [])
    for x, _old_y, setno in pts[:2]:
        rr = next((r for r in results if int(r.get('set', -1)) == int(setno)), None)
        if rr is None:
            continue
        dist = np.asarray(rr.get('dist', []), dtype=float)
        prof = np.asarray(rr.get('profile', []), dtype=float)
        if dist.size == 0 or prof.size != dist.size:
            continue
        xx = float(np.clip(x, float(dist[0]), float(dist[-1])))
        yy = float(np.interp(xx, dist, prof))
        refreshed.append((xx, yy, int(setno)))
    self._line_profile_measure_points = refreshed
    if len(refreshed) < 2 and getattr(self, '_line_profile_measure_mode', False):
        self._line_profile_measure_mode = True

def _line_profile_project_to_curve(self, event_x, event_y, set_hint=None, max_px=90.0):
    """Return the closest continuous position on any plotted profile curve.

    The returned x/y are interpolated along a curve segment, rather than being
    snapped to the nearest sampled point.
    """
    results = list(getattr(self, '_line_profile_results', []) or [])
    if set_hint is not None:
        hinted = [r for r in results if int(r.get('set', -1)) == int(set_hint)]
        if hinted:
            results = hinted
    if event_x is None or event_y is None or (not results):
        return None
    ax = self.line_profile_ax_graph
    target = ax.transData.transform((float(event_x), float(event_y)))
    best = None
    for r in results:
        dist = np.asarray(r.get('dist', []), dtype=float)
        prof = np.asarray(r.get('profile', []), dtype=float)
        if dist.size < 2 or prof.size != dist.size:
            continue
        screen = ax.transData.transform(np.column_stack((dist, prof)))
        a = screen[:-1]
        b = screen[1:]
        v = b - a
        vv = np.einsum('ij,ij->i', v, v)
        w = target - a
        with np.errstate(divide='ignore', invalid='ignore'):
            t = np.einsum('ij,ij->i', w, v) / np.where(vv > 0, vv, 1.0)
        t = np.clip(t, 0.0, 1.0)
        q = a + v * t[:, None]
        d2 = np.einsum('ij,ij->i', q - target, q - target)
        j = int(np.argmin(d2))
        dpx = float(np.sqrt(max(d2[j], 0.0)))
        if best is None or dpx < best[0]:
            x0, x1 = (float(dist[j]), float(dist[j + 1]))
            y0, y1 = (float(prof[j]), float(prof[j + 1]))
            tt = float(t[j])
            xx = x0 + tt * (x1 - x0)
            yy = y0 + tt * (y1 - y0)
            best = (dpx, xx, yy, int(r.get('set', 1)))
    if best is None or best[0] > float(max_px):
        return None
    return best

def _line_profile_graph_press_optimized(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.button not in (None, 1):
        return
    pts = list(getattr(self, '_line_profile_measure_points', []) or [])
    if len(pts) == 2 and event.x is not None and (event.y is not None):
        best_i, best_d = (None, float('inf'))
        for i, (px, py, _setno) in enumerate(pts):
            try:
                sx, sy = self.line_profile_ax_graph.transData.transform((float(px), float(py)))
                dd = float(np.hypot(float(event.x) - sx, float(event.y) - sy))
            except Exception:
                continue
            if dd < best_d:
                best_d, best_i = (dd, i)
        if best_i is not None and best_d <= 28.0:
            self._line_profile_measure_drag_index = best_i
            self._line_profile_measure_mode = False
            self.line_profile_measure_var.set(f'Dragging M{best_i + 1}: move along its profile; release to keep the sub-pixel position.')
            return
    if not getattr(self, '_line_profile_measure_mode', False):
        return
    if event.xdata is None or event.ydata is None:
        return
    best = self._line_profile_project_to_curve(event.xdata, event.ydata, max_px=75.0)
    if best is None:
        self.line_profile_measure_var.set('No curve found near the click. Click directly on or close to a profile.')
        return
    dpx, x, y, setno = best
    self._line_profile_measure_points.append((float(x), float(y), int(setno)))
    if len(self._line_profile_measure_points) >= 2:
        self._line_profile_measure_points = self._line_profile_measure_points[:2]
        self._line_profile_measure_mode = False
        try:
            self.line_profile_canvas_graph.get_tk_widget().configure(cursor='')
        except Exception:
            pass
    self._line_profile_update_measurement_status()
    self._line_profile_draw_measurement()

def _line_profile_graph_motion_optimized(self, event):
    i = getattr(self, '_line_profile_measure_drag_index', None)
    if i is None or event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    pts = list(getattr(self, '_line_profile_measure_points', []) or [])
    if not pts or i >= len(pts) or event.xdata is None:
        return
    target_set = int(pts[i][2])
    r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == target_set), None)
    if r is None:
        return
    dist = np.asarray(r.get('dist', []), dtype=float)
    prof = np.asarray(r.get('profile', []), dtype=float)
    if dist.size == 0 or prof.size != dist.size:
        return
    xx = float(np.clip(event.xdata, float(dist[0]), float(dist[-1])))
    yy = float(np.interp(xx, dist, prof))
    self._line_profile_measure_points[i] = (xx, yy, target_set)
    self._line_profile_update_measurement_status()
    self._line_profile_draw_measurement()

def _line_profile_graph_release_optimized(self, event):
    if getattr(self, '_line_profile_measure_drag_index', None) is not None:
        self._line_profile_update_measurement_status()
    self._line_profile_measure_drag_index = None
CDIWorkflowApp._line_profile_extract = _line_profile_extract_optimized
CDIWorkflowApp._line_profile_extrema_stats = _line_profile_extrema_stats_optimized
CDIWorkflowApp._line_profile_project_to_curve = _line_profile_project_to_curve
CDIWorkflowApp._line_profile_refresh_measurement_values = _line_profile_refresh_measurement_values
CDIWorkflowApp._line_profile_graph_press = _line_profile_graph_press_optimized
CDIWorkflowApp._line_profile_graph_motion = _line_profile_graph_motion_optimized
CDIWorkflowApp._line_profile_graph_release = _line_profile_graph_release_optimized
_old_redraw_profiles_optimized = CDIWorkflowApp._line_profile_redraw_profiles

def _line_profile_redraw_profiles_measurement_safe(self, idx=None):
    result = _old_redraw_profiles_optimized(self, idx=idx)
    try:
        self._line_profile_refresh_measurement_values()
        self._line_profile_update_measurement_status()
        self._line_profile_draw_measurement()
    except Exception:
        pass
    return result
CDIWorkflowApp._line_profile_redraw_profiles = _line_profile_redraw_profiles_measurement_safe

def _line_profile_frame_mousewheel(self, event):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        return 'break'
    n = len(stack)
    current = int(self.line_profile_frame_var.get())
    if getattr(event, 'num', None) == 4:
        step = 1
    elif getattr(event, 'num', None) == 5:
        step = -1
    else:
        step = 1 if getattr(event, 'delta', 0) > 0 else -1
    new_idx = max(0, min(current + step, n - 1))
    if new_idx != current:
        self.line_profile_frame_var.set(new_idx)
        self._line_profile_frame_slider_changed(new_idx)
    return 'break'

def _line_profile_plot_scroll(self, event):
    return self._line_profile_frame_mousewheel(event)

def _line_profile_frame_slider_changed(self, value=None):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        return
    try:
        idx = int(round(float(value if value is not None else self.line_profile_frame_var.get())))
    except Exception:
        idx = int(self.line_profile_frame_var.get())
    idx = max(0, min(idx, len(stack) - 1))
    self.line_profile_frame_var.set(idx)
    try:
        self.roi2_idx_var.set(idx)
    except Exception:
        pass
    self._line_profile_update_image(idx, preserve_lines=True)
def _line_profile_sync_from_secondary_roi_refined(self):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        if hasattr(self, 'line_profile_frame_slider'):
            self.line_profile_frame_slider.configure(from_=0, to=0, state='disabled')
        if hasattr(self, 'line_profile_frame_label'):
            self.line_profile_frame_label.config(text='--')
        self.line_profile_status_var.set('Waiting for a confirmed Secondary ROI. Select and confirm it first.')
        return
    try:
        idx = int(round(float(self.roi2_idx_var.get())))
    except Exception:
        idx = 0
    idx = max(0, min(idx, len(stack) - 1))
    self.line_profile_frame_var.set(idx)
    self.line_profile_frame_slider.configure(from_=0, to=max(0, len(stack) - 1), state='normal')
    self._line_profile_update_image(idx, preserve_lines=True)

def _line_profile_start_measurement_refined(self):
    if not getattr(self, '_line_profile_results', None):
        self.line_profile_measure_var.set('No profile data available. Draw the line sets first.')
        return
    self._line_profile_measure_mode = True
    self._line_profile_measure_points = []
    self._line_profile_measure_drag_index = None
    self._line_profile_measure_drag_set = None
    self._line_profile_measure_drag_point = None
    pending = len(self._line_profile_measure_measurements)
    total = len(self._line_profile_results)
    try:
        self.line_profile_canvas_graph.get_tk_widget().configure(cursor='crosshair')
    except Exception:
        pass
    self.line_profile_measure_var.set(f'DYNAMIC SCALE ACTIVE — measure {total} line profile(s). {pending} line(s) already complete. Click two points on the SAME profile to measure one line, then continue with the next line.')

def _line_profile_clear_measurement_refined(self, redraw=True):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    for a in getattr(self, '_line_profile_measure_image_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    self._line_profile_measure_image_artists = []
    self._line_profile_measure_measurements = {}
    self._line_profile_measure_points = []
    self._line_profile_measure_mode = False
    self._line_profile_measure_drag_index = None
    self._line_profile_measure_drag_set = None
    self._line_profile_measure_drag_point = None
    try:
        self.line_profile_canvas_graph.get_tk_widget().configure(cursor='')
    except Exception:
        pass
    if hasattr(self, 'line_profile_measure_var'):
        self.line_profile_measure_var.set('All measurements cleared. Click DYNAMIC MEASURING SCALE to measure multiple profiles.')
    if redraw:
        self._line_profile_redraw_image_lines()
        if hasattr(self, 'line_profile_canvas_graph'):
            self.line_profile_canvas_graph.draw_idle()

def _line_profile_measurement_from_x(self, setno, x):
    r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == int(setno)), None)
    if r is None:
        return None
    dist = np.asarray(r.get('dist', []), dtype=float)
    prof = np.asarray(r.get('profile', []), dtype=float)
    if dist.size == 0 or prof.size != dist.size:
        return None
    xx = float(np.clip(x, float(dist[0]), float(dist[-1])))
    yy = float(np.interp(xx, dist, prof))
    return (xx, yy, int(setno))

def _line_profile_refresh_measurement_values_refined(self):
    new = {}
    for setno, pts in getattr(self, '_line_profile_measure_measurements', {}).items():
        refreshed = []
        for pt in pts[:2]:
            v = self._line_profile_measurement_from_x(int(setno), float(pt[0]))
            if v is not None:
                refreshed.append(v)
        if refreshed:
            new[int(setno)] = refreshed
    self._line_profile_measure_measurements = new
    self._line_profile_measure_points = []

def _line_profile_find_profile_point_refined(self, x, y=None, set_hint=None, max_px=75.0):
    return self._line_profile_project_to_curve(x, y, set_hint=set_hint, max_px=max_px)

def _line_profile_select_measurement_point_refined(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.xdata is None or event.ydata is None:
        return
    best = self._line_profile_project_to_curve(event.xdata, event.ydata, max_px=75.0)
    if best is None:
        self.line_profile_measure_var.set('No profile close enough. Click on/near one of the curves.')
        return
    _screen_dist, x, y, setno = best
    setno = int(setno)
    if self._line_profile_measure_points:
        first = self._line_profile_measure_points[0]
        if int(first[2]) != setno:
            self.line_profile_measure_var.set(f'First point is on Set {first[2]}. Click the second point on Set {first[2]}, or click DYNAMIC MEASURING SCALE again to restart.')
            return
    self._line_profile_measure_points.append((float(x), float(y), setno))
    if len(self._line_profile_measure_points) == 2:
        pair = list(self._line_profile_measure_points[:2])
        self._line_profile_measure_measurements[setno] = pair
        self._line_profile_measure_points = []
        completed = len(self._line_profile_measure_measurements)
        total = len(self._line_profile_results)
        self.line_profile_measure_var.set(f'Set {setno} measured: ΔL = {abs(pair[1][0] - pair[0][0]):.2f} px. Completed {completed}/{total} line set(s). Continue clicking two points on another profile.')
    else:
        self.line_profile_measure_var.set(f'Set {setno}: first point selected at {x:.2f} px. Click the second point on the SAME profile.')
    self._line_profile_update_measurement_table()
    self._line_profile_draw_measurement()

def _line_profile_graph_press_refined(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.button not in (None, 1):
        return
    measurements = getattr(self, '_line_profile_measure_measurements', {})
    if event.x is not None and event.y is not None:
        best = None
        for setno, pts in measurements.items():
            for point_index, (px, py, _sn) in enumerate(pts[:2]):
                try:
                    sx, sy = self.line_profile_ax_graph.transData.transform((float(px), float(py)))
                    d = float(np.hypot(event.x - sx, event.y - sy))
                except Exception:
                    continue
                if best is None or d < best[0]:
                    best = (d, int(setno), point_index)
        if best is not None and best[0] <= 26.0:
            self._line_profile_measure_drag_set = best[1]
            self._line_profile_measure_drag_point = best[2]
            self._line_profile_measure_drag_index = best[2]
            self._line_profile_measure_mode = False
            self.line_profile_measure_var.set(f'Dragging Set {best[1]} M{best[2] + 1}. Release to keep the new position.')
            return
    if self._line_profile_measure_mode:
        self._line_profile_select_measurement_point_refined(event)

def _line_profile_graph_motion_refined(self, event):
    setno = getattr(self, '_line_profile_measure_drag_set', None)
    point_index = getattr(self, '_line_profile_measure_drag_point', None)
    if setno is None or point_index is None:
        return
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.xdata is None:
        return
    v = self._line_profile_measurement_from_x(setno, event.xdata)
    if v is None:
        return
    pts = list(self._line_profile_measure_measurements.get(setno, []))
    if len(pts) != 2:
        return
    pts[point_index] = v
    self._line_profile_measure_measurements[setno] = pts
    self._line_profile_update_measurement_table()
    self._line_profile_draw_measurement()

def _line_profile_graph_release_refined(self, event):
    if getattr(self, '_line_profile_measure_drag_set', None) is not None:
        setno = self._line_profile_measure_drag_set
        self.line_profile_measure_var.set(f'Set {setno} measurement updated.')
    self._line_profile_measure_drag_index = None
    self._line_profile_measure_drag_set = None
    self._line_profile_measure_drag_point = None

def _line_profile_redraw_image_lines_refined(self):
    for artist in getattr(self, '_line_profile_image_line_artists', []):
        try:
            artist.remove()
        except Exception:
            pass
    self._line_profile_image_line_artists = []
    for i, line in enumerate(getattr(self, '_line_profile_lines', [])):
        c = self._line_profile_line_color(i)
        a, = self.line_profile_ax_image.plot([line['p0'][0], line['p1'][0]], [line['p0'][1], line['p1'][1]], '-', lw=2.2, color=c, label=f'Set {i + 1}')
        self._line_profile_image_line_artists.append(a)
    for setno, pts in sorted(getattr(self, '_line_profile_measure_measurements', {}).items()):
        if len(pts) != 2:
            continue
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == int(setno)), None)
        if r is None:
            continue
        try:
            L = float(r.get('length', 0.0))
            if L <= 0:
                continue
            xy = []
            for x, _y, _sn in pts:
                frac = float(np.clip(float(x) / L, 0.0, 1.0))
                p0 = np.asarray(r['p0'], dtype=float)
                p1 = np.asarray(r['p1'], dtype=float)
                pos = p0 + frac * (p1 - p0)
                xy.append(pos)
            p0, p1 = xy
            under, = self.line_profile_ax_image.plot([p0[0], p1[0]], [p0[1], p1[1]], '-', lw=7.0, color='black', alpha=0.8, zorder=24)
            measure, = self.line_profile_ax_image.plot([p0[0], p1[0]], [p0[1], p1[1]], '-', lw=4.0, color='yellow', zorder=25)
            self._line_profile_measure_image_artists.extend([under, measure])
            mid = (p0 + p1) / 2.0
            label = self.line_profile_ax_image.text(float(mid[0]), float(mid[1]), f'ΔL={abs(float(pts[1][0]) - float(pts[0][0])):.2f} px', color='yellow', fontsize=9, weight='bold', ha='center', va='bottom', zorder=26, bbox=dict(boxstyle='round,pad=0.15', facecolor='black', edgecolor='yellow', alpha=0.78))
            self._line_profile_measure_image_artists.append(label)
        except Exception:
            pass
    self.line_profile_canvas_image.draw_idle()

def _line_profile_draw_measurement_refined(self):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    for setno, pts in sorted(getattr(self, '_line_profile_measure_measurements', {}).items()):
        if len(pts) != 2:
            continue
        x0, y0, _ = pts[0]
        x1, y1, _ = pts[1]
        for x, y, _ in pts:
            self._line_profile_measure_artists.append(self.line_profile_ax_graph.scatter([x], [y], s=34, facecolor='yellow', edgecolor='black', linewidths=0.9, zorder=9))
        self._line_profile_measure_artists.extend([self.line_profile_ax_graph.axvline(x0, color='yellow', ls='--', lw=1.0, alpha=0.65, zorder=5), self.line_profile_ax_graph.axvline(x1, color='yellow', ls='--', lw=1.0, alpha=0.65, zorder=5)])
        ymin, ymax = self.line_profile_ax_graph.get_ylim()
        ybar = ymax - 0.07 * max(ymax - ymin, 1e-09)
        self._line_profile_measure_artists.append(self.line_profile_ax_graph.annotate('', xy=(x1, ybar), xytext=(x0, ybar), arrowprops=dict(arrowstyle='<->', color='yellow', lw=1.6)))
    if len(self._line_profile_measure_points) == 1:
        x, y, _ = self._line_profile_measure_points[0]
        self._line_profile_measure_artists.extend([self.line_profile_ax_graph.scatter([x], [y], s=40, facecolor='yellow', edgecolor='black', linewidths=1.0, zorder=10), self.line_profile_ax_graph.axvline(x, color='yellow', ls=':', lw=1.2, alpha=0.8)])
    self._line_profile_redraw_image_lines_refined()
    self._line_profile_measure_artists = [*getattr(self, '_line_profile_measure_artists', [])]
    if hasattr(self, 'line_profile_canvas_graph'):
        self.line_profile_canvas_graph.draw_idle()

def _line_profile_update_measurement_table(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    for item in tree.get_children():
        tree.delete(item)
    result_by_set = {int(r.get('set', -1)): r for r in getattr(self, '_line_profile_results', [])}
    measurements = getattr(self, '_line_profile_measure_measurements', {})
    for setno, r in sorted(result_by_set.items()):
        length = float(r.get('length', 0.0))
        pts = measurements.get(setno, [])
        if len(pts) == 2:
            m1x, m1y = (pts[0][0], pts[0][1])
            m2x, m2y = (pts[1][0], pts[1][1])
            delta = abs(float(m2x) - float(m1x))
            status = 'Measured'
            vals = (setno, f'{length:.2f}', f'{m1x:.2f}', f'{m2x:.2f}', f'{delta:.2f}', f'{m1y:.4f}', f'{m2y:.4f}', status)
        else:
            status = 'Pending'
            vals = (setno, f'{length:.2f}', '—', '—', '—', '—', '—', status)
        tree.insert('', 'end', values=vals)

def _line_profile_update_table_refined(self):
    self._line_profile_update_measurement_table()

def _line_profile_recalculate_all_refined(self):
    self.line_profile_status_var.set('Extrema analysis has been removed from Tab 6. Measurements are now handled per line set.')

def _line_profile_export_results_refined(self):
    results = getattr(self, '_line_profile_results', [])
    if not results:
        messagebox.showwarning('Line Profile', 'Draw at least one line first.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Export dynamic measurements', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    try:
        idx = int(round(float(self.line_profile_frame_var.get())))
    except Exception:
        idx = 0
    rows = []
    measurements = getattr(self, '_line_profile_measure_measurements', {})
    for r in results:
        setno = int(r.get('set', -1))
        pts = measurements.get(setno, [])
        row = {'set': setno, 'frame': idx + 1, 'field_mT': float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else '', 'line_length_px': float(r.get('length', 0.0)), 'M1_position_px': '', 'M2_position_px': '', 'DeltaL_px': '', 'M1_intensity': '', 'M2_intensity': '', 'status': 'Pending'}
        if len(pts) == 2:
            row.update({'M1_position_px': float(pts[0][0]), 'M2_position_px': float(pts[1][0]), 'DeltaL_px': abs(float(pts[1][0]) - float(pts[0][0])), 'M1_intensity': float(pts[0][1]), 'M2_intensity': float(pts[1][1]), 'status': 'Measured'})
        rows.append(row)
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} line-set measurements: {fp}')

def _line_profile_redraw_profiles_refined(self, idx=None):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        self._line_profile_results = []
        self._line_profile_redraw_image_lines_refined()
        self._line_profile_update_measurement_table()
        return
    if idx is None:
        try:
            idx = int(round(float(self.line_profile_frame_var.get())))
        except Exception:
            try:
                idx = int(round(float(self.roi2_idx_var.get())))
            except Exception:
                idx = 0
    idx = max(0, min(int(idx), len(stack) - 1))
    self.line_profile_frame_var.set(idx)
    image = np.asarray(stack[idx], dtype=float)
    self.line_profile_ax_graph.clear()
    self.line_profile_ax_graph.set_xlabel('Length (pixels)')
    self.line_profile_ax_graph.set_ylabel('Intensity')
    self.line_profile_ax_graph.set_title(f'Secondary ROI line profiles — Frame {idx + 1}/{len(stack)}')
    self.line_profile_ax_graph.grid(True, alpha=0.22)
    self._line_profile_results = []
    self._line_profile_graph_artists = []
    for i, line in enumerate(self._line_profile_lines):
        dist, profile = self._line_profile_extract(image, line['p0'], line['p1'])
        result = {'set': i + 1, 'p0': line['p0'], 'p1': line['p1'], 'length': float(dist[-1]) if len(dist) else 0.0, 'dist': dist, 'profile': profile}
        self._line_profile_results.append(result)
        ln, = self.line_profile_ax_graph.plot(dist, profile, lw=2.0, color=self._line_profile_line_color(i), label=f'Set {i + 1}', picker=14, pickradius=14, zorder=4)
        ln._line_profile_setno = i + 1
        self._line_profile_graph_artists.append(ln)
    if self._line_profile_results:
        self.line_profile_ax_graph.legend(loc='best')
    self._line_profile_current_profile = self._line_profile_results[-1] if self._line_profile_results else None
    self._line_profile_refresh_measurement_values()
    self._line_profile_redraw_image_lines_refined()
    self._line_profile_update_measurement_table()
    self.line_profile_canvas_graph.draw_idle()
CDIWorkflowApp._line_profile_frame_slider_changed = _line_profile_frame_slider_changed
CDIWorkflowApp._line_profile_frame_mousewheel = _line_profile_frame_mousewheel
CDIWorkflowApp._line_profile_plot_scroll = _line_profile_plot_scroll
CDIWorkflowApp._line_profile_sync_from_secondary_roi = _line_profile_sync_from_secondary_roi_refined
CDIWorkflowApp._line_profile_start_measurement = _line_profile_start_measurement_refined
CDIWorkflowApp._line_profile_clear_measurement = _line_profile_clear_measurement_refined
CDIWorkflowApp._line_profile_refresh_measurement_values = _line_profile_refresh_measurement_values_refined
CDIWorkflowApp._line_profile_find_profile_point = _line_profile_find_profile_point_refined
CDIWorkflowApp._line_profile_graph_press = _line_profile_graph_press_refined
CDIWorkflowApp._line_profile_graph_motion = _line_profile_graph_motion_refined
CDIWorkflowApp._line_profile_graph_release = _line_profile_graph_release_refined
CDIWorkflowApp._line_profile_redraw_image_lines = _line_profile_redraw_image_lines_refined
CDIWorkflowApp._line_profile_redraw_image_lines_refined = _line_profile_redraw_image_lines_refined
CDIWorkflowApp._line_profile_draw_measurement = _line_profile_draw_measurement_refined
CDIWorkflowApp._line_profile_update_table = _line_profile_update_table_refined
CDIWorkflowApp._line_profile_update_measurement_table = _line_profile_update_measurement_table
CDIWorkflowApp._line_profile_recalculate_all = _line_profile_recalculate_all_refined
CDIWorkflowApp._line_profile_export_results = _line_profile_export_results_refined
CDIWorkflowApp._line_profile_redraw_profiles = _line_profile_redraw_profiles_refined

def _line_profile_graph_pick_refined(self, event):
    return

def _line_profile_set_count_changed_refined(self):
    try:
        target = max(1, min(20, int(self.line_profile_sets_var.get())))
    except Exception:
        target = 1
    self.line_profile_sets_var.set(target)
    if len(self._line_profile_lines) > target:
        self._line_profile_lines = self._line_profile_lines[:target]
        self._line_profile_measure_measurements = {k: v for k, v in self._line_profile_measure_measurements.items() if k <= target}
        self._line_profile_redraw_profiles()
    self.line_profile_status_var.set(f'Number of line sets = {target}. Press DRAW LINE SETS to add the remaining sets.')

def _line_profile_redo_last_refined(self):
    if not self._line_profile_lines:
        self.line_profile_status_var.set('No line to redo.')
        return
    removed_set = len(self._line_profile_lines)
    self._line_profile_lines.pop()
    self._line_profile_measure_measurements.pop(removed_set, None)
    self._line_profile_active = True
    self._line_profile_redraw_profiles()
    self.line_profile_status_var.set(f'Redraw Set {len(self._line_profile_lines) + 1}/{self.line_profile_sets_var.get()}.')

def _line_profile_clear_line_refined(self, index=None):
    if not self._line_profile_lines:
        return
    if index is None:
        index = len(self._line_profile_lines) - 1
    try:
        index = int(index)
    except Exception:
        index = len(self._line_profile_lines) - 1
    if 0 <= index < len(self._line_profile_lines):
        self._line_profile_lines.pop(index)
    old_measurements = dict(self._line_profile_measure_measurements)
    self._line_profile_measure_measurements = {}
    for i, line in enumerate(self._line_profile_lines):
        old_set = i + 1 if i < index else i + 2
        line['set'] = i + 1
        if old_set in old_measurements:
            self._line_profile_measure_measurements[i + 1] = [(float(p[0]), float(p[1]), i + 1) for p in old_measurements[old_set][:2]]
    self._line_profile_active = len(self._line_profile_lines) < int(self.line_profile_sets_var.get())
    self._line_profile_redraw_profiles()
CDIWorkflowApp._line_profile_graph_pick = _line_profile_graph_pick_refined
CDIWorkflowApp._line_profile_select_measurement_point_refined = _line_profile_select_measurement_point_refined
CDIWorkflowApp._line_profile_measurement_from_x = _line_profile_measurement_from_x
CDIWorkflowApp._line_profile_set_count_changed = _line_profile_set_count_changed_refined
CDIWorkflowApp._line_profile_redo_last = _line_profile_redo_last_refined
CDIWorkflowApp._line_profile_clear_line = _line_profile_clear_line_refined

def _noop_removed_tab_builder(self, *args, **kwargs):
    return None
for _removed_builder_name in ('_build_line_profile_tab', '_build_particle_density_tab', '_build_tab8_export_folder_controls'):
    if not hasattr(CDIWorkflowApp, _removed_builder_name):
        setattr(CDIWorkflowApp, _removed_builder_name, _noop_removed_tab_builder)

def _tab6_init_dynamic_pair_state(self):
    self._line_profile_measurements_list = []
    self._line_profile_measure_next_point = 1
    self._line_profile_pending_point = None
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    self._line_profile_measure_mode = False

def _tab6_build_line_profile_wrapper(self):
    _TAB6_ORIGINAL_BUILDER(self)
    _tab6_init_dynamic_pair_state(self)
    tree = getattr(self, 'line_profile_tree', None)
    if tree is not None:
        columns = ('Pair', 'Set', 'Line length (px)', 'M1 position (px)', 'M2 position (px)', 'ΔL (px)', 'M1 intensity', 'M2 intensity', 'Status')
        tree.configure(columns=columns, displaycolumns=columns)
        widths = (60, 55, 105, 105, 105, 85, 100, 100, 95)
        for col, wid in zip(columns, widths):
            tree.heading(col, text=col)
            tree.column(col, width=wid, anchor='center')
    if hasattr(self, 'line_profile_measure_var'):
        self.line_profile_measure_var.set('Click DYNAMIC MEASURING SCALE, then click two points on any one profile. After M1–M2 is complete, immediately select M3–M4 on any profile, then M5–M6, ...')

def _tab6_sync_legacy_measurement_dict(self):
    d = {}
    for m in getattr(self, '_line_profile_measurements_list', []):
        d[int(m['pair_id'])] = list(m['pts'])
    self._line_profile_measure_measurements_by_pair = d

def _tab6_start_dynamic_measurement(self):
    if not getattr(self, '_line_profile_results', None):
        self.line_profile_measure_var.set('Draw the line profiles first.')
        return
    self._line_profile_measure_mode = True
    self._line_profile_pending_point = None
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    next_id = len(getattr(self, '_line_profile_measurements_list', [])) * 2 + 1
    self._line_profile_measure_next_point = next_id
    try:
        self.line_profile_canvas_graph.get_tk_widget().configure(cursor='crosshair')
    except Exception:
        pass
    self.line_profile_measure_var.set(f'DYNAMIC SCALE ACTIVE — click M{next_id} and M{next_id + 1} on the same profile. When that pair is complete, the next pair can be selected on ANY profile.')

def _tab6_clear_dynamic_measurement(self, redraw=True):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    for a in getattr(self, '_line_profile_measure_image_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_image_artists = []
    _tab6_init_dynamic_pair_state(self)
    _tab6_sync_legacy_measurement_dict(self)
    try:
        self.line_profile_canvas_graph.get_tk_widget().configure(cursor='')
    except Exception:
        pass
    self.line_profile_measure_var.set('All measurements cleared. Click DYNAMIC MEASURING SCALE to start M1–M2, M3–M4, ...')
    if redraw:
        self._line_profile_redraw_image_lines()
        self.line_profile_canvas_graph.draw_idle()

def _tab6_measurement_from_x(self, setno, x):
    r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == int(setno)), None)
    if r is None:
        return None
    dist = np.asarray(r.get('dist', []), dtype=float)
    prof = np.asarray(r.get('profile', []), dtype=float)
    if dist.size == 0 or prof.size != dist.size:
        return None
    xx = float(np.clip(x, float(dist[0]), float(dist[-1])))
    yy = float(np.interp(xx, dist, prof))
    return (xx, yy, int(setno))

def _tab6_select_pair_point(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.xdata is None or event.ydata is None:
        return
    best = self._line_profile_project_to_curve(event.xdata, event.ydata, max_px=85.0)
    if best is None:
        self.line_profile_measure_var.set('No profile found near the cursor. Click directly on a profile curve.')
        return
    _dpx, x, y, setno = best
    setno = int(setno)
    pending = getattr(self, '_line_profile_pending_point', None)
    pair_no = len(getattr(self, '_line_profile_measurements_list', [])) + 1
    m1_id = 2 * pair_no - 1
    m2_id = 2 * pair_no
    if pending is None:
        self._line_profile_pending_point = (float(x), float(y), setno)
        self._line_profile_measure_next_point = m2_id
        self._line_profile_measure_mode = True
        self.line_profile_measure_var.set(f'M{m1_id} selected on Set {setno} at {x:.2f} px. Click M{m2_id} on the SAME profile (Set {setno}).')
    else:
        if int(pending[2]) != setno:
            self.line_profile_measure_var.set(f'M{m1_id} is on Set {pending[2]}. Click M{m2_id} on that same Set, or click the currently active pair again to choose a new M1.')
            return
        pts = [pending, (float(x), float(y), setno)]
        self._line_profile_measurements_list.append({'pair_id': pair_no, 'm1_id': m1_id, 'm2_id': m2_id, 'setno': setno, 'pts': pts})
        self._line_profile_pending_point = None
        _tab6_sync_legacy_measurement_dict(self)
        self._line_profile_update_measurement_table()
        self._line_profile_draw_measurement()
        next_pair = pair_no + 1
        next_m1 = 2 * next_pair - 1
        next_m2 = 2 * next_pair
        total = len(getattr(self, '_line_profile_results', []))
        self._line_profile_measure_mode = True
        self._line_profile_measure_next_point = next_m1
        self.line_profile_measure_var.set(f'M{m1_id}–M{m2_id} complete: Set {setno}, ΔL = {abs(pts[1][0] - pts[0][0]):.2f} px. Now click M{next_m1} and M{next_m2} on ANY of the {total} profile(s).')

def _tab6_graph_press(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.button not in (None, 1):
        return
    best = None
    for pi, m in enumerate(getattr(self, '_line_profile_measurements_list', [])):
        for point_index, (px, py, _sn) in enumerate(m['pts'][:2]):
            try:
                sx, sy = self.line_profile_ax_graph.transData.transform((float(px), float(py)))
                d = float(np.hypot(event.x - sx, event.y - sy))
            except Exception:
                continue
            if best is None or d < best[0]:
                best = (d, pi, point_index)
    if best is not None and best[0] <= 24.0:
        self._line_profile_measure_drag_pair = best[1]
        self._line_profile_measure_drag_point = best[2]
        self._line_profile_measure_mode = False
        pair = self._line_profile_measurements_list[best[1]]
        mid = pair['m1_id'] if best[2] == 0 else pair['m2_id']
        self.line_profile_measure_var.set(f"Dragging M{mid} of pair M{pair['m1_id']}–M{pair['m2_id']}. Release to keep the new position.")
        return
    if getattr(self, '_line_profile_measure_mode', False):
        _tab6_select_pair_point(self, event)

def _tab6_graph_motion(self, event):
    pi = getattr(self, '_line_profile_measure_drag_pair', None)
    point_index = getattr(self, '_line_profile_measure_drag_point', None)
    if pi is None or point_index is None:
        return
    if event is None or event.inaxes is not self.line_profile_ax_graph or event.xdata is None:
        return
    measurements = getattr(self, '_line_profile_measurements_list', [])
    if not 0 <= int(pi) < len(measurements):
        return
    m = measurements[int(pi)]
    v = _tab6_measurement_from_x(self, int(m['setno']), event.xdata)
    if v is None:
        return
    m['pts'][point_index] = v
    _tab6_sync_legacy_measurement_dict(self)
    self._line_profile_update_measurement_table()
    self._line_profile_draw_measurement()

def _tab6_graph_release(self, event):
    pi = getattr(self, '_line_profile_measure_drag_pair', None)
    if pi is not None and 0 <= int(pi) < len(getattr(self, '_line_profile_measurements_list', [])):
        m = self._line_profile_measurements_list[int(pi)]
        self.line_profile_measure_var.set(f"Pair M{m['m1_id']}–M{m['m2_id']} updated. ΔL = {abs(m['pts'][1][0] - m['pts'][0][0]):.2f} px. Ready for the next measurement pair.")
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    if getattr(self, '_line_profile_measurements_list', None):
        self._line_profile_measure_mode = True

def _tab6_redraw_image_lines(self):
    for a in getattr(self, '_line_profile_image_line_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_image_line_artists = []
    for a in getattr(self, '_line_profile_measure_image_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_image_artists = []
    for i, line in enumerate(getattr(self, '_line_profile_lines', [])):
        c = self._line_profile_line_color(i)
        a, = self.line_profile_ax_image.plot([line['p0'][0], line['p1'][0]], [line['p0'][1], line['p1'][1]], '-', lw=2.0, color=c, zorder=12)
        self._line_profile_image_line_artists.append(a)
    for m in getattr(self, '_line_profile_measurements_list', []):
        pts = m.get('pts', [])
        if len(pts) != 2:
            continue
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == int(m['setno'])), None)
        if r is None:
            continue
        L = float(r.get('length', 0.0))
        if L <= 0:
            continue
        p0 = np.asarray(r['p0'], dtype=float)
        p1 = np.asarray(r['p1'], dtype=float)
        xy = []
        for x, _y, _sn in pts:
            frac = float(np.clip(float(x) / L, 0.0, 1.0))
            xy.append(p0 + frac * (p1 - p0))
        a, b = xy
        shadow, = self.line_profile_ax_image.plot([a[0], b[0]], [a[1], b[1]], '-', lw=6.5, color='black', alpha=0.75, zorder=25)
        yellow, = self.line_profile_ax_image.plot([a[0], b[0]], [a[1], b[1]], '-', lw=3.8, color='yellow', zorder=26)
        self._line_profile_measure_image_artists.extend([shadow, yellow])
    self.line_profile_canvas_image.draw_idle()

def _tab6_draw_measurement(self):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    for m in getattr(self, '_line_profile_measurements_list', []):
        pts = m.get('pts', [])
        if len(pts) != 2:
            continue
        x0, y0, _ = pts[0]
        x1, y1, _ = pts[1]
        c = 'blue'
        line, = self.line_profile_ax_graph.plot([x0, x1], [y0, y1], '-', color=c, lw=2.8, alpha=0.95, zorder=20)
        self._line_profile_measure_artists.append(line)
        for pid, (x, y, _sn) in zip((m['m1_id'], m['m2_id']), pts):
            sc = self.line_profile_ax_graph.scatter([x], [y], s=42, facecolor='blue', edgecolor='white', linewidths=1.2, zorder=22)
            self._line_profile_measure_artists.append(sc)
            txt = self.line_profile_ax_graph.annotate(f'M{pid}', (x, y), xytext=(5, 7), textcoords='offset points', color='blue', fontsize=9, fontweight='bold', zorder=23, bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='blue', alpha=0.9))
            self._line_profile_measure_artists.append(txt)
        ymin, ymax = self.line_profile_ax_graph.get_ylim()
        ybar = ymax - 0.055 * max(ymax - ymin, 1e-09) - 0.035 * m['pair_id'] * max(ymax - ymin, 1e-09)
        arrow = self.line_profile_ax_graph.annotate('', xy=(x1, ybar), xytext=(x0, ybar), arrowprops=dict(arrowstyle='<->', color='blue', lw=1.8))
        self._line_profile_measure_artists.append(arrow)
        label = self.line_profile_ax_graph.text((x0 + x1) / 2.0, ybar, f"M{m['m1_id']}–M{m['m2_id']}: ΔL={abs(x1 - x0):.2f} px", color='blue', fontsize=9, fontweight='bold', ha='center', va='bottom', bbox=dict(boxstyle='round,pad=0.16', facecolor='white', edgecolor='blue', alpha=0.88), zorder=24)
        self._line_profile_measure_artists.append(label)
    pending = getattr(self, '_line_profile_pending_point', None)
    if pending is not None:
        x, y, _ = pending
        sc = self.line_profile_ax_graph.scatter([x], [y], s=44, facecolor='blue', edgecolor='white', linewidths=1.1, zorder=24)
        self._line_profile_measure_artists.append(sc)
        pid = getattr(self, '_line_profile_measure_next_point', 1) - 1
        txt = self.line_profile_ax_graph.annotate(f'M{pid}', (x, y), xytext=(5, 7), textcoords='offset points', color='blue', fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='blue', alpha=0.9), zorder=25)
        self._line_profile_measure_artists.append(txt)
    self._line_profile_redraw_image_lines()
    self.line_profile_canvas_graph.draw_idle()

def _tab6_update_table(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    for item in tree.get_children():
        tree.delete(item)
    result_by_set = {int(r.get('set', -1)): r for r in getattr(self, '_line_profile_results', [])}
    for m in getattr(self, '_line_profile_measurements_list', []):
        pts = m.get('pts', [])
        r = result_by_set.get(int(m['setno']))
        length = float(r.get('length', 0.0)) if r is not None else 0.0
        if len(pts) == 2:
            x0, y0, _ = pts[0]
            x1, y1, _ = pts[1]
            vals = (f"M{m['m1_id']}–M{m['m2_id']}", m['setno'], f'{length:.2f}', f'{x0:.2f}', f'{x1:.2f}', f'{abs(x1 - x0):.2f}', f'{y0:.4f}', f'{y1:.4f}', 'Measured')
            tree.insert('', 'end', values=vals)
    pending = getattr(self, '_line_profile_pending_point', None)
    if pending is not None:
        pid = getattr(self, '_line_profile_measure_next_point', 1) - 1
        tree.insert('', 'end', values=(f'M{pid}', pending[2], '—', f'{pending[0]:.2f}', '—', '—', f'{pending[1]:.4f}', '—', 'Pending'))

def _tab6_refresh_measurements(self):
    refreshed = []
    for m in getattr(self, '_line_profile_measurements_list', []):
        rpts = []
        for pt in m.get('pts', [])[:2]:
            v = _tab6_measurement_from_x(self, int(m['setno']), float(pt[0]))
            if v is not None:
                rpts.append(v)
        if len(rpts) == 2:
            nm = dict(m)
            nm['pts'] = rpts
            refreshed.append(nm)
    self._line_profile_measurements_list = refreshed
    _tab6_sync_legacy_measurement_dict(self)
    if hasattr(self, 'line_profile_measure_var') and refreshed:
        n = len(refreshed) + 1
        self.line_profile_measure_var.set(f'{len(refreshed)} measurement pair(s) retained on current frame. Ready for M{2 * n - 1}–M{2 * n} on any profile.')

def _tab6_export_measurements(self):
    if not getattr(self, '_line_profile_measurements_list', None):
        messagebox.showwarning('Line Profile', 'No dynamic measurements to export.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Export dynamic measurements', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    rows = []
    try:
        idx = int(round(float(self.line_profile_frame_var.get())))
    except Exception:
        idx = 0
    result_by_set = {int(r.get('set', -1)): r for r in getattr(self, '_line_profile_results', [])}
    for m in getattr(self, '_line_profile_measurements_list', []):
        if len(m.get('pts', [])) != 2:
            continue
        x0, y0, _ = m['pts'][0]
        x1, y1, _ = m['pts'][1]
        r = result_by_set.get(int(m['setno']))
        rows.append({'pair': f"M{m['m1_id']}-M{m['m2_id']}", 'set': int(m['setno']), 'frame': idx + 1, 'field_mT': float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else '', 'line_length_px': float(r.get('length', 0.0)) if r else '', 'M1_position_px': float(x0), 'M2_position_px': float(x1), 'DeltaL_px': abs(float(x1) - float(x0)), 'M1_intensity': float(y0), 'M2_intensity': float(y1)})
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} dynamic measurement pair(s): {fp}')
_TAB6_ORIGINAL_BUILDER = CDIWorkflowApp._build_line_profile_tab
CDIWorkflowApp._line_profile_start_measurement = _tab6_start_dynamic_measurement
CDIWorkflowApp._line_profile_clear_measurement = _tab6_clear_dynamic_measurement
CDIWorkflowApp._line_profile_graph_press = _tab6_graph_press
CDIWorkflowApp._line_profile_graph_motion = _tab6_graph_motion
CDIWorkflowApp._line_profile_graph_release = _tab6_graph_release
CDIWorkflowApp._line_profile_redraw_image_lines = _tab6_redraw_image_lines
CDIWorkflowApp._line_profile_draw_measurement = _tab6_draw_measurement
CDIWorkflowApp._line_profile_update_measurement_table = _tab6_update_table
CDIWorkflowApp._line_profile_update_table = _tab6_update_table
CDIWorkflowApp._line_profile_refresh_measurement_values = _tab6_refresh_measurements
CDIWorkflowApp._tab6_sync_legacy_measurement_dict = _tab6_sync_legacy_measurement_dict
CDIWorkflowApp._line_profile_export_results = _tab6_export_measurements
_OLD_TAB6_REDRAW_PROFILES = CDIWorkflowApp._line_profile_redraw_profiles

def _tab6_redraw_profiles_with_dynamic_measurements(self, idx=None):
    _OLD_TAB6_REDRAW_PROFILES(self, idx=idx)
    try:
        self._tab6_refresh_measurements()
        self._tab6_update_table()
        self._tab6_draw_measurement()
    except Exception:
        pass
CDIWorkflowApp._line_profile_redraw_profiles = _tab6_redraw_profiles_with_dynamic_measurements
_TAB6_BASE_BUILDER_ZOOM = CDIWorkflowApp._build_line_profile_tab
_TAB6_BASE_UPDATE_IMAGE_ZOOM = CDIWorkflowApp._line_profile_update_image

def _tab6_zoom_clamp_limits(self, xlim, ylim):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        return (xlim, ylim)
    try:
        arr = np.asarray(stack[0])
        h, w = arr.shape[:2]
    except Exception:
        return (xlim, ylim)
    xmin, xmax = (-0.5, float(w) - 0.5)
    ymin, ymax = (-0.5, float(h) - 0.5)

    def clamp_pair(a, b, lo, hi):
        width = float(b - a)
        full = float(hi - lo)
        if width >= full:
            return (lo, hi)
        if a < lo:
            b += lo - a
            a = lo
        if b > hi:
            a -= b - hi
            b = hi
        a = max(lo, a)
        b = min(hi, b)
        return (a, b)
    return (clamp_pair(float(xlim[0]), float(xlim[1]), xmin, xmax), clamp_pair(float(ylim[0]), float(ylim[1]), ymin, ymax))

def _tab6_zoom_restore_limits(self):
    limits = getattr(self, '_line_profile_zoom_limits', None)
    if not getattr(self, '_line_profile_zoomed', False) or limits is None:
        return
    xlim, ylim = limits
    xlim, ylim = _tab6_zoom_clamp_limits(self, xlim, ylim)
    self.line_profile_ax_image.set_xlim(*xlim)
    self.line_profile_ax_image.set_ylim(*ylim)
    self.line_profile_ax_image.set_aspect('equal', adjustable='box')
    self._line_profile_zoom_limits = (tuple(xlim), tuple(ylim))

def _line_profile_update_image_with_zoom(self, idx=None, preserve_lines=True):
    was_zoomed = bool(getattr(self, '_line_profile_zoomed', False))
    saved_limits = getattr(self, '_line_profile_zoom_limits', None)
    _TAB6_BASE_UPDATE_IMAGE_ZOOM(self, idx=idx, preserve_lines=preserve_lines)
    if was_zoomed and saved_limits is not None:
        self._line_profile_zoom_limits = saved_limits
        _tab6_zoom_restore_limits(self)
        self.line_profile_canvas_image.draw_idle()

def _line_profile_zoom_mousewheel(self, event):
    if event is None or getattr(event, 'inaxes', None) is not self.line_profile_ax_image:
        return
    if getattr(event, 'xdata', None) is None or getattr(event, 'ydata', None) is None:
        return
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        return
    try:
        x = float(event.xdata)
        y = float(event.ydata)
    except Exception:
        return
    if getattr(event, 'button', None) == 'up' or getattr(event, 'step', 0) > 0:
        zoom_factor = 1.22
    elif getattr(event, 'button', None) == 'down' or getattr(event, 'step', 0) < 0:
        zoom_factor = 1.0 / 1.22
    else:
        delta = getattr(event, 'delta', 0)
        zoom_factor = 1.22 if delta > 0 else 1.0 / 1.22 if delta < 0 else 1.0
    if zoom_factor == 1.0:
        return
    old_xlim = self.line_profile_ax_image.get_xlim()
    old_ylim = self.line_profile_ax_image.get_ylim()
    old_w = old_xlim[1] - old_xlim[0]
    old_h = old_ylim[1] - old_ylim[0]
    new_w = old_w / zoom_factor
    new_h = old_h / zoom_factor
    new_xlim = (x - (x - old_xlim[0]) / zoom_factor, x + (old_xlim[1] - x) / zoom_factor)
    new_ylim = (y - (y - old_ylim[0]) / zoom_factor, y + (old_ylim[1] - y) / zoom_factor)
    try:
        arr = np.asarray(stack[int(self.line_profile_frame_var.get())])
        h, w = arr.shape[:2]
        min_w = max(8.0, 0.015 * w)
        min_h = max(8.0, 0.015 * h)
        if new_xlim[1] - new_xlim[0] < min_w:
            cx = x
            half = min_w / 2.0
            new_xlim = (cx - half, cx + half)
        if new_ylim[1] - new_ylim[0] < min_h:
            cy = y
            half = min_h / 2.0
            new_ylim = (cy - half, cy + half)
    except Exception:
        pass
    new_xlim, new_ylim = _tab6_zoom_clamp_limits(self, new_xlim, new_ylim)
    self.line_profile_ax_image.set_xlim(*new_xlim)
    self.line_profile_ax_image.set_ylim(*new_ylim)
    self.line_profile_ax_image.set_aspect('equal', adjustable='box')
    self._line_profile_zoomed = True
    self._line_profile_zoom_limits = (tuple(new_xlim), tuple(new_ylim))
    self.line_profile_canvas_image.draw_idle()
    return 'break'

def _line_profile_plot_scroll_zoom_dispatch(self, event):
    if getattr(event, 'inaxes', None) is self.line_profile_ax_image:
        return _line_profile_zoom_mousewheel(self, event)
    if getattr(event, 'inaxes', None) is self.line_profile_ax_graph:
        return 'break'
    return None

def _line_profile_reset_zoom(self):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        return
    try:
        arr = np.asarray(stack[int(self.line_profile_frame_var.get())])
        h, w = arr.shape[:2]
    except Exception:
        return
    xlim = (-0.5, float(w) - 0.5)
    ylim = (-0.5, float(h) - 0.5)
    self.line_profile_ax_image.set_xlim(*xlim)
    self.line_profile_ax_image.set_ylim(*ylim)
    self.line_profile_ax_image.set_aspect('equal', adjustable='box')
    self._line_profile_zoomed = False
    self._line_profile_zoom_limits = None
    if hasattr(self, 'line_profile_status_var'):
        self.line_profile_status_var.set('ROI zoom reset to full Secondary ROI. Drawn and highlighted lines are preserved.')
    self.line_profile_canvas_image.draw_idle()
CDIWorkflowApp._line_profile_update_image = _line_profile_update_image_with_zoom
CDIWorkflowApp._line_profile_plot_scroll = _line_profile_plot_scroll_zoom_dispatch
CDIWorkflowApp._line_profile_reset_zoom = _line_profile_reset_zoom
CDIWorkflowApp._line_profile_zoom_mousewheel = _line_profile_zoom_mousewheel
_OLD_PD_REFRESH = getattr(CDIWorkflowApp, 'update_roi2_view', None)

def _add_export_folder_controls(self, parent):
    if not hasattr(self, '_export_folder_var'):
        self._export_folder_var = tk.StringVar(value=os.path.abspath(getattr(self, 'output_dir', os.getcwd())))
    box = ttk.Frame(parent, padding=(6, 3, 6, 5))
    box.pack(fill='x', pady=(0, 3))
    ttk.Label(box, text='EXPORT FOLDER PATH:').pack(side='left')
    ttk.Entry(box, textvariable=self._export_folder_var).pack(side='left', fill='x', expand=True, padx=5)

    def choose_folder():
        folder = filedialog.askdirectory(parent=self.root, title='Select export folder')
        if folder:
            self._export_folder_var.set(folder)
    ttk.Button(box, text='SELECT FOLDER', command=choose_folder).pack(side='left')
    return box

def _export_initialdir(self):
    try:
        folder = self._export_folder_var.get().strip()
        if folder and os.path.isdir(folder):
            return folder
    except Exception:
        pass
    return os.getcwd()
CDIWorkflowApp._add_export_folder_controls = _add_export_folder_controls
CDIWorkflowApp._export_initialdir = _export_initialdir

def _roi2_frame_slider_wheel(self, event):
    slider = getattr(self, 'roi2_slider', None)
    stack = getattr(self, 'roi2_stack', None)
    if slider is None or stack is None or len(stack) == 0:
        return 'break'
    try:
        rx, ry = (slider.winfo_rootx(), slider.winfo_rooty())
        rw, rh = (slider.winfo_width(), slider.winfo_height())
        px, py = (slider.winfo_toplevel().winfo_pointerx(), slider.winfo_toplevel().winfo_pointery())
        if not (rx <= px <= rx + rw and ry <= py <= ry + rh):
            return 'break'
    except Exception:
        return 'break'
    step = 1 if getattr(event, 'num', None) == 4 or getattr(event, 'delta', 0) > 0 or getattr(event, 'step', 0) > 0 else -1
    self.roi2_idx_var.set(max(0, min(len(stack) - 1, int(self.roi2_idx_var.get()) + step)))
    self._roi2_slider_changed()
    return 'break'
CDIWorkflowApp._roi2_frame_slider_wheel = _roi2_frame_slider_wheel
_old_init_export_roi2 = CDIWorkflowApp.__init__

def _init_export_roi2(self, *a, **kw):
    _old_init_export_roi2(self, *a, **kw)
    try:
        self.roi2_slider.bind('<MouseWheel>', self._roi2_frame_slider_wheel, add='+')
        self.roi2_slider.bind('<Button-4>', self._roi2_frame_slider_wheel, add='+')
        self.roi2_slider.bind('<Button-5>', self._roi2_frame_slider_wheel, add='+')
    except Exception:
        pass
CDIWorkflowApp.__init__ = _init_export_roi2
_old_add_export_folder_controls = getattr(CDIWorkflowApp, '_add_export_folder_controls', None)

def _common_export_folder_controls(self, parent):
    if getattr(self, 'tab_data', None) is not parent:
        return None
    if hasattr(self, '_common_export_folder_box'):
        try:
            self._common_export_folder_box.destroy()
        except Exception:
            pass
    if not hasattr(self, '_export_folder_var'):
        self._export_folder_var = tk.StringVar(value=os.path.abspath(getattr(self, 'output_dir', os.getcwd())))
    box = ttk.LabelFrame(parent, text='COMMON EXPORT FOLDER — ALL TABS', padding=6)
    box.pack(fill='x', padx=10, pady=(8, 4))
    ttk.Label(box, text='Export folder path:').pack(side='left')
    ttk.Entry(box, textvariable=self._export_folder_var).pack(side='left', fill='x', expand=True, padx=6)

    def choose_folder():
        folder = filedialog.askdirectory(parent=self.root, title='Select common export folder')
        if folder:
            self._export_folder_var.set(folder)
    ttk.Button(box, text='SELECT FOLDER', command=choose_folder).pack(side='left')
    self._common_export_folder_box = box
    return box
CDIWorkflowApp._add_export_folder_controls = _common_export_folder_controls

def _build_data_tab_with_common_export(self):
    _OLD_BUILD_DATA_TAB(self)
    box = self._add_export_folder_controls(self.tab_data)
    if box is not None:
        try:
            children = self.tab_data.winfo_children()
            if children and box in children:
                box.pack_configure(before=children[0])
        except Exception:
            pass
_OLD_BUILD_DATA_TAB = CDIWorkflowApp._build_data_tab
CDIWorkflowApp._build_data_tab = _build_data_tab_with_common_export
_OLD_BUILD_PD = CDIWorkflowApp._build_particle_density_tab

def _install_safe_slider_wheel(self):

    def walk(w):
        yield w
        try:
            for c in w.winfo_children():
                yield from walk(c)
        except Exception:
            pass
    for scale in [w for w in walk(self.root) if isinstance(w, (tk.Scale, ttk.Scale))]:
        try:
            scale.unbind('<MouseWheel>')
            scale.unbind('<Button-4>')
            scale.unbind('<Button-5>')
            scale._safe_slider_original_bindings_removed = True

            def arm(event, sc=scale):
                try:
                    sc.focus_set()
                except Exception:
                    pass
                sc._wheel_armed = True
                return None

            def wheel(event, sc=scale):
                try:
                    if sc.focus_get() is not sc or not getattr(sc, '_wheel_armed', False):
                        return 'break'
                    lo = float(sc.cget('from'))
                    hi = float(sc.cget('to'))
                    value = float(sc.get())
                    if isinstance(sc, tk.Scale):
                        step = float(sc.cget('resolution'))
                    else:
                        step = abs(hi - lo) / 100.0
                    if step <= 0:
                        step = 1.0
                    direction = 1 if getattr(event, 'num', None) == 4 or getattr(event, 'delta', 0) > 0 else -1
                    new = max(min(value + direction * step, max(lo, hi)), min(lo, hi))
                    sc.set(new)
                    cmd = ''
                    try:
                        cmd = str(sc.cget('command'))
                    except Exception:
                        pass
                    if cmd:
                        try:
                            sc.tk.call(cmd, str(new))
                        except Exception:
                            pass
                    return 'break'
                except Exception:
                    return 'break'
            scale.bind('<Button-1>', arm, add='+')
            scale.bind('<MouseWheel>', wheel, add='+')
            scale.bind('<Button-4>', wheel, add='+')
            scale.bind('<Button-5>', wheel, add='+')
            scale._wheel_armed = False
        except Exception:
            pass
CDIWorkflowApp._install_safe_slider_wheel = _install_safe_slider_wheel
_OLD_INIT_SAFE_SLIDERS = CDIWorkflowApp.__init__

def _init_safe_sliders(self, *args, **kwargs):
    _OLD_INIT_SAFE_SLIDERS(self, *args, **kwargs)
    try:
        self._install_safe_slider_wheel()
    except Exception:
        pass
    try:
        self.nb.bind('<<NotebookTabChanged>>', self._pd_on_tab_selected, add='+')
    except Exception:
        pass
    try:
        self.after(50, self._pd_on_tab_selected)
    except Exception:
        pass
CDIWorkflowApp.__init__ = _init_safe_sliders

def _tab5_export_initialdir(self):
    try:
        folder = self.output_var.get().strip()
        if folder and os.path.isdir(folder):
            return folder
    except Exception:
        pass
    return os.getcwd()

def _export_processed_tab5_folder(self, ext='png'):
    if self.current_processed is None:
        self.update_processing_view()
    fp = filedialog.asksaveasfilename(parent=self.root, initialdir=self._tab5_export_initialdir(), defaultextension='.' + ext, filetypes=[(ext.upper(), '*.' + ext)])
    if not fp:
        return
    Image.fromarray(np.uint8(np.clip(self.current_processed, 0, 1) * 255)).save(fp)
    self.log(f'Exported {fp}')
CDIWorkflowApp._tab5_export_initialdir = _tab5_export_initialdir
CDIWorkflowApp.export_processed = _export_processed_tab5_folder

def _wrap_tab8_export(method_name):
    original = getattr(CDIWorkflowApp, method_name)

    def wrapped(self, *args, **kwargs):
        had_old = hasattr(self, 'output_var')
        old_value = None
        if had_old:
            try:
                old_value = self.output_var.get()
            except Exception:
                old_value = None
        try:
            folder = self._tab8_export_initialdir()
            os.makedirs(folder, exist_ok=True)
            if not hasattr(self, 'output_var'):
                self.output_var = tk.StringVar(value=folder)
                created = True
            else:
                created = False
                self.output_var.set(folder)
            return original(self, *args, **kwargs)
        finally:
            try:
                if had_old and old_value is not None:
                    self.output_var.set(old_value)
                elif created:
                    self.output_var.set(os.path.abspath(getattr(self, 'output_dir', os.getcwd())))
            except Exception:
                pass
    return wrapped
CDIWorkflowApp.batch_export_roi2_mp4 = _wrap_tab8_export('batch_export_roi2_mp4')
CDIWorkflowApp.batch_export_roi2_images = _wrap_tab8_export('batch_export_roi2_images')
_old_init_export_policy = CDIWorkflowApp.__init__

def _init_export_policy(self, *args, **kwargs):
    _old_init_export_policy(self, *args, **kwargs)
    try:
        self._build_tab8_export_folder_controls()
    except Exception as exc:
        try:
            self.log(f'Tab 8 export-folder UI warning: {exc}')
        except Exception:
            pass
CDIWorkflowApp.__init__ = _init_export_policy

def _restricted_export_folder_controls(self, parent):
    if parent is self.tab_data:
        return self._common_export_folder_controls(self, parent) if False else None
    if parent is self.tab_proc:
        return None
    if parent is self.tab_roi2:
        return None
    return None

def _pd3_field_value(self, idx):
    try:
        arr = getattr(self, 'real_fields_mT', None)
        if arr is not None and len(arr) > int(idx):
            v = float(arr[int(idx)])
            if np.isfinite(v):
                return v
    except Exception:
        pass
    return float(idx)

def _pd3_normalize(self, image):
    a = np.asarray(image, dtype=float)
    if a.ndim != 2:
        a = np.squeeze(a)
    finite = np.isfinite(a)
    if not np.any(finite):
        return np.zeros_like(a, dtype=float)
    vals = a[finite]
    lo, hi = np.percentile(vals, [1.0, 99.0])
    if hi <= lo:
        lo, hi = (float(vals.min()), float(vals.max()))
    if hi <= lo:
        return np.zeros_like(a, dtype=float)
    return np.clip((a - lo) / (hi - lo), 0.0, 1.0)

def _pd3_detect_frame(self, image):
    """Noisy-image detector: denoise -> local background -> threshold -> CC -> shape."""
    from scipy import ndimage as ndi
    from skimage.measure import regionprops
    from skimage.morphology import opening, closing, disk, remove_small_objects
    z = self._pd3_normalize(image)
    try:
        threshold = float(self.pd_threshold_var.get())
        threshold = float(np.clip(threshold, 0.01, 0.5))
    except Exception:
        threshold = 0.1
    try:
        dmin = float(self.pd_min_diam_var.get())
        dmax = float(self.pd_max_diam_var.get())
    except Exception:
        dmin, dmax = (10.0, 50.0)
    dmin = max(3.0, dmin)
    dmax = max(dmin, dmax)
    try:
        min_sep = max(1.0, float(self.pd_min_sep_var.get()))
    except Exception:
        min_sep = 50.0
    try:
        ar_max = max(1.0, float(self.pd_max_aspect_var.get()))
    except Exception:
        ar_max = 1.8
    polarity = str(self.pd_polarity_var.get()).lower().strip() if hasattr(self, 'pd_polarity_var') else 'both'
    include_edge = bool(self.pd_include_edge_var.get()) if hasattr(self, 'pd_include_edge_var') else False
    med = ndi.median_filter(z, size=3, mode='nearest')
    smooth = ndi.gaussian_filter(med, sigma=1.0, mode='nearest')
    bg_sigma = max(5.0, 0.75 * dmax)
    bg = ndi.gaussian_filter(smooth, sigma=bg_sigma, mode='nearest')
    detail = smooth - bg
    h, w = z.shape
    candidates = []
    if polarity in ('bright', 'both'):
        polarity_list.append('bright') if False else None
    pols = []
    if polarity in ('bright', 'both'):
        pols.append(('bright', detail))
    if polarity in ('dark', 'both'):
        pols.append(('dark', -detail))
    if not pols:
        pols = [('bright', detail)]
    for pol, signal in pols:
        pos = signal[signal > 0]
        scale = float(np.percentile(pos, 99.0)) if pos.size else 1.0
        scale = max(scale, 1e-09)
        q = np.clip(signal / scale, 0.0, 1.0)
        mask = q >= threshold
        r = max(1, min(2, int(round(dmin / 20.0))))
        mask = opening(mask, disk(r))
        mask = closing(mask, disk(r))
        min_area = max(8, int(np.pi * (dmin / 2.0) ** 2 * 0.18))
        mask = remove_small_objects(mask, max_size=max(0, int(min_area)-1))
        labels, n = ndi.label(mask)
        props = regionprops(labels, intensity_image=q)
        for rp in props:
            area = float(rp.area)
            eqd = float(rp.equivalent_diameter_area)
            if eqd < dmin or eqd > dmax:
                continue
            cy, cx = map(float, rp.centroid)
            edge = cx < eqd / 2 or cy < eqd / 2 or cx > w - 1 - eqd / 2 or (cy > h - 1 - eqd / 2)
            if edge and (not include_edge):
                continue
            per = float(rp.perimeter)
            circ = float(4 * np.pi * area / (per * per)) if per > 1e-09 else 0.0
            sol = float(rp.solidity)
            maj = float(rp.axis_major_length)
            mino = float(rp.axis_minor_length)
            aspect = maj / max(mino, 1e-09)
            ecc = float(rp.eccentricity)
            rad = max(4, int(round(eqd * 0.75)))
            yy0 = max(0, int(cy - rad))
            yy1 = min(h, int(cy + rad + 1))
            xx0 = max(0, int(cx - rad))
            xx1 = min(w, int(cx + rad + 1))
            patch = q[yy0:yy1, xx0:xx1]
            obj = labels[yy0:yy1, xx0:xx1] == rp.label
            inside = patch[obj]
            outside = patch[~obj]
            contrast = float(abs(np.median(inside) - np.median(outside))) if inside.size and outside.size else float(np.max(inside) - np.median(patch))
            candidates.append(dict(x=cx, y=cy, diameter_px=eqd, radius_px=eqd / 2.0, area_px=area, circularity=circ, solidity=sol, eccentricity=ecc, aspect_ratio=aspect, contrast=contrast, score=float(contrast * (0.5 * circ + 0.3 * sol + 0.2 * (1 - min(ecc, 1.0)))), edge=bool(edge), polarity=pol, method='Threshold + Segmentation + Shape'))
    if not candidates:
        return []
    circs = np.array([d['circularity'] for d in candidates], float)
    sols = np.array([d['solidity'] for d in candidates], float)
    circ_gate = float(np.clip(max(0.45, np.percentile(circs, 20)), 0.45, 0.8))
    sol_gate = float(np.clip(max(0.65, np.percentile(sols, 20)), 0.65, 0.95))
    ecc_gate = float(np.sqrt(max(0.0, 1.0 - 1.0 / (ar_max * ar_max)))) if ar_max > 1 else 0.0
    self.pd_auto_shape_text = f'Auto shape gates: circularity ≥ {circ_gate:.2f} | solidity ≥ {sol_gate:.2f} | eccentricity ≤ {ecc_gate:.2f}'
    idx = int(self.pd_frame_var.get()) if hasattr(self, 'pd_frame_var') else 0
    nnc = getattr(self, 'pd_nnc_exclusions', {}).get(idx, []) if hasattr(self, 'pd_nnc_exclusions') else []
    filtered = []
    for d in candidates:
        if d['circularity'] < circ_gate or d['solidity'] < sol_gate or d['eccentricity'] > ecc_gate or (d['aspect_ratio'] > ar_max):
            continue
        blocked = False
        for ex in nnc:
            if np.hypot(d['x'] - ex[0], d['y'] - ex[1]) <= max(8.0, 0.6 * d['diameter_px']):
                blocked = True
                break
        if blocked:
            continue
        filtered.append(d)
    filtered.sort(key=lambda d: d['score'], reverse=True)
    accepted = []
    for d in filtered:
        if any((np.hypot(d['x'] - q['x'], d['y'] - q['y']) < min_sep for q in accepted)):
            continue
        accepted.append(d)
    return accepted

def _pd3_update_selected_particle_view(self):
    if not getattr(self, 'pd_current_detections', None):
        self.pd_particle_status.config(text='No particle selected. Click an accepted particle/centre in the ROI.')
        self.pd_3d_canvas.draw_idle()
        self.pd_fwhm_canvas.draw_idle()
        return
    j = getattr(self, 'pd_selected_detection', None)
    if j is None or j < 0 or j >= len(self.pd_current_detections):
        self.pd_particle_status.config(text='No particle selected. Click an accepted particle/centre in the ROI.')
        return
    d = self.pd_current_detections[int(j)]
    idx = int(self.pd_frame_var.get())
    img = np.asarray(self.roi2_stack[idx], dtype=float)
    analysis = self._pd3_particle_analysis(d, img)
    d.update(analysis)
    self.pd_selected_particle = d
    rad = max(12, int(np.ceil(0.9 * d['diameter_px'])))
    h, w = img.shape
    x0 = max(0, int(round(d['x'])) - rad)
    x1 = min(w, int(round(d['x'])) + rad + 1)
    y0 = max(0, int(round(d['y'])) - rad)
    y1 = min(h, int(round(d['y'])) + rad + 1)
    patch = img[y0:y1, x0:x1]
    yy, xx = np.mgrid[y0:y1, x0:x1]
    self.pd_3d_ax.cla()
    self.pd_3d_ax.plot_surface(xx, yy, patch, cmap='viridis', linewidth=0, antialiased=True)
    self.pd_3d_ax.set_xlabel('X pixel', fontsize=8)
    self.pd_3d_ax.set_ylabel('Y pixel', fontsize=8)
    self.pd_3d_ax.set_zlabel('Intensity', fontsize=8)
    self.pd_3d_ax.set_title(f'Particle #{int(j) + 1} — 3D intensity', fontsize=9, pad=8)
    self.pd_fwhm_ax.cla()
    for ang in (0.0, 45.0, 90.0, 135.0):
        r, p = analysis['profiles'][ang]
        lab = f"{int(ang)}°  FWHM={analysis[f'fwhm_{(int(ang) if ang in (0.0, 45.0, 90.0, 135.0) else 0)}']:.2f}"
        self.pd_fwhm_ax.plot(r, p, label=lab, linewidth=1.5)
    self.pd_fwhm_ax.axvline(0.0, ls='--', lw=0.8)
    self.pd_fwhm_ax.set_xlabel('Distance from center (pixel)', fontsize=8)
    self.pd_fwhm_ax.set_ylabel('Intensity', fontsize=8)
    self.pd_fwhm_ax.set_title(f"0° / 45° / 90° / 135° profiles | Avg FWHM={analysis['fwhm_avg']:.2f} px", fontsize=9)
    self.pd_fwhm_ax.legend(fontsize=7, loc='best')
    self.pd_fwhm_ax.grid(alpha=0.22)
    self.pd_3d_canvas.draw_idle()
    self.pd_fwhm_canvas.draw_idle()
    ftxt = f"Particle #{int(j) + 1}: center=({d['x']:.2f}, {d['y']:.2f}) px | Diameter={d['diameter_px']:.2f} px | Shape AR={d['aspect_ratio']:.2f}\nFWHM: 0°={analysis['fwhm_0']:.2f}, 45°={analysis['fwhm_45']:.2f}, 90°={analysis['fwhm_90']:.2f}, 135°={analysis['fwhm_135']:.2f} | Average={analysis['fwhm_avg']:.2f} ± {analysis['fwhm_sd']:.2f} px | FWHM-AR={analysis['fwhm_profile_aspect']:.2f}"
    self.pd_particle_status.config(text=ftxt)
    self._pd3_refresh_density_row(idx)

def _pd3_refresh_density_row(self, idx):
    dets = getattr(self, 'pd_frame_detections', {}).get(idx, []) or []
    valid = [d for d in dets if not d.get('edge', False) or bool(self.pd_include_edge_var.get())]
    for d in valid:
        if 'fwhm_avg' not in d:
            try:
                d.update(self._pd3_particle_analysis(d, np.asarray(self.roi2_stack[idx], dtype=float)))
            except Exception:
                pass
    count = len(valid)
    arr = np.array([d.get('fwhm_avg', np.nan) for d in valid], float)
    arr = arr[np.isfinite(arr)]
    dvals = np.array([d.get('diameter_px', np.nan) for d in valid], float)
    dvals = dvals[np.isfinite(dvals)]
    arvals = np.array([d.get('aspect_ratio', np.nan) for d in valid], float)
    arvals = arvals[np.isfinite(arvals)]
    h, w = np.asarray(self.roi2_stack[idx]).shape[:2]
    sx, sy = _tab8_authoritative_nm_scales(self)
    area_um2 = ((float(w) * float(sx)) * (float(h) * float(sy)) / 1.0e6) if (sx and sy and sx > 0 and sy > 0) else np.nan
    dens = count / area_um2 if area_um2 > 0 else np.nan
    row = (idx + 1, self._pd3_field_value(idx), count, area_um2, dens, float(np.mean(dvals)) if dvals.size else np.nan, float(np.mean(arr)) if arr.size else np.nan, float(np.mean(arvals)) if arvals.size else np.nan)
    self.pd_results[idx] = row
    self._pd3_refresh_table()
    self._pd3_update_density_plot()

def _pd3_refresh_table(self):
    if not hasattr(self, 'pd_table'):
        return
    for item in self.pd_table.get_children():
        self.pd_table.delete(item)
    for k in sorted(self.pd_results):
        self.pd_table.insert('', 'end', values=self.pd_results[k])

def _pd3_update_density_plot(self):
    if not hasattr(self, 'pd_plot_canvas'):
        return
    ax = self.pd_plot_ax
    ax2 = self.pd_plot_ax2
    ax.cla()
    ax2.cla()
    ax.set_zorder(2)
    ax.patch.set_alpha(0.0)
    ax2.set_zorder(1)
    ax2.patch.set_visible(False)
    rows = []
    for k in sorted(getattr(self, 'pd_results', {}) or {}):
        r = self.pd_results[k]
        if r and len(r) >= 5:
            rows.append(r)
    if rows:
        frames = np.array([r[0] for r in rows], float)
        fields = np.array([r[1] for r in rows], float)
        counts = np.array([r[2] for r in rows], float)
        dens = np.array([r[4] for r in rows], float)
        ax.plot(fields, counts, 'o-', lw=1.7, ms=4, label='Count')
        ax2.plot(frames, dens, 's--', lw=1.5, ms=3.5, label='Density')
        ax.set_xlabel('Field (mT)', fontsize=8)
        ax.set_ylabel('Particle count', fontsize=8)
        ax2.set_xlabel('Equivalent frame No.', fontsize=8, pad=4)
        ax2.xaxis.set_label_position('top')
        ax2.xaxis.tick_top()
        ax2.set_ylabel('Density (skyrmions/µm²)', fontsize=8)
        ax2.yaxis.set_label_position('right')
        ax2.yaxis.tick_right()
        ax.grid(alpha=0.22)
        ax.set_title('Count vs Field  |  Density vs Frame', fontsize=9, pad=18)
        ax.relim()
        ax.autoscale_view()
        ax2.relim()
        ax2.autoscale_view()
    else:
        ax.text(0.5, 0.5, 'Analyze a frame to plot results', ha='center', va='center', transform=ax.transAxes, fontsize=8)
        ax.set_xlabel('Field (mT)', fontsize=8)
        ax.set_ylabel('Particle count', fontsize=8)
        ax2.set_xlabel('Equivalent frame No.', fontsize=8)
        ax2.xaxis.set_label_position('top')
        ax2.xaxis.tick_top()
        ax2.set_ylabel('Density (skyrmions/µm²)', fontsize=8)
        ax2.yaxis.set_label_position('right')
        ax2.yaxis.tick_right()
        ax.set_title('Count vs Field  |  Density vs Frame', fontsize=9, pad=18)
    self.pd_plot_canvas.draw_idle()

def _pd3_analyze_current(self):
    try:
        self._pd_sync_secondary_roi()
    except Exception:
        pass
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        messagebox.showwarning('Skyrmion Analysis', 'Select/confirm the Secondary ROI first.', parent=self.root)
        return
    idx = max(0, min(int(self.pd_frame_var.get()), len(stack) - 1))
    self.pd_frame_var.set(idx)
    det = [dict(d) for d in self._pd3_detect_frame(np.asarray(stack[idx], dtype=float))]
    self.pd_current_detections = det
    self.pd_frame_detections[idx] = [dict(d) for d in det]
    self.pd_selected_detection = None
    self.pd_last_deleted = None
    self._pd3_update_roi_image(preserve_zoom=True)
    self._pd3_draw_overlay()
    self._pd3_refresh_density_row(idx)
    self.pd_status_var.set(f'Frame {idx + 1}: detected {len(det)} accepted particles. Click a particle to calculate 4-direction FWHM.')

def _pd3_analyze_selected_frames(self):
    try:
        self._pd_sync_secondary_roi()
    except Exception:
        pass
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        messagebox.showwarning('Skyrmion Analysis', 'Select/confirm the Secondary ROI first.', parent=self.root)
        return
    n = len(stack)
    try:
        a = max(1, int(self.pd_range_start_var.get()))
        b = min(n, int(self.pd_range_end_var.get()))
    except Exception:
        a, b = (1, n)
    if a > b:
        a, b = (b, a)
    self.pd_results = {}
    for idx in range(a - 1, b):
        det = [dict(d) for d in self._pd3_detect_frame(np.asarray(stack[idx], dtype=float))]
        self.pd_frame_detections[idx] = det
        self.pd_current_detections = det
        self.pd_selected_detection = None
        self._pd3_refresh_density_row(idx)
    keep = max(a - 1, min(int(self.pd_frame_var.get()), b - 1))
    self.pd_frame_var.set(keep)
    self.pd_current_detections = [dict(d) for d in self.pd_frame_detections.get(keep, [])]
    self._pd3_update_roi_image(preserve_zoom=True)
    self._pd3_draw_overlay()
    self._pd3_refresh_table()
    self._pd3_update_density_plot()
    self.pd_status_var.set(f'Analyzed selected frames {a}–{b} ({b - a + 1} frame(s)).')

def _pd3_update_roi_image(self, preserve_zoom=True):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        self.pd_frame_slider.configure(from_=0, to=0, state='disabled')
        return
    idx = max(0, min(int(self.pd_frame_var.get()), len(stack) - 1))
    self.pd_frame_var.set(idx)
    self.pd_frame_slider.configure(from_=0, to=len(stack) - 1, state='normal')
    img = np.asarray(stack[idx], dtype=float)
    self.pd_image_artist_gray.set_data(img)
    self.pd_image_artist_color.set_data(img)
    try:
        cmap = getattr(self, 'colormap_var', tk.StringVar(value='gray')).get() or 'gray'
        self.pd_image_artist_color.set_cmap(cmap)
    except Exception:
        self.pd_image_artist_color.set_cmap('gray')
    h, w = img.shape[:2]
    old = getattr(self, '_pd_zoom_limits', None) if preserve_zoom else None
    if old is None:
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            ax.set_xlim(-0.5, w - 0.5)
            ax.set_ylim(-0.5, h - 0.5)
    else:
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            ax.set_xlim(*old[0])
            ax.set_ylim(*old[1])
    field = self._pd3_field_value(idx)
    self.pd_frame_label.config(text=f'{idx + 1}/{len(stack)} | {field:+.2f} mT')
    self.pd_ax_gray.set_title(f'Selected ROI — Grayscale | Frame {idx + 1}/{len(stack)}', fontsize=9)
    self.pd_ax_color.set_title(f'Selected ROI — Color | Frame {idx + 1}/{len(stack)} | {field:+.2f} mT', fontsize=9)
    self.pd_canvas.draw_idle()

def _pd3_zoom_scroll(self, event):
    if event is None or event.inaxes not in (self.pd_ax_gray, self.pd_ax_color):
        return
    ax = event.inaxes
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x = float(event.xdata)
    y = float(event.ydata)
    if not np.isfinite(x + y):
        return
    fac = 0.8 if getattr(event, 'button', None) == 'up' else 1.25
    x0 = x - (x - xlim[0]) * fac
    x1 = x + (xlim[1] - x) * fac
    y0 = y - (y - ylim[0]) * fac
    y1 = y + (ylim[1] - y) * fac
    for aa in (self.pd_ax_gray, self.pd_ax_color):
        aa.set_xlim(x0, x1)
        aa.set_ylim(y0, y1)
    self._pd_zoom_limits = ((x0, x1), (y0, y1))
    self.pd_canvas.draw_idle()

def _pd3_reset_zoom(self):
    self._pd_zoom_limits = None
    self._pd3_update_roi_image(preserve_zoom=False)
    self.pd_status_var.set('ROI zoom reset.')

def _pd3_frame_changed(self, *args):
    self._pd3_update_roi_image(preserve_zoom=True)
    self._pd3_load_current_detections()

def _pd3_load_current_detections(self):
    idx = int(self.pd_frame_var.get())
    self.pd_current_detections = [dict(d) for d in self.pd_frame_detections.get(idx, [])]
    self.pd_selected_detection = None
    self._pd3_draw_overlay()
    self._pd3_refresh_density_row(idx) if idx in self.pd_results else None

def _pd3_frame_wheel(self, event):
    slider = self.pd_frame_slider
    try:
        rx = slider.winfo_rootx()
        ry = slider.winfo_rooty()
        rw = slider.winfo_width()
        rh = slider.winfo_height()
        px = slider.winfo_toplevel().winfo_pointerx()
        py = slider.winfo_toplevel().winfo_pointery()
        if not (rx <= px <= rx + rw and ry <= py <= ry + rh and getattr(slider, '_pd3_armed', False)):
            return 'break'
        delta = 1 if getattr(event, 'num', None) == 4 or getattr(event, 'delta', 0) > 0 else -1
        new = max(0, min(len(getattr(self, 'roi2_stack', [])) - 1, int(slider.get()) + delta))
        slider.set(new)
        self.pd_frame_var.set(new)
        self._pd3_frame_changed()
        return 'break'
    except Exception:
        return 'break'

def _pd3_select_detection(self, event):
    if event is None or event.inaxes not in (self.pd_ax_gray, self.pd_ax_color) or (not self.pd_current_detections) or (event.xdata is None) or (event.ydata is None):
        return
    ds = self.pd_current_detections
    dist = [np.hypot(d['x'] - event.xdata, d['y'] - event.ydata) for d in ds]
    j = int(np.argmin(dist))
    tol = max(8.0, 0.65 * float(ds[j].get('diameter_px', 12.0)))
    if dist[j] <= tol:
        ctrl = bool(getattr(event, 'key', None) in ('control', 'ctrl'))
        if ctrl:
            sel = getattr(self, 'pd_multi_selected', set())
            sel = set(sel)
            sel.remove(j) if j in sel else sel.add(j)
            self.pd_multi_selected = sel
        else:
            self.pd_multi_selected = {j}
            self.pd_selected_detection = j
            self._pd3_update_selected_particle_view()
        self._pd3_draw_overlay()

def _pd3_draw_overlay(self):
    for a in getattr(self, 'pd_overlay_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self.pd_overlay_artists = []
    if not getattr(self, 'pd_show_centers_var', tk.BooleanVar(value=True)).get():
        self.pd_canvas.draw_idle()
        return
    selected = set(getattr(self, 'pd_multi_selected', set()) or set())
    for i, d in enumerate(getattr(self, 'pd_current_detections', []) or []):
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            c = plt.Circle((d['x'], d['y']), d['diameter_px'] / 2, fill=False, color='yellow' if i in selected else 'cyan', lw=2.0, zorder=20)
            ax.add_patch(c)
            self.pd_overlay_artists.append(c)
            t = ax.text(d['x'], d['y'], str(i + 1), fontsize=8, fontweight='bold', ha='center', va='center', color='yellow' if i in selected else 'cyan', zorder=21)
            self.pd_overlay_artists.append(t)
    j = getattr(self, 'pd_selected_detection', None)
    if j is not None and 0 <= j < len(getattr(self, 'pd_current_detections', [])):
        d = self.pd_current_detections[j]
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            p = ax.plot([d['x']], [d['y']], marker='o', ms=7, mfc='none', mec='yellow', mew=1.6, zorder=25)[0]
            self.pd_overlay_artists.append(p)
    self.pd_canvas.draw_idle()

def _pd3_delete_selected(self):
    sel = sorted(set(getattr(self, 'pd_multi_selected', set()) or set()), reverse=True)
    dets = getattr(self, 'pd_current_detections', []) or []
    if not sel and getattr(self, 'pd_selected_detection', None) is not None:
        sel = [int(self.pd_selected_detection)]
    if not sel:
        self.pd_status_var.set('Select one or more particles first (Ctrl + left-click for multiple).')
        return
    idx = int(self.pd_frame_var.get())
    nnc_mode = bool(self.pd_nnc_var.get())
    deleted = []
    for j in sel:
        if 0 <= j < len(dets):
            deleted.append(dict(dets[j]))
    if nnc_mode:
        ex = self.pd_nnc_exclusions.setdefault(idx, [])
        for d in deleted:
            ex.append((float(d['x']), float(d['y'])))
        self.pd_last_deleted = None
        msg = f'Added {len(deleted)} particle(s) to NNC-Delete exclusion for frame {idx + 1}. They will be ignored in future analyses.'
    else:
        self.pd_last_deleted = (idx, deleted)
        for j in sel:
            if 0 <= j < len(dets):
                dets.pop(j)
        self.pd_current_detections = dets
        self.pd_frame_detections[idx] = [dict(d) for d in dets]
        msg = f'Deleted {len(deleted)} particle(s) from frame {idx + 1}.'
    self.pd_multi_selected = set()
    self.pd_selected_detection = None
    self._pd3_draw_overlay()
    self._pd3_refresh_density_row(idx)
    self.pd_status_var.set(msg)

def _pd3_undo_delete(self):
    last = getattr(self, 'pd_last_deleted', None)
    if not last:
        self.pd_status_var.set('No normal deletion to undo. NNC-Delete exclusions are intentionally persistent.')
        return
    idx, deleted = last
    cur = [dict(d) for d in self.pd_frame_detections.get(idx, [])]
    cur.extend(deleted)
    cur.sort(key=lambda d: d.get('x', 0))
    self.pd_frame_detections[idx] = cur
    if int(self.pd_frame_var.get()) == idx:
        self.pd_current_detections = [dict(d) for d in cur]
    self.pd_last_deleted = None
    self._pd3_draw_overlay()
    self._pd3_refresh_density_row(idx)
    self.pd_status_var.set(f'Restored {len(deleted)} particle(s) in frame {idx + 1}.')

def _pd3_clear_nnc(self):
    self.pd_nnc_exclusions = {}
    self.pd_status_var.set('NNC-Delete exclusions cleared. Re-run analysis to restore excluded detections.')
    self._pd3_analyze_current()

def _pd3_build_tab(self):
    self.pd_frame_var = tk.IntVar(value=0)
    self.pd_frame_label = None
    self.pd_polarity_var = tk.StringVar(value='both')
    self.pd_min_diam_var = tk.DoubleVar(value=10.0)
    self.pd_max_diam_var = tk.DoubleVar(value=50.0)
    self.pd_threshold_var = tk.DoubleVar(value=0.1)
    self.pd_min_sep_var = tk.DoubleVar(value=50.0)
    self.pd_max_aspect_var = tk.DoubleVar(value=1.8)
    self.pd_include_edge_var = tk.BooleanVar(value=False)
    self.pd_show_centers_var = tk.BooleanVar(value=True)
    self.pd_nnc_var = tk.BooleanVar(value=False)
    self.pd_status_var = tk.StringVar(value='Waiting for a confirmed Secondary ROI from Tab 5 / Secondary ROI workflow.')
    self.pd_auto_shape_text = 'Auto shape gates: --'
    self.pd_results = {}
    self.pd_current_detections = []
    self.pd_frame_detections = {}
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self.pd_overlay_artists = []
    self.pd_last_deleted = None
    self.pd_nnc_exclusions = {}
    self.pd_selected_particle = None
    self.pd_zoomed = False
    self._pd_zoom_limits = None
    for w in self.tab_particle_density.winfo_children():
        w.destroy()
    outer = ttk.Frame(self.tab_particle_density, padding=5)
    outer.pack(fill='both', expand=True)
    outer.columnconfigure(0, weight=5)
    outer.columnconfigure(1, weight=4)
    outer.columnconfigure(2, weight=3)
    outer.rowconfigure(1, weight=1)
    outer.rowconfigure(2, weight=1)
    head = ttk.Frame(outer)
    head.grid(row=0, column=0, columnspan=3, sticky='ew', pady=(0, 4))
    ttk.Label(head, text='SKYRMION / PARTICLE — THRESHOLD + SEGMENTATION + SHAPE ANALYSIS', font=('Arial', 12, 'bold')).pack(side='left')
    ttk.Label(head, text='Source: confirmed Secondary ROI', foreground='darkgreen').pack(side='left', padx=12)
    ttk.Label(head, text='Frame:').pack(side='left', padx=(18, 3))
    self.pd_frame_slider = tk.Scale(head, from_=0, to=0, orient='horizontal', variable=self.pd_frame_var, resolution=1, length=250, showvalue=True)
    self.pd_frame_slider.pack(side='left')
    self.pd_frame_label = ttk.Label(head, text='--', width=22)
    self.pd_frame_label.pack(side='left', padx=4)
    ttk.Button(head, text='RESET ZOOM', command=self._pd_reset_zoom).pack(side='left', padx=2)
    ttk.Button(head, text='ANALYZE CURRENT FRAME', command=self._pd_analyze_current).pack(side='left', padx=2)
    ttk.Button(head, text='ANALYZE SELECTED FRAMES', command=self._pd_analyze_selected_frames).pack(side='left', padx=2)
    ttk.Button(head, text='DELETE SELECTED', command=self._pd_delete_selected).pack(side='left', padx=2)
    ttk.Button(head, text='UNDO DELETE', command=self._pd_undo_delete).pack(side='left', padx=2)
    roi_box = ttk.LabelFrame(outer, text='SELECTED ROI — GRAYSCALE + COLOR', padding=2)
    roi_box.grid(row=1, column=0, rowspan=2, sticky='nsew', padx=(0, 4))
    ctrl_box = ttk.LabelFrame(outer, text='DETECTION / ANALYSIS CONTROLS', padding=2)
    ctrl_box.grid(row=1, column=2, rowspan=2, sticky='nsew', padx=(4, 0))
    three_box = ttk.LabelFrame(outer, text='SELECTED PARTICLE — 3D INTENSITY', padding=2)
    three_box.grid(row=1, column=1, sticky='nsew', padx=2, pady=(0, 3))
    fwhm_box = ttk.LabelFrame(outer, text='4-DIRECTION FWHM', padding=2)
    fwhm_box.grid(row=2, column=1, sticky='nsew', padx=2, pady=(3, 0))
    self.pd_fig, (self.pd_ax_gray, self.pd_ax_color) = plt.subplots(1, 2, figsize=(8.0, 4.6), dpi=100, constrained_layout=True)
    self.pd_image_artist_gray = self.pd_ax_gray.imshow(np.zeros((10, 10)), cmap='gray', origin='lower', interpolation='nearest')
    cmap = getattr(self, 'colormap_var', tk.StringVar(value='gray')).get() or 'gray'
    self.pd_image_artist_color = self.pd_ax_color.imshow(np.zeros((10, 10)), cmap=cmap, origin='lower', interpolation='nearest')
    for ax in (self.pd_ax_gray, self.pd_ax_color):
        ax.set_xlabel('X pixel')
        ax.set_ylabel('Y pixel')
    try:
        self.pd_colorbar = self.pd_fig.colorbar(self.pd_image_artist_color, ax=self.pd_ax_color, fraction=0.046, pad=0.04)
        self.pd_colorbar.set_label('Intensity')
    except Exception:
        self.pd_colorbar = None
    self.pd_canvas = FigureCanvasTkAgg(self.pd_fig, master=roi_box)
    self.pd_canvas.draw()
    self.pd_canvas.get_tk_widget().pack(fill='both', expand=True)
    self.pd_canvas.mpl_connect('scroll_event', self._pd_zoom_scroll)
    self.pd_canvas.mpl_connect('button_press_event', self._pd_select_detection)
    self.pd_3d_fig = plt.Figure(figsize=(5.2, 2.8), dpi=100)
    self.pd_3d_ax = self.pd_3d_fig.add_subplot(111, projection='3d')
    self.pd_3d_canvas = FigureCanvasTkAgg(self.pd_3d_fig, master=three_box)
    self.pd_3d_canvas.draw()
    self.pd_3d_canvas.get_tk_widget().pack(fill='both', expand=True)
    self.pd_fwhm_fig = plt.Figure(figsize=(5.2, 2.8), dpi=100)
    self.pd_fwhm_ax = self.pd_fwhm_fig.add_subplot(111)
    self.pd_fwhm_canvas = FigureCanvasTkAgg(self.pd_fwhm_fig, master=fwhm_box)
    self.pd_fwhm_canvas.draw()
    self.pd_fwhm_canvas.get_tk_widget().pack(fill='both', expand=True)
    cc = tk.Canvas(ctrl_box, highlightthickness=0, borderwidth=0)
    sy = ttk.Scrollbar(ctrl_box, orient='vertical', command=cc.yview)
    cf = ttk.Frame(cc, padding=(6, 4, 10, 8))
    win = cc.create_window((0, 0), window=cf, anchor='nw')
    cc.configure(yscrollcommand=sy.set)
    cc.pack(side='left', fill='both', expand=True)
    sy.pack(side='right', fill='y')
    cf.bind('<Configure>', lambda e: cc.configure(scrollregion=cc.bbox('all')))
    cc.bind('<Configure>', lambda e: cc.itemconfigure(win, width=e.width))

    def cw(e):
        if getattr(e, 'delta', 0):
            cc.yview_scroll(int(-e.delta / 120), 'units')
        elif getattr(e, 'num', None) == 4:
            cc.yview_scroll(-3, 'units')
        elif getattr(e, 'num', None) == 5:
            cc.yview_scroll(3, 'units')
        return 'break'
    cc.bind('<MouseWheel>', cw)
    cf.bind('<MouseWheel>', cw)
    cc.bind('<Button-4>', cw)
    cc.bind('<Button-5>', cw)
    cf.bind('<Button-4>', cw)
    cf.bind('<Button-5>', cw)
    ttk.Label(cf, text='DETECTION PARAMETERS', font=('Arial', 11, 'bold')).pack(anchor='w', pady=(1, 5))
    ttk.Label(cf, text='Polarity').pack(anchor='w')
    ttk.Combobox(cf, textvariable=self.pd_polarity_var, values=['bright', 'dark', 'both'], state='readonly', width=12).pack(anchor='w')

    def slabel(text, var, a, b, res=0.01):
        ttk.Label(cf, text=text).pack(anchor='w', pady=(7, 0))
        tk.Scale(cf, from_=a, to=b, orient='horizontal', resolution=res, variable=var, length=260, showvalue=True).pack(fill='x')
    slabel('Detection threshold', self.pd_threshold_var, 0.01, 0.5, 0.01)
    slabel('Maximum aspect ratio', self.pd_max_aspect_var, 1.0, 3.0, 0.05)
    ttk.Label(cf, text='Minimum diameter (px): 10   |   Maximum diameter (px): 50').pack(anchor='w', pady=(6, 0))
    ttk.Label(cf, text='Minimum separation (px): 50').pack(anchor='w')
    ttk.Checkbutton(cf, text='Include edge detections', variable=self.pd_include_edge_var).pack(anchor='w', pady=(6, 0))
    ttk.Checkbutton(cf, text='Show detected circles / centres', variable=self.pd_show_centers_var, command=self._pd3_draw_overlay).pack(anchor='w')
    ttk.Checkbutton(cf, text='NNC-Delete — No Need to Consider', variable=self.pd_nnc_var).pack(anchor='w')
    self.pd_auto_shape_label = ttk.Label(cf, text=self.pd_auto_shape_text, wraplength=300, justify='left')
    self.pd_auto_shape_label.pack(anchor='w', pady=(6, 0))
    ttk.Separator(cf).pack(fill='x', pady=7)
    ttk.Label(cf, text='SELECTED PARTICLE', font=('Arial', 10, 'bold')).pack(anchor='w')
    self.pd_particle_status = ttk.Label(cf, text='Click an accepted particle/centre in the ROI.', wraplength=300, justify='left')
    self.pd_particle_status.pack(anchor='w', pady=(3, 6))
    ttk.Label(cf, text='Start / End frame for selected-frame analysis').pack(anchor='w')
    rf = ttk.Frame(cf)
    rf.pack(fill='x', pady=(2, 5))
    self.pd_range_start_var = tk.IntVar(value=1)
    self.pd_range_end_var = tk.IntVar(value=1)
    ttk.Label(rf, text='Start').pack(side='left')
    self.pd_range_start_spin = tk.Spinbox(rf, from_=1, to=1, width=6, textvariable=self.pd_range_start_var, increment=1)
    self.pd_range_start_spin.pack(side='left', padx=3)
    ttk.Label(rf, text='End').pack(side='left')
    self.pd_range_end_spin = tk.Spinbox(rf, from_=1, to=1, width=6, textvariable=self.pd_range_end_var, increment=1)
    self.pd_range_end_spin.pack(side='left', padx=3)
    ttk.Button(cf, text='CLEAR NNC EXCLUSIONS', command=self._pd_clear_nnc).pack(fill='x', pady=2)
    ttk.Button(cf, text='EXPORT TABLE', command=self._pd_export_table).pack(fill='x', pady=2)
    ttk.Button(cf, text='EXPORT SELECTED ROI — TIFF', command=self._pd_export_tiff).pack(fill='x', pady=2)
    self.pd_status_label = ttk.Label(cf, textvariable=self.pd_status_var, wraplength=300, justify='left')
    self.pd_status_label.pack(anchor='w', pady=6)
    bottom_graph = ttk.LabelFrame(outer, text='COUNT / DENSITY', padding=2)
    bottom_graph.grid(row=3, column=0, columnspan=2, sticky='nsew', pady=(4, 0))
    outer.rowconfigure(3, weight=1)
    self.pd_plot_fig = plt.Figure(figsize=(8, 2.5), dpi=100)
    self.pd_plot_ax = self.pd_plot_fig.add_axes([0.1, 0.2, 0.68, 0.62])
    self.pd_plot_ax2 = self.pd_plot_fig.add_axes(self.pd_plot_ax.get_position(), frameon=False)
    self.pd_plot_ax2.patch.set_visible(False)
    self.pd_plot_ax2.set_zorder(1)
    self.pd_plot_ax.set_zorder(2)
    self.pd_plot_canvas = FigureCanvasTkAgg(self.pd_plot_fig, master=bottom_graph)
    self.pd_plot_canvas.draw()
    self.pd_plot_canvas.get_tk_widget().pack(fill='both', expand=True)
    table_box = ttk.LabelFrame(outer, text='DENSITY TABLE', padding=2)
    table_box.grid(row=3, column=2, sticky='nsew', padx=(4, 0), pady=(4, 0))
    tf = ttk.Frame(table_box)
    tf.pack(fill='both', expand=True)
    cols = ('Frame', 'Field (mT)', 'Count', 'ROI area (µm²)', 'Density (skyrmions/µm²)', 'Mean d (px)', 'Mean FWHM (px)', 'Mean Shape AR')
    self.pd_table = ttk.Treeview(tf, columns=cols, show='headings', height=8)
    widths = (52, 86, 55, 90, 120, 72, 95, 85)
    for c, wd in zip(cols, widths):
        self.pd_table.heading(c, text=c)
        self.pd_table.column(c, width=wd, anchor='center')
        sb = ttk.Scrollbar(tf, orient='vertical', command=self.pd_table.yview)
        self.pd_table.configure(yscrollcommand=sb.set)
        self.pd_table.pack(side='left', fill='both', expand=True)
        sb.pack(side='right', fill='y')

    def arm_slider(e):
        self.pd_frame_slider.focus_set()
        self.pd_frame_slider._pd3_armed = True
    self.pd_frame_slider._pd3_armed = False
    self.pd_frame_slider.bind('<Button-1>', arm_slider, add='+')
    self.pd_frame_slider.bind('<MouseWheel>', self._pd_frame_wheel, add='+')
    self.pd_frame_slider.bind('<Button-4>', self._pd_frame_wheel, add='+')
    self.pd_frame_slider.bind('<Button-5>', self._pd_frame_wheel, add='+')
    self.pd_frame_slider.configure(command=lambda *_: self._pd3_frame_changed())
    try:
        self._pd3_update_roi_image(preserve_zoom=False)
    except Exception:
        pass
    if hasattr(self, 'roi2_stack') and len(getattr(self, 'roi2_stack', []) or []):
        n = len(self.roi2_stack)
        self.pd_range_start_spin.configure(to=n)
        self.pd_range_end_spin.configure(to=n)
        self.pd_range_end_var.set(n)
    self._pd3_update_density_plot()
CDIWorkflowApp._pd3_field_value = _pd3_field_value
CDIWorkflowApp._pd3_draw_overlay = _pd3_draw_overlay
CDIWorkflowApp._pd3_frame_changed = _pd3_frame_changed
CDIWorkflowApp._pd3_update_density_plot = _pd3_update_density_plot
CDIWorkflowApp._pd3_update_roi_image = _pd3_update_roi_image
CDIWorkflowApp._pd3_clear_nnc = _pd3_clear_nnc

def _pd3_export_table(self):
    if not getattr(self, 'pd_results', None):
        self._pd_analyze_current()
    if not getattr(self, 'pd_results', None):
        return
    fp = filedialog.asksaveasfilename(parent=self.root, initialdir=self._export_initialdir(), title='Export skyrmion analysis table', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('Excel', '*.xlsx')])
    if not fp:
        return
    cols = ['Frame', 'Field (mT)', 'Count', 'ROI area (µm²)', 'Density (skyrmions/µm²)', 'Mean diameter (px)', 'Mean FWHM (px)', 'Mean Shape AR']
    df = pd.DataFrame([self.pd_results[k] for k in sorted(self.pd_results)], columns=cols)
    if fp.lower().endswith('.xlsx'):
        df.to_excel(fp, index=False)
    else:
        df.to_csv(fp, index=False)
    self.pd_status_var.set(f'Exported table: {fp}')

def _pd3_export_tiff(self):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        messagebox.showwarning('Export ROI', 'Select/confirm the Secondary ROI first.', parent=self.root)
        return
    idx = max(0, min(int(self.pd_frame_var.get()), len(stack) - 1))
    fp = filedialog.asksaveasfilename(parent=self.root, initialdir=self._export_initialdir(), title='Export selected ROI as TIFF (300 PPI)', defaultextension='.tiff', filetypes=[('TIFF image', '*.tif *.tiff')])
    if not fp:
        return
    arr = np.asarray(stack[idx], dtype=float)
    arr = self._pd3_normalize(arr)
    Image.fromarray(np.rint(arr * 65535).astype(np.uint16)).save(fp, format='TIFF', dpi=(300, 300))
    self.pd_status_var.set(f'Exported 300 PPI ROI TIFF: {fp}')

def _tab6_export_folder_open(self, folder):
    """Open the export folder in the native file browser."""
    try:
        folder = os.path.abspath(os.path.expanduser(str(folder)))
        if os.name == 'nt':
            os.startfile(folder)
        elif sys.platform == 'darwin':
            import subprocess
            subprocess.Popen(['open', folder])
        else:
            import subprocess
            subprocess.Popen(['xdg-open', folder])
    except Exception as exc:
        try:
            self.log(f'Could not open export folder: {exc}')
        except Exception:
            pass

def _tab6_export_display(self, idx):
    """Return the exact processed/display image plus view metadata for export."""
    image = _proc_image(self, idx)
    view = str(self.view_var.get())
    if view == 'Image':
        display = image
    elif view == 'FFT':
        _, display = _proc_fft(self, image)
    else:
        F, _ = _proc_fft(self, image)
        Fm, _ = _proc_fft_mask(self, F, self.fft_mask_var.get())
        r = np.abs(np.fft.ifft2(np.fft.ifftshift(Fm)))
        mn, mx = (float(r.min()), float(r.max()))
        display = (r - mn) / (mx - mn + 1e-12)
        display = np.where(self.local_mask, display, 0.0)
    return (np.asarray(display, dtype=float), view)

def _tab6_export_figure(self, idx, dpi=None):
    """Build a standalone Tab-6 export figure with axes + colorbar."""
    if dpi is None:
        try:
            dpi = int(self.proc_export_dpi_var.get())
        except Exception:
            dpi = 300
    dpi = int(dpi)
    display, view = _tab6_export_display(self, idx)
    h, w = display.shape[:2]
    base = None
    try:
        base = float(self._proc_get_tab5_base_pixel_size())
    except Exception:
        pass
    if base is not None and np.isfinite(base) and (base > 0):
        ow = int(getattr(self, 'original_roi_w', w) or w)
        oh = int(getattr(self, 'original_roi_h', h) or h)
        extent = (0.0, float(ow) * base, 0.0, float(oh) * base)
        xlabel, ylabel = ('X (nm)', 'Y (nm)')
    else:
        extent = (0.0, float(w), 0.0, float(h))
        xlabel, ylabel = ('X (pixel)', 'Y (pixel)')
    cmap_name = 'gray'
    try:
        cmap_name = str(self.colormap_var.get()).strip() or 'gray'
    except Exception:
        pass
    field = float(self.real_fields_mT[idx])
    fig = plt.Figure(figsize=(8.0, 6.8), dpi=dpi, facecolor='white')
    ax = fig.add_axes([0.11, 0.11, 0.74, 0.78])
    im = ax.imshow(display, origin='lower', extent=extent, interpolation='nearest', aspect='equal', cmap=cmap_name, vmin=0.0, vmax=1.0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'Tab 6 — {view} | Frame {idx + 1}/{len(self.recons)} | Field = {field:+.2f} mT')
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_aspect('equal', adjustable='box')
    cax = fig.add_axes([0.88, 0.11, 0.035, 0.78])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label('Intensity')
    return fig

def _tab6_save_single_with_axes(self, extension, fmt):
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Resolution Required', 'Apply SAME CONFIRMED ROI resolution first.', parent=self.root)
        return
    try:
        dpi = int(self.proc_export_dpi_var.get())
    except Exception:
        dpi = 300
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Export DPI', 'Select 150, 300, 600 or 900 DPI.', parent=self.root)
        return
    try:
        folder = os.path.abspath(os.path.expanduser(str(self.output_var.get()).strip() or 'ROI_exports'))
        os.makedirs(folder, exist_ok=True)
        i = max(0, min(int(round(float(self.proc_idx_var.get()))), len(self.recons) - 1))
        field = float(self.real_fields_mT[i])
        path = os.path.join(folder, f'ROI_{i:03d}_{field:+.2f}mT{extension}')
        fig = _tab6_export_figure(self, i, dpi=dpi)
        save_kwargs = dict(dpi=dpi, format=fmt, facecolor='white', bbox_inches='tight')
        if fmt == 'JPEG':
            save_kwargs['pil_kwargs'] = {'quality': 95, 'subsampling': 0}
        elif fmt == 'TIFF':
            save_kwargs['pil_kwargs'] = {'compression': 'tiff_lzw'}
        fig.savefig(path, **save_kwargs)
        plt.close(fig)
        self.proc_status_var.set(f'Saved with axes + colorbar: {path} | {dpi} DPI')
        self.log(f'Tab 6 single export complete: {path} | axes+colorbar | {dpi} DPI')
        self._tab6_export_folder_open(folder)
    except Exception as exc:
        try:
            plt.close(fig)
        except Exception:
            pass
        self.proc_status_var.set('Single-image export failed.')
        self.log(f'Tab 6 single export failed: {exc}')
        messagebox.showerror('Export Error', str(exc), parent=self.root)

def _tab6_batch_with_axes(self, extension, fmt):
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Resolution Required', 'Apply SAME CONFIRMED ROI resolution first.', parent=self.root)
        return
    try:
        dpi = int(self.proc_export_dpi_var.get())
    except Exception:
        dpi = 300
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Export DPI', 'Select 150, 300, 600 or 900 DPI.', parent=self.root)
        return
    folder = os.path.abspath(os.path.expanduser(str(self.output_var.get()).strip() or 'ROI_exports'))
    os.makedirs(folder, exist_ok=True)
    fig = None
    try:
        total = len(self.recons)
        for i in range(total):
            field = float(self.real_fields_mT[i])
            path = os.path.join(folder, f'ROI_{i:03d}_{field:+.2f}mT{extension}')
            fig = _tab6_export_figure(self, i, dpi=dpi)
            save_kwargs = dict(dpi=dpi, format=fmt, facecolor='white', bbox_inches='tight')
            if fmt == 'JPEG':
                save_kwargs['pil_kwargs'] = {'quality': 95, 'subsampling': 0}
            elif fmt == 'TIFF':
                save_kwargs['pil_kwargs'] = {'compression': 'tiff_lzw'}
            fig.savefig(path, **save_kwargs)
            plt.close(fig)
            fig = None
            self.proc_status_var.set(f'Exporting {fmt} with axes + colorbar: {i + 1}/{total} | {dpi} DPI')
            self.root.update_idletasks()
        self.proc_status_var.set(f'Batch {fmt} export complete | {total} images | axes + colorbar | {dpi} DPI')
        self.log(f'Tab 6 batch export complete: format={fmt} | count={total} | axes+colorbar | {dpi} DPI | folder={folder}')
        self._tab6_export_folder_open(folder)
    except Exception as exc:
        try:
            if fig is not None:
                plt.close(fig)
        except Exception:
            pass
        self.proc_status_var.set('Batch export failed.')
        self.log(f'Tab 6 batch export failed: {exc}')
        messagebox.showerror('Batch Export Error', str(exc), parent=self.root)

def _tab6_mp4_with_axes(self, indices, filename):
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Resolution Required', 'Apply SAME CONFIRMED ROI first.', parent=self.root)
        return
    imageio = _proc_imageio(self)
    if imageio is None:
        return
    try:
        dpi = int(self.proc_export_dpi_var.get())
    except Exception:
        dpi = 300
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Export DPI', 'Select 150, 300, 600 or 900 DPI.', parent=self.root)
        return
    folder = os.path.abspath(os.path.expanduser(str(self.output_var.get()).strip() or 'ROI_exports'))
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, filename)
    indices = list(indices)
    if not indices:
        messagebox.showwarning('MP4 Export', 'No frames selected.', parent=self.root)
        return
    fps = max(1, int(self.fps_var.get()))
    writer = None
    fig = None
    try:
        fig = None
        errors = []
        for codec in ('libx264', 'mpeg4', None):
            try:
                kwargs = {'fps': fps, 'format': 'ffmpeg', 'macro_block_size': None}
                if codec is not None:
                    kwargs['codec'] = codec
                writer = imageio.get_writer(path, **kwargs)
                break
            except Exception as exc:
                errors.append(f"{codec or 'default'}: {exc}")
        if writer is None:
            raise RuntimeError('Could not initialize an MP4 encoder.\n\n' + '\n'.join(errors))
        total = len(indices)
        for count, i in enumerate(indices, start=1):
            fig = _tab6_export_figure(self, i, dpi=dpi)
            fig.canvas.draw()
            frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            h, w = frame.shape[:2]
            if h % 2 or w % 2:
                frame = np.pad(frame, ((0, h % 2), (0, w % 2), (0, 0)), mode='edge')
            writer.append_data(frame)
            plt.close(fig)
            fig = None
            self.proc_status_var.set(f'MP4: {count}/{total} | axes + colorbar | {fps} FPS | {dpi} DPI')
            self.root.update_idletasks()
    except Exception as exc:
        if fig is not None:
            try:
                plt.close(fig)
            except Exception:
                pass
        self.proc_status_var.set('MP4 export failed.')
        self.log(f'Tab 6 MP4 export failed: {path} | {exc}')
        messagebox.showerror('MP4 Error', str(exc), parent=self.root)
        return
    finally:
        try:
            if writer is not None:
                writer.close()
        except Exception:
            pass
        try:
            if fig is not None:
                plt.close(fig)
        except Exception:
            pass
    self.proc_status_var.set(f'MP4 saved with axes + colorbar: {path} | {fps} FPS | {dpi} DPI')
    self.log(f'Tab 6 MP4 saved: {path} | frames={len(indices)} | axes+colorbar | fps={fps} | dpi={dpi}')
    self._tab6_export_folder_open(folder)

def _tab6_single_mp4_with_axes(self):
    i = int(self.proc_idx_var.get())
    frames = max(1, int(self.fps_var.get()) * 2)
    _tab6_mp4_with_axes(self, [i] * frames, f'ROI_single_{i:03d}_{self.real_fields_mT[i]:+.2f}mT.mp4')

def _tab6_batch_mp4_with_axes(self):
    try:
        indices, start, end, fps = _proc_get_batch_mp4_indices(self)
        self.fps_var.set(fps)
        _tab6_mp4_with_axes(self, indices, f'ROI_frames_{start:04d}-{end:04d}_{fps}fps_axes.mp4')
    except Exception as exc:
        messagebox.showerror('Batch MP4 Export', str(exc), parent=self.root)
CDIWorkflowApp.save_single = _tab6_save_single_with_axes
CDIWorkflowApp.batch_export = _tab6_batch_with_axes
CDIWorkflowApp.create_mp4 = _tab6_mp4_with_axes
CDIWorkflowApp.export_single_mp4 = _tab6_single_mp4_with_axes
CDIWorkflowApp.export_batch_mp4 = _tab6_batch_mp4_with_axes

def _final_common_export_folder_only_tab1(self):
    if not hasattr(self, '_export_folder_var'):
        self._export_folder_var = tk.StringVar(value=os.path.abspath(getattr(self, 'output_dir', os.getcwd())))
    self.output_var = self._export_folder_var

    def walk(widget):
        yield widget
        try:
            for child in widget.winfo_children():
                yield from walk(child)
        except Exception:
            pass
    old_output_var = getattr(self, 'output_var', None)
    for widget in list(walk(self.tab_proc)):
        try:
            if isinstance(widget, (ttk.LabelFrame, tk.LabelFrame)):
                text = str(widget.cget('text')).strip().upper()
                if 'EXPORT FOLDER' in text:
                    widget.destroy()
                    continue
            if isinstance(widget, (ttk.Label, tk.Label)):
                text = str(widget.cget('text')).strip().upper()
                if text in ('EXPORT FOLDER', 'EXPORT FOLDER PATH:', 'EXPORT FOLDER PATH'):
                    widget.destroy()
                    continue
            if isinstance(widget, (ttk.Entry, tk.Entry)):
                try:
                    tv = str(widget.cget('textvariable'))
                except Exception:
                    tv = ''
                if old_output_var is not None and tv == str(old_output_var):
                    widget.destroy()
                    continue
            if isinstance(widget, (ttk.Button, tk.Button)):
                text = str(widget.cget('text')).strip().upper()
                if text == 'SELECT FOLDER':
                    widget.destroy()
                    continue
        except Exception:
            pass
    box = getattr(self, '_tab8_export_folder_box', None)
    if box is not None:
        try:
            box.destroy()
        except Exception:
            pass
    self._tab8_export_folder_box = None

    def _common_tab8_export_initialdir():
        try:
            folder = self._export_folder_var.get().strip()
            if folder and os.path.isdir(folder):
                return folder
        except Exception:
            pass
        return os.getcwd()
    self._tab8_export_initialdir = _common_tab8_export_initialdir
    try:
        if hasattr(self, '_tab8_export_folder_var'):
            delattr(self, '_tab8_export_folder_var')
    except Exception:
        pass
_ORIG_CONFIRM_ROI2_FOR_PD = CDIWorkflowApp.confirm_roi2

def _confirm_roi2_and_refresh_tab7(self, *args, **kwargs):
    result = _ORIG_CONFIRM_ROI2_FOR_PD(self, *args, **kwargs)
    try:
        stack = getattr(self, 'roi2_stack', None)
        if stack is not None and len(stack) > 0 and hasattr(self, 'pd_frame_var'):
            idx = int(getattr(self, 'roi2_idx_var', tk.IntVar(value=0)).get())
            idx = max(0, min(idx, len(stack) - 1))
            self.pd_frame_var.set(idx)
            self._pd3_update_roi_image(preserve_zoom=False)
            if hasattr(self, '_pd3_load_current_detections'):
                self._pd3_load_current_detections()
            if hasattr(self, 'pd_status_var'):
                self.pd_status_var.set(f'Confirmed Secondary ROI loaded from Tab 5 | Frame {idx + 1}/{len(stack)}. Click ANALYZE CURRENT FRAME to detect particles.')
    except Exception as exc:
        try:
            if hasattr(self, 'pd_status_var'):
                self.pd_status_var.set(f'Tab 7 ROI refresh warning: {exc}')
        except Exception:
            pass
    return result
CDIWorkflowApp.confirm_roi2 = _confirm_roi2_and_refresh_tab7

def _pd7_normalize_for_detection(self, image):
    a = np.asarray(image, dtype=float)
    a = np.squeeze(a)
    if a.ndim != 2:
        raise ValueError('ROI image must be 2-D')
    finite = np.isfinite(a)
    if not np.any(finite):
        return np.zeros_like(a, dtype=float)
    med = float(np.nanmedian(a[finite]))
    vals = a[finite]
    lo, hi = np.percentile(vals, [1.0, 99.0])
    if hi <= lo:
        lo, hi = (float(np.min(vals)), float(np.max(vals)))
    if hi <= lo:
        return np.zeros_like(a, dtype=float)
    return np.clip((a - lo) / (hi - lo), 0.0, 1.0)

def _pd7_detect_frame(self, image):
    """Detect outer particles first, then classify using an inner contrast core.

    Outer region = yellow particle candidate from thresholded, denoised,
    background-corrected image.
    Core region = green sub-region using a relative contrast level inside each
    candidate.  The core major/minor-axis ratio is used for the final
    particle-vs-worm decision.
    """
    from scipy import ndimage as ndi
    from skimage.measure import regionprops, find_contours
    from skimage.morphology import remove_small_objects, closing, opening, disk
    z = self._pd7_normalize_for_detection(image)
    try:
        thr = float(np.clip(self.pd_threshold_var.get(), 0.01, 0.5))
    except Exception:
        thr = 0.1
    try:
        core_level = float(np.clip(self.pd_core_contrast_var.get(), 0.1, 0.95))
    except Exception:
        core_level = 0.55
    try:
        dmin = float(self.pd_min_diam_var.get())
    except Exception:
        dmin = 10.0
    try:
        dmax = float(self.pd_max_diam_var.get())
    except Exception:
        dmax = 50.0
    dmin = max(3.0, dmin)
    dmax = max(dmin, dmax)
    try:
        min_sep = max(1.0, float(self.pd_min_sep_var.get()))
    except Exception:
        min_sep = 50.0
    try:
        core_ar_max = max(1.0, float(self.pd_max_aspect_var.get()))
    except Exception:
        core_ar_max = 1.8
    polarity = str(self.pd_polarity_var.get()).strip().lower()
    include_edge = bool(self.pd_include_edge_var.get()) if hasattr(self, 'pd_include_edge_var') else False
    h, w = z.shape
    med = ndi.median_filter(z, size=3, mode='nearest')
    smooth = ndi.gaussian_filter(med, sigma=1.0, mode='nearest')
    bg = ndi.gaussian_filter(smooth, sigma=max(6.0, 0.55 * dmax), mode='nearest')
    detail = smooth - bg
    pols = []
    if polarity in ('bright', 'both'):
        pols.append(('bright', detail))
    if polarity in ('dark', 'both'):
        pols.append(('dark', -detail))
    if not pols:
        pols = [('bright', detail)]
    candidates = []
    for pol, signal in pols:
        pos = signal[signal > 0]
        scale = float(np.percentile(pos, 99.0)) if pos.size else 1.0
        scale = max(scale, 1e-09)
        q = np.clip(signal / scale, 0.0, 1.0)
        mask = q >= thr
        mask = remove_small_objects(mask, max_size=max(11, int(np.pi * (dmin / 2.0) ** 2 * 0.12)-1))
        mask = opening(mask, disk(1))
        mask = closing(mask, disk(1))
        labels, n = ndi.label(mask)
        for rp in regionprops(labels, intensity_image=q):
            eqd = float(rp.equivalent_diameter_area)
            if not dmin <= eqd <= dmax:
                continue
            cy, cx = map(float, rp.centroid)
            edge = cx < eqd / 2 or cy < eqd / 2 or cx > w - 1 - eqd / 2 or (cy > h - 1 - eqd / 2)
            if edge and (not include_edge):
                continue
            rad = max(5, int(round(0.9 * eqd)))
            y0 = max(0, int(round(cy)) - rad)
            y1 = min(h, int(round(cy)) + rad + 1)
            x0 = max(0, int(round(cx)) - rad)
            x1 = min(w, int(round(cx)) + rad + 1)
            patch = q[y0:y1, x0:x1]
            cand = labels[y0:y1, x0:x1] == rp.label
            if np.count_nonzero(cand) < 8:
                continue
            bgvals = patch[~cand]
            inside = patch[cand]
            ib = float(np.median(bgvals)) if bgvals.size else 0.0
            ip = float(np.percentile(inside, 95)) if inside.size else float(np.max(inside))
            if ip <= ib + 1e-09:
                continue
            rel = (patch - ib) / (ip - ib)
            core_local = cand & (rel >= core_level)
            core_local = remove_small_objects(core_local, max_size=max(5, int(np.pi * (eqd * 0.12) ** 2)-1))
            if not np.any(core_local):
                continue
            core_labels, cn = ndi.label(core_local)
            core_props = regionprops(core_labels, intensity_image=rel)
            if not core_props:
                continue
            cr = max(core_props, key=lambda rr: (rr.area, float(rr.max_intensity) if hasattr(rr, 'max_intensity') else rr.area))
            c_area = float(cr.area)
            c_eqd = float(cr.equivalent_diameter_area)
            if c_eqd < 2.0:
                continue
            cmaj = float(cr.axis_major_length)
            cmin = float(cr.axis_minor_length)
            c_ar = cmaj / max(cmin, 1e-09)
            c_ecc = float(cr.eccentricity)
            c_sol = float(cr.solidity)
            ccy_rel, ccx_rel = map(float, cr.centroid)
            ccy = ccy_rel + y0
            ccx = ccx_rel + x0
            if c_ar > max(3.5, core_ar_max * 2.5):
                continue
            core_mask_full = np.zeros_like(mask, dtype=bool)
            core_full_local = core_labels == cr.label
            core_mask_full[y0:y1, x0:x1] = core_full_local
            try:
                cts = find_contours(labels == rp.label, 0.5)
                outer_cont = max(cts, key=len) if cts else np.empty((0, 2))
            except Exception:
                outer_cont = np.empty((0, 2))
            try:
                cts = find_contours(core_mask_full, 0.5)
                core_cont = max(cts, key=len) if cts else np.empty((0, 2))
            except Exception:
                core_cont = np.empty((0, 2))
            classification = 'particle' if c_ar <= core_ar_max else 'worm'
            score = float(c_area * max(0.0, 1.0 - abs(c_ar - 1.0)))
            candidates.append(dict(x=float(ccx), y=float(ccy), outer_x=float(cx), outer_y=float(cy), diameter_px=eqd, radius_px=eqd / 2.0, area_px=float(rp.area), core_diameter_px=c_eqd, core_area_px=c_area, core_aspect_ratio=c_ar, core_eccentricity=c_ecc, core_solidity=c_sol, aspect_ratio=c_ar, eccentricity=c_ecc, solidity=c_sol, circularity=float(4 * np.pi * rp.area / max(float(rp.perimeter) ** 2, 1e-09)), core_contrast=float(core_level), contrast=float(ip - ib), edge=bool(edge), polarity=pol, classification=classification, accepted=bool(classification == 'particle'), score=score, outer_contour=np.asarray(outer_cont, dtype=float), core_contour=np.asarray(core_cont, dtype=float), method='Threshold + Segmentation + Core Shape'))
    if not candidates:
        self.pd_auto_shape_text = f'Core classification: aspect ratio ≤ {core_ar_max:.2f} → particle | > {core_ar_max:.2f} → worm'
        return []
    idx = int(getattr(self, 'pd_frame_var', tk.IntVar(value=0)).get())
    nnc = getattr(self, 'pd_nnc_exclusions', {}).get(idx, []) or []
    filtered = []
    for d in candidates:
        if not d.get('accepted', False):
            continue
        if any((np.hypot(d['x'] - x0, d['y'] - y0) <= max(5.0, 0.45 * d['diameter_px']) for x0, y0 in nnc)):
            continue
        filtered.append(d)
    filtered.sort(key=lambda d: d.get('score', 0.0), reverse=True)
    kept = []
    for d in filtered:
        if all((np.hypot(d['x'] - k['x'], d['y'] - k['y']) >= min_sep for k in kept)):
            kept.append(d)
    kept.sort(key=lambda d: (d['y'], d['x']))
    self.pd_auto_shape_text = f'Core classifier: AR ≤ {core_ar_max:.2f} = particle | AR > {core_ar_max:.2f} = worm | core contrast level = {core_level:.2f}'
    return kept

def _pd7_update_selected_particle_view(self):
    if not getattr(self, 'pd_current_detections', None):
        try:
            self.pd_particle_status.config(text='No particle selected. Click an accepted particle/centre in the ROI.')
        except Exception:
            pass
        if hasattr(self, 'pd_fwhm_canvas'):
            self.pd_fwhm_ax.cla()
            self.pd_fwhm_canvas.draw_idle()
        return
    j = getattr(self, 'pd_selected_detection', None)
    if j is None or j < 0 or j >= len(self.pd_current_detections):
        return
    d = self.pd_current_detections[int(j)]
    idx = int(self.pd_frame_var.get())
    img = np.asarray(self.roi2_stack[idx], dtype=float)
    try:
        analysis = self._pd3_particle_analysis(d, img)
    except Exception as exc:
        self.pd_particle_status.config(text=f'FWHM calculation failed: {exc}')
        return
    d.update(analysis)
    self.pd_selected_particle = d
    self.pd_fwhm_ax.cla()
    for ang in (0.0, 45.0, 90.0, 135.0):
        key = f'fwhm_{int(ang)}'
        try:
            rr, pp = analysis['profiles'][ang]
            self.pd_fwhm_ax.plot(rr, pp, label=f'{int(ang)}°  FWHM={analysis[key]:.2f} px', linewidth=1.5)
        except Exception:
            pass
    self.pd_fwhm_ax.axvline(0.0, ls='--', lw=0.8)
    self.pd_fwhm_ax.set_xlabel('Distance from core center (pixel)', fontsize=8)
    self.pd_fwhm_ax.set_ylabel('Intensity', fontsize=8)
    self.pd_fwhm_ax.set_title(f"4-direction FWHM | Average FWHM = {analysis.get('fwhm_avg', np.nan):.2f} px", fontsize=9)
    self.pd_fwhm_ax.legend(fontsize=7, loc='best')
    self.pd_fwhm_ax.grid(alpha=0.22)
    self.pd_fwhm_canvas.draw_idle()
    self.pd_particle_status.config(text=f"Particle #{int(j) + 1}: core center=({d.get('x', np.nan):.2f}, {d.get('y', np.nan):.2f}) px | Outer diameter={d.get('diameter_px', np.nan):.2f} px | Core diameter={d.get('core_diameter_px', np.nan):.2f} px | Core AR={d.get('core_aspect_ratio', np.nan):.2f}\nFWHM: 0°={analysis.get('fwhm_0', np.nan):.2f}, 45°={analysis.get('fwhm_45', np.nan):.2f}, 90°={analysis.get('fwhm_90', np.nan):.2f}, 135°={analysis.get('fwhm_135', np.nan):.2f} | Average={analysis.get('fwhm_avg', np.nan):.2f} ± {analysis.get('fwhm_sd', np.nan):.2f} px")
    self._pd3_refresh_density_row(idx)

def _pd7_draw_overlay(self):
    for a in getattr(self, 'pd_overlay_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self.pd_overlay_artists = []
    if not bool(getattr(self, 'pd_show_centers_var', tk.BooleanVar(value=True)).get()):
        self.pd_canvas.draw_idle()
        return
    sel = set(getattr(self, 'pd_multi_selected', set()) or set())
    for i, d in enumerate(getattr(self, 'pd_current_detections', []) or []):
        outer = d.get('outer_contour', np.empty((0, 2)))
        core = d.get('core_contour', np.empty((0, 2)))
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            if np.size(outer):
                ln = ax.plot(outer[:, 1], outer[:, 0], '-', color='yellow' if i in sel else 'yellow', lw=2.2, zorder=20)[0]
                self.pd_overlay_artists.append(ln)
            if np.size(core):
                ln = ax.plot(core[:, 1], core[:, 0], '-', color='limegreen', lw=2.0, zorder=22)[0]
                self.pd_overlay_artists.append(ln)
            txt = ax.text(d['x'], d['y'], str(i + 1), fontsize=8, fontweight='bold', ha='center', va='center', color='yellow' if i in sel else 'yellow', zorder=24, bbox=dict(boxstyle='round,pad=0.15', fc='black', ec='yellow' if i in sel else 'yellow', alpha=0.55))
            self.pd_overlay_artists.append(txt)
            p = ax.plot([d['x']], [d['y']], marker='o', ms=5, mfc='none', mec='yellow', mew=1.3, zorder=25)[0]
            self.pd_overlay_artists.append(p)
    self.pd_canvas.draw_idle()

def _tab6_frame_slider_arm(self, event=None):
    try:
        self._line_profile_slider_armed = True
        if hasattr(self, 'line_profile_status_var'):
            self.line_profile_status_var.set('Frame slider armed — mouse wheel changes only the Tab 6 frame.')
    except Exception:
        pass
    return None

def _line_profile_frame_mousewheel_independent(self, event):
    """Change Tab 6 frame only when the pointer is over its dedicated slider and it has been clicked."""
    slider = getattr(self, 'line_profile_frame_slider', None)
    stack = getattr(self, 'roi2_stack', None)
    if slider is None or stack is None or len(stack) == 0:
        return 'break'
    if not getattr(self, '_line_profile_slider_armed', False):
        return 'break'
    try:
        px = slider.winfo_toplevel().winfo_pointerx()
        py = slider.winfo_toplevel().winfo_pointery()
        rx = slider.winfo_rootx()
        ry = slider.winfo_rooty()
        rw = slider.winfo_width()
        rh = slider.winfo_height()
        if not (rx <= px <= rx + rw and ry <= py <= ry + rh):
            return 'break'
    except Exception:
        return 'break'
    if getattr(event, 'num', None) == 4:
        step = 1
    elif getattr(event, 'num', None) == 5:
        step = -1
    else:
        step = 1 if getattr(event, 'delta', 0) > 0 else -1
    cur = int(self.line_profile_frame_var.get())
    new = max(0, min(len(stack) - 1, cur + step))
    if new != cur:
        self.line_profile_frame_var.set(new)
        self._line_profile_frame_slider_changed(new)
    return 'break'

def _line_profile_frame_slider_changed_independent(self, value=None):
    """Update only Tab 6. The confirmed Secondary ROI stack remains fixed."""
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        return
    try:
        idx = int(round(float(value if value is not None else self.line_profile_frame_var.get())))
    except Exception:
        idx = int(self.line_profile_frame_var.get())
    idx = max(0, min(idx, len(stack) - 1))
    self.line_profile_frame_var.set(idx)
    self._line_profile_update_image(idx, preserve_lines=True)
CDIWorkflowApp._line_profile_frame_mousewheel = _line_profile_frame_mousewheel_independent
CDIWorkflowApp._line_profile_frame_slider_changed = _line_profile_frame_slider_changed_independent
_OLD_TAB6_BUILD = CDIWorkflowApp._build_line_profile_tab
CDIWorkflowApp._tab6_frame_slider_arm = _tab6_frame_slider_arm

def _pd7_update_roi_image_core_only(self, preserve_zoom=True):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        try:
            self.pd_status_var.set('Waiting for confirmed Secondary ROI from Tab 5.')
        except Exception:
            pass
        return False
    try:
        idx = int(self.pd_frame_var.get())
    except Exception:
        idx = 0
    idx = max(0, min(idx, len(stack) - 1))
    self.pd_frame_var.set(idx)
    img = np.asarray(stack[idx], dtype=float)
    if img.ndim != 2 or img.size == 0:
        return False
    if not hasattr(self, 'pd_image_artist_gray') or not hasattr(self, 'pd_image_artist_color'):
        return False
    self.pd_image_artist_gray.set_data(img)
    self.pd_image_artist_color.set_data(img)
    try:
        cmap = self.colormap_var.get() or 'gray'
    except Exception:
        cmap = 'gray'
    try:
        self.pd_image_artist_color.set_cmap(cmap)
    except Exception:
        pass
    finite = img[np.isfinite(img)]
    if finite.size:
        lo, hi = np.percentile(finite, [1, 99])
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = (float(np.nanmin(finite)), float(np.nanmax(finite)))
        if hi > lo:
            self.pd_image_artist_gray.set_clim(lo, hi)
            self.pd_image_artist_color.set_clim(lo, hi)
            try:
                self.pd_colorbar.update_normal(self.pd_image_artist_color)
            except Exception:
                pass
    h, w = img.shape
    old = getattr(self, '_pd_zoom_limits', None) if preserve_zoom else None
    if old is None:
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            ax.set_xlim(-0.5, w - 0.5)
            ax.set_ylim(-0.5, h - 0.5)
            ax.set_aspect('equal', adjustable='box')
    else:
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            ax.set_xlim(*old[0])
            ax.set_ylim(*old[1])
            ax.set_aspect('equal', adjustable='box')
    try:
        field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    except Exception:
        field = float(idx)
    self.pd_frame_label.config(text=f'{idx + 1}/{len(stack)} | {field:+.2f} mT')
    self.pd_ax_gray.set_title(f'Selected ROI — Grayscale | Frame {idx + 1}/{len(stack)}', fontsize=9)
    self.pd_ax_color.set_title(f'Selected ROI — Color | Frame {idx + 1}/{len(stack)} | {field:+.2f} mT', fontsize=9)
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass
    return True

def _pd7_refresh_from_confirmed_roi(self, preserve_zoom=True):
    return _pd7_update_roi_image_core_only(self, preserve_zoom=preserve_zoom)

def _pd7_frame_changed_independent(self, *args):
    idx = max(0, min(int(self.pd_frame_var.get()), len(getattr(self, 'roi2_stack', [])) - 1)) if getattr(self, 'roi2_stack', None) is not None and len(self.roi2_stack) else 0
    self.pd_frame_var.set(idx)
    self._pd7_refresh_from_confirmed_roi(preserve_zoom=True)
    try:
        if idx in getattr(self, 'pd_frame_detections', {}):
            self.pd_current_detections = [dict(d) for d in self.pd_frame_detections.get(idx, [])]
        else:
            self.pd_current_detections = []
    except Exception:
        pass
    self.pd_multi_selected = set()
    self.pd_selected_detection = None
    self._pd3_draw_overlay()

def _pd7_frame_wheel_independent(self, event):
    slider = getattr(self, 'pd_frame_slider', None)
    stack = getattr(self, 'roi2_stack', None)
    if slider is None or stack is None or len(stack) == 0 or (not getattr(slider, '_pd7_armed', False)):
        return 'break'
    try:
        px = slider.winfo_toplevel().winfo_pointerx()
        py = slider.winfo_toplevel().winfo_pointery()
        rx = slider.winfo_rootx()
        ry = slider.winfo_rooty()
        rw = slider.winfo_width()
        rh = slider.winfo_height()
        if not (rx <= px <= rx + rw and ry <= py <= ry + rh):
            return 'break'
    except Exception:
        return 'break'
    step = 1 if getattr(event, 'num', None) == 4 or getattr(event, 'delta', 0) > 0 else -1
    cur = int(slider.get())
    new = max(0, min(len(stack) - 1, cur + step))
    slider.set(new)
    self.pd_frame_var.set(new)
    self._pd7_frame_changed_independent()
    return 'break'

def _pd7_frame_slider_arm(self, event=None):
    try:
        self.pd_frame_slider._pd7_armed = True
    except Exception:
        pass
    return None

def _pd7_zoom_scroll_independent(self, event):
    if event is None or event.inaxes not in (getattr(self, 'pd_ax_gray', None), getattr(self, 'pd_ax_color', None)):
        return
    ax = event.inaxes
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x = float(event.xdata)
    y = float(event.ydata)
    fac = 0.8 if getattr(event, 'button', None) == 'up' else 1.25
    x0 = x - (x - xlim[0]) * fac
    x1 = x + (xlim[1] - x) * fac
    y0 = y - (y - ylim[0]) * fac
    y1 = y + (ylim[1] - y) * fac
    stack = getattr(self, 'roi2_stack', None)
    if stack is not None and len(stack):
        h, w = np.asarray(stack[int(self.pd_frame_var.get())]).shape[:2]
        x0 = max(-0.5, x0)
        x1 = min(w - 0.5, x1)
        y0 = max(-0.5, y0)
        y1 = min(h - 0.5, y1)
    for aa in (self.pd_ax_gray, self.pd_ax_color):
        aa.set_xlim(x0, x1)
        aa.set_ylim(y0, y1)
    self._pd_zoom_limits = ((x0, x1), (y0, y1))
    self.pd_canvas.draw_idle()

def _pd7_reset_zoom_independent(self):
    self._pd_zoom_limits = None
    self._pd7_refresh_from_confirmed_roi(preserve_zoom=False)
    self.pd_status_var.set('ROI zoom reset.')

def _pd7_update_selected_particle_core_only(self):
    if not getattr(self, 'pd_current_detections', None):
        try:
            self.pd_particle_status.config(text='No particle selected. Click an accepted particle/centre in the ROI.')
        except Exception:
            pass
        return
    j = getattr(self, 'pd_selected_detection', None)
    if j is None or j < 0 or j >= len(self.pd_current_detections):
        return
    d = self.pd_current_detections[int(j)]
    cls = str(d.get('classification', 'particle')).upper()
    status = 'PARTICLE' if d.get('accepted', True) else 'WORM'
    txt = f"Particle #{int(j) + 1}: center=({d.get('x', np.nan):.2f}, {d.get('y', np.nan):.2f}) px\nOuter diameter={d.get('diameter_px', np.nan):.2f} px | Outer area={d.get('area_px', np.nan):.1f} px²\nCore diameter={d.get('core_diameter_px', np.nan):.2f} px | Core AR={d.get('core_aspect_ratio', np.nan):.3f}\nCore eccentricity={d.get('core_eccentricity', np.nan):.3f} | Core solidity={d.get('core_solidity', np.nan):.3f}\nClassification: {status}"
    try:
        self.pd_particle_status.config(text=txt)
    except Exception:
        pass

def _pd7_select_detection_core_only(self, event):
    if event is None or event.inaxes not in (getattr(self, 'pd_ax_gray', None), getattr(self, 'pd_ax_color', None)):
        return
    dets = getattr(self, 'pd_current_detections', []) or []
    if not dets or event.xdata is None or event.ydata is None:
        return
    ds = [np.hypot(float(d['x']) - event.xdata, float(d['y']) - event.ydata) for d in dets]
    j = int(np.argmin(ds))
    tol = max(8.0, 0.65 * float(dets[j].get('diameter_px', 12.0)))
    if ds[j] > tol:
        return
    ctrl = bool(getattr(event, 'key', None) in ('control', 'ctrl'))
    if ctrl:
        sel = set(getattr(self, 'pd_multi_selected', set()) or set())
        if j in sel:
            sel.remove(j)
        else:
            sel.add(j)
        self.pd_multi_selected = sel
    else:
        self.pd_multi_selected = {j}
        self.pd_selected_detection = j
        self._pd7_update_selected_particle_core_only()
    self._pd3_draw_overlay()

def _pd7_draw_overlay_core_only(self):
    for a in getattr(self, 'pd_overlay_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self.pd_overlay_artists = []
    if not bool(getattr(self, 'pd_show_centers_var', tk.BooleanVar(value=True)).get()):
        self.pd_canvas.draw_idle()
        return
    selected = set(getattr(self, 'pd_multi_selected', set()) or set())
    for i, d in enumerate(getattr(self, 'pd_current_detections', []) or []):
        outer = np.asarray(d.get('outer_contour', np.empty((0, 2))), dtype=float)
        core = np.asarray(d.get('core_contour', np.empty((0, 2))), dtype=float)
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            if outer.size:
                self.pd_overlay_artists.append(ax.plot(outer[:, 1], outer[:, 0], '-', color='yellow', lw=2.1, zorder=20)[0])
            if core.size:
                self.pd_overlay_artists.append(ax.plot(core[:, 1], core[:, 0], '-', color='limegreen', lw=2.0, zorder=21)[0])
            col = 'yellow' if i in selected else 'cyan'
            self.pd_overlay_artists.append(ax.text(d['x'], d['y'], str(i + 1), fontsize=8, fontweight='bold', ha='center', va='center', color=col, zorder=24, bbox=dict(boxstyle='round,pad=0.15', fc='black', ec=col, alpha=0.55)))
    self.pd_canvas.draw_idle()

def _pd7_refresh_density_row_core_only(self, idx):
    dets = getattr(self, 'pd_frame_detections', {}).get(idx, []) or []
    valid = [d for d in dets if d.get('accepted', True) and (not d.get('edge', False) or bool(self.pd_include_edge_var.get()))]
    count = len(valid)
    h, w = np.asarray(self.roi2_stack[idx]).shape[:2]
    sx, sy = _tab8_authoritative_nm_scales(self)
    area_um2 = ((float(w) * float(sx)) * (float(h) * float(sy)) / 1.0e6) if (sx and sy and sx > 0 and sy > 0) else np.nan
    dens = count / area_um2 if area_um2 > 0 else np.nan
    dvals = np.array([d.get('diameter_px', np.nan) for d in valid], float)
    dvals = dvals[np.isfinite(dvals)]
    corevals = np.array([d.get('core_diameter_px', np.nan) for d in valid], float)
    corevals = corevals[np.isfinite(corevals)]
    arvals = np.array([d.get('core_aspect_ratio', np.nan) for d in valid], float)
    arvals = arvals[np.isfinite(arvals)]
    row = (idx + 1, self._pd3_field_value(idx), count, area_um2, dens, float(np.mean(dvals)) if dvals.size else np.nan, float(np.mean(corevals)) if corevals.size else np.nan, float(np.mean(arvals)) if arvals.size else np.nan)
    self.pd_results[idx] = row
    self._pd3_refresh_table()
    self._pd3_update_density_plot()

def _pd7_analyze_current_core_only(self):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        messagebox.showwarning('Skyrmion Analysis', 'Select/confirm the Secondary ROI first.', parent=self.root)
        return
    idx = max(0, min(int(self.pd_frame_var.get()), len(stack) - 1))
    self.pd_frame_var.set(idx)
    det = [dict(d) for d in self._pd7_detect_frame(np.asarray(stack[idx], dtype=float))]
    self.pd_current_detections = det
    self.pd_frame_detections[idx] = [dict(d) for d in det]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self.pd_last_deleted = None
    self._pd7_refresh_from_confirmed_roi(preserve_zoom=True)
    self._pd7_draw_overlay_core_only()
    self._pd7_refresh_density_row_core_only(idx)
    self.pd_status_var.set(f'Frame {idx + 1}: detected {len(det)} accepted particles using outer-particle + inner-core symmetry classification.')

def _pd7_analyze_selected_core_only(self):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        messagebox.showwarning('Skyrmion Analysis', 'Select/confirm the Secondary ROI first.', parent=self.root)
        return
    n = len(stack)
    try:
        a = max(1, min(n, int(self.pd_range_start_var.get())))
        b = max(1, min(n, int(self.pd_range_end_var.get())))
    except Exception:
        a, b = (1, n)
    if a > b:
        a, b = (b, a)
    self.pd_results = {}
    for idx in range(a - 1, b):
        det = [dict(d) for d in self._pd7_detect_frame(np.asarray(stack[idx], dtype=float))]
        self.pd_frame_detections[idx] = det
        self._pd7_refresh_density_row_core_only(idx)
    keep = max(a - 1, min(int(self.pd_frame_var.get()), b - 1))
    self.pd_frame_var.set(keep)
    self.pd_current_detections = [dict(d) for d in self.pd_frame_detections.get(keep, [])]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._pd7_refresh_from_confirmed_roi(preserve_zoom=True)
    self._pd7_draw_overlay_core_only()
    self._pd3_refresh_table()
    self._pd3_update_density_plot()
    self.pd_status_var.set(f'Analyzed selected frames {a}–{b}.')

def _pd7_build_core_only(self):
    self.pd_frame_var = tk.IntVar(value=0)
    self.pd_frame_label = None
    self.pd_polarity_var = tk.StringVar(value='bright')
    self.pd_min_diam_var = tk.DoubleVar(value=10.0)
    self.pd_max_diam_var = tk.DoubleVar(value=50.0)
    self.pd_threshold_var = tk.DoubleVar(value=0.1)
    self.pd_min_sep_var = tk.DoubleVar(value=50.0)
    self.pd_max_aspect_var = tk.DoubleVar(value=1.8)
    self.pd_core_contrast_var = tk.DoubleVar(value=0.55)
    self.pd_core_min_diam_var = tk.DoubleVar(value=4.0)
    self.pd_core_max_diam_var = tk.DoubleVar(value=40.0)
    self.pd_include_edge_var = tk.BooleanVar(value=False)
    self.pd_show_centers_var = tk.BooleanVar(value=True)
    self.pd_nnc_var = tk.BooleanVar(value=False)
    self.pd_status_var = tk.StringVar(value='Waiting for confirmed Secondary ROI from Tab 5.')
    self.pd_results = {}
    self.pd_current_detections = []
    self.pd_frame_detections = {}
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self.pd_overlay_artists = []
    self.pd_last_deleted = None
    self.pd_nnc_exclusions = {}
    self.pd_selected_particle = None
    self._pd_zoom_limits = None
    for w in self.tab_particle_density.winfo_children():
        w.destroy()
    outer = ttk.Frame(self.tab_particle_density, padding=5)
    outer.pack(fill='both', expand=True)
    outer.columnconfigure(0, weight=5)
    outer.columnconfigure(1, weight=3)
    outer.rowconfigure(1, weight=3)
    outer.rowconfigure(2, weight=2)
    head = ttk.Frame(outer)
    head.grid(row=0, column=0, columnspan=2, sticky='ew', pady=(0, 4))
    ttk.Label(head, text='SKYRMION / PARTICLE — OUTER SEGMENTATION + INNER CORE + CORE SYMMETRY', font=('Arial', 11, 'bold')).pack(side='left')
    ttk.Label(head, text='Source: confirmed Secondary ROI', foreground='darkgreen').pack(side='left', padx=10)
    ttk.Label(head, text='Frame:').pack(side='left', padx=(12, 3))
    self.pd_frame_slider = tk.Scale(head, from_=0, to=0, orient='horizontal', variable=self.pd_frame_var, resolution=1, length=230, showvalue=True, command=self._pd_frame_changed)
    self.pd_frame_slider.pack(side='left')
    self.pd_frame_slider.bind('<Button-1>', _pd7_frame_slider_arm, add='+')
    self.pd_frame_slider._pd7_armed = False
    self.pd_frame_slider.bind('<MouseWheel>', self._pd_frame_wheel, add='+')
    self.pd_frame_slider.bind('<Button-4>', self._pd_frame_wheel, add='+')
    self.pd_frame_slider.bind('<Button-5>', self._pd_frame_wheel, add='+')
    self.pd_frame_label = ttk.Label(head, text='--', width=23)
    self.pd_frame_label.pack(side='left', padx=4)
    ttk.Button(head, text='RESET ZOOM', command=self._pd_reset_zoom).pack(side='left', padx=2)
    ttk.Button(head, text='ANALYZE CURRENT FRAME', command=self._pd_analyze_current).pack(side='left', padx=2)
    ttk.Button(head, text='ANALYZE SELECTED FRAMES', command=self._pd_analyze_selected_frames).pack(side='left', padx=2)
    ttk.Button(head, text='DELETE SELECTED', command=self._pd_delete_selected).pack(side='left', padx=2)
    ttk.Button(head, text='UNDO DELETE', command=self._pd_undo_delete).pack(side='left', padx=2)
    roi_box = ttk.LabelFrame(outer, text='SELECTED ROI — GRAYSCALE + COLOR', padding=2)
    roi_box.grid(row=1, column=0, sticky='nsew', padx=(0, 4), pady=(0, 3))
    ctrl_box = ttk.LabelFrame(outer, text='DETECTION / ANALYSIS CONTROLS', padding=2)
    ctrl_box.grid(row=1, column=1, rowspan=2, sticky='nsew', padx=(4, 0), pady=(0, 3))
    graph_box = ttk.LabelFrame(outer, text='COUNT / DENSITY GRAPH', padding=2)
    graph_box.grid(row=2, column=0, sticky='nsew', padx=(0, 4), pady=(3, 0))
    self.pd_fig, (self.pd_ax_gray, self.pd_ax_color) = plt.subplots(1, 2, figsize=(8.5, 5.0), dpi=100, constrained_layout=True)
    self.pd_image_artist_gray = self.pd_ax_gray.imshow(np.zeros((10, 10)), cmap='gray', origin='lower', interpolation='nearest')
    try:
        cmap = self.colormap_var.get() or 'gray'
    except Exception:
        cmap = 'gray'
    self.pd_image_artist_color = self.pd_ax_color.imshow(np.zeros((10, 10)), cmap=cmap, origin='lower', interpolation='nearest')
    for ax in (self.pd_ax_gray, self.pd_ax_color):
        ax.set_xlabel('X pixel')
        ax.set_ylabel('Y pixel')
        ax.set_aspect('equal', adjustable='box')
    self.pd_ax_gray.set_title('Selected ROI — Grayscale')
    self.pd_ax_color.set_title('Selected ROI — Color')
    self.pd_colorbar = self.pd_fig.colorbar(self.pd_image_artist_color, ax=self.pd_ax_color, fraction=0.046, pad=0.04)
    self.pd_colorbar.set_label('Intensity')
    self.pd_canvas = FigureCanvasTkAgg(self.pd_fig, master=roi_box)
    self.pd_canvas.draw()
    self.pd_canvas.get_tk_widget().pack(fill='both', expand=True)
    self.pd_canvas.mpl_connect('scroll_event', self._pd_zoom_scroll)
    self.pd_canvas.mpl_connect('button_press_event', self._pd_select_detection)
    cc = tk.Canvas(ctrl_box, highlightthickness=0, borderwidth=0)
    sy = ttk.Scrollbar(ctrl_box, orient='vertical', command=cc.yview)
    cf = ttk.Frame(cc, padding=7)
    cw = cc.create_window((0, 0), window=cf, anchor='nw')
    cc.configure(yscrollcommand=sy.set)
    cc.pack(side='left', fill='both', expand=True)
    sy.pack(side='right', fill='y')
    cf.bind('<Configure>', lambda e: cc.configure(scrollregion=cc.bbox('all')))
    cc.bind('<Configure>', lambda e: cc.itemconfigure(cw, width=e.width))

    def cwheel(e):
        try:
            delta = e.delta if hasattr(e, 'delta') else 0
            if delta:
                cc.yview_scroll(int(-delta / 120), 'units')
            elif getattr(e, 'num', None) == 4:
                cc.yview_scroll(-3, 'units')
            elif getattr(e, 'num', None) == 5:
                cc.yview_scroll(3, 'units')
        except Exception:
            pass
        return 'break'
    for ww in (cc, cf):
        ww.bind('<MouseWheel>', cwheel)
        ww.bind('<Button-4>', cwheel)
        ww.bind('<Button-5>', cwheel)
    ttk.Label(cf, text='OUTER PARTICLE DETECTION', font=('Arial', 10, 'bold')).pack(anchor='w', pady=(0, 5))
    ttk.Label(cf, text='Detection threshold').pack(anchor='w')
    tk.Scale(cf, from_=0.01, to=0.5, resolution=0.01, orient='horizontal', variable=self.pd_threshold_var, length=270, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Polarity').pack(anchor='w')
    ttk.Combobox(cf, textvariable=self.pd_polarity_var, values=['bright', 'dark', 'both'], state='readonly', width=12).pack(anchor='w')
    ttk.Label(cf, text='Particle diameter (px): 10–50').pack(anchor='w', pady=(5, 0))
    ttk.Label(cf, textvariable=tk.StringVar(value='Fixed diameter range: 10–50 px')).pack(anchor='w')
    ttk.Label(cf, text='Minimum separation (px): 50').pack(anchor='w')
    ttk.Label(cf, text='INNER CORE DETECTION', font=('Arial', 10, 'bold')).pack(anchor='w', pady=(10, 5))
    ttk.Label(cf, text='Core contrast level (relative)').pack(anchor='w')
    tk.Scale(cf, from_=0.1, to=0.95, resolution=0.01, orient='horizontal', variable=self.pd_core_contrast_var, length=270, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Core minimum diameter (px)').pack(anchor='w')
    tk.Scale(cf, from_=2, to=30, resolution=1, orient='horizontal', variable=self.pd_core_min_diam_var, length=270, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Core maximum diameter (px)').pack(anchor='w')
    tk.Scale(cf, from_=5, to=60, resolution=1, orient='horizontal', variable=self.pd_core_max_diam_var, length=270, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='CORE SYMMETRY CLASSIFICATION', font=('Arial', 10, 'bold')).pack(anchor='w', pady=(10, 5))
    ttk.Label(cf, text='Maximum core aspect ratio').pack(anchor='w')
    tk.Scale(cf, from_=1.0, to=3.0, resolution=0.01, orient='horizontal', variable=self.pd_max_aspect_var, length=270, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Core AR ≤ limit → PARTICLE | Core AR > limit → WORM', wraplength=300, justify='left').pack(anchor='w', pady=4)
    ttk.Checkbutton(cf, text='Include edge detections', variable=self.pd_include_edge_var).pack(anchor='w')
    ttk.Checkbutton(cf, text='Show outer/core overlays', variable=self.pd_show_centers_var, command=self._pd_draw_overlay).pack(anchor='w')
    ttk.Checkbutton(cf, text='NNC-Delete — No Need to Consider', variable=self.pd_nnc_var).pack(anchor='w')
    ttk.Button(cf, text='CLEAR NNC EXCLUSIONS', command=self._pd_clear_nnc).pack(fill='x', pady=3)
    ttk.Label(cf, text='Selected particle', font=('Arial', 10, 'bold')).pack(anchor='w', pady=(9, 2))
    self.pd_particle_status = ttk.Label(cf, text='Click an accepted particle in the ROI.', wraplength=300, justify='left')
    self.pd_particle_status.pack(anchor='w')
    ttk.Label(cf, text='Start frame').pack(anchor='w', pady=(9, 0))
    self.pd_range_start_var = tk.IntVar(value=1)
    tk.Spinbox(cf, from_=1, to=9999, textvariable=self.pd_range_start_var, width=8, increment=1).pack(anchor='w')
    ttk.Label(cf, text='End frame').pack(anchor='w', pady=(5, 0))
    self.pd_range_end_var = tk.IntVar(value=1)
    tk.Spinbox(cf, from_=1, to=9999, textvariable=self.pd_range_end_var, width=8, increment=1).pack(anchor='w')
    ttk.Label(cf, textvariable=self.pd_status_var, wraplength=300, justify='left').pack(anchor='w', pady=6)
    ttk.Separator(cf).pack(fill='x', pady=4)
    ttk.Button(cf, text='EXPORT TABLE (CSV / XLSX)', command=self._pd_export_table).pack(fill='x', pady=2)
    ttk.Button(cf, text='EXPORT SELECTED ROI — TIFF 300 PPI', command=self._pd_export_tiff).pack(fill='x', pady=2)
    ttk.Button(cf, text='EXPORT 3 FIGURES — ANNOTATED GRAY + ANNOTATED COLOR + RAW COLOR | PNG 300 PPI', command=self._pd_export_three_figures_300ppi).pack(fill='x', pady=2)
    self.pd_plot_fig = plt.Figure(figsize=(7.0, 2.6), dpi=100)
    self.pd_plot_ax = self.pd_plot_fig.add_axes([0.1, 0.2, 0.72, 0.62])
    self.pd_plot_ax2 = self.pd_plot_fig.add_axes(self.pd_plot_ax.get_position(), frameon=False)
    self.pd_plot_ax2.patch.set_visible(False)
    self.pd_plot_canvas = FigureCanvasTkAgg(self.pd_plot_fig, master=graph_box)
    self.pd_plot_canvas.draw()
    self.pd_plot_canvas.get_tk_widget().pack(fill='both', expand=True)
    table_box = ttk.LabelFrame(outer, text='DENSITY TABLE')
    table_box.grid(row=2, column=1, sticky='nsew', padx=(4, 0), pady=(3, 0))
    tf = ttk.Frame(table_box)
    tf.pack(fill='both', expand=True)
    cols = ('Frame', 'Field (mT)', 'Count', 'ROI area (µm²)', 'Density (skyrmions/µm²)', 'Mean outer d (px)', 'Mean core d (px)', 'Mean core AR')
    self.pd_table = ttk.Treeview(tf, columns=cols, show='headings')
    for c in cols:
        self.pd_table.heading(c, text=c)
        self.pd_table.column(c, width=105, anchor='center')
    sy2 = ttk.Scrollbar(tf, orient='vertical', command=self.pd_table.yview)
    self.pd_table.configure(yscrollcommand=sy2.set)
    self.pd_table.pack(side='left', fill='both', expand=True)
    sy2.pack(side='right', fill='y')
    self._pd7_refresh_from_confirmed_roi(preserve_zoom=False)
    try:
        n = len(self.roi2_stack) if getattr(self, 'roi2_stack', None) is not None else 0
        self.pd_frame_slider.configure(from_=0, to=max(0, n - 1), state='normal' if n else 'disabled')
        self.pd_range_end_var.set(max(1, n))
    except Exception:
        pass
CDIWorkflowApp._pd7_refresh_from_confirmed_roi = _pd7_refresh_from_confirmed_roi
CDIWorkflowApp._pd3_update_roi_image = _pd7_update_roi_image_core_only
CDIWorkflowApp._pd3_frame_changed = _pd7_frame_changed_independent
CDIWorkflowApp._pd3_frame_wheel = _pd7_frame_wheel_independent
CDIWorkflowApp._pd3_zoom_scroll = _pd7_zoom_scroll_independent
CDIWorkflowApp._pd3_reset_zoom = _pd7_reset_zoom_independent
CDIWorkflowApp._pd3_select_detection = _pd7_select_detection_core_only
CDIWorkflowApp._pd3_draw_overlay = _pd7_draw_overlay_core_only
CDIWorkflowApp._pd3_update_selected_particle_view = _pd7_update_selected_particle_core_only
CDIWorkflowApp._pd3_refresh_density_row = _pd7_refresh_density_row_core_only
CDIWorkflowApp._pd3_analyze_current = _pd7_analyze_current_core_only
CDIWorkflowApp._pd3_analyze_selected_frames = _pd7_analyze_selected_core_only

def _pd7_delete_selected_core(self):
    sel = sorted(set(getattr(self, 'pd_multi_selected', set()) or set()), reverse=True)
    dets = getattr(self, 'pd_current_detections', []) or []
    if not sel and getattr(self, 'pd_selected_detection', None) is not None:
        sel = [int(self.pd_selected_detection)]
    if not sel:
        self.pd_status_var.set('Select one or more particles first (Ctrl + left-click for multiple).')
        return
    idx = int(self.pd_frame_var.get())
    deleted = []
    nnc_mode = bool(self.pd_nnc_var.get())
    for j in sel:
        if 0 <= j < len(dets):
            deleted.append(dict(dets[j]))
    if nnc_mode:
        ex = self.pd_nnc_exclusions.setdefault(idx, [])
        for d in deleted:
            ex.append((float(d.get('x', 0)), float(d.get('y', 0))))
        keep = [d for k, d in enumerate(dets) if k not in set(sel)]
        self.pd_current_detections = keep
        self.pd_frame_detections[idx] = [dict(d) for d in keep]
        self.pd_last_deleted = None
        msg = f'NNC-Delete: {len(deleted)} particle(s) excluded from frame {idx + 1} for future analysis.'
    else:
        self.pd_last_deleted = (idx, deleted)
        for j in sel:
            if 0 <= j < len(dets):
                dets.pop(j)
        self.pd_current_detections = dets
        self.pd_frame_detections[idx] = [dict(d) for d in dets]
        msg = f'Deleted {len(deleted)} particle(s) from frame {idx + 1}.'
    self.pd_multi_selected = set()
    self.pd_selected_detection = None
    self._pd_draw_overlay()
    self._pd3_refresh_density_row(idx)
    self.pd_status_var.set(msg)

def _pd7_undo_delete_core(self):
    last = getattr(self, 'pd_last_deleted', None)
    if not last:
        self.pd_status_var.set('No normal deletion to undo. NNC-Delete exclusions are permanent.')
        return
    idx, deleted = last
    cur = [dict(d) for d in self.pd_frame_detections.get(idx, [])]
    cur.extend(deleted)
    cur.sort(key=lambda d: (d.get('y', 0), d.get('x', 0)))
    self.pd_frame_detections[idx] = cur
    if int(self.pd_frame_var.get()) == idx:
        self.pd_current_detections = [dict(d) for d in cur]
    self.pd_last_deleted = None
    self._pd_draw_overlay()
    self._pd3_refresh_density_row(idx)
    self.pd_status_var.set(f'Restored {len(deleted)} particle(s) in frame {idx + 1}.')

def _pd7_clear_nnc_core(self):
    self.pd_nnc_exclusions = {}
    self.pd_status_var.set('NNC-Delete exclusions cleared. Re-run analysis for affected frames.')

def _pd7_refresh_table_core(self):
    if not hasattr(self, 'pd_table'):
        return
    for item in self.pd_table.get_children():
        self.pd_table.delete(item)
    for k in sorted(getattr(self, 'pd_results', {}) or {}):
        self.pd_table.insert('', 'end', values=self.pd_results[k])

def _pd7_update_density_plot_core(self):
    if not hasattr(self, 'pd_plot_canvas'):
        return
    ax = self.pd_plot_ax
    ax2 = self.pd_plot_ax2
    ax.cla()
    ax2.cla()
    ax.set_zorder(2)
    ax2.set_zorder(1)
    ax2.patch.set_visible(False)
    ax.patch.set_alpha(0.0)
    rows = [self.pd_results[k] for k in sorted(getattr(self, 'pd_results', {}) or {}) if self.pd_results[k] is not None]
    if rows:
        frames = np.asarray([r[0] for r in rows], float)
        fields = np.asarray([r[1] for r in rows], float)
        counts = np.asarray([r[2] for r in rows], float)
        dens = np.asarray([r[4] for r in rows], float)
        ax.plot(fields, counts, 'o-', lw=1.6, ms=4)
        ax2.plot(frames, dens, 's--', lw=1.4, ms=3)
        ax.set_xlabel('Field (mT)')
        ax.set_ylabel('Particle count')
        ax2.set_xlabel('Equivalent frame No.')
        ax2.xaxis.set_label_position('top')
        ax2.xaxis.tick_top()
        ax2.set_ylabel('Density (skyrmions/µm²)')
        ax2.yaxis.set_label_position('right')
        ax2.yaxis.tick_right()
        ax.set_title('Count vs Field | Density vs Frame', fontsize=9, pad=18)
        ax.grid(alpha=0.2)
    else:
        ax.text(0.5, 0.5, 'Analyze a frame to plot results', ha='center', va='center', transform=ax.transAxes)
        ax.set_xlabel('Field (mT)')
        ax.set_ylabel('Particle count')
        ax2.set_xlabel('Equivalent frame No.')
        ax2.xaxis.set_label_position('top')
        ax2.xaxis.tick_top()
        ax2.set_ylabel('Density (skyrmions/µm²)')
        ax2.yaxis.set_label_position('right')
        ax2.yaxis.tick_right()
        ax.set_title('Count vs Field | Density vs Frame', fontsize=9, pad=18)
    self.pd_plot_canvas.draw_idle()
try:
    _TAB5_CONFIRM_ORIG = CDIWorkflowApp.confirm_roi2

    def _tab5_confirm_roi2_keep_tab7(self, *args, **kwargs):
        out = _TAB5_CONFIRM_ORIG(self, *args, **kwargs)
        try:
            if getattr(self, 'roi2_stack', None) is not None:
                self._pd7_refresh_from_confirmed_roi(preserve_zoom=False)
        except Exception:
            pass
        return out
    CDIWorkflowApp.confirm_roi2 = _tab5_confirm_roi2_keep_tab7
except Exception:
    pass

def _roi2_preserve_confirmed_geometry_final(self):
    c = getattr(self, 'roi2_confirmed_coords', None)
    if c is not None:
        self.roi2_coords = dict(c)
        self._roi2_geometry_locked = True
try:
    _ORIG_ROI2_UPDATE_FINAL = CDIWorkflowApp.update_roi2_view

    def _roi2_update_view_final(self, *args, **kwargs):
        out = _ORIG_ROI2_UPDATE_FINAL(self, *args, **kwargs)
        _roi2_preserve_confirmed_geometry_final(self)
        return out
    CDIWorkflowApp.update_roi2_view = _roi2_update_view_final
except Exception:
    pass
CDIWorkflowApp._pd7_detect_frame = _pd7_detect_frame
CDIWorkflowApp._pd7_normalize_for_detection = _pd7_normalize_for_detection
CDIWorkflowApp._pd3_field_value = _pd3_field_value
CDIWorkflowApp._pd3_refresh_table = _pd7_refresh_table_core
CDIWorkflowApp._pd3_update_density_plot = _pd7_update_density_plot_core

def _tab7_get_stack(self):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None:
        return None
    try:
        arr = np.asarray(stack)
        if arr.ndim != 3 or arr.shape[0] == 0:
            return None
        return arr
    except Exception:
        return None

def _tab7_field(self, idx):
    try:
        a = getattr(self, 'real_fields_mT', None)
        if a is not None and len(a) > int(idx):
            v = float(a[int(idx)])
            if np.isfinite(v):
                return v
    except Exception:
        pass
    return float(idx)

def _tab7_normalize(self, image):
    a = np.asarray(image, dtype=float)
    finite = np.isfinite(a)
    if not finite.any():
        return np.zeros_like(a, dtype=float)
    vals = a[finite]
    lo, hi = np.percentile(vals, [1.0, 99.0])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = (float(np.nanmin(vals)), float(np.nanmax(vals)))
    if hi <= lo:
        return np.zeros_like(a, dtype=float)
    return np.clip((a - lo) / (hi - lo), 0.0, 1.0)

def _tab7_detect(self, image):
    """Two-level contrast detection.

    1) Outer particle: denoise + local background removal + threshold.
    2) Core: a second relative threshold applied INSIDE each outer particle.
    3) Core symmetry: core major/minor-axis ratio is used for classification.
    No LoG/DoG and no watershed are used.
    """
    from scipy import ndimage as ndi
    from skimage.measure import regionprops
    from skimage.morphology import remove_small_objects, opening, closing, disk
    from skimage.draw import polygon2mask
    z = self._tab7_normalize(image)
    h, w = z.shape
    try:
        threshold = float(np.clip(self.pd_threshold_var.get(), 0.01, 0.9))
    except Exception:
        threshold = 0.1
    try:
        core_level = float(np.clip(self.pd_core_contrast_var.get(), 0.05, 0.95))
    except Exception:
        core_level = 0.55
    try:
        dmin = max(3.0, float(self.pd_min_diam_var.get()))
    except Exception:
        dmin = 10.0
    try:
        dmax = max(dmin, float(self.pd_max_diam_var.get()))
    except Exception:
        dmax = 50.0
    try:
        sep = max(1.0, float(self.pd_min_sep_var.get()))
    except Exception:
        sep = 50.0
    try:
        ar_max = max(1.0, float(self.pd_max_aspect_var.get()))
    except Exception:
        ar_max = 1.8
    try:
        cmin = max(1.0, float(self.pd_core_min_diam_var.get()))
    except Exception:
        cmin = 4.0
    try:
        cmax = max(cmin, float(self.pd_core_max_diam_var.get()))
    except Exception:
        cmax = 40.0
    polarity = str(self.pd_polarity_var.get()).lower().strip()
    include_edge = bool(self.pd_include_edge_var.get()) if hasattr(self, 'pd_include_edge_var') else False
    sm = ndi.gaussian_filter(ndi.median_filter(z, size=3, mode='nearest'), sigma=1.0, mode='nearest')
    bg_sigma = max(8.0, 0.8 * dmax)
    bg = ndi.gaussian_filter(sm, sigma=bg_sigma, mode='nearest')
    detail = sm - bg
    polarity_list = []
    if polarity in ('bright', 'both'):
        polarity_list.append(('bright', detail))
    if polarity in ('dark', 'both'):
        polarity_list.append(('dark', -detail))
    if not polarity_list:
        polarity_list = [('bright', detail)]
    all_candidates = []
    for pol, sig in polarity_list:
        positive = sig[sig > 0]
        scale = float(np.percentile(positive, 99.0)) if positive.size else 1.0
        scale = max(scale, 1e-12)
        q = np.clip(sig / scale, 0.0, 1.0)
        mask = q >= threshold
        min_area = max(8, int(np.pi * (dmin / 2.0) ** 2 * 0.08))
        mask = remove_small_objects(mask, max_size=max(0, int(min_area)-1))
        mask = opening(mask, footprint=disk(1))
        mask = closing(mask, footprint=disk(1))
        labels, nlab = ndi.label(mask)
        if nlab == 0:
            continue
        for rp in regionprops(labels, intensity_image=q):
            eqd = float(rp.equivalent_diameter_area)
            if eqd < dmin or eqd > dmax:
                continue
            cy, cx = map(float, rp.centroid)
            edge = cx < eqd * 0.5 or cy < eqd * 0.5 or cx > w - 1 - eqd * 0.5 or (cy > h - 1 - eqd * 0.5)
            if edge and (not include_edge):
                continue
            rad = max(6, int(np.ceil(0.75 * eqd)))
            y0 = max(0, int(round(cy)) - rad)
            y1 = min(h, int(round(cy)) + rad + 1)
            x0 = max(0, int(round(cx)) - rad)
            x1 = min(w, int(round(cx)) + rad + 1)
            cand = labels[y0:y1, x0:x1] == rp.label
            patch = q[y0:y1, x0:x1]
            if cand.sum() < 8:
                continue
            outside = patch[~cand]
            inside = patch[cand]
            local_bg = float(np.median(outside)) if outside.size else 0.0
            peak = float(np.percentile(inside, 98.0)) if inside.size else float(np.max(inside))
            denom = peak - local_bg
            if denom <= 1e-09:
                continue
            rel = np.clip((patch - local_bg) / denom, 0.0, 1.0)
            core = cand & (rel >= core_level)
            core_min_area = max(5, int(np.pi * (cmin / 2.0) ** 2 * 0.12))
            core = remove_small_objects(core, max_size=max(0, int(core_min_area)-1))
            if not core.any():
                continue
            clab, _ = ndi.label(core)
            props = regionprops(clab, intensity_image=rel)
            if not props:
                continue
            cr = max(props, key=lambda p: (float(p.intensity_mean), float(p.area)))
            c_eqd = float(cr.equivalent_diameter_area)
            if c_eqd < cmin or c_eqd > cmax:
                continue
            cmaj = float(cr.axis_major_length)
            cminor = float(cr.axis_minor_length)
            car = cmaj / max(cminor, 1e-12)
            cecc = float(getattr(cr, 'eccentricity', 0.0))
            csol = float(getattr(cr, 'solidity', 1.0))
            ccy, ccx = map(float, cr.centroid)
            ccx += x0
            ccy += y0
            outer_circ = 4.0 * np.pi * float(rp.area) / max(float(rp.perimeter) ** 2, 1e-12)
            classification = 'particle' if car <= ar_max else 'worm'
            all_candidates.append({'x': ccx, 'y': ccy, 'outer_x': cx, 'outer_y': cy, 'diameter_px': eqd, 'radius_px': eqd / 2.0, 'area_px': float(rp.area), 'outer_circularity': outer_circ, 'core_diameter_px': c_eqd, 'core_area_px': float(cr.area), 'core_aspect_ratio': car, 'core_eccentricity': cecc, 'core_solidity': csol, 'aspect_ratio': car, 'core_contrast': core_level, 'contrast': float(denom), 'polarity': pol, 'edge': bool(edge), 'classification': classification, 'accepted': classification == 'particle'})
    accepted = [d for d in all_candidates if d.get('accepted')]
    idx = int(getattr(self, 'pd_frame_var', tk.IntVar(value=0)).get())
    nnc = getattr(self, 'pd_nnc_exclusions', {}).get(idx, []) or []
    accepted = [d for d in accepted if not any((np.hypot(d['x'] - ex[0], d['y'] - ex[1]) <= max(6.0, 0.45 * d['diameter_px']) for ex in nnc))]
    accepted.sort(key=lambda d: (d['core_contrast'], d['core_area_px']), reverse=True)
    kept = []
    for d in accepted:
        if all((np.hypot(d['x'] - k['x'], d['y'] - k['y']) >= sep for k in kept)):
            kept.append(d)
    kept.sort(key=lambda d: (d['y'], d['x']))
    self.pd_auto_shape_text = f'Core symmetry: AR ≤ {ar_max:.2f} = PARTICLE | AR > {ar_max:.2f} = WORM'
    self.pd_last_all_candidates = all_candidates
    return kept

def _tab7_clear_artists(self):
    for art in getattr(self, 'pd_overlay_artists', []):
        try:
            art.remove()
        except Exception:
            pass
    self.pd_overlay_artists = []

def _tab7_draw_overlay(self):
    self._tab7_clear_artists()
    if not getattr(self, 'pd_show_centers_var', None) or not self.pd_show_centers_var.get():
        try:
            self.pd_canvas.draw_idle()
        except Exception:
            pass
        return
    dets = getattr(self, 'pd_current_detections', []) or []
    for i, d in enumerate(dets, 1):
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            x, y = (float(d['outer_x']), float(d['outer_y']))
            r = max(2.0, float(d['diameter_px']) / 2.0)
            theta = np.linspace(0, 2 * np.pi, 160)
            yy = y + r * np.sin(theta)
            xx = x + r * np.cos(theta)
            a1, = ax.plot(xx, yy, color='yellow', lw=2.0, zorder=20)
            a2, = ax.plot([d['x']], [d['y']], 'o', ms=4.0, mfc='none', mec='green', mew=1.5, zorder=22)
            cr = max(1.5, float(d.get('core_diameter_px', 6.0)) / 2.0)
            car = max(1.0, float(d.get('core_aspect_ratio', 1.0)))
            tt = np.linspace(0, 2 * np.pi, 120)
            ex = cr * car / 2.0
            ey = cr / 2.0
            gx = float(d['x']) + ex * np.cos(tt)
            gy = float(d['y']) + ey * np.sin(tt)
            a3, = ax.plot(gx, gy, color='lime', lw=1.8, zorder=21)
            a4 = ax.text(x, y, str(i), color='black', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='circle,pad=0.15', fc='yellow', ec='none'), zorder=23)
            self.pd_overlay_artists.extend([a1, a2, a3, a4])
            if i - 1 == getattr(self, 'pd_selected_detection', None):
                a5, = ax.plot([d['x']], [d['y']], 'o', ms=11, mfc='none', mec='white', mew=2.0, zorder=24)
                self.pd_overlay_artists.append(a5)
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass

def _tab7_update_display(self, preserve_zoom=True):
    stack = _tab7_get_stack(self)
    if stack is None:
        self.pd_status_var.set('Waiting for a confirmed Secondary ROI from Tab 5.')
        if hasattr(self, 'pd_frame_slider'):
            self.pd_frame_slider.configure(from_=0, to=0, state='disabled')
        if hasattr(self, 'pd_frame_label'):
            self.pd_frame_label.config(text='--')
        return False
    n = stack.shape[0]
    try:
        idx = int(self.pd_frame_var.get())
    except Exception:
        idx = 0
    idx = max(0, min(idx, n - 1))
    self.pd_frame_var.set(idx)
    try:
        self.pd_frame_slider.configure(from_=0, to=max(0, n - 1), state='normal')
    except Exception:
        pass
    img = np.asarray(stack[idx], dtype=float)
    finite = img[np.isfinite(img)]
    nz = finite[np.abs(finite) > 1e-12] if finite.size else finite
    vals = nz if nz.size >= max(20, int(0.01 * max(1, img.size))) else finite
    if vals.size:
        lo, hi = np.percentile(vals, [1, 99])
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = (float(vals.min()), float(vals.max()))
    else:
        lo, hi = (0.0, 1.0)
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    self.pd_image_artist_gray.set_data(img)
    self.pd_image_artist_gray.set_clim(lo, hi)
    self.pd_image_artist_color.set_data(img)
    self.pd_image_artist_color.set_clim(lo, hi)
    try:
        self.pd_image_artist_color.set_cmap(self.colormap_var.get() or 'gray')
    except Exception:
        pass
    try:
        self.pd_colorbar.update_normal(self.pd_image_artist_color)
    except Exception:
        pass
    h, w = img.shape[:2]
    if preserve_zoom and getattr(self, '_pd_zoom_limits', None) is not None:
        xl, yl = self._pd_zoom_limits
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            ax.set_xlim(*xl)
            ax.set_ylim(*yl)
    else:
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            ax.set_xlim(-0.5, w - 0.5)
            ax.set_ylim(-0.5, h - 0.5)
            ax.set_aspect('equal', adjustable='box')
        self._pd_zoom_limits = None
    field = self._tab7_field(idx)
    self.pd_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT')
    self.pd_ax_gray.set_title(f'Selected ROI — Grayscale | Frame {idx + 1}/{n}', fontsize=9)
    self.pd_ax_color.set_title(f'Selected ROI — Color | Frame {idx + 1}/{n} | {field:+.2f} mT', fontsize=9)
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass
    return True

def _tab7_refresh_from_roi(self, preserve_zoom=True):
    ok = _tab7_update_display(self, preserve_zoom=preserve_zoom)
    if ok:
        idx = int(self.pd_frame_var.get())
        self.pd_current_detections = [dict(d) for d in getattr(self, 'pd_frame_detections', {}).get(idx, []) or []]
        self.pd_selected_detection = None
        self.pd_multi_selected = set()
        self._tab7_draw_overlay()
    return ok

def _tab7_slider_changed(self, value=None):
    stack = _tab7_get_stack(self)
    if stack is None:
        return
    try:
        idx = int(round(float(value if value is not None else self.pd_frame_var.get())))
    except Exception:
        idx = int(self.pd_frame_var.get())
    idx = max(0, min(idx, stack.shape[0] - 1))
    self.pd_frame_var.set(idx)
    _tab7_update_display(self, preserve_zoom=True)
    self.pd_current_detections = [dict(d) for d in getattr(self, 'pd_frame_detections', {}).get(idx, []) or []]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._tab7_draw_overlay()

def _tab7_arm_slider(self, event=None):
    self.pd_frame_slider._tab7_armed = True
    try:
        self.pd_frame_slider.focus_set()
    except Exception:
        pass
    return None

def _tab7_slider_wheel(self, event):
    slider = getattr(self, 'pd_frame_slider', None)
    stack = _tab7_get_stack(self)
    if slider is None or stack is None or (not getattr(slider, '_tab7_armed', False)):
        return 'break'
    try:
        px = slider.winfo_toplevel().winfo_pointerx()
        py = slider.winfo_toplevel().winfo_pointery()
        rx, ry = (slider.winfo_rootx(), slider.winfo_rooty())
        rw, rh = (slider.winfo_width(), slider.winfo_height())
        if not (rx <= px <= rx + rw and ry <= py <= ry + rh):
            return 'break'
    except Exception:
        return 'break'
    step = 1 if getattr(event, 'num', None) == 4 or getattr(event, 'delta', 0) > 0 else -1
    cur = int(slider.get())
    new = max(0, min(cur + step, stack.shape[0] - 1))
    slider.set(new)
    self.pd_frame_var.set(new)
    _tab7_slider_changed(self, new)
    return 'break'

def _tab7_zoom(self, event):
    if event is None or event.inaxes not in (getattr(self, 'pd_ax_gray', None), getattr(self, 'pd_ax_color', None)) or event.xdata is None or (event.ydata is None):
        return
    ax = event.inaxes
    xl = ax.get_xlim()
    yl = ax.get_ylim()
    x = float(event.xdata)
    y = float(event.ydata)
    f = 0.8 if getattr(event, 'button', None) == 'up' else 1.25
    x0 = x - (x - xl[0]) * f
    x1 = x + (xl[1] - x) * f
    y0 = y - (y - yl[0]) * f
    y1 = y + (yl[1] - y) * f
    stack = _tab7_get_stack(self)
    if stack is not None:
        h, w = stack[int(self.pd_frame_var.get())].shape[:2]
        x0 = max(-0.5, x0)
        x1 = min(w - 0.5, x1)
        y0 = max(-0.5, y0)
        y1 = min(h - 0.5, y1)
    for aa in (self.pd_ax_gray, self.pd_ax_color):
        aa.set_xlim(x0, x1)
        aa.set_ylim(y0, y1)
    self._pd_zoom_limits = ((x0, x1), (y0, y1))
    self.pd_canvas.draw_idle()

def _tab7_reset_zoom(self):
    self._pd_zoom_limits = None
    _tab7_update_display(self, preserve_zoom=False)
    self.pd_status_var.set('ROI zoom reset.')

def _tab7_select(self, event):
    if event is None or event.inaxes not in (getattr(self, 'pd_ax_gray', None), getattr(self, 'pd_ax_color', None)) or event.xdata is None or (event.ydata is None):
        return
    dets = getattr(self, 'pd_current_detections', []) or []
    if not dets:
        return
    x, y = (float(event.xdata), float(event.ydata))
    ds = [(np.hypot(d['x'] - x, d['y'] - y), i) for i, d in enumerate(dets)]
    dist, i = min(ds)
    if dist > max(8.0, 0.6 * float(dets[i].get('diameter_px', 10))):
        return
    ctrl = bool(getattr(event, 'key', None) in ('control', 'ctrl', 'Control'))
    if not hasattr(self, 'pd_multi_selected'):
        self.pd_multi_selected = set()
    if ctrl:
        if i in self.pd_multi_selected:
            self.pd_multi_selected.remove(i)
        else:
            self.pd_multi_selected.add(i)
        self.pd_selected_detection = i
    else:
        self.pd_multi_selected = {i}
        self.pd_selected_detection = i
    d = dets[i]
    self.pd_particle_status.config(text=f"Particle #{i + 1}\nOuter diameter = {d['diameter_px']:.2f} px\nCore diameter = {d['core_diameter_px']:.2f} px\nCore aspect ratio = {d['core_aspect_ratio']:.3f}\nCore eccentricity = {d['core_eccentricity']:.3f}\nCore solidity = {d['core_solidity']:.3f}\nClassification = PARTICLE")
    self._tab7_draw_overlay()

def _tab7_refresh_density_row(self, idx):
    stack = _tab7_get_stack(self)
    if stack is None:
        return
    dets = [d for d in getattr(self, 'pd_frame_detections', {}).get(idx, []) if d.get('accepted', True)]
    h, w = stack[int(idx)].shape[:2]
    sx, sy = _tab8_authoritative_nm_scales(self)
    area_um2 = ((float(w) * float(sx)) * (float(h) * float(sy)) / 1.0e6) if (sx and sy and sx > 0 and sy > 0) else np.nan
    count = len(dets)
    dens = count / area_um2 if area_um2 > 0 else 0.0
    md = float(np.mean([d['diameter_px'] for d in dets])) if dets else np.nan
    mcd = float(np.mean([d['core_diameter_px'] for d in dets])) if dets else np.nan
    mar = float(np.mean([d['core_aspect_ratio'] for d in dets])) if dets else np.nan
    field = self._tab7_field(idx)
    self.pd_results[int(idx)] = (int(idx) + 1, field, count, area_um2, dens, md, mcd, mar)
    self._tab7_refresh_table()
    self._tab7_plot_density()

def _tab7_refresh_table(self):
    if not hasattr(self, 'pd_table'):
        return
    for it in self.pd_table.get_children():
        self.pd_table.delete(it)
    for k in sorted(getattr(self, 'pd_results', {}) or {}):
        self.pd_table.insert('', 'end', values=self.pd_results[k])

def _tab7_plot_density(self):
    if not hasattr(self, 'pd_plot_canvas'):
        return
    ax = self.pd_plot_ax
    ax2 = self.pd_plot_ax2
    ax.clear()
    ax2.clear()
    ax2.patch.set_visible(False)
    rows = [self.pd_results[k] for k in sorted(getattr(self, 'pd_results', {}) or {})]
    if rows:
        frames = np.array([r[0] for r in rows], float)
        fields = np.array([r[1] for r in rows], float)
        counts = np.array([r[2] for r in rows], float)
        dens = np.array([r[4] for r in rows], float)
        ax.plot(fields, counts, 'o-', label='Count')
        ax.set_xlabel('Field (mT)')
        ax.set_ylabel('Particle count')
        ax.grid(True, alpha=0.25)
        ax2.plot(frames, dens, 's--', label='Density')
        ax2.set_ylabel('Density (skyrmions/µm²)')
        ax2.yaxis.tick_right()
        ax2.yaxis.set_label_position('right')
        ax2.xaxis.tick_top()
        ax2.set_xlabel('Equivalent frame no.')
        ax2.xaxis.set_label_position('top')
    else:
        ax.text(0.5, 0.5, 'Analyze a frame to plot results', ha='center', va='center', transform=ax.transAxes)
    try:
        self.pd_plot_canvas.draw_idle()
    except Exception:
        pass

def _tab7_analyze_current(self):
    stack = _tab7_get_stack(self)
    if stack is None:
        messagebox.showwarning('Skyrmion Analysis', 'Confirm the Secondary ROI in Tab 5 first.', parent=self.root)
        return
    idx = max(0, min(int(self.pd_frame_var.get()), stack.shape[0] - 1))
    self.pd_frame_var.set(idx)
    det = self._tab7_detect(np.asarray(stack[idx], dtype=float))
    self.pd_current_detections = [dict(d) for d in det]
    self.pd_frame_detections[idx] = [dict(d) for d in det]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    _tab7_update_display(self, preserve_zoom=True)
    self._tab7_draw_overlay()
    _tab7_refresh_density_row(self, idx)
    self.pd_status_var.set(f'Frame {idx + 1}: {len(det)} accepted particle(s). Core AR ≤ {float(self.pd_max_aspect_var.get()):.2f} classified as PARTICLE.')

def _tab7_analyze_selected(self):
    stack = _tab7_get_stack(self)
    if stack is None:
        messagebox.showwarning('Skyrmion Analysis', 'Confirm the Secondary ROI in Tab 5 first.', parent=self.root)
        return
    n = stack.shape[0]
    try:
        a = int(self.pd_range_start_var.get()) - 1
        b = int(self.pd_range_end_var.get()) - 1
    except Exception:
        a, b = (0, n - 1)
    a = max(0, min(a, n - 1))
    b = max(a, min(b, n - 1))
    old = int(self.pd_frame_var.get())
    for idx in range(a, b + 1):
        det = self._tab7_detect(np.asarray(stack[idx], dtype=float))
        self.pd_frame_detections[idx] = [dict(d) for d in det]
        _tab7_refresh_density_row(self, idx)
    self.pd_frame_var.set(old if a <= old <= b else a)
    _tab7_slider_changed(self, self.pd_frame_var.get())
    self.pd_status_var.set(f'Analyzed selected frames {a + 1}–{b + 1}.')

def _tab7_delete(self):
    dets = getattr(self, 'pd_current_detections', []) or []
    sel = sorted(set(getattr(self, 'pd_multi_selected', set()) or set()), reverse=True)
    if not sel and getattr(self, 'pd_selected_detection', None) is not None:
        sel = [int(self.pd_selected_detection)]
    if not sel:
        self.pd_status_var.set('Select one or more particles first. Ctrl + left-click for multiple.')
        return
    idx = int(self.pd_frame_var.get())
    deleted = [dict(dets[j]) for j in sel if 0 <= j < len(dets)]
    if self.pd_nnc_var.get():
        ex = self.pd_nnc_exclusions.setdefault(idx, [])
        ex.extend([(float(d['x']), float(d['y'])) for d in deleted])
        self.pd_last_deleted = None
    else:
        self.pd_last_deleted = (idx, deleted)
    dets = [d for j, d in enumerate(dets) if j not in set(sel)]
    self.pd_current_detections = dets
    self.pd_frame_detections[idx] = [dict(d) for d in dets]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._tab7_draw_overlay()
    _tab7_refresh_density_row(self, idx)
    self.pd_status_var.set(f'Deleted {len(deleted)} particle(s) from frame {idx + 1}.')

def _tab7_undo(self):
    rec = getattr(self, 'pd_last_deleted', None)
    if not rec:
        self.pd_status_var.set('Nothing to undo. NNC deletions are permanent.')
        return
    idx, deleted = rec
    cur = [dict(d) for d in self.pd_frame_detections.get(idx, [])]
    cur.extend(deleted)
    cur.sort(key=lambda d: (d['y'], d['x']))
    self.pd_frame_detections[idx] = cur
    if int(self.pd_frame_var.get()) == idx:
        self.pd_current_detections = [dict(d) for d in cur]
    self.pd_last_deleted = None
    self._tab7_draw_overlay()
    _tab7_refresh_density_row(self, idx)
    self.pd_status_var.set(f'Restored {len(deleted)} particle(s).')

def _tab7_clear_nnc(self):
    self.pd_nnc_exclusions = {}
    self.pd_status_var.set('NNC exclusions cleared. Re-analyze affected frames.')

def _tab7_export_table(self):
    if not getattr(self, 'pd_results', None):
        self._tab7_analyze_current()
    if not getattr(self, 'pd_results', None):
        return
    fp = filedialog.asksaveasfilename(parent=self.root, initialdir=self._export_initialdir(), title='Export skyrmion table', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('Excel', '*.xlsx')])
    if not fp:
        return
    rows = [self.pd_results[k] for k in sorted(self.pd_results)]
    cols = ['Frame', 'Field (mT)', 'Count', 'ROI area (µm²)', 'Density (skyrmions/µm²)', 'Mean outer diameter (px)', 'Mean core diameter (px)', 'Mean core aspect ratio']
    df = pd.DataFrame(rows, columns=cols)
    try:
        if fp.lower().endswith('.xlsx'):
            df.to_excel(fp, index=False)
        else:
            df.to_csv(fp, index=False)
        self.pd_status_var.set(f'Exported table: {fp}')
    except Exception as exc:
        messagebox.showerror('Export table', str(exc), parent=self.root)

def _tab7_export_tiff(self):
    stack = _tab7_get_stack(self)
    if stack is None:
        return
    idx = max(0, min(int(self.pd_frame_var.get()), stack.shape[0] - 1))
    fp = filedialog.asksaveasfilename(parent=self.root, initialdir=self._export_initialdir(), title='Export selected ROI — TIFF 300 PPI', defaultextension='.tif', filetypes=[('TIFF', '*.tif *.tiff')])
    if not fp:
        return
    from PIL import Image
    img = np.asarray(stack[idx], dtype=float)
    finite = img[np.isfinite(img)]
    nz = finite[np.abs(finite) > 1e-12] if finite.size else finite
    vals = nz if nz.size > 20 else finite
    lo, hi = np.percentile(vals, [1, 99]) if vals.size else (0, 1)
    u8 = np.uint8(np.clip((img - lo) / max(hi - lo, 1e-12) * 255, 0, 255))
    Image.fromarray(u8).save(fp, format='TIFF', dpi=(300, 300))
    self.pd_status_var.set(f'Exported 300 PPI ROI TIFF: {fp}')

def _tab7_build(self):
    self.pd_frame_var = tk.IntVar(value=0)
    self.pd_threshold_var = tk.DoubleVar(value=0.1)
    self.pd_polarity_var = tk.StringVar(value='both')
    self.pd_min_diam_var = tk.DoubleVar(value=10.0)
    self.pd_max_diam_var = tk.DoubleVar(value=50.0)
    self.pd_min_sep_var = tk.DoubleVar(value=50.0)
    self.pd_max_aspect_var = tk.DoubleVar(value=1.8)
    self.pd_core_contrast_var = tk.DoubleVar(value=0.55)
    self.pd_core_min_diam_var = tk.DoubleVar(value=4.0)
    self.pd_core_max_diam_var = tk.DoubleVar(value=40.0)
    self.pd_include_edge_var = tk.BooleanVar(value=False)
    self.pd_show_centers_var = tk.BooleanVar(value=True)
    self.pd_nnc_var = tk.BooleanVar(value=False)
    self.pd_status_var = tk.StringVar(value='Confirm Secondary ROI in Tab 5.')
    self.pd_auto_shape_text = '--'
    self.pd_results = {}
    self.pd_frame_detections = {}
    self.pd_current_detections = []
    self.pd_overlay_artists = []
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self.pd_last_deleted = None
    self.pd_nnc_exclusions = {}
    self.pd_last_all_candidates = []
    self._pd_zoom_limits = None
    for w in self.tab_particle_density.winfo_children():
        w.destroy()
    outer = ttk.Frame(self.tab_particle_density, padding=6)
    outer.pack(fill='both', expand=True)
    outer.columnconfigure(0, weight=4)
    outer.columnconfigure(1, weight=3)
    outer.rowconfigure(1, weight=1)
    outer.rowconfigure(2, weight=1)
    head = ttk.Frame(outer)
    head.grid(row=0, column=0, columnspan=2, sticky='ew', pady=(0, 4))
    ttk.Label(head, text='SKYRMION / PARTICLE — OUTER PARTICLE + INNER CORE + CORE SYMMETRY', font=('Arial', 11, 'bold')).pack(side='left')
    ttk.Label(head, text='Source: confirmed Secondary ROI (Tab 5)', foreground='darkgreen').pack(side='left', padx=10)
    ttk.Label(head, text='Frame:').pack(side='left')
    self.pd_frame_slider = tk.Scale(head, from_=0, to=0, orient='horizontal', variable=self.pd_frame_var, resolution=1, length=250, showvalue=True, highlightthickness=0, command=self._tab7_slider_changed)
    self.pd_frame_slider.pack(side='left', padx=4)
    self.pd_frame_slider._tab7_armed = False
    self.pd_frame_slider.bind('<Button-1>', self._tab7_arm_slider, add='+')
    self.pd_frame_slider.bind('<MouseWheel>', self._tab7_slider_wheel, add='+')
    self.pd_frame_slider.bind('<Button-4>', self._tab7_slider_wheel, add='+')
    self.pd_frame_slider.bind('<Button-5>', self._tab7_slider_wheel, add='+')
    self.pd_frame_label = ttk.Label(head, text='--', width=24)
    self.pd_frame_label.pack(side='left', padx=3)
    ttk.Button(head, text='RESET ZOOM', command=self._tab7_reset_zoom).pack(side='left', padx=2)
    ttk.Button(head, text='ANALYZE CURRENT FRAME', command=self._tab7_analyze_current).pack(side='left', padx=2)
    ttk.Button(head, text='ANALYZE SELECTED FRAMES', command=self._tab7_analyze_selected).pack(side='left', padx=2)
    ttk.Button(head, text='DELETE SELECTED', command=self._tab7_delete).pack(side='left', padx=2)
    ttk.Button(head, text='UNDO DELETE', command=self._tab7_undo).pack(side='left', padx=2)
    roi_box = ttk.LabelFrame(outer, text='SELECTED ROI — GRAYSCALE + COLOR', padding=2)
    roi_box.grid(row=1, column=0, sticky='nsew', padx=(0, 4), pady=(0, 3))
    ctrl_box = ttk.LabelFrame(outer, text='DETECTION / ANALYSIS CONTROLS', padding=2)
    ctrl_box.grid(row=1, column=1, rowspan=2, sticky='nsew', padx=(4, 0), pady=(0, 3))
    graph_box = ttk.LabelFrame(outer, text='COUNT / DENSITY', padding=2)
    graph_box.grid(row=2, column=0, sticky='nsew', padx=(0, 4), pady=(3, 0))
    table_box = ttk.LabelFrame(outer, text='DENSITY TABLE', padding=2)
    table_box.grid(row=2, column=1, sticky='nsew', padx=(4, 0), pady=(3, 0))
    self.pd_fig, (self.pd_ax_gray, self.pd_ax_color) = plt.subplots(1, 2, figsize=(8.8, 4.8), dpi=100, constrained_layout=True)
    self.pd_image_artist_gray = self.pd_ax_gray.imshow(np.zeros((2, 2)), cmap='gray', origin='lower', interpolation='nearest')
    try:
        cmap = self.colormap_var.get() or 'gray'
    except Exception:
        cmap = 'gray'
    self.pd_image_artist_color = self.pd_ax_color.imshow(np.zeros((2, 2)), cmap=cmap, origin='lower', interpolation='nearest')
    self.pd_colorbar = self.pd_fig.colorbar(self.pd_image_artist_color, ax=self.pd_ax_color, fraction=0.046, pad=0.04)
    self.pd_colorbar.set_label('Intensity')
    for ax in (self.pd_ax_gray, self.pd_ax_color):
        ax.set_xlabel('X pixel')
        ax.set_ylabel('Y pixel')
        ax.set_aspect('equal', adjustable='box')
    self.pd_canvas = FigureCanvasTkAgg(self.pd_fig, master=roi_box)
    self.pd_canvas.draw()
    self.pd_canvas.get_tk_widget().pack(fill='both', expand=True)
    self.pd_canvas.mpl_connect('scroll_event', self._tab7_zoom)
    self.pd_canvas.mpl_connect('button_press_event', self._tab7_select)
    cc = tk.Canvas(ctrl_box, highlightthickness=0)
    sy = ttk.Scrollbar(ctrl_box, orient='vertical', command=cc.yview)
    cf = ttk.Frame(cc, padding=7)
    cw = cc.create_window((0, 0), window=cf, anchor='nw')
    cc.configure(yscrollcommand=sy.set)
    cc.pack(side='left', fill='both', expand=True)
    sy.pack(side='right', fill='y')
    cf.bind('<Configure>', lambda e: cc.configure(scrollregion=cc.bbox('all')))
    cc.bind('<Configure>', lambda e: cc.itemconfigure(cw, width=e.width))
    ttk.Label(cf, text='OUTER PARTICLE SEGMENTATION', font=('Arial', 10, 'bold')).pack(anchor='w')
    ttk.Label(cf, text='Detection threshold').pack(anchor='w')
    tk.Scale(cf, from_=0.01, to=0.5, resolution=0.01, orient='horizontal', variable=self.pd_threshold_var, length=260, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Polarity').pack(anchor='w')
    ttk.Combobox(cf, textvariable=self.pd_polarity_var, values=['bright', 'dark', 'both'], state='readonly').pack(anchor='w')
    ttk.Label(cf, text='Particle diameter: 10–50 px').pack(anchor='w')
    ttk.Label(cf, text='Minimum separation: 50 px').pack(anchor='w')
    ttk.Separator(cf).pack(fill='x', pady=6)
    ttk.Label(cf, text='INNER CORE SEGMENTATION', font=('Arial', 10, 'bold')).pack(anchor='w')
    ttk.Label(cf, text='Core contrast level (relative)').pack(anchor='w')
    tk.Scale(cf, from_=0.1, to=0.95, resolution=0.01, orient='horizontal', variable=self.pd_core_contrast_var, length=260, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Core minimum diameter (px)').pack(anchor='w')
    tk.Scale(cf, from_=1, to=30, resolution=1, orient='horizontal', variable=self.pd_core_min_diam_var, length=260, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Core maximum diameter (px)').pack(anchor='w')
    tk.Scale(cf, from_=5, to=60, resolution=1, orient='horizontal', variable=self.pd_core_max_diam_var, length=260, showvalue=True).pack(fill='x')
    ttk.Separator(cf).pack(fill='x', pady=6)
    ttk.Label(cf, text='CORE SYMMETRY CLASSIFICATION', font=('Arial', 10, 'bold')).pack(anchor='w')
    ttk.Label(cf, text='Maximum core aspect ratio').pack(anchor='w')
    tk.Scale(cf, from_=1.0, to=3.0, resolution=0.01, orient='horizontal', variable=self.pd_max_aspect_var, length=260, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='AR ≤ limit → PARTICLE | AR > limit → WORM', wraplength=290).pack(anchor='w')
    ttk.Label(cf, textvariable=tk.StringVar(value='Classification uses ONLY the inner core.'), foreground='darkgreen', wraplength=290).pack(anchor='w', pady=3)
    ttk.Checkbutton(cf, text='Include edge detections', variable=self.pd_include_edge_var).pack(anchor='w')
    ttk.Checkbutton(cf, text='Show outer/core overlays', variable=self.pd_show_centers_var, command=self._tab7_draw_overlay).pack(anchor='w')
    ttk.Checkbutton(cf, text='NNC-Delete — No Need to Consider', variable=self.pd_nnc_var).pack(anchor='w')
    ttk.Button(cf, text='CLEAR NNC EXCLUSIONS', command=self._tab7_clear_nnc).pack(fill='x', pady=2)
    ttk.Label(cf, text='SELECTED PARTICLE', font=('Arial', 10, 'bold')).pack(anchor='w', pady=(6, 2))
    self.pd_particle_status = ttk.Label(cf, text='Click an accepted particle/centre in the ROI.', wraplength=290)
    self.pd_particle_status.pack(anchor='w')
    ttk.Label(cf, text='Start frame').pack(anchor='w', pady=(6, 0))
    self.pd_range_start_var = tk.IntVar(value=1)
    tk.Spinbox(cf, from_=1, to=9999, textvariable=self.pd_range_start_var, width=8, increment=1).pack(anchor='w')
    ttk.Label(cf, text='End frame').pack(anchor='w')
    self.pd_range_end_var = tk.IntVar(value=1)
    tk.Spinbox(cf, from_=1, to=9999, textvariable=self.pd_range_end_var, width=8, increment=1).pack(anchor='w')
    ttk.Label(cf, textvariable=self.pd_status_var, wraplength=290).pack(anchor='w', pady=5)
    ttk.Button(cf, text='EXPORT TABLE', command=self._tab7_export_table).pack(fill='x', pady=2)
    ttk.Button(cf, text='EXPORT SELECTED ROI — TIFF', command=self._tab7_export_tiff).pack(fill='x', pady=2)
    self.pd_plot_fig = plt.Figure(figsize=(7, 2.5), dpi=100)
    self.pd_plot_ax = self.pd_plot_fig.add_axes([0.1, 0.2, 0.72, 0.62])
    self.pd_plot_ax2 = self.pd_plot_fig.add_axes(self.pd_plot_ax.get_position(), frameon=False)
    self.pd_plot_ax2.patch.set_visible(False)
    self.pd_plot_canvas = FigureCanvasTkAgg(self.pd_plot_fig, master=graph_box)
    self.pd_plot_canvas.draw()
    self.pd_plot_canvas.get_tk_widget().pack(fill='both', expand=True)
    tf = ttk.Frame(table_box)
    tf.pack(fill='both', expand=True)
    cols = ('Frame', 'Field (mT)', 'Count', 'ROI area (µm²)', 'Density (skyrmions/µm²)', 'Mean outer d (px)', 'Mean core d (px)', 'Mean core AR')
    self.pd_table = ttk.Treeview(tf, columns=cols, show='headings')
    for c in cols:
        self.pd_table.heading(c, text=c)
        self.pd_table.column(c, width=110, anchor='center')
    tv = ttk.Scrollbar(tf, orient='vertical', command=self.pd_table.yview)
    self.pd_table.configure(yscrollcommand=tv.set)
    self.pd_table.pack(side='left', fill='both', expand=True)
    tv.pack(side='right', fill='y')
    self._tab7_sync_secondary_roi(preserve_zoom=False)

def _tab7_sync_secondary_roi(self, preserve_zoom=True):
    return _tab7_refresh_from_roi(self, preserve_zoom=preserve_zoom)

def _tab7_tab_changed(self, event=None):
    try:
        if self.nb.select() == str(self.tab_particle_density):
            self._tab7_sync_secondary_roi(preserve_zoom=True)
    except Exception:
        pass
CDIWorkflowApp._tab7_get_stack = _tab7_get_stack
CDIWorkflowApp._tab7_field = _tab7_field
CDIWorkflowApp._tab7_normalize = _tab7_normalize
CDIWorkflowApp._tab7_detect = _tab7_detect
CDIWorkflowApp._tab7_clear_artists = _tab7_clear_artists
CDIWorkflowApp._tab7_draw_overlay = _tab7_draw_overlay
CDIWorkflowApp._tab7_update_display = _tab7_update_display
CDIWorkflowApp._tab7_refresh_from_roi = _tab7_refresh_from_roi
CDIWorkflowApp._tab7_slider_changed = _tab7_slider_changed
CDIWorkflowApp._tab7_arm_slider = _tab7_arm_slider
CDIWorkflowApp._tab7_slider_wheel = _tab7_slider_wheel
CDIWorkflowApp._tab7_zoom = _tab7_zoom
CDIWorkflowApp._tab7_reset_zoom = _tab7_reset_zoom
CDIWorkflowApp._tab7_select = _tab7_select
CDIWorkflowApp._tab7_analyze_current = _tab7_analyze_current
CDIWorkflowApp._tab7_analyze_selected = _tab7_analyze_selected
CDIWorkflowApp._tab7_delete = _tab7_delete
CDIWorkflowApp._tab7_undo = _tab7_undo
CDIWorkflowApp._tab7_clear_nnc = _tab7_clear_nnc
CDIWorkflowApp._tab7_export_table = _tab7_export_table
CDIWorkflowApp._tab7_export_tiff = _tab7_export_tiff
CDIWorkflowApp._tab7_refresh_density_row = _tab7_refresh_density_row
CDIWorkflowApp._tab7_refresh_table = _tab7_refresh_table
CDIWorkflowApp._tab7_plot_density = _tab7_plot_density
CDIWorkflowApp._tab7_sync_secondary_roi = _tab7_sync_secondary_roi
try:
    CDIWorkflowApp._tab7_tab_changed = _tab7_tab_changed
except Exception:
    pass

def _tab7_exact_source_stack(self):
    """Return the exact confirmed Secondary ROI stack created by Tab 5."""
    stack = getattr(self, 'roi2_stack', None)
    if stack is None:
        return None
    try:
        arr = np.asarray(stack)
    except Exception:
        return None
    if arr.ndim != 3 or arr.shape[0] < 1:
        return None
    return arr

def _tab7_exact_update_display(self, preserve_zoom=True):
    stack = _tab7_exact_source_stack(self)
    if stack is None:
        try:
            self.pd_status_var.set('Waiting for confirmed Secondary ROI from Tab 5.')
            self.pd_frame_slider.configure(from_=0, to=0, state='disabled')
            self.pd_frame_label.config(text='--')
        except Exception:
            pass
        return False
    n = int(stack.shape[0])
    try:
        idx = int(round(float(self.pd_frame_var.get())))
    except Exception:
        idx = 0
    idx = max(0, min(idx, n - 1))
    self.pd_frame_var.set(idx)
    try:
        self.pd_frame_slider.configure(from_=0, to=n - 1, state='normal')
    except Exception:
        pass
    image = np.asarray(stack[idx], dtype=float)
    if image.ndim != 2 or image.size == 0:
        return False
    cmap_name = 'gray'
    try:
        cmap_name = self.colormap_var.get() or 'gray'
    except Exception:
        pass
    self.pd_image_artist_gray.set_data(image)
    self.pd_image_artist_gray.set_cmap('gray')
    self.pd_image_artist_gray.set_clim(0.0, 1.0)
    self.pd_image_artist_gray.set_extent((-0.5, image.shape[1] - 0.5, -0.5, image.shape[0] - 0.5))
    self.pd_image_artist_color.set_data(image)
    self.pd_image_artist_color.set_cmap(cmap_name)
    self.pd_image_artist_color.set_clim(0.0, 1.0)
    self.pd_image_artist_color.set_extent((-0.5, image.shape[1] - 0.5, -0.5, image.shape[0] - 0.5))
    try:
        self.pd_colorbar.update_normal(self.pd_image_artist_color)
    except Exception:
        pass
    h, w = image.shape
    old_zoom = getattr(self, '_pd_zoom_limits', None) if preserve_zoom else None
    if old_zoom is None:
        xlim = (-0.5, float(w) - 0.5)
        ylim = (-0.5, float(h) - 0.5)
        self._pd_zoom_limits = None
    else:
        xlim, ylim = old_zoom
    for ax in (self.pd_ax_gray, self.pd_ax_color):
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_aspect('equal', adjustable='box')
    try:
        field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    except Exception:
        field = float(idx)
    self.pd_ax_gray.set_title(f'Secondary ROI — Grayscale | Frame {idx + 1}/{n} | Field = {field:+.2f} mT')
    self.pd_ax_color.set_title(f'Secondary ROI — Color | Frame {idx + 1}/{n} | Field = {field:+.2f} mT')
    self.pd_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT')
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass
    return True

def _tab7_exact_slider_changed(self, value=None):
    stack = _tab7_exact_source_stack(self)
    if stack is None:
        return
    try:
        idx = int(round(float(value if value is not None else self.pd_frame_var.get())))
    except Exception:
        idx = int(self.pd_frame_var.get())
    idx = max(0, min(idx, stack.shape[0] - 1))
    self.pd_frame_var.set(idx)
    _tab7_exact_update_display(self, preserve_zoom=True)
    self.pd_current_detections = [dict(d) for d in getattr(self, 'pd_frame_detections', {}).get(idx, []) or []]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    try:
        self._tab7_draw_overlay()
    except Exception:
        pass

def _tab7_exact_arm_slider(self, event=None):
    try:
        self.pd_frame_slider._tab7_exact_armed = True
        self.pd_frame_slider.focus_set()
    except Exception:
        pass
    return None

def _tab7_exact_slider_wheel(self, event):
    slider = getattr(self, 'pd_frame_slider', None)
    stack = _tab7_exact_source_stack(self)
    if slider is None or stack is None:
        return 'break'
    if not getattr(slider, '_tab7_exact_armed', False):
        return 'break'
    try:
        px = slider.winfo_toplevel().winfo_pointerx()
        py = slider.winfo_toplevel().winfo_pointery()
        rx, ry = (slider.winfo_rootx(), slider.winfo_rooty())
        rw, rh = (slider.winfo_width(), slider.winfo_height())
        if not (rx <= px <= rx + rw and ry <= py <= ry + rh):
            return 'break'
    except Exception:
        return 'break'
    step = 1 if getattr(event, 'num', None) == 4 or getattr(event, 'delta', 0) > 0 or getattr(event, 'step', 0) > 0 else -1
    cur = int(slider.get())
    new = max(0, min(cur + step, stack.shape[0] - 1))
    slider.set(new)
    self.pd_frame_var.set(new)
    _tab7_exact_slider_changed(self, new)
    return 'break'

def _tab7_exact_zoom(self, event):
    if event is None or event.inaxes not in (getattr(self, 'pd_ax_gray', None), getattr(self, 'pd_ax_color', None)):
        return
    if event.xdata is None or event.ydata is None:
        return
    ax = event.inaxes
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x = float(event.xdata)
    y = float(event.ydata)
    if getattr(event, 'button', None) == 'up' or getattr(event, 'step', 0) > 0:
        factor = 1.22
    else:
        factor = 1.0 / 1.22
    width = xlim[1] - xlim[0]
    height = ylim[1] - ylim[0]
    new_width = width / factor
    new_height = height / factor
    px = (x - xlim[0]) / width if width else 0.5
    py = (y - ylim[0]) / height if height else 0.5
    nx0 = x - px * new_width
    nx1 = x + (1.0 - px) * new_width
    ny0 = y - py * new_height
    ny1 = y + (1.0 - py) * new_height
    stack = _tab7_exact_source_stack(self)
    if stack is not None:
        h, w = stack[int(self.pd_frame_var.get())].shape[:2]
        nx0 = max(-0.5, nx0)
        nx1 = min(w - 0.5, nx1)
        ny0 = max(-0.5, ny0)
        ny1 = min(h - 0.5, ny1)
    limits = ((nx0, nx1), (ny0, ny1))
    for a in (self.pd_ax_gray, self.pd_ax_color):
        a.set_xlim(*limits[0])
        a.set_ylim(*limits[1])
        a.set_aspect('equal', adjustable='box')
    self._pd_zoom_limits = limits
    self.pd_canvas.draw_idle()
    return 'break'

def _tab7_exact_reset_zoom(self):
    self._pd_zoom_limits = None
    _tab7_exact_update_display(self, preserve_zoom=False)
    try:
        self.pd_status_var.set('ROI zoom reset.')
    except Exception:
        pass

def _tab7_exact_build(self):
    """Clean Tab 7 UI using the confirmed Tab-5 Secondary ROI as its only source."""
    self.pd_frame_var = tk.IntVar(value=0)
    self.pd_threshold_var = tk.DoubleVar(value=0.1)
    self.pd_polarity_var = tk.StringVar(value='both')
    self.pd_min_diam_var = tk.DoubleVar(value=10.0)
    self.pd_max_diam_var = tk.DoubleVar(value=50.0)
    self.pd_min_sep_var = tk.DoubleVar(value=50.0)
    self.pd_max_aspect_var = tk.DoubleVar(value=1.8)
    self.pd_core_contrast_var = tk.DoubleVar(value=0.55)
    self.pd_core_min_diam_var = tk.DoubleVar(value=4.0)
    self.pd_core_max_diam_var = tk.DoubleVar(value=40.0)
    self.pd_include_edge_var = tk.BooleanVar(value=False)
    self.pd_show_centers_var = tk.BooleanVar(value=True)
    self.pd_nnc_var = tk.BooleanVar(value=False)
    self.pd_status_var = tk.StringVar(value='Confirm Secondary ROI in Tab 5.')
    self.pd_results = {}
    self.pd_frame_detections = {}
    self.pd_current_detections = []
    self.pd_overlay_artists = []
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self.pd_last_deleted = None
    self.pd_nnc_exclusions = {}
    self.pd_last_all_candidates = []
    self.pd_auto_shape_text = '--'
    self._pd_zoom_limits = None
    for wdg in self.tab_particle_density.winfo_children():
        wdg.destroy()
    outer = ttk.Frame(self.tab_particle_density, padding=6)
    outer.pack(fill='both', expand=True)
    outer.columnconfigure(0, weight=4)
    outer.columnconfigure(1, weight=3)
    outer.rowconfigure(1, weight=1)
    outer.rowconfigure(2, weight=1)
    head = ttk.Frame(outer)
    head.grid(row=0, column=0, columnspan=2, sticky='ew', pady=(0, 4))
    ttk.Label(head, text='SKYRMION / PARTICLE — OUTER PARTICLE + INNER CORE + CORE SYMMETRY', font=('Arial', 11, 'bold')).pack(side='left')
    ttk.Label(head, text='Source: confirmed Secondary ROI (Tab 5)', foreground='darkgreen').pack(side='left', padx=10)
    ttk.Label(head, text='Frame:').pack(side='left', padx=(8, 3))
    self.pd_frame_slider = tk.Scale(head, from_=0, to=0, orient='horizontal', variable=self.pd_frame_var, resolution=1, length=250, showvalue=True, highlightthickness=0, command=self._pd_exact_slider_changed)
    self.pd_frame_slider.pack(side='left', padx=4)
    self.pd_frame_slider._tab7_exact_armed = False
    self.pd_frame_slider.bind('<Button-1>', self._pd_exact_arm_slider, add='+')
    self.pd_frame_slider.bind('<MouseWheel>', self._pd_exact_slider_wheel, add='+')
    self.pd_frame_slider.bind('<Button-4>', self._pd_exact_slider_wheel, add='+')
    self.pd_frame_slider.bind('<Button-5>', self._pd_exact_slider_wheel, add='+')
    self.pd_frame_label = ttk.Label(head, text='--', width=24)
    self.pd_frame_label.pack(side='left', padx=3)
    ttk.Button(head, text='RESET ZOOM', command=self._pd_exact_reset_zoom).pack(side='left', padx=2)
    ttk.Button(head, text='ANALYZE CURRENT FRAME', command=self._tab7_analyze_current).pack(side='left', padx=2)
    ttk.Button(head, text='ANALYZE SELECTED FRAMES', command=self._tab7_analyze_selected).pack(side='left', padx=2)
    ttk.Button(head, text='DELETE SELECTED', command=self._tab7_delete).pack(side='left', padx=2)
    ttk.Button(head, text='UNDO DELETE', command=self._tab7_undo).pack(side='left', padx=2)
    roi_box = ttk.LabelFrame(outer, text='SELECTED ROI — GRAYSCALE + COLOR', padding=2)
    roi_box.grid(row=1, column=0, sticky='nsew', padx=(0, 4), pady=(0, 3))
    ctrl_box = ttk.LabelFrame(outer, text='DETECTION / ANALYSIS CONTROLS', padding=2)
    ctrl_box.grid(row=1, column=1, rowspan=2, sticky='nsew', padx=(4, 0), pady=(0, 3))
    graph_box = ttk.LabelFrame(outer, text='COUNT / DENSITY', padding=2)
    graph_box.grid(row=2, column=0, sticky='nsew', padx=(0, 4), pady=(3, 0))
    table_box = ttk.LabelFrame(outer, text='DENSITY TABLE', padding=2)
    table_box.grid(row=2, column=1, sticky='nsew', padx=(4, 0), pady=(3, 0))
    self.pd_fig, (self.pd_ax_gray, self.pd_ax_color) = plt.subplots(1, 2, figsize=(8.8, 4.8), dpi=100, constrained_layout=True)
    self.pd_image_artist_gray = self.pd_ax_gray.imshow(np.zeros((2, 2)), cmap='gray', origin='lower', vmin=0, vmax=1, interpolation='nearest', extent=(-0.5, 1.5, -0.5, 1.5))
    try:
        cmap = self.colormap_var.get() or 'gray'
    except Exception:
        cmap = 'gray'
    self.pd_image_artist_color = self.pd_ax_color.imshow(np.zeros((2, 2)), cmap=cmap, origin='lower', vmin=0, vmax=1, interpolation='nearest', extent=(-0.5, 1.5, -0.5, 1.5))
    for ax in (self.pd_ax_gray, self.pd_ax_color):
        ax.set_xlabel('X pixel')
        ax.set_ylabel('Y pixel')
        ax.set_aspect('equal', adjustable='box')
    self.pd_colorbar = self.pd_fig.colorbar(self.pd_image_artist_color, ax=self.pd_ax_color, fraction=0.046, pad=0.04)
    self.pd_colorbar.set_label('Intensity')
    self.pd_canvas = FigureCanvasTkAgg(self.pd_fig, master=roi_box)
    self.pd_canvas.draw()
    self.pd_canvas.get_tk_widget().pack(fill='both', expand=True)
    self.pd_canvas.mpl_connect('scroll_event', self._pd_exact_zoom)
    self.pd_canvas.mpl_connect('button_press_event', self._tab7_select)
    ccanvas = tk.Canvas(ctrl_box, highlightthickness=0, borderwidth=0)
    cscroll = ttk.Scrollbar(ctrl_box, orient='vertical', command=ccanvas.yview)
    cframe = ttk.Frame(ccanvas, padding=7)
    cwin = ccanvas.create_window((0, 0), window=cframe, anchor='nw')
    ccanvas.configure(yscrollcommand=cscroll.set)
    ccanvas.pack(side='left', fill='both', expand=True)
    cscroll.pack(side='right', fill='y')
    cframe.bind('<Configure>', lambda e: ccanvas.configure(scrollregion=ccanvas.bbox('all')))
    ccanvas.bind('<Configure>', lambda e: ccanvas.itemconfigure(cwin, width=e.width))

    def _control_wheel(e):
        try:
            delta = getattr(e, 'delta', 0)
            if delta:
                ccanvas.yview_scroll(int(-delta / 120), 'units')
            elif getattr(e, 'num', None) == 4:
                ccanvas.yview_scroll(-3, 'units')
            elif getattr(e, 'num', None) == 5:
                ccanvas.yview_scroll(3, 'units')
        except Exception:
            pass
        return 'break'
    ccanvas.bind('<MouseWheel>', _control_wheel)
    ccanvas.bind('<Button-4>', _control_wheel)
    ccanvas.bind('<Button-5>', _control_wheel)
    cframe.bind('<MouseWheel>', _control_wheel)
    cframe.bind('<Button-4>', _control_wheel)
    cframe.bind('<Button-5>', _control_wheel)
    ttk.Label(cframe, text='OUTER PARTICLE SEGMENTATION', font=('Arial', 10, 'bold')).pack(anchor='w')
    ttk.Label(cframe, text='Detection threshold').pack(anchor='w', pady=(4, 0))
    tk.Scale(cframe, from_=0.01, to=0.5, resolution=0.01, orient='horizontal', variable=self.pd_threshold_var, length=270, showvalue=True).pack(fill='x')
    ttk.Label(cframe, text='Polarity').pack(anchor='w', pady=(3, 0))
    ttk.Combobox(cframe, textvariable=self.pd_polarity_var, values=['bright', 'dark', 'both'], state='readonly', width=12).pack(anchor='w')
    ttk.Label(cframe, text='Outer particle diameter: 10–50 px').pack(anchor='w', pady=(5, 0))
    ttk.Label(cframe, text='Minimum separation: 50 px').pack(anchor='w')
    ttk.Separator(cframe).pack(fill='x', pady=6)
    ttk.Label(cframe, text='INNER CORE SEGMENTATION', font=('Arial', 10, 'bold')).pack(anchor='w')
    ttk.Label(cframe, text='Core contrast level (relative)').pack(anchor='w', pady=(4, 0))
    tk.Scale(cframe, from_=0.1, to=0.95, resolution=0.01, orient='horizontal', variable=self.pd_core_contrast_var, length=270, showvalue=True).pack(fill='x')
    ttk.Label(cframe, text='Core minimum diameter (px)').pack(anchor='w')
    tk.Scale(cframe, from_=1, to=30, resolution=1, orient='horizontal', variable=self.pd_core_min_diam_var, length=270, showvalue=True).pack(fill='x')
    ttk.Label(cframe, text='Core maximum diameter (px)').pack(anchor='w')
    tk.Scale(cframe, from_=5, to=60, resolution=1, orient='horizontal', variable=self.pd_core_max_diam_var, length=270, showvalue=True).pack(fill='x')
    ttk.Separator(cframe).pack(fill='x', pady=6)
    ttk.Label(cframe, text='CORE SYMMETRY CLASSIFICATION', font=('Arial', 10, 'bold')).pack(anchor='w')
    ttk.Label(cframe, text='Maximum core aspect ratio').pack(anchor='w')
    tk.Scale(cframe, from_=1.0, to=3.0, resolution=0.01, orient='horizontal', variable=self.pd_max_aspect_var, length=270, showvalue=True).pack(fill='x')
    ttk.Label(cframe, text='Core AR ≤ limit → PARTICLE | Core AR > limit → WORM', wraplength=300).pack(anchor='w', pady=3)
    ttk.Checkbutton(cframe, text='Include edge detections', variable=self.pd_include_edge_var).pack(anchor='w')
    ttk.Checkbutton(cframe, text='Show outer/core overlays', variable=self.pd_show_centers_var, command=self._tab7_draw_overlay).pack(anchor='w')
    ttk.Checkbutton(cframe, text='NNC-Delete — No Need to Consider', variable=self.pd_nnc_var).pack(anchor='w')
    ttk.Button(cframe, text='CLEAR NNC EXCLUSIONS', command=self._tab7_clear_nnc).pack(fill='x', pady=2)
    ttk.Label(cframe, text='SELECTED PARTICLE', font=('Arial', 10, 'bold')).pack(anchor='w', pady=(6, 2))
    self.pd_particle_status = ttk.Label(cframe, text='Click an accepted particle/centre in the ROI.', wraplength=300, justify='left')
    self.pd_particle_status.pack(anchor='w')
    ttk.Label(cframe, text='Start frame').pack(anchor='w', pady=(6, 0))
    self.pd_range_start_var = tk.IntVar(value=1)
    tk.Spinbox(cframe, from_=1, to=9999, textvariable=self.pd_range_start_var, width=8, increment=1).pack(anchor='w')
    ttk.Label(cframe, text='End frame').pack(anchor='w')
    self.pd_range_end_var = tk.IntVar(value=1)
    tk.Spinbox(cframe, from_=1, to=9999, textvariable=self.pd_range_end_var, width=8, increment=1).pack(anchor='w')
    ttk.Label(cframe, textvariable=self.pd_status_var, wraplength=300, justify='left').pack(anchor='w', pady=5)
    ttk.Separator(cframe).pack(fill='x', pady=4)
    ttk.Button(cframe, text='EXPORT TABLE', command=self._tab7_export_table).pack(fill='x', pady=2)
    ttk.Button(cframe, text='EXPORT SELECTED ROI — TIFF', command=self._tab7_export_tiff).pack(fill='x', pady=2)
    self.pd_plot_fig = plt.Figure(figsize=(7.0, 2.5), dpi=100)
    self.pd_plot_ax = self.pd_plot_fig.add_axes([0.1, 0.2, 0.72, 0.62])
    self.pd_plot_ax2 = self.pd_plot_fig.add_axes(self.pd_plot_ax.get_position(), frameon=False)
    self.pd_plot_ax2.patch.set_visible(False)
    self.pd_plot_canvas = FigureCanvasTkAgg(self.pd_plot_fig, master=graph_box)
    self.pd_plot_canvas.draw()
    self.pd_plot_canvas.get_tk_widget().pack(fill='both', expand=True)
    tf = ttk.Frame(table_box)
    tf.pack(fill='both', expand=True)
    cols = ('Frame', 'Field (mT)', 'Count', 'ROI area (µm²)', 'Density (skyrmions/µm²)', 'Mean outer d (px)', 'Mean core d (px)', 'Mean core AR')
    self.pd_table = ttk.Treeview(tf, columns=cols, show='headings')
    for c in cols:
        self.pd_table.heading(c, text=c)
        self.pd_table.column(c, width=110, anchor='center')
    tscroll = ttk.Scrollbar(tf, orient='vertical', command=self.pd_table.yview)
    self.pd_table.configure(yscrollcommand=tscroll.set)
    self.pd_table.pack(side='left', fill='both', expand=True)
    tscroll.pack(side='right', fill='y')
    self._tab7_exact_update_display(preserve_zoom=False)
    if _tab7_exact_source_stack(self) is not None:
        n = int(_tab7_exact_source_stack(self).shape[0])
        self.pd_frame_slider.configure(from_=0, to=n - 1, state='normal')
        self.pd_range_end_var.set(max(1, n))
    try:
        self.nb.bind('<<NotebookTabChanged>>', self._tab7_clean_tab_changed, add='+')
    except Exception:
        pass

def _tab7_clean_tab_changed(self, event=None):
    try:
        if self.nb.select() == str(self.tab_particle_density):
            self._tab7_exact_update_display(preserve_zoom=True)
            idx = int(self.pd_frame_var.get())
            self.pd_current_detections = [dict(d) for d in getattr(self, 'pd_frame_detections', {}).get(idx, []) or []]
            self._tab7_draw_overlay()
    except Exception:
        pass
CDIWorkflowApp._tab7_exact_source_stack = _tab7_exact_source_stack
CDIWorkflowApp._tab7_exact_update_display = _tab7_exact_update_display
CDIWorkflowApp._tab7_exact_slider_changed = _tab7_exact_slider_changed
CDIWorkflowApp._tab7_exact_arm_slider = _tab7_exact_arm_slider
CDIWorkflowApp._tab7_exact_slider_wheel = _tab7_exact_slider_wheel
CDIWorkflowApp._tab7_exact_zoom = _tab7_exact_zoom
CDIWorkflowApp._tab7_exact_reset_zoom = _tab7_exact_reset_zoom
CDIWorkflowApp._tab7_clean_tab_changed = _tab7_clean_tab_changed
CDIWorkflowApp._tab7_update_display = _tab7_exact_update_display
CDIWorkflowApp._tab7_slider_changed = _tab7_exact_slider_changed
CDIWorkflowApp._tab7_arm_slider = _tab7_exact_arm_slider
CDIWorkflowApp._tab7_slider_wheel = _tab7_exact_slider_wheel
CDIWorkflowApp._tab7_zoom = _tab7_exact_zoom
CDIWorkflowApp._tab7_reset_zoom = _tab7_exact_reset_zoom
CDIWorkflowApp._pd3_update_roi_image = _tab7_exact_update_display
CDIWorkflowApp._pd3_frame_changed = _tab7_exact_slider_changed
CDIWorkflowApp._pd3_frame_wheel = _tab7_exact_slider_wheel
CDIWorkflowApp._pd3_zoom_scroll = _tab7_exact_zoom
CDIWorkflowApp._pd3_reset_zoom = _tab7_exact_reset_zoom
CDIWorkflowApp._tab7_analyze_current = _tab7_analyze_current
CDIWorkflowApp._tab7_analyze_selected = _tab7_analyze_selected
CDIWorkflowApp._tab7_select = _tab7_select
CDIWorkflowApp._tab7_delete = _tab7_delete
CDIWorkflowApp._tab7_undo = _tab7_undo
CDIWorkflowApp._tab7_draw_overlay = _tab7_draw_overlay
CDIWorkflowApp._tab7_clear_nnc = _tab7_clear_nnc
CDIWorkflowApp._tab7_export_table = _tab7_export_table
CDIWorkflowApp._tab7_export_tiff = _tab7_export_tiff
CDIWorkflowApp._tab7_detect = _tab7_detect
CDIWorkflowApp._tab7_refresh_density_row = _tab7_refresh_density_row
CDIWorkflowApp._tab7_refresh_table = _tab7_refresh_table
CDIWorkflowApp._tab7_plot_density = _tab7_plot_density

def _tab7_v3_stack(self):
    return _tab7_get_stack(self)

def _tab7_v3_field(self, idx):
    return _tab7_field(self, idx)

def _tab7_v3_refresh_display(self, preserve_zoom=True):
    stack = _tab7_v3_stack(self)
    if stack is None:
        try:
            self.pd_frame_slider.configure(from_=0, to=0, state='disabled')
            self.pd_frame_label.config(text='--')
            self.pd_status_var.set('Waiting for a confirmed Secondary ROI from Tab 5.')
        except Exception:
            pass
        return False
    n = int(stack.shape[0])
    idx = max(0, min(int(self.pd_frame_var.get()), n - 1))
    self.pd_frame_var.set(idx)
    try:
        self.pd_frame_slider.configure(from_=0, to=n - 1, state='normal')
        _set_frame_slider_midpoint_once(self,self.pd_frame_slider,n,self.pd_frame_var,zero_based=True,token='tab7v3')
    except Exception:
        pass
    img = np.asarray(stack[idx], dtype=float)
    finite = img[np.isfinite(img)]
    nz = finite[np.abs(finite) > 1e-12] if finite.size else finite
    vals = nz if nz.size >= max(20, int(0.01 * max(1, img.size))) else finite
    if vals.size:
        lo, hi = np.percentile(vals, [1, 99])
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = (float(np.min(vals)), float(np.max(vals)))
    else:
        lo, hi = (0.0, 1.0)
    if hi <= lo:
        hi = lo + 1.0
    self.pd_image_artist_gray.set_data(img)
    self.pd_image_artist_gray.set_clim(lo, hi)
    self.pd_image_artist_gray.set_extent((-0.5, img.shape[1] - 0.5, -0.5, img.shape[0] - 0.5))
    self.pd_image_artist_color.set_data(img)
    self.pd_image_artist_color.set_clim(lo, hi)
    self.pd_image_artist_color.set_extent((-0.5, img.shape[1] - 0.5, -0.5, img.shape[0] - 0.5))
    try:
        self.pd_image_artist_color.set_cmap(self.colormap_var.get() or 'gray')
    except Exception:
        pass
    try:
        self.pd_colorbar.update_normal(self.pd_image_artist_color)
    except Exception:
        pass
    h, w = img.shape[:2]
    zlim = getattr(self, '_pd_zoom_limits', None) if preserve_zoom else None
    if zlim is None:
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            ax.set_xlim(-0.5, w - 0.5)
            ax.set_ylim(-0.5, h - 0.5)
            ax.set_aspect('equal', adjustable='box')
        self._pd_zoom_limits = None
    else:
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            ax.set_xlim(*zlim[0])
            ax.set_ylim(*zlim[1])
            ax.set_aspect('equal', adjustable='box')
    field = _tab7_v3_field(self, idx)
    self.pd_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT')
    self.pd_ax_gray.set_title(f'Selected ROI — Grayscale | Frame {idx + 1}/{n}', fontsize=9)
    self.pd_ax_color.set_title(f'Selected ROI — Color | Frame {idx + 1}/{n} | {field:+.2f} mT', fontsize=9)
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass
    return True

def _tab7_v3_draw_overlay(self):
    for a in getattr(self, 'pd_overlay_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self.pd_overlay_artists = []
    if not getattr(self, 'pd_show_centers_var', None) or not self.pd_show_centers_var.get():
        try:
            self.pd_canvas.draw_idle()
        except Exception:
            pass
        return
    selected = set(getattr(self, 'pd_multi_selected', set()) or set())
    for i, d in enumerate(getattr(self, 'pd_current_detections', []) or []):
        x = float(d.get('outer_x', d.get('x', 0.0)))
        y = float(d.get('outer_y', d.get('y', 0.0)))
        r = max(2.0, float(d.get('diameter_px', 10.0)) / 2.0)
        t = np.linspace(0, 2 * np.pi, 160)
        for ax in (self.pd_ax_gray, self.pd_ax_color):
            outer_line, = ax.plot(x + r * np.cos(t), y + r * np.sin(t), color='yellow', lw=2.0, zorder=20)
            cx = float(d.get('x', x))
            cy = float(d.get('y', y))
            core_d = max(2.0, float(d.get('core_diameter_px', 6.0)))
            car = max(1.0, float(d.get('core_aspect_ratio', 1.0)))
            a = core_d * car / 2.0
            b = core_d / 2.0
            core_line, = ax.plot(cx + a * np.cos(t), cy + b * np.sin(t), color='lime', lw=1.8, zorder=21)
            mark, = ax.plot([cx], [cy], 'o', ms=5.0, mfc='none', mec='white' if i in selected else 'lime', mew=1.7, zorder=22)
            if not getattr(self, '_tab8_hide_roi_labels', False):
                label = ax.text(cx, cy, str(i + 1), color='black', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='circle,pad=0.12', fc='yellow', ec='none'), zorder=23)
                self.pd_overlay_artists.extend([outer_line, core_line, mark, label])
            else:
                self.pd_overlay_artists.extend([outer_line, core_line, mark])
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass

def _tab7_v3_update_current(self):
    stack = _tab7_v3_stack(self)
    if stack is None:
        return
    idx = max(0, min(int(self.pd_frame_var.get()), stack.shape[0] - 1))
    self.pd_frame_var.set(idx)
    _tab7_v3_refresh_display(self, preserve_zoom=True)
    self.pd_current_detections = [dict(d) for d in getattr(self, 'pd_frame_detections', {}).get(idx, []) or []]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    _tab7_v3_draw_overlay(self)

def _tab7_v3_arm_slider(self, event=None):
    try:
        self.pd_frame_slider.focus_set()
    except Exception:
        pass
    self.pd_frame_slider._tab7_v3_armed = True
    return None

def _tab7_v3_slider_wheel(self, event):
    slider = getattr(self, 'pd_frame_slider', None)
    stack = _tab7_v3_stack(self)
    if slider is None or stack is None or (not getattr(slider, '_tab7_v3_armed', False)):
        return 'break'
    try:
        root = slider.winfo_toplevel()
        px = root.winfo_pointerx()
        py = root.winfo_pointery()
        rx, ry = (slider.winfo_rootx(), slider.winfo_rooty())
        rw, rh = (slider.winfo_width(), slider.winfo_height())
        if not (rx <= px <= rx + rw and ry <= py <= ry + rh):
            return 'break'
    except Exception:
        return 'break'
    step = 1 if getattr(event, 'num', None) == 4 or getattr(event, 'delta', 0) > 0 else -1
    cur = int(slider.get())
    new = max(0, min(cur + step, stack.shape[0] - 1))
    slider.set(new)
    self.pd_frame_var.set(new)
    _tab7_v3_update_current(self)
    return 'break'

def _tab7_v3_select(self, event):
    if event is None or event.inaxes not in (getattr(self, 'pd_ax_gray', None), getattr(self, 'pd_ax_color', None)) or event.xdata is None or (event.ydata is None):
        return
    dets = getattr(self, 'pd_current_detections', []) or []
    if not dets:
        return
    x, y = (float(event.xdata), float(event.ydata))
    best = min(((np.hypot(float(d.get('x', 0)) - x, float(d.get('y', 0)) - y), i) for i, d in enumerate(dets)), default=(1000000000.0, -1))
    dist, i = best
    if i < 0 or dist > max(8.0, 0.65 * float(dets[i].get('diameter_px', 10.0))):
        return
    ctrl = str(getattr(event, 'key', None)).lower() in ('control', 'ctrl')
    if ctrl:
        if i in self.pd_multi_selected:
            self.pd_multi_selected.remove(i)
        else:
            self.pd_multi_selected.add(i)
    else:
        self.pd_multi_selected = {i}
    self.pd_selected_detection = i
    d = dets[i]
    cls = str(d.get('classification', 'particle')).upper()
    if hasattr(self, 'pd_particle_status'):
        self.pd_particle_status.config(text=f"Particle #{i + 1}\nOuter diameter = {float(d.get('diameter_px', np.nan)):.2f} px\nCore diameter = {float(d.get('core_diameter_px', np.nan)):.2f} px\nCore aspect ratio = {float(d.get('core_aspect_ratio', np.nan)):.3f}\nCore eccentricity = {float(d.get('core_eccentricity', np.nan)):.3f}\nCore solidity = {float(d.get('core_solidity', np.nan)):.3f}\nClassification = {cls}")
    _tab7_v3_draw_overlay(self)

def _tab7_v3_normalize_nnc(self, x, y, w, h):
    return (float(x) / max(1, w - 1), float(y) / max(1, h - 1))

def _tab7_v3_delete(self):
    dets = getattr(self, 'pd_current_detections', []) or []
    sel = sorted(set(getattr(self, 'pd_multi_selected', set()) or set()), reverse=True)
    if not sel and getattr(self, 'pd_selected_detection', None) is not None:
        sel = [int(self.pd_selected_detection)]
    if not sel:
        self.pd_status_var.set('Select one or more detected particles first. Ctrl + left-click selects multiple.')
        return
    stack = _tab7_v3_stack(self)
    if stack is None:
        return
    idx = int(self.pd_frame_var.get())
    h, w = np.asarray(stack[idx]).shape[:2]
    deleted = [dict(dets[j]) for j in sel if 0 <= j < len(dets)]
    if not deleted:
        return
    if bool(self.pd_nnc_var.get()):
        if not hasattr(self, 'pd_nnc_exclusions_global'):
            self.pd_nnc_exclusions_global = []
        for d in deleted:
            xn, yn = _tab7_v3_normalize_nnc(self, float(d.get('x', 0)), float(d.get('y', 0)), w, h)
            tol = max(8.0, 0.6 * float(d.get('diameter_px', 10.0)))
            self.pd_nnc_exclusions_global.append((xn, yn, tol))
        self.pd_last_deleted = None
        msg = f'NNC-Delete: {len(deleted)} particle(s) permanently excluded from subsequent frame/field analysis.'
    else:
        self.pd_last_deleted = (idx, deleted)
    for j in sel:
        if 0 <= j < len(dets):
            dets.pop(j)
    self.pd_current_detections = dets
    self.pd_frame_detections[idx] = [dict(d) for d in dets]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    _tab7_v3_draw_overlay(self)
    _tab7_v3_refresh_density_row(self, idx)
    if bool(self.pd_nnc_var.get()):
        det = [dict(d) for d in self._tab7_detect(np.asarray(stack[idx], dtype=float))]
        self.pd_current_detections = det
        self.pd_frame_detections[idx] = [dict(d) for d in det]
        _tab7_v3_refresh_density_row(self, idx)
        _tab7_v3_draw_overlay(self)
    self.pd_status_var.set(msg if bool(self.pd_nnc_var.get()) else f'Deleted {len(deleted)} particle(s) from frame {idx + 1}.')

def _tab7_v3_undo(self):
    rec = getattr(self, 'pd_last_deleted', None)
    if not rec:
        self.pd_status_var.set('Nothing to undo. NNC-Delete exclusions are permanent until CLEAR NNC EXCLUSIONS.')
        return
    idx, deleted = rec
    cur = [dict(d) for d in self.pd_frame_detections.get(idx, []) or []]
    cur.extend([dict(d) for d in deleted])
    cur.sort(key=lambda d: (float(d.get('y', 0)), float(d.get('x', 0))))
    self.pd_frame_detections[idx] = cur
    if int(self.pd_frame_var.get()) == idx:
        self.pd_current_detections = [dict(d) for d in cur]
    self.pd_last_deleted = None
    _tab7_v3_draw_overlay(self)
    _tab7_v3_refresh_density_row(self, idx)
    self.pd_status_var.set(f'Restored {len(deleted)} particle(s) in frame {idx + 1}.')

def _tab7_v3_clear_nnc(self):
    self.pd_nnc_exclusions_global = []
    self.pd_nnc_exclusions = {}
    self.pd_status_var.set('NNC exclusions cleared. Re-analyze frames to restore previously excluded candidates.')

def _tab7_v3_analyze_current(self):
    stack = _tab7_v3_stack(self)
    if stack is None:
        messagebox.showwarning('Skyrmion Analysis', 'Confirm the Secondary ROI in Tab 5 first.', parent=self.root)
        return
    idx = max(0, min(int(self.pd_frame_var.get()), stack.shape[0] - 1))
    self.pd_frame_var.set(idx)
    det = [dict(d) for d in self._tab7_detect(np.asarray(stack[idx], dtype=float))]
    self.pd_current_detections = det
    self.pd_frame_detections[idx] = [dict(d) for d in det]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    _tab7_v3_refresh_display(self, preserve_zoom=True)
    _tab7_v3_draw_overlay(self)
    _tab7_v3_refresh_density_row(self, idx)
    self.pd_status_var.set(f'Analyzed frame {idx + 1}: {len(det)} accepted particle(s).')

def _tab7_v3_analyze_selected(self):
    stack = _tab7_v3_stack(self)
    if stack is None:
        messagebox.showwarning('Skyrmion Analysis', 'Confirm the Secondary ROI in Tab 5 first.', parent=self.root)
        return
    n = stack.shape[0]
    try:
        a = max(1, min(int(self.pd_range_start_var.get()), n))
        b = max(1, min(int(self.pd_range_end_var.get()), n))
    except Exception:
        a, b = (1, n)
    if a > b:
        a, b = (b, a)
    self.pd_status_var.set(f'Analyzing frames {a}–{b} with current parameters…')
    self.root.update_idletasks()
    for idx in range(a - 1, b):
        det = [dict(d) for d in self._tab7_detect(np.asarray(stack[idx], dtype=float))]
        self.pd_frame_detections[idx] = det
        _tab7_v3_refresh_density_row(self, idx)
    keep = max(a - 1, min(int(self.pd_frame_var.get()), b - 1))
    self.pd_frame_var.set(keep)
    self.pd_current_detections = [dict(d) for d in self.pd_frame_detections.get(keep, [])]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    _tab7_v3_refresh_display(self, preserve_zoom=True)
    _tab7_v3_draw_overlay(self)
    _tab7_v3_refresh_table(self)
    _tab7_v3_plot_density(self)
    self.pd_status_var.set(f'Completed analysis for selected frames {a}–{b}. Each frame used the current detection parameters.')

def _tab7_v3_refresh_density_row(self, idx):
    stack = _tab7_v3_stack(self)
    if stack is None:
        return
    dets = [d for d in getattr(self, 'pd_frame_detections', {}).get(idx, []) if d.get('accepted', True)]
    h, w = np.asarray(stack[int(idx)]).shape[:2]
    sx, sy = _tab8_authoritative_nm_scales(self)
    if sx is None or sy is None:
        self.pd_status_var.set('Tab 6 Measurement Scale is unavailable. Confirm Tab 5/Tab 6 calibration first.')
        return
    area_um2 = (float(w) * float(sx)) * (float(h) * float(sy)) / 1.0e6
    count = len(dets)
    dens = count / area_um2 if area_um2 > 0 else np.nan
    md = float(np.mean([float(d.get('diameter_px', np.nan)) for d in dets])) if dets else np.nan
    mcd = float(np.mean([float(d.get('core_diameter_px', np.nan)) for d in dets])) if dets else np.nan
    mar = float(np.mean([float(d.get('core_aspect_ratio', np.nan)) for d in dets])) if dets else np.nan
    self.pd_results[int(idx)] = (int(idx) + 1, _tab7_v3_field(self, idx), count, area_um2, dens, md, mcd, mar)
    _tab7_v3_refresh_table(self)
    _tab7_v3_plot_density(self)

def _tab7_v3_refresh_table(self):
    if not hasattr(self, 'pd_table'):
        return
    for it in self.pd_table.get_children():
        self.pd_table.delete(it)
    for k in sorted(getattr(self, 'pd_results', {}) or {}):
        self.pd_table.insert('', 'end', values=self.pd_results[k])

def _tab7_v3_plot_density(self):
    """Plot only valid analyzed frames, preserving acquisition order and sweep branches."""
    if not hasattr(self, 'pd_plot_canvas'):
        return
    ax, ax2 = self.pd_plot_ax, self.pd_plot_ax2
    ax.clear(); ax2.clear()
    ax.set_zorder(2); ax2.set_zorder(1)
    ax.patch.set_alpha(0.0); ax2.patch.set_visible(False)
    results = getattr(self, 'pd_results', {}) or {}
    analyzed = getattr(self, 'pd_frame_detections', {}) or {}
    rows = []
    for k in sorted(results):
        if k not in analyzed:
            continue
        r = results.get(k)
        if not isinstance(r, (tuple, list)) or len(r) < 5:
            continue
        try:
            frame, field, count, density = float(r[0]), float(r[1]), float(r[2]), float(r[4])
        except Exception:
            continue
        if not all(np.isfinite(v) for v in (frame, field, count, density)):
            continue
        rows.append((int(k), frame, field, count, density))
    ax.set_xlabel('Field (mT)'); ax.set_ylabel('Particle count')
    ax2.set_ylabel('Density (skyrmions/µm²)')
    ax2.yaxis.tick_right(); ax2.yaxis.set_label_position('right')
    ax2.xaxis.tick_top(); ax2.xaxis.set_label_position('top')
    ax2.set_xlabel('Acquisition frame')
    if not rows:
        ax.text(0.5, 0.5, 'Analyze current or selected frames', ha='center', va='center', transform=ax.transAxes, fontsize=9)
        ax.set_title('Particle count / Density — valid analyzed frames only', fontsize=9, pad=18)
        self.pd_plot_canvas.draw_idle(); return
    # Split the actual acquisition sequence at field-sweep direction reversals.
    segments=[]; current=[rows[0]]; prev_dir=0
    for row in rows[1:]:
        df=row[2]-current[-1][2]
        direction=1 if df>0 else (-1 if df<0 else prev_dir)
        if prev_dir and direction and direction!=prev_dir:
            segments.append(current); current=[current[-1],row]
        else:
            current.append(row)
        if direction: prev_dir=direction
    segments.append(current)
    for j,seg in enumerate(segments):
        x=np.asarray([r[2] for r in seg],float)
        yc=np.asarray([r[3] for r in seg],float)
        yd=np.asarray([r[4] for r in seg],float)
        ax.plot(x,yc,'o-',lw=1.5,ms=4,label='Count' if j==0 else None)
        ax2.plot(x,yd,'s--',lw=1.4,ms=3,label='Density' if j==0 else None)
    fields=np.asarray([r[2] for r in rows],float)
    frames=np.asarray([r[1] for r in rows],float)
    tick_idx=np.arange(len(rows)) if len(rows)<=12 else np.unique(np.linspace(0,len(rows)-1,12).round().astype(int))
    ax2.set_xticks(fields[tick_idx]); ax2.set_xticklabels([str(int(frames[i])) for i in tick_idx])
    ax.grid(True,alpha=0.2)
    ax.set_title('Particle count / Density — acquisition-aware, valid analyzed frames',fontsize=9,pad=18)
    self.pd_plot_canvas.draw_idle()


def _tab7_v3_export_roi_png(self):
    """Export three current-ROI PNGs at 300 PPI:
    1) annotated grayscale ROI,
    2) annotated color ROI,
    3) raw color ROI with no particle/core overlays.
    One save dialog selects the filename stem; all three files are written together.
    """
    stack = _tab7_v3_stack(self)
    if stack is None or len(stack) == 0:
        messagebox.showwarning('Tab 7', 'No confirmed Secondary ROI is available. Confirm the Secondary ROI in Tab 5 first.', parent=self.root)
        return
    try:
        idx = int(max(0, min(int(self.pd_frame_var.get()), len(stack) - 1)))
        image = np.asarray(stack[idx], dtype=float)
        field = float(_tab7_v3_field(self, idx))
    except Exception as exc:
        messagebox.showerror('Tab 7', f'Unable to read current ROI: {exc}', parent=self.root)
        return
    outdir = self._export_initialdir() if hasattr(self, '_export_initialdir') else os.getcwd()
    base = filedialog.asksaveasfilename(parent=self.root, initialdir=outdir, title='Export Current ROI — Annotated Grayscale + Annotated Color + Raw Color (300 PPI PNG)', defaultextension='.png', filetypes=[('PNG image', '*.png')])
    if not base:
        return
    stem, _ = os.path.splitext(base)
    cmap_name = 'gray'
    try:
        cmap_name = self.colormap_var.get() or 'gray'
    except Exception:
        pass
    finite = image[np.isfinite(image)]
    if finite.size:
        lo, hi = np.percentile(finite, [0.5, 99.5])
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = (float(np.nanmin(finite)), float(np.nanmax(finite)))
    else:
        lo, hi = (0.0, 1.0)
    detections = [dict(d) for d in getattr(self, 'pd_current_detections', []) or []]

    def _apply_overlay(ax):
        for i, d in enumerate(detections):
            x = float(d.get('outer_x', d.get('x', 0.0)))
            y = float(d.get('outer_y', d.get('y', 0.0)))
            r = max(2.0, float(d.get('diameter_px', 10.0)) / 2.0)
            t = np.linspace(0, 2 * np.pi, 160)
            ax.plot(x + r * np.cos(t), y + r * np.sin(t), color='yellow', lw=2.0, zorder=20)
            cx = float(d.get('x', x))
            cy = float(d.get('y', y))
            core_d = max(2.0, float(d.get('core_diameter_px', 6.0)))
            car = max(1.0, float(d.get('core_aspect_ratio', 1.0)))
            a = core_d * car / 2.0
            b = core_d / 2.0
            ax.plot(cx + a * np.cos(t), cy + b * np.sin(t), color='lime', lw=1.8, zorder=21)
            ax.text(cx, cy, str(i + 1), color='black', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='circle,pad=0.12', fc='yellow', ec='none'), zorder=23)

    def _save_figure(kind, cmap, annotated):
        fig, ax = plt.subplots(figsize=(7.5, 6.0), dpi=100)
        artist = ax.imshow(image, origin='lower', cmap=cmap, vmin=lo, vmax=hi, extent=(-0.5, image.shape[1] - 0.5, -0.5, image.shape[0] - 0.5), interpolation='nearest', aspect='equal')
        ax.set_xlabel('X pixel')
        ax.set_ylabel('Y pixel')
        ax.set_title(f'Selected ROI — {kind} | Frame {idx + 1}/{len(stack)} | Field = {field:+.2f} mT')
        cb = fig.colorbar(artist, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label('Intensity')
        if annotated:
            _apply_overlay(ax)
        path = f"{stem}_{kind.lower().replace(' ', '_')}.png"
        fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return path
    try:
        p1 = _save_figure('Annotated Grayscale', 'gray', True)
        p2 = _save_figure('Annotated Color', cmap_name, True)
        p3 = _save_figure('Raw Color', cmap_name, False)
        self.pd_status_var.set(f'Exported 3 current-ROI PNGs at 300 PPI: {os.path.basename(p1)}, {os.path.basename(p2)}, {os.path.basename(p3)}')
    except Exception as exc:
        messagebox.showerror('Tab 7 export', f'ROI export failed: {exc}', parent=self.root)

def _tab7_v3_export_table(self):
    if not getattr(self, 'pd_results', None):
        self._tab7_analyze_current()
    if not getattr(self, 'pd_results', None):
        return
    fp = filedialog.asksaveasfilename(parent=self.root, initialdir=self._export_initialdir(), title='Export particle / skyrmion table', defaultextension='.csv', filetypes=[('CSV', '*.csv')])
    if not fp:
        return
    cols = ['Frame', 'Field (mT)', 'Count', 'ROI area (µm²)', 'Density (skyrmions/µm²)', 'Mean outer diameter (px)', 'Mean core diameter (px)', 'Mean core aspect ratio']
    df = pd.DataFrame([self.pd_results[k] for k in sorted(self.pd_results)], columns=cols)
    df.to_csv(fp, index=False)
    self.pd_status_var.set(f'Exported CSV table: {fp}')

def _tab7_v3_export_figure(self):
    if hasattr(self, 'pd_plot_canvas'):
        self._tab7_v3_plot_density()
    fp = filedialog.asksaveasfilename(parent=self.root, initialdir=self._export_initialdir(), title='Export Count / Density Figure — PNG 300 PPI', defaultextension='.png', filetypes=[('PNG image', '*.png')])
    if not fp:
        return
    self.pd_plot_fig.savefig(fp, dpi=300, bbox_inches='tight', facecolor='white')
    self.pd_status_var.set(f'Exported 300 PPI PNG figure: {fp}')

def _tab7_v3_tab_changed(self, event=None):
    try:
        if self.nb.select() == str(self.tab_particle_density):
            _tab7_v3_refresh_display(self, preserve_zoom=True)
            idx = int(self.pd_frame_var.get())
            self.pd_current_detections = [dict(d) for d in getattr(self, 'pd_frame_detections', {}).get(idx, []) or []]
            _tab7_v3_draw_overlay(self)
            n = len(_tab7_v3_stack(self) or [])
            if n and hasattr(self, 'pd_range_end_var'):
                self.pd_range_start_spin.configure(to=n)
                self.pd_range_end_spin.configure(to=n)
    except Exception:
        pass

def _tab7_v3_build(self):
    self.pd_frame_var = tk.IntVar(value=0)
    self.pd_threshold_var = tk.DoubleVar(value=0.1)
    self.pd_polarity_var = tk.StringVar(value='both')
    self.pd_min_diam_var = tk.DoubleVar(value=10.0)
    self.pd_max_diam_var = tk.DoubleVar(value=50.0)
    self.pd_min_sep_var = tk.DoubleVar(value=50.0)
    self.pd_max_aspect_var = tk.DoubleVar(value=1.8)
    self.pd_core_contrast_var = tk.DoubleVar(value=0.55)
    self.pd_core_min_diam_var = tk.DoubleVar(value=4.0)
    self.pd_core_max_diam_var = tk.DoubleVar(value=40.0)
    self.pd_include_edge_var = tk.BooleanVar(value=False)
    self.pd_show_centers_var = tk.BooleanVar(value=True)
    self.pd_nnc_var = tk.BooleanVar(value=False)
    self.pd_status_var = tk.StringVar(value='Confirm Secondary ROI in Tab 5.')
    self.pd_results = {}
    self.pd_frame_detections = {}
    self.pd_current_detections = []
    self.pd_overlay_artists = []
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self.pd_last_deleted = None
    self.pd_nnc_exclusions = {}
    self.pd_nnc_exclusions_global = []
    self._pd_zoom_limits = None
    self._tab8_hide_roi_labels = False
    for w in self.tab_particle_density.winfo_children():
        w.destroy()
    outer = ttk.Frame(self.tab_particle_density, padding=5)
    outer.pack(fill='both', expand=True)
    ttk.Label(outer, text='SKYRMION / PARTICLE — OUTER PARTICLE + INNER CORE + CORE SYMMETRY', font=('Arial', 11, 'bold')).pack(fill='x', pady=(0, 3))
    ttk.Label(outer, text='Source: fixed confirmed Secondary ROI from Tab 5', foreground='darkgreen').pack(anchor='w')
    self.pd_scale_status_var = tk.StringVar(value='Scale: Tab 6 Measurement Scale (X,Y)')
    ttk.Label(outer, textvariable=self.pd_scale_status_var, foreground='navy').pack(anchor='w')
    nav = ttk.Frame(outer)
    nav.pack(fill='x', pady=(3, 2))
    ttk.Label(nav, text='Frame:').pack(side='left')
    self.pd_frame_slider = tk.Scale(nav, from_=0, to=0, orient='horizontal', variable=self.pd_frame_var, resolution=1, length=240, showvalue=True, highlightthickness=0, command=lambda v: self._tab7_v3_update_current())
    self.pd_frame_slider.pack(side='left', padx=5)
    self.pd_frame_slider._tab7_v3_armed = False
    self.pd_frame_slider.bind('<Button-1>', self._tab7_v3_arm_slider, add='+')
    self.pd_frame_slider.bind('<MouseWheel>', self._tab7_v3_slider_wheel, add='+')
    self.pd_frame_slider.bind('<Button-4>', self._tab7_v3_slider_wheel, add='+')
    self.pd_frame_slider.bind('<Button-5>', self._tab7_v3_slider_wheel, add='+')
    self.pd_frame_label = ttk.Label(nav, text='--', width=24)
    self.pd_frame_label.pack(side='left', padx=5)
    actions = ttk.Frame(outer)
    actions.pack(fill='x', pady=(0, 4))
    ttk.Button(actions, text='RESET ZOOM', command=self._tab7_v3_reset_zoom).pack(side='left', padx=2)
    ttk.Button(actions, text='ANALYZE CURRENT FRAME', command=self._tab7_v3_analyze_current).pack(side='left', padx=2)
    ttk.Button(actions, text='ANALYZE SELECTED FRAMES', command=self._tab7_v3_analyze_selected).pack(side='left', padx=2)
    ttk.Button(actions, text='DELETE SELECTED', command=self._tab7_v3_delete).pack(side='left', padx=2)
    ttk.Button(actions, text='UNDO DELETE', command=self._tab7_v3_undo).pack(side='left', padx=2)
    ttk.Label(actions, text='|  Start').pack(side='left', padx=(8, 2))
    self.pd_range_start_var = tk.IntVar(value=1)
    self.pd_range_end_var = tk.IntVar(value=1)
    self.pd_range_start_spin = ttk.Spinbox(actions, from_=1, to=1, width=6, textvariable=self.pd_range_start_var, increment=1)
    self.pd_range_start_spin.pack(side='left')
    ttk.Label(actions, text='End').pack(side='left', padx=(6, 2))
    self.pd_range_end_spin = ttk.Spinbox(actions, from_=1, to=1, width=6, textvariable=self.pd_range_end_var, increment=1)
    self.pd_range_end_spin.pack(side='left')
    ttk.Checkbutton(actions, text='NNC-Delete — No Need to Consider', variable=self.pd_nnc_var).pack(side='left', padx=8)
    ttk.Button(actions, text='CLEAR NNC', command=self._tab7_v3_clear_nnc).pack(side='left', padx=2)
    ttk.Button(actions, text='CLEAR', command=self._tab8_clear_roi_labels).pack(side='left', padx=2)
    ttk.Button(actions, text='REFRESH', command=self._tab8_refresh_table_plot).pack(side='left', padx=2)
    panes = ttk.Panedwindow(outer, orient='horizontal')
    panes.pack(fill='both', expand=True)
    left = ttk.Frame(panes)
    right = ttk.Frame(panes)
    panes.add(left, weight=3)
    panes.add(right, weight=2)
    self.pd_main_panes = panes
    leftv = ttk.Panedwindow(left, orient='vertical')
    leftv.pack(fill='both', expand=True)
    panes_left_top = ttk.Frame(leftv)
    panes_left_bottom = ttk.Frame(leftv)
    leftv.add(panes_left_top, weight=3)
    leftv.add(panes_left_bottom, weight=1)
    self.pd_left_panes = leftv
    rightv = ttk.Panedwindow(right, orient='vertical')
    rightv.pack(fill='both', expand=True)
    right_top = ttk.Frame(rightv)
    right_bottom = ttk.Frame(rightv)
    rightv.add(right_top, weight=3)
    rightv.add(right_bottom, weight=1)
    self.pd_right_panes = rightv
    roi_box = ttk.LabelFrame(panes_left_top, text='SELECTED ROI — GRAYSCALE + COLOR', padding=2)
    roi_box.pack(fill='both', expand=True)
    self.pd_fig, (self.pd_ax_gray, self.pd_ax_color) = plt.subplots(1, 2, figsize=(8.8, 4.8), dpi=100, constrained_layout=True)
    self.pd_image_artist_gray = self.pd_ax_gray.imshow(np.zeros((2, 2)), cmap='gray', origin='lower', vmin=0, vmax=1, interpolation='nearest', extent=(-0.5, 1.5, -0.5, 1.5))
    try:
        cmap = self.colormap_var.get() or 'gray'
    except Exception:
        cmap = 'gray'
    self.pd_image_artist_color = self.pd_ax_color.imshow(np.zeros((2, 2)), cmap=cmap, origin='lower', vmin=0, vmax=1, interpolation='nearest', extent=(-0.5, 1.5, -0.5, 1.5))
    for ax in (self.pd_ax_gray, self.pd_ax_color):
        ax.set_xlabel('X pixel')
        ax.set_ylabel('Y pixel')
        ax.set_aspect('equal', adjustable='box')
    self.pd_colorbar = self.pd_fig.colorbar(self.pd_image_artist_color, ax=self.pd_ax_color, fraction=0.046, pad=0.04)
    self.pd_colorbar.set_label('Processed ROI intensity')
    self.pd_canvas = FigureCanvasTkAgg(self.pd_fig, master=roi_box)
    self.pd_canvas.draw()
    self.pd_canvas.get_tk_widget().pack(fill='both', expand=True)
    self.pd_canvas.mpl_connect('scroll_event', self._tab7_exact_zoom)
    self.pd_canvas.mpl_connect('button_press_event', self._tab7_v3_select)
    ctrl_box = ttk.LabelFrame(right_top, text='DETECTION / ANALYSIS CONTROLS', padding=2)
    ctrl_box.pack(fill='both', expand=True)
    cc = tk.Canvas(ctrl_box, highlightthickness=0, borderwidth=0)
    cs = ttk.Scrollbar(ctrl_box, orient='vertical', command=cc.yview)
    cf = ttk.Frame(cc, padding=7)
    win = cc.create_window((0, 0), window=cf, anchor='nw')
    cc.configure(yscrollcommand=cs.set)
    cc.pack(side='left', fill='both', expand=True)
    cs.pack(side='right', fill='y')
    cf.bind('<Configure>', lambda e: cc.configure(scrollregion=cc.bbox('all')))
    cc.bind('<Configure>', lambda e: cc.itemconfigure(win, width=e.width))

    def wheel_controls(e):
        step = -3 if getattr(e, 'num', None) == 4 else 3
        if getattr(e, 'delta', 0):
            step = int(-e.delta / 120)
        cc.yview_scroll(step, 'units')
        return 'break'
    for widget in (cc, cf):
        widget.bind('<MouseWheel>', wheel_controls)
        widget.bind('<Button-4>', wheel_controls)
        widget.bind('<Button-5>', wheel_controls)
    ttk.Label(cf, text='OUTER PARTICLE SEGMENTATION', font=('Arial', 10, 'bold')).pack(anchor='w')
    ttk.Label(cf, text='Detection threshold').pack(anchor='w')
    tk.Scale(cf, from_=0.01, to=0.5, resolution=0.01, orient='horizontal', variable=self.pd_threshold_var, length=250, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Polarity').pack(anchor='w')
    ttk.Combobox(cf, textvariable=self.pd_polarity_var, values=['bright', 'dark', 'both'], state='readonly', width=10).pack(anchor='w')
    ttk.Label(cf, text='Outer diameter: 10–50 px  |  Separation: 50 px').pack(anchor='w', pady=(3, 5))
    ttk.Separator(cf).pack(fill='x', pady=5)
    ttk.Label(cf, text='INNER CORE SEGMENTATION', font=('Arial', 10, 'bold')).pack(anchor='w')
    ttk.Label(cf, text='Core contrast level (relative)').pack(anchor='w')
    tk.Scale(cf, from_=0.1, to=0.95, resolution=0.01, orient='horizontal', variable=self.pd_core_contrast_var, length=250, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Core minimum diameter (px)').pack(anchor='w')
    tk.Scale(cf, from_=1, to=30, resolution=1, orient='horizontal', variable=self.pd_core_min_diam_var, length=250, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Core maximum diameter (px)').pack(anchor='w')
    tk.Scale(cf, from_=5, to=60, resolution=1, orient='horizontal', variable=self.pd_core_max_diam_var, length=250, showvalue=True).pack(fill='x')
    ttk.Separator(cf).pack(fill='x', pady=5)
    ttk.Label(cf, text='CORE SYMMETRY CLASSIFICATION', font=('Arial', 10, 'bold')).pack(anchor='w')
    ttk.Label(cf, text='Maximum core aspect ratio').pack(anchor='w')
    tk.Scale(cf, from_=1.0, to=3.0, resolution=0.01, orient='horizontal', variable=self.pd_max_aspect_var, length=250, showvalue=True).pack(fill='x')
    ttk.Label(cf, text='Core AR ≤ limit → PARTICLE | Core AR > limit → WORM', wraplength=280).pack(anchor='w', pady=3)
    ttk.Checkbutton(cf, text='Include edge detections', variable=self.pd_include_edge_var).pack(anchor='w')
    ttk.Checkbutton(cf, text='Show outer/core overlays', variable=self.pd_show_centers_var, command=self._tab7_v3_draw_overlay).pack(anchor='w')
    ttk.Label(cf, text='SELECTED PARTICLE', font=('Arial', 10, 'bold')).pack(anchor='w', pady=(6, 2))
    self.pd_particle_status = ttk.Label(cf, text='Click an accepted particle/centre in the ROI.', wraplength=290, justify='left')
    self.pd_particle_status.pack(anchor='w')
    self.pd_status_label = ttk.Label(cf, textvariable=self.pd_status_var, wraplength=290, justify='left')
    self.pd_status_label.pack(anchor='w', pady=5)
    ttk.Separator(cf).pack(fill='x', pady=5)
    ttk.Button(cf, text='EXPORT TABLE', command=self._tab7_v3_export_table).pack(fill='x', pady=2)
    ttk.Button(cf, text='EXPORT COUNT/DENSITY FIGURE — PNG', command=self._tab7_v3_export_figure).pack(fill='x', pady=2)
    ttk.Button(cf, text='EXPORT GRAY+COLOR ROI — PNG', command=self._tab7_v3_export_roi_png).pack(fill='x', pady=2)
    graph_box = ttk.LabelFrame(panes_left_bottom, text='COUNT / DENSITY GRAPH', padding=2)
    graph_box.pack(fill='both', expand=True)
    self.pd_plot_fig = plt.Figure(figsize=(7, 2.5), dpi=100)
    self.pd_plot_ax = self.pd_plot_fig.add_axes([0.1, 0.2, 0.72, 0.62])
    self.pd_plot_ax2 = self.pd_plot_fig.add_axes(self.pd_plot_ax.get_position(), frameon=False)
    self.pd_plot_ax2.patch.set_visible(False)
    self.pd_plot_canvas = FigureCanvasTkAgg(self.pd_plot_fig, master=graph_box)
    self.pd_plot_canvas.draw()
    self.pd_plot_canvas.get_tk_widget().pack(fill='both', expand=True)
    table_box = ttk.LabelFrame(right_bottom, text='DENSITY TABLE', padding=2)
    table_box.pack(fill='both', expand=True)
    tf = ttk.Frame(table_box)
    tf.pack(fill='both', expand=True)
    cols = ('Frame', 'Field (mT)', 'Count', 'ROI area (µm²)', 'Density (skyrmions/µm²)', 'Mean outer d (px)', 'Mean core d (px)', 'Mean core AR')
    self.pd_table = ttk.Treeview(tf, columns=cols, show='headings')
    for c in cols:
        self.pd_table.heading(c, text=c)
        self.pd_table.column(c, width=105, anchor='center')
    tv = ttk.Scrollbar(tf, orient='vertical', command=self.pd_table.yview)
    self.pd_table.configure(yscrollcommand=tv.set)
    self.pd_table.pack(side='left', fill='both', expand=True)
    tv.pack(side='right', fill='y')
    stack = _tab7_v3_stack(self)
    if stack is not None:
        n = int(stack.shape[0])
        self.pd_frame_slider.configure(from_=0, to=n - 1, state='normal')
        self.pd_range_start_spin.configure(to=n)
        self.pd_range_end_spin.configure(to=n)
        self.pd_range_start_var.set(1)
        self.pd_range_end_var.set(n)
    else:
        self.pd_frame_slider.configure(state='disabled')
    _tab7_v3_refresh_display(self, preserve_zoom=False)
    _tab7_v3_plot_density(self)
CDIWorkflowApp._tab7_v3_update_current = _tab7_v3_update_current
CDIWorkflowApp._tab7_v3_arm_slider = _tab7_v3_arm_slider
CDIWorkflowApp._tab7_v3_slider_wheel = _tab7_v3_slider_wheel
CDIWorkflowApp._tab7_v3_select = _tab7_v3_select
CDIWorkflowApp._tab7_v3_draw_overlay = _tab7_v3_draw_overlay
CDIWorkflowApp._tab7_v3_delete = _tab7_v3_delete
CDIWorkflowApp._tab7_v3_undo = _tab7_v3_undo
CDIWorkflowApp._tab7_v3_clear_nnc = _tab7_v3_clear_nnc
CDIWorkflowApp._tab7_v3_analyze_current = _tab7_v3_analyze_current
CDIWorkflowApp._tab7_v3_analyze_selected = _tab7_v3_analyze_selected
CDIWorkflowApp._tab7_v3_refresh_density_row = _tab7_v3_refresh_density_row
CDIWorkflowApp._tab7_v3_refresh_table = _tab7_v3_refresh_table
CDIWorkflowApp._tab7_v3_plot_density = _tab7_v3_plot_density
CDIWorkflowApp._tab7_v3_export_table = _tab7_v3_export_table
CDIWorkflowApp._tab7_v3_export_figure = _tab7_v3_export_figure
CDIWorkflowApp._tab7_v3_export_roi_png = _tab7_v3_export_roi_png
CDIWorkflowApp._tab7_v3_build = _tab7_v3_build
CDIWorkflowApp._tab7_delete = _tab7_v3_delete
CDIWorkflowApp._tab7_undo = _tab7_v3_undo
CDIWorkflowApp._tab7_clear_nnc = _tab7_v3_clear_nnc
CDIWorkflowApp._tab7_select = _tab7_v3_select
CDIWorkflowApp._tab7_draw_overlay = _tab7_v3_draw_overlay
CDIWorkflowApp._tab7_analyze_current = _tab7_v3_analyze_current
CDIWorkflowApp._tab7_analyze_selected = _tab7_v3_analyze_selected
CDIWorkflowApp._tab7_export_table = _tab7_v3_export_table
CDIWorkflowApp._tab7_export_figure = _tab7_v3_export_figure
_old_tab7_detect_v3 = _tab7_detect

def _tab7_detect_with_global_nnc(self, image):
    dets = _old_tab7_detect_v3(self, image)
    if not getattr(self, 'pd_nnc_exclusions_global', None):
        return dets
    h, w = np.asarray(image).shape[:2]
    out = []
    for d in dets:
        x = float(d.get('x', 0.0))
        y = float(d.get('y', 0.0))
        diam = float(d.get('diameter_px', 10.0))
        blocked = False
        for xn, yn, tol in self.pd_nnc_exclusions_global:
            px = xn * (w - 1)
            py = yn * (h - 1)
            if np.hypot(x - px, y - py) <= max(tol, 0.6 * diam):
                blocked = True
                break
        if not blocked:
            out.append(d)
    return out
CDIWorkflowApp._tab7_detect = _tab7_detect_with_global_nnc
CDIWorkflowApp._proc_update_physical_axes_and_colorbar = _proc_update_physical_axes_and_colorbar
CDIWorkflowApp._proc_reset_zoom = _proc_reset_zoom
CDIWorkflowApp._proc_zoom_button = _proc_zoom_button
CDIWorkflowApp._proc_restore_zoom_limits = _proc_restore_zoom_limits
CDIWorkflowApp._proc_zoom_scroll = _proc_zoom_scroll
CDIWorkflowApp._proc_update = _proc_update
CDIWorkflowApp.update_processing_view = _proc_update
CDIWorkflowApp.save_single = _proc_save_single
CDIWorkflowApp.batch_export = _proc_batch

def _tab6_physical_geometry(self):
    """Return (physical_width_nm, physical_height_nm, output_width, output_height)."""
    arr = self._proc_image_display.get_array()
    if arr is None:
        return None
    h, w = np.asarray(arr).shape[:2]
    base = self._proc_get_tab5_base_pixel_size()
    if base is None or not np.isfinite(base) or base <= 0:
        return None
    ow = int(getattr(self, 'original_roi_w', 0) or w)
    oh = int(getattr(self, 'original_roi_h', 0) or h)
    if ow <= 0 or oh <= 0 or w <= 0 or (h <= 0):
        return None
    return (float(ow) * float(base), float(oh) * float(base), int(w), int(h))

def _tab6_nm_to_output_px(self, x_nm, y_nm):
    """Convert physical nm coordinates from the displayed axes to output pixels."""
    geom = self._tab6_physical_geometry()
    if geom is None:
        return (float(x_nm), float(y_nm))
    pw, ph, w, h = geom
    px = float(x_nm) * w / pw
    py = float(y_nm) * h / ph
    return (px, py)

def _tab6_output_px_to_nm_rect(self, xmin_px, xmax_px, ymin_px, ymax_px):
    """Convert an internal pixel ROI rectangle to displayed physical nm bounds."""
    geom = self._tab6_physical_geometry()
    if geom is None:
        return (float(xmin_px), float(xmax_px), float(ymin_px), float(ymax_px))
    pw, ph, w, h = geom
    return (float(xmin_px) * pw / w, float(xmax_px + 1) * pw / w, float(ymin_px) * ph / h, float(ymax_px + 1) * ph / h)

def _tab6_secondary_status_nm(self):
    """Return a human-readable Secondary ROI status in nm."""
    c = getattr(self, 'roi2_coords', None)
    if c is None:
        return 'Secondary ROI: not selected'
    x0, x1, y0, y1 = self._tab6_output_px_to_nm_rect(int(c['xmin']), int(c['xmax']), int(c['ymin']), int(c['ymax']))
    return f'Secondary ROI: X={x0:.3g}–{x1:.3g} nm, Y={y0:.3g}–{y1:.3g} nm | Width={x1 - x0:.3g} nm, Height={y1 - y0:.3g} nm'

def _proc_secondary_on_release_nm(self, event):
    """Select Secondary ROI on the physical-nm display while storing output pixels internally."""
    if not getattr(self, '_proc_secondary_roi_selecting', False) or self._proc_secondary_press_xy is None:
        return
    if event.inaxes is not self._proc_ax or event.xdata is None or event.ydata is None:
        self._proc_secondary_press_xy = None
        return
    x0_nm, y0_nm = self._proc_secondary_press_xy
    x1_nm, y1_nm = (float(event.xdata), float(event.ydata))
    geom = self._tab6_physical_geometry()
    if geom is None:
        self.proc_secondary_status_var.set('Primary ROI has no valid Tab-5 final nm/pixel calibration.')
        self._proc_secondary_press_xy = None
        return
    physical_w_nm = float(geom[0])
    physical_h_nm = float(geom[1])
    w = int(geom[2])
    h = int(geom[3])
    x0_nm = max(0.0, min(x0_nm, physical_w_nm))
    x1_nm = max(0.0, min(x1_nm, physical_w_nm))
    y0_nm = max(0.0, min(y0_nm, physical_h_nm))
    y1_nm = max(0.0, min(y1_nm, physical_h_nm))
    xmin_nm, xmax_nm = sorted((x0_nm, x1_nm))
    ymin_nm, ymax_nm = sorted((y0_nm, y1_nm))
    xmin = int(np.floor(xmin_nm * w / physical_w_nm))
    xmax = int(np.ceil(xmax_nm * w / physical_w_nm) - 1)
    ymin = int(np.floor(ymin_nm * h / physical_h_nm))
    ymax = int(np.ceil(ymax_nm * h / physical_h_nm) - 1)
    xmin = max(0, min(xmin, w - 1))
    xmax = max(0, min(xmax, w - 1))
    ymin = max(0, min(ymin, h - 1))
    ymax = max(0, min(ymax, h - 1))
    self._proc_secondary_press_xy = None
    if xmax <= xmin or ymax <= ymin:
        self.proc_secondary_status_var.set('Secondary ROI too small. Drag a larger rectangle in nm coordinates.')
        return
    self.roi2_coords = {'xmin': xmin, 'xmax': xmax, 'ymin': ymin, 'ymax': ymax}
    if hasattr(self, 'roi2_idx_var'):
        self.roi2_idx_var.set(max(0, min(int(self.proc_idx_var.get()), len(self.recons) - 1)))
    self.proc_secondary_status_var.set(self._tab6_secondary_status_nm() + ' | Click CONFIRM SECONDARY ROI.')
    self._proc_secondary_roi_selecting = False
    try:
        self.proc_secondary_select_button.configure(text='SELECT SECONDARY ROI')
    except Exception:
        pass
    self._proc_draw_secondary_roi_patch(confirmed=False)

def _proc_secondary_on_move_nm(self, event):
    """Draw the Secondary ROI preview in physical nm coordinates."""
    if not getattr(self, '_proc_secondary_roi_selecting', False) or self._proc_secondary_press_xy is None:
        return
    if event.inaxes is not self._proc_ax or event.xdata is None or event.ydata is None:
        return
    x0, y0 = self._proc_secondary_press_xy
    x1, y1 = (float(event.xdata), float(event.ydata))
    xmin, xmax = sorted((x0, x1))
    ymin, ymax = sorted((y0, y1))
    geom = self._tab6_physical_geometry()
    if geom is not None:
        pw = float(geom[0])
        ph = float(geom[1])
        xmin = max(0.0, min(xmin, pw))
        xmax = max(0.0, min(xmax, pw))
        ymin = max(0.0, min(ymin, ph))
        ymax = max(0.0, min(ymax, ph))
    if self._proc_secondary_patch is None:
        self._proc_secondary_patch = plt.Rectangle((xmin, ymin), max(1e-12, xmax - xmin), max(1e-12, ymax - ymin), fill=False, edgecolor='red', linewidth=2.5, zorder=20)
        self._proc_ax.add_patch(self._proc_secondary_patch)
    else:
        self._proc_secondary_patch.set_xy((xmin, ymin))
        self._proc_secondary_patch.set_width(max(1e-12, xmax - xmin))
        self._proc_secondary_patch.set_height(max(1e-12, ymax - ymin))
    self._proc_canvas.draw_idle()

def _proc_draw_secondary_roi_patch_nm(self, confirmed=False, redraw=True):
    """Draw stored Secondary ROI as a physical-nm rectangle."""
    if self._proc_secondary_patch is not None:
        try:
            self._proc_secondary_patch.remove()
        except Exception:
            pass
        self._proc_secondary_patch = None
    c = getattr(self, 'roi2_coords', None)
    if c is None:
        if redraw:
            self._proc_canvas.draw_idle()
        return
    x0, x1, y0, y1 = self._tab6_output_px_to_nm_rect(int(c['xmin']), int(c['xmax']), int(c['ymin']), int(c['ymax']))
    edge = 'lime' if confirmed else 'red'
    self._proc_secondary_patch = plt.Rectangle((x0, y0), max(1e-12, x1 - x0), max(1e-12, y1 - y0), fill=False, edgecolor=edge, linewidth=2.5, zorder=20)
    self._proc_ax.add_patch(self._proc_secondary_patch)
    if redraw:
        self._proc_canvas.draw_idle()
CDIWorkflowApp._tab6_physical_geometry = _tab6_physical_geometry
CDIWorkflowApp._tab6_nm_to_output_px = _tab6_nm_to_output_px
CDIWorkflowApp._tab6_output_px_to_nm_rect = _tab6_output_px_to_nm_rect
CDIWorkflowApp._tab6_secondary_status_nm = _tab6_secondary_status_nm
CDIWorkflowApp._proc_secondary_on_release = _proc_secondary_on_release_nm
CDIWorkflowApp._proc_secondary_on_move = _proc_secondary_on_move_nm
CDIWorkflowApp._proc_draw_secondary_roi_patch = _proc_draw_secondary_roi_patch_nm
_old_proc_update_physical_axes = CDIWorkflowApp._proc_update_physical_axes_and_colorbar

def _proc_update_physical_axes_and_colorbar_nm(self, display):
    _old_proc_update_physical_axes(self, display)
    base = self._proc_get_tab5_base_pixel_size()
    if base is not None and np.isfinite(base) and (base > 0):
        self._proc_ax.set_xlabel('X (nm)')
        self._proc_ax.set_ylabel('Y (nm)')
        self._proc_ax.set_aspect('equal', adjustable='box')
        self._proc_ax.set_anchor('C')
CDIWorkflowApp._proc_update_physical_axes_and_colorbar = _proc_update_physical_axes_and_colorbar_nm

def _pixel_cal_zoom_scroll_safe(self, which, event):
    """Stable cursor-centred mouse-wheel zoom for Tab 5 SEM/Primary ROI.

    Only the mouse wheel controls zoom. The completed calibration line/ROI line
    remains untouched. Zoom limits are stored in the same coordinates used by
    the corresponding Matplotlib image, with the native SEM/ROI Y orientation
    preserved.
    """
    if which not in ('sem', 'roi'):
        return
    ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
    canvas = self.pixel_cal_sem_canvas if which == 'sem' else self.pixel_cal_roi_canvas
    if getattr(event, 'inaxes', None) is not ax or event.xdata is None or event.ydata is None:
        return
    step = getattr(event, 'step', 0)
    if step == 0:
        return
    image = self._pixel_cal_get_display_image(which)
    if image is None:
        return
    h, w = np.asarray(image).shape[:2]
    if h <= 0 or w <= 0:
        return
    x0, x1 = [float(v) for v in ax.get_xlim()]
    y0, y1 = [float(v) for v in ax.get_ylim()]
    sx = x1 - x0
    sy = y1 - y0
    if abs(sx) < 1e-12 or abs(sy) < 1e-12:
        return
    factor = 1.2 ** (-float(step))
    cx, cy = (float(event.xdata), float(event.ydata))
    rx = (cx - x0) / sx
    ry = (cy - y0) / sy
    nsx = sx * factor
    nsy = sy * factor
    nx0 = cx - rx * nsx
    nx1 = nx0 + nsx
    ny0 = cy - ry * nsy
    ny1 = ny0 + nsy
    fx0, fx1 = (-0.5, w - 0.5)
    fy_top, fy_bottom = (-0.5, h - 0.5)
    xdesc = x0 > x1
    ydesc = y0 > y1
    xlo, xhi = sorted((nx0, nx1))
    ylo, yhi = sorted((ny0, ny1))
    full_w = fx1 - fx0
    full_h = fy_bottom - fy_top
    if xhi - xlo >= 0.999999 * full_w and yhi - ylo >= 0.999999 * full_h:
        self._pixel_cal_reset_zoom(which)
        return
    if xlo < fx0:
        d = fx0 - xlo
        xlo += d
        xhi += d
    if xhi > fx1:
        d = xhi - fx1
        xlo -= d
        xhi -= d
    if ylo < fy_top:
        d = fy_top - ylo
        ylo += d
        yhi += d
    if yhi > fy_bottom:
        d = yhi - fy_bottom
        ylo -= d
        yhi -= d
    xlo, xhi = (max(fx0, xlo), min(fx1, xhi))
    ylo, yhi = (max(fy_top, ylo), min(fy_bottom, yhi))
    xlims = (xhi, xlo) if xdesc else (xlo, xhi)
    ylims = (yhi, ylo) if ydesc else (ylo, yhi)
    self._pixel_cal_zoom_limits[which] = (xlims, ylims)
    ax.set_autoscale_on(False)
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.set_aspect('equal', adjustable='box')
    canvas.draw_idle()
CDIWorkflowApp._pixel_cal_zoom_scroll = _pixel_cal_zoom_scroll_safe

def _proc_tab5_base_nm_per_px_safe(self):
    """Read only Tab 5's final Primary ROI calibration."""
    try:
        v = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
        if np.isfinite(v) and v > 0:
            return v
    except Exception:
        pass
    return None

def _proc_update_physical_axes_safe(self, display):
    """Force Tab 6 Primary ROI axes to real physical nm coordinates."""
    arr = np.asarray(display)
    if arr.ndim < 2:
        return
    h, w = arr.shape[:2]
    if h <= 0 or w <= 0:
        return
    base = _proc_tab5_base_nm_per_px_safe(self)
    ax = self._proc_ax
    self._proc_zoom_limits = getattr(self, '_proc_zoom_limits', None)
    if base is not None:
        ow = int(getattr(self, 'original_roi_w', 0) or w)
        oh = int(getattr(self, 'original_roi_h', 0) or h)
        physical_w_nm = float(ow) * base
        physical_h_nm = float(oh) * base
        if physical_w_nm <= 0 or physical_h_nm <= 0:
            base = None
        else:
            ax.set_axis_on()
            self._proc_image_display.set_extent((0.0, physical_w_nm, 0.0, physical_h_nm))
            ax.set_xlim(0.0, physical_w_nm)
            ax.set_ylim(0.0, physical_h_nm)
            ax.set_xlabel('X (nm)')
            ax.set_ylabel('Y (nm)')
            ax.tick_params(axis='both', which='both', labelsize=9)
            ax.set_aspect('equal', adjustable='box')
            ax.set_anchor('C')
            ax.set_autoscale_on(False)
            self._proc_full_physical_extent = (0.0, physical_w_nm, 0.0, physical_h_nm)
            self.proc_pixel_size_var.set(f'Pixel Size = {base:.9g} nm/px')
            limits = getattr(self, '_proc_zoom_limits', None)
            if limits is not None and getattr(self, '_proc_zoomed', False):
                x0, x1, y0, y1 = [float(v) for v in limits]
                x0 = max(0.0, min(x0, physical_w_nm))
                x1 = max(0.0, min(x1, physical_w_nm))
                y0 = max(0.0, min(y0, physical_h_nm))
                y1 = max(0.0, min(y1, physical_h_nm))
                if x1 > x0 and y1 > y0:
                    ax.set_xlim(x0, x1)
                    ax.set_ylim(y0, y1)
                else:
                    self._proc_zoom_limits = None
                    self._proc_zoomed = False
            return
    ax.set_axis_on()
    self._proc_image_display.set_extent((-0.5, w - 0.5, -0.5, h - 0.5))
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(-0.5, h - 0.5)
    ax.set_xlabel('X (pixel)')
    ax.set_ylabel('Y (pixel)')
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    self._proc_full_physical_extent = (-0.5, w - 0.5, -0.5, h - 0.5)

def _proc_update_physical_axes_and_colorbar_safe(self, display):
    _proc_update_physical_axes_and_colorbar(self, display)
    _proc_update_physical_axes_safe(self, display)
    try:
        if getattr(self, '_proc_colorbar', None) is not None:
            self._proc_colorbar.update_normal(self._proc_image_display)
            self._proc_image_display.set_clim(0.0, 1.0)
            self._proc_colorbar.set_label('Intensity')
    except Exception:
        pass
CDIWorkflowApp._proc_update_physical_axes_and_colorbar = _proc_update_physical_axes_and_colorbar_safe

def _proc_restore_zoom_limits_safe(self, width=None, height=None):
    base = _proc_tab5_base_nm_per_px_safe(self)
    arr = self._proc_image_display.get_array()
    if arr is None:
        return
    hh, ww = np.asarray(arr).shape[:2]
    if base is not None:
        ow = int(getattr(self, 'original_roi_w', 0) or ww)
        oh = int(getattr(self, 'original_roi_h', 0) or hh)
        full = (0.0, float(ow) * base, 0.0, float(oh) * base)
    else:
        full = (-0.5, float(ww) - 0.5, -0.5, float(hh) - 0.5)
    limits = getattr(self, '_proc_zoom_limits', None)
    if not limits:
        return
    x0, x1, y0, y1 = [float(v) for v in limits]
    x0 = max(full[0], min(x0, full[1]))
    x1 = max(full[0], min(x1, full[1]))
    y0 = max(full[2], min(y0, full[3]))
    y1 = max(full[2], min(y1, full[3]))
    if x1 <= x0 or y1 <= y0:
        self._proc_zoom_limits = None
        self._proc_zoomed = False
        self._proc_ax.set_xlim(full[0], full[1])
        self._proc_ax.set_ylim(full[2], full[3])
        return
    self._proc_zoom_limits = (x0, x1, y0, y1)
    self._proc_zoomed = True
    self._proc_ax.set_autoscale_on(False)
    self._proc_ax.set_xlim(x0, x1)
    self._proc_ax.set_ylim(y0, y1)
    self._proc_ax.set_aspect('equal', adjustable='box')
    self._proc_ax.set_anchor('C')

def _proc_reset_zoom_safe(self):
    arr = self._proc_image_display.get_array()
    if arr is None:
        return
    hh, ww = np.asarray(arr).shape[:2]
    base = _proc_tab5_base_nm_per_px_safe(self)
    if base is not None:
        ow = int(getattr(self, 'original_roi_w', 0) or ww)
        oh = int(getattr(self, 'original_roi_h', 0) or hh)
        full = (0.0, float(ow) * base, 0.0, float(oh) * base)
    else:
        full = (-0.5, float(ww) - 0.5, -0.5, float(hh) - 0.5)
    self._proc_full_physical_extent = full
    self._proc_zoom_limits = None
    self._proc_zoomed = False
    self._proc_ax.set_autoscale_on(False)
    self._proc_ax.set_xlim(full[0], full[1])
    self._proc_ax.set_ylim(full[2], full[3])
    self._proc_ax.set_aspect('equal', adjustable='box')
    self._proc_ax.set_anchor('C')
    self._proc_ax.set_axis_on()
    self._proc_canvas.draw_idle()

def _proc_zoom_scroll_safe(self, event):
    """Cursor-centred mouse-wheel zoom in the physical nm coordinate system."""
    if getattr(event, 'inaxes', None) is not self._proc_ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    step = getattr(event, 'step', 0)
    if not step:
        return
    arr = self._proc_image_display.get_array()
    if arr is None:
        return
    hh, ww = np.asarray(arr).shape[:2]
    base = _proc_tab5_base_nm_per_px_safe(self)
    if base is not None:
        ow = int(getattr(self, 'original_roi_w', 0) or ww)
        oh = int(getattr(self, 'original_roi_h', 0) or hh)
        full = (0.0, float(ow) * base, 0.0, float(oh) * base)
    else:
        full = (-0.5, float(ww) - 0.5, -0.5, float(hh) - 0.5)
    ax = self._proc_ax
    x0, x1 = [float(v) for v in ax.get_xlim()]
    y0, y1 = [float(v) for v in ax.get_ylim()]
    sx, sy = (x1 - x0, y1 - y0)
    if abs(sx) < 1e-12 or abs(sy) < 1e-12:
        return
    factor = 1.2 ** (-float(step))
    cx, cy = (float(event.xdata), float(event.ydata))
    rx = (cx - x0) / sx
    ry = (cy - y0) / sy
    nsx, nsy = (sx * factor, sy * factor)
    nx0, nx1 = (cx - rx * nsx, cx + (1.0 - rx) * nsx)
    ny0, ny1 = (cy - ry * nsy, cy + (1.0 - ry) * nsy)
    xlo, xhi = sorted((nx0, nx1))
    ylo, yhi = sorted((ny0, ny1))
    if xhi - xlo >= 0.999999 * (full[1] - full[0]) and yhi - ylo >= 0.999999 * (full[3] - full[2]):
        return _proc_reset_zoom_safe(self)
    if xlo < full[0]:
        d = full[0] - xlo
        xlo += d
        xhi += d
    if xhi > full[1]:
        d = xhi - full[1]
        xlo -= d
        xhi -= d
    if ylo < full[2]:
        d = full[2] - ylo
        ylo += d
        yhi += d
    if yhi > full[3]:
        d = yhi - full[3]
        ylo -= d
        yhi -= d
    xlo, xhi = (max(full[0], xlo), min(full[1], xhi))
    ylo, yhi = (max(full[2], ylo), min(full[3], yhi))
    self._proc_zoom_limits = (xlo, xhi, ylo, yhi)
    self._proc_zoomed = True
    ax.set_autoscale_on(False)
    ax.set_xlim(xlo, xhi)
    ax.set_ylim(ylo, yhi)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    self._proc_canvas.draw_idle()
CDIWorkflowApp._proc_restore_zoom_limits = _proc_restore_zoom_limits_safe
CDIWorkflowApp._proc_reset_zoom = _proc_reset_zoom_safe
CDIWorkflowApp._proc_zoom_scroll = _proc_zoom_scroll_safe
_old_build_proc_tab_final = CDIWorkflowApp._build_proc_tab

def _build_proc_tab_with_safe_mouse_handlers(self):
    _old_build_proc_tab_final(self)
    canvas = self._proc_canvas
    try:
        for signal in ('scroll_event', 'button_press_event', 'motion_notify_event', 'button_release_event'):
            registry = getattr(canvas.callbacks, 'callbacks', {}).get(signal, {})
            for cid in list(registry.keys()):
                try:
                    canvas.mpl_disconnect(cid)
                except Exception:
                    pass
    except Exception:
        pass
    canvas.mpl_connect('scroll_event', self._proc_zoom_scroll)
    canvas.mpl_connect('button_press_event', self._proc_secondary_on_press)
    canvas.mpl_connect('motion_notify_event', self._proc_secondary_on_move)
    canvas.mpl_connect('button_release_event', self._proc_secondary_on_release)
    canvas.mpl_connect('button_press_event', self._proc_profile_on_press)
    canvas.mpl_connect('motion_notify_event', self._proc_profile_on_move)
    canvas.mpl_connect('button_release_event', self._proc_profile_on_release)
    try:
        self._proc_update()
    except Exception:
        pass
CDIWorkflowApp._build_proc_tab = _build_proc_tab_with_safe_mouse_handlers
CDIWorkflowApp._tab6_physical_geometry = _tab6_physical_geometry
CDIWorkflowApp._tab6_nm_to_output_px = _tab6_nm_to_output_px
CDIWorkflowApp._tab6_output_px_to_nm_rect = _tab6_output_px_to_nm_rect
CDIWorkflowApp._tab6_secondary_status_nm = _tab6_secondary_status_nm
CDIWorkflowApp._proc_secondary_on_release = _proc_secondary_on_release_nm
CDIWorkflowApp._proc_secondary_on_move = _proc_secondary_on_move_nm
CDIWorkflowApp._proc_draw_secondary_roi_patch = _proc_draw_secondary_roi_patch_nm
_old_proc_update_physical_axes = CDIWorkflowApp._proc_update_physical_axes_and_colorbar

def _proc_update_physical_axes_and_colorbar_nm(self, display):
    _old_proc_update_physical_axes(self, display)
    base = self._proc_get_tab5_base_pixel_size()
    if base is not None and np.isfinite(base) and (base > 0):
        self._proc_ax.set_xlabel('X (nm)')
        self._proc_ax.set_ylabel('Y (nm)')
        self._proc_ax.set_aspect('equal', adjustable='box')
        self._proc_ax.set_anchor('C')
CDIWorkflowApp._proc_update_physical_axes_and_colorbar = _proc_update_physical_axes_and_colorbar_nm
CDIWorkflowApp._proc_secondary_on_move = _proc_secondary_on_move_nm_safe
CDIWorkflowApp._proc_secondary_on_release = _proc_secondary_on_release_nm_safe

def _tab6_primary_physical_extent(self):
    """Return the fixed physical extent of the confirmed Primary ROI in nm."""
    arr = getattr(self, '_proc_image_display', None)
    if arr is None:
        return None
    try:
        data = arr.get_array()
        if data is None:
            return None
        h, w = np.asarray(data).shape[:2]
    except Exception:
        return None
    try:
        base = self._proc_get_tab5_base_pixel_size()
        if base is None or not np.isfinite(base) or base <= 0:
            return (-0.5, float(w) - 0.5, -0.5, float(h) - 0.5)
        ow = int(getattr(self, 'original_roi_w', 0) or w)
        oh = int(getattr(self, 'original_roi_h', 0) or h)
        if ow <= 0 or oh <= 0:
            return (-0.5, float(w) - 0.5, -0.5, float(h) - 0.5)
        return (0.0, float(ow) * float(base), 0.0, float(oh) * float(base))
    except Exception:
        return (-0.5, float(w) - 0.5, -0.5, float(h) - 0.5)

def _tab6_fit_primary_axes(self):
    """Fit the complete Primary ROI image, nm axes and colorbar in the pane."""
    try:
        ax = self._proc_ax
        fig = self._proc_fig
        image = self._proc_image_display
        if image.get_array() is None:
            return
        full = _tab6_primary_physical_extent(self)
        if full is None:
            return
        fig.subplots_adjust(left=0.105, right=0.865, bottom=0.105, top=0.925)
        ax.set_axis_on()
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
        ax.set_xlabel('X (nm)', labelpad=5)
        ax.set_ylabel('Y (nm)', labelpad=5)
        ax.tick_params(axis='both', which='both', labelsize=8)
        try:
            ax.locator_params(axis='x', nbins=6)
            ax.locator_params(axis='y', nbins=6)
        except Exception:
            pass
        image.set_extent(full)
        ax.set_autoscale_on(False)
        limits = getattr(self, '_proc_zoom_limits', None)
        zoomed = bool(getattr(self, '_proc_zoomed', False)) and limits is not None
        if zoomed:
            x0, x1, y0, y1 = [float(v) for v in limits]
            fx0, fx1, fy0, fy1 = full
            x0, x1 = (max(fx0, min(x0, fx1)), max(fx0, min(x1, fx1)))
            y0, y1 = (max(fy0, min(y0, fy1)), max(fy0, min(y1, fy1)))
            if x1 > x0 and y1 > y0:
                ax.set_xlim(x0, x1)
                ax.set_ylim(y0, y1)
            else:
                self._proc_zoom_limits = None
                self._proc_zoomed = False
                ax.set_xlim(fx0, fx1)
                ax.set_ylim(fy0, fy1)
        else:
            ax.set_xlim(full[0], full[1])
            ax.set_ylim(full[2], full[3])
        try:
            cb = getattr(self, '_proc_colorbar', None)
            if cb is not None:
                cb.ax.set_position([0.885, 0.105, 0.035, 0.82])
                cb.set_label('Intensity', labelpad=6)
                cb.ax.tick_params(labelsize=8)
        except Exception:
            pass
        self._proc_canvas.draw_idle()
    except Exception as exc:
        try:
            self.log(f'Tab 6 primary ROI fit warning: {exc}')
        except Exception:
            pass

def _tab6_zoom_scroll_final(self, event):
    """Final Tab-6 cursor-centred wheel zoom in the displayed nm coordinates."""
    ax = getattr(self, '_proc_ax', None)
    canvas = getattr(self, '_proc_canvas', None)
    image_artist = getattr(self, '_proc_image_display', None)
    if ax is None or canvas is None or image_artist is None:
        return
    if getattr(event, 'inaxes', None) is not ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        return
    full = _tab6_primary_physical_extent(self)
    if full is None:
        return
    fx0, fx1, fy0, fy1 = [float(v) for v in full]
    x0, x1 = [float(v) for v in ax.get_xlim()]
    y0, y1 = [float(v) for v in ax.get_ylim()]
    sx = abs(x1 - x0)
    sy = abs(y1 - y0)
    if sx <= 1e-12 or sy <= 1e-12:
        return
    factor = 1.2 ** (-step)
    cx, cy = (float(event.xdata), float(event.ydata))
    xmin, xmax = (min(x0, x1), max(x0, x1))
    ymin, ymax = (min(y0, y1), max(y0, y1))
    rx = 0.5 if sx <= 1e-12 else (cx - xmin) / sx
    ry = 0.5 if sy <= 1e-12 else (cy - ymin) / sy
    rx = min(1.0, max(0.0, rx))
    ry = min(1.0, max(0.0, ry))
    nsx = sx * factor
    nsy = sy * factor
    nxmin = cx - rx * nsx
    nxmax = nxmin + nsx
    nymin = cy - ry * nsy
    nymax = nymin + nsy
    if nxmin < fx0:
        d = fx0 - nxmin
        nxmin += d
        nxmax += d
    if nxmax > fx1:
        d = nxmax - fx1
        nxmin -= d
        nxmax -= d
    if nymin < fy0:
        d = fy0 - nymin
        nymin += d
        nymax += d
    if nymax > fy1:
        d = nymax - fy1
        nymin -= d
        nymax -= d
    if nxmax - nxmin >= 0.999999 * (fx1 - fx0) and nymax - nymin >= 0.999999 * (fy1 - fy0):
        self._proc_reset_zoom()
        _tab6_fit_primary_axes(self)
        return
    nxmin = max(fx0, nxmin)
    nxmax = min(fx1, nxmax)
    nymin = max(fy0, nymin)
    nymax = min(fy1, nymax)
    if nxmax <= nxmin or nymax <= nymin:
        return
    self._proc_zoom_limits = (nxmin, nxmax, nymin, nymax)
    self._proc_zoomed = True
    ax.set_autoscale_on(False)
    ax.set_xlim(nxmin, nxmax)
    ax.set_ylim(nymin, nymax)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    canvas.draw_idle()
_previous_tab6_axis_update = CDIWorkflowApp._proc_update_physical_axes_and_colorbar

def _proc_update_physical_axes_and_colorbar_final(self, display):
    _previous_tab6_axis_update(self, display)
    _tab6_fit_primary_axes(self)
CDIWorkflowApp._proc_update_physical_axes_and_colorbar = _proc_update_physical_axes_and_colorbar_final
_previous_tab6_build_proc = CDIWorkflowApp._build_proc_tab

def _build_proc_tab_final_fit_and_zoom(self):
    _previous_tab6_build_proc(self)
    canvas = self._proc_canvas
    for signal in ('scroll_event', 'button_press_event', 'motion_notify_event', 'button_release_event'):
        try:
            registry = getattr(canvas.callbacks, 'callbacks', {}).get(signal, {})
            for cid in list(registry.keys()):
                try:
                    canvas.mpl_disconnect(cid)
                except Exception:
                    pass
        except Exception:
            pass
    canvas.mpl_connect('scroll_event', self._tab6_wheel_zoom_handler)
    canvas.mpl_connect('button_press_event', self._proc_secondary_on_press)
    canvas.mpl_connect('motion_notify_event', self._proc_secondary_on_move)
    canvas.mpl_connect('button_release_event', self._proc_secondary_on_release)
    canvas.mpl_connect('button_press_event', self._proc_profile_on_press)
    canvas.mpl_connect('motion_notify_event', self._proc_profile_on_move)
    canvas.mpl_connect('button_release_event', self._proc_profile_on_release)
    _tab6_fit_primary_axes(self)

def _tab6_wheel_zoom_handler(self, event):
    _tab6_zoom_scroll_final(self, event)
CDIWorkflowApp._tab6_wheel_zoom_handler = _tab6_wheel_zoom_handler
CDIWorkflowApp._build_proc_tab = _build_proc_tab_final_fit_and_zoom

def _tab5_final_primary_nm_per_pixel(self):
    """Return the authoritative Final Primary ROI nm/pixel from Tab 5."""
    try:
        v = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
        if np.isfinite(v) and v > 0:
            return v
    except Exception:
        pass
    return None

def _tab5_primary_roi_pixel_shape(self):
    """Return the exact Primary ROI pixel dimensions represented in Tab 5."""
    shape = getattr(self, '_pixel_cal_roi_display_shape', None)
    if shape is not None:
        try:
            h, w = (int(shape[0]), int(shape[1]))
            if h > 0 and w > 0:
                return (w, h)
        except Exception:
            pass
    try:
        w = int(getattr(self, 'original_roi_w', 0) or 0)
        h = int(getattr(self, 'original_roi_h', 0) or 0)
        if w > 0 and h > 0:
            return (w, h)
    except Exception:
        pass
    try:
        arr = self._proc_image_display.get_array()
        if arr is not None:
            h, w = np.asarray(arr).shape[:2]
            if h > 0 and w > 0:
                return (int(w), int(h))
    except Exception:
        pass
    return None

def _proc_get_tab5_physical_extent(self, display=None):
    """
    Calculate the fixed physical Primary ROI extent from Tab 5.

    The displayed Tab-6 array may be resampled, but its physical field of view
    is always the original Tab-5 ROI pixel dimensions times the final nm/px.
    """
    base = _tab5_final_primary_nm_per_pixel(self)
    shape = _tab5_primary_roi_pixel_shape(self)
    if base is None or shape is None:
        if display is not None:
            try:
                h, w = np.asarray(display).shape[:2]
                return (-0.5, float(w) - 0.5, -0.5, float(h) - 0.5)
            except Exception:
                pass
        return None
    w_px, h_px = shape
    return (0.0, float(w_px) * base, 0.0, float(h_px) * base)

def _proc_update_physical_axes_authoritative(self, display):
    """
    Display Tab 6 Primary ROI with X/Y axes in nm using Tab 5 calibration.

    IMPORTANT:
    - Tab 5 Final Primary ROI nm/pixel is authoritative.
    - Tab 5 ROI width/height are authoritative for the physical FOV.
    - Tab 6 resampling changes only sampling density, never physical extent.
    """
    arr = np.asarray(display)
    if arr.ndim < 2:
        return
    extent = _proc_get_tab5_physical_extent(self, display)
    if extent is None:
        return
    x0, x1, y0, y1 = extent
    ax = self._proc_ax
    im = self._proc_image_display
    im.set_extent((x0, x1, y0, y1))
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    ax.set_axis_on()
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    ax.set_autoscale_on(False)
    self._proc_full_physical_extent = extent
    limits = getattr(self, '_proc_zoom_limits', None)
    if limits is None:
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)
        self._proc_zoomed = False
    else:
        _proc_restore_authoritative_zoom(self)
    try:
        im.set_clim(0.0, 1.0)
        if getattr(self, '_proc_colorbar', None) is not None:
            self._proc_colorbar.update_normal(im)
            self._proc_colorbar.set_label('Intensity')
    except Exception:
        pass

def _proc_restore_authoritative_zoom(self):
    """Restore/clamp Tab-6 zoom in the physical nm coordinate system."""
    extent = getattr(self, '_proc_full_physical_extent', None)
    if extent is None:
        extent = _proc_get_tab5_physical_extent(self)
    if extent is None:
        return
    fx0, fx1, fy0, fy1 = [float(v) for v in extent]
    limits = getattr(self, '_proc_zoom_limits', None)
    if limits is None:
        self._proc_ax.set_xlim(fx0, fx1)
        self._proc_ax.set_ylim(fy0, fy1)
        self._proc_zoomed = False
        return
    try:
        x0, x1, y0, y1 = [float(v) for v in limits]
    except Exception:
        self._proc_zoom_limits = None
        self._proc_zoomed = False
        self._proc_ax.set_xlim(fx0, fx1)
        self._proc_ax.set_ylim(fy0, fy1)
        return
    x0 = max(fx0, min(x0, fx1))
    x1 = max(fx0, min(x1, fx1))
    y0 = max(fy0, min(y0, fy1))
    y1 = max(fy0, min(y1, fy1))
    if x1 <= x0 or y1 <= y0:
        self._proc_zoom_limits = None
        self._proc_zoomed = False
        self._proc_ax.set_xlim(fx0, fx1)
        self._proc_ax.set_ylim(fy0, fy1)
        return
    self._proc_zoom_limits = (x0, x1, y0, y1)
    self._proc_zoomed = True
    self._proc_ax.set_autoscale_on(False)
    self._proc_ax.set_xlim(x0, x1)
    self._proc_ax.set_ylim(y0, y1)
    self._proc_ax.set_aspect('equal', adjustable='box')
    self._proc_ax.set_anchor('C')

def _proc_reset_zoom_authoritative(self):
    """Reset Tab 6 Primary ROI to the complete physical ROI."""
    extent = _proc_get_tab5_physical_extent(self)
    if extent is None:
        return
    x0, x1, y0, y1 = extent
    self._proc_full_physical_extent = extent
    self._proc_zoom_limits = None
    self._proc_zoomed = False
    self._proc_ax.set_autoscale_on(False)
    self._proc_ax.set_xlim(x0, x1)
    self._proc_ax.set_ylim(y0, y1)
    self._proc_ax.set_aspect('equal', adjustable='box')
    self._proc_ax.set_anchor('C')
    self._proc_ax.set_axis_on()
    self._proc_canvas.draw_idle()

def _proc_zoom_scroll_authoritative(self, event):
    """
    Mouse-wheel-only cursor-centred zoom for Tab 6 in physical nm coordinates.
    """
    if getattr(event, 'inaxes', None) is not self._proc_ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        return
    extent = _proc_get_tab5_physical_extent(self)
    if extent is None:
        return
    fx0, fx1, fy0, fy1 = extent
    ax = self._proc_ax
    x0, x1 = [float(v) for v in ax.get_xlim()]
    y0, y1 = [float(v) for v in ax.get_ylim()]
    factor = 1.2 ** (-step)
    cx, cy = (float(event.xdata), float(event.ydata))
    rx = (cx - x0) / (x1 - x0) if x1 != x0 else 0.5
    ry = (cy - y0) / (y1 - y0) if y1 != y0 else 0.5
    nx0 = cx - rx * (x1 - x0) * factor
    nx1 = cx + (1.0 - rx) * (x1 - x0) * factor
    ny0 = cy - ry * (y1 - y0) * factor
    ny1 = cy + (1.0 - ry) * (y1 - y0) * factor
    if nx1 - nx0 >= 0.999999 * (fx1 - fx0):
        nx0, nx1 = (fx0, fx1)
    else:
        if nx0 < fx0:
            d = fx0 - nx0
            nx0 += d
            nx1 += d
        if nx1 > fx1:
            d = nx1 - fx1
            nx0 -= d
            nx1 -= d
        if nx0 < fx0:
            nx0 = fx0
        if nx1 > fx1:
            nx1 = fx1
    if ny1 - ny0 >= 0.999999 * (fy1 - fy0):
        ny0, ny1 = (fy0, fy1)
    else:
        if ny0 < fy0:
            d = fy0 - ny0
            ny0 += d
            ny1 += d
        if ny1 > fy1:
            d = ny1 - fy1
            ny0 -= d
            ny1 -= d
        if ny0 < fy0:
            ny0 = fy0
        if ny1 > fy1:
            ny1 = fy1
    if nx1 - nx0 >= 0.999999 * (fx1 - fx0) and ny1 - ny0 >= 0.999999 * (fy1 - fy0):
        return _proc_reset_zoom_authoritative(self)
    self._proc_zoom_limits = (nx0, nx1, ny0, ny1)
    self._proc_zoomed = True
    ax.set_autoscale_on(False)
    ax.set_xlim(nx0, nx1)
    ax.set_ylim(ny0, ny1)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    self._proc_canvas.draw_idle()
CDIWorkflowApp._proc_update_physical_axes_and_colorbar = _proc_update_physical_axes_authoritative
CDIWorkflowApp._proc_restore_zoom_limits = _proc_restore_authoritative_zoom
CDIWorkflowApp._proc_reset_zoom = _proc_reset_zoom_authoritative
CDIWorkflowApp._proc_zoom_scroll = _proc_zoom_scroll_authoritative

def _pixel_cal_zoom_scroll_authoritative(self, which, event):
    if which not in ('sem', 'roi'):
        return
    ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
    canvas = self.pixel_cal_sem_canvas if which == 'sem' else self.pixel_cal_roi_canvas
    if getattr(event, 'inaxes', None) is not ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        return
    image = self._pixel_cal_get_display_image(which)
    if image is None:
        return
    h, w = np.asarray(image).shape[:2]
    if h <= 0 or w <= 0:
        return
    x0, x1 = [float(v) for v in ax.get_xlim()]
    y0, y1 = [float(v) for v in ax.get_ylim()]
    factor = 1.2 ** (-step)
    cx, cy = (float(event.xdata), float(event.ydata))
    rx = (cx - x0) / (x1 - x0) if x1 != x0 else 0.5
    ry = (cy - y0) / (y1 - y0) if y1 != y0 else 0.5
    nsx = (x1 - x0) * factor
    nsy = (y1 - y0) * factor
    nx0 = cx - rx * nsx
    nx1 = nx0 + nsx
    ny0 = cy - ry * nsy
    ny1 = ny0 + nsy
    xdesc = x0 > x1
    ydesc = y0 > y1
    fx0, fx1 = (-0.5, float(w) - 0.5)
    fy0, fy1 = (-0.5, float(h) - 0.5)
    xlo, xhi = sorted((nx0, nx1))
    ylo, yhi = sorted((ny0, ny1))
    if xhi - xlo >= 0.999999 * (fx1 - fx0) and yhi - ylo >= 0.999999 * (fy1 - fy0):
        self._pixel_cal_reset_zoom(which)
        return
    if xlo < fx0:
        d = fx0 - xlo
        xlo += d
        xhi += d
    if xhi > fx1:
        d = xhi - fx1
        xlo -= d
        xhi -= d
    if ylo < fy0:
        d = fy0 - ylo
        ylo += d
        yhi += d
    if yhi > fy1:
        d = yhi - fy1
        ylo -= d
        yhi -= d
    xlo, xhi = (max(fx0, xlo), min(fx1, xhi))
    ylo, yhi = (max(fy0, ylo), min(fy1, yhi))
    xlims = (xhi, xlo) if xdesc else (xlo, xhi)
    ylims = (yhi, ylo) if ydesc else (ylo, yhi)
    self._pixel_cal_zoom_limits[which] = (xlims, ylims)
    ax.set_autoscale_on(False)
    ax.set_xlim(*xlims)
    ax.set_ylim(*ylims)
    ax.set_aspect('equal', adjustable='box')
    canvas.draw_idle()
CDIWorkflowApp._pixel_cal_zoom_scroll = _pixel_cal_zoom_scroll_authoritative

def _tab5_final_roi_display_shape_nm(self):
    """Return the exact Primary ROI image size used for Tab-5 final calibration."""
    shape = getattr(self, '_pixel_cal_roi_display_shape', None)
    if shape is not None:
        try:
            h, w = (int(shape[0]), int(shape[1]))
            if h > 0 and w > 0:
                return (w, h)
        except Exception:
            pass
    try:
        w = int(getattr(self, 'original_roi_w', 0) or 0)
        h = int(getattr(self, 'original_roi_h', 0) or 0)
        if w > 0 and h > 0:
            return (w, h)
    except Exception:
        pass
    return None

def _tab6_effective_nm_per_output_px(self):
    """Return the physical nm/output-pixel scale used by Tab 6.

    Tab 5 Final Primary ROI calibration is authoritative for the original
    physical field.  Tab 6 may resize/resample the image, so the effective
    output-pixel scale must be:

        original Primary ROI physical size / actual Tab-6 output size

    This keeps Tab 6 and Tab 8 on exactly the same physical X/Y scale.
    """
    try:
        base = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
    except Exception:
        base = np.nan

    if not np.isfinite(base) or base <= 0:
        return None

    # Actual image represented by Tab 6.
    shape = None
    try:
        arr = getattr(self, '_proc_image_display', None)
        if arr is not None and arr.get_array() is not None:
            h, w = np.asarray(arr.get_array()).shape[:2]
            if w > 0 and h > 0:
                shape = (int(w), int(h))
    except Exception:
        shape = None

    if shape is None:
        try:
            rw, rh = getattr(self, 'roi_resolution', (0, 0))
            if int(rw) > 0 and int(rh) > 0:
                shape = (int(rw), int(rh))
        except Exception:
            pass

    if shape is None:
        return None

    out_w, out_h = shape

    # IMPORTANT:
    # original_roi_w/h are the physical Primary ROI dimensions selected
    # before Tab-6 output resizing.  Do not use the already-resampled
    # Tab-5 display shape here, otherwise the physical FOV can shrink to
    # the output dimensions.
    try:
        original_w = int(getattr(self, 'original_roi_w', 0) or 0)
        original_h = int(getattr(self, 'original_roi_h', 0) or 0)
    except Exception:
        original_w = original_h = 0

    if original_w > 0 and original_h > 0:
        physical_w_nm = float(original_w) * base
        physical_h_nm = float(original_h) * base
        return (
            physical_w_nm / float(out_w),
            physical_h_nm / float(out_h)
        )

    # Fallback only when original Primary ROI dimensions are unavailable.
    return (float(base), float(base))

def _tab6_physical_geometry_nm(self):
    """Return physical FOV and current output-pixel shape for Tab 6."""
    arr = getattr(self, '_proc_image_display', None)
    if arr is None or arr.get_array() is None:
        return None
    data = np.asarray(arr.get_array())
    if data.ndim < 2:
        return None
    h, w = data.shape[:2]
    scales = _tab6_effective_nm_per_output_px(self)
    if scales is None:
        return None
    sx, sy = scales
    return (float(w) * sx, float(h) * sy, int(w), int(h), float(sx), float(sy))

def _tab6_output_px_to_nm_rect_physical(self, xmin_px, xmax_px, ymin_px, ymax_px):
    geom = _tab6_physical_geometry_nm(self)
    if geom is None:
        return (float(xmin_px), float(xmax_px), float(ymin_px), float(ymax_px))
    _pw, _ph, w, h, sx, sy = geom
    return (float(xmin_px) * sx, float(xmax_px + 1) * sx, float(ymin_px) * sy, float(ymax_px + 1) * sy)

def _tab6_secondary_status_nm_physical(self):
    c = getattr(self, 'roi2_coords', None)
    if c is None:
        return 'Secondary ROI: not selected'
    x0, x1, y0, y1 = _tab6_output_px_to_nm_rect_physical(self, int(c['xmin']), int(c['xmax']), int(c['ymin']), int(c['ymax']))
    return f'Secondary ROI: X={x0:.6g}–{x1:.6g} nm, Y={y0:.6g}–{y1:.6g} nm | Width={x1 - x0:.6g} nm, Height={y1 - y0:.6g} nm'
CDIWorkflowApp._tab6_physical_geometry = _tab6_physical_geometry_nm
CDIWorkflowApp._tab6_output_px_to_nm_rect = _tab6_output_px_to_nm_rect_physical
CDIWorkflowApp._tab6_secondary_status_nm = _tab6_secondary_status_nm_physical
_prev_tab6_physical_update_nm = CDIWorkflowApp._proc_update_physical_axes_and_colorbar

def _proc_update_physical_axes_and_colorbar_integrated_nm(self, display):
    arr = np.asarray(display)
    if arr.ndim < 2:
        return
    h, w = arr.shape[:2]
    if h <= 0 or w <= 0:
        return
    base = _tab5_final_roi_display_shape_nm(self)
    try:
        final_base = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
    except Exception:
        final_base = np.nan
    scales = _tab6_effective_nm_per_output_px(self)
    ax = self._proc_ax
    im = self._proc_image_display
    ax.set_axis_on()
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    ax.set_autoscale_on(False)
    if scales is not None and np.isfinite(final_base) and (final_base > 0):
        sx, sy = scales
        physical_w = float(w) * sx
        physical_h = float(h) * sy
        extent = (0.0, physical_w, 0.0, physical_h)
        im.set_extent(extent)
        ax.set_xlabel('X (nm)')
        ax.set_ylabel('Y (nm)')
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        self._proc_full_physical_extent = extent
        self.proc_pixel_size_var.set(f'Pixel Size = {final_base:.9g} nm/px')
        self.proc_effective_pixel_size_var.set(f'Tab 6 measurement scale = X {sx:.9g}, Y {sy:.9g} nm/output-px (output {w} × {h})')
        try:
            if getattr(self, '_proc_colorbar', None) is not None:
                self._proc_colorbar.update_normal(im)
                self._proc_colorbar.set_label('Intensity')
                im.set_clim(0.0, 1.0)
        except Exception:
            pass
        if getattr(self, '_proc_zoomed', False) and getattr(self, '_proc_zoom_limits', None) is not None:
            try:
                x0, x1, y0, y1 = [float(v) for v in self._proc_zoom_limits]
                x0 = max(0.0, min(x0, physical_w))
                x1 = max(0.0, min(x1, physical_w))
                y0 = max(0.0, min(y0, physical_h))
                y1 = max(0.0, min(y1, physical_h))
                if x1 > x0 and y1 > y0:
                    ax.set_xlim(x0, x1)
                    ax.set_ylim(y0, y1)
                else:
                    self._proc_zoom_limits = None
                    self._proc_zoomed = False
            except Exception:
                self._proc_zoom_limits = None
                self._proc_zoomed = False
    else:
        im.set_extent((-0.5, w - 0.5, -0.5, h - 0.5))
        ax.set_xlabel('X (pixel)')
        ax.set_ylabel('Y (pixel)')
        ax.set_xlim(-0.5, w - 0.5)
        ax.set_ylim(-0.5, h - 0.5)
        self._proc_full_physical_extent = (-0.5, w - 0.5, -0.5, h - 0.5)
CDIWorkflowApp._proc_update_physical_axes_and_colorbar = _proc_update_physical_axes_and_colorbar_integrated_nm

def _tab6_update_processing_pixel_size_integrated(self):
    try:
        base = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
    except Exception:
        base = np.nan
    scales = _tab6_effective_nm_per_output_px(self)
    if not np.isfinite(base) or base <= 0:
        self.proc_pixel_size_var.set('Pixel Size = -- nm/px')
        self.proc_effective_pixel_size_var.set('Tab 5 Final Primary ROI calibration unavailable')
        self.proc_profile_measure_nm_per_px = None
        return
    self.proc_pixel_size_var.set(f'Pixel Size = {base:.9g} nm/px')
    if scales is None:
        self.proc_effective_pixel_size_var.set('Tab 6 effective sampling scale unavailable')
        self.proc_profile_measure_nm_per_px = None
        return
    sx, sy = scales
    self.proc_profile_measure_nm_per_px = float(sx)
    arr = getattr(self, '_proc_image_display', None)
    try:
        h, w = np.asarray(arr.get_array()).shape[:2]
    except Exception:
        h, w = (0, 0)
    self.proc_effective_pixel_size_var.set(f'Tab 6 measurement scale = X {sx:.9g}, Y {sy:.9g} nm/output-px (output {w} × {h}; Tab 5 remains {base:.9g} nm/px)')
CDIWorkflowApp.update_processing_pixel_size = _tab6_update_processing_pixel_size_integrated

def _tab6_confirm_secondary_status_wrap(self, *args, **kwargs):
    result = _TAB6_CONFIRM_SECONDARY_BASE(self, *args, **kwargs)
    try:
        self.proc_secondary_status_var.set('✓ SECONDARY ROI CONFIRMED. ' + _tab6_secondary_status_nm_physical(self) + ' | Same internal pixel ROI is retained for processing.')
    except Exception:
        pass
    return result
_TAB6_CONFIRM_SECONDARY_BASE = CDIWorkflowApp.confirm_secondary_roi_from_processing
CDIWorkflowApp.confirm_secondary_roi_from_processing = _tab6_confirm_secondary_status_wrap

def _tab7_line_profile_scales_nm(self):
    """Return x/y nm per output pixel for the confirmed Secondary ROI stack."""
    scales = _tab6_effective_nm_per_output_px(self)
    if scales is not None:
        return scales
    try:
        stack = np.asarray(self.roi2_stack)
        if stack.ndim == 3 and stack.shape[0] > 0:
            h, w = stack.shape[1:3]
            base = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
            tab5_shape = _tab5_final_roi_display_shape_nm(self)
            if np.isfinite(base) and base > 0 and (tab5_shape is not None):
                tw, th = tab5_shape
                return (base * tw / w, base * th / h)
    except Exception:
        pass
    return None

def _tab7_px_to_nm_point(self, p):
    scales = _tab7_line_profile_scales_nm(self)
    if scales is None:
        return (float(p[0]), float(p[1]))
    sx, sy = scales
    return (float(p[0]) * sx, float(p[1]) * sy)

def _tab7_line_distance_nm(self, p0, p1, dist_px):
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    dx, dy = (float(p1[0] - p0[0]), float(p1[1] - p0[1]))
    length_px = float(np.hypot(dx, dy))
    scales = _tab7_line_profile_scales_nm(self)
    if scales is None or length_px <= 0:
        return np.asarray(dist_px, dtype=float)
    sx, sy = scales
    factor = float(np.hypot(dx / length_px * sx, dy / length_px * sy))
    return np.asarray(dist_px, dtype=float) * factor

def _tab7_nm_to_profile_px(self, result, x_nm):
    dnm = np.asarray(result.get('dist_nm', []), dtype=float)
    dpx = np.asarray(result.get('dist', []), dtype=float)
    if dnm.size == 0 or dpx.size != dnm.size:
        return None
    xx = float(np.clip(x_nm, float(dnm[0]), float(dnm[-1])))
    return float(np.interp(xx, dnm, dpx))

def _tab7_redraw_image_lines_nm(self):
    for a in getattr(self, '_line_profile_image_line_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_image_line_artists = []
    for i, line in enumerate(getattr(self, '_line_profile_lines', [])):
        c = self._line_profile_line_color(i)
        p0n = _tab7_px_to_nm_point(self, line['p0'])
        p1n = _tab7_px_to_nm_point(self, line['p1'])
        a, = self.line_profile_ax_image.plot([p0n[0], p1n[0]], [p0n[1], p1n[1]], '-', lw=2.2, color=c, zorder=12)
        self._line_profile_image_line_artists.append(a)
    for a in getattr(self, '_line_profile_measure_image_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_image_artists = []
    for m in getattr(self, '_line_profile_measurements_list', []):
        pts = m.get('pts', [])
        if len(pts) != 2:
            continue
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(m.get('setno', -1))), None)
        if r is None:
            continue
        Lpx = float(r.get('length', 0.0))
        if Lpx <= 0:
            continue
        p0 = np.asarray(r['p0'], dtype=float)
        p1 = np.asarray(r['p1'], dtype=float)
        xy_px = []
        for x, _y, _sn in pts:
            frac = float(np.clip(float(x) / Lpx, 0.0, 1.0))
            xy_px.append(p0 + frac * (p1 - p0))
        a_px, b_px = xy_px
        a_nm = _tab7_px_to_nm_point(self, a_px)
        b_nm = _tab7_px_to_nm_point(self, b_px)
        shadow, = self.line_profile_ax_image.plot([a_nm[0], b_nm[0]], [a_nm[1], b_nm[1]], '-', lw=6.5, color='black', alpha=0.75, zorder=25)
        yellow, = self.line_profile_ax_image.plot([a_nm[0], b_nm[0]], [a_nm[1], b_nm[1]], '-', lw=3.8, color='yellow', zorder=26)
        self._line_profile_measure_image_artists.extend([shadow, yellow])
    self.line_profile_canvas_image.draw_idle()

def _tab7_extrema_stats_nm(self, dist_px, profile, dist_nm):
    from scipy.signal import find_peaks
    dist_px = np.asarray(dist_px, dtype=float)
    dist_nm = np.asarray(dist_nm, dtype=float)
    profile = np.asarray(profile, dtype=float)
    if dist_nm.size < 3 or profile.size != dist_nm.size:
        return {'min_x': np.array([], float), 'max_x': np.array([], float), 'minmin': (np.nan, np.nan, 0), 'maxmax': (np.nan, np.nan, 0), 'minmax': (np.nan, np.nan, 0), 'all': (np.nan, np.nan, 0)}
    try:
        spacing_nm = max(0.0, float(self.line_profile_min_distance_var.get()))
    except Exception:
        spacing_nm = 3.0
    try:
        prom = max(0.0, min(1.0, float(self.line_profile_prominence_var.get())))
    except Exception:
        prom = 0.03
    dstep = float(np.median(np.diff(dist_nm))) if dist_nm.size > 1 else 1.0
    sample_distance = max(1, int(np.ceil(spacing_nm / max(dstep, 1e-12))))
    max_idx, _ = find_peaks(profile, distance=sample_distance, prominence=prom)
    min_idx, _ = find_peaks(-profile, distance=sample_distance, prominence=prom)

    def refine(indices):
        out = []
        for j in np.asarray(indices, dtype=int):
            if j <= 0 or j >= len(profile) - 1:
                out.append(float(dist_nm[j]))
                continue
            y1, y2, y3 = (float(profile[j - 1]), float(profile[j]), float(profile[j + 1]))
            den = y1 - 2 * y2 + y3
            delta = 0.0 if abs(den) < 1e-14 else float(np.clip(0.5 * (y1 - y3) / den, -1.0, 1.0))
            out.append(float(dist_nm[j] + delta * dstep))
        return np.asarray(out, float)
    min_x, max_x = (refine(min_idx), refine(max_idx))
    mm = np.abs(np.diff(min_x)) if min_x.size >= 2 else np.array([], float)
    xx = np.abs(np.diff(max_x)) if max_x.size >= 2 else np.array([], float)
    events = sorted([(float(x), 'min') for x in min_x] + [(float(x), 'max') for x in max_x], key=lambda q: q[0])
    mixed = []
    all_d = []
    for a, b in zip(events[:-1], events[1:]):
        d = abs(b[0] - a[0])
        all_d.append(d)
        if a[1] != b[1]:
            mixed.append(d)

    def stat(a):
        a = np.asarray(a, float)
        if a.size == 0:
            return (np.nan, np.nan, 0)
        return (float(np.mean(a)), float(np.std(a, ddof=1)) if a.size > 1 else 0.0, int(a.size))
    return {'min_x': min_x, 'max_x': max_x, 'minmin': stat(mm), 'maxmax': stat(xx), 'minmax': stat(mixed), 'all': stat(all_d)}

def _tab7_update_image_nm(self, idx=None, preserve_lines=True):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        self._line_profile_sync_from_secondary_roi()
        return
    if idx is None:
        try:
            idx = int(round(float(self.line_profile_frame_var.get())))
        except Exception:
            idx = 0
    idx = max(0, min(int(idx), len(stack) - 1))
    try:
        self.line_profile_frame_var.set(idx)
    except Exception:
        pass
    image = np.asarray(stack[idx], dtype=float)
    h, w = image.shape[:2]
    scales = _tab7_line_profile_scales_nm(self)
    sx, sy = scales if scales is not None else (1.0, 1.0)
    self.line_profile_image.set_data(image)
    try:
        self.line_profile_image.set_cmap(self.colormap_var.get() or 'gray')
    except Exception:
        pass
    self.line_profile_image.set_clim(0, 1)
    self.line_profile_image.set_extent((0.0, float(w) * sx, 0.0, float(h) * sy))
    self.line_profile_ax_image.set_xlim(0.0, float(w) * sx)
    self.line_profile_ax_image.set_ylim(0.0, float(h) * sy)
    self.line_profile_ax_image.set_aspect('equal', adjustable='box')
    self.line_profile_ax_image.set_anchor('C')
    self.line_profile_ax_image.set_xlabel('X (nm)')
    self.line_profile_ax_image.set_ylabel('Y (nm)')
    field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    self.line_profile_ax_image.set_title(f'Secondary ROI — Frame {idx + 1}/{len(stack)} | Field = {field:+.2f} mT')
    self.line_profile_colorbar.update_normal(self.line_profile_image)
    self.line_profile_frame_label.config(text=f'{idx + 1}/{len(stack)} | {field:+.2f} mT')
    if preserve_lines and getattr(self, '_line_profile_lines', None):
        self._line_profile_redraw_profiles(idx)
    else:
        self.line_profile_canvas_graph.draw_idle()
    self._line_profile_redraw_image_lines()
    self.line_profile_canvas_image.draw_idle()

def _tab7_redraw_profiles_nm(self, idx=None):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        self._line_profile_results = []
        self._line_profile_redraw_image_lines()
        self._line_profile_update_table()
        return
    if idx is None:
        try:
            idx = int(round(float(self.line_profile_frame_var.get())))
        except Exception:
            idx = 0
    idx = max(0, min(int(idx), len(stack) - 1))
    image = np.asarray(stack[idx], dtype=float)
    self.line_profile_ax_graph.clear()
    self.line_profile_ax_graph.set_xlabel('Distance (nm)')
    self.line_profile_ax_graph.set_ylabel('Intensity')
    self.line_profile_ax_graph.set_title(f'Secondary ROI line profiles — Frame {idx + 1}/{len(stack)}')
    self.line_profile_ax_graph.grid(True, alpha=0.22)
    self._line_profile_results = []
    self._line_profile_graph_artists = []
    for i, line in enumerate(getattr(self, '_line_profile_lines', [])):
        dist_px, profile = self._line_profile_extract(image, line['p0'], line['p1'])
        dist_nm = _tab7_line_distance_nm(self, line['p0'], line['p1'], dist_px)
        stats = _tab7_extrema_stats_nm(self, dist_px, profile, dist_nm)
        result = {'set': i + 1, 'p0': line['p0'], 'p1': line['p1'], 'length': float(dist_px[-1]) if len(dist_px) else 0.0, 'length_nm': float(dist_nm[-1]) if len(dist_nm) else 0.0, 'dist': dist_px, 'dist_nm': dist_nm, 'profile': profile, 'stats': stats}
        self._line_profile_results.append(result)
        ln, = self.line_profile_ax_graph.plot(dist_nm, profile, lw=2.0, color=self._line_profile_line_color(i), label=f'Set {i + 1}', picker=14, pickradius=14, zorder=4)
        ln._line_profile_setno = i + 1
        self._line_profile_graph_artists.append(ln)
    if self._line_profile_results:
        self.line_profile_ax_graph.legend(loc='best')
    self._line_profile_current_profile = self._line_profile_results[-1] if self._line_profile_results else None
    self._line_profile_redraw_image_lines()
    self._line_profile_update_table()
    self._line_profile_draw_measurement()
    self.line_profile_canvas_graph.draw_idle()

def _tab7_project_to_curve_nm(self, event_x_nm, event_y, set_hint=None, max_px=90.0):
    results = list(getattr(self, '_line_profile_results', []) or [])
    if set_hint is not None:
        results = [r for r in results if int(r.get('set', -1)) == int(set_hint)] or results
    if event_x_nm is None or event_y is None or (not results):
        return None
    ax = self.line_profile_ax_graph
    target = ax.transData.transform((float(event_x_nm), float(event_y)))
    best = None
    for r in results:
        dnm = np.asarray(r.get('dist_nm', []), float)
        dpx = np.asarray(r.get('dist', []), float)
        prof = np.asarray(r.get('profile', []), float)
        if dnm.size < 2 or dpx.size != dnm.size or prof.size != dnm.size:
            continue
        screen = ax.transData.transform(np.column_stack((dnm, prof)))
        a = screen[:-1]
        b = screen[1:]
        v = b - a
        vv = np.einsum('ij,ij->i', v, v)
        wv = target - a
        with np.errstate(divide='ignore', invalid='ignore'):
            t = np.einsum('ij,ij->i', wv, v) / np.where(vv > 0, vv, 1.0)
        t = np.clip(t, 0.0, 1.0)
        q = a + v * t[:, None]
        d2 = np.einsum('ij,ij->i', q - target, q - target)
        j = int(np.argmin(d2))
        dscreen = float(np.sqrt(max(d2[j], 0.0)))
        if best is None or dscreen < best[0]:
            xnm0, xnm1 = (float(dnm[j]), float(dnm[j + 1]))
            y0, y1 = (float(prof[j]), float(prof[j + 1]))
            tt = float(t[j])
            xnm = xnm0 + tt * (xnm1 - xnm0)
            yy = y0 + tt * (y1 - y0)
            xpx = float(np.interp(xnm, dnm, dpx))
            best = (dscreen, xpx, yy, int(r.get('set', 1)))
    if best is None or best[0] > float(max_px):
        return None
    return best

def _tab7_graph_press_nm(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph or event.button not in (None, 1):
        return
    best = None
    for pi, m in enumerate(getattr(self, '_line_profile_measurements_list', []) or []):
        setno = int(m.get('setno', 1))
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == setno), None)
        if r is None:
            continue
        dnm = np.asarray(r.get('dist_nm', []), float)
        dpx = np.asarray(r.get('dist', []), float)
        for point_index, (px, _py, _sn) in enumerate(m.get('pts', [])[:2]):
            xnm = float(np.interp(float(px), dpx, dnm)) if dnm.size and dpx.size == dnm.size else float(px)
            sx, sy = self.line_profile_ax_graph.transData.transform((xnm, float(_py)))
            dd = float(np.hypot(event.x - sx, event.y - sy))
            if best is None or dd < best[0]:
                best = (dd, pi, point_index)
    if best is not None and best[0] <= 24.0:
        self._line_profile_measure_drag_pair = best[1]
        self._line_profile_measure_drag_point = best[2]
        self._line_profile_measure_mode = False
        pair = self._line_profile_measurements_list[best[1]]
        mid = pair['m1_id'] if best[2] == 0 else pair['m2_id']
        self.line_profile_measure_var.set(f"Dragging M{mid} of pair M{pair['m1_id']}–M{pair['m2_id']}. Release to keep the new physical position.")
        return
    if getattr(self, '_line_profile_measure_mode', False):
        _tab7_select_pair_point_nm(self, event)

def _tab7_select_pair_point_nm(self, event):
    best = _tab7_project_to_curve_nm(self, event.xdata, event.ydata, max_px=85.0)
    if best is None:
        self.line_profile_measure_var.set('No profile found near the cursor. Click directly on a profile curve.')
        return
    _dscreen, xpx, y, setno = best
    setno = int(setno)
    pending = getattr(self, '_line_profile_pending_point', None)
    pair_no = len(getattr(self, '_line_profile_measurements_list', [])) + 1
    m1_id = 2 * pair_no - 1
    m2_id = 2 * pair_no
    r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == setno), None)
    dnm = np.asarray(r.get('dist_nm', []), float) if r else np.array([])
    dpx = np.asarray(r.get('dist', []), float) if r else np.array([])
    xnm = float(np.interp(xpx, dpx, dnm)) if dnm.size and dpx.size == dnm.size else float(xpx)
    if pending is None:
        self._line_profile_pending_point = (float(xpx), float(y), setno)
        self._line_profile_measure_next_point = m2_id
        self._line_profile_measure_mode = True
        self.line_profile_measure_var.set(f'M{m1_id} selected on Set {setno} at {xnm:.6g} nm. Click M{m2_id} on the SAME profile (Set {setno}).')
    else:
        if int(pending[2]) != setno:
            self.line_profile_measure_var.set(f'M{m1_id} is on Set {pending[2]}. Click M{m2_id} on that same Set.')
            return
        pts = [pending, (float(xpx), float(y), setno)]
        self._line_profile_measurements_list.append({'pair_id': pair_no, 'm1_id': m1_id, 'm2_id': m2_id, 'setno': setno, 'pts': pts})
        self._line_profile_pending_point = None
        _tab6_sync_legacy_measurement_dict(self)
        self._line_profile_update_measurement_table()
        self._line_profile_draw_measurement()
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == setno), None)
        dnm = np.asarray(r.get('dist_nm', []), float) if r else np.array([])
        dpx = np.asarray(r.get('dist', []), float) if r else np.array([])
        delta_nm = abs(float(np.interp(pts[1][0], dpx, dnm) - np.interp(pts[0][0], dpx, dnm))) if dnm.size and dpx.size == dnm.size else abs(pts[1][0] - pts[0][0])
        self._line_profile_measure_mode = True
        self._line_profile_measure_next_point = m2_id + 1
        self.line_profile_measure_var.set(f'M{m1_id}–M{m2_id} complete: Set {setno}, ΔL = {delta_nm:.6g} nm. Now click the next pair on any profile.')

def _tab7_graph_motion_nm(self, event):
    pi = getattr(self, '_line_profile_measure_drag_pair', None)
    point_index = getattr(self, '_line_profile_measure_drag_point', None)
    if pi is None or point_index is None or event is None or (event.inaxes is not self.line_profile_ax_graph) or (event.xdata is None):
        return
    measurements = getattr(self, '_line_profile_measurements_list', [])
    if not 0 <= int(pi) < len(measurements):
        return
    m = measurements[int(pi)]
    r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == int(m['setno'])), None)
    if r is None:
        return
    xpx = _tab7_nm_to_profile_px(self, r, float(event.xdata))
    if xpx is None:
        return
    v = self._tab6_measurement_from_x(int(m['setno']), xpx)
    if v is None:
        return
    m['pts'][point_index] = v
    _tab6_sync_legacy_measurement_dict(self)
    self._line_profile_update_measurement_table()
    self._line_profile_draw_measurement()

def _tab7_graph_release_nm(self, event):
    pi = getattr(self, '_line_profile_measure_drag_pair', None)
    if pi is not None and 0 <= int(pi) < len(getattr(self, '_line_profile_measurements_list', [])):
        m = self._line_profile_measurements_list[int(pi)]
        setno = int(m['setno'])
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == setno), None)
        if r is not None:
            dnm = np.asarray(r.get('dist_nm', []), float)
            dpx = np.asarray(r.get('dist', []), float)
            delta = abs(np.interp(m['pts'][1][0], dpx, dnm) - np.interp(m['pts'][0][0], dpx, dnm)) if dnm.size and dpx.size == dnm.size else abs(m['pts'][1][0] - m['pts'][0][0])
            self.line_profile_measure_var.set(f"Pair M{m['m1_id']}–M{m['m2_id']} updated. ΔL = {delta:.6g} nm. Ready for the next measurement pair.")
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    if getattr(self, '_line_profile_measurements_list', None):
        self._line_profile_measure_mode = True

def _tab7_draw_measurement_nm(self):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    for m in getattr(self, '_line_profile_measurements_list', []):
        pts = m.get('pts', [])
        if len(pts) != 2:
            continue
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == int(m.get('setno', -1))), None)
        if r is None:
            continue
        dnm = np.asarray(r.get('dist_nm', []), float)
        dpx = np.asarray(r.get('dist', []), float)
        if dnm.size == 0 or dpx.size != dnm.size:
            continue
        x0n = float(np.interp(pts[0][0], dpx, dnm))
        x1n = float(np.interp(pts[1][0], dpx, dnm))
        y0, y1 = (float(pts[0][1]), float(pts[1][1]))
        line, = self.line_profile_ax_graph.plot([x0n, x1n], [y0, y1], '-', color='blue', lw=2.8, alpha=0.95, zorder=20)
        self._line_profile_measure_artists.append(line)
        for pid, xn, y in ((m['m1_id'], x0n, y0), (m['m2_id'], x1n, y1)):
            sc = self.line_profile_ax_graph.scatter([xn], [y], s=42, facecolor='blue', edgecolor='white', linewidths=1.2, zorder=22)
            self._line_profile_measure_artists.append(sc)
            txt = self.line_profile_ax_graph.annotate(f'M{pid}', (xn, y), xytext=(5, 7), textcoords='offset points', color='blue', fontsize=9, fontweight='bold', zorder=23, bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='blue', alpha=0.9))
            self._line_profile_measure_artists.append(txt)
        ymin, ymax = self.line_profile_ax_graph.get_ylim()
        ybar = ymax - 0.055 * max(ymax - ymin, 1e-09) - 0.035 * m['pair_id'] * max(ymax - ymin, 1e-09)
        arrow = self.line_profile_ax_graph.annotate('', xy=(x1n, ybar), xytext=(x0n, ybar), arrowprops=dict(arrowstyle='<->', color='blue', lw=1.8))
        self._line_profile_measure_artists.append(arrow)
        label = self.line_profile_ax_graph.text((x0n + x1n) / 2.0, ybar, f"M{m['m1_id']}–M{m['m2_id']}: ΔL={abs(x1n - x0n):.6g} nm", color='blue', fontsize=9, fontweight='bold', ha='center', va='bottom', bbox=dict(boxstyle='round,pad=0.16', facecolor='white', edgecolor='blue', alpha=0.88), zorder=24)
        self._line_profile_measure_artists.append(label)
    pending = getattr(self, '_line_profile_pending_point', None)
    if pending is not None:
        setno = int(pending[2])
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == setno), None)
        dnm = np.asarray(r.get('dist_nm', []), float) if r else np.array([])
        dpx = np.asarray(r.get('dist', []), float) if r else np.array([])
        xnm = float(np.interp(pending[0], dpx, dnm)) if dnm.size and dpx.size == dnm.size else float(pending[0])
        sc = self.line_profile_ax_graph.scatter([xnm], [pending[1]], s=44, facecolor='blue', edgecolor='white', linewidths=1.1, zorder=24)
        self._line_profile_measure_artists.append(sc)
        pid = getattr(self, '_line_profile_measure_next_point', 1) - 1
        txt = self.line_profile_ax_graph.annotate(f'M{pid}', (xnm, pending[1]), xytext=(5, 7), textcoords='offset points', color='blue', fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='blue', alpha=0.9), zorder=25)
        self._line_profile_measure_artists.append(txt)
    self._line_profile_redraw_image_lines()
    self.line_profile_canvas_graph.draw_idle()

def _tab7_update_measurement_table_nm(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    for item in tree.get_children():
        tree.delete(item)
    result_by_set = {int(r.get('set', -1)): r for r in getattr(self, '_line_profile_results', [])}
    for m in getattr(self, '_line_profile_measurements_list', []):
        pts = m.get('pts', [])
        r = result_by_set.get(int(m.get('setno', -1)))
        if r is None or len(pts) != 2:
            continue
        dnm = np.asarray(r.get('dist_nm', []), float)
        dpx = np.asarray(r.get('dist', []), float)
        if dnm.size == 0 or dpx.size != dnm.size:
            continue
        x0 = float(np.interp(pts[0][0], dpx, dnm))
        x1 = float(np.interp(pts[1][0], dpx, dnm))
        length_nm = float(r.get('length_nm', 0.0))
        vals = (f"M{m['m1_id']}–M{m['m2_id']}", m['setno'], f'{length_nm:.6g}', f'{x0:.6g}', f'{x1:.6g}', f'{abs(x1 - x0):.6g}', f'{pts[0][1]:.6g}', f'{pts[1][1]:.6g}', 'Measured')
        tree.insert('', 'end', values=vals)
    pending = getattr(self, '_line_profile_pending_point', None)
    if pending is not None:
        setno = int(pending[2])
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == setno), None)
        dnm = np.asarray(r.get('dist_nm', []), float) if r else np.array([])
        dpx = np.asarray(r.get('dist', []), float) if r else np.array([])
        xnm = float(np.interp(pending[0], dpx, dnm)) if dnm.size and dpx.size == dnm.size else float(pending[0])
        pid = getattr(self, '_line_profile_measure_next_point', 1) - 1
        tree.insert('', 'end', values=(f'M{pid}', setno, '—', f'{xnm:.6g}', '—', '—', f'{pending[1]:.6g}', '—', 'Pending'))

def _tab7_measurement_status_nm(self):
    pts = getattr(self, '_line_profile_measure_points', []) or []
    if pts:
        _x0, _y0, _sn = pts[0]
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == int(_sn)), None)
        if r is not None:
            dnm = np.asarray(r.get('dist_nm', []), float)
            dpx = np.asarray(r.get('dist', []), float)
            xnm = float(np.interp(_x0, dpx, dnm)) if dnm.size and dpx.size == dnm.size else float(_x0)
            if len(pts) == 1:
                self.line_profile_measure_var.set(f'M1: Set {int(_sn)}, position = {xnm:.6g} nm, intensity={_y0:.6g}')
            else:
                x1 = pts[1][0]
                x1nm = float(np.interp(x1, dpx, dnm)) if dnm.size and dpx.size == dnm.size else float(x1)
                self.line_profile_measure_var.set(f'Dynamic scale ΔL = {abs(x1nm - xnm):.6g} nm | M1={xnm:.6g} nm | M2={x1nm:.6g} nm')
        return
    pending = getattr(self, '_line_profile_pending_point', None)
    if pending is not None:
        setno = int(pending[2])
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == setno), None)
        dnm = np.asarray(r.get('dist_nm', []), float) if r else np.array([])
        dpx = np.asarray(r.get('dist', []), float) if r else np.array([])
        xnm = float(np.interp(pending[0], dpx, dnm)) if dnm.size and dpx.size == dnm.size else float(pending[0])
        self.line_profile_measure_var.set(f"M{getattr(self, '_line_profile_measure_next_point', 2) - 1}: position = {xnm:.6g} nm, intensity={pending[1]:.6g}")

def _tab7_export_profiles_nm(self):
    if not getattr(self, '_line_profile_results', []):
        messagebox.showwarning('Line Profile', 'Draw at least one line first.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Export line-profile graph data (nm)', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    try:
        idx = int(round(float(self.line_profile_frame_var.get())))
    except Exception:
        idx = 0
    rows = []
    for r in self._line_profile_results:
        for xn, y in zip(np.asarray(r.get('dist_nm', []), float), np.asarray(r.get('profile', []), float)):
            rows.append({'set': int(r['set']), 'frame': idx + 1, 'field_mT': float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else '', 'distance_nm': float(xn), 'intensity': float(y)})
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} profile points (nm): {fp}')

def _tab7_export_measurements_nm(self):
    if not getattr(self, '_line_profile_measurements_list', None):
        messagebox.showwarning('Line Profile', 'No dynamic measurements to export.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Export dynamic measurements (nm)', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    try:
        idx = int(round(float(self.line_profile_frame_var.get())))
    except Exception:
        idx = 0
    rows = []
    for m in self._line_profile_measurements_list:
        if len(m.get('pts', [])) != 2:
            continue
        setno = int(m['setno'])
        r = next((r for r in getattr(self, '_line_profile_results', []) if int(r.get('set', -1)) == setno), None)
        if r is None:
            continue
        dnm = np.asarray(r.get('dist_nm', []), float)
        dpx = np.asarray(r.get('dist', []), float)
        x0 = float(np.interp(m['pts'][0][0], dpx, dnm))
        x1 = float(np.interp(m['pts'][1][0], dpx, dnm))
        rows.append({'pair': f"M{m['m1_id']}-M{m['m2_id']}", 'set': setno, 'frame': idx + 1, 'field_mT': float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else '', 'line_length_nm': float(r.get('length_nm', 0.0)), 'M1_position_nm': x0, 'M2_position_nm': x1, 'DeltaL_nm': abs(x1 - x0), 'M1_intensity': float(m['pts'][0][1]), 'M2_intensity': float(m['pts'][1][1])})
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} dynamic measurement pair(s) (nm): {fp}')

def _tab7_image_press_nm(self, event):
    if event.inaxes is not self.line_profile_ax_image or event.xdata is None or event.ydata is None or (not getattr(self, '_line_profile_active', False)):
        return
    scales = _tab7_line_profile_scales_nm(self)
    sx, sy = scales if scales is not None else (1.0, 1.0)
    self._line_profile_press_xy = (float(event.xdata) / sx, float(event.ydata) / sy)
    if self._line_profile_temp_artist is not None:
        try:
            self._line_profile_temp_artist.remove()
        except Exception:
            pass
    self._line_profile_temp_artist, = self.line_profile_ax_image.plot([event.xdata, event.xdata], [event.ydata, event.ydata], '--', lw=1.8, color=self._line_profile_line_color(len(self._line_profile_lines)))
    self.line_profile_canvas_image.draw_idle()

def _tab7_image_motion_nm(self, event):
    if self._line_profile_press_xy is None or event.inaxes is not self.line_profile_ax_image or event.xdata is None or (event.ydata is None):
        return
    if self._line_profile_temp_artist is not None:
        self._line_profile_temp_artist.set_data([self._tab7_px_to_nm_point(self, self._line_profile_press_xy)[0], float(event.xdata)], [self._tab7_px_to_nm_point(self, self._line_profile_press_xy)[1], float(event.ydata)])
    self.line_profile_canvas_image.draw_idle()

def _tab7_image_release_nm(self, event):
    if self._line_profile_press_xy is None:
        return
    p0 = self._line_profile_press_xy
    self._line_profile_press_xy = None
    if self._line_profile_temp_artist is not None:
        try:
            self._line_profile_temp_artist.remove()
        except Exception:
            pass
        self._line_profile_temp_artist = None
    if event.inaxes is not self.line_profile_ax_image or event.xdata is None or event.ydata is None:
        self.line_profile_canvas_image.draw_idle()
        return
    scales = _tab7_line_profile_scales_nm(self)
    sx, sy = scales if scales is not None else (1.0, 1.0)
    p1 = (float(event.xdata) / sx, float(event.ydata) / sy)
    if np.hypot(p1[0] - p0[0], p1[1] - p0[1]) < 1.0:
        self.line_profile_status_var.set('Line too short. Draw a longer line.')
        return
    target = max(1, min(20, int(self.line_profile_sets_var.get())))
    if len(self._line_profile_lines) >= target:
        self._line_profile_active = False
        self.line_profile_status_var.set(f'All {target} requested line sets are already drawn.')
        return
    self._line_profile_lines.append({'p0': p0, 'p1': p1, 'set': len(self._line_profile_lines) + 1})
    self._line_profile_active = len(self._line_profile_lines) < target
    self._line_profile_redraw_profiles()
    p0nm = _tab7_px_to_nm_point(self, p0)
    p1nm = _tab7_px_to_nm_point(self, p1)
    self.line_profile_status_var.set(f'Line set {len(self._line_profile_lines)}/{target} added | length={float(np.hypot(p1nm[0] - p0nm[0], p1nm[1] - p0nm[1])):.6g} nm')
_TAB7_BUILDER_BEFORE_NM = CDIWorkflowApp._build_line_profile_tab
CDIWorkflowApp._line_profile_update_image = _tab7_update_image_nm
CDIWorkflowApp._line_profile_redraw_profiles = _tab7_redraw_profiles_nm
CDIWorkflowApp._line_profile_redraw_image_lines = _tab7_redraw_image_lines_nm
CDIWorkflowApp._line_profile_image_press = _tab7_image_press_nm
CDIWorkflowApp._line_profile_image_motion = _tab7_image_motion_nm
CDIWorkflowApp._line_profile_image_release = _tab7_image_release_nm
CDIWorkflowApp._line_profile_project_to_curve = _tab7_project_to_curve_nm
CDIWorkflowApp._line_profile_graph_press = _tab7_graph_press_nm
CDIWorkflowApp._line_profile_graph_motion = _tab7_graph_motion_nm
CDIWorkflowApp._line_profile_graph_release = _tab7_graph_release_nm
CDIWorkflowApp._line_profile_draw_measurement = _tab7_draw_measurement_nm
CDIWorkflowApp._line_profile_update_table = _tab7_update_measurement_table_nm
CDIWorkflowApp._line_profile_update_measurement_table = _tab7_update_measurement_table_nm
CDIWorkflowApp._line_profile_export_profiles = _tab7_export_profiles_nm
CDIWorkflowApp._line_profile_export_results = _tab7_export_measurements_nm
_OLD_TAB7_LOAD_SECONDARY = CDIWorkflowApp.load_line_from_tab6

def _tab7_load_line_from_tab6_nm(self, *args, **kwargs):
    ok = _OLD_TAB7_LOAD_SECONDARY(self, *args, **kwargs)
    try:
        if not ok:
            return ok
        c = getattr(self, 'line_source_coords', None)
        scales = _tab7_line_profile_scales_nm(self)
        if c is not None and scales is not None:
            sx, sy = scales
            x0 = float(c['xmin']) * sx
            x1 = float(c['xmax'] + 1) * sx
            y0 = float(c['ymin']) * sy
            y1 = float(c['ymax'] + 1) * sy
            self.line_source_status_var.set(f'✓ Source = Tab 6 confirmed Secondary ROI | X={x0:.6g}–{x1:.6g} nm, Y={y0:.6g}–{y1:.6g} nm | crop physical size={x1 - x0:.6g} × {y1 - y0:.6g} nm | frames={len(self.line_source_stack)}')
        return ok
    except Exception:
        return ok
CDIWorkflowApp.load_line_from_tab6 = _tab7_load_line_from_tab6_nm
_OLD_TAB7_FRAME_SYNC = CDIWorkflowApp._line_profile_sync_from_secondary_roi

def _tab7_sync_from_secondary_roi_nm(self):
    result = _OLD_TAB7_FRAME_SYNC(self)
    try:
        if getattr(self, 'line_profile_image', None) is not None:
            self._line_profile_update_image(idx=int(self.line_profile_frame_var.get()), preserve_lines=True)
    except Exception:
        pass
    return result
CDIWorkflowApp._line_profile_sync_from_secondary_roi = _tab7_sync_from_secondary_roi_nm
_OLD_APPLY_RES = CDIWorkflowApp.apply_roi_resolution_from_gui

def _tab6_apply_resolution_physical(self, *args, **kwargs):
    result = _OLD_APPLY_RES(self, *args, **kwargs)
    try:
        self.update_processing_pixel_size()
        if getattr(self, 'current_processed', None) is not None:
            self._proc_update_physical_axes_and_colorbar(self.current_processed)
        self._proc_secondary_status_nm = _tab6_secondary_status_nm_physical(self)
    except Exception:
        pass
    return result
CDIWorkflowApp.apply_roi_resolution_from_gui = _tab6_apply_resolution_physical

def _frame_count(self):
    try:
        return max(0, int(len(self.recons)))
    except Exception:
        return 0

def _clamp_frame(self, idx):
    n = self._frame_count()
    if n <= 0:
        return 0
    try:
        idx = int(round(float(idx)))
    except Exception:
        idx = 0
    return max(0, min(idx, n - 1))

def _update_frame_ui(self, idx):
    """Update frame variables/labels only; do not trigger cross-tab redraws."""
    idx = self._clamp_frame(idx)
    n = self._frame_count()
    self.current_index = idx
    for name in ('proc_idx_var', 'roi2_idx_var', 'line_profile_frame_var', 'pd_frame_var'):
        var = getattr(self, name, None)
        if var is not None:
            try:
                var.set(idx)
            except Exception:
                pass
    for name in ('proc_slider', 'line_profile_frame_slider', 'pd_frame_slider'):
        sl = getattr(self, name, None)
        if sl is not None:
            try:
                sl.set(idx)
            except Exception:
                try:
                    sl.configure(value=idx)
                except Exception:
                    pass
    spin = getattr(self, '_tab4_roi_index_spinbox', None)
    if spin is not None:
        try:
            spin.configure(from_=0, to=max(0, n - 1), state='normal')
            spin.set(idx)
        except Exception:
            pass
    field = None
    try:
        if self.real_fields_mT is not None and len(self.real_fields_mT) > idx:
            field = float(self.real_fields_mT[idx])
    except Exception:
        field = None
    if hasattr(self, 'proc_field_var'):
        try:
            self.proc_field_var.set(f'Frame {idx + 1}/{n} | Field = {field:+.2f} mT' if field is not None else f'Frame {idx + 1}/{n}')
        except Exception:
            pass
    if hasattr(self, 'line_profile_frame_label'):
        try:
            self.line_profile_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT' if field is not None else f'{idx + 1}/{n}')
        except Exception:
            pass
    if hasattr(self, 'pd_frame_label'):
        try:
            self.pd_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT' if field is not None else f'{idx + 1}/{n}')
        except Exception:
            pass
    return idx
CDIWorkflowApp._frame_count = _frame_count
CDIWorkflowApp._clamp_frame = _clamp_frame
CDIWorkflowApp._update_frame_ui = _update_frame_ui
_OLD_TAB4_BUILD_FRAME_SYNC = CDIWorkflowApp._build_roi_tab

def _tab4_build_roi_frame_sync(self):
    _OLD_TAB4_BUILD_FRAME_SYNC(self)

    def walk(widget):
        for child in widget.winfo_children():
            yield child
            yield from walk(child)
    spin = None
    try:
        for w in walk(self.tab_roi):
            cls = w.winfo_class()
            if str(cls).lower().endswith('spinbox'):
                spin = w
                break
    except Exception:
        spin = None
    if spin is not None:
        self._tab4_roi_index_spinbox = spin
        try:
            spin.configure(state='normal', from_=0, to=0)
        except Exception:
            pass

        def tab4_step(*_):
            try:
                idx = int(round(float(self.roi_index_var.get())))
            except Exception:
                idx = 0
            idx = self._clamp_frame(idx)
            self.roi_index_var.set(idx)
            self.refresh_roi_image()
            return 'break'
        try:
            spin.configure(command=tab4_step)
            spin.bind('<Up>', tab4_step, add='+')
            spin.bind('<Down>', tab4_step, add='+')
            spin.bind('<Return>', tab4_step, add='+')
        except Exception:
            pass
    self._tab4_sync_roi_index_range()
CDIWorkflowApp._build_roi_tab = _tab4_build_roi_frame_sync

def _tab4_sync_roi_index_range(self):
    n = self._frame_count()
    if n <= 0:
        return
    spin = getattr(self, '_tab4_roi_index_spinbox', None)
    idx = self._clamp_frame(getattr(self, 'roi_index_var', tk.IntVar(value=0)).get())
    try:
        self.roi_index_var.set(idx)
    except Exception:
        pass
    if spin is not None:
        try:
            spin.configure(from_=0, to=n - 1, state='normal')
            spin.set(idx)
        except Exception:
            pass
CDIWorkflowApp._tab4_sync_roi_index_range = _tab4_sync_roi_index_range
for _method_name in ('load_reconstructions', 'load_all_reconstructions', 'full_load', 'test_single_hdf5'):
    _old = getattr(CDIWorkflowApp, _method_name, None)
    if _old is None:
        continue

    def _make_loader_wrapper(old_method):

        def _wrapped(self, *args, **kwargs):
            result = old_method(self, *args, **kwargs)
            try:
                self._tab4_sync_roi_index_range()
            except Exception:
                pass
            return result
        return _wrapped
    setattr(CDIWorkflowApp, _method_name, _make_loader_wrapper(_old))
_OLD_TAB4_REFRESH_FRAME_SYNC = CDIWorkflowApp.refresh_roi_image

def _tab4_refresh_frame_sync(self, *args, **kwargs):
    try:
        self._tab4_sync_roi_index_range()
    except Exception:
        pass
    return _OLD_TAB4_REFRESH_FRAME_SYNC(self, *args, **kwargs)
CDIWorkflowApp.refresh_roi_image = _tab4_refresh_frame_sync

def _tab6_proc_slider_frame_sync(self, value=None):
    if getattr(self, '_frame_sync_guard', False):
        return
    self._frame_sync_guard = True
    try:
        try:
            idx = int(round(float(value))) if value is not None else int(round(float(self.proc_idx_var.get())))
        except Exception:
            idx = 0
        idx = self._clamp_frame(idx)
        self.proc_idx_var.set(idx)
        self._update_frame_ui(idx)
        self.update_processing_view()
        try:
            if getattr(self, 'line_profile_image', None) is not None:
                self.line_profile_frame_var.set(idx)
                self._line_profile_update_image(idx=idx, preserve_lines=True)
        except Exception:
            pass
        try:
            if getattr(self, 'roi2_stack', None) is not None and len(self.roi2_stack) > idx:
                self.roi2_idx_var.set(idx)
        except Exception:
            pass
        try:
            if getattr(self, 'pd_frame_var', None) is not None:
                self.pd_frame_var.set(idx)
                if getattr(self, 'roi2_stack', None) is not None and len(self.roi2_stack):
                    self._pd7_refresh_from_confirmed_roi(preserve_zoom=True)
        except Exception:
            pass
    finally:
        self._frame_sync_guard = False
CDIWorkflowApp._proc_slider_update = _tab6_proc_slider_frame_sync

def _bind_tab6_frame_slider_sync(self):
    sl = getattr(self, 'proc_slider', None)
    if sl is None:
        return
    try:
        sl.configure(command=lambda value: self._proc_slider_update(value))
        sl.configure(from_=0, to=max(0, self._frame_count() - 1), resolution=1)
        sl.set(self._clamp_frame(self.proc_idx_var.get()))
    except Exception:
        pass
CDIWorkflowApp._bind_tab6_frame_slider_sync = _bind_tab6_frame_slider_sync

def _tab7_line_profile_frame_changed_sync(self, value=None):
    if getattr(self, '_frame_sync_guard', False):
        return
    self._frame_sync_guard = True
    try:
        try:
            idx = int(round(float(value))) if value is not None else int(round(float(self.line_profile_frame_var.get())))
        except Exception:
            idx = 0
        idx = self._clamp_frame(idx)
        self.line_profile_frame_var.set(idx)
        self._update_frame_ui(idx)
        self._line_profile_update_image(idx=idx, preserve_lines=True)
        try:
            self.proc_idx_var.set(idx)
            if getattr(self, 'proc_slider', None) is not None:
                self.proc_slider.set(idx)
        except Exception:
            pass
        try:
            self.roi2_idx_var.set(idx)
        except Exception:
            pass
    finally:
        self._frame_sync_guard = False
CDIWorkflowApp._line_profile_frame_slider_changed = _tab7_line_profile_frame_changed_sync

def _tab7_particle_frame_changed_sync(self, value=None):
    if getattr(self, '_frame_sync_guard', False):
        return
    self._frame_sync_guard = True
    try:
        try:
            idx = int(round(float(value))) if value is not None else int(round(float(self.pd_frame_var.get())))
        except Exception:
            idx = 0
        stack = getattr(self, 'roi2_stack', None)
        if stack is not None and len(stack):
            idx = max(0, min(idx, len(stack) - 1))
        else:
            idx = self._clamp_frame(idx)
        self.pd_frame_var.set(idx)
        self._update_frame_ui(idx)
        try:
            self._pd7_refresh_from_confirmed_roi(preserve_zoom=True)
        except Exception:
            pass
        try:
            self.proc_idx_var.set(idx)
            if getattr(self, 'proc_slider', None) is not None:
                self.proc_slider.set(idx)
            self.roi2_idx_var.set(idx)
        except Exception:
            pass
    finally:
        self._frame_sync_guard = False
CDIWorkflowApp._tab7_slider_changed = _tab7_particle_frame_changed_sync
_OLD_BUILD_UI_FRAME_SYNC = CDIWorkflowApp._build_ui

def _build_ui_frame_sync(self):
    _OLD_BUILD_UI_FRAME_SYNC(self)
    self._frame_sync_guard = False
    try:
        self._tab4_sync_roi_index_range()
    except Exception:
        pass
    try:
        self._bind_tab6_frame_slider_sync()
    except Exception:
        pass
    try:
        sl = getattr(self, 'line_profile_frame_slider', None)
        if sl is not None:
            sl.configure(from_=0, to=max(0, self._frame_count() - 1), resolution=1)
            sl.configure(command=lambda value: self._line_profile_frame_slider_changed(value))
            sl.set(self._clamp_frame(getattr(self, 'proc_idx_var', tk.IntVar(value=0)).get()))
    except Exception:
        pass
    try:
        sl = getattr(self, 'pd_frame_slider', None)
        if sl is not None:
            sl.configure(from_=0, to=max(0, self._frame_count() - 1), resolution=1)
    except Exception:
        pass
CDIWorkflowApp._build_ui = _build_ui_frame_sync

def _tab7_v14_init_measurement_state(self):
    if not hasattr(self, '_line_profile_measurements_list'):
        self._line_profile_measurements_list = []
    if not hasattr(self, '_line_profile_pending_point'):
        self._line_profile_pending_point = None
    if not hasattr(self, '_line_profile_measure_next_point'):
        self._line_profile_measure_next_point = 1
    if not hasattr(self, '_line_profile_measure_drag_pair'):
        self._line_profile_measure_drag_pair = None
    if not hasattr(self, '_line_profile_measure_drag_point'):
        self._line_profile_measure_drag_point = None
    if not hasattr(self, '_line_profile_measure_mode'):
        self._line_profile_measure_mode = False
    if not hasattr(self, '_line_profile_measure_artists'):
        self._line_profile_measure_artists = []
    if not hasattr(self, '_line_profile_measure_image_artists'):
        self._line_profile_measure_image_artists = []

def _tab7_v14_result(self, setno):
    for r in getattr(self, '_line_profile_results', []) or []:
        try:
            if int(r.get('set', -1)) == int(setno):
                return r
        except Exception:
            pass
    return None

def _tab7_v14_xpx_to_nm(self, result, xpx):
    if result is None:
        return float(xpx)
    dpx = np.asarray(result.get('dist', []), dtype=float)
    dnm = np.asarray(result.get('dist_nm', []), dtype=float)
    if dpx.size >= 2 and dpx.size == dnm.size:
        return float(np.interp(float(np.clip(xpx, dpx[0], dpx[-1])), dpx, dnm))
    return float(xpx)

def _tab7_v14_measurement_from_xpx(self, setno, xpx):
    r = _tab7_v14_result(self, setno)
    if r is None:
        return None
    dpx = np.asarray(r.get('dist', []), dtype=float)
    prof = np.asarray(r.get('profile', []), dtype=float)
    if dpx.size == 0 or prof.size != dpx.size:
        return None
    xx = float(np.clip(xpx, dpx[0], dpx[-1]))
    yy = float(np.interp(xx, dpx, prof))
    return (xx, yy, int(setno))

def _tab7_v14_project_to_curve(self, x_nm, y, set_hint=None, max_screen_px=85.0):
    """Project a mouse click onto a plotted profile segment in screen space.

    Returns (screen_distance_px, x_pixel_along_profile, y_intensity, setno).
    The x-position is continuously interpolated between sampled profile points.
    """
    results = list(getattr(self, '_line_profile_results', []) or [])
    if set_hint is not None:
        hinted = [r for r in results if int(r.get('set', -1)) == int(set_hint)]
        if hinted:
            results = hinted
    if x_nm is None or y is None or (not results):
        return None
    ax = self.line_profile_ax_graph
    target = ax.transData.transform((float(x_nm), float(y)))
    best = None
    for r in results:
        dnm = np.asarray(r.get('dist_nm', []), dtype=float)
        dpx = np.asarray(r.get('dist', []), dtype=float)
        prof = np.asarray(r.get('profile', []), dtype=float)
        if dnm.size < 2 or dpx.size != dnm.size or prof.size != dnm.size:
            continue
        screen = ax.transData.transform(np.column_stack((dnm, prof)))
        a = screen[:-1]
        b = screen[1:]
        v = b - a
        vv = np.einsum('ij,ij->i', v, v)
        w = target - a
        with np.errstate(divide='ignore', invalid='ignore'):
            t = np.einsum('ij,ij->i', w, v) / np.where(vv > 0, vv, 1.0)
        t = np.clip(t, 0.0, 1.0)
        q = a + v * t[:, None]
        d2 = np.einsum('ij,ij->i', q - target, q - target)
        j = int(np.argmin(d2))
        dscreen = float(np.sqrt(max(d2[j], 0.0)))
        if best is None or dscreen < best[0]:
            dnm0, dnm1 = (float(dnm[j]), float(dnm[j + 1]))
            y0, y1 = (float(prof[j]), float(prof[j + 1]))
            tt = float(t[j])
            x_nm_hit = dnm0 + tt * (dnm1 - dnm0)
            y_hit = y0 + tt * (y1 - y0)
            x_px_hit = float(np.interp(x_nm_hit, dnm, dpx))
            best = (dscreen, x_px_hit, y_hit, int(r.get('set', 1)))
    if best is None or best[0] > float(max_screen_px):
        return None
    return best

def _tab7_v14_start_measurement(self):
    _tab7_v14_init_measurement_state(self)
    if not getattr(self, '_line_profile_results', None):
        self.line_profile_measure_var.set('Draw at least one line profile first.')
        return
    self._line_profile_measure_mode = True
    self._line_profile_pending_point = None
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    next_id = 2 * len(self._line_profile_measurements_list) + 1
    self._line_profile_measure_next_point = next_id
    try:
        self.line_profile_canvas_graph.get_tk_widget().configure(cursor='crosshair')
    except Exception:
        pass
    self.line_profile_measure_var.set(f'DYNAMIC MEASURING SCALE — click M{next_id} and M{next_id + 1} on the SAME profile. Endpoint positions are continuous/sub-pixel. After a pair is complete, the next pair may be selected on any profile.')

def _tab7_v14_select_pair_point(self, event):
    _tab7_v14_init_measurement_state(self)
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.xdata is None or event.ydata is None:
        return
    best = _tab7_v14_project_to_curve(self, event.xdata, event.ydata, max_screen_px=90.0)
    if best is None:
        self.line_profile_measure_var.set('No profile found near the cursor. Click directly on or close to a profile.')
        return
    _screen_d, xpx, y, setno = best
    setno = int(setno)
    pending = getattr(self, '_line_profile_pending_point', None)
    pair_no = len(self._line_profile_measurements_list) + 1
    m1_id = 2 * pair_no - 1
    m2_id = 2 * pair_no
    xnm = _tab7_v14_xpx_to_nm(self, _tab7_v14_result(self, setno), xpx)
    if pending is None:
        self._line_profile_pending_point = (float(xpx), float(y), setno)
        self._line_profile_measure_next_point = m2_id
        self.line_profile_measure_var.set(f'M{m1_id} selected on Set {setno} at {xnm:.6g} nm. Click M{m2_id} on the SAME profile.')
        self._line_profile_draw_measurement()
        return
    if int(pending[2]) != setno:
        first_no = m1_id
        self.line_profile_measure_var.set(f'M{first_no} is on Set {pending[2]}. Click M{m2_id} on that same profile.')
        return
    pts = [pending, (float(xpx), float(y), setno)]
    self._line_profile_measurements_list.append({'pair_id': pair_no, 'm1_id': m1_id, 'm2_id': m2_id, 'setno': setno, 'pts': pts})
    self._line_profile_pending_point = None
    try:
        _tab6_sync_legacy_measurement_dict(self)
    except Exception:
        pass
    self._line_profile_update_measurement_table()
    self._line_profile_draw_measurement()
    r = _tab7_v14_result(self, setno)
    x0nm = _tab7_v14_xpx_to_nm(self, r, pts[0][0])
    x1nm = _tab7_v14_xpx_to_nm(self, r, pts[1][0])
    next_pair = pair_no + 1
    next_m1 = 2 * next_pair - 1
    next_m2 = 2 * next_pair
    total = len(getattr(self, '_line_profile_results', []) or [])
    self._line_profile_measure_mode = True
    self._line_profile_measure_next_point = next_m1
    self.line_profile_measure_var.set(f'M{m1_id}–M{m2_id} complete: Set {setno}, ΔL = {abs(x1nm - x0nm):.6g} nm. Next pair: M{next_m1}–M{next_m2} on any of {total} profile(s).')

def _tab7_v14_graph_press(self, event):
    _tab7_v14_init_measurement_state(self)
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.button not in (None, 1):
        return
    best = None
    for pi, m in enumerate(getattr(self, '_line_profile_measurements_list', []) or []):
        setno = int(m.get('setno', 1))
        r = _tab7_v14_result(self, setno)
        if r is None:
            continue
        for point_index, (px, py, _sn) in enumerate(m.get('pts', [])[:2]):
            xnm = _tab7_v14_xpx_to_nm(self, r, px)
            try:
                sx, sy = self.line_profile_ax_graph.transData.transform((xnm, float(py)))
                d = float(np.hypot(event.x - sx, event.y - sy))
            except Exception:
                continue
            if best is None or d < best[0]:
                best = (d, pi, point_index)
    if best is not None and best[0] <= 28.0:
        self._line_profile_measure_drag_pair = best[1]
        self._line_profile_measure_drag_point = best[2]
        self._line_profile_measure_mode = False
        m = self._line_profile_measurements_list[best[1]]
        mid = m['m1_id'] if best[2] == 0 else m['m2_id']
        self.line_profile_measure_var.set(f"Dragging M{mid} of pair M{m['m1_id']}–M{m['m2_id']}. Move along the profile; release to keep the new position.")
        return
    if getattr(self, '_line_profile_measure_mode', False):
        _tab7_v14_select_pair_point(self, event)

def _tab7_v14_graph_motion(self, event):
    pi = getattr(self, '_line_profile_measure_drag_pair', None)
    point_index = getattr(self, '_line_profile_measure_drag_point', None)
    if pi is None or point_index is None:
        return
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.xdata is None or event.ydata is None:
        return
    measurements = getattr(self, '_line_profile_measurements_list', [])
    if not 0 <= int(pi) < len(measurements):
        return
    m = measurements[int(pi)]
    setno = int(m['setno'])
    r = _tab7_v14_result(self, setno)
    if r is None:
        return
    xpx = _tab7_nm_to_profile_px(self, r, float(event.xdata))
    if xpx is None:
        return
    v = _tab7_v14_measurement_from_xpx(self, setno, xpx)
    if v is None:
        return
    m['pts'][point_index] = v
    try:
        _tab6_sync_legacy_measurement_dict(self)
    except Exception:
        pass
    self._line_profile_update_measurement_table()
    self._line_profile_draw_measurement()

def _tab7_v14_graph_release(self, event):
    pi = getattr(self, '_line_profile_measure_drag_pair', None)
    measurements = getattr(self, '_line_profile_measurements_list', [])
    if pi is not None and 0 <= int(pi) < len(measurements):
        m = measurements[int(pi)]
        r = _tab7_v14_result(self, int(m['setno']))
        if r is not None:
            x0nm = _tab7_v14_xpx_to_nm(self, r, m['pts'][0][0])
            x1nm = _tab7_v14_xpx_to_nm(self, r, m['pts'][1][0])
            self.line_profile_measure_var.set(f"Pair M{m['m1_id']}–M{m['m2_id']} updated. ΔL = {abs(x1nm - x0nm):.6g} nm.")
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    if measurements:
        self._line_profile_measure_mode = True

def _tab7_v14_refresh_measurements(self):
    """Keep measurement x positions fixed in profile coordinates and update y."""
    _tab7_v14_init_measurement_state(self)
    refreshed = []
    for m in getattr(self, '_line_profile_measurements_list', []) or []:
        new_pts = []
        for pt in m.get('pts', [])[:2]:
            v = _tab7_v14_measurement_from_xpx(self, int(m['setno']), float(pt[0]))
            if v is not None:
                new_pts.append(v)
        if len(new_pts) == 2:
            mm = dict(m)
            mm['pts'] = new_pts
            refreshed.append(mm)
    self._line_profile_measurements_list = refreshed
    try:
        _tab6_sync_legacy_measurement_dict(self)
    except Exception:
        pass

def _tab7_v14_redraw_image_measurements(self):
    """Redraw normal profile lines plus thick yellow measurement segments."""
    for a in getattr(self, '_line_profile_image_line_artists', []) or []:
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_image_line_artists = []
    for a in getattr(self, '_line_profile_measure_image_artists', []) or []:
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_image_artists = []
    for i, line in enumerate(getattr(self, '_line_profile_lines', []) or []):
        p0n = _tab7_px_to_nm_point(self, line['p0'])
        p1n = _tab7_px_to_nm_point(self, line['p1'])
        c = self._line_profile_line_color(i)
        a, = self.line_profile_ax_image.plot([p0n[0], p1n[0]], [p0n[1], p1n[1]], '-', lw=2.2, color=c, zorder=12)
        self._line_profile_image_line_artists.append(a)
    for m in getattr(self, '_line_profile_measurements_list', []) or []:
        pts = m.get('pts', [])
        if len(pts) != 2:
            continue
        r = _tab7_v14_result(self, int(m['setno']))
        if r is None:
            continue
        Lpx = float(r.get('length', 0.0))
        if Lpx <= 0:
            continue
        p0 = np.asarray(r['p0'], dtype=float)
        p1 = np.asarray(r['p1'], dtype=float)
        xy = []
        for xpx, _yy, _sn in pts:
            frac = float(np.clip(float(xpx) / Lpx, 0.0, 1.0))
            xy.append(p0 + frac * (p1 - p0))
        a_px, b_px = xy
        a_nm = _tab7_px_to_nm_point(self, a_px)
        b_nm = _tab7_px_to_nm_point(self, b_px)
        shadow, = self.line_profile_ax_image.plot([a_nm[0], b_nm[0]], [a_nm[1], b_nm[1]], '-', lw=7.0, color='black', alpha=0.78, zorder=25)
        yellow, = self.line_profile_ax_image.plot([a_nm[0], b_nm[0]], [a_nm[1], b_nm[1]], '-', lw=4.0, color='yellow', zorder=26)
        self._line_profile_measure_image_artists.extend([shadow, yellow])
        for q in (a_nm, b_nm):
            marker, = self.line_profile_ax_image.plot([q[0]], [q[1]], 'o', ms=6.5, mfc='yellow', mec='black', mew=1.0, zorder=27)
            self._line_profile_measure_image_artists.append(marker)
    pending = getattr(self, '_line_profile_pending_point', None)
    if pending is not None:
        q = _tab7_px_to_nm_point(self, np.asarray([pending[0], pending[1]], dtype=float))
        marker, = self.line_profile_ax_image.plot([q[0]], [q[1]], 'o', ms=6.5, mfc='yellow', mec='black', mew=1.0, zorder=27)
        self._line_profile_measure_image_artists.append(marker)
    self.line_profile_canvas_image.draw_idle()

def _tab7_v14_draw_measurement(self):
    for a in getattr(self, '_line_profile_measure_artists', []) or []:
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    for m in getattr(self, '_line_profile_measurements_list', []) or []:
        pts = m.get('pts', [])
        if len(pts) != 2:
            continue
        setno = int(m['setno'])
        r = _tab7_v14_result(self, setno)
        if r is None:
            continue
        x0nm = _tab7_v14_xpx_to_nm(self, r, pts[0][0])
        x1nm = _tab7_v14_xpx_to_nm(self, r, pts[1][0])
        y0 = float(pts[0][1])
        y1 = float(pts[1][1])
        line, = self.line_profile_ax_graph.plot([x0nm, x1nm], [y0, y1], '-', color='blue', lw=2.8, alpha=0.95, zorder=20)
        self._line_profile_measure_artists.append(line)
        for pid, xn, yy in ((m['m1_id'], x0nm, y0), (m['m2_id'], x1nm, y1)):
            sc = self.line_profile_ax_graph.scatter([xn], [yy], s=44, facecolor='blue', edgecolor='white', linewidths=1.2, zorder=22)
            self._line_profile_measure_artists.append(sc)
            txt = self.line_profile_ax_graph.annotate(f'M{pid}', (xn, yy), xytext=(5, 7), textcoords='offset points', color='blue', fontsize=9, fontweight='bold', zorder=23, bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='blue', alpha=0.9))
            self._line_profile_measure_artists.append(txt)
        ymin, ymax = self.line_profile_ax_graph.get_ylim()
        yspan = max(ymax - ymin, 1e-09)
        ybar = ymax - 0.055 * yspan - 0.035 * int(m['pair_id']) * yspan
        arrow, = self.line_profile_ax_graph.plot([x0nm, x1nm], [ybar, ybar], color='blue', lw=1.8, alpha=0.95, zorder=21)
        self._line_profile_measure_artists.append(arrow)
        a = self.line_profile_ax_graph.annotate('', xy=(x1nm, ybar), xytext=(x0nm, ybar), arrowprops=dict(arrowstyle='<->', color='blue', lw=1.8))
        self._line_profile_measure_artists.append(a)
        label = self.line_profile_ax_graph.text((x0nm + x1nm) / 2.0, ybar, f"M{m['m1_id']}–M{m['m2_id']}: ΔL={abs(x1nm - x0nm):.6g} nm", color='blue', fontsize=9, fontweight='bold', ha='center', va='bottom', bbox=dict(boxstyle='round,pad=0.16', facecolor='white', edgecolor='blue', alpha=0.88), zorder=24)
        self._line_profile_measure_artists.append(label)
    pending = getattr(self, '_line_profile_pending_point', None)
    if pending is not None:
        setno = int(pending[2])
        r = _tab7_v14_result(self, setno)
        xnm = _tab7_v14_xpx_to_nm(self, r, pending[0])
        sc = self.line_profile_ax_graph.scatter([xnm], [pending[1]], s=46, facecolor='blue', edgecolor='white', linewidths=1.1, zorder=24)
        self._line_profile_measure_artists.append(sc)
        pid = getattr(self, '_line_profile_measure_next_point', 2) - 1
        txt = self.line_profile_ax_graph.annotate(f'M{pid}', (xnm, pending[1]), xytext=(5, 7), textcoords='offset points', color='blue', fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='blue', alpha=0.9), zorder=25)
        self._line_profile_measure_artists.append(txt)
    _tab7_v14_redraw_image_measurements(self)
    if hasattr(self, 'line_profile_canvas_graph'):
        self.line_profile_canvas_graph.draw_idle()

def _tab7_v14_update_measurement_table(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    for item in tree.get_children():
        tree.delete(item)
    result_by_set = {int(r.get('set', -1)): r for r in getattr(self, '_line_profile_results', []) or []}
    for m in getattr(self, '_line_profile_measurements_list', []) or []:
        pts = m.get('pts', [])
        r = result_by_set.get(int(m.get('setno', -1)))
        if r is None:
            continue
        length_nm = float(r.get('length_nm', 0.0))
        if len(pts) == 2:
            x0nm = _tab7_v14_xpx_to_nm(self, r, pts[0][0])
            x1nm = _tab7_v14_xpx_to_nm(self, r, pts[1][0])
            vals = (f"M{m['m1_id']}–M{m['m2_id']}", int(m['setno']), f'{length_nm:.6g}', f'{x0nm:.6g}', f'{x1nm:.6g}', f'{abs(x1nm - x0nm):.6g}', f'{pts[0][1]:.6g}', f'{pts[1][1]:.6g}', 'Measured')
            tree.insert('', 'end', values=vals)
    pending = getattr(self, '_line_profile_pending_point', None)
    if pending is not None:
        setno = int(pending[2])
        r = result_by_set.get(setno)
        xnm = _tab7_v14_xpx_to_nm(self, r, pending[0])
        pid = getattr(self, '_line_profile_measure_next_point', 2) - 1
        tree.insert('', 'end', values=(f'M{pid}', setno, '—', f'{xnm:.6g}', '—', '—', f'{pending[1]:.6g}', '—', 'Pending'))

def _tab7_v14_frame_redraw_wrapper(self, idx=None):
    result = _TAB7_V14_BASE_REDRAW_PROFILES(self, idx=idx)
    try:
        _tab7_v14_refresh_measurements(self)
        _tab7_v14_update_measurement_table(self)
        _tab7_v14_draw_measurement(self)
    except Exception:
        pass
    return result
CDIWorkflowApp._tab7_v14_init_measurement_state = _tab7_v14_init_measurement_state
CDIWorkflowApp._tab7_v14_result = _tab7_v14_result
CDIWorkflowApp._tab7_v14_measurement_from_xpx = _tab7_v14_measurement_from_xpx
CDIWorkflowApp._tab7_v14_project_to_curve = _tab7_v14_project_to_curve
CDIWorkflowApp._tab7_v14_xpx_to_nm = _tab7_v14_xpx_to_nm
_TAB7_V14_BASE_REDRAW_PROFILES = CDIWorkflowApp._line_profile_redraw_profiles
CDIWorkflowApp._line_profile_start_measurement = _tab7_v14_start_measurement
CDIWorkflowApp._line_profile_graph_press = _tab7_v14_graph_press
CDIWorkflowApp._line_profile_graph_motion = _tab7_v14_graph_motion
CDIWorkflowApp._line_profile_graph_release = _tab7_v14_graph_release
CDIWorkflowApp._line_profile_refresh_measurement_values = _tab7_v14_refresh_measurements
CDIWorkflowApp._line_profile_redraw_image_lines = _tab7_v14_redraw_image_measurements
CDIWorkflowApp._line_profile_draw_measurement = _tab7_v14_draw_measurement
CDIWorkflowApp._line_profile_update_measurement_table = _tab7_v14_update_measurement_table
CDIWorkflowApp._line_profile_update_table = _tab7_v14_update_measurement_table
_TAB7_V14_OLD_FRAME_REDRAW = CDIWorkflowApp._line_profile_redraw_profiles

def _tab7_v14_redraw_profiles_final(self, idx=None):
    result = _TAB7_V14_OLD_FRAME_REDRAW(self, idx=idx)
    try:
        _tab7_v14_refresh_measurements(self)
        _tab7_v14_update_measurement_table(self)
        _tab7_v14_draw_measurement(self)
    except Exception:
        pass
    return result
CDIWorkflowApp._line_profile_redraw_profiles = _tab7_v14_redraw_profiles_final
_TAB7_V14_BASE_BUILD_UI = CDIWorkflowApp._build_ui

def _build_ui_tab7_v14_final(self):
    _TAB7_V14_BASE_BUILD_UI(self)
    _tab7_v14_init_measurement_state(self)
    try:
        self.line_profile_ax_image.set_xlabel('X (nm)')
        self.line_profile_ax_image.set_ylabel('Y (nm)')
        self.line_profile_ax_graph.set_xlabel('Distance (nm)')
    except Exception:
        pass
    try:
        self._line_profile_update_image(idx=int(self.line_profile_frame_var.get()), preserve_lines=True)
    except Exception:
        pass
CDIWorkflowApp._build_ui = _build_ui_tab7_v14_final

def _ui_frame_to_internal(self, value):
    n = self._frame_count()
    if n <= 0:
        return 0
    try:
        display_no = int(round(float(value)))
    except Exception:
        display_no = 1
    return max(0, min(display_no - 1, n - 1))

def _internal_to_ui_frame(self, idx):
    return self._clamp_frame(idx) + 1
CDIWorkflowApp._ui_frame_to_internal = _ui_frame_to_internal
CDIWorkflowApp._internal_to_ui_frame = _internal_to_ui_frame
_OLD_TAB4_BUILD_ONE_BASED = CDIWorkflowApp._build_roi_tab

def _tab4_build_one_based(self):
    _OLD_TAB4_BUILD_ONE_BASED(self)
    self._tab4_display_var = tk.StringVar(value='1')

    def walk(widget):
        for child in widget.winfo_children():
            yield child
            yield from walk(child)
    spin = None
    try:
        for w in walk(self.tab_roi):
            if str(w.winfo_class()).lower().endswith('spinbox'):
                spin = w
                break
    except Exception:
        spin = None
    self._tab4_roi_index_spinbox = spin
    if spin is None:
        return
    try:
        spin.configure(textvariable=self._tab4_display_var, state='normal', from_=1, to=1, increment=1, wrap=False)
    except Exception:
        pass

    def apply_display_frame(*_):
        idx = self._ui_frame_to_internal(self._tab4_display_var.get())
        try:
            self.roi_index_var.set(idx)
        except Exception:
            pass
        self._tab4_display_var.set(str(idx + 1))
        try:
            self._tab4_sync_roi_index_range()
        except Exception:
            pass
        try:
            _OLD_TAB4_REFRESH_FRAME_SYNC(self)
        except Exception:
            pass
        return 'break'
    try:
        spin.configure(command=apply_display_frame)
        spin.bind('<Up>', apply_display_frame, add='+')
        spin.bind('<Down>', apply_display_frame, add='+')
        spin.bind('<Return>', apply_display_frame, add='+')
    except Exception:
        pass
    self._tab4_sync_roi_index_range()
CDIWorkflowApp._build_roi_tab = _tab4_build_one_based

def _tab4_sync_roi_index_range_one_based(self):
    n = self._frame_count()
    if n <= 0:
        return
    spin = getattr(self, '_tab4_roi_index_spinbox', None)
    idx = self._clamp_frame(getattr(self, 'roi_index_var', tk.IntVar(value=0)).get())
    try:
        self.roi_index_var.set(idx)
    except Exception:
        pass
    try:
        self._tab4_display_var.set(str(idx + 1))
    except Exception:
        pass
    if spin is not None:
        try:
            spin.configure(from_=1, to=n, increment=1, state='normal')
            spin.set(idx + 1)
        except Exception:
            pass
CDIWorkflowApp._tab4_sync_roi_index_range = _tab4_sync_roi_index_range_one_based

def _bind_tab6_frame_slider_one_based(self):
    sl = getattr(self, 'proc_slider', None)
    if sl is None:
        return
    if not hasattr(self, '_proc_display_frame_var'):
        self._proc_display_frame_var = tk.DoubleVar(value=1)
    try:
        sl.configure(variable=self._proc_display_frame_var, from_=1, to=max(1, self._frame_count()), resolution=1, command=lambda value: self._proc_slider_update_one_based(value))
        sl.set(self._internal_to_ui_frame(getattr(self, 'proc_idx_var', tk.IntVar(value=0)).get()))
    except Exception:
        pass

def _proc_slider_update_one_based(self, value=None):
    if getattr(self, '_frame_sync_guard', False):
        return
    self._frame_sync_guard = True
    try:
        idx = self._ui_frame_to_internal(value if value is not None else getattr(self, '_proc_display_frame_var', tk.DoubleVar(value=1)).get())
        self.proc_idx_var.set(idx)
        try:
            self._proc_display_frame_var.set(idx + 1)
        except Exception:
            pass
        self._update_frame_ui(idx)
        self.update_processing_view()
        try:
            self.line_profile_frame_var.set(idx)
            if getattr(self, 'line_profile_image', None) is not None:
                self._line_profile_update_image(idx=idx, preserve_lines=True)
        except Exception:
            pass
        try:
            self.roi2_idx_var.set(idx)
        except Exception:
            pass
        try:
            self.pd_frame_var.set(idx)
            if getattr(self, 'roi2_stack', None) is not None and len(self.roi2_stack):
                self._pd7_refresh_from_confirmed_roi(preserve_zoom=True)
        except Exception:
            pass
    finally:
        self._frame_sync_guard = False
CDIWorkflowApp._bind_tab6_frame_slider_sync = _bind_tab6_frame_slider_one_based
CDIWorkflowApp._proc_slider_update = _proc_slider_update_one_based

def _line_profile_slider_update_one_based(self, value=None):
    if getattr(self, '_frame_sync_guard', False):
        return
    self._frame_sync_guard = True
    try:
        idx = self._ui_frame_to_internal(value if value is not None else getattr(self, '_line_profile_display_frame_var', tk.DoubleVar(value=1)).get())
        self.line_profile_frame_var.set(idx)
        if hasattr(self, '_line_profile_display_frame_var'):
            self._line_profile_display_frame_var.set(idx + 1)
        self._update_frame_ui(idx)
        try:
            self._line_profile_update_image(idx=idx, preserve_lines=True)
        except Exception:
            pass
        try:
            self.proc_idx_var.set(idx)
            if getattr(self, 'proc_slider', None) is not None and hasattr(self, '_proc_display_frame_var'):
                self._proc_display_frame_var.set(idx + 1)
                self.proc_slider.set(idx + 1)
        except Exception:
            pass
        try:
            self.roi2_idx_var.set(idx)
        except Exception:
            pass
    finally:
        self._frame_sync_guard = False
CDIWorkflowApp._line_profile_frame_slider_changed = _line_profile_slider_update_one_based

def _particle_slider_update_one_based(self, value=None):
    if getattr(self, '_frame_sync_guard', False):
        return
    self._frame_sync_guard = True
    try:
        idx = self._ui_frame_to_internal(value if value is not None else getattr(self, '_pd_display_frame_var', tk.DoubleVar(value=1)).get())
        stack = getattr(self, 'roi2_stack', None)
        n = len(stack) if stack is not None and len(stack) else self._frame_count()
        idx = max(0, min(idx, max(0, n - 1)))
        self.pd_frame_var.set(idx)
        if hasattr(self, '_pd_display_frame_var'):
            self._pd_display_frame_var.set(idx + 1)
        self._update_frame_ui(idx)
        try:
            self._pd7_refresh_from_confirmed_roi(preserve_zoom=True)
        except Exception:
            pass
        try:
            self.proc_idx_var.set(idx)
            if getattr(self, 'proc_slider', None) is not None and hasattr(self, '_proc_display_frame_var'):
                self._proc_display_frame_var.set(idx + 1)
                self.proc_slider.set(idx + 1)
        except Exception:
            pass
        try:
            self.line_profile_frame_var.set(idx)
        except Exception:
            pass
    finally:
        self._frame_sync_guard = False
CDIWorkflowApp._particle_density_frame_slider_changed = _particle_slider_update_one_based

def _update_frame_ui_one_based(self, idx):
    idx = self._clamp_frame(idx)
    n = self._frame_count()
    self.current_index = idx
    for name in ('proc_idx_var', 'roi2_idx_var', 'line_profile_frame_var', 'pd_frame_var'):
        var = getattr(self, name, None)
        if var is not None:
            try:
                var.set(idx)
            except Exception:
                pass
    try:
        if hasattr(self, '_proc_display_frame_var'):
            self._proc_display_frame_var.set(idx + 1)
        if getattr(self, 'proc_slider', None) is not None:
            self.proc_slider.set(idx + 1)
    except Exception:
        pass
    try:
        if hasattr(self, '_line_profile_display_frame_var'):
            self._line_profile_display_frame_var.set(idx + 1)
        if getattr(self, 'line_profile_frame_slider', None) is not None:
            self.line_profile_frame_slider.set(idx + 1)
    except Exception:
        pass
    try:
        if hasattr(self, '_pd_display_frame_var'):
            self._pd_display_frame_var.set(idx + 1)
        if getattr(self, 'pd_frame_slider', None) is not None:
            self.pd_frame_slider.set(idx + 1)
    except Exception:
        pass
    try:
        if hasattr(self, '_tab4_display_var'):
            self._tab4_display_var.set(str(idx + 1))
        if getattr(self, '_tab4_roi_index_spinbox', None) is not None:
            self._tab4_roi_index_spinbox.set(idx + 1)
    except Exception:
        pass
    field = None
    try:
        if self.real_fields_mT is not None and len(self.real_fields_mT) > idx:
            field = float(self.real_fields_mT[idx])
    except Exception:
        field = None
    frame_text = f'Frame {idx + 1}/{n}'
    if field is not None:
        frame_text += f' | Field = {field:+.2f} mT'
    for attr in ('proc_field_var',):
        var = getattr(self, attr, None)
        if var is not None:
            try:
                var.set(frame_text)
            except Exception:
                pass
    for attr in ('line_profile_frame_label', 'pd_frame_label'):
        w = getattr(self, attr, None)
        if w is not None:
            try:
                w.config(text=f'{idx + 1}/{n} | {field:+.2f} mT' if field is not None else f'{idx + 1}/{n}')
            except Exception:
                pass
    return idx
CDIWorkflowApp._update_frame_ui = _update_frame_ui_one_based


def _set_frame_slider_midpoint_once(self, slider, n, var=None, zero_based=True, token=None):
    """Initialize a frame slider to the middle frame once per widget."""
    try:
        n=int(n)
    except Exception:
        return
    if slider is None or n<=0:
        return
    key=str(token or slider).replace('-','_').replace('.','_').replace('/','_')
    flag="_frame_midpoint_initialized_"+key
    if getattr(self,flag,False):
        return
    mid=(n-1)//2 if zero_based else (n+1)//2
    try:
        if var is not None:
            var.set(mid)
        slider.set(mid)
        setattr(self,flag,True)
    except Exception:
        pass


def _sync_all_visible_frame_ranges(self):
    n = self._frame_count()
    if n <= 0:
        return
    idx = self._clamp_frame(getattr(self, 'current_index', 0))
    try:
        self._tab4_sync_roi_index_range()
    except Exception:
        pass
    try:
        if getattr(self, 'proc_slider', None) is not None:
            if not hasattr(self, '_proc_display_frame_var'):
                self._proc_display_frame_var = tk.DoubleVar(value=idx + 1)
            self.proc_slider.configure(from_=1, to=n, resolution=1, variable=self._proc_display_frame_var, command=lambda v: self._proc_slider_update(v))
            self._proc_display_frame_var.set(idx + 1)
            self.proc_slider.set(idx + 1)
            _set_frame_slider_midpoint_once(self,self.proc_slider,n,self._proc_display_frame_var,zero_based=False,token='proc')
    except Exception:
        pass
    try:
        if getattr(self, 'line_profile_frame_slider', None) is not None:
            if not hasattr(self, '_line_profile_display_frame_var'):
                self._line_profile_display_frame_var = tk.DoubleVar(value=idx + 1)
            self.line_profile_frame_slider.configure(from_=1, to=n, resolution=1, variable=self._line_profile_display_frame_var, command=lambda v: self._line_profile_frame_slider_changed(v))
            self._line_profile_display_frame_var.set(idx + 1)
            self.line_profile_frame_slider.set(idx + 1)
            _set_frame_slider_midpoint_once(self,self.line_profile_frame_slider,n,self._line_profile_display_frame_var,zero_based=False,token='line_profile')
    except Exception:
        pass
    try:
        if getattr(self, 'pd_frame_slider', None) is not None:
            if not hasattr(self, '_pd_display_frame_var'):
                self._pd_display_frame_var = tk.DoubleVar(value=idx + 1)
            self.pd_frame_slider.configure(from_=1, to=n, resolution=1, variable=self._pd_display_frame_var, command=lambda v: self._particle_density_frame_slider_changed(v))
            self._pd_display_frame_var.set(idx + 1)
            self.pd_frame_slider.set(idx + 1)
            _set_frame_slider_midpoint_once(self,self.pd_frame_slider,n,self._pd_display_frame_var,zero_based=False,token='pd_common')
    except Exception:
        pass
    self._update_frame_ui(idx)
CDIWorkflowApp._sync_all_visible_frame_ranges = _sync_all_visible_frame_ranges
_OLD_BUILD_UI_FRAME_FINAL = CDIWorkflowApp._build_ui

def _build_ui_one_based_frames(self):
    _OLD_BUILD_UI_FRAME_FINAL(self)
    self._frame_sync_guard = False
    try:
        self._sync_all_visible_frame_ranges()
    except Exception:
        pass
CDIWorkflowApp._build_ui = _build_ui_one_based_frames

def _confirm_secondary_roi_from_processing_robust(self):
    if getattr(self, 'roi2_coords', None) is None:
        messagebox.showwarning('Secondary ROI', 'First click SELECT SECONDARY ROI and drag a rectangle on the processed image.', parent=self.root)
        return False
    if self.recons is None:
        messagebox.showwarning('Secondary ROI', 'Load reconstructions first.', parent=self.root)
        return False
    try:
        self._roi2_stop_playback(update_status=False)
    except Exception:
        pass
    idx = self._clamp_frame(getattr(self, 'proc_idx_var', tk.IntVar(value=0)).get())
    self.proc_idx_var.set(idx)
    try:
        self.roi2_idx_var.set(idx)
    except Exception:
        pass
    try:
        reference = np.asarray(self._roi2_source_image(idx))
        h, w = reference.shape[:2]
        x0 = int(self.roi2_coords['xmin'])
        x1 = int(self.roi2_coords['xmax'])
        y0 = int(self.roi2_coords['ymin'])
        y1 = int(self.roi2_coords['ymax'])
        x0, x1 = (max(0, min(x0, w - 1)), max(0, min(x1, w - 1)))
        y0, y1 = (max(0, min(y0, h - 1)), max(0, min(y1, h - 1)))
        if x1 <= x0 or y1 <= y0:
            raise ValueError('Selected Secondary ROI has zero width or height.')
        mask = np.zeros((h, w), dtype=bool)
        mask[y0:y1 + 1, x0:x1 + 1] = True
        lm = getattr(self, 'local_mask', None)
        if lm is not None and np.asarray(lm).shape == mask.shape:
            mask &= np.asarray(lm, dtype=bool)
        if not np.any(mask):
            raise ValueError('Selected Secondary ROI contains no valid processed pixels.')
        self.roi2_mask = mask
        self.roi2_confirmed_coords = {'xmin': x0, 'xmax': x1, 'ymin': y0, 'ymax': y1}
        self.roi2_coords = dict(self.roi2_confirmed_coords)
        stack = np.zeros((len(self.recons), y1 - y0 + 1, x1 - x0 + 1), dtype=np.float32)
        local_mask = mask[y0:y1 + 1, x0:x1 + 1]
        for i in range(len(self.recons)):
            img = np.asarray(self._roi2_source_image(i), dtype=np.float32)
            crop = np.asarray(img[y0:y1 + 1, x0:x1 + 1], dtype=np.float32).copy()
            crop[~local_mask] = 0.0
            stack[i] = crop
        self.roi2_stack = stack
        try:
            self.roi2_idx_var.set(idx)
        except Exception:
            pass
        self.proc_secondary_status_var.set(f'✓ SECONDARY ROI CONFIRMED. X={x0}:{x1}, Y={y0}:{y1} | {x1 - x0 + 1} × {y1 - y0 + 1} px | Frame {idx + 1}/{len(self.recons)}')
        self.log(f'Secondary ROI confirmed: X={x0}:{x1}, Y={y0}:{y1}, size={x1 - x0 + 1}x{y1 - y0 + 1}, frames={len(self.recons)}')
        try:
            self._proc_draw_secondary_roi_patch(confirmed=True)
        except Exception:
            pass
        try:
            if getattr(self, 'roi2_ax', None) is not None and getattr(self, 'roi2_canvas', None) is not None:
                self.update_roi2_view()
        except Exception:
            pass
        try:
            self.line_profile_frame_var.set(idx)
            if getattr(self, 'line_profile_image', None) is not None:
                self._line_profile_update_image(idx=idx, preserve_lines=True)
        except Exception:
            pass
        try:
            self._pd7_refresh_from_confirmed_roi(preserve_zoom=True)
        except Exception:
            pass
        return True
    except Exception as exc:
        messagebox.showerror('Secondary ROI', f'Failed to confirm secondary ROI:\n{type(exc).__name__}: {exc}', parent=self.root)
        self.log(f'Secondary ROI confirmation failed: {type(exc).__name__}: {exc}')
        return False
CDIWorkflowApp.confirm_secondary_roi_from_processing = _confirm_secondary_roi_from_processing_robust

def _rebind_secondary_confirm_button(self):
    btn = getattr(self, 'proc_secondary_confirm_button', None)
    if btn is not None:
        try:
            btn.configure(command=self.confirm_secondary_roi_from_processing)
        except Exception:
            pass
CDIWorkflowApp._rebind_secondary_confirm_button = _rebind_secondary_confirm_button
for _loader_name in ('load_reconstructions', 'load_all_reconstructions', 'full_load', 'test_single_hdf5'):
    _loader_old = getattr(CDIWorkflowApp, _loader_name, None)
    if _loader_old is None:
        continue

    def _make_one_based_loader(old_method):

        def _wrapped(self, *args, **kwargs):
            result = old_method(self, *args, **kwargs)
            try:
                self._sync_all_visible_frame_ranges()
            except Exception:
                pass
            return result
        return _wrapped
    setattr(CDIWorkflowApp, _loader_name, _make_one_based_loader(_loader_old))
for _res_name in ('apply_roi_resolution_from_gui', 'build_roi_stack'):
    _res_old = getattr(CDIWorkflowApp, _res_name, None)
    if _res_old is None:
        continue

    def _make_one_based_resolution(old_method):

        def _wrapped(self, *args, **kwargs):
            result = old_method(self, *args, **kwargs)
            try:
                self._sync_all_visible_frame_ranges()
            except Exception:
                pass
            return result
        return _wrapped
    setattr(CDIWorkflowApp, _res_name, _make_one_based_resolution(_res_old))

def _final_disconnect_canvas_event(canvas, name):
    try:
        callbacks = list(canvas.callbacks.callbacks.get(name, {}).keys())
        for cid in callbacks:
            try:
                canvas.mpl_disconnect(cid)
            except Exception:
                pass
    except Exception:
        pass

def _final_tab5_rebind_wheel_zoom(self):
    """Keep exactly one wheel-zoom callback for each Tab-5 canvas."""
    for which, canvas in (('sem', getattr(self, 'pixel_cal_sem_canvas', None)), ('roi', getattr(self, 'pixel_cal_roi_canvas', None))):
        if canvas is None:
            continue
        _final_disconnect_canvas_event(canvas, 'scroll_event')
        canvas.mpl_connect('scroll_event', lambda e, w=which: self._pixel_cal_zoom_scroll(w, e))
        ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')

def _final_tab6_ensure_colorbar(self):
    """Ensure the Tab-6 Primary ROI always has one visible 0–1 intensity colorbar."""
    fig = getattr(self, '_proc_fig', None)
    ax = getattr(self, '_proc_ax', None)
    im = getattr(self, '_proc_image_display', None)
    if fig is None or ax is None or im is None:
        return
    try:
        im.set_clim(0.0, 1.0)
    except Exception:
        pass
    try:
        cbar = getattr(self, '_proc_colorbar', None)
        if cbar is None:
            fig.subplots_adjust(left=0.1, right=0.84, bottom=0.1, top=0.91)
            cax = fig.add_axes([0.86, 0.12, 0.035, 0.74])
            cbar = fig.colorbar(im, cax=cax)
            cbar.set_label('Intensity')
            self._proc_colorbar = cbar
            self._proc_colorbar_mappable = im
        else:
            cbar.update_normal(im)
            cbar.set_label('Intensity')
            try:
                cbar.ax.set_position([0.86, 0.12, 0.035, 0.74])
            except Exception:
                pass
    except Exception as exc:
        try:
            self.log(f'Tab 6 colorbar refresh warning: {exc}')
        except Exception:
            pass

def _final_tab6_after_update(self, display):
    """Run the authoritative physical-axis update, then make the figure visible."""
    result = _FINAL_TAB6_UPDATE_PHYSICAL(self, display)
    try:
        self._proc_ax.set_axis_on()
        self._proc_ax.set_xlabel('X (nm)')
        self._proc_ax.set_ylabel('Y (nm)')
        self._proc_ax.set_aspect('equal', adjustable='box')
        self._proc_ax.set_anchor('C')
    except Exception:
        pass
    _final_tab6_ensure_colorbar(self)
    try:
        self._proc_canvas.draw_idle()
    except Exception:
        pass
    return result
_FINAL_TAB6_UPDATE_PHYSICAL = CDIWorkflowApp._proc_update_physical_axes_and_colorbar
CDIWorkflowApp._proc_update_physical_axes_and_colorbar = _final_tab6_after_update

def _final_tab7_graph_press_nm(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if getattr(event, 'button', 1) != 1:
        return
    if event.x is None or event.y is None:
        return
    measurements = list(getattr(self, '_line_profile_measurements_list', []) or [])
    if not measurements:
        if getattr(self, '_line_profile_measure_mode', False):
            try:
                _tab7_select_pair_point_nm(self, event)
            except Exception:
                pass
        return
    best = None
    scales = _tab7_line_profile_scales_nm(self)
    for pair_index, m in enumerate(measurements):
        setno = int(m.get('setno', 1))
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == setno), None)
        if r is None:
            continue
        dpx = np.asarray(r.get('dist', []), dtype=float)
        dnm = np.asarray(r.get('dist_nm', []), dtype=float)
        if dpx.size < 2 or dnm.size != dpx.size:
            continue
        for point_index, pt in enumerate(m.get('pts', [])[:2]):
            xpx = float(pt[0])
            yn = float(pt[1])
            xnm = float(np.interp(xpx, dpx, dnm))
            sx, sy = self.line_profile_ax_graph.transData.transform((xnm, yn))
            dd = float(np.hypot(float(event.x) - sx, float(event.y) - sy))
            if best is None or dd < best[0]:
                best = (dd, pair_index, point_index, xnm, yn)
    if best is not None and best[0] <= 42.0:
        _, pair_index, point_index, _, _ = best
        self._line_profile_measure_drag_pair = int(pair_index)
        self._line_profile_measure_drag_point = int(point_index)
        self._line_profile_measure_mode = False
        m = measurements[pair_index]
        self.line_profile_measure_var.set(f"Dragging M{(m['m1_id'] if point_index == 0 else m['m2_id'])}. Move horizontally along the profile; release to fix the new position.")
        return
    if getattr(self, '_line_profile_measure_mode', False):
        try:
            _tab7_select_pair_point_nm(self, event)
        except Exception:
            pass

def _final_tab7_graph_motion_nm(self, event):
    pair_index = getattr(self, '_line_profile_measure_drag_pair', None)
    point_index = getattr(self, '_line_profile_measure_drag_point', None)
    if pair_index is None or point_index is None:
        return
    if event is None or event.inaxes is not self.line_profile_ax_graph or event.xdata is None:
        return
    measurements = getattr(self, '_line_profile_measurements_list', []) or []
    if not 0 <= int(pair_index) < len(measurements):
        return
    m = measurements[int(pair_index)]
    setno = int(m.get('setno', 1))
    r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == setno), None)
    if r is None:
        return
    xpx = _tab7_nm_to_profile_px(self, r, float(event.xdata))
    if xpx is None:
        return
    v = self._tab6_measurement_from_x(setno, float(xpx))
    if v is None:
        return
    m['pts'][int(point_index)] = v
    try:
        _tab6_sync_legacy_measurement_dict(self)
    except Exception:
        pass
    try:
        self._line_profile_update_measurement_table()
        self._line_profile_draw_measurement()
    except Exception:
        pass

def _final_tab7_graph_release_nm(self, event):
    pair_index = getattr(self, '_line_profile_measure_drag_pair', None)
    if pair_index is not None:
        measurements = getattr(self, '_line_profile_measurements_list', []) or []
        if 0 <= int(pair_index) < len(measurements):
            m = measurements[int(pair_index)]
            setno = int(m.get('setno', 1))
            r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == setno), None)
            if r is not None:
                dpx = np.asarray(r.get('dist', []), dtype=float)
                dnm = np.asarray(r.get('dist_nm', []), dtype=float)
                if dpx.size >= 2 and dnm.size == dpx.size:
                    x0 = float(np.interp(float(m['pts'][0][0]), dpx, dnm))
                    x1 = float(np.interp(float(m['pts'][1][0]), dpx, dnm))
                    delta_nm = abs(x1 - x0)
                    self.line_profile_measure_var.set(f"Pair M{m['m1_id']}–M{m['m2_id']} fixed. ΔL = {delta_nm:.6g} nm.")
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    if getattr(self, '_line_profile_measurements_list', None):
        self._line_profile_measure_mode = True
CDIWorkflowApp._line_profile_graph_press = _final_tab7_graph_press_nm
CDIWorkflowApp._line_profile_graph_motion = _final_tab7_graph_motion_nm
CDIWorkflowApp._line_profile_graph_release = _final_tab7_graph_release_nm
_OLD_FINAL_BUILD_UI = CDIWorkflowApp._build_ui

def _build_ui_final_visual_fixes(self):
    _OLD_FINAL_BUILD_UI(self)
    try:
        _final_tab5_rebind_wheel_zoom(self)
    except Exception as exc:
        self.log(f'Tab 5 wheel-zoom hookup warning: {exc}')
    try:
        canvas = getattr(self, '_proc_canvas', None)
        if canvas is not None:
            _final_disconnect_canvas_event(canvas, 'scroll_event')
            canvas.mpl_connect('scroll_event', self._proc_zoom_scroll)
    except Exception as exc:
        self.log(f'Tab 6 wheel-zoom hookup warning: {exc}')
    try:
        _final_tab6_ensure_colorbar(self)
        if getattr(self, '_proc_ax', None) is not None:
            self._proc_ax.set_axis_on()
            self._proc_ax.set_xlabel('X (nm)')
            self._proc_ax.set_ylabel('Y (nm)')
            self._proc_ax.set_aspect('equal', adjustable='box')
            self._proc_ax.set_anchor('C')
    except Exception as exc:
        self.log(f'Tab 6 visual setup warning: {exc}')
    try:
        canvas = getattr(self, 'line_profile_canvas_graph', None)
        if canvas is not None:
            _final_disconnect_canvas_event(canvas, 'button_press_event')
            _final_disconnect_canvas_event(canvas, 'motion_notify_event')
            _final_disconnect_canvas_event(canvas, 'button_release_event')
            canvas.mpl_connect('button_press_event', self._line_profile_graph_press)
            canvas.mpl_connect('motion_notify_event', self._line_profile_graph_motion)
            canvas.mpl_connect('button_release_event', self._line_profile_graph_release)
    except Exception as exc:
        self.log(f'Tab 7 M1/M2 event hookup warning: {exc}')
CDIWorkflowApp._build_ui = _build_ui_final_visual_fixes

def _tab5_roi_display_array_final(self):
    """Return exactly the ROI array currently displayed in Tab 5."""
    try:
        cached = getattr(self, '_pixel_cal_roi_display_image', None)
        if cached is not None:
            a = np.asarray(cached, dtype=float)
            if a.ndim == 2 and a.size:
                return a
    except Exception:
        pass
    try:
        ax = getattr(self, 'pixel_cal_roi_ax', None)
        if ax is not None and getattr(ax, 'images', None):
            a = np.asarray(ax.images[-1].get_array(), dtype=float)
            if a.ndim == 2 and a.size:
                return a
    except Exception:
        pass
    return None
_TAB5_OLD_REFRESH_ROI_FINAL = CDIWorkflowApp._pixel_cal_refresh_primary_roi

def _pixel_cal_refresh_primary_roi_final(self):
    result = _TAB5_OLD_REFRESH_ROI_FINAL(self)
    ax = getattr(self, 'pixel_cal_roi_ax', None)
    fig = getattr(self, 'pixel_cal_roi_fig', None)
    canvas = getattr(self, 'pixel_cal_roi_canvas', None)
    if ax is None or fig is None or canvas is None:
        return result
    try:
        arr = _tab5_roi_display_array_final(self)
        if arr is None:
            return result
        h, w = arr.shape[:2]
        self._pixel_cal_roi_display_image = arr.copy()
        self._pixel_cal_zoom_limits['roi'] = None
        ax.set_autoscale_on(False)
        ax.set_xlim(-0.5, float(w) - 0.5)
        ax.set_ylim(float(h) - 0.5, -0.5)
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
        ax.set_axis_on()
        ax.set_xlabel('ROI X pixel')
        ax.set_ylabel('ROI Y pixel')
        fig.subplots_adjust(left=0.13, right=0.98, bottom=0.11, top=0.9)
        canvas.draw_idle()
    except Exception as exc:
        try:
            self.log(f'Tab 5 ROI fit warning: {exc}')
        except Exception:
            pass
    return result
CDIWorkflowApp._pixel_cal_refresh_primary_roi = _pixel_cal_refresh_primary_roi_final

def _pixel_cal_get_display_image_final(self, which):
    if which == 'sem':
        image=getattr(self,'sem_image',None)
    elif which == 'roi':
        image=getattr(self,'_pixel_cal_roi_display_image',None)
        if image is None:
            try:
                ax=getattr(self,'pixel_cal_roi_ax',None)
                images=getattr(ax,'images',[]) if ax is not None else []
                image=images[-1].get_array() if images else None
            except Exception:
                image=None
    else:
        return None
    try:
        arr=np.asarray(image)
        return arr if arr.ndim==2 and arr.size else None
    except Exception:
        return None


CDIWorkflowApp._pixel_cal_get_display_image = _pixel_cal_get_display_image_final

def _pixel_cal_zoom_scroll_final_robust(self, which, event):
    """Single reliable cursor-centred wheel zoom for Tab-5 SEM/ROI panes."""
    if event is None:
        return
    ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
    canvas = self.pixel_cal_sem_canvas if which == 'sem' else self.pixel_cal_roi_canvas
    if getattr(event, 'inaxes', None) is not ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    image = _pixel_cal_get_display_image_final(self, which)
    if image is None:
        return
    image = np.asarray(image)
    if image.ndim < 2:
        return
    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        b = getattr(event, 'button', None)
        if b in ('up', 4):
            step = 1.0
        elif b in ('down', 5):
            step = -1.0
    if step == 0:
        return
    h, w = image.shape[:2]
    fx0, fx1 = (-0.5, float(w) - 0.5)
    fy0, fy1 = (-0.5, float(h) - 0.5)
    x0, x1 = [float(v) for v in ax.get_xlim()]
    y0, y1 = [float(v) for v in ax.get_ylim()]
    sx = x1 - x0
    sy = y1 - y0
    if abs(sx) < 1e-12 or abs(sy) < 1e-12:
        return
    factor = 1.2 ** (-step)
    cx, cy = (float(event.xdata), float(event.ydata))
    rx = (cx - x0) / sx
    ry = (cy - y0) / sy
    rx = float(np.clip(rx, 0.0, 1.0))
    ry = float(np.clip(ry, 0.0, 1.0))
    nsx = sx * factor
    nsy = sy * factor
    nx0 = cx - rx * nsx
    nx1 = nx0 + nsx
    ny0 = cy - ry * nsy
    ny1 = ny0 + nsy
    if nx0 < fx0:
        d = fx0 - nx0
        nx0 += d
        nx1 += d
    if nx1 > fx1:
        d = nx1 - fx1
        nx0 -= d
        nx1 -= d
    ytop = min(ny0, ny1)
    ybot = max(ny0, ny1)
    if ytop < fy0:
        d = fy0 - ytop
        ny0 += d
        ny1 += d
    if ybot > fy1:
        d = ybot - fy1
        ny0 -= d
        ny1 -= d
    if abs(nx1 - nx0) >= 0.999999 * (fx1 - fx0) and abs(ny1 - ny0) >= 0.999999 * (fy1 - fy0):
        ax.set_xlim(fx0, fx1)
        ax.set_ylim(fy1, fy0)
        self._pixel_cal_zoom_limits[which] = None
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
        canvas.draw_idle()
        return
    if nx0 > nx1:
        nx0, nx1 = (nx1, nx0)
    if ny0 < ny1:
        ydir = -1
    else:
        ydir = 1
    nx0 = max(fx0, nx0)
    nx1 = min(fx1, nx1)
    ylo = max(fy0, min(ny0, ny1))
    yhi = min(fy1, max(ny0, ny1))
    if ydir < 0:
        ylimits = (yhi, ylo)
    else:
        ylimits = (ylo, yhi)
    xlimits = (nx0, nx1)
    self._pixel_cal_zoom_limits[which] = (xlimits, ylimits)
    ax.set_autoscale_on(False)
    ax.set_xlim(*xlimits)
    ax.set_ylim(*ylimits)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    canvas.draw_idle()
CDIWorkflowApp._pixel_cal_zoom_scroll = _pixel_cal_zoom_scroll_final_robust

def _pixel_cal_reset_zoom_final(self, which):
    image = _pixel_cal_get_display_image_final(self, which)
    if image is None:
        return
    h, w = np.asarray(image).shape[:2]
    ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
    canvas = self.pixel_cal_sem_canvas if which == 'sem' else self.pixel_cal_roi_canvas
    self._pixel_cal_zoom_limits[which] = None
    ax.set_autoscale_on(False)
    ax.set_xlim(-0.5, float(w) - 0.5)
    ax.set_ylim(float(h) - 0.5, -0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    canvas.draw_idle()
CDIWorkflowApp._pixel_cal_reset_zoom = _pixel_cal_reset_zoom_final
_TAB5_BUILD_BEFORE_FINAL_ROI_ZOOM = CDIWorkflowApp._build_ui

def _build_ui_final_tab5_roi_zoom(self):
    _TAB5_BUILD_BEFORE_FINAL_ROI_ZOOM(self)
    for which, canvas in (('sem', getattr(self, 'pixel_cal_sem_canvas', None)), ('roi', getattr(self, 'pixel_cal_roi_canvas', None))):
        if canvas is None:
            continue
        _final_disconnect_canvas_event(canvas, 'scroll_event')
        canvas.mpl_connect('scroll_event', lambda event, w=which: self._pixel_cal_zoom_scroll(w, event))
        ax = self.pixel_cal_sem_ax if which == 'sem' else self.pixel_cal_roi_ax
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
CDIWorkflowApp._build_ui = _build_ui_final_tab5_roi_zoom

def _tab7_line_profile_full_nm_extent(self):
    scales = _tab7_line_profile_scales_nm(self)
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        return None
    try:
        idx = int(self.line_profile_frame_var.get())
        idx = max(0, min(idx, len(stack) - 1))
    except Exception:
        idx = 0
    h, w = np.asarray(stack[idx]).shape[:2]
    if scales is None:
        sx = sy = 1.0
    else:
        sx, sy = [float(v) for v in scales]
    return (0.0, float(w) * sx, 0.0, float(h) * sy)

def _tab7_line_profile_zoom_nm_final(self, event):
    ax = getattr(self, 'line_profile_ax_image', None)
    canvas = getattr(self, 'line_profile_canvas_image', None)
    if event is None or ax is None or canvas is None:
        return
    if getattr(event, 'inaxes', None) is not ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    full = _tab7_line_profile_full_nm_extent(self)
    if full is None:
        return
    fx0, fx1, fy0, fy1 = full
    x0, x1 = [float(v) for v in ax.get_xlim()]
    y0, y1 = [float(v) for v in ax.get_ylim()]
    sx_view = x1 - x0
    sy_view = y1 - y0
    if abs(sx_view) < 1e-12 or abs(sy_view) < 1e-12:
        return
    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        b = getattr(event, 'button', None)
        if b in ('up', 4):
            step = 1.0
        elif b in ('down', 5):
            step = -1.0
    if step == 0:
        return
    factor = 1.2 ** (-step)
    cx, cy = (float(event.xdata), float(event.ydata))
    xmin, xmax = (min(x0, x1), max(x0, x1))
    ymin, ymax = (min(y0, y1), max(y0, y1))
    rx = float(np.clip((cx - xmin) / max(1e-12, xmax - xmin), 0.0, 1.0))
    ry = float(np.clip((cy - ymin) / max(1e-12, ymax - ymin), 0.0, 1.0))
    nwx = (xmax - xmin) * factor
    nwy = (ymax - ymin) * factor
    nxmin = cx - rx * nwx
    nxmax = nxmin + nwx
    nymin = cy - ry * nwy
    nymax = nymin + nwy
    if nxmin < fx0:
        d = fx0 - nxmin
        nxmin += d
        nxmax += d
    if nxmax > fx1:
        d = nxmax - fx1
        nxmin -= d
        nxmax -= d
    if nymin < fy0:
        d = fy0 - nymin
        nymin += d
        nymax += d
    if nymax > fy1:
        d = nymax - fy1
        nymin -= d
        nymax -= d
    if nxmax - nxmin >= 0.999999 * (fx1 - fx0) and nymax - nymin >= 0.999999 * (fy1 - fy0):
        self._line_profile_zoomed = False
        self._line_profile_zoom_limits = None
        ax.set_xlim(fx0, fx1)
        ax.set_ylim(fy0, fy1)
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
        canvas.draw_idle()
        return
    self._line_profile_zoomed = True
    self._line_profile_zoom_limits = ((nxmin, nxmax), (nymin, nymax))
    ax.set_autoscale_on(False)
    ax.set_xlim(nxmin, nxmax)
    ax.set_ylim(nymin, nymax)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    canvas.draw_idle()

def _tab7_line_profile_reset_zoom_final(self):
    full = _tab7_line_profile_full_nm_extent(self)
    if full is None:
        return
    fx0, fx1, fy0, fy1 = full
    ax = self.line_profile_ax_image
    self._line_profile_zoomed = False
    self._line_profile_zoom_limits = None
    ax.set_autoscale_on(False)
    ax.set_xlim(fx0, fx1)
    ax.set_ylim(fy0, fy1)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    self.line_profile_canvas_image.draw_idle()
CDIWorkflowApp._line_profile_zoom_mousewheel = _tab7_line_profile_zoom_nm_final
CDIWorkflowApp._line_profile_reset_zoom = _tab7_line_profile_reset_zoom_final
_TAB7_IMAGE_UPDATE_BEFORE_FINAL_VIEW = CDIWorkflowApp._line_profile_update_image

def _tab7_line_profile_update_image_final(self, idx=None, preserve_lines=True):
    result = _TAB7_IMAGE_UPDATE_BEFORE_FINAL_VIEW(self, idx=idx, preserve_lines=preserve_lines)
    try:
        full = _tab7_line_profile_full_nm_extent(self)
        if full is not None:
            ax = self.line_profile_ax_image
            ax.set_axis_on()
            ax.set_xlabel('X (nm)')
            ax.set_ylabel('Y (nm)')
            ax.set_aspect('equal', adjustable='box')
            ax.set_anchor('C')
            if getattr(self, '_line_profile_zoomed', False) and getattr(self, '_line_profile_zoom_limits', None) is not None:
                z = self._line_profile_zoom_limits
                ax.set_xlim(*z[0])
                ax.set_ylim(*z[1])
            else:
                ax.set_xlim(full[0], full[1])
                ax.set_ylim(full[2], full[3])
            self.line_profile_canvas_image.draw_idle()
    except Exception:
        pass
    return result
CDIWorkflowApp._line_profile_update_image = _tab7_line_profile_update_image_final
_TAB7_BUILD_UI_BEFORE_ROI_ZOOM = CDIWorkflowApp._build_ui

def _build_ui_final_tab5_tab7_roi_zoom(self):
    _TAB7_BUILD_UI_BEFORE_ROI_ZOOM(self)
    canvas = getattr(self, 'line_profile_canvas_image', None)
    if canvas is not None:
        _final_disconnect_canvas_event(canvas, 'scroll_event')
        canvas.mpl_connect('scroll_event', self._line_profile_zoom_mousewheel)
    ax = getattr(self, 'line_profile_ax_image', None)
    if ax is not None:
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
CDIWorkflowApp._build_ui = _build_ui_final_tab5_tab7_roi_zoom

def _tab7_sync_private_only(self):
    if _tab7_private_stack(self) is None:
        return
    try:
        idx = int(self.line_profile_frame_var.get())
    except Exception:
        idx = 0
    self._line_profile_update_image(idx=idx, preserve_lines=True)
CDIWorkflowApp._line_profile_sync_from_secondary_roi = _tab7_sync_private_only

def _pixel_cal_zoom_scroll_orientation_safe(self, which, event):
    """Cursor-centred wheel zoom for Tab-5 images without Y-axis inversion."""
    if event is None:
        return
    if which == 'sem':
        ax = self.pixel_cal_sem_ax
        canvas = self.pixel_cal_sem_canvas
        image = getattr(self, 'sem_image', None)
    else:
        ax = self.pixel_cal_roi_ax
        canvas = self.pixel_cal_roi_canvas
        image = _pixel_cal_get_display_image_final(self, which)
    if getattr(event, 'inaxes', None) is not ax:
        return
    if event.xdata is None or event.ydata is None or image is None:
        return
    image = np.asarray(image)
    if image.ndim < 2:
        return
    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        button = getattr(event, 'button', None)
        if button in ('up', 4):
            step = 1.0
        elif button in ('down', 5):
            step = -1.0
    if step == 0:
        return
    h, w = image.shape[:2]
    xmin_full, xmax_full = (-0.5, float(w) - 0.5)
    ymin_full, ymax_full = (-0.5, float(h) - 0.5)
    cx, cy = (float(event.xdata), float(event.ydata))
    cur_x0, cur_x1 = map(float, ax.get_xlim())
    cur_y0, cur_y1 = map(float, ax.get_ylim())
    xlo, xhi = (min(cur_x0, cur_x1), max(cur_x0, cur_x1))
    ylo, yhi = (min(cur_y0, cur_y1), max(cur_y0, cur_y1))
    xlo = max(xmin_full, min(xlo, xmax_full))
    xhi = max(xmin_full, min(xhi, xmax_full))
    ylo = max(ymin_full, min(ylo, ymax_full))
    yhi = max(ymin_full, min(yhi, ymax_full))
    if xhi <= xlo or yhi <= ylo:
        xlo, xhi = (xmin_full, xmax_full)
        ylo, yhi = (ymin_full, ymax_full)
    fx = (cx - xlo) / max(xhi - xlo, 1e-12)
    fy = (cy - ylo) / max(yhi - ylo, 1e-12)
    fx = float(np.clip(fx, 0.0, 1.0))
    fy = float(np.clip(fy, 0.0, 1.0))
    factor = 1.2 ** (-step)
    new_w = (xhi - xlo) * factor
    new_h = (yhi - ylo) * factor
    nxlo = cx - fx * new_w
    nxhi = nxlo + new_w
    nylo = cy - fy * new_h
    nyhi = nylo + new_h
    if nxlo < xmin_full:
        shift = xmin_full - nxlo
        nxlo += shift
        nxhi += shift
    if nxhi > xmax_full:
        shift = nxhi - xmax_full
        nxlo -= shift
        nxhi -= shift
    if nylo < ymin_full:
        shift = ymin_full - nylo
        nylo += shift
        nyhi += shift
    if nyhi > ymax_full:
        shift = nyhi - ymax_full
        nylo -= shift
        nyhi -= shift
    nxlo = max(xmin_full, min(nxlo, xmax_full))
    nxhi = max(xmin_full, min(nxhi, xmax_full))
    nylo = max(ymin_full, min(nylo, ymax_full))
    nyhi = max(ymin_full, min(nyhi, ymax_full))
    full_w = xmax_full - xmin_full
    full_h = ymax_full - ymin_full
    if nxhi - nxlo >= 0.999999 * full_w and nyhi - nylo >= 0.999999 * full_h:
        self._pixel_cal_zoom_limits[which] = None
        ax.set_autoscale_on(False)
        ax.set_xlim(xmin_full, xmax_full)
        ax.set_ylim(ymax_full, ymin_full)
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
        canvas.draw_idle()
        return
    self._pixel_cal_zoom_limits[which] = ((nxlo, nxhi), (nyhi, nylo))
    ax.set_autoscale_on(False)
    ax.set_xlim(nxlo, nxhi)
    ax.set_ylim(nyhi, nylo)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    canvas.draw_idle()

def _pixel_cal_reset_zoom_orientation_safe(self, which):
    """Reset Tab-5 zoom to the native image orientation without redraw inversion."""
    if which == 'sem':
        image = getattr(self, 'sem_image', None)
        ax = self.pixel_cal_sem_ax
        canvas = self.pixel_cal_sem_canvas
    else:
        image = _pixel_cal_get_display_image_final(self, which)
        ax = self.pixel_cal_roi_ax
        canvas = self.pixel_cal_roi_canvas
    if image is None:
        return
    image = np.asarray(image)
    if image.ndim < 2:
        return
    h, w = image.shape[:2]
    self._pixel_cal_zoom_limits[which] = None
    ax.set_autoscale_on(False)
    ax.set_xlim(-0.5, float(w) - 0.5)
    ax.set_ylim(float(h) - 0.5, -0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    canvas.draw_idle()
CDIWorkflowApp._pixel_cal_zoom_scroll = _pixel_cal_zoom_scroll_orientation_safe
CDIWorkflowApp._pixel_cal_reset_zoom = _pixel_cal_reset_zoom_orientation_safe

def _tab7_private_stack(self):
    s = getattr(self, 'line_source_stack', None)
    if s is not None:
        try:
            if np.asarray(s).ndim == 3 and len(s):
                return s
        except Exception:
            pass
    return getattr(self, 'roi2_stack', None)

def _tab7_private_scale(self):
    f = getattr(self, '_tab7_fixed_nm_per_output_px', None)
    if f is not None:
        try:
            sx, sy = (float(f[0]), float(f[1]))
            if sx > 0 and sy > 0 and np.isfinite(sx) and np.isfinite(sy):
                return (sx, sy)
        except Exception:
            pass
    try:
        sc = _tab6_effective_nm_per_output_px(self)
        if sc is not None:
            sx, sy = map(float, sc)
            return (sx, sy)
    except Exception:
        pass
    try:
        v = float(getattr(self, 'primary_roi_final_nm_per_pixel', 1.0))
        if np.isfinite(v) and v > 0:
            return (v, v)
    except Exception:
        pass
    return (1.0, 1.0)
_tab7_line_profile_scales_nm = _tab7_private_scale

def _tab7_capture_roi2(self):
    s = getattr(self, 'roi2_stack', None)
    if s is None:
        return False
    a = np.asarray(s, dtype=np.float64)
    if a.ndim != 3 or a.shape[0] == 0:
        return False
    self.line_source_stack = a.copy()
    c = getattr(self, 'roi2_confirmed_coords', None)
    self.line_source_coords = dict(c) if isinstance(c, dict) else None
    try:
        sc = _tab6_effective_nm_per_output_px(self)
    except Exception:
        sc = None
    if sc is None:
        sc = (1.0, 1.0)
    self._tab7_fixed_nm_per_output_px = (float(sc[0]), float(sc[1]))
    self._line_profile_lines = []
    self._line_profile_results = []
    self._line_profile_measurements_list = []
    self._line_profile_pending_point = None
    self._line_profile_measure_mode = False
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    try:
        idx = int(self.roi2_idx_var.get())
    except Exception:
        idx = 0
    idx = max(0, min(idx, len(a) - 1))
    self.line_profile_frame_var.set(idx)
    try:
        self.line_profile_frame_slider.configure(from_=0, to=len(a) - 1, state='normal')
        self.line_profile_frame_slider.set(idx)
    except Exception:
        pass
    try:
        self._line_profile_update_image(idx=idx, preserve_lines=False)
    except Exception:
        pass
    try:
        self.line_profile_status_var.set(f'Secondary ROI copied from Tab 6 | independent | scale={sc[0]:.9g} x {sc[1]:.9g} nm/output-px')
    except Exception:
        pass
    try:
        self.nb.select(self.tab_line_profile)
    except Exception:
        pass
    return True
_old_confirm_secondary_final = CDIWorkflowApp.confirm_secondary_roi_from_processing

def _confirm_secondary_final_independent(self, *args, **kwargs):
    result = _old_confirm_secondary_final(self, *args, **kwargs)
    try:
        ok = result is not False and getattr(self, 'roi2_stack', None) is not None
        if ok:
            _tab7_capture_roi2(self)
    except Exception:
        pass
    return result
CDIWorkflowApp.confirm_secondary_roi_from_processing = _confirm_secondary_final_independent

def _tab7_full_nm(self):
    s = _tab7_private_stack(self)
    if s is None or len(s) == 0:
        return None
    h, w = np.asarray(s[0]).shape[:2]
    sx, sy = _tab7_private_scale(self)
    return (0.0, float(w) * sx, 0.0, float(h) * sy)

def _tab7_update_image_ind(self, idx=None, preserve_lines=True):
    s = _tab7_private_stack(self)
    if s is None or len(s) == 0:
        return
    n = len(s)
    try:
        idx = int(self.line_profile_frame_var.get()) if idx is None else int(idx)
    except Exception:
        idx = 0
    idx = max(0, min(idx, n - 1))
    self.line_profile_frame_var.set(idx)
    img = np.asarray(s[idx], float)
    h, w = img.shape[:2]
    sx, sy = _tab7_private_scale(self)
    im = self.line_profile_image
    im.set_data(img)
    im.set_extent((0, w * sx, 0, h * sy))
    im.set_clim(0, 1)
    ax = self.line_profile_ax_image
    ax.set_axis_on()
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    if getattr(self, '_tab7_roi_zoomed', False) and getattr(self, '_tab7_roi_zoom_limits', None):
        try:
            ax.set_xlim(*self._tab7_roi_zoom_limits[0])
            ax.set_ylim(*self._tab7_roi_zoom_limits[1])
        except Exception:
            ax.set_xlim(0, w * sx)
            ax.set_ylim(0, h * sy)
    else:
        ax.set_xlim(0, w * sx)
        ax.set_ylim(0, h * sy)
    try:
        field = float(self.real_fields_mT[idx])
    except Exception:
        field = float(idx)
    ax.set_title(f'Secondary ROI — Frame {idx + 1}/{n} | Field = {field:+.2f} mT')
    try:
        self.line_profile_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT')
        self.line_profile_colorbar.update_normal(im)
    except Exception:
        pass
    if preserve_lines:
        try:
            self._line_profile_redraw_profiles(idx)
        except Exception:
            pass
    try:
        self._line_profile_redraw_image_lines()
    except Exception:
        pass
    self.line_profile_canvas_image.draw_idle()

def _tab7_redraw_profiles_ind(self, idx=None):
    s = _tab7_private_stack(self)
    if s is None or len(s) == 0:
        return
    n = len(s)
    try:
        idx = int(self.line_profile_frame_var.get()) if idx is None else int(idx)
    except Exception:
        idx = 0
    idx = max(0, min(idx, n - 1))
    self.line_profile_frame_var.set(idx)
    img = np.asarray(s[idx], float)
    ax = self.line_profile_ax_graph
    ax.clear()
    ax.set_xlabel('Distance (nm)')
    ax.set_ylabel('Intensity')
    ax.grid(True, alpha=0.22)
    ax.set_title(f'Secondary ROI line profiles — Frame {idx + 1}/{n}')
    sx, sy = _tab7_private_scale(self)
    self._line_profile_results = []
    self._line_profile_graph_artists = []
    for i, line in enumerate(getattr(self, '_line_profile_lines', []) or []):
        dpx, prof = self._line_profile_extract(img, line['p0'], line['p1'])
        p0 = np.asarray(line['p0'], float)
        p1 = np.asarray(line['p1'], float)
        dx = p1[0] - p0[0]
        dy = p1[1] - p0[1]
        L = float(np.hypot(dx, dy))
        fac = float(np.hypot(dx / L * sx, dy / L * sy)) if L > 0 else sx
        dnm = np.asarray(dpx, float) * fac
        r = {'set': i + 1, 'p0': line['p0'], 'p1': line['p1'], 'length': float(dpx[-1]) if len(dpx) else 0.0, 'length_nm': float(dnm[-1]) if len(dnm) else 0.0, 'dist': np.asarray(dpx), 'dist_nm': dnm, 'profile': np.asarray(prof), 'stats': _tab7_extrema_stats_nm(self, dpx, prof, dnm)}
        self._line_profile_results.append(r)
        ln, = ax.plot(dnm, prof, lw=2, color=self._line_profile_line_color(i), label=f'Set {i + 1}', picker=18, pickradius=18, zorder=4)
        ln._line_profile_setno = i + 1
        self._line_profile_graph_artists.append(ln)
    if self._line_profile_results:
        ax.legend(loc='best')
    self._line_profile_current_profile = self._line_profile_results[-1] if self._line_profile_results else None
    try:
        self._line_profile_update_measurement_table()
    except Exception:
        pass
    try:
        self._line_profile_draw_measurement()
    except Exception:
        pass
    try:
        self._line_profile_redraw_image_lines()
    except Exception:
        pass
    self.line_profile_canvas_graph.draw_idle()

def _tab7_frame_ind(self, value=None):
    s = _tab7_private_stack(self)
    if s is None or len(s) == 0:
        return
    try:
        i = int(round(float(value))) if value is not None else int(self.line_profile_frame_var.get())
    except Exception:
        i = 0
    i = max(0, min(i, len(s) - 1))
    self.line_profile_frame_var.set(i)
    try:
        self.line_profile_frame_slider.set(i)
    except Exception:
        pass
    self._line_profile_update_image(idx=i, preserve_lines=True)

def _tab7_zoom_ind(self, event):
    ax = getattr(self, 'line_profile_ax_image', None)
    if event is None or getattr(event, 'inaxes', None) is not ax or event.xdata is None or (event.ydata is None):
        return
    full = _tab7_full_nm(self)
    if full is None:
        return
    fx0, fx1, fy0, fy1 = full
    x0, x1 = map(float, ax.get_xlim())
    y0, y1 = map(float, ax.get_ylim())
    wx = abs(x1 - x0)
    wy = abs(y1 - y0)
    step = float(getattr(event, 'step', 0) or 0)
    b = getattr(event, 'button', None)
    if step == 0:
        step = 1.0 if b in ('up', 4) else -1.0 if b in ('down', 5) else 0.0
    if step == 0:
        return
    f = 1.2 ** (-step)
    cx, cy = (float(event.xdata), float(event.ydata))
    rx = np.clip((cx - min(x0, x1)) / max(wx, 1e-12), 0, 1)
    ry = np.clip((cy - min(y0, y1)) / max(wy, 1e-12), 0, 1)
    nw = wx * f
    nh = wy * f
    if nw >= 0.999999 * (fx1 - fx0) and nh >= 0.999999 * (fy1 - fy0):
        return _tab7_roi_reset_zoom_ind(self)
    nx0 = cx - rx * nw
    nx1 = nx0 + nw
    ny0 = cy - ry * nh
    ny1 = ny0 + nh
    if nx0 < fx0:
        nx1 += fx0 - nx0
        nx0 = fx0
    if nx1 > fx1:
        nx0 -= nx1 - fx1
        nx1 = fx1
    if ny0 < fy0:
        ny1 += fy0 - ny0
        ny0 = fy0
    if ny1 > fy1:
        ny0 -= ny1 - fy1
        ny1 = fy1
    self._tab7_roi_zoomed = True
    self._tab7_roi_zoom_limits = ((nx0, nx1), (ny0, ny1))
    ax.set_xlim(nx0, nx1)
    ax.set_ylim(ny0, ny1)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    self.line_profile_canvas_image.draw_idle()
    return 'break'

def _tab7_roi_reset_zoom_ind(self):
    full = _tab7_full_nm(self)
    if full is None:
        return
    fx0, fx1, fy0, fy1 = full
    self._tab7_roi_zoomed = False
    self._tab7_roi_zoom_limits = None
    ax = self.line_profile_ax_image
    ax.set_xlim(fx0, fx1)
    ax.set_ylim(fy0, fy1)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    self.line_profile_canvas_image.draw_idle()

def _tab7_curve_hit(self, xnm, yv, maxscreen=42.0):
    ax = self.line_profile_ax_graph
    target = ax.transData.transform((xnm, yv))
    best = None
    for r in getattr(self, '_line_profile_results', []) or []:
        d = np.asarray(r.get('dist_nm', []), float)
        p = np.asarray(r.get('profile', []), float)
        px = np.asarray(r.get('dist', []), float)
        if d.size < 2 or p.size != d.size:
            continue
        scr = ax.transData.transform(np.column_stack((d, p)))
        a = scr[:-1]
        b = scr[1:]
        v = b - a
        vv = np.einsum('ij,ij->i', v, v)
        w = target - a
        tt = np.einsum('ij,ij->i', w, v) / np.where(vv > 0, vv, 1)
        tt = np.clip(tt, 0, 1)
        q = a + v * tt[:, None]
        d2 = np.einsum('ij,ij->i', q - target, q - target)
        j = int(np.argmin(d2))
        dd = float(np.sqrt(max(d2[j], 0)))
        if best is None or dd < best[0]:
            xn = float(d[j] + tt[j] * (d[j + 1] - d[j]))
            yp = float(p[j] + tt[j] * (p[j + 1] - p[j]))
            xp = float(np.interp(xn, d, px))
            best = (dd, xp, yp, int(r.get('set', 1)), xn)
    return best if best is None or best[0] > maxscreen else best

def _tab7_measure_press_ind(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph or event.button not in (1, None):
        return
    ms = getattr(self, '_line_profile_measurements_list', []) or []
    hit = None
    for pi, m in enumerate(ms):
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(m['setno'])), None)
        if r is None:
            continue
        d = np.asarray(r.get('dist_nm', []), float)
        px = np.asarray(r.get('dist', []), float)
        for k, pt in enumerate(m.get('pts', [])[:2]):
            xn = float(np.interp(pt[0], px, d))
            sx, sy = self.line_profile_ax_graph.transData.transform((xn, pt[1]))
            dd = float(np.hypot(event.x - sx, event.y - sy))
            if hit is None or dd < hit[0]:
                hit = (dd, pi, k)
    if hit is not None and hit[0] <= 30:
        self._line_profile_measure_drag_pair = hit[1]
        self._line_profile_measure_drag_point = hit[2]
        self._line_profile_measure_mode = False
        self.line_profile_measure_var.set(f"Dragging M{(ms[hit[1]]['m1_id'] if hit[2] == 0 else ms[hit[1]]['m2_id'])}; move cursor and release to fix.")
        return
    if getattr(self, '_line_profile_measure_mode', False):
        h = _tab7_curve_hit(self, float(event.xdata), float(event.ydata), 45)
        if h is None:
            return
        _, xp, yp, setno, xn = h
        pending = getattr(self, '_line_profile_pending_point', None)
        pair = len(ms) + 1
        m1 = 2 * pair - 1
        m2 = 2 * pair
        if pending is None:
            self._line_profile_pending_point = (xp, yp, setno)
            self._line_profile_measure_next_point = m2
            self.line_profile_measure_var.set(f'M{m1} fixed at {xn:.6g} nm. Click M{m2} on the same profile.')
        else:
            if int(pending[2]) != setno:
                return
            ms.append({'pair_id': pair, 'm1_id': m1, 'm2_id': m2, 'setno': setno, 'pts': [pending, (xp, yp, setno)]})
            self._line_profile_pending_point = None
            self._line_profile_measure_next_point = m2 + 1
            self._line_profile_measure_mode = True
            self._line_profile_update_measurement_table()
            self._line_profile_draw_measurement()
            r = self._line_profile_results[setno - 1]
            d = np.asarray(r['dist_nm'])
            px = np.asarray(r['dist'])
            x0 = float(np.interp(pending[0], px, d))
            self.line_profile_measure_var.set(f'M{m1}–M{m2} complete: ΔL = {abs(xn - x0):.6g} nm')

def _tab7_measure_motion_ind(self, event):
    pi = getattr(self, '_line_profile_measure_drag_pair', None)
    k = getattr(self, '_line_profile_measure_drag_point', None)
    if pi is None or k is None or event is None or (event.inaxes is not self.line_profile_ax_graph) or (event.xdata is None):
        return
    ms = self._line_profile_measurements_list
    if not 0 <= pi < len(ms):
        return
    m = ms[pi]
    r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(m['setno'])), None)
    if r is None:
        return
    d = np.asarray(r['dist_nm'], float)
    px = np.asarray(r['dist'], float)
    p = np.asarray(r['profile'], float)
    xn = float(np.clip(event.xdata, d[0], d[-1]))
    yp = float(np.interp(xn, d, p))
    xp = float(np.interp(xn, d, px))
    m['pts'][k] = (xp, yp, int(m['setno']))
    self._line_profile_update_measurement_table()
    self._line_profile_draw_measurement()
    x0 = float(np.interp(m['pts'][0][0], px, d))
    x1 = float(np.interp(m['pts'][1][0], px, d))
    self.line_profile_measure_var.set(f"Dynamic scale: M{m['m1_id']}={x0:.6g} nm | M{m['m2_id']}={x1:.6g} nm | ΔL={abs(x1 - x0):.6g} nm")

def _tab7_measure_release_ind(self, event):
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    if getattr(self, '_line_profile_measurements_list', None):
        self._line_profile_measure_mode = True
    try:
        self._line_profile_draw_measurement()
    except Exception:
        pass
CDIWorkflowApp._line_profile_update_image = _tab7_update_image_ind
CDIWorkflowApp._line_profile_redraw_profiles = _tab7_redraw_profiles_ind
CDIWorkflowApp._line_profile_frame_slider_changed = _tab7_frame_ind
CDIWorkflowApp._line_profile_zoom_mousewheel = _tab7_zoom_ind
CDIWorkflowApp._line_profile_reset_zoom = _tab7_roi_reset_zoom_ind
CDIWorkflowApp._line_profile_graph_press = _tab7_measure_press_ind
CDIWorkflowApp._line_profile_graph_motion = _tab7_measure_motion_ind
CDIWorkflowApp._line_profile_graph_release = _tab7_measure_release_ind
_old_build_ui_tab7_ind = CDIWorkflowApp._build_ui

def _build_ui_tab7_ind(self):
    _old_build_ui_tab7_ind(self)
    try:
        self._tab7_roi_zoomed = False
        self._tab7_roi_zoom_limits = None
        c = getattr(self, 'line_profile_canvas_image', None)
        if c is not None:
            _final_disconnect_canvas_event(c, 'scroll_event')
            c.mpl_connect('scroll_event', self._line_profile_zoom_mousewheel)
        g = getattr(self, 'line_profile_canvas_graph', None)
        if g is not None:
            _final_disconnect_canvas_event(g, 'button_press_event')
            _final_disconnect_canvas_event(g, 'motion_notify_event')
            _final_disconnect_canvas_event(g, 'button_release_event')
            g.mpl_connect('button_press_event', self._line_profile_graph_press)
            g.mpl_connect('motion_notify_event', self._line_profile_graph_motion)
            g.mpl_connect('button_release_event', self._line_profile_graph_release)
        s = getattr(self, 'line_profile_frame_slider', None)
        if s is not None:
            s.configure(command=lambda v: self._line_profile_frame_slider_changed(v))
    except Exception:
        pass
CDIWorkflowApp._build_ui = _build_ui_tab7_ind

def _tab5_authoritative_nm_per_pixel(self):
    """Return the committed Tab-5 Primary ROI calibration only."""
    for attr in (
        'pixel_cal_active_nm_per_pixel',
        'primary_roi_final_nm_per_pixel',
    ):
        try:
            v=float(getattr(self,attr))
            if np.isfinite(v) and v>0:
                return v
        except Exception:
            pass
    return None


def _tab5_effective_pixel_size_final(self):
    return _tab5_authoritative_nm_per_pixel(self)


try:
    if hasattr(CDIWorkflowApp, '_proc_get_tab5_base_pixel_size'):
        _old_base_px_final = CDIWorkflowApp._proc_get_tab5_base_pixel_size

        def _proc_get_tab5_base_pixel_size_final(self):
            v = _tab5_effective_pixel_size_final(self)
            return v if v is not None else _old_base_px_final(self)
        CDIWorkflowApp._proc_get_tab5_base_pixel_size = _proc_get_tab5_base_pixel_size_final
except Exception:
    pass
try:
    if hasattr(CDIWorkflowApp, '_tab5_final_primary_nm_per_pixel'):
        _old_tab5_final_getter = CDIWorkflowApp._tab5_final_primary_nm_per_pixel

        def _tab5_final_primary_nm_per_pixel_final(self):
            v = _tab5_effective_pixel_size_final(self)
            return v if v is not None else _old_tab5_final_getter(self)
        CDIWorkflowApp._tab5_final_primary_nm_per_pixel = _tab5_final_primary_nm_per_pixel_final
except Exception:
    pass
if hasattr(CDIWorkflowApp, '_pixel_cal_apply_direct'):
    _old_direct_apply_final = CDIWorkflowApp._pixel_cal_apply_direct
else:
    _old_direct_apply_final = None

def _tab7_capture_roi2_final(self):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None:
        return False
    arr = np.asarray(stack, dtype=np.float64)
    if arr.ndim != 3 or arr.shape[0] == 0:
        return False
    self.line_source_stack = arr.copy()
    coords = getattr(self, 'roi2_confirmed_coords', None)
    self.line_source_coords = dict(coords) if isinstance(coords, dict) else None
    try:
        sc = _tab6_effective_nm_per_output_px(self)
    except Exception:
        sc = None
    if sc is None:
        v = _tab5_effective_pixel_size_final(self)
        sc = (v, v) if v is not None else (1.0, 1.0)
    self._tab7_fixed_nm_per_output_px = (float(sc[0]), float(sc[1]))
    try:
        idx = int(self.roi2_idx_var.get())
    except Exception:
        idx = 0
    idx = max(0, min(idx, len(arr) - 1))
    self.line_profile_frame_var.set(idx)
    self._line_profile_lines = []
    self._line_profile_results = []
    self._line_profile_measurements_list = []
    self._line_profile_pending_point = None
    self._line_profile_measure_mode = False
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    try:
        self._line_profile_update_image(idx=idx, preserve_lines=False)
    except Exception:
        pass
    try:
        self.line_profile_status_var.set(f'ROI2 copied from Tab 6 | independent Tab 7 copy | scale = {sc[0]:.9g} × {sc[1]:.9g} nm/output-px')
    except Exception:
        pass
    try:
        self.nb.select(self.tab_line_profile)
    except Exception:
        pass
    return True
CDIWorkflowApp._tab7_capture_roi2 = _tab7_capture_roi2_final
_old_confirm_secondary_tab7_final = CDIWorkflowApp.confirm_secondary_roi_from_processing

def _confirm_secondary_roi_tab7_final(self, *args, **kwargs):
    result = _old_confirm_secondary_tab7_final(self, *args, **kwargs)
    try:
        if result is not False and getattr(self, 'roi2_stack', None) is not None:
            _tab7_capture_roi2_final(self)
    except Exception as exc:
        try:
            self.log(f'Tab 7 ROI2 capture warning: {exc}')
        except Exception:
            pass
    return result
CDIWorkflowApp.confirm_secondary_roi_from_processing = _confirm_secondary_roi_tab7_final

def _tab7_update_image_color_final(self, idx=None, preserve_lines=True):
    stack = getattr(self, 'line_source_stack', None)
    if stack is None:
        stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        return
    n = len(stack)
    try:
        idx = int(self.line_profile_frame_var.get()) if idx is None else int(idx)
    except Exception:
        idx = 0
    idx = max(0, min(idx, n - 1))
    self.line_profile_frame_var.set(idx)
    img = np.asarray(stack[idx], dtype=float)
    h, w = img.shape[:2]
    sx, sy = _tab7_private_scale(self)
    artist = self.line_profile_image
    cmap_name = 'RdBu_r'
    try:
        candidate = str(self.colormap_var.get()).strip()
        if candidate:
            cmap_name = candidate
    except Exception:
        pass
    try:
        artist.set_cmap(cmap_name)
    except Exception:
        artist.set_cmap('RdBu_r')
    artist.set_data(np.clip(img, 0.0, 1.0))
    artist.set_extent((0.0, w * sx, 0.0, h * sy))
    artist.set_clim(0.0, 1.0)
    ax = self.line_profile_ax_image
    ax.set_axis_on()
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    full = (0.0, float(w * sx), 0.0, float(h * sy))
    if getattr(self, '_tab7_roi_zoomed', False) and getattr(self, '_tab7_roi_zoom_limits', None):
        try:
            ax.set_xlim(*self._tab7_roi_zoom_limits[0])
            ax.set_ylim(*self._tab7_roi_zoom_limits[1])
        except Exception:
            ax.set_xlim(full[0], full[1])
            ax.set_ylim(full[2], full[3])
    else:
        ax.set_xlim(full[0], full[1])
        ax.set_ylim(full[2], full[3])
    try:
        field = float(self.real_fields_mT[idx])
    except Exception:
        field = float(idx)
    ax.set_title(f'Secondary ROI — Frame {idx + 1}/{n} | Field = {field:+.2f} mT')
    try:
        self.line_profile_colorbar.update_normal(artist)
        self.line_profile_colorbar.set_label('Selected Secondary ROI contrast')
    except Exception:
        pass
    if preserve_lines:
        try:
            self._line_profile_redraw_profiles(idx)
        except Exception:
            pass
    try:
        self._line_profile_redraw_image_lines()
    except Exception:
        pass
    self.line_profile_canvas_image.draw_idle()
CDIWorkflowApp._line_profile_update_image = _tab7_update_image_color_final

def _tab7_draw_measurement_final(self):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    ax = getattr(self, 'line_profile_ax_graph', None)
    if ax is None:
        return
    measurements = getattr(self, '_line_profile_measurements_list', []) or []
    for m in measurements:
        setno = int(m.get('setno', 1))
        pts = m.get('pts', [])
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == setno), None)
        if r is None or len(pts) < 1:
            continue
        d = np.asarray(r.get('dist_nm', r.get('dist', [])), dtype=float)
        px = np.asarray(r.get('dist', []), dtype=float)
        prof = np.asarray(r.get('profile', []), dtype=float)
        if d.size < 2 or px.size != d.size:
            continue
        xs = []
        ys = []
        for j, pt in enumerate(pts[:2]):
            xp = float(pt[0])
            y = float(pt[1])
            xnm = float(np.interp(xp, px, d))
            xs.append(xnm)
            ys.append(y)
            color = 'gold' if j == 0 else 'lime'
            self._line_profile_measure_artists.extend([ax.axvline(xnm, color=color, ls='--', lw=1.5, alpha=0.95, zorder=8), ax.scatter([xnm], [y], s=72, color=color, edgecolors='black', linewidths=1.2, zorder=9), ax.text(xnm, y, f" M{(m.get('m1_id', 1) if j == 0 else m.get('m2_id', 2))} ", color='black', fontsize=8, weight='bold', ha='center', va='bottom', bbox=dict(boxstyle='round,pad=0.12', facecolor='white', edgecolor='black', alpha=0.9), zorder=10)])
        if len(xs) == 2:
            y0, y1 = ax.get_ylim()
            ybar = y1 - 0.1 * max(y1 - y0, 1e-09)
            self._line_profile_measure_artists.append(ax.annotate('', xy=(xs[0], ybar), xytext=(xs[1], ybar), arrowprops=dict(arrowstyle='<->', color='blue', lw=2)))
            self._line_profile_measure_artists.append(ax.text((xs[0] + xs[1]) / 2, ybar, f' ΔL = {abs(xs[1] - xs[0]):.3f} nm ', ha='center', va='bottom', color='black', bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='blue', alpha=0.92)))
    try:
        self.line_profile_canvas_graph.draw_idle()
    except Exception:
        pass

def _tab7_start_measurement_final(self):
    if not getattr(self, '_line_profile_results', []):
        self.line_profile_measure_var.set('Draw at least one profile line first.')
        return
    self._line_profile_measurements_list = getattr(self, '_line_profile_measurements_list', []) or []
    self._line_profile_pending_point = None
    self._line_profile_measure_mode = True
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    self.line_profile_measure_var.set('DYNAMIC SCALE: click M1 and M2 on the profile. Then drag either marker to adjust it.')

def _tab7_profile_hit_final(self, xnm, yv, maxscreen=30.0):
    ax = self.line_profile_ax_graph
    target = ax.transData.transform((float(xnm), float(yv)))
    best = None
    for r in getattr(self, '_line_profile_results', []) or []:
        d = np.asarray(r.get('dist_nm', []), float)
        p = np.asarray(r.get('profile', []), float)
        px = np.asarray(r.get('dist', []), float)
        if d.size < 2 or p.size != d.size:
            continue
        scr = ax.transData.transform(np.column_stack((d, p)))
        a = scr[:-1]
        b = scr[1:]
        v = b - a
        vv = np.einsum('ij,ij->i', v, v)
        w = target - a
        t = np.einsum('ij,ij->i', w, v) / np.where(vv > 0, vv, 1)
        t = np.clip(t, 0, 1)
        q = a + v * t[:, None]
        dd = np.sqrt(np.einsum('ij,ij->i', q - target, q - target))
        j = int(np.argmin(dd))
        dist = float(dd[j])
        if best is None or dist < best[0]:
            xn = float(d[j] + t[j] * (d[j + 1] - d[j]))
            yp = float(p[j] + t[j] * (p[j + 1] - p[j]))
            xp = float(np.interp(xn, d, px))
            best = (dist, xp, yp, int(r.get('set', 1)), xn)
    return best if best is None or best[0] <= maxscreen else None

def _tab7_measure_press_final(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph or event.button != 1:
        return
    measurements = getattr(self, '_line_profile_measurements_list', []) or []
    for pi, m in enumerate(measurements):
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(m.get('setno', 1))), None)
        if r is None:
            continue
        d = np.asarray(r.get('dist_nm', []), float)
        px = np.asarray(r.get('dist', []), float)
        for k, pt in enumerate((m.get('pts', []) or [])[:2]):
            xnm = float(np.interp(float(pt[0]), px, d))
            sx, sy = self.line_profile_ax_graph.transData.transform((xnm, float(pt[1])))
            dd = float(np.hypot(event.x - sx, event.y - sy))
            if dd <= 22:
                self._line_profile_measure_drag_pair = pi
                self._line_profile_measure_drag_point = k
                self._line_profile_measure_mode = False
                self.line_profile_measure_var.set(f"Dragging M{(m.get('m1_id', 1) if k == 0 else m.get('m2_id', 2))} — release mouse to fix.")
                return
    if not getattr(self, '_line_profile_measure_mode', False):
        return
    h = _tab7_profile_hit_final(self, float(event.xdata), float(event.ydata), 42.0)
    if h is None:
        return
    _, xp, yp, setno, xnm = h
    pending = getattr(self, '_line_profile_pending_point', None)
    if pending is None:
        self._line_profile_pending_point = (xp, yp, setno)
        self.line_profile_measure_var.set(f'M1 fixed at {xnm:.6g} nm. Click M2 on the same profile.')
    else:
        if int(pending[2]) != int(setno):
            self.line_profile_measure_var.set('M2 must be selected on the same profile line as M1.')
            return
        pair = len(measurements) + 1
        m = {'pair_id': pair, 'm1_id': 2 * pair - 1, 'm2_id': 2 * pair, 'setno': int(setno), 'pts': [pending, (xp, yp, setno)]}
        measurements.append(m)
        self._line_profile_measurements_list = measurements
        self._line_profile_pending_point = None
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(setno)))
        d = np.asarray(r.get('dist_nm', []), float)
        px = np.asarray(r.get('dist', []), float)
        x0 = float(np.interp(pending[0], px, d))
        self.line_profile_measure_var.set(f"M{m['m1_id']}–M{m['m2_id']}: ΔL = {abs(xnm - x0):.6g} nm")
        self._line_profile_update_measurement_table()
        _tab7_draw_measurement_final(self)

def _tab7_measure_motion_final(self, event):
    pi = getattr(self, '_line_profile_measure_drag_pair', None)
    k = getattr(self, '_line_profile_measure_drag_point', None)
    if pi is None or k is None or event is None or (event.inaxes is not self.line_profile_ax_graph) or (event.xdata is None):
        return
    ms = getattr(self, '_line_profile_measurements_list', []) or []
    if not 0 <= pi < len(ms):
        return
    m = ms[pi]
    r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(m.get('setno', 1))), None)
    if r is None:
        return
    d = np.asarray(r.get('dist_nm', []), float)
    px = np.asarray(r.get('dist', []), float)
    p = np.asarray(r.get('profile', []), float)
    xnm = float(np.clip(event.xdata, d[0], d[-1]))
    y = float(np.interp(xnm, d, p))
    xp = float(np.interp(xnm, d, px))
    m['pts'][k] = (xp, y, int(m.get('setno', 1)))
    m['pts'][k] = tuple(m['pts'][k])
    self._line_profile_update_measurement_table()
    _tab7_draw_measurement_final(self)
    x0 = float(np.interp(m['pts'][0][0], px, d))
    x1 = float(np.interp(m['pts'][1][0], px, d))
    self.line_profile_measure_var.set(f"DYNAMIC SCALE: M{m['m1_id']}={x0:.6g} nm | M{m['m2_id']}={x1:.6g} nm | ΔL={abs(x1 - x0):.6g} nm")

def _tab7_measure_release_final(self, event):
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    if getattr(self, '_line_profile_measurements_list', None):
        self._line_profile_measure_mode = True
    _tab7_draw_measurement_final(self)
CDIWorkflowApp._line_profile_start_measurement = _tab7_start_measurement_final
CDIWorkflowApp._line_profile_graph_press = _tab7_measure_press_final
CDIWorkflowApp._line_profile_graph_motion = _tab7_measure_motion_final
CDIWorkflowApp._line_profile_graph_release = _tab7_measure_release_final
CDIWorkflowApp._line_profile_draw_measurement = _tab7_draw_measurement_final
_old_build_ui_tab7_final = CDIWorkflowApp._build_ui

def _build_ui_tab7_final(self):
    _old_build_ui_tab7_final(self)
    try:
        c = getattr(self, 'line_profile_canvas_graph', None)
        if c is not None:
            cb = c.callbacks.callbacks
            for ev in ('button_press_event', 'motion_notify_event', 'button_release_event'):
                cb.pop(ev, None)
            c.mpl_connect('button_press_event', self._line_profile_graph_press)
            c.mpl_connect('motion_notify_event', self._line_profile_graph_motion)
            c.mpl_connect('button_release_event', self._line_profile_graph_release)
        self._tab7_roi_zoomed = False
        self._tab7_roi_zoom_limits = None
    except Exception:
        pass
CDIWorkflowApp._build_ui = _build_ui_tab7_final

def _tab7_capture_current_view_if_zoomed(self):
    ax = getattr(self, 'line_profile_ax_image', None)
    if ax is None:
        return
    if getattr(self, '_tab7_roi_zoomed', False):
        try:
            self._tab7_roi_zoom_limits = (tuple(map(float, ax.get_xlim())), tuple(map(float, ax.get_ylim())))
        except Exception:
            pass

def _tab7_update_image_preserve_view_final(self, idx=None, preserve_lines=True):
    ax = getattr(self, 'line_profile_ax_image', None)
    old_zoomed = bool(getattr(self, '_tab7_roi_zoomed', False))
    old_limits = getattr(self, '_tab7_roi_zoom_limits', None)
    if old_zoomed and ax is not None:
        try:
            old_limits = (tuple(map(float, ax.get_xlim())), tuple(map(float, ax.get_ylim())))
        except Exception:
            pass
    _tab7_update_image_color_final(self, idx=idx, preserve_lines=preserve_lines)
    if old_zoomed and old_limits and (ax is not None):
        try:
            full = _tab7_full_nm(self)
            if full is not None:
                fx0, fx1, fy0, fy1 = map(float, full)
                (x0, x1), (y0, y1) = old_limits
                wx = min(abs(x1 - x0), abs(fx1 - fx0))
                wy = min(abs(y1 - y0), abs(fy1 - fy0))
                cx = 0.5 * (x0 + x1)
                cy = 0.5 * (y0 + y1)
                x0 = max(fx0, min(cx - 0.5 * wx, fx1 - wx))
                y0 = max(fy0, min(cy - 0.5 * wy, fy1 - wy))
                x1 = x0 + wx
                y1 = y0 + wy
                ax.set_xlim(x0, x1)
                ax.set_ylim(y0, y1)
                ax.set_aspect('equal', adjustable='box')
                ax.set_anchor('C')
                self._tab7_roi_zoomed = not (abs(wx - (fx1 - fx0)) < 1e-09 and abs(wy - (fy1 - fy0)) < 1e-09)
                self._tab7_roi_zoom_limits = ((x0, x1), (y0, y1))
                self.line_profile_canvas_image.draw_idle()
        except Exception:
            pass
CDIWorkflowApp._line_profile_update_image = _tab7_update_image_preserve_view_final

def _tab7_frame_preserve_view_final(self, value=None):
    s = _tab7_private_stack(self)
    if s is None or len(s) == 0:
        return
    try:
        i = int(round(float(value))) if value is not None else int(self.line_profile_frame_var.get())
    except Exception:
        i = 0
    i = max(0, min(i, len(s) - 1))
    self.line_profile_frame_var.set(i)
    try:
        self.line_profile_frame_slider.set(i)
    except Exception:
        pass
    self._line_profile_update_image(idx=i, preserve_lines=True)
CDIWorkflowApp._line_profile_frame_slider_changed = _tab7_frame_preserve_view_final

def _tab7_draw_measurement_sync_final(self):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    ax = getattr(self, 'line_profile_ax_graph', None)
    if ax is None:
        return
    pts = getattr(self, '_line_profile_measure_points', []) or []
    for k, pt in enumerate(pts[:2]):
        try:
            xp, yp, setno = (float(pt[0]), float(pt[1]), int(pt[2]))
        except Exception:
            continue
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == setno), None)
        if r is None:
            continue
        px = np.asarray(r.get('dist', []), dtype=float)
        dnm = np.asarray(r.get('dist_nm', []), dtype=float)
        prof = np.asarray(r.get('profile', []), dtype=float)
        if px.size < 2 or dnm.size != px.size:
            continue
        xnm = float(np.interp(xp, px, dnm))
        color = 'gold' if k == 0 else 'lime'
        mlabel = f'M{k + 1}'
        self._line_profile_measure_artists.extend([ax.axvline(xnm, color=color, ls='--', lw=1.5, alpha=0.95, zorder=8), ax.scatter([xnm], [yp], s=72, color=color, edgecolors='black', linewidths=1.2, zorder=9), ax.text(xnm, yp, f' {mlabel} ', color='black', fontsize=8, weight='bold', ha='center', va='bottom', bbox=dict(boxstyle='round,pad=0.12', facecolor='white', edgecolor='black', alpha=0.9), zorder=10)])
    if len(pts) >= 2:
        try:
            p0 = pts[0]
            p1 = pts[1]
            r0 = next((rr for rr in self._line_profile_results if int(rr.get('set', -1)) == int(p0[2])))
            r1 = next((rr for rr in self._line_profile_results if int(rr.get('set', -1)) == int(p1[2])))
            d0 = np.asarray(r0.get('dist_nm', []), dtype=float)
            px0 = np.asarray(r0.get('dist', []), dtype=float)
            d1 = np.asarray(r1.get('dist_nm', []), dtype=float)
            px1 = np.asarray(r1.get('dist', []), dtype=float)
            x0 = float(np.interp(float(p0[0]), px0, d0))
            x1 = float(np.interp(float(p1[0]), px1, d1))
            ymin, ymax = ax.get_ylim()
            ybar = ymax - 0.1 * max(ymax - ymin, 1e-09)
            self._line_profile_measure_artists.append(ax.annotate('', xy=(x0, ybar), xytext=(x1, ybar), arrowprops=dict(arrowstyle='<->', color='blue', lw=2)))
            self._line_profile_measure_artists.append(ax.text(0.5 * (x0 + x1), ybar, f' M1–M2: ΔL = {abs(x1 - x0):.3f} nm ', ha='center', va='bottom', color='black', bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='blue', alpha=0.92)))
            self.line_profile_measure_var.set(f'Dynamic scale: M1={x0:.6g} nm | M2={x1:.6g} nm | ΔL={abs(x1 - x0):.6g} nm')
        except Exception:
            pass
    try:
        self._line_profile_redraw_image_lines()
    except Exception:
        pass
    try:
        self.line_profile_canvas_graph.draw_idle()
        self.line_profile_canvas_image.draw_idle()
    except Exception:
        pass
CDIWorkflowApp._line_profile_draw_measurement = _tab7_draw_measurement_sync_final

def _tab7_start_measurement_sync_final(self):
    if not getattr(self, '_line_profile_results', []):
        self.line_profile_measure_var.set('Draw at least one profile line first.')
        return
    self._line_profile_measure_points = []
    self._line_profile_measure_mode = True
    self._line_profile_measure_drag_index = None
    self._line_profile_pending_point = None
    self.line_profile_measure_var.set('DYNAMIC SCALE: click M1 and M2 on the profile. Then drag either marker to adjust it.')
CDIWorkflowApp._line_profile_start_measurement = _tab7_start_measurement_sync_final

def _tab7_clear_measurement_sync_final(self):
    for a in getattr(self, '_line_profile_measure_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    for a in getattr(self, '_line_profile_measure_image_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    self._line_profile_measure_image_artists = []
    self._line_profile_measure_points = []
    self._line_profile_measure_mode = False
    self._line_profile_measure_drag_index = None
    self._line_profile_measurements_list = []
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    self._line_profile_pending_point = None
    try:
        self.line_profile_measure_var.set('Measurement cleared. Click DYNAMIC MEASURING SCALE to select M1 and M2.')
    except Exception:
        pass
    try:
        self._line_profile_redraw_image_lines()
        self.line_profile_canvas_graph.draw_idle()
        self.line_profile_canvas_image.draw_idle()
    except Exception:
        pass
CDIWorkflowApp._line_profile_clear_measurement = _tab7_clear_measurement_sync_final

def _tab7_measure_press_sync_final(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return
    if event.button not in (1, None):
        return
    pts = getattr(self, '_line_profile_measure_points', []) or []
    if pts and event.x is not None and (event.y is not None):
        best_i = None
        best_d = float('inf')
        for i, (xp, yp, setno) in enumerate(pts[:2]):
            r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(setno)), None)
            if r is None:
                continue
            px = np.asarray(r.get('dist', []), dtype=float)
            dnm = np.asarray(r.get('dist_nm', []), dtype=float)
            if px.size < 2 or dnm.size != px.size:
                continue
            xnm = float(np.interp(float(xp), px, dnm))
            sx, sy = self.line_profile_ax_graph.transData.transform((xnm, float(yp)))
            dd = float(np.hypot(event.x - sx, event.y - sy))
            if dd < best_d:
                best_d, best_i = (dd, i)
        if best_i is not None and best_d <= 30.0:
            self._line_profile_measure_drag_index = best_i
            self._line_profile_measure_mode = False
            self.line_profile_measure_var.set(f'Dragging M{best_i + 1}: move along the profile and release to fix.')
            return
    if not getattr(self, '_line_profile_measure_mode', False):
        return
    if event.xdata is None or event.ydata is None:
        return
    best = None
    target = self.line_profile_ax_graph.transData.transform((float(event.xdata), float(event.ydata)))
    for r in getattr(self, '_line_profile_results', []) or []:
        dnm = np.asarray(r.get('dist_nm', []), dtype=float)
        prof = np.asarray(r.get('profile', []), dtype=float)
        px = np.asarray(r.get('dist', []), dtype=float)
        if dnm.size < 2 or prof.size != dnm.size or px.size != dnm.size:
            continue
        scr = self.line_profile_ax_graph.transData.transform(np.column_stack((dnm, prof)))
        diff = scr - target
        d2 = np.einsum('ij,ij->i', diff, diff)
        j = int(np.argmin(d2))
        dd = float(np.sqrt(max(d2[j], 0.0)))
        if best is None or dd < best[0]:
            best = (dd, float(px[j]), float(prof[j]), int(r.get('set', 1)), float(dnm[j]))
    if best is None or best[0] > 45.0:
        return
    _, xp, yp, setno, xnm = best
    if not pts:
        self._line_profile_measure_points = [(xp, yp, setno)]
        self._line_profile_measure_mode = True
        self.line_profile_measure_var.set(f'M1 fixed at {xnm:.6g} nm. Click M2 on the same profile.')
    else:
        if int(pts[0][2]) != int(setno):
            self.line_profile_measure_var.set('M2 must be selected on the same profile as M1.')
            return
        self._line_profile_measure_points = [pts[0], (xp, yp, setno)]
        self._line_profile_measure_mode = False
        self.line_profile_measure_var.set(f'M1–M2 complete: ΔL is being shown in nm. M1={xnm:.6g} nm is the second selected endpoint.')
    self._line_profile_update_measurement_table()
    self._line_profile_draw_measurement()

def _tab7_measure_motion_sync_final(self, event):
    i = getattr(self, '_line_profile_measure_drag_index', None)
    if i is None or event is None:
        return
    if event.inaxes is not self.line_profile_ax_graph or event.xdata is None:
        return
    pts = getattr(self, '_line_profile_measure_points', []) or []
    if not 0 <= int(i) < len(pts):
        return
    xp0, yp0, setno = pts[int(i)]
    r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(setno)), None)
    if r is None:
        return
    dnm = np.asarray(r.get('dist_nm', []), dtype=float)
    px = np.asarray(r.get('dist', []), dtype=float)
    prof = np.asarray(r.get('profile', []), dtype=float)
    if dnm.size < 2 or px.size != dnm.size or prof.size != dnm.size:
        return
    xnm = float(np.clip(float(event.xdata), dnm[0], dnm[-1]))
    xp = float(np.interp(xnm, dnm, px))
    yp = float(np.interp(xnm, dnm, prof))
    pts[int(i)] = (xp, yp, int(setno))
    self._line_profile_measure_points = pts
    self._line_profile_update_measurement_table()
    self._line_profile_draw_measurement()

def _tab7_measure_release_sync_final(self, event):
    if getattr(self, '_line_profile_measure_drag_index', None) is not None:
        self._line_profile_measure_drag_index = None
        self._line_profile_measure_mode = False
        self._line_profile_draw_measurement()
CDIWorkflowApp._line_profile_graph_press = _tab7_measure_press_sync_final
CDIWorkflowApp._line_profile_graph_motion = _tab7_measure_motion_sync_final
CDIWorkflowApp._line_profile_graph_release = _tab7_measure_release_sync_final
_old_build_ui_tab7_sync_final = CDIWorkflowApp._build_ui

def _build_ui_tab7_sync_final(self):
    _old_build_ui_tab7_sync_final(self)
    try:
        self._tab7_roi_zoomed = bool(getattr(self, '_tab7_roi_zoomed', False))
        self._tab7_roi_zoom_limits = getattr(self, '_tab7_roi_zoom_limits', None)
        c = getattr(self, 'line_profile_canvas_image', None)
        if c is not None:
            _final_disconnect_canvas_event(c, 'scroll_event')
            c.mpl_connect('scroll_event', self._line_profile_zoom_mousewheel)
        g = getattr(self, 'line_profile_canvas_graph', None)
        if g is not None:
            cb = g.callbacks.callbacks
            for ev in ('button_press_event', 'motion_notify_event', 'button_release_event'):
                cb.pop(ev, None)
            g.mpl_connect('button_press_event', self._line_profile_graph_press)
            g.mpl_connect('motion_notify_event', self._line_profile_graph_motion)
            g.mpl_connect('button_release_event', self._line_profile_graph_release)
    except Exception:
        pass
CDIWorkflowApp._build_ui = _build_ui_tab7_sync_final

def _tab7_sync_measurement_table_state(self):
    pts = getattr(self, '_line_profile_measure_points', []) or []
    mapping = {}
    for pt in pts[:2]:
        try:
            xp, yp, setno = (float(pt[0]), float(pt[1]), int(pt[2]))
        except Exception:
            continue
        mapping.setdefault(setno, []).append((xp, yp, setno))
    self._line_profile_measure_measurements = mapping

def _tab7_measure_press_sync_final2(self, event):
    result = _tab7_measure_press_sync_final(self, event)
    _tab7_sync_measurement_table_state(self)
    return result

def _tab7_measure_motion_sync_final2(self, event):
    result = _tab7_measure_motion_sync_final(self, event)
    _tab7_sync_measurement_table_state(self)
    return result

def _tab7_measure_release_sync_final2(self, event):
    result = _tab7_measure_release_sync_final(self, event)
    _tab7_sync_measurement_table_state(self)
    return result

def _tab7_start_measurement_sync_final2(self):
    result = _tab7_start_measurement_sync_final(self)
    self._line_profile_measure_measurements = {}
    return result

def _tab7_clear_measurement_sync_final2(self, redraw=True):
    result = _tab7_clear_measurement_sync_final(self, redraw=redraw)
    self._line_profile_measure_measurements = {}
    return result
CDIWorkflowApp._line_profile_start_measurement = _tab7_start_measurement_sync_final2
CDIWorkflowApp._line_profile_graph_press = _tab7_measure_press_sync_final2
CDIWorkflowApp._line_profile_graph_motion = _tab7_measure_motion_sync_final2
CDIWorkflowApp._line_profile_graph_release = _tab7_measure_release_sync_final2
CDIWorkflowApp._line_profile_clear_measurement = _tab7_clear_measurement_sync_final2

def _tab7_final_disconnect_event(canvas, event_name):
    if canvas is None:
        return
    try:
        registry = canvas.callbacks.callbacks.get(event_name, {})
        for cid in list(registry.keys()):
            try:
                canvas.mpl_disconnect(cid)
            except Exception:
                pass
    except Exception:
        pass

def _tab7_final_scale(self):
    try:
        sc = self._tab7_private_scale()
        sx, sy = (float(sc[0]), float(sc[1]))
        if np.isfinite(sx) and np.isfinite(sy) and (sx > 0) and (sy > 0):
            return (sx, sy)
    except Exception:
        pass
    v = _tab5_effective_pixel_size_final(self)
    if v is not None and np.isfinite(v) and (v > 0):
        return (float(v), float(v))
    return (1.0, 1.0)

def _tab7_final_full_extent(self):
    stack = getattr(self, 'line_source_stack', None)
    if stack is None:
        stack = getattr(self, 'roi2_stack', None)
    if stack is None:
        return None
    a = np.asarray(stack)
    if a.ndim != 3 or a.shape[0] == 0:
        return None
    h, w = a.shape[1:3]
    sx, sy = _tab7_final_scale(self)
    return (0.0, float(w) * sx, 0.0, float(h) * sy)

def _tab7_final_clamp_view(full, x0, x1, y0, y1):
    fx0, fx1, fy0, fy1 = map(float, full)
    wx = min(abs(float(x1) - float(x0)), fx1 - fx0)
    wy = min(abs(float(y1) - float(y0)), fy1 - fy0)
    if wx <= 0 or wy <= 0:
        return ((fx0, fx1), (fy0, fy1))
    cx = 0.5 * (float(x0) + float(x1))
    cy = 0.5 * (float(y0) + float(y1))
    nx0 = min(max(cx - 0.5 * wx, fx0), fx1 - wx)
    ny0 = min(max(cy - 0.5 * wy, fy0), fy1 - wy)
    return ((nx0, nx0 + wx), (ny0, ny0 + wy))

def _tab7_final_apply_view(self, xlim=None, ylim=None):
    ax = getattr(self, 'line_profile_ax_image', None)
    canvas = getattr(self, 'line_profile_canvas_image', None)
    full = _tab7_final_full_extent(self)
    if ax is None or full is None:
        return
    if xlim is None or ylim is None:
        xlim = (full[0], full[1])
        ylim = (full[2], full[3])
        self._tab7_roi_zoomed = False
        self._tab7_roi_zoom_limits = None
        self._tab7_zoom_view_nm = None
    else:
        (x0, x1), (y0, y1) = _tab7_final_clamp_view(full, xlim[0], xlim[1], ylim[0], ylim[1])
        xlim, ylim = ((x0, x1), (y0, y1))
        full_w = max(full[1] - full[0], 1e-12)
        full_h = max(full[3] - full[2], 1e-12)
        self._tab7_roi_zoomed = x1 - x0 < 0.999999 * full_w or y1 - y0 < 0.999999 * full_h
        self._tab7_roi_zoom_limits = (tuple(xlim), tuple(ylim))
        self._tab7_zoom_view_nm = (tuple(xlim), tuple(ylim))
    ax.set_autoscale_on(False)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    ax.set_axis_on()
    try:
        ax.figure.tight_layout(pad=0.8)
    except Exception:
        pass
    if canvas is not None:
        canvas.draw_idle()

def _tab7_final_reset_zoom(self):
    _tab7_final_apply_view(self)

def _tab7_final_zoom_scroll(self, event):
    ax = getattr(self, 'line_profile_ax_image', None)
    if event is None or getattr(event, 'inaxes', None) is not ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    full = _tab7_final_full_extent(self)
    if full is None:
        return
    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        btn = getattr(event, 'button', None)
        if btn in ('up', 4):
            step = 1.0
        elif btn in ('down', 5):
            step = -1.0
    if step == 0:
        return
    x0, x1 = map(float, ax.get_xlim())
    y0, y1 = map(float, ax.get_ylim())
    xlo, xhi = sorted((x0, x1))
    ylo, yhi = sorted((y0, y1))
    wx = xhi - xlo
    wy = yhi - ylo
    if wx <= 0 or wy <= 0:
        return
    factor = 1.2 ** (-step)
    cx = float(event.xdata)
    cy = float(event.ydata)
    rx = np.clip((cx - xlo) / wx, 0.0, 1.0)
    ry = np.clip((cy - ylo) / wy, 0.0, 1.0)
    nwx = wx * factor
    nwy = wy * factor
    nx0 = cx - rx * nwx
    nx1 = nx0 + nwx
    ny0 = cy - ry * nwy
    ny1 = ny0 + nwy
    if nwx >= 0.999999 * (full[1] - full[0]) and nwy >= 0.999999 * (full[3] - full[2]):
        _tab7_final_reset_zoom(self)
        return 'break'
    (nx0, nx1), (ny0, ny1) = _tab7_final_clamp_view(full, nx0, nx1, ny0, ny1)
    _tab7_final_apply_view(self, (nx0, nx1), (ny0, ny1))
    return 'break'

def _tab7_final_update_image(self, idx=None, preserve_lines=True):
    stack = getattr(self, 'line_source_stack', None)
    if stack is None:
        stack = getattr(self, 'roi2_stack', None)
    if stack is None:
        return
    arr = np.asarray(stack)
    if arr.ndim != 3 or arr.shape[0] == 0:
        return
    n = int(arr.shape[0])
    old_view = getattr(self, '_tab7_zoom_view_nm', None)
    was_zoomed = bool(getattr(self, '_tab7_roi_zoomed', False))
    try:
        i = int(round(float(self.line_profile_frame_var.get()))) if idx is None else int(idx)
    except Exception:
        i = 0
    i = max(0, min(i, n - 1))
    self.line_profile_frame_var.set(i)
    try:
        self.line_profile_frame_slider.set(i)
    except Exception:
        pass
    img = np.asarray(arr[i], dtype=float)
    h, w = img.shape[:2]
    sx, sy = _tab7_final_scale(self)
    im = self.line_profile_image
    im.set_data(img)
    im.set_extent((0.0, float(w) * sx, 0.0, float(h) * sy))
    im.set_clim(0.0, 1.0)
    ax = self.line_profile_ax_image
    ax.set_axis_on()
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    try:
        field = float(self.real_fields_mT[i])
    except Exception:
        field = float(i)
    ax.set_title(f'Secondary ROI — Frame {i + 1}/{n} | Field = {field:+.2f} mT')
    try:
        self.line_profile_frame_label.config(text=f'{i + 1}/{n} | {field:+.2f} mT')
        self.line_profile_colorbar.update_normal(im)
        self.line_profile_colorbar.set_label('Selected Secondary ROI contrast')
    except Exception:
        pass
    if preserve_lines:
        try:
            self._line_profile_redraw_profiles(i)
        except Exception:
            pass
    try:
        self._line_profile_redraw_image_lines()
    except Exception:
        pass
    if was_zoomed and old_view is not None:
        try:
            _tab7_final_apply_view(self, old_view[0], old_view[1])
        except Exception:
            _tab7_final_apply_view(self)
    else:
        _tab7_final_apply_view(self)
    self.line_profile_canvas_image.draw_idle()

def _tab7_final_frame_slider_changed(self, value=None):
    stack = getattr(self, 'line_source_stack', None)
    if stack is None:
        stack = getattr(self, 'roi2_stack', None)
    if stack is None:
        return
    try:
        idx = int(round(float(value))) if value is not None else int(self.line_profile_frame_var.get())
    except Exception:
        idx = 0
    idx = max(0, min(idx, len(stack) - 1))
    ax = getattr(self, 'line_profile_ax_image', None)
    if getattr(self, '_tab7_roi_zoomed', False) and ax is not None:
        try:
            self._tab7_zoom_view_nm = (tuple(map(float, ax.get_xlim())), tuple(map(float, ax.get_ylim())))
            self._tab7_roi_zoom_limits = self._tab7_zoom_view_nm
        except Exception:
            pass
    self._tab7_frame_guard = True
    try:
        self._tab7_update_image(idx=idx, preserve_lines=True)
    finally:
        self._tab7_frame_guard = False

def _tab7_final_redraw_image_lines(self):
    for a in getattr(self, '_line_profile_image_line_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_image_line_artists = []
    sx, sy = _tab7_final_scale(self)
    lines = getattr(self, '_line_profile_lines', []) or []
    measured_sets = set()
    for pt in getattr(self, '_line_profile_measure_points', []) or []:
        try:
            measured_sets.add(int(pt[2]))
        except Exception:
            pass
    for m in getattr(self, '_line_profile_measurements_list', []) or []:
        try:
            measured_sets.add(int(m.get('setno')))
        except Exception:
            pass
    for i, line in enumerate(lines):
        p0 = np.asarray(line.get('p0', (0, 0)), dtype=float)
        p1 = np.asarray(line.get('p1', (0, 0)), dtype=float)
        x0, y0 = (float(p0[0] * sx), float(p0[1] * sy))
        x1, y1 = (float(p1[0] * sx), float(p1[1] * sy))
        setno = i + 1
        base_color = self._line_profile_line_color(i)
        if setno in measured_sets:
            under, = self.line_profile_ax_image.plot([x0, x1], [y0, y1], '-', lw=9.0, color='black', alpha=0.75, zorder=24)
            hi, = self.line_profile_ax_image.plot([x0, x1], [y0, y1], '-', lw=6.0, color='yellow', alpha=0.95, zorder=25)
            self._line_profile_image_line_artists.extend([under, hi])
        else:
            a, = self.line_profile_ax_image.plot([x0, x1], [y0, y1], '-', lw=2.2, color=base_color, zorder=15)
            self._line_profile_image_line_artists.append(a)
    for a in getattr(self, '_line_profile_measure_image_artists', []):
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_image_artists = []
    for k, pt in enumerate(getattr(self, '_line_profile_measure_points', [])[:2]):
        try:
            xp, _yp, setno = (float(pt[0]), float(pt[1]), int(pt[2]))
            r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == setno))
            L = float(r.get('length', 0.0))
            p0 = np.asarray(r.get('p0'), dtype=float)
            p1 = np.asarray(r.get('p1'), dtype=float)
            if L <= 0:
                continue
            frac = float(np.clip(xp / L, 0.0, 1.0))
            xy = p0 + frac * (p1 - p0)
            xy_nm = (float(xy[0] * sx), float(xy[1] * sy))
            hnd = self.line_profile_ax_image.scatter([xy_nm[0]], [xy_nm[1]], s=85, facecolor='yellow', edgecolor='black', linewidth=1.4, zorder=30)
            tag = self.line_profile_ax_image.text(xy_nm[0] + 3.0 * sx, xy_nm[1] + 3.0 * sy, f'M{k + 1}', color='black', fontsize=9, weight='bold', bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='black', alpha=0.9), zorder=31)
            self._line_profile_measure_image_artists.extend([hnd, tag])
        except Exception:
            pass
    self.line_profile_canvas_image.draw_idle()

def _tab7_final_measure_motion(self, event):
    i = getattr(self, '_line_profile_measure_drag_index', None)
    if i is None:
        return
    ax = getattr(self, 'line_profile_ax_graph', None)
    if event is None or event.inaxes is not ax or event.xdata is None:
        return
    pts = getattr(self, '_line_profile_measure_points', []) or []
    if not 0 <= int(i) < len(pts):
        return
    xp, yp, setno = pts[int(i)]
    r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(setno)), None)
    if r is None:
        return
    dnm = np.asarray(r.get('dist_nm', []), dtype=float)
    px = np.asarray(r.get('dist', []), dtype=float)
    prof = np.asarray(r.get('profile', []), dtype=float)
    if dnm.size < 2 or px.size != dnm.size or prof.size != dnm.size:
        return
    xnm = float(np.clip(float(event.xdata), dnm[0], dnm[-1]))
    xp_new = float(np.interp(xnm, dnm, px))
    yp_new = float(np.interp(xnm, dnm, prof))
    pts[int(i)] = (xp_new, yp_new, int(setno))
    self._line_profile_measure_points = pts
    for m in getattr(self, '_line_profile_measurements_list', []) or []:
        if int(m.get('setno', -1)) == int(setno):
            mp = m.get('pts', [])
            if len(mp) > int(i):
                mp[int(i)] = (xp_new, yp_new, int(setno))
    try:
        self._line_profile_update_measurement_table()
    except Exception:
        pass
    try:
        self._line_profile_draw_measurement()
    except Exception:
        pass

def _tab7_final_measure_release(self, event):
    if getattr(self, '_line_profile_measure_drag_index', None) is not None:
        self._line_profile_measure_drag_index = None
        self._line_profile_measure_mode = bool(len(getattr(self, '_line_profile_measure_points', []) or []) < 2)
    try:
        self._line_profile_draw_measurement()
    except Exception:
        pass
CDIWorkflowApp._tab7_private_stack = _tab7_private_stack
CDIWorkflowApp._tab7_private_scale = _tab7_private_scale
CDIWorkflowApp._line_profile_zoom_mousewheel = _tab7_final_zoom_scroll
CDIWorkflowApp._line_profile_reset_zoom = _tab7_final_reset_zoom
CDIWorkflowApp._line_profile_update_image = _tab7_final_update_image
CDIWorkflowApp._line_profile_frame_slider_changed = _tab7_final_frame_slider_changed
CDIWorkflowApp._line_profile_redraw_image_lines = _tab7_final_redraw_image_lines
CDIWorkflowApp._line_profile_graph_motion = _tab7_final_measure_motion
CDIWorkflowApp._line_profile_graph_release = _tab7_final_measure_release
_old_build_ui_tab7_last = CDIWorkflowApp._build_ui

def _build_ui_tab7_last(self):
    _old_build_ui_tab7_last(self)
    try:
        self._tab7_roi_zoomed = False
        self._tab7_roi_zoom_limits = None
        self._tab7_zoom_view_nm = None
        self._tab7_frame_guard = False
    except Exception:
        pass
    try:
        c = getattr(self, 'line_profile_canvas_image', None)
        if c is not None:
            _tab7_final_disconnect_event(c, 'scroll_event')
            c.mpl_connect('scroll_event', self._line_profile_zoom_mousewheel)
    except Exception:
        pass
    try:
        g = getattr(self, 'line_profile_canvas_graph', None)
        if g is not None:
            for ev in ('button_press_event', 'motion_notify_event', 'button_release_event'):
                _tab7_final_disconnect_event(g, ev)
            g.mpl_connect('button_press_event', self._line_profile_graph_press)
            g.mpl_connect('motion_notify_event', self._line_profile_graph_motion)
            g.mpl_connect('button_release_event', self._line_profile_graph_release)
    except Exception:
        pass
    try:
        sl = getattr(self, 'line_profile_frame_slider', None)
        parent = sl.master if sl is not None else None
        if parent is not None:
            children = list(parent.winfo_children())
            for w in children:
                try:
                    w.pack_forget()
                except Exception:
                    pass
            row0 = ttk.Frame(parent)
            row1 = ttk.Frame(parent)
            row0.grid(row=0, column=0, sticky='ew', padx=2, pady=(0, 2))
            row1.grid(row=1, column=0, sticky='ew', padx=2, pady=(0, 2))
            parent.grid_columnconfigure(0, weight=1)

            def _put(w, row, **kw):
                if w is not None:
                    try:
                        w.grid(in_=row, **kw)
                    except Exception:
                        pass
            labels = [w for w in children if isinstance(w, ttk.Label)]
            buttons = [w for w in children if isinstance(w, ttk.Button)]
            scales = [w for w in children if isinstance(w, tk.Scale)]
            spins = [w for w in children if isinstance(w, tk.Spinbox)]
            ordered0 = []
            for w in children:
                if w is sl or w is getattr(self, 'line_profile_frame_label', None) or w is getattr(self, 'line_profile_sets_spin', None):
                    ordered0.append(w)
            col = 0
            for w in children:
                if w is getattr(self, 'line_profile_frame_label', None):
                    _put(w, row0, row=0, column=col, padx=(0, 5), sticky='w')
                    col += 1
                elif w is sl:
                    _put(w, row0, row=0, column=col, padx=(0, 8), sticky='ew')
                    parent.grid_columnconfigure(col, weight=1)
                    col += 1
                elif w is getattr(self, 'line_profile_sets_spin', None):
                    _put(w, row0, row=0, column=col, padx=(0, 6), sticky='w')
                    col += 1
            action_btns = []
            preferred = ['DRAW LINE SETS', 'CLEAR ALL LINES', 'REDO LAST LINE', 'DYNAMIC MEASURING SCALE', 'CLEAR MEASUREMENT', 'EXPORT LINE PROFILES CSV', 'EXPORT MEASUREMENTS CSV']
            for text in preferred:
                for w in children:
                    try:
                        if isinstance(w, ttk.Button) and w.cget('text') == text and (w not in action_btns):
                            action_btns.append(w)
                            break
                    except Exception:
                        pass
            for j, w in enumerate(action_btns):
                _put(w, row1, row=0, column=j, padx=2, sticky='ew')
                row1.grid_columnconfigure(j, weight=1)
            misc = [w for w in children if w not in action_btns and w is not sl and (w is not getattr(self, 'line_profile_frame_label', None)) and (w is not getattr(self, 'line_profile_sets_spin', None))]
            used = []
            col = 0
            for w in misc:
                try:
                    if isinstance(w, ttk.Label):
                        txt = str(w.cget('text'))
                        if txt in ('Frame:', 'No. of sets:') or txt.startswith('No. of'):
                            _put(w, row0, row=0, column=col, padx=(3, 3), sticky='w')
                            col += 1
                except Exception:
                    pass
            parent.update_idletasks()
    except Exception:
        pass
    try:
        if getattr(self, 'line_source_stack', None) is not None:
            self._line_profile_update_image(idx=getattr(self, 'line_profile_frame_var', tk.IntVar(value=0)).get(), preserve_lines=True)
    except Exception:
        pass
CDIWorkflowApp._build_ui = _build_ui_tab7_last
_old_tab7_capture_latest = getattr(CDIWorkflowApp, '_tab7_capture_roi2_final', None)

def _tab7_capture_roi2_authoritative(self):
    if _old_tab7_capture_latest is not None:
        ok = _old_tab7_capture_latest(self)
    else:
        ok = False
    if not ok:
        return ok
    try:
        self.line_source_stack = np.asarray(self.roi2_stack, dtype=np.float64).copy()
        coords = getattr(self, 'roi2_confirmed_coords', None)
        self.line_source_coords = dict(coords) if isinstance(coords, dict) else None
        sc = _tab6_effective_nm_per_output_px(self)
        if sc is None:
            v = _tab5_effective_pixel_size_final(self)
            sc = (v, v) if v is not None else (1.0, 1.0)
        self._tab7_fixed_nm_per_output_px = (float(sc[0]), float(sc[1]))
        self._tab7_roi_zoomed = False
        self._tab7_roi_zoom_limits = None
        self._tab7_zoom_view_nm = None
        idx = int(getattr(self, 'roi2_idx_var', tk.IntVar(value=0)).get())
        idx = max(0, min(idx, len(self.line_source_stack) - 1))
        self.line_profile_frame_var.set(idx)
        self._line_profile_update_image(idx=idx, preserve_lines=False)
    except Exception:
        pass
    return ok
if _old_tab7_capture_latest is not None:
    CDIWorkflowApp._tab7_capture_roi2_final = _tab7_capture_roi2_authoritative

def _tab7_final_stack(self):
    s = getattr(self, 'line_source_stack', None)
    if s is None:
        s = getattr(self, 'roi2_stack', None)
    try:
        a = np.asarray(s, dtype=float)
        if a.ndim == 3 and a.shape[0] > 0:
            return a
    except Exception:
        pass
    return None

def _tab7_final_scale(self):
    sc = getattr(self, '_tab7_fixed_nm_per_output_px', None)
    if sc is not None:
        try:
            sx, sy = (float(sc[0]), float(sc[1]))
            if np.isfinite(sx) and np.isfinite(sy) and (sx > 0) and (sy > 0):
                return (sx, sy)
        except Exception:
            pass
    try:
        v = float(_tab6_effective_nm_per_output_px(self)[0])
        if np.isfinite(v) and v > 0:
            return (v, v)
    except Exception:
        pass
    try:
        v = float(_tab5_effective_pixel_size_final(self))
        if np.isfinite(v) and v > 0:
            return (v, v)
    except Exception:
        pass
    return (1.0, 1.0)

def _tab7_final_init_state(self):
    if not hasattr(self, '_tab7_measure_pairs'):
        self._tab7_measure_pairs = []
    if not hasattr(self, '_tab7_measure_pending'):
        self._tab7_measure_pending = None
    if not hasattr(self, '_tab7_measure_drag'):
        self._tab7_measure_drag = None
    if not hasattr(self, '_tab7_measure_mode'):
        self._tab7_measure_mode = False
    if not hasattr(self, '_tab7_lines_px'):
        self._tab7_lines_px = getattr(self, '_line_profile_lines', []) or []
    if not hasattr(self, '_tab7_measure_artists'):
        self._tab7_measure_artists = []

def _tab7_final_set_frame(self, idx):
    stack = _tab7_final_stack(self)
    if stack is None:
        return
    n = int(stack.shape[0])
    idx = max(0, min(int(idx), n - 1))
    try:
        self.line_profile_frame_var.set(idx)
    except Exception:
        pass
    try:
        self._tab7_frame_ui_var.set(idx + 1)
    except Exception:
        pass
    try:
        self.line_profile_frame_slider.set(idx + 1)
    except Exception:
        pass
    self._tab7_final_render_frame(idx, preserve_zoom=True)

def _tab7_final_slider_changed(self, value=None):
    stack = _tab7_final_stack(self)
    if stack is None:
        return 'break'
    try:
        ui = int(round(float(value if value is not None else self._tab7_frame_ui_var.get())))
    except Exception:
        ui = 1
    idx = max(0, min(ui - 1, len(stack) - 1))
    self._tab7_final_set_frame(idx)
    return 'break'

def _tab7_final_slider_key(self, event=None):
    try:
        cur = int(round(float(self._tab7_frame_ui_var.get())))
    except Exception:
        cur = 1
    step = 1
    if event is not None and getattr(event, 'keysym', '') in ('Left', 'Down'):
        step = -1
    elif event is not None and getattr(event, 'keysym', '') in ('Right', 'Up'):
        step = 1
    self._tab7_final_slider_changed(cur + step)
    return 'break'

def _tab7_final_slider_wheel(self, event):
    widget = getattr(self, 'line_profile_frame_slider', None)
    if widget is None:
        return 'break'
    try:
        pointer = widget.winfo_containing(widget.winfo_pointerx(), widget.winfo_pointery())
        if pointer is not widget:
            return 'break'
    except Exception:
        return 'break'
    delta = getattr(event, 'delta', 0)
    if getattr(event, 'num', None) == 4:
        step = 1
    elif getattr(event, 'num', None) == 5:
        step = -1
    elif delta:
        step = 1 if delta > 0 else -1
    else:
        return 'break'
    try:
        cur = int(round(float(self._tab7_frame_ui_var.get())))
    except Exception:
        cur = 1
    self._tab7_final_slider_changed(cur + step)
    return 'break'

def _tab7_final_render_frame(self, idx, preserve_zoom=True):
    stack = _tab7_final_stack(self)
    if stack is None:
        return
    n = int(stack.shape[0])
    idx = max(0, min(int(idx), n - 1))
    image = np.asarray(stack[idx], dtype=float)
    h, w = image.shape[:2]
    sx, sy = _tab7_final_scale(self)
    if not hasattr(self, '_tab7_view_nm'):
        self._tab7_view_nm = None
    ax = self.line_profile_ax_image
    artist = self.line_profile_image
    artist.set_data(np.clip(image, 0.0, 1.0))
    try:
        cmap = str(self.colormap_var.get()).strip()
        artist.set_cmap(cmap if cmap else 'RdBu_r')
    except Exception:
        artist.set_cmap('RdBu_r')
    artist.set_clim(0.0, 1.0)
    full = (0.0, float(w * sx), 0.0, float(h * sy))
    ax.set_axis_on()
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    if preserve_zoom and getattr(self, '_tab7_view_nm', None) is not None:
        xlim, ylim = self._tab7_view_nm
        x0, x1 = xlim
        y0, y1 = ylim
        fx0, fx1, fy0, fy1 = full
        wx = min(abs(x1 - x0), fx1 - fx0)
        wy = min(abs(y1 - y0), fy1 - fy0)
        cx = 0.5 * (x0 + x1)
        cy = 0.5 * (y0 + y1)
        nx0 = max(fx0, min(cx - 0.5 * wx, fx1 - wx))
        nx1 = nx0 + wx
        ny0 = max(fy0, min(cy - 0.5 * wy, fy1 - wy))
        ny1 = ny0 + wy
        ax.set_xlim(nx0, nx1)
        ax.set_ylim(ny0, ny1)
        self._tab7_view_nm = ((nx0, nx1), (ny0, ny1))
    else:
        ax.set_xlim(full[0], full[1])
        ax.set_ylim(full[2], full[3])
        self._tab7_view_nm = (tuple(ax.get_xlim()), tuple(ax.get_ylim()))
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    try:
        field = float(self.real_fields_mT[idx])
    except Exception:
        field = float(idx)
    ax.set_title(f'Secondary ROI — Frame {idx + 1}/{n} | Field = {field:+.2f} mT')
    try:
        self.line_profile_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT')
        self.line_profile_colorbar.update_normal(artist)
        self.line_profile_colorbar.set_label('Selected Secondary ROI contrast')
    except Exception:
        pass
    _tab7_final_redraw_profiles(self, idx)
    _tab7_final_draw_measurements(self)
    try:
        self.line_profile_canvas_image.draw_idle()
        self.line_profile_canvas_graph.draw_idle()
    except Exception:
        pass

def _tab7_final_extract_profile(self, image, p0, p1):
    x0, y0 = map(float, p0)
    x1, y1 = map(float, p1)
    lp = float(np.hypot(x1 - x0, y1 - y0))
    if lp <= 0:
        ix = max(0, min(image.shape[1] - 1, int(round(x0))))
        iy = max(0, min(image.shape[0] - 1, int(round(y0))))
        return (np.array([0.0]), np.array([float(image[iy, ix])]))
    n = max(2, int(np.ceil(lp * 2.0)) + 1)
    dpx = np.linspace(0.0, lp, n)
    xs = x0 + (x1 - x0) * dpx / lp
    ys = y0 + (y1 - y0) * dpx / lp
    prof = map_coordinates(image, [ys, xs], order=1, mode='nearest')
    return (dpx, np.asarray(prof, dtype=float))

def _tab7_final_redraw_profiles(self, idx=None):
    stack = _tab7_final_stack(self)
    if stack is None:
        return
    try:
        idx = int(self.line_profile_frame_var.get()) if idx is None else int(idx)
    except Exception:
        idx = 0
    idx = max(0, min(idx, len(stack) - 1))
    image = np.asarray(stack[idx], dtype=float)
    sx, sy = _tab7_final_scale(self)
    ax = self.line_profile_ax_graph
    ax.clear()
    ax.set_ylabel('Intensity')
    ax.set_xlabel('Distance (nm)')
    ax.set_title(f'Secondary ROI line profiles — Frame {idx + 1}/{len(stack)}')
    ax.grid(True, alpha=0.22)
    lines = list(getattr(self, '_line_profile_lines', []) or [])
    self._line_profile_results = []
    max_len_nm = 0.0
    for i, line in enumerate(lines):
        p0 = tuple(map(float, line['p0']))
        p1 = tuple(map(float, line['p1']))
        dpx, prof = _tab7_final_extract_profile(self, image, p0, p1)
        px_len = float(dpx[-1]) if dpx.size else 0.0
        phys_len = float(np.hypot((p1[0] - p0[0]) * sx, (p1[1] - p0[1]) * sy))
        dnm = np.linspace(0.0, phys_len, dpx.size) if dpx.size else np.array([0.0])
        max_len_nm = max(max_len_nm, phys_len)
        stats = self._line_profile_extrema_stats(dnm, prof)
        result = {'set': i + 1, 'p0': p0, 'p1': p1, 'length': px_len, 'length_nm': phys_len, 'dist': dpx, 'dist_nm': dnm, 'profile': prof, 'stats': stats}
        self._line_profile_results.append(result)
        color = self._line_profile_line_color(i)
        ln, = ax.plot(dnm, prof, lw=2.0, color=color, label=f'Set {i + 1}', zorder=4)
        ln._line_profile_setno = i + 1
    if self._line_profile_results:
        ax.legend(loc='best')
        xmax = max_len_nm if max_len_nm > 0 else 1.0
        ax.set_xlim(0.0, xmax)
        ax.set_ylim(0.0, 1.05)
    else:
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(0.0, 1.0)
    self._line_profile_current_profile = self._line_profile_results[-1] if self._line_profile_results else None
    _tab7_final_redraw_image_lines(self)
    _tab7_final_update_table(self)

def _tab7_final_redraw_image_lines(self):
    ax = self.line_profile_ax_image
    for a in getattr(self, '_line_profile_image_line_artists', []) or []:
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_image_line_artists = []
    sx, sy = _tab7_final_scale(self)
    for i, line in enumerate(getattr(self, '_line_profile_lines', []) or []):
        p0 = np.asarray(line['p0'], dtype=float)
        p1 = np.asarray(line['p1'], dtype=float)
        q0 = p0 * np.asarray([sx, sy])
        q1 = p1 * np.asarray([sx, sy])
        c = self._line_profile_line_color(i)
        a, = ax.plot([q0[0], q1[0]], [q0[1], q1[1]], '-', lw=2.0, color=c, zorder=12)
        self._line_profile_image_line_artists.append(a)
    self.line_profile_canvas_image.draw_idle()

def _tab7_final_project_curve(self, xnm, yn, set_hint=None, max_screen_px=90.0):
    ax = self.line_profile_ax_graph
    target = ax.transData.transform((float(xnm), float(yn)))
    best = None
    for r in self._line_profile_results:
        if set_hint is not None and int(r.get('set', -1)) != int(set_hint):
            continue
        dnm = np.asarray(r.get('dist_nm', []), dtype=float)
        dpx = np.asarray(r.get('dist', []), dtype=float)
        prof = np.asarray(r.get('profile', []), dtype=float)
        if dnm.size < 2 or dpx.size != dnm.size or prof.size != dnm.size:
            continue
        screen = ax.transData.transform(np.column_stack((dnm, prof)))
        a = screen[:-1]
        b = screen[1:]
        v = b - a
        vv = np.einsum('ij,ij->i', v, v)
        w = target - a
        t = np.einsum('ij,ij->i', w, v) / np.where(vv > 0, vv, 1.0)
        t = np.clip(t, 0.0, 1.0)
        q = a + v * t[:, None]
        d2 = np.einsum('ij,ij->i', q - target, q - target)
        j = int(np.argmin(d2))
        ds = float(np.sqrt(max(d2[j], 0.0)))
        if best is None or ds < best[0]:
            xnm_hit = float(dnm[j] + t[j] * (dnm[j + 1] - dnm[j]))
            y_hit = float(prof[j] + t[j] * (prof[j + 1] - prof[j]))
            xpx_hit = float(np.interp(xnm_hit, dnm, dpx))
            best = (ds, xpx_hit, y_hit, int(r['set']))
    return best if best is not None and best[0] <= max_screen_px else None

def _tab7_final_select_measure_point(self, event):
    best = _tab7_final_project_curve(self, event.xdata, event.ydata, max_screen_px=90.0)
    if best is None:
        self.line_profile_measure_var.set('Click on or close to a plotted profile curve.')
        return
    _ds, xpx, y, setno = best
    pending = self._tab7_measure_pending
    if pending is None:
        self._tab7_measure_pending = (float(xpx), float(y), int(setno))
        self.line_profile_measure_var.set(f'M1 selected on Set {setno}. Click M2 on the same profile.')
    else:
        if int(pending[2]) != int(setno):
            self.line_profile_measure_var.set(f'M1 is on Set {pending[2]}. M2 must be selected on the same profile.')
            return
        pair_id = len(self._tab7_measure_pairs) + 1
        self._tab7_measure_pairs.append({'pair_id': pair_id, 'm1_id': 2 * pair_id - 1, 'm2_id': 2 * pair_id, 'setno': int(setno), 'pts': [pending, (float(xpx), float(y), int(setno))]})
        self._tab7_measure_pending = None
        self.line_profile_measure_var.set(f'M{2 * pair_id - 1}–M{2 * pair_id} selected. Drag either endpoint to adjust.')
        self._tab7_measure_mode = False
    _tab7_final_draw_measurements(self)

def _tab7_final_start_measurement(self):
    if not getattr(self, '_line_profile_results', None):
        self.line_profile_measure_var.set('Draw at least one line on ROI2 first.')
        return
    self._tab7_final_init_state(self)
    self._tab7_measure_pending = None
    self._tab7_measure_drag = None
    self._tab7_measure_mode = True
    self.line_profile_measure_var.set('DYNAMIC MEASURING SCALE: click M1 and M2 on the SAME profile. Then drag either endpoint.')

def _tab7_final_graph_press(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph or event.button != 1:
        return
    self._tab7_final_init_state(self)
    for pi, m in enumerate(self._tab7_measure_pairs):
        r = next((rr for rr in self._line_profile_results if int(rr.get('set', -1)) == int(m['setno'])), None)
        if r is None:
            continue
        dnm = np.asarray(r.get('dist_nm', []), dtype=float)
        dpx = np.asarray(r.get('dist', []), dtype=float)
        for k, pt in enumerate(m['pts'][:2]):
            xnm = float(np.interp(float(pt[0]), dpx, dnm))
            disp = self.line_profile_ax_graph.transData.transform((xnm, float(pt[1])))
            if np.hypot(event.x - disp[0], event.y - disp[1]) <= 24.0:
                self._tab7_measure_drag = (pi, k)
                self._tab7_measure_mode = False
                self.line_profile_measure_var.set(f"Dragging M{(m['m1_id'] if k == 0 else m['m2_id'])}. Move horizontally along the profile and release.")
                return
    if self._tab7_measure_mode:
        _tab7_final_select_measure_point(self, event)

def _tab7_final_graph_motion(self, event):
    drag = getattr(self, '_tab7_measure_drag', None)
    if drag is None or event is None or event.inaxes is not self.line_profile_ax_graph or (event.xdata is None):
        return
    pi, k = drag
    if not 0 <= pi < len(self._tab7_measure_pairs):
        return
    m = self._tab7_measure_pairs[pi]
    r = next((rr for rr in self._line_profile_results if int(rr.get('set', -1)) == int(m['setno'])), None)
    if r is None:
        return
    dnm = np.asarray(r.get('dist_nm', []), float)
    dpx = np.asarray(r.get('dist', []), float)
    prof = np.asarray(r.get('profile', []), float)
    if dnm.size < 2:
        return
    xnm = float(np.clip(event.xdata, dnm[0], dnm[-1]))
    xpx = float(np.interp(xnm, dnm, dpx))
    y = float(np.interp(xpx, dpx, prof))
    m['pts'][k] = (xpx, y, int(m['setno']))
    _tab7_final_draw_measurements(self)

def _tab7_final_graph_release(self, event=None):
    if getattr(self, '_tab7_measure_drag', None) is not None:
        self.line_profile_measure_var.set('Endpoint fixed. Drag another endpoint or start a new measurement pair.')
    self._tab7_measure_drag = None

def _tab7_final_draw_measurements(self):
    for a in getattr(self, '_line_profile_measure_artists', []) or []:
        try:
            a.remove()
        except Exception:
            pass
    self._line_profile_measure_artists = []
    ax = self.line_profile_ax_graph
    iax = self.line_profile_ax_image
    for a in getattr(self, '_tab7_measure_image_artists', []) or []:
        try:
            a.remove()
        except Exception:
            pass
    self._tab7_measure_image_artists = []
    sx, sy = _tab7_final_scale(self)
    for m in self._tab7_measure_pairs:
        r = next((rr for rr in self._line_profile_results if int(rr.get('set', -1)) == int(m['setno'])), None)
        if r is None:
            continue
        dnm = np.asarray(r.get('dist_nm', []), float)
        dpx = np.asarray(r.get('dist', []), float)
        prof = np.asarray(r.get('profile', []), float)
        if dnm.size < 2:
            continue
        x0nm = float(np.interp(m['pts'][0][0], dpx, dnm))
        x1nm = float(np.interp(m['pts'][1][0], dpx, dnm))
        y0 = float(m['pts'][0][1])
        y1 = float(m['pts'][1][1])
        line, = ax.plot([x0nm, x1nm], [y0, y1], color='blue', lw=2.5, zorder=20)
        self._line_profile_measure_artists.append(line)
        for pid, xn, yy, col in ((m['m1_id'], x0nm, y0, 'gold'), (m['m2_id'], x1nm, y1, 'lime')):
            sc = ax.scatter([xn], [yy], s=60, color=col, edgecolors='black', linewidths=1.1, zorder=22)
            self._line_profile_measure_artists.append(sc)
            tx = ax.annotate(f'M{pid}', (xn, yy), xytext=(5, 7), textcoords='offset points', fontsize=8, fontweight='bold', color='black', bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='black', alpha=0.9), zorder=23)
            self._line_profile_measure_artists.append(tx)
        ylo, yhi = ax.get_ylim()
        ybar = yhi - 0.08 * max(yhi - ylo, 1e-09) - 0.04 * max(0, m['pair_id'] - 1) * (yhi - ylo)
        ar = ax.annotate('', xy=(x1nm, ybar), xytext=(x0nm, ybar), arrowprops=dict(arrowstyle='<->', color='blue', lw=1.8))
        self._line_profile_measure_artists.append(ar)
        lab = ax.text((x0nm + x1nm) / 2, ybar, f"M{m['m1_id']}–M{m['m2_id']}: ΔL={abs(x1nm - x0nm):.6g} nm", ha='center', va='bottom', fontsize=9, color='blue', fontweight='bold', bbox=dict(boxstyle='round,pad=0.16', facecolor='white', edgecolor='blue', alpha=0.88), zorder=24)
        self._line_profile_measure_artists.append(lab)
        p0 = np.asarray(r['p0'], float)
        p1 = np.asarray(r['p1'], float)
        Lpx = float(r['length'])
        q = []
        for pt in m['pts'][:2]:
            frac = float(np.clip(pt[0] / Lpx, 0, 1))
            qp = p0 + frac * (p1 - p0)
            q.append(qp * np.asarray([sx, sy]))
        q0, q1 = q
        shadow, = iax.plot([q0[0], q1[0]], [q0[1], q1[1]], '-', lw=7, color='black', alpha=0.75, zorder=30)
        yellow, = iax.plot([q0[0], q1[0]], [q0[1], q1[1]], '-', lw=4, color='yellow', zorder=31)
        self._tab7_measure_image_artists += [shadow, yellow]
        for qq in (q0, q1):
            mk, = iax.plot([qq[0]], [qq[1]], 'o', ms=6.5, mfc='yellow', mec='black', mew=1.0, zorder=32)
            self._tab7_measure_image_artists.append(mk)
    if self._tab7_measure_pending is not None:
        m = self._tab7_measure_pending
        r = next((rr for rr in self._line_profile_results if int(rr.get('set', -1)) == int(m[2])), None)
        if r is not None:
            xnm = float(np.interp(m[0], np.asarray(r['dist'], float), np.asarray(r['dist_nm'], float)))
            sc = ax.scatter([xnm], [m[1]], s=60, color='gold', edgecolors='black', zorder=24)
            self._line_profile_measure_artists.append(sc)
            tx = ax.annotate('M1', (xnm, m[1]), xytext=(5, 7), textcoords='offset points', fontsize=8, fontweight='bold', color='black', bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='black', alpha=0.9), zorder=25)
            self._line_profile_measure_artists.append(tx)
    _tab7_final_update_table(self)
    try:
        self.line_profile_canvas_graph.draw_idle()
        self.line_profile_canvas_image.draw_idle()
    except Exception:
        pass

def _tab7_final_update_table(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    for item in tree.get_children():
        tree.delete(item)
    rows = []
    for m in self._tab7_measure_pairs:
        r = next((rr for rr in self._line_profile_results if int(rr.get('set', -1)) == int(m['setno'])), None)
        if r is None:
            continue
        dnm = np.asarray(r.get('dist_nm', []), float)
        dpx = np.asarray(r.get('dist', []), float)
        x0 = float(np.interp(m['pts'][0][0], dpx, dnm))
        x1 = float(np.interp(m['pts'][1][0], dpx, dnm))
        rows.append((f"M{m['m1_id']}–M{m['m2_id']}", int(m['setno']), f"{float(r.get('length_nm', 0)):.6g}", f'{x0:.6g}', f'{x1:.6g}', f'{abs(x1 - x0):.6g}', f"{m['pts'][0][1]:.6g}", f"{m['pts'][1][1]:.6g}", 'Measured'))
    for row in rows:
        tree.insert('', 'end', values=row)
    if self._tab7_measure_pending is not None:
        m = self._tab7_measure_pending
        r = next((rr for rr in self._line_profile_results if int(rr.get('set', -1)) == int(m[2])), None)
        if r is not None:
            xnm = float(np.interp(m[0], np.asarray(r['dist'], float), np.asarray(r['dist_nm'], float)))
            tree.insert('', 'end', values=('M1', int(m[2]), '—', f'{xnm:.6g}', '—', '—', f'{m[1]:.6g}', '—', 'Pending'))

def _tab7_final_clear_measurement(self):
    self._tab7_measure_pairs = []
    self._tab7_measure_pending = None
    self._tab7_measure_drag = None
    self._tab7_measure_mode = False
    self.line_profile_measure_var.set('Measurement cleared. Click DYNAMIC MEASURING SCALE to select M1 and M2.')
    _tab7_final_draw_measurements(self)

def _tab7_final_reset_image_zoom(self):
    stack = _tab7_final_stack(self)
    if stack is None:
        return
    try:
        idx = int(self.line_profile_frame_var.get())
    except Exception:
        idx = 0
    img = np.asarray(stack[max(0, min(idx, len(stack) - 1))])
    h, w = img.shape[:2]
    sx, sy = _tab7_final_scale(self)
    full = (0.0, float(w * sx), 0.0, float(h * sy))
    self._tab7_view_nm = ((full[0], full[1]), (full[2], full[3]))
    ax = self.line_profile_ax_image
    ax.set_xlim(full[0], full[1])
    ax.set_ylim(full[2], full[3])
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    self.line_profile_canvas_image.draw_idle()

def _tab7_final_image_scroll(self, event):
    ax = self.line_profile_ax_image
    if event is None or event.inaxes is not ax or event.xdata is None or (event.ydata is None):
        return
    stack = _tab7_final_stack(self)
    if stack is None:
        return
    step = getattr(event, 'step', None)
    if step is None:
        d = getattr(event, 'button', None)
        step = 1 if d == 'up' else -1 if d == 'down' else 0
    factor = 0.85 if step > 0 else 1 / 0.85
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    cx, cy = (float(event.xdata), float(event.ydata))
    nx0 = cx + (x0 - cx) * factor
    nx1 = cx + (x1 - cx) * factor
    ny0 = cy + (y0 - cy) * factor
    ny1 = cy + (y1 - cy) * factor
    img = np.asarray(stack[int(self.line_profile_frame_var.get())])
    h, w = img.shape[:2]
    fx0, fx1, fy0, fy1 = (0.0, w * _tab7_final_scale(self)[0], 0.0, h * _tab7_final_scale(self)[1])
    wx = min(nx1 - nx0, fx1 - fx0)
    wy = min(ny1 - ny0, fy1 - fy0)
    if wx >= fx1 - fx0 and wy >= fy1 - fy0:
        _tab7_final_reset_image_zoom(self)
        return
    nx0 = max(fx0, min(nx0, fx1 - wx))
    nx1 = nx0 + wx
    ny0 = max(fy0, min(ny0, fy1 - wy))
    ny1 = ny0 + wy
    ax.set_xlim(nx0, nx1)
    ax.set_ylim(ny0, ny1)
    ax.set_aspect('equal', adjustable='box')
    self._tab7_view_nm = ((nx0, nx1), (ny0, ny1))
    self.line_profile_canvas_image.draw_idle()

def _tab7_final_build_wrapper(self):
    _TAB7_FINAL_BASE_BUILD(self)
    _tab7_final_init_state(self)
    stack = _tab7_final_stack(self)
    n = int(stack.shape[0]) if stack is not None else 0
    self._tab7_frame_ui_var = tk.IntVar(value=1 if n else 1)
    slider = getattr(self, 'line_profile_frame_slider', None)
    if slider is not None:
        slider.configure(variable=self._tab7_frame_ui_var, from_=1, to=max(1, n), showvalue=True, resolution=1, command=self._line_profile_frame_slider_changed)
        for seq in ('<MouseWheel>', '<Button-4>', '<Button-5>', '<KeyPress-Up>', '<KeyPress-Down>', '<KeyPress-Left>', '<KeyPress-Right>'):
            try:
                slider.unbind(seq)
            except Exception:
                pass
        slider.bind('<MouseWheel>', self._tab7_frame_slider_wheel, add='+')
        slider.bind('<Button-4>', self._tab7_frame_slider_wheel, add='+')
        slider.bind('<Button-5>', self._tab7_frame_slider_wheel, add='+')
    gc = getattr(self, 'line_profile_canvas_graph', None)
    if gc is not None:
        _final_disconnect_canvas_event(gc, 'scroll_event')
        try:
            gc.get_tk_widget().unbind('<MouseWheel>')
            gc.get_tk_widget().unbind('<Button-4>')
            gc.get_tk_widget().unbind('<Button-5>')
        except Exception:
            pass
    if gc is not None:
        _final_disconnect_canvas_event(gc, 'button_press_event')
        _final_disconnect_canvas_event(gc, 'motion_notify_event')
        _final_disconnect_canvas_event(gc, 'button_release_event')
        gc.mpl_connect('button_press_event', self._line_profile_graph_press)
        gc.mpl_connect('motion_notify_event', self._line_profile_graph_motion)
        gc.mpl_connect('button_release_event', self._line_profile_graph_release)
    ic = getattr(self, 'line_profile_canvas_image', None)
    if ic is not None:
        _final_disconnect_canvas_event(ic, 'scroll_event')
        ic.mpl_connect('scroll_event', self._tab7_image_scroll)
    if stack is not None and n:
        self._tab7_final_set_frame(0)

def _tab7_final_refresh_after_roi2(self, *args, **kwargs):
    try:
        ok = _TAB7_FINAL_OLD_CAPTURE(self, *args, **kwargs)
    except Exception:
        ok = False
    stack = getattr(self, 'line_source_stack', None)
    if stack is None:
        stack = getattr(self, 'roi2_stack', None)
    if stack is not None:
        a = np.asarray(stack)
        if a.ndim == 3 and a.shape[0]:
            self._tab7_final_init_state(self)
            self._tab7_measure_pairs = []
            self._tab7_measure_pending = None
            self._tab7_measure_drag = None
            self._tab7_measure_mode = False
            try:
                idx = max(0, min(int(getattr(self, 'roi2_idx_var').get()), a.shape[0] - 1))
            except Exception:
                idx = 0
            self._tab7_final_set_frame(idx)
            try:
                self.nb.select(self.tab_line_profile)
            except Exception:
                pass
            return True
    return ok
_TAB7_FINAL_BASE_BUILD = CDIWorkflowApp._build_ui
_TAB7_FINAL_OLD_CAPTURE = getattr(CDIWorkflowApp, '_tab7_capture_roi2', None)
CDIWorkflowApp._tab7_frame_slider_changed = _tab7_final_slider_changed
CDIWorkflowApp._tab7_frame_slider_wheel = _tab7_final_slider_wheel
CDIWorkflowApp._tab7_image_scroll = _tab7_final_image_scroll
CDIWorkflowApp._tab7_final_set_frame = _tab7_final_set_frame
CDIWorkflowApp._tab7_final_render_frame = _tab7_final_render_frame
CDIWorkflowApp._line_profile_update_image = _tab7_final_render_frame
CDIWorkflowApp._line_profile_redraw_profiles = _tab7_final_redraw_profiles
CDIWorkflowApp._line_profile_start_measurement = _tab7_final_start_measurement
CDIWorkflowApp._line_profile_graph_press = _tab7_final_graph_press
CDIWorkflowApp._line_profile_graph_motion = _tab7_final_graph_motion
CDIWorkflowApp._line_profile_graph_release = _tab7_final_graph_release
CDIWorkflowApp._line_profile_draw_measurement = _tab7_final_draw_measurements
CDIWorkflowApp._line_profile_update_measurement_table = _tab7_final_update_table
CDIWorkflowApp._line_profile_update_table = _tab7_final_update_table
CDIWorkflowApp._line_profile_clear_measurement = _tab7_final_clear_measurement
CDIWorkflowApp._line_profile_reset_zoom = _tab7_final_reset_image_zoom
CDIWorkflowApp._tab7_final_reset_image_zoom = _tab7_final_reset_image_zoom
CDIWorkflowApp._build_ui = _tab7_final_build_wrapper
if _TAB7_FINAL_OLD_CAPTURE is not None:

    def _tab7_capture_final_wrapper(self, *args, **kwargs):
        ok = _TAB7_FINAL_OLD_CAPTURE(self, *args, **kwargs)
        try:
            stack = _tab7_final_stack(self)
            if stack is not None:
                self._tab7_final_init_state(self)
                self._tab7_measure_pairs = []
                self._tab7_measure_pending = None
                self._tab7_measure_drag = None
                self._tab7_measure_mode = False
                idx = max(0, min(int(getattr(self, 'roi2_idx_var').get()), len(stack) - 1))
                self._tab7_final_set_frame(idx)
        except Exception:
            pass
        return ok
    CDIWorkflowApp._tab7_capture_roi2 = _tab7_capture_final_wrapper

def _tab7_final_configure_table(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    cols = ('Set', 'Length (nm)', 'M1 (nm)', 'M2 (nm)', 'ΔL (nm)', 'I(M1)', 'I(M2)', 'Status')
    try:
        tree.configure(columns=cols, show='headings')
        widths = (55, 105, 95, 95, 95, 85, 85, 90)
        for col, wid in zip(cols, widths):
            tree.heading(col, text=col)
            tree.column(col, width=wid, anchor='center')
    except Exception:
        pass

def _tab7_final_update_table8(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    _tab7_final_configure_table(self)
    for item in tree.get_children():
        tree.delete(item)
    for m in getattr(self, '_tab7_measure_pairs', []) or []:
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(m.get('setno', -1))), None)
        if r is None:
            continue
        dpx = np.asarray(r.get('dist', []), dtype=float)
        dnm = np.asarray(r.get('dist_nm', []), dtype=float)
        if dpx.size < 2 or dnm.size != dpx.size:
            continue
        x0 = float(np.interp(float(m['pts'][0][0]), dpx, dnm))
        x1 = float(np.interp(float(m['pts'][1][0]), dpx, dnm))
        tree.insert('', 'end', values=(int(m['setno']), f"{float(r.get('length_nm', 0.0)):.6g}", f'{x0:.6g}', f'{x1:.6g}', f'{abs(x1 - x0):.6g}', f"{float(m['pts'][0][1]):.6g}", f"{float(m['pts'][1][1]):.6g}", 'Measured'))
    pending = getattr(self, '_tab7_measure_pending', None)
    if pending is not None:
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(pending[2])), None)
        if r is not None:
            dpx = np.asarray(r.get('dist', []), dtype=float)
            dnm = np.asarray(r.get('dist_nm', []), dtype=float)
            xnm = float(np.interp(float(pending[0]), dpx, dnm)) if dpx.size else float(pending[0])
            tree.insert('', 'end', values=(int(pending[2]), '—', f'{xnm:.6g}', '—', '—', f'{float(pending[1]):.6g}', '—', 'Pending'))

def _tab7_final_clear_all_lines(self):
    try:
        self._line_profile_lines = []
    except Exception:
        pass
    self._line_profile_results = []
    self._tab7_measure_pairs = []
    self._tab7_measure_pending = None
    self._tab7_measure_drag = None
    self._tab7_measure_mode = False
    self._line_profile_active = False
    self._line_profile_redraw_profiles()
    if hasattr(self, 'line_profile_status_var'):
        self.line_profile_status_var.set('All ROI2 line profiles cleared.')
    if hasattr(self, 'line_profile_measure_var'):
        self.line_profile_measure_var.set('Measurement cleared. Draw line profiles, then start Dynamic Measuring Scale.')

def _tab7_final_redo_last(self):
    lines = getattr(self, '_line_profile_lines', []) or []
    if lines:
        lines = list(lines[:-1])
        self._line_profile_lines = lines
        self._line_profile_results = []
        self._tab7_measure_pairs = []
        self._tab7_measure_pending = None
        self._tab7_measure_drag = None
        self._tab7_measure_mode = False
        self._line_profile_redraw_profiles()
        if hasattr(self, 'line_profile_status_var'):
            self.line_profile_status_var.set('Last ROI2 line removed. Draw again as required.')
CDIWorkflowApp._line_profile_update_measurement_table = _tab7_final_update_table8
CDIWorkflowApp._line_profile_update_table = _tab7_final_update_table8
CDIWorkflowApp._line_profile_clear_all_lines = _tab7_final_clear_all_lines
CDIWorkflowApp._line_profile_redo_last = _tab7_final_redo_last

def _tab7_duplicate_roi2_from_tab6(self):
    """
    Make an independent snapshot of the processed/confirmed ROI2 from Tab 6.

    Analysis uses the copied intensity stack, while the copied color settings
    are used only for visualization. The original Tab-6 ROI2 is never modified.
    """
    stack = getattr(self, 'roi2_stack', None)
    coords = getattr(self, 'roi2_confirmed_coords', None)
    if stack is None or coords is None:
        return False
    try:
        arr = np.asarray(stack, dtype=np.float64)
    except Exception:
        return False
    if arr.ndim != 3 or arr.shape[0] == 0:
        return False
    self._tab7_roi2_copy_stack = arr.copy()
    self._tab7_roi2_copy_coords = dict(coords)
    cmap_name = 'RdBu_r'
    clim = (0.0, 1.0)
    try:
        roi2_im = getattr(self, 'roi2_image', None)
        if roi2_im is not None:
            cmap = roi2_im.get_cmap()
            if cmap is not None:
                cmap_name = str(getattr(cmap, 'name', cmap_name))
            c0, c1 = roi2_im.get_clim()
            if np.isfinite(c0) and np.isfinite(c1) and (c1 > c0):
                clim = (float(c0), float(c1))
    except Exception:
        pass
    try:
        selected_cmap = str(self.colormap_var.get()).strip()
        if selected_cmap:
            cmap_name = selected_cmap
    except Exception:
        pass
    self._tab7_roi2_copy_cmap = cmap_name
    self._tab7_roi2_copy_clim = clim
    try:
        sc = _tab6_effective_nm_per_output_px(self)
    except Exception:
        sc = None
    if sc is None:
        try:
            v = float(_tab5_effective_pixel_size_final(self))
            sc = (v, v) if np.isfinite(v) and v > 0 else (1.0, 1.0)
        except Exception:
            sc = (1.0, 1.0)
    self._tab7_fixed_nm_per_output_px = (float(sc[0]), float(sc[1]))
    try:
        idx = int(round(float(self.roi2_idx_var.get())))
    except Exception:
        idx = 0
    idx = max(0, min(idx, arr.shape[0] - 1))
    self._tab7_roi2_copy_frame = idx
    self._line_profile_lines = []
    self._line_profile_results = []
    self._line_profile_measurements_list = []
    self._line_profile_pending_point = None
    self._line_profile_measure_mode = False
    self._line_profile_measure_drag_pair = None
    self._line_profile_measure_drag_point = None
    if hasattr(self, '_tab7_measure_pairs'):
        self._tab7_measure_pairs = []
    if hasattr(self, '_tab7_measure_pending'):
        self._tab7_measure_pending = None
    if hasattr(self, '_tab7_measure_drag'):
        self._tab7_measure_drag = None
    if hasattr(self, '_tab7_measure_mode'):
        self._tab7_measure_mode = False
    self.line_source_stack = self._tab7_roi2_copy_stack.copy()
    self.line_source_coords = dict(self._tab7_roi2_copy_coords)
    try:
        self.line_profile_frame_var.set(idx)
    except Exception:
        pass
    try:
        self._tab7_frame_ui_var.set(idx + 1)
    except Exception:
        pass
    try:
        self.line_profile_frame_slider.configure(from_=1, to=max(1, arr.shape[0]), state='normal', resolution=1)
        self.line_profile_frame_slider.set(idx + 1)
    except Exception:
        pass
    try:
        self.line_profile_image.set_cmap(self._tab7_roi2_copy_cmap)
        self.line_profile_image.set_clim(*self._tab7_roi2_copy_clim)
    except Exception:
        pass
    try:
        self._line_profile_update_image(idx=idx, preserve_lines=False)
    except Exception:
        pass
    try:
        self.line_profile_status_var.set(f'✓ Independent copy of confirmed Tab-6 ROI2 | frames={arr.shape[0]} | crop={arr.shape[2]} × {arr.shape[1]} px | Color={self._tab7_roi2_copy_cmap} | scale={self._tab7_fixed_nm_per_output_px[0]:.6g} × {self._tab7_fixed_nm_per_output_px[1]:.6g} nm/output-px')
    except Exception:
        pass
    return True
CDIWorkflowApp._tab7_duplicate_roi2_from_tab6 = _tab7_duplicate_roi2_from_tab6

def _tab7_duplicate_authoritative_stack(self):
    s = getattr(self, '_tab7_roi2_copy_stack', None)
    if s is None:
        s = getattr(self, 'line_source_stack', None)
    if s is None:
        s = getattr(self, 'roi2_stack', None)
    try:
        a = np.asarray(s, dtype=float)
        if a.ndim == 3 and a.shape[0] > 0:
            return a
    except Exception:
        pass
    return None
CDIWorkflowApp._tab7_final_stack = _tab7_duplicate_authoritative_stack
CDIWorkflowApp._tab7_private_stack = _tab7_duplicate_authoritative_stack
_TAB7_DUP_OLD_RENDER = CDIWorkflowApp._tab7_final_render_frame

def _tab7_duplicate_render(self, idx, preserve_zoom=True):
    try:
        cmap = getattr(self, '_tab7_roi2_copy_cmap', None)
        clim = getattr(self, '_tab7_roi2_copy_clim', None)
        if cmap and getattr(self, 'line_profile_image', None) is not None:
            self.line_profile_image.set_cmap(cmap)
        if clim is not None and getattr(self, 'line_profile_image', None) is not None:
            self.line_profile_image.set_clim(*clim)
    except Exception:
        pass
    return _TAB7_DUP_OLD_RENDER(self, idx, preserve_zoom=preserve_zoom)
CDIWorkflowApp._tab7_final_render_frame = _tab7_duplicate_render
CDIWorkflowApp._line_profile_update_image = _tab7_duplicate_render
_OLD_CONFIRM_SECONDARY_DUP = CDIWorkflowApp.confirm_secondary_roi_from_processing

def _confirm_secondary_duplicate_final(self, *args, **kwargs):
    result = _OLD_CONFIRM_SECONDARY_DUP(self, *args, **kwargs)
    try:
        if result is not False:
            self._tab7_duplicate_roi2_from_tab6()
    except Exception as exc:
        try:
            self.log(f'Tab 7 ROI2 duplicate-copy warning: {type(exc).__name__}: {exc}')
        except Exception:
            pass
    return result
CDIWorkflowApp.confirm_secondary_roi_from_processing = _confirm_secondary_duplicate_final

def _tab7_duplicate_set_frame(self, idx):
    stack = self._tab7_duplicate_authoritative_stack()
    if stack is None:
        return
    n = int(stack.shape[0])
    idx = max(0, min(int(idx), n - 1))
    try:
        self.line_profile_frame_var.set(idx)
    except Exception:
        pass
    try:
        self._tab7_frame_ui_var.set(idx + 1)
    except Exception:
        pass
    try:
        self._tab7_roi2_copy_frame = idx
    except Exception:
        pass
    self._tab7_duplicate_render(idx, preserve_zoom=True)

def _tab7_duplicate_frame_changed(self, value=None):
    stack = self._tab7_duplicate_authoritative_stack()
    if stack is None:
        return 'break'
    try:
        ui = int(round(float(value if value is not None else self._tab7_frame_ui_var.get())))
    except Exception:
        ui = 1
    idx = max(0, min(ui - 1, len(stack) - 1))
    self._tab7_duplicate_set_frame(idx)
    return 'break'
CDIWorkflowApp._tab7_final_set_frame = _tab7_duplicate_set_frame
CDIWorkflowApp._tab7_final_slider_changed = _tab7_duplicate_frame_changed
CDIWorkflowApp._line_profile_frame_slider_changed = _tab7_duplicate_frame_changed

def _tab7_compact_action_buttons(self):
    tab = getattr(self, 'tab_line_profile', None)
    if tab is None:
        return

    def walk(widget):
        yield widget
        try:
            for ch in widget.winfo_children():
                yield from walk(ch)
        except Exception:
            pass
    found = []
    for w in walk(tab):
        try:
            if isinstance(w, (ttk.Button, tk.Button)):
                found.append(w)
        except Exception:
            pass
    for b in found:
        try:
            text = str(b.cget('text'))
        except Exception:
            continue
        replacements = {'DYNAMIC MEASURING SCALE': 'DYNAMIC SCALE', 'EXPORT LINE PROFILES CSV': 'EXPORT PROFILES', 'EXPORT RESULTS TABLE CSV': 'EXPORT RESULTS', 'EXPORT MEASUREMENTS CSV': 'EXPORT MEASUREMENTS', 'CLEAR MEASUREMENT': 'CLEAR MEASURE', 'CLEAR ALL LINES': 'CLEAR LINES', 'REDO LAST LINE': 'REDO LAST', 'DRAW LINE SETS': 'DRAW LINES'}
        new_text = text
        for old, new in replacements.items():
            new_text = new_text.replace(old, new)
        try:
            if new_text != text:
                b.configure(text=new_text)
        except Exception:
            pass
        try:
            b.configure(width=16)
        except Exception:
            pass
    try:
        for child in tab.winfo_children():
            if isinstance(child, ttk.Frame):
                try:
                    child.pack_configure(fill='x', expand=False, pady=2)
                except Exception:
                    pass
    except Exception:
        pass
_OLD_BUILD_UI_DUP_FINAL = CDIWorkflowApp._build_ui

def _build_ui_duplicate_roi2_final(self, *args, **kwargs):
    result = _OLD_BUILD_UI_DUP_FINAL(self, *args, **kwargs)
    self._tab7_roi2_copy_stack = None
    self._tab7_roi2_copy_coords = None
    self._tab7_roi2_copy_cmap = 'RdBu_r'
    self._tab7_roi2_copy_clim = (0.0, 1.0)
    self._tab7_roi2_copy_frame = 0
    try:
        self._tab7_compact_action_buttons()
    except Exception:
        pass
    try:
        if getattr(self, 'roi2_stack', None) is not None and getattr(self, 'roi2_confirmed_coords', None) is not None:
            self._tab7_duplicate_roi2_from_tab6()
    except Exception:
        pass
    return result
CDIWorkflowApp._build_ui = _build_ui_duplicate_roi2_final
CDIWorkflowApp._tab7_compact_action_buttons = _tab7_compact_action_buttons

def _tab7_refresh_duplicate_on_select(self, event=None):
    try:
        if event is not None and event.widget.select() != str(self.tab_line_profile):
            return
        if getattr(self, '_tab7_roi2_copy_stack', None) is None:
            self._tab7_duplicate_roi2_from_tab6()
            return
        try:
            idx = int(self.line_profile_frame_var.get())
        except Exception:
            idx = 0
        stack = self._tab7_duplicate_authoritative_stack()
        if stack is not None:
            idx = max(0, min(idx, len(stack) - 1))
            self._tab7_duplicate_render(idx, preserve_zoom=True)
    except Exception:
        pass
try:
    CDIWorkflowApp._tab7_refresh_duplicate_on_select = _tab7_refresh_duplicate_on_select
    _DUP_OLD_BUILD = CDIWorkflowApp._build_ui

    def _build_ui_duplicate_final(self, *args, **kwargs):
        result = _DUP_OLD_BUILD(self, *args, **kwargs)
        try:
            self.nb.bind('<<NotebookTabChanged>>', self._tab7_refresh_duplicate_on_select, add='+')
        except Exception:
            pass
        return result
    CDIWorkflowApp._build_ui = _build_ui_duplicate_final
except Exception:
    pass

def _tab7_fetch_confirmed_roi2_from_tab6(self, auto_select=False):
    """
    Snapshot the confirmed processed ROI2 from Tab 6.

    The copy contains:
      - all processed ROI2 frames
      - confirmed ROI2 geometry
      - Tab-6 ROI2 colormap
      - Tab-6 ROI2 color limits
      - exact Tab-6 nm/output-pixel scale

    Tab 7 then performs all line-profile analysis on this independent copy.
    """
    stack = getattr(self, 'roi2_stack', None)
    coords = getattr(self, 'roi2_confirmed_coords', None)
    if stack is None or coords is None:
        try:
            self.line_profile_status_var.set('No confirmed ROI2 in Tab 6. Select ROI2 and click CONFIRM SECONDARY ROI first.')
        except Exception:
            pass
        return False
    try:
        arr = np.asarray(stack, dtype=np.float64)
    except Exception as exc:
        try:
            self.line_profile_status_var.set(f'ROI2 copy error: {exc}')
        except Exception:
            pass
        return False
    if arr.ndim != 3 or arr.shape[0] == 0:
        try:
            self.line_profile_status_var.set(f'Invalid confirmed ROI2 stack shape: {arr.shape}')
        except Exception:
            pass
        return False
    self._tab7_roi2_copy_stack = arr.copy()
    self._tab7_roi2_copy_coords = dict(coords)
    cmap_name = 'RdBu_r'
    clim = (0.0, 1.0)
    try:
        roi_im = getattr(self, 'roi2_image', None)
        if roi_im is not None:
            cmap_obj = roi_im.get_cmap()
            if cmap_obj is not None:
                cmap_name = str(getattr(cmap_obj, 'name', cmap_name))
            c0, c1 = roi_im.get_clim()
            if np.isfinite(c0) and np.isfinite(c1) and (float(c1) > float(c0)):
                clim = (float(c0), float(c1))
    except Exception:
        pass
    try:
        c = str(self.colormap_var.get()).strip()
        if c:
            cmap_name = c
    except Exception:
        pass
    self._tab7_roi2_copy_cmap = cmap_name
    self._tab7_roi2_copy_clim = tuple(map(float, clim))
    sc = None
    try:
        sc = _tab6_effective_nm_per_output_px(self)
    except Exception:
        pass
    if sc is None:
        try:
            v = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
            if np.isfinite(v) and v > 0:
                sc = (v, v)
        except Exception:
            pass
    if sc is None:
        sc = (1.0, 1.0)
    self._tab7_roi2_copy_nm_per_output_px = (float(sc[0]), float(sc[1]))
    self._tab7_fixed_nm_per_output_px = self._tab7_roi2_copy_nm_per_output_px
    try:
        idx = int(round(float(self.roi2_idx_var.get())))
    except Exception:
        idx = 0
    idx = max(0, min(idx, arr.shape[0] - 1))
    self._tab7_roi2_copy_frame = idx
    try:
        self.line_profile_frame_var.set(idx)
    except Exception:
        pass
    for attr in ('_line_profile_lines', '_line_profile_results', '_line_profile_measurements_list', '_line_profile_measure_points', '_line_profile_measure_artists', '_line_profile_image_line_artists'):
        if hasattr(self, attr):
            try:
                setattr(self, attr, [])
            except Exception:
                pass
    try:
        self._line_profile_measure_mode = False
        self._line_profile_measure_drag_index = None
        self._line_profile_measure_drag_point = None
    except Exception:
        pass
    self.line_source_stack = self._tab7_roi2_copy_stack
    self.line_source_coords = dict(self._tab7_roi2_copy_coords)
    try:
        self._tab7_frame_ui_var.set(idx + 1)
    except Exception:
        pass
    try:
        sl = self.line_profile_frame_slider
        sl.configure(from_=1, to=max(1, int(arr.shape[0])), state='normal', resolution=1)
        sl.set(idx + 1)
    except Exception:
        pass
    try:
        self.line_profile_image.set_cmap(self._tab7_roi2_copy_cmap)
        self.line_profile_image.set_clim(*self._tab7_roi2_copy_clim)
    except Exception:
        pass
    try:
        self._tab7_final_render_frame(idx, preserve_zoom=False)
    except Exception:
        try:
            self._line_profile_update_image(idx=idx, preserve_lines=False)
        except Exception:
            pass
    try:
        c = self._tab7_roi2_copy_coords
        self.line_profile_status_var.set(f"✓ ROI2 FETCHED FROM TAB 6 | independent Tab-7 copy | X={c['xmin']}:{c['xmax']}, Y={c['ymin']}:{c['ymax']} | size={arr.shape[2]} × {arr.shape[1]} px | frames={arr.shape[0]} | Color={self._tab7_roi2_copy_cmap} | nm/px={self._tab7_roi2_copy_nm_per_output_px[0]:.9g} × {self._tab7_roi2_copy_nm_per_output_px[1]:.9g}")
    except Exception:
        pass
    if auto_select:
        try:
            self.nb.select(self.tab_line_profile)
        except Exception:
            pass
    return True
CDIWorkflowApp._tab7_fetch_confirmed_roi2_from_tab6 = _tab7_fetch_confirmed_roi2_from_tab6

def _tab7_roi2_copy_stack_authoritative(self):
    s = getattr(self, '_tab7_roi2_copy_stack', None)
    if s is None:
        s = getattr(self, 'line_source_stack', None)
    if s is None:
        s = getattr(self, 'roi2_stack', None)
    try:
        a = np.asarray(s, dtype=float)
        if a.ndim == 3 and a.shape[0] > 0:
            return a
    except Exception:
        pass
    return None
CDIWorkflowApp._tab7_final_stack = _tab7_roi2_copy_stack_authoritative
CDIWorkflowApp._tab7_private_stack = _tab7_roi2_copy_stack_authoritative

def _tab7_roi2_copy_scale(self):
    sc = getattr(self, '_tab7_roi2_copy_nm_per_output_px', None)
    if sc is None:
        sc = getattr(self, '_tab7_fixed_nm_per_output_px', None)
    if sc is not None:
        try:
            sx, sy = (float(sc[0]), float(sc[1]))
            if sx > 0 and sy > 0 and np.isfinite(sx) and np.isfinite(sy):
                return (sx, sy)
        except Exception:
            pass
    return (1.0, 1.0)
CDIWorkflowApp._tab7_final_scale = _tab7_roi2_copy_scale
CDIWorkflowApp._tab7_private_scale = _tab7_roi2_copy_scale

def _tab7_copy_scales_nm(self):
    return _tab7_roi2_copy_scale(self)
CDIWorkflowApp._tab7_line_profile_scales_nm = _tab7_copy_scales_nm

def _tab7_render_copied_roi2_final(self, idx=0, preserve_zoom=True):
    stack = _tab7_roi2_copy_stack_authoritative(self)
    if stack is None:
        try:
            self.line_profile_status_var.set('Waiting for confirmed ROI2 from Tab 6.')
        except Exception:
            pass
        return
    n = int(stack.shape[0])
    idx = max(0, min(int(idx), n - 1))
    img = np.asarray(stack[idx], dtype=float)
    h, w = img.shape[:2]
    sx, sy = _tab7_roi2_copy_scale(self)
    ax = self.line_profile_ax_image
    artist = self.line_profile_image
    cmap = getattr(self, '_tab7_roi2_copy_cmap', 'RdBu_r')
    clim = getattr(self, '_tab7_roi2_copy_clim', (0.0, 1.0))
    try:
        artist.set_cmap(cmap)
    except Exception:
        artist.set_cmap('RdBu_r')
    artist.set_data(np.clip(img, 0.0, 1.0))
    artist.set_clim(float(clim[0]), float(clim[1]))
    full = (0.0, float(w * sx), 0.0, float(h * sy))
    ax.set_axis_on()
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    old_view = getattr(self, '_tab7_view_nm', None)
    if preserve_zoom and old_view is not None:
        try:
            ax.set_xlim(*old_view[0])
            ax.set_ylim(*old_view[1])
        except Exception:
            ax.set_xlim(full[0], full[1])
            ax.set_ylim(full[2], full[3])
    else:
        ax.set_xlim(full[0], full[1])
        ax.set_ylim(full[2], full[3])
        self._tab7_view_nm = (tuple(ax.get_xlim()), tuple(ax.get_ylim()))
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    try:
        field = float(self.real_fields_mT[idx])
    except Exception:
        field = float(idx)
    ax.set_title(f'Secondary ROI — Frame {idx + 1}/{n} | Field = {field:+.2f} mT')
    try:
        self.line_profile_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT')
        self.line_profile_colorbar.update_normal(artist)
        self.line_profile_colorbar.set_label(f'Tab-6 ROI2 processed contrast ({cmap})')
    except Exception:
        pass
    try:
        _tab7_final_redraw_profiles(self, idx)
    except Exception:
        pass
    try:
        _tab7_final_redraw_image_lines(self)
    except Exception:
        pass
    try:
        self.line_profile_canvas_image.draw_idle()
        self.line_profile_canvas_graph.draw_idle()
    except Exception:
        pass
CDIWorkflowApp._tab7_final_render_frame = _tab7_render_copied_roi2_final
CDIWorkflowApp._line_profile_update_image = _tab7_render_copied_roi2_final
_TAB6_CONFIRM_FOR_COPY = CDIWorkflowApp.confirm_secondary_roi_from_processing

def _tab6_confirm_then_fetch_roi2(self, *args, **kwargs):
    result = _TAB6_CONFIRM_FOR_COPY(self, *args, **kwargs)
    try:
        if result is not False:
            self._tab7_fetch_confirmed_roi2_from_tab6(auto_select=False)
    except Exception as exc:
        try:
            self.log(f'Tab 7 ROI2 fetch warning: {type(exc).__name__}: {exc}')
        except Exception:
            pass
    return result
CDIWorkflowApp.confirm_secondary_roi_from_processing = _tab6_confirm_then_fetch_roi2

def _tab7_copy_frame_changed(self, value=None):
    stack = _tab7_roi2_copy_stack_authoritative(self)
    if stack is None:
        return 'break'
    try:
        ui = int(round(float(value if value is not None else self._tab7_frame_ui_var.get())))
    except Exception:
        ui = 1
    idx = max(0, min(ui - 1, len(stack) - 1))
    self._tab7_roi2_copy_frame = idx
    try:
        self.line_profile_frame_var.set(idx)
    except Exception:
        pass
    self._tab7_render_copied_roi2_final(idx, preserve_zoom=True)
    return 'break'
CDIWorkflowApp._tab7_final_slider_changed = _tab7_copy_frame_changed
CDIWorkflowApp._line_profile_frame_slider_changed = _tab7_copy_frame_changed

def _tab7_copy_set_frame(self, idx):
    stack = _tab7_roi2_copy_stack_authoritative(self)
    if stack is None:
        return
    idx = max(0, min(int(idx), len(stack) - 1))
    try:
        self._tab7_frame_ui_var.set(idx + 1)
        self.line_profile_frame_var.set(idx)
        self.line_profile_frame_slider.set(idx + 1)
    except Exception:
        pass
    self._tab7_render_copied_roi2_final(idx, preserve_zoom=True)
CDIWorkflowApp._tab7_final_set_frame = _tab7_copy_set_frame

def _tab7_add_fetch_button(self):
    if getattr(self, '_tab7_fetch_roi2_button', None) is not None:
        return
    try:
        ref = getattr(self, 'line_profile_frame_label', None)
        parent = ref.master if ref is not None else self.tab_line_profile
        self._tab7_fetch_roi2_button = ttk.Button(parent, text='FETCH ROI2 FROM TAB 6', command=lambda: self._tab7_fetch_confirmed_roi2_from_tab6(auto_select=False))
        if ref is not None:
            self._tab7_fetch_roi2_button.pack(side='left', before=ref, padx=3, pady=1)
        else:
            self._tab7_fetch_roi2_button.pack(side='left', padx=3, pady=1)
    except Exception:
        pass
    try:
        tab = self.tab_line_profile

        def walk(w):
            yield w
            try:
                for ch in w.winfo_children():
                    yield from walk(ch)
            except Exception:
                pass
        compact = {'DYNAMIC MEASURING SCALE': 'DYNAMIC SCALE', 'CLEAR MEASUREMENT': 'CLEAR MEASURE', 'EXPORT LINE PROFILES CSV': 'EXPORT PROFILES', 'EXPORT RESULTS TABLE CSV': 'EXPORT RESULTS', 'EXPORT MEASUREMENTS CSV': 'EXPORT MEASURE', 'CLEAR ALL LINES': 'CLEAR LINES', 'REDO LAST LINE': 'REDO LAST', 'DRAW LINE SETS': 'DRAW LINES'}
        for w in walk(tab):
            try:
                if isinstance(w, (ttk.Button, tk.Button)):
                    text = str(w.cget('text'))
                    new = compact.get(text, text)
                    if new != text:
                        w.configure(text=new)
                    try:
                        w.configure(padding=(3, 1))
                    except Exception:
                        pass
            except Exception:
                pass
    except Exception:
        pass
CDIWorkflowApp._tab7_add_fetch_button = _tab7_add_fetch_button
_TAB7_COPY_PREVIOUS_BUILD_UI = CDIWorkflowApp._build_ui

def _tab7_copy_final_build_ui(self, *args, **kwargs):
    result = _TAB7_COPY_PREVIOUS_BUILD_UI(self, *args, **kwargs)
    self._tab7_roi2_copy_stack = getattr(self, '_tab7_roi2_copy_stack', None)
    self._tab7_roi2_copy_coords = getattr(self, '_tab7_roi2_copy_coords', None)
    self._tab7_roi2_copy_cmap = getattr(self, '_tab7_roi2_copy_cmap', 'RdBu_r')
    self._tab7_roi2_copy_clim = getattr(self, '_tab7_roi2_copy_clim', (0.0, 1.0))
    self._tab7_roi2_copy_frame = getattr(self, '_tab7_roi2_copy_frame', 0)
    try:
        self._tab7_add_fetch_button()
    except Exception:
        pass
    try:
        if getattr(self, 'roi2_stack', None) is not None and getattr(self, 'roi2_confirmed_coords', None) is not None:
            self._tab7_fetch_confirmed_roi2_from_tab6(auto_select=False)
    except Exception:
        pass
    return result
CDIWorkflowApp._build_ui = _tab7_copy_final_build_ui

def _tab7_copy_on_tab_selected(self, event=None):
    try:
        if event is not None and event.widget.select() != str(self.tab_line_profile):
            return
        if getattr(self, '_tab7_roi2_copy_stack', None) is None:
            self._tab7_fetch_confirmed_roi2_from_tab6(auto_select=False)
            return
        idx = int(self.line_profile_frame_var.get() if hasattr(self, 'line_profile_frame_var') else 0)
        stack = _tab7_roi2_copy_stack_authoritative(self)
        if stack is not None:
            idx = max(0, min(idx, len(stack) - 1))
            self._tab7_render_copied_roi2_final(idx, preserve_zoom=True)
    except Exception:
        pass
try:
    CDIWorkflowApp._tab7_copy_on_tab_selected = _tab7_copy_on_tab_selected
    _TAB7_COPY_BUILD_HOOK = CDIWorkflowApp._build_ui

    def _tab7_copy_build_ui_last(self, *args, **kwargs):
        result = _TAB7_COPY_BUILD_HOOK(self, *args, **kwargs)
        try:
            self.nb.bind('<<NotebookTabChanged>>', self._tab7_copy_on_tab_selected, add='+')
        except Exception:
            pass
        return result
    CDIWorkflowApp._build_ui = _tab7_copy_build_ui_last
except Exception:
    pass

def _tab7r_scale_nm(self):
    try:
        sc = _tab6_effective_nm_per_output_px(self)
        if sc is not None:
            sx, sy = (float(sc[0]), float(sc[1]))
            if np.isfinite(sx) and np.isfinite(sy) and (sx > 0) and (sy > 0):
                return (sx, sy)
    except Exception:
        pass
    for name in ('primary_roi_final_nm_per_pixel', 'pixel_cal_active_nm_per_pixel'):
        try:
            v = float(getattr(self, name))
            if np.isfinite(v) and v > 0:
                return (v, v)
        except Exception:
            pass
    return (1.0, 1.0)

def _tab7r_copy_roi2_from_tab6(self, select_tab=False):
    """Fetch and freeze the CURRENT confirmed processed ROI2 from Tab 6."""
    stack = getattr(self, 'roi2_stack', None)
    coords = getattr(self, 'roi2_confirmed_coords', None)
    if stack is None or coords is None:
        self._tab7r_loaded = False
        try:
            self.line_profile_status_var.set('ROI2 not available. In Tab 6 select Secondary ROI and click CONFIRM SECONDARY ROI.')
        except Exception:
            pass
        return False
    try:
        src = np.asarray(stack, dtype=np.float64)
    except Exception as exc:
        try:
            self.line_profile_status_var.set(f'ROI2 fetch error: {exc}')
        except Exception:
            pass
        return False
    if src.ndim != 3 or src.shape[0] == 0:
        try:
            self.line_profile_status_var.set(f'Invalid confirmed ROI2 stack: {src.shape}')
        except Exception:
            pass
        return False
    self._tab7r_stack = src.copy()
    self._tab7r_coords = {'xmin': int(coords['xmin']), 'xmax': int(coords['xmax']), 'ymin': int(coords['ymin']), 'ymax': int(coords['ymax'])}
    cmap = 'RdBu_r'
    clim = (0.0, 1.0)
    try:
        im6 = getattr(self, 'roi2_image', None)
        if im6 is not None:
            cm = im6.get_cmap()
            if cm is not None:
                cmap = str(getattr(cm, 'name', cmap))
            lo, hi = im6.get_clim()
            if np.isfinite(lo) and np.isfinite(hi) and (hi > lo):
                clim = (float(lo), float(hi))
    except Exception:
        pass
    try:
        cmv = str(self.colormap_var.get()).strip()
        if cmv:
            cmap = cmv
    except Exception:
        pass
    self._tab7r_cmap = cmap
    self._tab7r_clim = clim
    self._tab7r_nm_per_px = _tab7r_scale_nm(self)
    try:
        idx = int(round(float(self.roi2_idx_var.get())))
    except Exception:
        idx = 0
    idx = max(0, min(idx, src.shape[0] - 1))
    self._tab7r_frame = idx
    self._tab7r_lines = []
    self._tab7r_measurements = []
    self._tab7r_draw_mode = False
    self._tab7r_pending_line = None
    self._tab7r_measure_mode = False
    self._tab7r_pending_measure = None
    self._tab7r_zoom = None
    self._tab7r_results = []
    self.line_source_stack = self._tab7r_stack
    self.line_source_coords = dict(self._tab7r_coords)
    self._tab7_fixed_nm_per_output_px = self._tab7r_nm_per_px
    try:
        self.line_profile_frame_var.set(idx)
    except Exception:
        pass
    try:
        self.line_profile_frame_slider.configure(from_=1, to=max(1, src.shape[0]), state='normal', resolution=1)
        self.line_profile_frame_slider.set(idx + 1)
    except Exception:
        pass
    try:
        self._tab7r_render_frame(idx, preserve_zoom=False)
    except Exception:
        pass
    try:
        c = self._tab7r_coords
        sx, sy = self._tab7r_nm_per_px
        self.line_profile_status_var.set(f"✓ CONFIRMED ROI2 FETCHED FROM TAB 6 | copy={src.shape[2]} × {src.shape[1]} px × {src.shape[0]} frames | X={c['xmin']}:{c['xmax']}, Y={c['ymin']}:{c['ymax']} | scale={sx:.9g} × {sy:.9g} nm/output-px | Color={self._tab7r_cmap}")
    except Exception:
        pass
    if select_tab:
        try:
            self.nb.select(self.tab_line_profile)
        except Exception:
            pass
    self._tab7r_loaded = True
    return True

def _tab7r_stack(self):
    s = getattr(self, '_tab7r_stack', None)
    if s is None:
        return None
    try:
        a = np.asarray(s, dtype=float)
        if a.ndim == 3 and a.shape[0] > 0:
            return a
    except Exception:
        pass
    return None

def _tab7r_sample_line(self, image, p0, p1):
    """Bilinear line sampling. Returns distance in px and intensity."""
    x0, y0 = map(float, p0)
    x1, y1 = map(float, p1)
    length = float(np.hypot(x1 - x0, y1 - y0))
    n = max(2, int(np.ceil(length * 4.0)) + 1)
    dpx = np.linspace(0.0, length, n)
    if length <= 0:
        ix = max(0, min(image.shape[1] - 1, int(round(x0))))
        iy = max(0, min(image.shape[0] - 1, int(round(y0))))
        return (np.array([0.0]), np.array([float(image[iy, ix])]))
    xs = x0 + (x1 - x0) * dpx / length
    ys = y0 + (y1 - y0) * dpx / length
    prof = map_coordinates(image, [ys, xs], order=1, mode='nearest')
    return (dpx, np.asarray(prof, dtype=float))

def _tab7r_px_to_nm(self, dpx):
    sx, sy = self._tab7r_nm_per_px
    return np.asarray(dpx, dtype=float)

def _tab7r_line_distance_nm(self, p0, p1):
    sx, sy = self._tab7r_nm_per_px
    x0, y0 = map(float, p0)
    x1, y1 = map(float, p1)
    return float(np.hypot((x1 - x0) * sx, (y1 - y0) * sy))

def _tab7r_line_profile_nm(self, image, p0, p1):
    x0, y0 = map(float, p0)
    x1, y1 = map(float, p1)
    sx, sy = self._tab7r_nm_per_px
    physical_len = float(np.hypot((x1 - x0) * sx, (y1 - y0) * sy))
    px_len = float(np.hypot(x1 - x0, y1 - y0))
    n = max(2, int(np.ceil(max(px_len, 1.0) * 4.0)) + 1)
    t = np.linspace(0.0, 1.0, n)
    xs = x0 + (x1 - x0) * t
    ys = y0 + (y1 - y0) * t
    prof = map_coordinates(image, [ys, xs], order=1, mode='nearest').astype(float)
    d_nm = t * physical_len
    return (d_nm, prof)

def _tab7r_render_frame(self, idx=0, preserve_zoom=True):
    stack = self._tab7r_stack()
    if stack is None:
        return
    n = int(stack.shape[0])
    idx = max(0, min(int(idx), n - 1))
    img = np.asarray(stack[idx], dtype=float)
    h, w = img.shape[:2]
    sx, sy = self._tab7r_nm_per_px
    artist = self.line_profile_image
    ax = self.line_profile_ax_image
    try:
        artist.set_cmap(self._tab7r_cmap)
    except Exception:
        artist.set_cmap('RdBu_r')
    artist.set_data(np.clip(img, 0.0, 1.0))
    artist.set_clim(*self._tab7r_clim)
    artist.set_extent((0.0, w * sx, 0.0, h * sy))
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    full = (0.0, w * sx, 0.0, h * sy)
    if preserve_zoom and self._tab7r_zoom is not None:
        try:
            ax.set_xlim(*self._tab7r_zoom[0])
            ax.set_ylim(*self._tab7r_zoom[1])
        except Exception:
            ax.set_xlim(full[0], full[1])
            ax.set_ylim(full[2], full[3])
    else:
        ax.set_xlim(full[0], full[1])
        ax.set_ylim(full[2], full[3])
        self._tab7r_zoom = (tuple(ax.get_xlim()), tuple(ax.get_ylim()))
    field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    self.line_profile_frame_var.set(idx)
    try:
        self.line_profile_frame_slider.set(idx + 1)
    except Exception:
        pass
    self.line_profile_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT')
    ax.set_title(f'Confirmed Tab-6 ROI2 — Frame {idx + 1}/{n} | {field:+.2f} mT')
    try:
        self.line_profile_colorbar.update_normal(artist)
        self.line_profile_colorbar.set_label(f'Tab-6 ROI2 processed contrast | {self._tab7r_cmap}')
    except Exception:
        pass
    self._tab7r_redraw_profiles(idx)
    self._tab7r_redraw_lines()
    self.line_profile_canvas_image.draw_idle()
    self.line_profile_canvas_graph.draw_idle()

def _tab7r_redraw_lines(self):
    ax = self.line_profile_ax_image
    for art in getattr(self, '_tab7r_image_artists', []):
        try:
            art.remove()
        except Exception:
            pass
    self._tab7r_image_artists = []
    sx, sy = self._tab7r_nm_per_px
    colors = ['yellow', 'cyan', 'lime', 'magenta', 'orange', 'white']
    for j, item in enumerate(self._tab7r_lines):
        p0 = item['p0']
        p1 = item['p1']
        x = [p0[0] * sx, p1[0] * sx]
        y = [p0[1] * sy, p1[1] * sy]
        try:
            ln, = ax.plot(x, y, '--', lw=1.8, color=colors[j % len(colors)], zorder=40)
            self._tab7r_image_artists.append(ln)
        except Exception:
            pass

def _tab7r_redraw_profiles(self, idx=None):
    stack = self._tab7r_stack()
    if stack is None:
        return
    if idx is None:
        idx = self._tab7r_frame
    image = np.asarray(stack[idx], dtype=float)
    ax = self.line_profile_ax_graph
    ax.clear()
    ax.set_xlabel('Distance (nm)')
    ax.set_ylabel('Intensity')
    ax.set_title('Line Profiles — Confirmed Tab-6 ROI2')
    ax.grid(True, alpha=0.22)
    sx, sy = self._tab7r_nm_per_px
    profiles = []
    for j, item in enumerate(self._tab7r_lines):
        dnm, prof = self._tab7r_line_profile_nm(image, item['p0'], item['p1'])
        profiles.append((dnm, prof))
        item['length_nm'] = float(dnm[-1])
        item['distance_nm'] = dnm
        item['profile'] = prof
        ax.plot(dnm, prof, lw=1.4, label=f"Line {j + 1} ({item.get('angle_deg', 0.0):.1f}°)")
    if profiles:
        dmax = max((float(x[0][-1]) for x in profiles))
        common = np.linspace(0.0, dmax, 600)
        vals = []
        for dnm, prof in profiles:
            vals.append(np.interp(common, dnm, prof))
        avg = np.mean(vals, axis=0)
        self._tab7r_results = [{'frame': int(idx), 'line_count': len(profiles), 'avg_distance_nm': common, 'avg_profile': avg}]
        ax.plot(common, avg, lw=2.2, label='AVERAGE')
        ax.legend(loc='best', fontsize=8)
    self.line_profile_canvas_graph.draw_idle()

def _tab7r_start_lines(self):
    stack = self._tab7r_stack()
    if stack is None:
        self.line_profile_status_var.set('Fetch the confirmed ROI2 from Tab 6 first.')
        return
    try:
        target = max(1, min(12, int(self.line_profile_sets_var.get())))
    except Exception:
        target = 1
    if len(self._tab7r_lines) >= target:
        self.line_profile_status_var.set(f'{target} line set(s) already defined.')
        return
    self._tab7r_draw_mode = True
    self._tab7r_pending_line = None
    self.line_profile_status_var.set(f'DRAW MODE: click-drag one line for set {len(self._tab7r_lines) + 1}/{target}.')

def _tab7r_clear_lines(self):
    self._tab7r_lines = []
    self._tab7r_measurements = []
    self._tab7r_draw_mode = False
    self._tab7r_pending_line = None
    self._tab7r_redraw_profiles(self._tab7r_frame)
    self._tab7r_redraw_lines()
    self.line_profile_status_var.set('All profile lines cleared.')

def _tab7r_redo_last(self):
    if self._tab7r_lines:
        self._tab7r_lines.pop()
        self._tab7r_measurements = [m for m in self._tab7r_measurements if m.get('setno') <= len(self._tab7r_lines)]
        self._tab7r_redraw_profiles(self._tab7r_frame)
        self._tab7r_redraw_lines()
        self.line_profile_status_var.set(f'Last line removed. {len(self._tab7r_lines)} line set(s) remain.')

def _tab7r_image_press(self, event):
    if event.inaxes is not self.line_profile_ax_image:
        return
    if event.xdata is None or event.ydata is None:
        return
    if not self._tab7r_draw_mode:
        return
    sx, sy = self._tab7r_nm_per_px
    p0 = (float(event.xdata) / sx, float(event.ydata) / sy)
    self._tab7r_pending_line = p0

def _tab7r_image_release(self, event):
    if not self._tab7r_draw_mode:
        return
    p0 = self._tab7r_pending_line
    self._tab7r_pending_line = None
    if p0 is None or event.inaxes is not self.line_profile_ax_image:
        return
    if event.xdata is None or event.ydata is None:
        return
    sx, sy = self._tab7r_nm_per_px
    p1 = (float(event.xdata) / sx, float(event.ydata) / sy)
    px_len = float(np.hypot(p1[0] - p0[0], p1[1] - p0[1]))
    if px_len < 1.0:
        self.line_profile_status_var.set('Line too short.')
        return
    target = max(1, min(12, int(self.line_profile_sets_var.get())))
    angle = float(np.degrees(np.arctan2((p1[1] - p0[1]) * sy, (p1[0] - p0[0]) * sx))) % 180.0
    self._tab7r_lines.append({'p0': p0, 'p1': p1, 'setno': len(self._tab7r_lines) + 1, 'angle_deg': angle})
    self._tab7r_draw_mode = len(self._tab7r_lines) < target
    self._tab7r_redraw_profiles(self._tab7r_frame)
    self._tab7r_redraw_lines()
    length_nm = self._tab7r_line_distance_nm(p0, p1)
    if self._tab7r_draw_mode:
        msg = f'Line {len(self._tab7r_lines)}/{target} added | length={length_nm:.6g} nm | angle={angle:.1f}° | draw next.'
    else:
        msg = f'All {target} requested lines drawn. Last length={length_nm:.6g} nm, angle={angle:.1f}°.'
    self.line_profile_status_var.set(msg)

def _tab7r_measure(self):
    if not self._tab7r_lines:
        self.line_profile_status_var.set('Draw at least one profile line first.')
        return
    self._tab7r_measure_mode = True
    self._tab7r_pending_measure = None
    self.line_profile_measure_var.set('MEASURE MODE: click two points on a profile curve to define M1 and M2.')

def _tab7r_graph_press(self, event):
    if not self._tab7r_measure_mode:
        return
    if event.inaxes is not self.line_profile_ax_graph:
        return
    if event.xdata is None or event.ydata is None:
        return
    if self._tab7r_pending_measure is None:
        self._tab7r_pending_measure = (float(event.xdata), float(event.ydata))
        self.line_profile_measure_var.set('M1 selected. Click the M2 point.')
    else:
        p1 = self._tab7r_pending_measure
        p2 = (float(event.xdata), float(event.ydata))
        self._tab7r_pending_measure = None
        self._tab7r_measure_mode = False
        best = None
        bestdist = np.inf
        for j, item in enumerate(self._tab7r_lines):
            dnm = np.asarray(item.get('distance_nm', []), dtype=float)
            prof = np.asarray(item.get('profile', []), dtype=float)
            if dnm.size < 2:
                continue
            d1 = float(abs(np.interp(p1[0], dnm, dnm) - p1[0]))
            d2 = float(abs(np.interp(p2[0], dnm, dnm) - p2[0]))
            d = d1 + d2
            if d < bestdist:
                bestdist = d
                best = j
        if best is None:
            self.line_profile_measure_var.set('No profile available for measurement.')
            return
        item = self._tab7r_lines[best]
        length_nm = float(item.get('length_nm', self._tab7r_line_distance_nm(item['p0'], item['p1'])))
        m1 = max(0.0, min(length_nm, p1[0]))
        m2 = max(0.0, min(length_nm, p2[0]))
        prof = np.asarray(item.get('profile', []), dtype=float)
        dnm = np.asarray(item.get('distance_nm', []), dtype=float)
        i1 = float(np.interp(m1, dnm, prof))
        i2 = float(np.interp(m2, dnm, prof))
        measurement = {'setno': int(item['setno']), 'm1_nm': min(m1, m2), 'm2_nm': max(m1, m2), 'delta_nm': abs(m2 - m1), 'm1_intensity': i1 if m1 <= m2 else i2, 'm2_intensity': i2 if m1 <= m2 else i1}
        self._tab7r_measurements.append(measurement)
        self._tab7r_update_table()
        self.line_profile_measure_var.set(f"Line {measurement['setno']}: M1={measurement['m1_nm']:.6g} nm, M2={measurement['m2_nm']:.6g} nm, ΔL={measurement['delta_nm']:.6g} nm")

def _tab7r_update_table(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    for item in tree.get_children():
        tree.delete(item)
    for m in self._tab7r_measurements:
        tree.insert('', 'end', values=(m['setno'], f"{m['m1_nm']:.6g}", f"{m['m2_nm']:.6g}", f"{m['delta_nm']:.6g}", f"{m['m1_intensity']:.6g}", f"{m['m2_intensity']:.6g}"))

def _tab7r_frame_changed(self, value=None):
    stack = self._tab7r_stack()
    if stack is None:
        return 'break'
    try:
        ui = int(round(float(value if value is not None else self._tab7r_frame_ui.get())))
    except Exception:
        ui = 1
    idx = max(0, min(ui - 1, len(stack) - 1))
    self._tab7r_frame = idx
    self.line_profile_frame_var.set(idx)
    self._tab7r_render_frame(idx, preserve_zoom=True)
    return 'break'

def _tab7r_wheel(self, event):
    stack = self._tab7r_stack()
    if stack is None:
        return 'break'
    delta = getattr(event, 'delta', 0)
    if getattr(event, 'num', None) == 4:
        step = 1
    elif getattr(event, 'num', None) == 5:
        step = -1
    elif delta:
        step = 1 if delta > 0 else -1
    else:
        return 'break'
    cur = int(self._tab7r_frame_ui.get())
    self._tab7r_frame_ui.set(max(1, min(cur + step, len(stack))))
    _tab7r_frame_changed(self, self._tab7r_frame_ui.get())
    return 'break'

def _tab7r_zoom(self, event):
    if event.inaxes is not self.line_profile_ax_image:
        return
    if event.xdata is None or event.ydata is None:
        return
    ax = self.line_profile_ax_image
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    step = getattr(event, 'step', None)
    if step is None:
        step = 1 if getattr(event, 'button', None) == 'up' else -1
    factor = 1.2 ** (-float(step))
    cx, cy = (float(event.xdata), float(event.ydata))
    nx0 = cx - (cx - x0) * factor
    nx1 = cx + (x1 - cx) * factor
    ny0 = cy - (cy - y0) * factor
    ny1 = cy + (y1 - cy) * factor
    ax.set_xlim(nx0, nx1)
    ax.set_ylim(ny0, ny1)
    self._tab7r_zoom = (tuple(ax.get_xlim()), tuple(ax.get_ylim()))
    self.line_profile_canvas_image.draw_idle()

def _tab7r_reset_zoom(self):
    stack = self._tab7r_stack()
    if stack is None:
        return
    img = np.asarray(stack[self._tab7r_frame], dtype=float)
    h, w = img.shape[:2]
    sx, sy = self._tab7r_nm_per_px
    full = (0.0, w * sx, 0.0, h * sy)
    self._tab7r_zoom = (full[:2], full[2:])
    self.line_profile_ax_image.set_xlim(full[0], full[1])
    self.line_profile_ax_image.set_ylim(full[2], full[3])
    self.line_profile_canvas_image.draw_idle()

def _tab7r_export_profiles(self):
    if not self._tab7r_lines:
        messagebox.showwarning('Line Profile', 'No lines have been drawn.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(parent=self.root, title='Export line profiles', defaultextension='.csv', filetypes=[('CSV', '*.csv')])
    if not fp:
        return
    rows = []
    idx = int(self._tab7r_frame)
    field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    for line in self._tab7r_lines:
        dnm = np.asarray(line.get('distance_nm', []), dtype=float)
        prof = np.asarray(line.get('profile', []), dtype=float)
        for d, y in zip(dnm, prof):
            rows.append({'Frame': idx + 1, 'Field_mT': field, 'Set': int(line['setno']), 'Angle_deg': float(line.get('angle_deg', np.nan)), 'Distance_nm': float(d), 'Intensity': float(y)})
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} profile points to CSV.')

def _tab7r_export_measurements(self):
    if not self._tab7r_measurements:
        messagebox.showwarning('Line Profile', 'No measurements have been made.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(parent=self.root, title='Export line measurements', defaultextension='.csv', filetypes=[('CSV', '*.csv')])
    if not fp:
        return
    pd.DataFrame(self._tab7r_measurements).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(self._tab7r_measurements)} measurement(s) to CSV.')
CDIWorkflowApp._tab7r_stack = _tab7r_stack
CDIWorkflowApp._tab7r_copy_roi2_from_tab6 = _tab7r_copy_roi2_from_tab6
CDIWorkflowApp._tab7r_render_frame = _tab7r_render_frame
CDIWorkflowApp._tab7r_redraw_profiles = _tab7r_redraw_profiles
CDIWorkflowApp._tab7r_redraw_lines = _tab7r_redraw_lines
CDIWorkflowApp._tab7r_start_lines = _tab7r_start_lines
CDIWorkflowApp._tab7r_clear_lines = _tab7r_clear_lines
CDIWorkflowApp._tab7r_redo_last = _tab7r_redo_last
CDIWorkflowApp._tab7r_image_press = _tab7r_image_press
CDIWorkflowApp._tab7r_image_release = _tab7r_image_release
CDIWorkflowApp._tab7r_measure = _tab7r_measure
CDIWorkflowApp._tab7r_graph_press = _tab7r_graph_press
CDIWorkflowApp._tab7r_update_table = _tab7r_update_table
CDIWorkflowApp._tab7r_frame_changed = _tab7r_frame_changed
CDIWorkflowApp._tab7r_wheel = _tab7r_wheel
CDIWorkflowApp._tab7r_zoom = _tab7r_zoom
CDIWorkflowApp._tab7r_reset_zoom = _tab7r_reset_zoom
CDIWorkflowApp._tab7r_export_profiles = _tab7r_export_profiles
CDIWorkflowApp._tab7r_export_measurements = _tab7r_export_measurements
CDIWorkflowApp._line_profile_update_image = _tab7r_render_frame
CDIWorkflowApp._line_profile_redraw_profiles = _tab7r_redraw_profiles
CDIWorkflowApp._line_profile_redraw_image_lines = _tab7r_redraw_lines
CDIWorkflowApp._line_profile_image_press = _tab7r_image_press
CDIWorkflowApp._line_profile_image_release = _tab7r_image_release
CDIWorkflowApp._line_profile_graph_press = _tab7r_graph_press
CDIWorkflowApp._line_profile_frame_slider_changed = _tab7r_frame_changed
_TAB7R_OLD_CONFIRM = CDIWorkflowApp.confirm_secondary_roi_from_processing

def _tab7r_confirm_then_copy(self, *args, **kwargs):
    result = _TAB7R_OLD_CONFIRM(self, *args, **kwargs)
    try:
        if result is not False:
            self._tab7r_copy_roi2_from_tab6(select_tab=False)
    except Exception:
        pass
    return result
CDIWorkflowApp.confirm_secondary_roi_from_processing = _tab7r_confirm_then_copy

def _tab7r_tab_changed(self, event=None):
    try:
        if event is not None and event.widget.select() != str(self.tab_line_profile):
            return
        if not getattr(self, '_tab7r_loaded', False):
            self._tab7r_copy_roi2_from_tab6(select_tab=False)
    except Exception:
        pass
_TAB7R_BUILD_UI_OLD = CDIWorkflowApp._build_ui

def _tab7r_build_ui(self, *args, **kwargs):
    result = _TAB7R_BUILD_UI_OLD(self, *args, **kwargs)
    try:
        self.nb.bind('<<NotebookTabChanged>>', self._tab7r_tab_changed, add='+')
    except Exception:
        pass
    return result
CDIWorkflowApp._build_ui = _tab7r_build_ui
CDIWorkflowApp._tab7r_tab_changed = _tab7r_tab_changed

def _t7_scale_from_tab6(self):
    try:
        base = self._proc_get_tab5_base_pixel_size()
        ow = int(getattr(self, 'original_roi_w', 0) or 0)
        oh = int(getattr(self, 'original_roi_h', 0) or 0)
        rw, rh = getattr(self, 'roi_resolution', (0, 0))
        rw, rh = (int(rw), int(rh))
        if base is not None and np.isfinite(base) and (base > 0) and (ow > 0) and (oh > 0) and (rw > 0) and (rh > 0):
            return (float(base) * ow / rw, float(base) * oh / rh)
    except Exception:
        pass
    try:
        v = float(getattr(self, 'proc_profile_measure_nm_per_px', np.nan))
        if np.isfinite(v) and v > 0:
            return (v, v)
    except Exception:
        pass
    try:
        v = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
        if np.isfinite(v) and v > 0:
            return (v, v)
    except Exception:
        pass
    return (1.0, 1.0)

def _t7_clim_like_tab6(self, image):
    a = np.asarray(image, dtype=float)
    finite = np.isfinite(a)
    if not np.any(finite):
        return (0.0, 1.0)
    lo = float(np.nanmin(a[finite]))
    hi = float(np.nanmax(a[finite]))
    if np.isclose(lo, hi):
        pad = max(abs(lo) * 0.05, 1e-06)
        lo, hi = (lo - pad, hi + pad)
    elif lo < 0.0 < hi:
        lim = max(abs(lo), abs(hi))
        lo, hi = (-lim, lim)
    if hi <= lo:
        hi = lo + 1e-12
    return (lo, hi)

def _t7_fetch_tab6_roi2(self):
    stack = getattr(self, 'roi2_stack', None)
    coords = getattr(self, 'roi2_confirmed_coords', None)
    if stack is None or coords is None:
        try:
            self.line_profile_status_var.set('No confirmed ROI2. Confirm Secondary ROI in Tab 6 first.')
        except Exception:
            pass
        return False
    arr = np.asarray(stack, dtype=np.float64)
    if arr.ndim != 3 or arr.shape[0] == 0:
        try:
            self.line_profile_status_var.set(f'Invalid ROI2 stack: {arr.shape}')
        except Exception:
            pass
        return False
    self._t7_stack_copy = np.array(arr, dtype=np.float64, copy=True)
    self._t7_coords_copy = {'xmin': int(coords['xmin']), 'xmax': int(coords['xmax']), 'ymin': int(coords['ymin']), 'ymax': int(coords['ymax'])}
    try:
        cmap = str(self.colormap_var.get()).strip() or 'RdBu_r'
    except Exception:
        cmap = 'RdBu_r'
    self._t7_cmap = cmap
    self._t7_sx_nm_px, self._t7_sy_nm_px = _t7_scale_from_tab6(self)
    try:
        idx = int(round(float(self.roi2_idx_var.get())))
    except Exception:
        idx = 0
    idx = max(0, min(idx, arr.shape[0] - 1))
    self._t7_frame = idx
    self._t7_lines = []
    self._t7_measurements = []
    self._t7_average = None
    self._t7_draw_mode = False
    self._t7_drag_start = None
    self._t7_temp_line = None
    self._t7_measure_mode = False
    self._t7_measure_first = None
    self._t7_zoom = None
    self._t7_image_artists = []
    self._t7_loaded = True
    self.line_source_stack = self._t7_stack_copy
    self.line_source_coords = dict(self._t7_coords_copy)
    self._tab7_fixed_nm_per_output_px = (self._t7_sx_nm_px, self._t7_sy_nm_px)
    try:
        self.line_profile_frame_var.set(idx)
    except Exception:
        pass
    try:
        self._t7_frame_ui_var.set(idx + 1)
    except Exception:
        pass
    try:
        self.line_profile_frame_slider.configure(from_=1, to=arr.shape[0], state='normal')
        self.line_profile_frame_slider.set(idx + 1)
    except Exception:
        pass
    _t7_render(self, idx, preserve_zoom=False)
    c = self._t7_coords_copy
    self.line_profile_status_var.set(f"✓ CONFIRMED ROI2 FETCHED FROM TAB 6 | independent copy | {arr.shape[2]} × {arr.shape[1]} px × {arr.shape[0]} frames | X={c['xmin']}:{c['xmax']}, Y={c['ymin']}:{c['ymax']} | scale={self._t7_sx_nm_px:.9g} × {self._t7_sy_nm_px:.9g} nm/output-px | Color={self._t7_cmap}")
    return True

def _t7_stack(self):
    s = getattr(self, '_t7_stack_copy', None)
    if s is None:
        return None
    try:
        a = np.asarray(s, dtype=float)
        if a.ndim == 3 and a.shape[0] > 0:
            return a
    except Exception:
        pass
    return None

def _t7_render(self, idx=0, preserve_zoom=True):
    stack = _t7_stack(self)
    if stack is None:
        return
    n = int(stack.shape[0])
    idx = max(0, min(int(idx), n - 1))
    self._t7_frame = idx
    image = np.asarray(stack[idx], dtype=float)
    h, w = image.shape
    sx, sy = (self._t7_sx_nm_px, self._t7_sy_nm_px)
    ax = self.line_profile_ax_image
    im = self.line_profile_image
    lo, hi = _t7_clim_like_tab6(self, image)
    try:
        im.set_cmap(self._t7_cmap)
    except Exception:
        im.set_cmap('RdBu_r')
    im.set_data(image)
    im.set_clim(lo, hi)
    im.set_extent((0, w * sx, 0, h * sy))
    full = (0, w * sx, 0, h * sy)
    if preserve_zoom and self._t7_zoom is not None:
        try:
            ax.set_xlim(*self._t7_zoom[0])
            ax.set_ylim(*self._t7_zoom[1])
        except Exception:
            ax.set_xlim(full[0], full[1])
            ax.set_ylim(full[2], full[3])
    else:
        ax.set_xlim(full[0], full[1])
        ax.set_ylim(full[2], full[3])
        self._t7_zoom = (tuple(ax.get_xlim()), tuple(ax.get_ylim()))
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    try:
        field = float(self.real_fields_mT[idx])
    except Exception:
        field = float(idx)
    ax.set_title(f'Confirmed Tab-6 ROI2 — Frame {idx + 1}/{n} | Field = {field:+.2f} mT')
    try:
        self.line_profile_frame_label.config(text=f'{idx + 1}/{n} | {field:+.2f} mT')
        self.line_profile_colorbar.update_normal(im)
        self.line_profile_colorbar.set_label(f'Tab-6 ROI2 processed contrast | {self._t7_cmap} | [{lo:.4g}, {hi:.4g}]')
    except Exception:
        pass
    _t7_redraw_profiles(self, idx)
    _t7_redraw_lines(self)
    try:
        self.line_profile_canvas_image.draw_idle()
        self.line_profile_canvas_graph.draw_idle()
    except Exception:
        pass

def _t7_profile(self, image, p0, p1):
    x0, y0 = map(float, p0)
    x1, y1 = map(float, p1)
    px_len = float(np.hypot(x1 - x0, y1 - y0))
    sx, sy = (self._t7_sx_nm_px, self._t7_sy_nm_px)
    nm_len = float(np.hypot((x1 - x0) * sx, (y1 - y0) * sy))
    samples = max(2, int(np.ceil(px_len * 4.0)) + 1)
    t = np.linspace(0.0, 1.0, samples)
    xs = x0 + (x1 - x0) * t
    ys = y0 + (y1 - y0) * t
    prof = map_coordinates(np.asarray(image, dtype=float), [ys, xs], order=1, mode='nearest')
    return (t * nm_len, np.asarray(prof, dtype=float))

def _t7_redraw_profiles(self, idx=None):
    stack = _t7_stack(self)
    if stack is None:
        return
    if idx is None:
        idx = self._t7_frame
    image = np.asarray(stack[idx], dtype=float)
    ax = self.line_profile_ax_graph
    ax.clear()
    ax.set_xlabel('Distance (nm)')
    ax.set_ylabel('Intensity')
    ax.set_title('Line Profiles — Confirmed Tab-6 ROI2')
    ax.grid(True, alpha=0.22)
    curves = []
    for j, line in enumerate(self._t7_lines):
        d, p = _t7_profile(self, image, line['p0'], line['p1'])
        line['distance_nm'] = d
        line['profile'] = p
        line['length_nm'] = float(d[-1])
        ax.plot(d, p, lw=1.35, label=f"Line {j + 1} | {line['angle_deg']:.1f}°")
        curves.append((d, p))
    if curves:
        dmax = max((float(d[-1]) for d, _ in curves))
        common = np.linspace(0.0, dmax, 600)
        y = np.vstack([np.interp(common, d, p) for d, p in curves])
        avg = np.mean(y, axis=0)
        self._t7_average = (common, avg)
        ax.plot(common, avg, lw=2.2, label='AVERAGE')
        ax.legend(loc='best', fontsize=8)
    else:
        self._t7_average = None
    self.line_profile_canvas_graph.draw_idle()

def _t7_redraw_lines(self):
    ax = self.line_profile_ax_image
    for artist in getattr(self, '_t7_image_artists', []):
        try:
            artist.remove()
        except Exception:
            pass
    self._t7_image_artists = []
    sx, sy = (self._t7_sx_nm_px, self._t7_sy_nm_px)
    colors = ['yellow', 'cyan', 'lime', 'magenta', 'orange', 'white', 'red', 'blue']
    for j, line in enumerate(self._t7_lines):
        ln, = ax.plot([line['p0'][0] * sx, line['p1'][0] * sx], [line['p0'][1] * sy, line['p1'][1] * sy], '--', lw=1.8, color=colors[j % len(colors)], zorder=50)
        self._t7_image_artists.append(ln)

def _t7_start_draw(self):
    if _t7_stack(self) is None:
        self.line_profile_status_var.set('Fetch confirmed ROI2 from Tab 6 first.')
        return
    try:
        target = max(1, min(12, int(self._t7_n_lines_var.get())))
    except Exception:
        target = 1
    if len(self._t7_lines) >= target:
        self.line_profile_status_var.set(f'{target} line set(s) already drawn.')
        return
    self._t7_draw_mode = True
    self.line_profile_status_var.set(f'DRAW MODE — drag line {len(self._t7_lines) + 1}/{target} on the ROI.')

def _t7_img_press(self, event):
    if event.inaxes is not self.line_profile_ax_image or event.xdata is None or event.ydata is None or (not self._t7_draw_mode):
        return
    sx, sy = (self._t7_sx_nm_px, self._t7_sy_nm_px)
    self._t7_drag_start = (float(event.xdata) / sx, float(event.ydata) / sy)
    if self._t7_temp_line is not None:
        try:
            self._t7_temp_line.remove()
        except Exception:
            pass
    self._t7_temp_line, = self.line_profile_ax_image.plot([event.xdata, event.xdata], [event.ydata, event.ydata], '--', lw=1.6, color='yellow', zorder=60)
    self.line_profile_canvas_image.draw_idle()

def _t7_img_move(self, event):
    if self._t7_drag_start is None or event.inaxes is not self.line_profile_ax_image or event.xdata is None or (event.ydata is None) or (self._t7_temp_line is None):
        return
    sx, sy = (self._t7_sx_nm_px, self._t7_sy_nm_px)
    p0 = self._t7_drag_start
    self._t7_temp_line.set_data([p0[0] * sx, float(event.xdata)], [p0[1] * sy, float(event.ydata)])
    self.line_profile_canvas_image.draw_idle()

def _t7_img_release(self, event):
    if self._t7_drag_start is None:
        return
    p0 = self._t7_drag_start
    self._t7_drag_start = None
    if self._t7_temp_line is not None:
        try:
            self._t7_temp_line.remove()
        except Exception:
            pass
        self._t7_temp_line = None
    if event.inaxes is not self.line_profile_ax_image or event.xdata is None or event.ydata is None:
        return
    sx, sy = (self._t7_sx_nm_px, self._t7_sy_nm_px)
    p1 = (float(event.xdata) / sx, float(event.ydata) / sy)
    pxlen = float(np.hypot(p1[0] - p0[0], p1[1] - p0[1]))
    if pxlen < 1.0:
        self.line_profile_status_var.set('Line too short.')
        return
    try:
        target = max(1, min(12, int(self._t7_n_lines_var.get())))
    except Exception:
        target = 1
    angle = float(np.degrees(np.arctan2((p1[1] - p0[1]) * sy, (p1[0] - p0[0]) * sx))) % 180.0
    self._t7_lines.append({'p0': p0, 'p1': p1, 'angle_deg': angle})
    self._t7_draw_mode = len(self._t7_lines) < target
    L = float(np.hypot((p1[0] - p0[0]) * sx, (p1[1] - p0[1]) * sy))
    _t7_redraw_profiles(self, self._t7_frame)
    _t7_redraw_lines(self)
    self.line_profile_status_var.set(f'Line {len(self._t7_lines)}/{target} added | length={L:.6g} nm | angle={angle:.1f}°')

def _t7_clear_lines(self):
    self._t7_lines = []
    self._t7_measurements = []
    self._t7_draw_mode = False
    self._t7_measure_mode = False
    self._t7_measure_first = None
    _t7_redraw_profiles(self, self._t7_frame)
    _t7_redraw_lines(self)
    _t7_update_table(self)
    self.line_profile_status_var.set('All profile lines cleared.')

def _t7_redo_last(self):
    if not self._t7_lines:
        self.line_profile_status_var.set('No line to remove.')
        return
    last = len(self._t7_lines)
    self._t7_lines.pop()
    self._t7_measurements = [m for m in self._t7_measurements if int(m['set']) != last]
    _t7_redraw_profiles(self, self._t7_frame)
    _t7_redraw_lines(self)
    _t7_update_table(self)
    self.line_profile_status_var.set('Last line removed.')

def _t7_zoom(self, event=None, factor=None):
    ax = self.line_profile_ax_image
    if event is not None:
        if event.inaxes is not ax or event.xdata is None or event.ydata is None:
            return
        cx, cy = (float(event.xdata), float(event.ydata))
        f = float(factor if factor is not None else 1 / 1.25 if getattr(event, 'button', None) == 'up' else 1.25)
    else:
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        cx = (x0 + x1) / 2
        cy = (y0 + y1) / 2
        f = float(factor)
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    nx0 = cx - (cx - x0) * f
    nx1 = cx + (x1 - cx) * f
    ny0 = cy - (cy - y0) * f
    ny1 = cy + (y1 - cy) * f
    st = _t7_stack(self)
    if st is not None:
        h, w = st[self._t7_frame].shape
        fx0, fx1, fy0, fy1 = (0.0, w * self._t7_sx_nm_px, 0.0, h * self._t7_sy_nm_px)
        if nx0 < fx0:
            nx1 += fx0 - nx0
            nx0 = fx0
        if nx1 > fx1:
            nx0 -= nx1 - fx1
            nx1 = fx1
        if ny0 < fy0:
            ny1 += fy0 - ny0
            ny0 = fy0
        if ny1 > fy1:
            ny0 -= ny1 - fy1
            ny1 = fy1
        nx0 = max(fx0, nx0)
        nx1 = min(fx1, nx1)
        ny0 = max(fy0, ny0)
        ny1 = min(fy1, ny1)
    ax.set_xlim(nx0, nx1)
    ax.set_ylim(ny0, ny1)
    self._t7_zoom = (tuple(ax.get_xlim()), tuple(ax.get_ylim()))
    self.line_profile_canvas_image.draw_idle()

def _t7_wheel(self, event):
    if getattr(event, 'num', None) == 4 or getattr(event, 'delta', 0) > 0:
        _t7_zoom(self, event, 1 / 1.25)
    elif getattr(event, 'num', None) == 5 or getattr(event, 'delta', 0) < 0:
        _t7_zoom(self, event, 1.25)
    return 'break'

def _t7_reset_zoom(self):
    st = _t7_stack(self)
    if st is None:
        return
    h, w = st[self._t7_frame].shape
    full = (0.0, w * self._t7_sx_nm_px, 0.0, h * self._t7_sy_nm_px)
    self.line_profile_ax_image.set_xlim(full[0], full[1])
    self.line_profile_ax_image.set_ylim(full[2], full[3])
    self._t7_zoom = (tuple(self.line_profile_ax_image.get_xlim()), tuple(self.line_profile_ax_image.get_ylim()))
    self.line_profile_canvas_image.draw_idle()

def _t7_frame_changed(self, value=None):
    st = _t7_stack(self)
    if st is None:
        return 'break'
    try:
        ui = int(round(float(value if value is not None else self._t7_frame_ui_var.get())))
    except Exception:
        ui = 1
    idx = max(0, min(ui - 1, len(st) - 1))
    self._t7_frame = idx
    self._t7_frame_ui_var.set(idx + 1)
    self.line_profile_frame_var.set(idx)
    _t7_render(self, idx, preserve_zoom=True)
    return 'break'

def _t7_wheel_frame(self, event):
    st = _t7_stack(self)
    if st is None:
        return 'break'
    step = 1 if getattr(event, 'num', None) == 4 or getattr(event, 'delta', 0) > 0 else -1
    cur = int(self._t7_frame_ui_var.get())
    new = max(1, min(cur + step, len(st)))
    _t7_frame_changed(self, new)
    return 'break'

def _t7_measure_start(self):
    if not self._t7_lines:
        self.line_profile_measure_var.set('Draw at least one profile line before measuring.')
        return
    self._t7_measure_mode = True
    self._t7_measure_first = None
    self.line_profile_measure_var.set('MEASURE MODE — click M1 and M2 on the same profile curve.')

def _t7_graph_click(self, event):
    if not self._t7_measure_mode or event.inaxes is not self.line_profile_ax_graph or event.xdata is None or (event.ydata is None):
        return
    x = float(event.xdata)
    y = float(event.ydata)
    best = None
    besterr = np.inf
    for j, line in enumerate(self._t7_lines):
        d = np.asarray(line.get('distance_nm', []), float)
        p = np.asarray(line.get('profile', []), float)
        if d.size < 2:
            continue
        xp = float(np.clip(x, d[0], d[-1]))
        yp = float(np.interp(xp, d, p))
        err = abs(yp - y)
        if err < besterr:
            best = j
            besterr = err
    if best is None:
        return
    line = self._t7_lines[best]
    d = np.asarray(line['distance_nm'], float)
    p = np.asarray(line['profile'], float)
    xm = float(np.clip(x, d[0], d[-1]))
    ym = float(np.interp(xm, d, p))
    if self._t7_measure_first is None:
        self._t7_measure_first = {'set': best + 1, 'x': xm, 'y': ym}
        self.line_profile_measure_var.set(f'M1 selected on Line {best + 1}: {xm:.6g} nm. Click M2.')
        return
    first = self._t7_measure_first
    if first['set'] != best + 1:
        self._t7_measure_first = None
        self.line_profile_measure_var.set('M1 and M2 must be on the same line. Click M1 again.')
        return
    m1 = min(first['x'], xm)
    m2 = max(first['x'], xm)
    i1 = float(np.interp(m1, d, p))
    i2 = float(np.interp(m2, d, p))
    self._t7_measurements.append({'set': best + 1, 'm1_nm': m1, 'm2_nm': m2, 'delta_nm': m2 - m1, 'm1_intensity': i1, 'm2_intensity': i2})
    self._t7_measure_first = None
    self._t7_measure_mode = False
    _t7_update_table(self)
    self.line_profile_measure_var.set(f'Line {best + 1}: M1={m1:.6g} nm | M2={m2:.6g} nm | ΔL={m2 - m1:.6g} nm')

def _t7_update_table(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    for iid in tree.get_children():
        tree.delete(iid)
    for m in self._t7_measurements:
        tree.insert('', 'end', values=(m['set'], f"{m['m1_nm']:.6g}", f"{m['m2_nm']:.6g}", f"{m['delta_nm']:.6g}", f"{m['m1_intensity']:.6g}", f"{m['m2_intensity']:.6g}"))

def _t7_export_profiles(self):
    if not self._t7_lines:
        messagebox.showwarning('Line Profile', 'No profile lines to export.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(parent=self.root, title='Export line profiles', defaultextension='.csv', filetypes=[('CSV', '*.csv')])
    if not fp:
        return
    rows = []
    idx = self._t7_frame
    field = float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else float(idx)
    for j, line in enumerate(self._t7_lines, 1):
        for d, y in zip(line.get('distance_nm', []), line.get('profile', [])):
            rows.append({'Frame': idx + 1, 'Field_mT': field, 'Set': j, 'Angle_deg': float(line['angle_deg']), 'Distance_nm': float(d), 'Intensity': float(y)})
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} profile points.')

def _t7_export_measurements(self):
    if not self._t7_measurements:
        messagebox.showwarning('Line Profile', 'No measurements to export.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(parent=self.root, title='Export line measurements', defaultextension='.csv', filetypes=[('CSV', '*.csv')])
    if not fp:
        return
    pd.DataFrame(self._t7_measurements).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(self._t7_measurements)} measurements.')
CDIWorkflowApp._t7_fetch_tab6_roi2 = _t7_fetch_tab6_roi2
CDIWorkflowApp._t7_stack = _t7_stack
CDIWorkflowApp._t7_render = _t7_render
CDIWorkflowApp._t7_start_draw = _t7_start_draw
CDIWorkflowApp._t7_img_press = _t7_img_press
CDIWorkflowApp._t7_img_move = _t7_img_move
CDIWorkflowApp._t7_img_release = _t7_img_release
CDIWorkflowApp._t7_clear_lines = _t7_clear_lines
CDIWorkflowApp._t7_redo_last = _t7_redo_last
CDIWorkflowApp._t7_zoom = _t7_zoom
CDIWorkflowApp._t7_wheel = _t7_wheel
CDIWorkflowApp._t7_reset_zoom = _t7_reset_zoom
CDIWorkflowApp._t7_frame_changed = _t7_frame_changed
CDIWorkflowApp._t7_wheel_frame = _t7_wheel_frame
CDIWorkflowApp._t7_measure_start = _t7_measure_start
CDIWorkflowApp._t7_graph_click = _t7_graph_click
CDIWorkflowApp._t7_update_table = _t7_update_table
CDIWorkflowApp._t7_export_profiles = _t7_export_profiles
CDIWorkflowApp._t7_export_measurements = _t7_export_measurements
CDIWorkflowApp._t7_redraw_profiles = _t7_redraw_profiles
CDIWorkflowApp._t7_redraw_lines = _t7_redraw_lines

def _t7_legacy_sync(self, *args, **kwargs):
    if getattr(self, '_t7_loaded', False):
        _t7_render(self, self._t7_frame, preserve_zoom=True)
    return None
CDIWorkflowApp._line_profile_sync_from_secondary_roi = _t7_legacy_sync
CDIWorkflowApp._line_profile_update_image = lambda self, idx=None, preserve_lines=True: _t7_render(self, self._t7_frame if idx is None else idx, preserve_zoom=True)
CDIWorkflowApp._line_profile_frame_slider_changed = _t7_frame_changed
_T7_OLD_CONFIRM = CDIWorkflowApp.confirm_secondary_roi_from_processing

def _t7_confirm_then_fetch(self, *args, **kwargs):
    result = _T7_OLD_CONFIRM(self, *args, **kwargs)
    try:
        if getattr(self, 'roi2_stack', None) is not None and getattr(self, 'roi2_confirmed_coords', None) is not None:
            self._t7_fetch_tab6_roi2()
    except Exception as exc:
        try:
            self.log(f'Tab 7 ROI2 fetch warning: {type(exc).__name__}: {exc}')
        except Exception:
            pass
    return result
CDIWorkflowApp.confirm_secondary_roi_from_processing = _t7_confirm_then_fetch

def _t7_tab_changed(self, event=None):
    try:
        if event is not None and event.widget.select() != str(self.tab_line_profile):
            return
        if not getattr(self, '_t7_loaded', False):
            self._t7_fetch_tab6_roi2()
        elif _t7_stack(self) is not None:
            _t7_render(self, self._t7_frame, preserve_zoom=True)
    except Exception:
        pass
CDIWorkflowApp._t7_tab_changed = _t7_tab_changed
_T7_BUILD_UI_PREV = CDIWorkflowApp._build_ui

def _t7_build_ui_final(self, *args, **kwargs):
    result = _T7_BUILD_UI_PREV(self, *args, **kwargs)
    try:
        self.nb.bind('<<NotebookTabChanged>>', self._t7_tab_changed, add='+')
    except Exception:
        pass
    return result
CDIWorkflowApp._build_ui = _t7_build_ui_final

def _tab7_layout_fixed_rebuild(self):
    tab = self.tab_line_profile
    for child in list(tab.winfo_children()):
        try:
            child.destroy()
        except Exception:
            pass
    self._t7_frame = int(getattr(self, '_t7_frame', 0))
    self._t7_lines = getattr(self, '_t7_lines', [])
    self._t7_measurements = getattr(self, '_t7_measurements', [])
    self._t7_average = getattr(self, '_t7_average', None)
    self._t7_draw_mode = False
    self._t7_drag_start = None
    self._t7_temp_line = None
    self._t7_measure_mode = False
    self._t7_measure_first = None
    self._t7_zoom = getattr(self, '_t7_zoom', None)
    self._t7_image_artists = []
    self._t7_frame_ui_var = tk.IntVar(value=max(1, int(getattr(self, '_t7_frame', 0)) + 1))
    self._t7_n_lines_var = tk.IntVar(value=max(1, min(12, int(getattr(self, '_t7_n_lines_var', tk.IntVar(value=3)).get()))) if hasattr(getattr(self, '_t7_n_lines_var', None), 'get') else 3)
    self.line_profile_frame_var = tk.IntVar(value=max(0, int(getattr(self, '_t7_frame', 0))))
    self.line_profile_status_var = tk.StringVar(value='Confirm Secondary ROI2 in Tab 6, then click FETCH ROI2 FROM TAB 6.')
    self.line_profile_measure_var = tk.StringVar(value='Draw profile lines, then use MEASURE M1 / M2 on the graph.')
    outer = ttk.Frame(tab, padding=7)
    outer.pack(fill='both', expand=True)
    header = ttk.Frame(outer)
    header.pack(fill='x', pady=(0, 5))
    ttk.Label(header, text='LINE PROFILE — TAB-6 CONFIRMED ROI', font=('Arial', 12, 'bold')).pack(side='left')
    ttk.Label(header, text='Independent processed ROI copy', foreground='darkgreen').pack(side='left', padx=12)
    controls = ttk.LabelFrame(outer, text='FRAME / PROFILE CONTROLS', padding=5)
    controls.pack(fill='x', expand=False, pady=(0, 5))
    controls.grid_columnconfigure(3, weight=1)
    ttk.Button(controls, text='FETCH ROI2 FROM TAB 6', command=self._t7_fetch_tab6_roi2, width=23).grid(row=0, column=0, padx=(2, 8), pady=2, sticky='w')
    ttk.Label(controls, text='Frame:').grid(row=0, column=1, padx=(0, 5), pady=2, sticky='w')
    self.line_profile_frame_slider = tk.Scale(controls, from_=1, to=1, resolution=1, orient='horizontal', variable=self._t7_frame_ui_var, showvalue=True, length=420, highlightthickness=0, bd=1, command=self._t7_frame_changed)
    self.line_profile_frame_slider.grid(row=0, column=2, padx=3, pady=0, sticky='w')
    self.line_profile_frame_label = ttk.Label(controls, text='Frame --', width=26)
    self.line_profile_frame_label.grid(row=0, column=3, padx=(10, 2), pady=2, sticky='w')
    ttk.Label(controls, text='No. of lines:').grid(row=0, column=4, padx=(12, 3), pady=2, sticky='e')
    tk.Spinbox(controls, from_=1, to=12, width=4, textvariable=self._t7_n_lines_var).grid(row=0, column=5, padx=(0, 2), pady=2, sticky='w')
    buttons1 = ttk.Frame(controls)
    buttons1.grid(row=1, column=0, columnspan=6, sticky='w', pady=(4, 1))
    button_specs = [('DRAW LINES', self._t7_start_draw, 14), ('CLEAR LINES', self._t7_clear_lines, 14), ('REDO LAST', self._t7_redo_last, 13), ('ZOOM IN', lambda: self._t7_zoom(factor=1 / 1.35), 11), ('ZOOM OUT', lambda: self._t7_zoom(factor=1.35), 11), ('RESET ZOOM', self._t7_reset_zoom, 13), ('MEASURE', self._t7_measure_start, 12)]
    for j, (label, cmd, width) in enumerate(button_specs):
        ttk.Button(buttons1, text=label, command=cmd, width=width).pack(side='left', padx=2)
    buttons2 = ttk.Frame(controls)
    buttons2.grid(row=2, column=0, columnspan=6, sticky='w', pady=(3, 1))
    ttk.Button(buttons2, text='EXPORT PROFILES CSV', command=self._t7_export_profiles, width=19).pack(side='left', padx=2)
    ttk.Button(buttons2, text='EXPORT MEASUREMENTS CSV', command=self._t7_export_measurements, width=22).pack(side='left', padx=2)
    ttk.Label(buttons2, text='Physical distance = nm').pack(side='left', padx=10)
    ttk.Label(outer, textvariable=self.line_profile_status_var, wraplength=1600, justify='left').pack(anchor='w', pady=(2, 1))
    ttk.Label(outer, textvariable=self.line_profile_measure_var, wraplength=1600, justify='left').pack(anchor='w', pady=(1, 4))
    main = ttk.Panedwindow(outer, orient='vertical')
    main.pack(fill='both', expand=True)
    plots = ttk.Frame(main)
    table = ttk.Frame(main)
    main.add(plots, weight=6)
    main.add(table, weight=2)
    horizontal = ttk.Panedwindow(plots, orient='horizontal')
    horizontal.pack(fill='both', expand=True)
    image_frame = ttk.Frame(horizontal)
    graph_frame = ttk.Frame(horizontal)
    horizontal.add(image_frame, weight=1)
    horizontal.add(graph_frame, weight=1)
    self.line_profile_fig, self.line_profile_ax_image = plt.subplots(figsize=(7.2, 5.7), dpi=100)
    self.line_profile_image = self.line_profile_ax_image.imshow(np.zeros((10, 10)), origin='lower', cmap='RdBu_r', vmin=0, vmax=1, interpolation='nearest', extent=(0, 10, 0, 10), aspect='equal')
    self.line_profile_colorbar = self.line_profile_fig.colorbar(self.line_profile_image, ax=self.line_profile_ax_image, fraction=0.046, pad=0.04)
    self.line_profile_colorbar.set_label('Tab-6 ROI2 processed contrast')
    self.line_profile_ax_image.set_xlabel('X (nm)')
    self.line_profile_ax_image.set_ylabel('Y (nm)')
    self.line_profile_ax_image.set_title('Waiting for confirmed ROI2 from Tab 6')
    self.line_profile_canvas_image = FigureCanvasTkAgg(self.line_profile_fig, master=image_frame)
    self.line_profile_canvas_image.draw()
    self.line_profile_canvas_image.get_tk_widget().pack(fill='both', expand=True)
    self.line_profile_fig2, self.line_profile_ax_graph = plt.subplots(figsize=(7.2, 5.7), dpi=100)
    self.line_profile_ax_graph.set_xlabel('Distance (nm)')
    self.line_profile_ax_graph.set_ylabel('Intensity')
    self.line_profile_ax_graph.set_title('Line Profiles')
    self.line_profile_ax_graph.grid(True, alpha=0.22)
    self.line_profile_canvas_graph = FigureCanvasTkAgg(self.line_profile_fig2, master=graph_frame)
    self.line_profile_canvas_graph.draw()
    self.line_profile_canvas_graph.get_tk_widget().pack(fill='both', expand=True)
    columns = ('Set', 'M1 position (nm)', 'M2 position (nm)', 'ΔL (nm)', 'M1 intensity', 'M2 intensity')
    self.line_profile_tree = ttk.Treeview(table, columns=columns, show='headings', height=7)
    widths = (65, 150, 150, 110, 150, 150)
    for col, width in zip(columns, widths):
        self.line_profile_tree.heading(col, text=col)
        self.line_profile_tree.column(col, width=width, anchor='center')
    self.line_profile_tree.pack(side='left', fill='both', expand=True)
    table_scroll = ttk.Scrollbar(table, orient='vertical', command=self.line_profile_tree.yview)
    table_scroll.pack(side='right', fill='y')
    self.line_profile_tree.configure(yscrollcommand=table_scroll.set)
    self._t7_connections = [self.line_profile_canvas_image.mpl_connect('button_press_event', self._t7_img_press), self.line_profile_canvas_image.mpl_connect('motion_notify_event', self._t7_img_move), self.line_profile_canvas_image.mpl_connect('button_release_event', self._t7_img_release), self.line_profile_canvas_image.mpl_connect('scroll_event', self._t7_wheel), self.line_profile_canvas_graph.mpl_connect('button_press_event', self._t7_graph_click)]
    self.line_profile_frame_slider.bind('<MouseWheel>', self._t7_wheel_frame, add='+')
    self.line_profile_frame_slider.bind('<Button-4>', self._t7_wheel_frame, add='+')
    self.line_profile_frame_slider.bind('<Button-5>', self._t7_wheel_frame, add='+')
    try:
        if getattr(self, 'roi2_stack', None) is not None and getattr(self, 'roi2_confirmed_coords', None) is not None:
            self._t7_fetch_tab6_roi2()
    except Exception:
        pass
_TAB7_LAYOUT_BUILD_OLD = CDIWorkflowApp._build_ui

def _tab7_layout_build_final(self, *args, **kwargs):
    result = _TAB7_LAYOUT_BUILD_OLD(self, *args, **kwargs)
    try:
        self.after(100, self._tab7_force_layout_rebuild)
    except Exception:
        pass
    return result

def _tab7_force_layout_rebuild(self):
    try:
        _tab7_layout_fixed_rebuild(self)
        if getattr(self, 'roi2_stack', None) is not None and getattr(self, 'roi2_confirmed_coords', None) is not None:
            self._t7_fetch_tab6_roi2()
    except Exception as exc:
        try:
            self.log(f'Tab 7 layout rebuild warning: {type(exc).__name__}: {exc}')
        except Exception:
            pass
CDIWorkflowApp._build_ui = _tab7_layout_build_final
CDIWorkflowApp._tab7_force_layout_rebuild = _tab7_force_layout_rebuild

def _t7_dynamic_scale_start(self):
    if _t7_stack(self) is None:
        try:
            self.line_profile_measure_var.set('Fetch the confirmed ROI2 from Tab 6 first.')
        except Exception:
            pass
        return
    if not getattr(self, '_t7_lines', None):
        try:
            self.line_profile_measure_var.set('Draw at least one profile line before using DYNAMIC MEASURING SCALE.')
        except Exception:
            pass
        return
    self._t7_dynamic_scale_active = True
    self._t7_dynamic_measure_line = None
    self._t7_dynamic_m1_t = None
    self._t7_dynamic_m2_t = None
    self._t7_dynamic_hover_t = None
    self._t7_remove_dynamic_artists()
    try:
        self.line_profile_measure_var.set('DYNAMIC SCALE ON — click a drawn line to set M1; move to preview M2; click again to set M2.')
    except Exception:
        pass
    _t7_redraw_lines(self)
    self._t7_draw_dynamic_scale()

def _t7_remove_dynamic_artists(self):
    for artist in getattr(self, '_t7_dynamic_artists', []):
        try:
            artist.remove()
        except Exception:
            pass
    self._t7_dynamic_artists = []
    for artist in getattr(self, '_t7_dynamic_graph_artists', []):
        try:
            artist.remove()
        except Exception:
            pass
    self._t7_dynamic_graph_artists = []

def _t7_find_nearest_drawn_line(self, x_nm, y_nm):
    sx, sy = (self._t7_sx_nm_px, self._t7_sy_nm_px)
    qx = float(x_nm) / sx
    qy = float(y_nm) / sy
    best = None
    best_d2 = np.inf
    for i, line in enumerate(getattr(self, '_t7_lines', [])):
        p0 = np.asarray(line['p0'], dtype=float)
        p1 = np.asarray(line['p1'], dtype=float)
        v = p1 - p0
        vv = float(np.dot(v, v))
        if vv <= 1e-15:
            t = 0.0
            p = p0
        else:
            t = float(np.dot(np.asarray([qx, qy]) - p0, v) / vv)
            t = max(0.0, min(1.0, t))
            p = p0 + t * v
        dx = (qx - p[0]) * sx
        dy = (qy - p[1]) * sy
        d2 = dx * dx + dy * dy
        if d2 < best_d2:
            best_d2 = d2
            best = (i, t)
    if best is None:
        return None
    return best

def _t7_dynamic_point_on_line(self, line, t):
    p0 = np.asarray(line['p0'], dtype=float)
    p1 = np.asarray(line['p1'], dtype=float)
    p = p0 + float(t) * (p1 - p0)
    return (float(p[0]), float(p[1]))

def _t7_dynamic_distance_nm(self, line, t0, t1):
    p0 = np.asarray(line['p0'], dtype=float)
    p1 = np.asarray(line['p1'], dtype=float)
    delta = (float(t1) - float(t0)) * (p1 - p0)
    return float(np.hypot(delta[0] * self._t7_sx_nm_px, delta[1] * self._t7_sy_nm_px))

def _t7_dynamic_scale_update(self, line_index, t_m1=None, t_m2=None, tentative=False):
    self._t7_remove_dynamic_artists()
    if line_index is None or line_index < 0 or line_index >= len(self._t7_lines):
        return
    line = self._t7_lines[line_index]
    if t_m1 is None:
        return
    if t_m2 is None:
        t_m2 = t_m1
    p1 = self._t7_dynamic_point_on_line(line, t_m1)
    p2 = self._t7_dynamic_point_on_line(line, t_m2)
    sx, sy = self._t7_sx_nm_px
    a = (float(line['p0'][0] * sx), float(line['p0'][1] * sy))
    b = (float(line['p1'][0] * sx), float(line['p1'][1] * sy))
    m1_nm = (float(p1[0] * sx), float(p1[1] * sy))
    m2_nm = (float(p2[0] * sx), float(p2[1] * sy))
    seg, = self.line_profile_ax_image.plot([m1_nm[0], m2_nm[0]], [m1_nm[1], m2_nm[1]], color='yellow', lw=4.5, solid_capstyle='round', zorder=110)
    self._t7_dynamic_artists.append(seg)
    m1_artist, = self.line_profile_ax_image.plot([m1_nm[0]], [m1_nm[1]], marker='o', ms=8, markerfacecolor='yellow', markeredgecolor='black', markeredgewidth=1.2, zorder=112)
    self._t7_dynamic_artists.append(m1_artist)
    m2_artist, = self.line_profile_ax_image.plot([m2_nm[0]], [m2_nm[1]], marker='o', ms=8, markerfacecolor='yellow', markeredgecolor='black', markeredgewidth=1.2, zorder=112)
    self._t7_dynamic_artists.append(m2_artist)
    length_nm = self._t7_dynamic_distance_nm(line, t_m1, t_m2)
    tx = 0.5 * (m1_nm[0] + m2_nm[0])
    ty = 0.5 * (m1_nm[1] + m2_nm[1])
    label = self.line_profile_ax_image.text(tx, ty, f'ΔL = {length_nm:.4f} nm', color='yellow', fontsize=9, fontweight='bold', ha='center', va='bottom', zorder=113, bbox=dict(boxstyle='round,pad=0.2', facecolor='black', edgecolor='yellow', alpha=0.7))
    self._t7_dynamic_artists.append(label)
    try:
        d = np.asarray(line.get('distance_nm', []), dtype=float)
        p = np.asarray(line.get('profile', []), dtype=float)
        if d.size >= 2 and p.size == d.size:
            line_len = float(d[-1])
            x1 = max(0.0, min(line_len, float(t_m1) * line_len))
            x2 = max(0.0, min(line_len, float(t_m2) * line_len))
            lo, hi = (min(x1, x2), max(x1, x2))
            yl = self.line_profile_ax_graph.axvspan(lo, hi, color='yellow', alpha=0.18, zorder=4)
            self._t7_dynamic_graph_artists.append(yl)
            v1 = self.line_profile_ax_graph.axvline(x1, color='yellow', lw=2.2, zorder=8)
            v2 = self.line_profile_ax_graph.axvline(x2, color='yellow', lw=2.2, zorder=8)
            self._t7_dynamic_graph_artists.extend([v1, v2])
            g1, = self.line_profile_ax_graph.plot([x1], [float(np.interp(x1, d, p))], 'o', ms=7, color='yellow', markeredgecolor='black', zorder=9)
            g2, = self.line_profile_ax_graph.plot([x2], [float(np.interp(x2, d, p))], 'o', ms=7, color='yellow', markeredgecolor='black', zorder=9)
            self._t7_dynamic_graph_artists.extend([g1, g2])
    except Exception:
        pass
    try:
        self.line_profile_canvas_image.draw_idle()
        self.line_profile_canvas_graph.draw_idle()
    except Exception:
        pass
    if tentative:
        self.line_profile_measure_var.set(f'DYNAMIC SCALE — Line {line_index + 1} | M1 → current pointer | ΔL = {length_nm:.4f} nm | Click to set M2.')
    else:
        self.line_profile_measure_var.set(f'DYNAMIC SCALE — Line {line_index + 1} | M1/M2 distance = {length_nm:.4f} nm')

def _t7_dynamic_image_press(self, event):
    if not getattr(self, '_t7_dynamic_scale_active', False):
        return
    if event.inaxes is not self.line_profile_ax_image:
        return
    if event.xdata is None or event.ydata is None:
        return
    found = _t7_find_nearest_drawn_line(self, float(event.xdata), float(event.ydata))
    if found is None:
        return
    line_index, t = found
    if self._t7_dynamic_measure_line is None:
        self._t7_dynamic_measure_line = int(line_index)
        self._t7_dynamic_m1_t = float(t)
        self._t7_dynamic_m2_t = None
        self._t7_dynamic_hover_t = float(t)
        self._t7_dynamic_scale_update(line_index, t, t, tentative=True)
    else:
        if int(line_index) != int(self._t7_dynamic_measure_line):
            self.line_profile_measure_var.set(f'Keep M1 and M2 on the same drawn line (Line {self._t7_dynamic_measure_line + 1}).')
            return
        self._t7_dynamic_m2_t = float(t)
        self._t7_dynamic_hover_t = float(t)
        self._t7_dynamic_scale_update(self._t7_dynamic_measure_line, self._t7_dynamic_m1_t, self._t7_dynamic_m2_t, tentative=False)

def _t7_dynamic_image_motion(self, event):
    if not getattr(self, '_t7_dynamic_scale_active', False):
        return
    if self._t7_dynamic_measure_line is None:
        return
    if event.inaxes is not self.line_profile_ax_image:
        return
    if event.xdata is None or event.ydata is None:
        return
    if self._t7_dynamic_m1_t is None:
        return
    found = _t7_find_nearest_drawn_line(self, float(event.xdata), float(event.ydata))
    if found is None:
        return
    line_index, t = found
    if int(line_index) != int(self._t7_dynamic_measure_line):
        return
    self._t7_dynamic_hover_t = float(t)
    self._t7_dynamic_scale_update(self._t7_dynamic_measure_line, self._t7_dynamic_m1_t, self._t7_dynamic_hover_t, tentative=True)

def _t7_dynamic_image_release(self, event):
    return

def _t7_dynamic_finish(self):
    self._t7_dynamic_scale_active = False
    self._t7_dynamic_measure_line = None
    self._t7_dynamic_m1_t = None
    self._t7_dynamic_m2_t = None
    self._t7_dynamic_hover_t = None
    self._t7_remove_dynamic_artists()
    try:
        self.line_profile_measure_var.set('Dynamic measuring scale OFF.')
        self.line_profile_canvas_image.draw_idle()
        self.line_profile_canvas_graph.draw_idle()
    except Exception:
        pass
CDIWorkflowApp._t7_dynamic_scale_start = _t7_dynamic_scale_start
CDIWorkflowApp._t7_remove_dynamic_artists = _t7_remove_dynamic_artists
CDIWorkflowApp._t7_find_nearest_drawn_line = _t7_find_nearest_drawn_line
CDIWorkflowApp._t7_dynamic_scale_update = _t7_dynamic_scale_update
CDIWorkflowApp._t7_dynamic_image_press = _t7_dynamic_image_press
CDIWorkflowApp._t7_dynamic_image_motion = _t7_dynamic_image_motion
CDIWorkflowApp._t7_dynamic_image_release = _t7_dynamic_image_release
CDIWorkflowApp._t7_dynamic_finish = _t7_dynamic_finish
CDIWorkflowApp._t7_dynamic_point_on_line = _t7_dynamic_point_on_line
CDIWorkflowApp._t7_dynamic_distance_nm = _t7_dynamic_distance_nm
_TAB7_PREVIOUS_BUILD = CDIWorkflowApp._build_line_profile_tab

def _tab7_build_with_dynamic_scale(self):
    _TAB7_PREVIOUS_BUILD(self)
    try:
        containers = []

        def walk(w):
            containers.append(w)
            try:
                for ch in w.winfo_children():
                    walk(ch)
            except Exception:
                pass
        walk(self.tab_line_profile)
        line_row = None
        for w in containers:
            try:
                texts = [str(ch.cget('text')) for ch in w.winfo_children() if isinstance(ch, (ttk.Button, tk.Button))]
                if 'DRAW LINES' in texts and 'RESET ZOOM' in texts:
                    line_row = w
                    break
            except Exception:
                pass
        if line_row is not None:
            for ch in list(line_row.winfo_children()):
                try:
                    if str(ch.cget('text')) == 'DYNAMIC MEASURING SCALE':
                        ch.destroy()
                except Exception:
                    pass
            ttk.Button(line_row, text='DYNAMIC MEASURING SCALE', command=self._t7_dynamic_scale_start, width=22).pack(side='left', padx=2)
    except Exception:
        pass
_TAB7_BUILD_UI_BEFORE_DYNAMIC = CDIWorkflowApp._build_ui

def _tab7_dynamic_build_ui(self, *args, **kwargs):
    result = _TAB7_BUILD_UI_BEFORE_DYNAMIC(self, *args, **kwargs)
    try:
        canvas = self.line_profile_canvas_image
        self._t7_dynamic_connections = [canvas.mpl_connect('button_press_event', self._t7_dynamic_image_press), canvas.mpl_connect('motion_notify_event', self._t7_dynamic_image_motion), canvas.mpl_connect('button_release_event', self._t7_dynamic_image_release)]
        self._t7_dynamic_scale_active = False
        self._t7_dynamic_measure_line = None
        self._t7_dynamic_m1_t = None
        self._t7_dynamic_m2_t = None
        self._t7_dynamic_hover_t = None
        self._t7_dynamic_artists = []
        self._t7_dynamic_graph_artists = []
    except Exception:
        pass
    return result
CDIWorkflowApp._build_ui = _tab7_dynamic_build_ui
_TAB7_OLD_FRAME_CHANGED_DYNAMIC = CDIWorkflowApp._t7_frame_changed

def _t7_frame_changed_dynamic(self, value=None):
    result = _TAB7_OLD_FRAME_CHANGED_DYNAMIC(self, value)
    try:
        if getattr(self, '_t7_dynamic_measure_line', None) is not None and getattr(self, '_t7_dynamic_m1_t', None) is not None and (getattr(self, '_t7_dynamic_m2_t', None) is not None):
            self._t7_dynamic_scale_update(self._t7_dynamic_measure_line, self._t7_dynamic_m1_t, self._t7_dynamic_m2_t, tentative=False)
    except Exception:
        pass
    return result
CDIWorkflowApp._t7_frame_changed = _t7_frame_changed_dynamic

def _ensure_tab6_processing_vars(self):
    """Guarantee the variables required by _proc_image/_proc_update exist."""
    if not hasattr(self, 'filter_var'):
        self.filter_var = tk.StringVar(value='Gaussian')
    if not hasattr(self, 'strength_var'):
        self.strength_var = tk.DoubleVar(value=3.0)
    if not hasattr(self, 'colormap_var'):
        self.colormap_var = tk.StringVar(value='RdBu_r')
    if not hasattr(self, 'view_var'):
        self.view_var = tk.StringVar(value='IFFT')
    if not hasattr(self, 'fft_mask_var'):
        self.fft_mask_var = tk.DoubleVar(value=0.6)
    if not hasattr(self, 'brightness_var'):
        self.brightness_var = tk.DoubleVar(value=0.0)
    if not hasattr(self, 'contrast_var'):
        self.contrast_var = tk.DoubleVar(value=1.0)
    if not hasattr(self, 'proc_idx_var'):
        self.proc_idx_var = tk.IntVar(value=0)
    if not hasattr(self, 'fps_var'):
        self.fps_var = tk.IntVar(value=10)
_TAB6_SAFE_UPDATE_ORIGINAL = CDIWorkflowApp.update_processing_view

def _tab6_safe_update_processing_view(self, *args, **kwargs):
    _ensure_tab6_processing_vars(self)
    return _TAB6_SAFE_UPDATE_ORIGINAL(self, *args, **kwargs)
CDIWorkflowApp._ensure_tab6_processing_vars = _ensure_tab6_processing_vars
CDIWorkflowApp.update_processing_view = _tab6_safe_update_processing_view
CDIWorkflowApp._line_profile_frame_mousewheel = _line_profile_frame_mousewheel_independent
CDIWorkflowApp._line_profile_frame_slider_changed = _line_profile_frame_slider_changed_independent
CDIWorkflowApp._line_profile_on_tab_selected = _line_profile_on_tab_selected
CDIWorkflowApp._line_profile_sync_from_secondary_roi = _line_profile_sync_from_secondary_roi_refined
CDIWorkflowApp._line_profile_sync_from_tab5 = _line_profile_sync_from_secondary_roi_refined
CDIWorkflowApp._line_profile_start_measurement = _line_profile_start_measurement_refined
CDIWorkflowApp._line_profile_clear_measurement = _line_profile_clear_measurement_refined
CDIWorkflowApp._line_profile_refresh_measurement_values = _line_profile_refresh_measurement_values_refined
CDIWorkflowApp._line_profile_find_profile_point = _line_profile_find_profile_point_refined
CDIWorkflowApp._line_profile_graph_press = _line_profile_graph_press_refined
CDIWorkflowApp._line_profile_graph_motion = _line_profile_graph_motion_refined
CDIWorkflowApp._line_profile_graph_release = _line_profile_graph_release_refined
CDIWorkflowApp._line_profile_redraw_image_lines = _line_profile_redraw_image_lines_refined
CDIWorkflowApp._line_profile_redraw_image_lines_refined = _line_profile_redraw_image_lines_refined
CDIWorkflowApp._line_profile_draw_measurement = _line_profile_draw_measurement_refined
CDIWorkflowApp._line_profile_update_table = _line_profile_update_table_refined
CDIWorkflowApp._line_profile_update_measurement_table = _line_profile_update_measurement_table
CDIWorkflowApp._line_profile_recalculate_all = _line_profile_recalculate_all_refined
CDIWorkflowApp._line_profile_export_results = _line_profile_export_results_refined
CDIWorkflowApp._line_profile_redraw_profiles = _line_profile_redraw_profiles_refined

def _lp_nm_scale_from_tab6(self):
    """
    Return X/Y nm per Tab-6 output pixel.

    Tab 5 calibration is authoritative. Tab 6 output resolution determines
    the local sampling scale:
        X scale = base_nm_per_px * original_width / output_width
        Y scale = base_nm_per_px * original_height / output_height
    """
    try:
        sx = float(getattr(self, 'proc_profile_measure_nm_per_px'))
        if np.isfinite(sx) and sx > 0:
            ow = int(getattr(self, 'original_roi_w', 0) or 0)
            oh = int(getattr(self, 'original_roi_h', 0) or 0)
            rw, rh = getattr(self, 'roi_resolution', (0, 0))
            rw, rh = (int(rw), int(rh))
            if ow > 0 and oh > 0 and (rw > 0) and (rh > 0):
                sy = sx * (float(oh) / float(ow))
                return (sx, sy)
            return (sx, sx)
    except Exception:
        pass
    try:
        base = float(self._proc_get_tab5_base_pixel_size())
        ow = int(getattr(self, 'original_roi_w', 0) or 0)
        oh = int(getattr(self, 'original_roi_h', 0) or 0)
        rw, rh = getattr(self, 'roi_resolution', (0, 0))
        rw, rh = (int(rw), int(rh))
        if np.isfinite(base) and base > 0 and (ow > 0) and (oh > 0) and (rw > 0) and (rh > 0):
            return (base * float(ow) / float(rw), base * float(oh) / float(rh))
    except Exception:
        pass
    try:
        v = float(getattr(self, 'primary_roi_final_nm_per_pixel'))
        if np.isfinite(v) and v > 0:
            return (v, v)
    except Exception:
        pass
    return (1.0, 1.0)
CDIWorkflowApp._line_profile_nm_scale = _lp_nm_scale_from_tab6
_LP_REF_BUILD_FINAL = CDIWorkflowApp._build_line_profile_tab
_LP_REF_UPDATE_IMAGE = CDIWorkflowApp._line_profile_update_image

def _line_profile_update_image_px_to_nm(self, idx=None, preserve_lines=True):
    result = _LP_REF_UPDATE_IMAGE(self, idx, preserve_lines)
    try:
        stack = getattr(self, 'roi2_stack', None)
        if stack is None or len(stack) == 0:
            return result
        if idx is None:
            idx = int(round(float(self.roi2_idx_var.get())))
        idx = max(0, min(int(idx), len(stack) - 1))
        img = np.asarray(stack[idx], dtype=float)
        h, w = img.shape[:2]
        sx, sy = self._line_profile_nm_scale()
        self.line_profile_image.set_extent((0.0, float(w) * sx, 0.0, float(h) * sy))
        self.line_profile_ax_image.set_xlim(0.0, float(w) * sx)
        self.line_profile_ax_image.set_ylim(0.0, float(h) * sy)
        self.line_profile_ax_image.set_xlabel('X (nm)')
        self.line_profile_ax_image.set_ylabel('Y (nm)')
        self._line_profile_redraw_image_lines()
        self.line_profile_canvas_image.draw_idle()
    except Exception:
        pass
    return result
CDIWorkflowApp._line_profile_update_image = _line_profile_update_image_px_to_nm
_LP_REF_REDRAW = CDIWorkflowApp._line_profile_redraw_profiles

def _line_profile_redraw_profiles_px_to_nm(self, idx=None):
    stack = getattr(self, 'roi2_stack', None)
    if stack is None or len(stack) == 0:
        self._line_profile_results = []
        self._line_profile_redraw_image_lines()
        self._line_profile_update_table()
        return
    if idx is None:
        try:
            idx = int(round(float(self.roi2_idx_var.get())))
        except Exception:
            idx = 0
    idx = max(0, min(int(idx), len(stack) - 1))
    self.line_profile_frame_var.set(idx)
    image = np.asarray(stack[idx], dtype=float)
    sx, sy = self._line_profile_nm_scale()
    self.line_profile_ax_graph.clear()
    self.line_profile_ax_graph.set_xlabel('Distance (nm)')
    self.line_profile_ax_graph.set_ylabel('Intensity')
    self.line_profile_ax_graph.set_title(f'Secondary ROI line profiles — Frame {idx + 1}/{len(stack)}')
    self.line_profile_ax_graph.grid(True, alpha=0.22)
    self._line_profile_results = []
    self._line_profile_graph_artists = []
    for i, line in enumerate(getattr(self, '_line_profile_lines', [])):
        dist_px, profile = self._line_profile_extract(image, line['p0'], line['p1'])
        dx_px = float(line['p1'][0] - line['p0'][0])
        dy_px = float(line['p1'][1] - line['p0'][1])
        physical_length_nm = float(np.hypot(dx_px * sx, dy_px * sy))
        px_length = float(dist_px[-1]) if len(dist_px) else 0.0
        if px_length > 0 and physical_length_nm > 0:
            dist_nm = np.asarray(dist_px, dtype=float) * (physical_length_nm / px_length)
        else:
            dist_nm = np.zeros_like(np.asarray(dist_px, dtype=float))
        result = {'set': i + 1, 'p0': line['p0'], 'p1': line['p1'], 'length': float(dist_nm[-1]) if len(dist_nm) else 0.0, 'length_px': px_length, 'dist': dist_nm, 'dist_px': np.asarray(dist_px, dtype=float), 'profile': np.asarray(profile, dtype=float)}
        self._line_profile_results.append(result)
        ln, = self.line_profile_ax_graph.plot(dist_nm, profile, lw=2.0, color=self._line_profile_line_color(i), label=f'Set {i + 1}', picker=14, pickradius=14, zorder=4)
        ln._line_profile_setno = i + 1
        self._line_profile_graph_artists.append(ln)
    if self._line_profile_results:
        self.line_profile_ax_graph.legend(loc='best')
    self._line_profile_current_profile = self._line_profile_results[-1] if self._line_profile_results else None
    self._line_profile_refresh_measurement_values()
    self._line_profile_redraw_image_lines()
    self._line_profile_update_measurement_table()
    self.line_profile_canvas_graph.draw_idle()
CDIWorkflowApp._line_profile_redraw_profiles = _line_profile_redraw_profiles_px_to_nm
_LP_REF_IMAGE_PRESS = CDIWorkflowApp._line_profile_image_press
_LP_REF_IMAGE_MOTION = CDIWorkflowApp._line_profile_image_motion
_LP_REF_IMAGE_RELEASE = CDIWorkflowApp._line_profile_image_release

def _line_profile_image_press_px_to_nm(self, event):
    if event.inaxes is not self.line_profile_ax_image:
        return
    if event.xdata is None or event.ydata is None:
        return
    if not getattr(self, '_line_profile_active', False):
        return
    sx, sy = self._line_profile_nm_scale()
    px = float(event.xdata) / sx
    py = float(event.ydata) / sy
    old_xdata, old_ydata = (event.xdata, event.ydata)
    event.xdata, event.ydata = (px, py)
    try:
        return _LP_REF_IMAGE_PRESS(self, event)
    finally:
        event.xdata, event.ydata = (old_xdata, old_ydata)

def _line_profile_image_motion_px_to_nm(self, event):
    if event.inaxes is not self.line_profile_ax_image:
        return
    if event.xdata is None or event.ydata is None:
        return
    if getattr(self, '_line_profile_press_xy', None) is None:
        return
    sx, sy = self._line_profile_nm_scale()
    px = float(event.xdata) / sx
    py = float(event.ydata) / sy
    old_xdata, old_ydata = (event.xdata, event.ydata)
    event.xdata, event.ydata = (px, py)
    try:
        return _LP_REF_IMAGE_MOTION(self, event)
    finally:
        event.xdata, event.ydata = (old_xdata, old_ydata)

def _line_profile_image_release_px_to_nm(self, event):
    if getattr(self, '_line_profile_press_xy', None) is None:
        return
    sx, sy = self._line_profile_nm_scale()
    old_xdata = getattr(event, 'xdata', None)
    old_ydata = getattr(event, 'ydata', None)
    if event.xdata is not None and event.ydata is not None:
        event.xdata = float(event.xdata) / sx
        event.ydata = float(event.ydata) / sy
    try:
        return _LP_REF_IMAGE_RELEASE(self, event)
    finally:
        event.xdata = old_xdata
        event.ydata = old_ydata
CDIWorkflowApp._line_profile_image_press = _line_profile_image_press_px_to_nm
CDIWorkflowApp._line_profile_image_motion = _line_profile_image_motion_px_to_nm
CDIWorkflowApp._line_profile_image_release = _line_profile_image_release_px_to_nm

def _line_profile_redraw_image_lines_px_to_nm(self):
    for artist in getattr(self, '_line_profile_image_line_artists', []):
        try:
            artist.remove()
        except Exception:
            pass
    self._line_profile_image_line_artists = []
    sx, sy = self._line_profile_nm_scale()
    for i, line in enumerate(getattr(self, '_line_profile_lines', [])):
        c = self._line_profile_line_color(i)
        a, = self.line_profile_ax_image.plot([float(line['p0'][0]) * sx, float(line['p1'][0]) * sx], [float(line['p0'][1]) * sy, float(line['p1'][1]) * sy], '-', lw=2.2, color=c, label=f'Set {i + 1}')
        self._line_profile_image_line_artists.append(a)
    for setno, pts in sorted(getattr(self, '_line_profile_measure_measurements', {}).items()):
        if len(pts) != 2:
            continue
        r = next((rr for rr in getattr(self, '_line_profile_results', []) if int(rr.get('set', -1)) == int(setno)), None)
        if r is None:
            continue
        try:
            L_nm = float(r.get('length', 0.0))
            if L_nm <= 0:
                continue
            xy = []
            for x_nm, _y, _sn in pts:
                frac = float(np.clip(float(x_nm) / L_nm, 0.0, 1.0))
                p0 = np.asarray(r['p0'], dtype=float)
                p1 = np.asarray(r['p1'], dtype=float)
                pos_px = p0 + frac * (p1 - p0)
                xy.append(np.array([pos_px[0] * sx, pos_px[1] * sy], dtype=float))
            p0_nm, p1_nm = xy
            under, = self.line_profile_ax_image.plot([p0_nm[0], p1_nm[0]], [p0_nm[1], p1_nm[1]], '-', lw=7.0, color='black', alpha=0.8, zorder=24)
            measure, = self.line_profile_ax_image.plot([p0_nm[0], p1_nm[0]], [p0_nm[1], p1_nm[1]], '-', lw=4.0, color='yellow', zorder=25)
            self._line_profile_measure_image_artists.extend([under, measure])
            mid = (p0_nm + p1_nm) / 2.0
            label = self.line_profile_ax_image.text(float(mid[0]), float(mid[1]), f'ΔL={abs(float(pts[1][0]) - float(pts[0][0])):.2f} nm', color='yellow', fontsize=9, weight='bold', ha='center', va='bottom', zorder=26, bbox=dict(boxstyle='round,pad=0.15', facecolor='black', edgecolor='yellow', alpha=0.78))
            self._line_profile_measure_image_artists.append(label)
        except Exception:
            pass
    self.line_profile_canvas_image.draw_idle()
CDIWorkflowApp._line_profile_redraw_image_lines = _line_profile_redraw_image_lines_px_to_nm

def _line_profile_update_measurement_table_px_to_nm(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    for item in tree.get_children():
        tree.delete(item)
    result_by_set = {int(r.get('set', -1)): r for r in getattr(self, '_line_profile_results', [])}
    measurements = getattr(self, '_line_profile_measure_measurements', {})
    for setno, r in sorted(result_by_set.items()):
        length = float(r.get('length', 0.0))
        pts = measurements.get(setno, [])
        if len(pts) == 2:
            m1x, m1y = (float(pts[0][0]), float(pts[0][1]))
            m2x, m2y = (float(pts[1][0]), float(pts[1][1]))
            delta = abs(m2x - m1x)
            vals = (setno, f'{length:.2f}', f'{m1x:.2f}', f'{m2x:.2f}', f'{delta:.2f}', f'{m1y:.4f}', f'{m2y:.4f}', 'Measured')
        else:
            vals = (setno, f'{length:.2f}', '—', '—', '—', '—', '—', 'Pending')
        tree.insert('', 'end', values=vals)
CDIWorkflowApp._line_profile_update_measurement_table = _line_profile_update_measurement_table_px_to_nm
CDIWorkflowApp._line_profile_update_table = _line_profile_update_measurement_table_px_to_nm
CDIWorkflowApp._line_profile_measurement_from_x = _line_profile_measurement_from_x

def _line_profile_export_profiles_px_to_nm(self):
    results = getattr(self, '_line_profile_results', [])
    if not results:
        messagebox.showwarning('Line Profile', 'Draw at least one line first.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Export line-profile graph data', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    try:
        idx = int(round(float(self.roi2_idx_var.get())))
    except Exception:
        idx = 0
    rows = []
    for r in results:
        for x_nm, y in zip(np.asarray(r['dist'], dtype=float), np.asarray(r['profile'], dtype=float)):
            rows.append({'set': int(r['set']), 'frame': idx + 1, 'field_mT': float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else '', 'distance_nm': float(x_nm), 'intensity': float(y)})
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} profile points (nm): {fp}')

def _line_profile_export_results_px_to_nm(self):
    results = getattr(self, '_line_profile_results', [])
    if not results:
        messagebox.showwarning('Line Profile', 'Draw at least one line first.', parent=self.root)
        return
    fp = filedialog.asksaveasfilename(initialdir=self._export_initialdir(), parent=self.root, title='Export dynamic measurements (nm)', defaultextension='.csv', filetypes=[('CSV', '*.csv'), ('All files', '*.*')])
    if not fp:
        return
    try:
        idx = int(round(float(self.line_profile_frame_var.get())))
    except Exception:
        idx = 0
    measurements = getattr(self, '_line_profile_measure_measurements', {})
    rows = []
    for r in results:
        setno = int(r.get('set', -1))
        pts = measurements.get(setno, [])
        row = {'set': setno, 'frame': idx + 1, 'field_mT': float(self.real_fields_mT[idx]) if self.real_fields_mT is not None else '', 'line_length_nm': float(r.get('length', 0.0)), 'M1_position_nm': '', 'M2_position_nm': '', 'DeltaL_nm': '', 'M1_intensity': '', 'M2_intensity': '', 'status': 'Pending'}
        if len(pts) == 2:
            row.update({'M1_position_nm': float(pts[0][0]), 'M2_position_nm': float(pts[1][0]), 'DeltaL_nm': abs(float(pts[1][0]) - float(pts[0][0])), 'M1_intensity': float(pts[0][1]), 'M2_intensity': float(pts[1][1]), 'status': 'Measured'})
        rows.append(row)
    pd.DataFrame(rows).to_csv(fp, index=False)
    self.line_profile_status_var.set(f'Exported {len(rows)} line-set measurements (nm): {fp}')
CDIWorkflowApp._line_profile_export_profiles = _line_profile_export_profiles_px_to_nm
CDIWorkflowApp._line_profile_export_results = _line_profile_export_results_px_to_nm
_LP_FINAL_SYNC_BASE = CDIWorkflowApp._line_profile_sync_from_secondary_roi

def _line_profile_sync_nm_final(self):
    result = _LP_FINAL_SYNC_BASE(self)
    try:
        if getattr(self, 'line_profile_image', None) is not None:
            idx = int(round(float(self.line_profile_frame_var.get())))
            self._line_profile_update_image(idx=idx, preserve_lines=True)
    except Exception:
        pass
    return result
CDIWorkflowApp._line_profile_sync_from_secondary_roi = _line_profile_sync_nm_final
CDIWorkflowApp._line_profile_frame_slider_changed = _line_profile_frame_slider_changed_independent
CDIWorkflowApp._line_profile_frame_mousewheel = _line_profile_frame_mousewheel_independent

def _tab7_mouse_zoom_roi2(self, event):
    """Cursor-centred zoom on the Tab-7 ROI2 image."""
    try:
        ax = self.line_profile_ax_image
    except Exception:
        return
    if event is None or getattr(event, 'inaxes', None) is not ax:
        return
    if getattr(event, 'xdata', None) is None or getattr(event, 'ydata', None) is None:
        return
    delta = getattr(event, 'step', None)
    if delta is None:
        raw = getattr(event, 'delta', 0)
        if raw == 0:
            return
        delta = 1 if raw > 0 else -1
    if delta == 0:
        return
    factor = 1.2 ** (-float(delta))
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    cx = float(event.xdata)
    cy = float(event.ydata)
    if not np.all(np.isfinite(xlim + ylim)):
        return
    xmin, xmax = map(float, xlim)
    ymin, ymax = map(float, ylim)
    nxmin = cx - (cx - xmin) * factor
    nxmax = cx + (xmax - cx) * factor
    nymin = cy - (cy - ymin) * factor
    nymax = cy + (ymax - cy) * factor
    try:
        stack = getattr(self, 'line_source_stack', None)
        if stack is None:
            stack = getattr(self, 'roi2_stack', None)
        arr = np.asarray(stack, dtype=float)
        h, w = arr.shape[1:3]
        sx, sy = _tab6_effective_nm_per_output_px(self)
        if sx is None:
            sx, sy = (1.0, 1.0)
        sx, sy = (float(sx), float(sy))
        full_x = (0.0, float(w) * sx)
        full_y = (0.0, float(h) * sy)
    except Exception:
        full_x = None
        full_y = None
    if full_x is not None and full_y is not None:
        full_xmin, full_xmax = full_x
        full_ymin, full_ymax = full_y
        full_w = full_xmax - full_xmin
        full_h = full_ymax - full_ymin
        new_w = min(abs(nxmax - nxmin), full_w)
        new_h = min(abs(nymax - nymin), full_h)
        if new_w <= 0 or new_h <= 0:
            return
        rx = 0.0 if full_w <= 0 else (cx - full_xmin) / full_w
        ry = 0.0 if full_h <= 0 else (cy - full_ymin) / full_h
        rx = min(1.0, max(0.0, rx))
        ry = min(1.0, max(0.0, ry))
        nxmin = cx - rx * new_w
        nxmax = nxmin + new_w
        nymin = cy - ry * new_h
        nymax = nymin + new_h
        if nxmin < full_xmin:
            nxmin, nxmax = (full_xmin, full_xmin + new_w)
        if nxmax > full_xmax:
            nxmax, nxmin = (full_xmax, full_xmax - new_w)
        if nymin < full_ymin:
            nymin, nymax = (full_ymin, full_ymin + new_h)
        if nymax > full_ymax:
            nymax, nymin = (full_ymax, full_ymax - new_h)
    ax.set_xlim(nxmin, nxmax)
    ax.set_ylim(nymin, nymax)
    ax.set_aspect('equal', adjustable='box')
    try:
        self._tab7_mouse_zoom_view = (tuple(map(float, ax.get_xlim())), tuple(map(float, ax.get_ylim())))
        self._tab7_roi_zoomed = True
        self._tab7_view_nm = self._tab7_mouse_zoom_view
    except Exception:
        pass
    try:
        self.line_profile_canvas_image.draw_idle()
    except Exception:
        pass

def _tab7_mouse_zoom_reset(self):
    """Restore full Tab-7 ROI2 physical-nm extent."""
    try:
        stack = getattr(self, 'line_source_stack', None)
        if stack is None:
            stack = getattr(self, 'roi2_stack', None)
        arr = np.asarray(stack, dtype=float)
        h, w = arr.shape[1:3]
        sx, sy = _tab6_effective_nm_per_output_px(self)
        if sx is None:
            sx, sy = (1.0, 1.0)
        sx, sy = (float(sx), float(sy))
        xlim = (0.0, float(w) * sx)
        ylim = (0.0, float(h) * sy)
        self.line_profile_ax_image.set_xlim(*xlim)
        self.line_profile_ax_image.set_ylim(*ylim)
        self.line_profile_ax_image.set_aspect('equal', adjustable='box')
        self._tab7_mouse_zoom_view = (xlim, ylim)
        self._tab7_view_nm = self._tab7_mouse_zoom_view
        self._tab7_roi_zoomed = False
        self.line_profile_canvas_image.draw_idle()
    except Exception:
        pass
_TAB7_ZOOM_BUILD_OLD = CDIWorkflowApp._build_ui

def _tab7_zoom_build_ui(self, *args, **kwargs):
    result = _TAB7_ZOOM_BUILD_OLD(self, *args, **kwargs)
    try:
        canvas = self.line_profile_canvas_image
        self._tab7_mouse_zoom_cid = canvas.mpl_connect('scroll_event', self._tab7_mouse_zoom_roi2)
    except Exception:
        pass
    return result
CDIWorkflowApp._build_ui = _tab7_zoom_build_ui
CDIWorkflowApp._tab7_mouse_zoom_roi2 = _tab7_mouse_zoom_roi2
CDIWorkflowApp._tab7_mouse_zoom_reset = _tab7_mouse_zoom_reset
try:
    _TAB7_OLD_RESET_ZOOM = CDIWorkflowApp._t7_reset_zoom

    def _tab7_reset_zoom_with_mouse_state(self, *args, **kwargs):
        result = _TAB7_OLD_RESET_ZOOM(self, *args, **kwargs)
        try:
            self._tab7_mouse_zoom_reset()
        except Exception:
            pass
        return result
    CDIWorkflowApp._t7_reset_zoom = _tab7_reset_zoom_with_mouse_state
except Exception:
    pass

def _tab7_zoom_full_extent_nm(self):
    stack = getattr(self, '_tab7r_stack', None)
    if stack is None:
        stack = getattr(self, '_tab7_roi2_copy_stack', None)
    if stack is None:
        stack = getattr(self, 'line_source_stack', None)
    if stack is None:
        stack = getattr(self, 'roi2_stack', None)
    if stack is None:
        return None
    arr = np.asarray(stack)
    if arr.ndim != 3 or arr.shape[0] == 0:
        return None
    h, w = (int(arr.shape[1]), int(arr.shape[2]))
    sx = sy = 1.0
    for name in ('_tab7r_nm_per_px', '_tab7_roi2_copy_nm_per_output_px', '_tab7_fixed_nm_per_output_px'):
        sc = getattr(self, name, None)
        if sc is not None:
            try:
                sx, sy = (float(sc[0]), float(sc[1]))
                if sx > 0 and sy > 0 and np.isfinite(sx) and np.isfinite(sy):
                    break
            except Exception:
                pass
    else:
        try:
            sc = _tab6_effective_nm_per_output_px(self)
            if sc is not None:
                sx, sy = (float(sc[0]), float(sc[1]))
        except Exception:
            pass
    return (0.0, float(w) * sx, 0.0, float(h) * sy)

def _tab7_zoom_apply(self, factor, x=None, y=None):
    ax = getattr(self, 'line_profile_ax_image', None)
    canvas = getattr(self, 'line_profile_canvas_image', None)
    if ax is None or canvas is None:
        return 'break'
    full = _tab7_zoom_full_extent_nm(self)
    if full is None:
        return 'break'
    fx0, fx1, fy0, fy1 = full
    try:
        x0, x1 = map(float, ax.get_xlim())
        y0, y1 = map(float, ax.get_ylim())
    except Exception:
        x0, x1, y0, y1 = (fx0, fx1, fy0, fy1)
    if not all(np.isfinite([x0, x1, y0, y1])) or x1 <= x0 or y1 <= y0:
        x0, x1, y0, y1 = (fx0, fx1, fy0, fy1)
    if x is None or y is None:
        x = 0.5 * (x0 + x1)
        y = 0.5 * (y0 + y1)
    factor = float(factor)
    if not np.isfinite(factor) or factor <= 0:
        return 'break'
    old_w = max(1e-15, x1 - x0)
    old_h = max(1e-15, y1 - y0)
    new_w = min(old_w * factor, fx1 - fx0)
    new_h = min(old_h * factor, fy1 - fy0)
    try:
        stack = getattr(self, '_tab7r_stack', None)
        if stack is None:
            stack = getattr(self, 'line_source_stack', None)
        arr = np.asarray(stack)
        hpx, wpx = (arr.shape[1], arr.shape[2])
        sx = (fx1 - fx0) / max(1, wpx)
        sy = (fy1 - fy0) / max(1, hpx)
        new_w = max(new_w, sx)
        new_h = max(new_h, sy)
    except Exception:
        pass
    rx = min(1.0, max(0.0, (x - x0) / old_w))
    ry = min(1.0, max(0.0, (y - y0) / old_h))
    nx0 = x - rx * new_w
    nx1 = nx0 + new_w
    ny0 = y - ry * new_h
    ny1 = ny0 + new_h
    if nx0 < fx0:
        nx1 += fx0 - nx0
        nx0 = fx0
    if nx1 > fx1:
        nx0 -= nx1 - fx1
        nx1 = fx1
    if ny0 < fy0:
        ny1 += fy0 - ny0
        ny0 = fy0
    if ny1 > fy1:
        ny0 -= ny1 - fy1
        ny1 = fy1
    ax.set_xlim(nx0, nx1)
    ax.set_ylim(ny0, ny1)
    ax.set_aspect('equal', adjustable='box')
    self._tab7_mouse_zoom_view = ((float(nx0), float(nx1)), (float(ny0), float(ny1)))
    self._tab7_mouse_zoomed = True
    try:
        canvas.draw_idle()
    except Exception:
        pass
    return 'break'

def _tab7_zoom_from_tk(self, event):
    """Direct Tk mouse-wheel handler; converts cursor from screen to axes nm."""
    canvas_widget = getattr(getattr(self, 'line_profile_canvas_image', None), 'get_tk_widget', lambda: None)()
    ax = getattr(self, 'line_profile_ax_image', None)
    if canvas_widget is None or ax is None:
        return 'break'
    try:
        px = float(event.x)
        py = float(event.y)
        width = max(1.0, float(canvas_widget.winfo_width()))
        height = max(1.0, float(canvas_widget.winfo_height()))
        display_x = px
        display_y = height - py
        x_nm, y_nm = ax.transData.inverted().transform((display_x, display_y))
    except Exception:
        try:
            x_nm = 0.5 * sum(ax.get_xlim())
            y_nm = 0.5 * sum(ax.get_ylim())
        except Exception:
            return 'break'
    raw = getattr(event, 'delta', 0)
    if raw > 0:
        factor = 1.0 / 1.25
    elif raw < 0:
        factor = 1.25
    else:
        factor = 1.0 / 1.25
    return _tab7_zoom_apply(self, factor, x_nm, y_nm)

def _tab7_zoom_linux(self, event):
    try:
        if getattr(event, 'num', None) == 4:
            factor = 1.0 / 1.25
        elif getattr(event, 'num', None) == 5:
            factor = 1.25
        else:
            return 'break'
        widget = getattr(getattr(self, 'line_profile_canvas_image', None), 'get_tk_widget', lambda: None)()
        ax = getattr(self, 'line_profile_ax_image', None)
        x_nm = y_nm = None
        if widget is not None and ax is not None:
            try:
                height = max(1.0, float(widget.winfo_height()))
                dx, dy = ax.transData.inverted().transform((float(event.x), height - float(event.y)))
                x_nm, y_nm = (float(dx), float(dy))
            except Exception:
                pass
        return _tab7_zoom_apply(self, factor, x_nm, y_nm)
    except Exception:
        return 'break'

def _tab7_zoom_reset_robust(self):
    full = _tab7_zoom_full_extent_nm(self)
    if full is None:
        return 'break'
    _, fx1, _, fy1 = full
    ax = self.line_profile_ax_image
    ax.set_xlim(0.0, fx1)
    ax.set_ylim(0.0, fy1)
    ax.set_aspect('equal', adjustable='box')
    self._tab7_mouse_zoomed = False
    self._tab7_mouse_zoom_view = ((0.0, float(fx1)), (0.0, float(fy1)))
    try:
        self.line_profile_canvas_image.draw_idle()
    except Exception:
        pass
    return 'break'

def _tab7_install_tk_zoom(self):
    """Bind directly to the TkAgg canvas; safely replace stale wheel bindings."""
    try:
        widget = self.line_profile_canvas_image.get_tk_widget()
    except Exception:
        return
    try:
        self._tab7_zoom_bind_ids = []
        self._tab7_zoom_bind_ids.append(widget.bind('<MouseWheel>', self._tab7_zoom_from_tk, add='+'))
        self._tab7_zoom_bind_ids.append(widget.bind('<Button-4>', self._tab7_zoom_linux, add='+'))
        self._tab7_zoom_bind_ids.append(widget.bind('<Button-5>', self._tab7_zoom_linux, add='+'))
    except Exception:
        pass
CDIWorkflowApp._tab7_zoom_full_extent_nm = _tab7_zoom_full_extent_nm
CDIWorkflowApp._tab7_zoom_apply = _tab7_zoom_apply
CDIWorkflowApp._tab7_zoom_from_tk = _tab7_zoom_from_tk
CDIWorkflowApp._tab7_zoom_linux = _tab7_zoom_linux
CDIWorkflowApp._tab7_zoom_reset_robust = _tab7_zoom_reset_robust
CDIWorkflowApp._tab7_install_tk_zoom = _tab7_install_tk_zoom
_TAB7_ZOOM_UI_BUILD_OLD = CDIWorkflowApp._build_ui

def _tab7_zoom_ui_build_final(self, *args, **kwargs):
    result = _TAB7_ZOOM_UI_BUILD_OLD(self, *args, **kwargs)

    def install():
        try:
            self._tab7_install_tk_zoom()
        except Exception:
            pass
    try:
        self.after(250, install)
        self.after(1000, install)
    except Exception:
        install()
    return result
CDIWorkflowApp._build_ui = _tab7_zoom_ui_build_final
try:
    _TAB7_RESET_OLD = CDIWorkflowApp._t7_reset_zoom

    def _tab7_reset_zoom_final(self, *args, **kwargs):
        try:
            return self._tab7_zoom_reset_robust()
        finally:
            try:
                _TAB7_RESET_OLD(self, *args, **kwargs)
            except Exception:
                pass
    CDIWorkflowApp._t7_reset_zoom = _tab7_reset_zoom_final
except Exception:
    pass
_OLD_BUILD_UI_BEFORE_NEW_LPA = CDIWorkflowApp._build_ui

def _lpa_scale_nm(self):
    candidates = []
    for name in ('_tab6_effective_nm_per_output_px', '_tab6_get_nm_per_output_px', '_tab7_private_scale'):
        try:
            fn = getattr(self, name, None)
            if callable(fn):
                sc = fn()
                if sc is not None and len(sc) >= 2:
                    sx, sy = (float(sc[0]), float(sc[1]))
                    if np.isfinite(sx) and np.isfinite(sy) and (sx > 0) and (sy > 0):
                        return (sx, sy)
        except Exception:
            pass
    for attr in ('_tab7_fixed_nm_per_output_px', '_tab6_nm_per_output_px'):
        try:
            sc = getattr(self, attr, None)
            if sc is not None and len(sc) >= 2:
                sx, sy = (float(sc[0]), float(sc[1]))
                if np.isfinite(sx) and np.isfinite(sy) and (sx > 0) and (sy > 0):
                    return (sx, sy)
        except Exception:
            pass
    try:
        v = float(getattr(self, 'pixel_cal_active_nm_per_pixel', np.nan))
        if np.isfinite(v) and v > 0:
            return (v, v)
    except Exception:
        pass
    return (1.0, 1.0)

def _lpa_full_extent(self):
    stack = getattr(self, '_lpa_stack', None)
    if stack is None:
        return None
    a = np.asarray(stack)
    if a.ndim != 3 or a.shape[0] == 0:
        return None
    h, w = a.shape[1:3]
    sx, sy = _lpa_scale_nm(self)
    return (0.0, float(w) * sx, 0.0, float(h) * sy)

def _lpa_clamp_view(full, x0, x1, y0, y1):
    fx0, fx1, fy0, fy1 = map(float, full)
    wx = min(abs(float(x1) - float(x0)), fx1 - fx0)
    wy = min(abs(float(y1) - float(y0)), fy1 - fy0)
    if wx <= 0 or wy <= 0:
        return ((fx0, fx1), (fy0, fy1))
    cx = 0.5 * (float(x0) + float(x1))
    cy = 0.5 * (float(y0) + float(y1))
    nx0 = min(max(cx - 0.5 * wx, fx0), fx1 - wx)
    ny0 = min(max(cy - 0.5 * wy, fy0), fy1 - wy)
    return ((nx0, nx0 + wx), (ny0, ny0 + wy))

def _lpa_apply_view(self, xlim=None, ylim=None, redraw=True):
    ax = getattr(self, '_lpa_ax', None)
    canvas = getattr(self, '_lpa_canvas', None)
    full = _lpa_full_extent(self)
    if ax is None or full is None:
        return
    if xlim is None or ylim is None:
        xlim = (full[0], full[1])
        ylim = (full[2], full[3])
        self._lpa_zoomed = False
        self._lpa_zoom_limits = None
    else:
        xlim, ylim = _lpa_clamp_view(full, xlim[0], xlim[1], ylim[0], ylim[1])
        self._lpa_zoomed = xlim[1] - xlim[0] < 0.999999 * (full[1] - full[0]) or ylim[1] - ylim[0] < 0.999999 * (full[3] - full[2])
        self._lpa_zoom_limits = (tuple(xlim), tuple(ylim))
    ax.set_autoscale_on(False)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    if redraw and canvas is not None:
        canvas.draw_idle()

def _lpa_zoom_scroll(self, event):
    ax = getattr(self, '_lpa_ax', None)
    if event is None or getattr(event, 'inaxes', None) is not ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    full = _lpa_full_extent(self)
    if full is None:
        return
    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        btn = getattr(event, 'button', None)
        if btn in ('up', 4):
            step = 1.0
        elif btn in ('down', 5):
            step = -1.0
    if step == 0:
        return
    x0, x1 = map(float, ax.get_xlim())
    y0, y1 = map(float, ax.get_ylim())
    xlo, xhi = sorted((x0, x1))
    ylo, yhi = sorted((y0, y1))
    wx = xhi - xlo
    wy = yhi - ylo
    if wx <= 0 or wy <= 0:
        return
    factor = 1.2 ** (-step)
    cx, cy = (float(event.xdata), float(event.ydata))
    rx = np.clip((cx - xlo) / wx, 0.0, 1.0)
    ry = np.clip((cy - ylo) / wy, 0.0, 1.0)
    nwx, nwy = (wx * factor, wy * factor)
    if nwx >= 0.999999 * (full[1] - full[0]) and nwy >= 0.999999 * (full[3] - full[2]):
        _lpa_apply_view(self)
        return 'break'
    nx0, nx1 = (cx - rx * nwx, cx + (1 - rx) * nwx)
    ny0, ny1 = (cy - ry * nwy, cy + (1 - ry) * nwy)
    (nx0, nx1), (ny0, ny1) = _lpa_clamp_view(full, nx0, nx1, ny0, ny1)
    _lpa_apply_view(self, (nx0, nx1), (ny0, ny1))
    return 'break'

def _lpa_reset_zoom(self):
    _lpa_apply_view(self)

def _lpa_tk_wheel(self, event):
    canvas = getattr(self, '_lpa_canvas', None)
    ax = getattr(self, '_lpa_ax', None)
    if canvas is None or ax is None:
        return
    try:
        rootx = canvas.get_tk_widget().winfo_rootx()
        rooty = canvas.get_tk_widget().winfo_rooty()
        x = float(event.x_root - rootx)
        y_top = float(event.y_root - rooty)
        h = float(canvas.get_tk_widget().winfo_height())
        y = h - y_top
        disp = np.array([[x, y]], dtype=float)
        data = ax.transData.inverted().transform(disp)[0]
        if getattr(event, 'num', None) == 4:
            step = 1.0
        elif getattr(event, 'num', None) == 5:
            step = -1.0
        else:
            step = 1.0 if getattr(event, 'delta', 0) > 0 else -1.0

        class E:
            pass
        e = E()
        e.inaxes = ax
        e.xdata = float(data[0])
        e.ydata = float(data[1])
        e.step = step
        e.button = 'up' if step > 0 else 'down'
        _lpa_zoom_scroll(self, e)
    except Exception:
        pass
    return 'break'

def _lpa_fetch_roi2(self, select_tab=False):
    stack = getattr(self, 'roi2_stack', None)
    confirmed = getattr(self, 'roi2_confirmed_coords', None)
    if stack is None or np.asarray(stack).ndim != 3 or len(stack) == 0 or not isinstance(confirmed, dict):
        self._lpa_stack = None
        try: self._lpa_status_var.set('No confirmed/processed ROI2 available. Confirm Secondary ROI in Tab 6.')
        except Exception: pass
        return False
    arr = np.asarray(stack, dtype=float)
    self._lpa_stack = arr.copy()
    self._lpa_roi2_coords = dict(confirmed)
    n = arr.shape[0]
    # Start from the SAME frame currently shown/selected in Tab 6.
    cur = 0
    for name in ('roi2_idx_var', 'proc_idx_var'):
        try:
            cur = int(getattr(self, name).get())
            break
        except Exception: pass
    cur = max(0, min(cur, n - 1))
    self._lpa_frame_var.set(cur)
    self._lpa_slider.configure(from_=0, to=max(0, n - 1), state='normal')
    self._lpa_render_frame(cur, preserve_zoom=True)
    try: field = float(self.real_fields_mT[cur])
    except Exception: field = float(cur)
    self._lpa_status_var.set(f'Confirmed ROI2 from Tab 6 | Frame {cur + 1}/{n} | Field = {field:+.2f} mT')
    if select_tab:
        try: self.nb.select(self.tab_line_profile_analysis)
        except Exception: pass
    return True

def _lpa_render_frame(self, idx=0, preserve_zoom=True):
    stack = getattr(self, '_lpa_stack', None)
    if stack is None: return
    n = len(stack); idx = max(0, min(int(idx), n - 1))
    old_zoom = getattr(self, '_lpa_zoom_limits', None) if preserve_zoom and getattr(self, '_lpa_zoomed', False) else None
    self._lpa_frame_var.set(idx)
    img = np.asarray(stack[idx], dtype=float); im = self._lpa_image
    im.set_data(img)
    sx, sy = _lpa_scale_nm(self); h, w = img.shape[:2]
    im.set_extent((0.0, w * sx, 0.0, h * sy))
    cmap = 'RdBu_r'
    try:
        v = str(self.colormap_var.get()).strip()
        if v: cmap = v
    except Exception: pass
    try: im.set_cmap(cmap)
    except Exception: pass
    im.set_clim(0.0, 1.0)
    try: field = float(self.real_fields_mT[idx])
    except Exception: field = None
    title = f'Confirmed ROI — Frame {idx + 1}/{n}'
    if field is not None: title += f' | Field = {field:+.2f} mT'
    self._lpa_ax.set_title(title); self._lpa_ax.set_xlabel('X (nm)'); self._lpa_ax.set_ylabel('Y (nm)')
    self._lpa_ax.set_aspect('equal', adjustable='box'); self._lpa_ax.set_anchor('C')
    try: self._lpa_frame_label.config(text=f'Frame No. {idx + 1}/{n} | Equivalent Field: {field:+.2f} mT')
    except Exception: pass
    try: self._lpa_colorbar.update_normal(im)
    except Exception: pass
    if old_zoom is not None: _lpa_apply_view(self, old_zoom[0], old_zoom[1], redraw=False)
    else: _lpa_apply_view(self, redraw=False)
    _lpa_redraw_lines_and_plot(self)
    self._lpa_canvas.draw_idle()

def _lpa_slider_changed(self, value=None):
    if getattr(self, '_lpa_stack', None) is None and not _lpa_fetch_roi2(self): return
    try: idx = int(round(float(value))) if value is not None else int(self._lpa_frame_var.get())
    except Exception: idx = 0
    _lpa_render_frame(self, idx, preserve_zoom=True)

def _lpa_tab_selected(self, event=None):
    try:
        if self.nb.select() == str(self.tab_line_profile_analysis): _lpa_fetch_roi2(self)
    except Exception: pass

def _lpa_measure_clear_artists(self):
    for attr in ('_lpa_measure_artists', '_lpa_measure_image_artists'):
        for a in getattr(self, attr, []) or []:
            try: a.remove()
            except Exception: pass
        setattr(self, attr, [])


def _lpa_measure_pick_curve(self, event, max_screen=28.0):
    if event is None or event.inaxes is not self._lpa_plot_ax or event.xdata is None or event.ydata is None:
        return None
    target = self._lpa_plot_ax.transData.transform((float(event.xdata), float(event.ydata)))
    best = None
    stack = getattr(self, '_lpa_stack', None)
    if stack is None or len(stack) == 0:
        return None
    idx = max(0, min(int(self._lpa_frame_var.get()), len(stack)-1))
    image = np.asarray(stack[idx], dtype=float)
    for j, (p0, p1) in enumerate(getattr(self, '_lpa_lines', []) or []):
        d, prof = _lpa_profile_for_line(self, image, p0, p1)
        if d.size < 2 or prof.size != d.size: continue
        pts = self._lpa_plot_ax.transData.transform(np.column_stack((d, prof)))
        a, b = pts[:-1], pts[1:]; v = b-a; vv = np.einsum('ij,ij->i', v, v)
        w = target-a; t = np.einsum('ij,ij->i', w, v) / np.where(vv > 0, vv, 1.0); t=np.clip(t,0,1)
        q=a+v*t[:,None]; dd=np.hypot(q[:,0]-target[0], q[:,1]-target[1]); k=int(np.argmin(dd))
        if best is None or float(dd[k]) < best[0]:
            xnm=float(d[k]+t[k]*(d[k+1]-d[k])); yv=float(prof[k]+t[k]*(prof[k+1]-prof[k]))
            best=(float(dd[k]), j, xnm, yv, d, prof)
    return best if best is not None and best[0] <= max_screen else None


def _lpa_measure_interpolate(self, line_idx, xnm):
    stack=getattr(self,'_lpa_stack',None)
    if stack is None or len(stack)==0 or line_idx < 0 or line_idx >= len(self._lpa_lines): return None
    idx=max(0,min(int(self._lpa_frame_var.get()),len(stack)-1)); image=np.asarray(stack[idx],dtype=float)
    line=self._lpa_lines[line_idx]; d,p=_lpa_profile_for_line(self,image,line[0],line[1])
    if d.size<2 or p.size!=d.size: return None
    x=float(np.clip(float(xnm),float(d[0]),float(d[-1]))); y=float(np.interp(x,d,p)); return x,y,d,p


def _lpa_measure_redraw(self):
    _lpa_measure_clear_artists(self)
    # Graph endpoint markers/measurement arrows.
    for mi,m in enumerate(getattr(self,'_lpa_measurements',[]) or [],1):
        got=_lpa_measure_interpolate(self,int(m['line']),float(m['m1_nm'])); got2=_lpa_measure_interpolate(self,int(m['line']),float(m['m2_nm']))
        if got is None or got2 is None: continue
        x0,y0,_,_=got; x1,y1,_,_=got2
        a=self._lpa_plot_ax.axvline(x0,color='gold',ls='--',lw=1.5,zorder=8); b=self._lpa_plot_ax.axvline(x1,color='lime',ls='--',lw=1.5,zorder=8)
        p0=self._lpa_plot_ax.plot([x0],[y0],'o',ms=8,mfc='gold',mec='black',mew=1.1,zorder=10)[0]
        p1=self._lpa_plot_ax.plot([x1],[y1],'o',ms=8,mfc='lime',mec='black',mew=1.1,zorder=10)[0]
        self._lpa_measure_artists.extend([a,b,p0,p1])
        ymin,ymax=self._lpa_plot_ax.get_ylim(); ybar=ymax-0.08*max(ymax-ymin,1e-12)
        ar=self._lpa_plot_ax.annotate('',xy=(x1,ybar),xytext=(x0,ybar),arrowprops=dict(arrowstyle='<->',color='blue',lw=2.0),zorder=9)
        tx=self._lpa_plot_ax.text((x0+x1)/2,ybar,f'ΔL = {abs(x1-x0):.6g} nm',ha='center',va='bottom',fontsize=8,fontweight='bold',bbox=dict(boxstyle='round,pad=0.15',facecolor='white',edgecolor='blue',alpha=0.9),zorder=11)
        self._lpa_measure_artists.extend([ar,tx])
        self._lpa_measure_artists.append(self._lpa_plot_ax.text(x0,y0,' M%d '%(2*mi-1),fontsize=8,fontweight='bold',ha='center',va='bottom',bbox=dict(boxstyle='round,pad=0.1',facecolor='white',edgecolor='black',alpha=.9),zorder=11))
        self._lpa_measure_artists.append(self._lpa_plot_ax.text(x1,y1,' M%d '%(2*mi),fontsize=8,fontweight='bold',ha='center',va='bottom',bbox=dict(boxstyle='round,pad=0.1',facecolor='white',edgecolor='black',alpha=.9),zorder=11))
    # Pending M1 marker.
    if getattr(self,'_lpa_measure_first',None) is not None:
        f=self._lpa_measure_first; got=_lpa_measure_interpolate(self,int(f['line']),float(f['x']))
        if got is not None:
            x,y,_,_ = got; self._lpa_measure_artists.append(self._lpa_plot_ax.plot([x],[y],'o',ms=8,mfc='gold',mec='black',mew=1.1,zorder=12)[0]); self._lpa_measure_artists.append(self._lpa_plot_ax.text(x,y,' M1 ',fontsize=8,fontweight='bold',ha='center',va='bottom',bbox=dict(boxstyle='round,pad=0.1',facecolor='white',edgecolor='black',alpha=.9),zorder=12))
    # Highlight measured segment and endpoints on ROI2 image.
    for mi,m in enumerate(getattr(self,'_lpa_measurements',[]) or [],1):
        line=self._lpa_lines[int(m['line'])]; total=float(np.hypot(line[1][0]-line[0][0],line[1][1]-line[0][1]));
        if total<=0: continue
        t0=float(m['m1_nm'])/total; t1=float(m['m2_nm'])/total
        q0=(line[0][0]+t0*(line[1][0]-line[0][0]), line[0][1]+t0*(line[1][1]-line[0][1])); q1=(line[0][0]+t1*(line[1][0]-line[0][0]), line[0][1]+t1*(line[1][1]-line[0][1]))
        sh=self._lpa_ax.plot([q0[0],q1[0]],[q0[1],q1[1]],'-',lw=6,color='black',alpha=.7,zorder=40)[0]; yy=self._lpa_ax.plot([q0[0],q1[0]],[q0[1],q1[1]],'-',lw=3.5,color='yellow',zorder=41)[0]; self._lpa_measure_image_artists.extend([sh,yy])
        for q in (q0,q1): self._lpa_measure_image_artists.append(self._lpa_ax.plot([q[0]],[q[1]],'o',ms=7,mfc='yellow',mec='black',mew=1,zorder=42)[0])
    try: self._lpa_plot_canvas.draw_idle(); self._lpa_canvas.draw_idle()
    except Exception: pass


def _lpa_measure_update_table(self):
    tree=getattr(self,'_lpa_measure_tree',None)
    if tree is None: return
    for iid in tree.get_children(): tree.delete(iid)
    for i,m in enumerate(getattr(self,'_lpa_measurements',[]) or [],1):
        tree.insert('', 'end', values=(f'M{2*i-1}–M{2*i}', int(m['line'])+1, f"{float(m['m1_nm']):.6g}", f"{float(m['m2_nm']):.6g}", f"{float(abs(m['m2_nm']-m['m1_nm'])):.6g}", f"{float(m['m1_intensity']):.6g}", f"{float(m['m2_intensity']):.6g}"))


def _lpa_measure_start(self):
    if not getattr(self,'_lpa_lines',[]):
        try:self._lpa_status_var.set('Draw at least one line before measuring.')
        except Exception:pass
        return
    self._lpa_measure_mode=True; self._lpa_measure_first=None; self._lpa_measure_drag=None
    try:self._lpa_status_var.set('MEASURE: click M1 then M2 on the SAME line profile. Drag either endpoint with left mouse button.')
    except Exception:pass
    _lpa_measure_redraw(self)


def _lpa_measure_graph_press(self,event):
    if event is None or event.inaxes is not self._lpa_plot_ax or event.button != 1: return
    # Existing endpoint gets drag priority.
    best=None
    for mi,m in enumerate(getattr(self,'_lpa_measurements',[]) or []):
        for k,xnm in enumerate((float(m['m1_nm']),float(m['m2_nm']))):
            got=_lpa_measure_interpolate(self,int(m['line']),xnm)
            if got is None: continue
            y=got[1]; sxp,syp=self._lpa_plot_ax.transData.transform((xnm,y)); dd=float(np.hypot(event.x-sxp,event.y-syp))
            if best is None or dd<best[0]: best=(dd,mi,k)
    if best is not None and best[0] <= 18.0:
        self._lpa_measure_drag=(best[1],best[2]); self._lpa_measure_mode=False
        try:self._lpa_status_var.set(f'Dragging M{2*best[1]+best[2] + 1}. Release to set the new endpoint.')
        except Exception:pass
        return
    if not getattr(self,'_lpa_measure_mode',False): return
    hit=_lpa_measure_pick_curve(self,event)
    if hit is None:
        try:self._lpa_status_var.set('Click directly on a line-profile curve.')
        except Exception:pass
        return
    _,li,xnm,yv,_,_=hit
    if self._lpa_measure_first is None:
        self._lpa_measure_first={'line':int(li),'x':float(xnm)}
        try:self._lpa_status_var.set(f'M1 selected on Line {li+1} at {xnm:.6g} nm. Click M2 on the same line.')
        except Exception:pass
    else:
        f=self._lpa_measure_first
        if int(f['line']) != int(li):
            try:self._lpa_status_var.set(f'M2 must be on the same Line {int(f["line"])+1} as M1.')
            except Exception:pass
            return
        x0=float(f['x']); x1=float(xnm); i0=float(np.interp(x0,hit[4],hit[5])); i1=float(np.interp(x1,hit[4],hit[5]))
        self._lpa_measurements.append({'line':int(li),'m1_nm':x0,'m2_nm':x1,'delta_nm':abs(x1-x0),'m1_intensity':i0,'m2_intensity':i1})
        self._lpa_measure_first=None; self._lpa_measure_mode=False
        _lpa_measure_update_table(self); _lpa_redraw_lines_and_plot(self); _lpa_measure_redraw(self)
        try:self._lpa_status_var.set(f'Line {li+1}: M1={x0:.6g} nm, M2={x1:.6g} nm, measured distance ΔL={abs(x1-x0):.6g} nm')
        except Exception:pass


def _lpa_measure_graph_motion(self,event):
    drag=getattr(self,'_lpa_measure_drag',None)
    if drag is None or event is None or event.inaxes is not self._lpa_plot_ax or event.xdata is None: return
    mi,k=drag
    if mi<0 or mi>=len(getattr(self,'_lpa_measurements',[]) or []): return
    m=self._lpa_measurements[mi]; got=_lpa_measure_interpolate(self,int(m['line']),float(event.xdata))
    if got is None: return
    x,y,d,p=got
    if k==0:m['m1_nm']=x; m['m1_intensity']=y
    else:m['m2_nm']=x; m['m2_intensity']=y
    m['delta_nm']=abs(float(m['m2_nm'])-float(m['m1_nm']))
    _lpa_measure_update_table(self); _lpa_redraw_lines_and_plot(self); _lpa_measure_redraw(self)
    try:self._lpa_status_var.set(f'Line {int(m["line"])+1}: ΔL={m["delta_nm"]:.6g} nm. Drag M1/M2 with left mouse button.')
    except Exception:pass


def _lpa_measure_graph_release(self,event=None):
    if getattr(self,'_lpa_measure_drag',None) is not None:
        try:self._lpa_status_var.set('Measurement endpoint fixed. Press MEASURE for another pair or drag an endpoint again.')
        except Exception:pass
    self._lpa_measure_drag=None

def _lpa_clear_lines(self):
    self._lpa_lines = []
    self._lpa_measurements = []
    self._lpa_measure_mode = False
    self._lpa_measure_first = None
    self._lpa_measure_drag = None
    _lpa_measure_clear_artists(self)
    _lpa_redraw_lines_and_plot(self)
    _lpa_measure_update_table(self)
    try: self._lpa_status_var.set('ROI2 loaded. All drawn lines, profile plots and measurements cleared.')
    except Exception: pass

def _lpa_start_draw(self):
    if getattr(self, '_lpa_stack', None) is None:
        if not _lpa_fetch_roi2(self): return
    try: n = max(1, int(self._lpa_nlines_var.get()))
    except Exception: n = 1
    self._lpa_draw_target = n
    self._lpa_draw_mode = True
    self._lpa_drag_start = None
    try: self._lpa_status_var.set(f'Draw line 1/{n} on ROI2 (click-drag-release).')
    except Exception: pass

def _lpa_line_press(self, event):
    if not getattr(self, '_lpa_draw_mode', False) or event is None or event.inaxes is not self._lpa_ax or event.button != 1 or event.xdata is None or event.ydata is None: return
    self._lpa_drag_start = (float(event.xdata), float(event.ydata))
    self._lpa_temp_line = self._lpa_ax.plot([event.xdata, event.xdata], [event.ydata, event.ydata], '--', lw=2, alpha=0.8)[0]

def _lpa_line_motion(self, event):
    p0 = getattr(self, '_lpa_drag_start', None); t = getattr(self, '_lpa_temp_line', None)
    if p0 is None or t is None or event is None or event.inaxes is not self._lpa_ax or event.xdata is None or event.ydata is None: return
    t.set_data([p0[0], float(event.xdata)], [p0[1], float(event.ydata)]); self._lpa_canvas.draw_idle()

def _lpa_line_release(self, event):
    p0 = getattr(self, '_lpa_drag_start', None)
    t = getattr(self, '_lpa_temp_line', None)
    self._lpa_drag_start = None; self._lpa_temp_line = None
    if t is not None:
        try: t.remove()
        except Exception: pass
    if p0 is None or event is None or event.inaxes is not self._lpa_ax or event.xdata is None or event.ydata is None: return
    p1 = (float(event.xdata), float(event.ydata))
    if np.hypot(p1[0] - p0[0], p1[1] - p0[1]) <= 1e-12: return
    self._lpa_lines.append((p0, p1))
    _lpa_redraw_lines_and_plot(self)
    target = getattr(self, '_lpa_draw_target', 1)
    if len(self._lpa_lines) >= target:
        self._lpa_draw_mode = False
        self._lpa_status_var.set(f'{len(self._lpa_lines)} line(s) drawn. Intensity profiles updated.')
    else:
        self._lpa_status_var.set(f'Draw line {len(self._lpa_lines) + 1}/{target} on ROI2.')
    self._lpa_canvas.draw_idle()

def _lpa_profile_for_line(self, image, p0, p1):
    sx, sy = _lpa_scale_nm(self)
    x0, y0 = float(p0[0]) / max(sx, 1e-12), float(p0[1]) / max(sy, 1e-12)
    x1, y1 = float(p1[0]) / max(sx, 1e-12), float(p1[1]) / max(sy, 1e-12)
    px_len = float(np.hypot(x1 - x0, y1 - y0))
    count = max(50, int(np.ceil(px_len * 4)) + 1)
    xx = np.linspace(x0, x1, count); yy = np.linspace(y0, y1, count)
    prof = map_coordinates(np.asarray(image, dtype=float), [yy, xx], order=1, mode='nearest')
    dist = np.linspace(0.0, float(np.hypot((x1 - x0) * sx, (y1 - y0) * sy)), count)
    return dist, prof

def _lpa_redraw_lines_and_plot(self):
    ax = self._lpa_ax
    for a in getattr(self, '_lpa_line_artists', []):
        try: a.remove()
        except Exception: pass
    self._lpa_line_artists = []
    lines = getattr(self, '_lpa_lines', []) or []
    colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', ['C0', 'C1', 'C2', 'C3', 'C4', 'C5'])
    for i, (p0, p1) in enumerate(lines):
        ln, = ax.plot([p0[0], p1[0]], [p0[1], p1[1]], lw=2.5, color=colors[i % len(colors)], zorder=20)
        self._lpa_line_artists.append(ln)
    self._lpa_plot_ax.clear(); self._lpa_plot_ax.set_xlabel('Line Length (nm)'); self._lpa_plot_ax.set_ylabel('Intensity'); self._lpa_plot_ax.grid(True, alpha=0.25)
    stack = getattr(self, '_lpa_stack', None)
    if stack is None or len(stack)==0:
        _lpa_measure_redraw(self); return
    try: idx = int(self._lpa_frame_var.get())
    except Exception: idx = 0
    idx = max(0, min(idx, len(stack)-1)); image = np.asarray(stack[idx], dtype=float)
    max_len=0.0; max_int=0.0
    for i,(p0,p1) in enumerate(lines):
        d,prof=_lpa_profile_for_line(self,image,p0,p1); self._lpa_plot_ax.plot(d,prof,lw=2,label=f'Line {i+1}')
        if d.size: max_len=max(max_len,float(d[-1]))
        if prof.size and np.any(np.isfinite(prof)): max_int=max(max_int,float(np.nanmax(prof)))
    if lines:
        self._lpa_plot_ax.set_xlim(0.0,max(max_len*1.05,1e-9)); self._lpa_plot_ax.set_ylim(0.0,max(max_int*1.10,1e-9)); self._lpa_plot_ax.legend(loc='best')
    else:
        self._lpa_plot_ax.text(0.5,0.5,'Draw a line on ROI2 to plot intensity',ha='center',va='center',transform=self._lpa_plot_ax.transAxes)
    self._lpa_plot_canvas.draw_idle()
    _lpa_measure_redraw(self)
    _lpa_measure_update_table(self)

def _build_line_profile_analysis_tab(self):
    outer = ttk.Frame(self.tab_line_profile_analysis, padding=7); outer.pack(fill='both', expand=True)
    top = ttk.Frame(outer); top.pack(fill='x', pady=(0, 5))
    ttk.Label(top, text='LINE PROFILE ANALYSIS — CONFIRMED ROI2', font=('Arial', 12, 'bold')).pack(side='left')
    self._lpa_status_var = tk.StringVar(value='Waiting for confirmed/processed ROI2 from Tab 6.')
    ttk.Label(top, textvariable=self._lpa_status_var, foreground='darkgreen').pack(side='left', padx=12)

    controls = ttk.Frame(outer); controls.pack(fill='x', pady=(0, 5))
    ttk.Button(controls, text='ROI2 FROM TAB 6', command=lambda: _lpa_fetch_roi2(self, select_tab=False)).pack(side='left', padx=(0, 5))
    ttk.Button(controls, text='RESET ZOOM', command=lambda: _lpa_reset_zoom(self)).pack(side='left', padx=5)
    ttk.Label(controls, text='No. of lines:').pack(side='left', padx=(15, 3))
    self._lpa_nlines_var = tk.IntVar(value=1)
    tk.Spinbox(controls, from_=1, to=50, textvariable=self._lpa_nlines_var, width=5).pack(side='left', padx=3)
    ttk.Button(controls, text='DRAW LINE', command=lambda: _lpa_start_draw(self)).pack(side='left', padx=5)
    ttk.Button(controls, text='CLEAR LINES', command=lambda: _lpa_clear_lines(self)).pack(side='left', padx=5)
    ttk.Button(controls, text='MEASURE', command=lambda: _lpa_measure_start(self)).pack(side='left', padx=5)
    ttk.Label(controls, text='Frame:').pack(side='left', padx=(15, 3))
    self._lpa_frame_var = tk.IntVar(value=0)
    self._lpa_slider = tk.Scale(controls, from_=0, to=0, orient='horizontal', resolution=1, showvalue=True, variable=self._lpa_frame_var, command=lambda v: _lpa_slider_changed(self, v), length=360)
    self._lpa_slider.pack(side='left', padx=3)
    self._lpa_frame_label = ttk.Label(controls, text='Frame No. 1/0 | Equivalent Field: -- mT'); self._lpa_frame_label.pack(side='left', padx=7)

    # Vertical splitter: plots above, measurement table below.
    vertical = ttk.Panedwindow(outer, orient='vertical'); vertical.pack(fill='both', expand=True)
    pane = ttk.Panedwindow(vertical, orient='horizontal')
    table = ttk.Frame(vertical)
    vertical.add(pane, weight=7)
    vertical.add(table, weight=2)

    left = ttk.Frame(pane); right = ttk.Frame(pane); pane.add(left, weight=1); pane.add(right, weight=1)
    self._lpa_fig, self._lpa_ax = plt.subplots(figsize=(7, 6)); self._lpa_ax.set_aspect('equal', adjustable='box')
    self._lpa_image = self._lpa_ax.imshow(np.zeros((2, 2), dtype=float), cmap='RdBu_r', origin='lower', extent=(0, 2, 0, 2), vmin=0, vmax=1, interpolation='nearest')
    self._lpa_colorbar = self._lpa_fig.colorbar(self._lpa_image, ax=self._lpa_ax, pad=0.02); self._lpa_colorbar.set_label('Intensity')
    self._lpa_ax.set_xlabel('X (nm)'); self._lpa_ax.set_ylabel('Y (nm)')
    self._lpa_canvas = FigureCanvasTkAgg(self._lpa_fig, master=left); self._lpa_canvas.draw(); self._lpa_canvas.get_tk_widget().pack(fill='both', expand=True)
    self._lpa_canvas.mpl_connect('scroll_event', lambda e: _lpa_zoom_scroll(self, e))
    tkc = self._lpa_canvas.get_tk_widget(); tkc.bind('<MouseWheel>', lambda e: _lpa_tk_wheel(self, e), add='+'); tkc.bind('<Button-4>', lambda e: _lpa_tk_wheel(self, e), add='+'); tkc.bind('<Button-5>', lambda e: _lpa_tk_wheel(self, e), add='+')

    self._lpa_fig2, self._lpa_plot_ax = plt.subplots(figsize=(7, 6))
    self._lpa_plot_ax.set_xlabel('Line Length (nm)'); self._lpa_plot_ax.set_ylabel('Intensity'); self._lpa_plot_ax.grid(True, alpha=0.25)
    self._lpa_plot_canvas = FigureCanvasTkAgg(self._lpa_fig2, master=right); self._lpa_plot_canvas.draw(); self._lpa_plot_canvas.get_tk_widget().pack(fill='both', expand=True)
    # Existing drawing callbacks on ROI2 image.
    self._lpa_canvas.mpl_connect('button_press_event', lambda e: _lpa_line_press(self, e))
    self._lpa_canvas.mpl_connect('motion_notify_event', lambda e: _lpa_line_motion(self, e))
    self._lpa_canvas.mpl_connect('button_release_event', lambda e: _lpa_line_release(self, e))
    # Measurement callbacks on the profile graph.
    self._lpa_plot_canvas.mpl_connect('button_press_event', lambda e: _lpa_measure_graph_press(self, e))
    self._lpa_plot_canvas.mpl_connect('motion_notify_event', lambda e: _lpa_measure_graph_motion(self, e))
    self._lpa_plot_canvas.mpl_connect('button_release_event', lambda e: _lpa_measure_graph_release(self, e))

    # Measurement table.
    ttk.Label(table, text='Measured distances', font=('Arial', 10, 'bold')).pack(anchor='w', padx=4, pady=(3, 1))
    columns = ('Measurement', 'Line', 'M1 (nm)', 'M2 (nm)', 'Distance (nm)', 'M1 Intensity', 'M2 Intensity')
    self._lpa_measure_tree = ttk.Treeview(table, columns=columns, show='headings', height=6)
    widths = (100, 60, 110, 110, 120, 120, 120)
    for col, width in zip(columns, widths):
        self._lpa_measure_tree.heading(col, text=col)
        self._lpa_measure_tree.column(col, width=width, minwidth=55, anchor='center')
    self._lpa_measure_tree.pack(side='left', fill='both', expand=True, padx=(4, 0), pady=(0, 4))
    tscroll = ttk.Scrollbar(table, orient='vertical', command=self._lpa_measure_tree.yview)
    tscroll.pack(side='right', fill='y', padx=(0, 4), pady=(0, 4)); self._lpa_measure_tree.configure(yscrollcommand=tscroll.set)

    self._lpa_stack = None; self._lpa_roi2_coords = None; self._lpa_zoomed = False; self._lpa_zoom_limits = None
    self._lpa_lines = []; self._lpa_line_artists = []; self._lpa_draw_mode = False; self._lpa_draw_target = 1; self._lpa_drag_start = None; self._lpa_temp_line = None
    self._lpa_measurements = []; self._lpa_measure_mode = False; self._lpa_measure_first = None; self._lpa_measure_drag = None; self._lpa_measure_artists = []; self._lpa_measure_image_artists = []


def _build_ui_with_new_lpa(self):
    _OLD_BUILD_UI_BEFORE_NEW_LPA(self)
    try:
        current_tabs = list(self.nb.tabs())
        for tab_id in current_tabs[6:]:
            try:
                self.nb.forget(tab_id)
            except Exception:
                pass
        for attr in ('tab_line_profile', 'tab_particle_density', 'tab_roi2', 'tab_gauss2d', 'tab_multi_roi', 'tab_multi_gauss', 'tab_frc', 'tab_log'):
            try:
                frame = getattr(self, attr, None)
                if frame is not None:
                    frame.destroy()
            except Exception:
                pass
    except Exception:
        pass
    self.tab_line_profile_analysis = ttk.Frame(self.nb)
    self.nb.add(self.tab_line_profile_analysis, text='7. Line Profile Analysis')
    self.tab_line_profile = self.tab_line_profile_analysis
    _build_line_profile_analysis_tab(self)
    self.nb.bind('<<NotebookTabChanged>>', lambda e: _lpa_tab_selected(self, e), add='+')
    try:
        _lpa_fetch_roi2(self, select_tab=False)
    except Exception:
        pass
CDIWorkflowApp._build_ui = _build_ui_with_new_lpa
CDIWorkflowApp._lpa_fetch_roi2 = _lpa_fetch_roi2
CDIWorkflowApp._lpa_render_frame = _lpa_render_frame
CDIWorkflowApp._lpa_slider_changed = _lpa_slider_changed
CDIWorkflowApp._lpa_reset_zoom = _lpa_reset_zoom
if hasattr(CDIWorkflowApp, 'confirm_secondary_roi_from_processing'):
    _LPA_OLD_CONFIRM_ROI2 = CDIWorkflowApp.confirm_secondary_roi_from_processing

    def _LPA_CONFIRM_ROI2_WRAPPER(self, *args, **kwargs):
        result = _LPA_OLD_CONFIRM_ROI2(self, *args, **kwargs)
        try:
            if result is not False and hasattr(self, 'tab_line_profile_analysis'):
                _lpa_fetch_roi2(self, select_tab=True)
        except Exception:
            pass
        return result
    CDIWorkflowApp.confirm_secondary_roi_from_processing = _LPA_CONFIRM_ROI2_WRAPPER

def _tab6_open_output_folder(folder):
    """Open an existing export directory in the native file manager."""
    try:
        folder = os.path.abspath(os.path.expanduser(str(folder)))
        if not os.path.isdir(folder):
            return False
        if os.name == 'nt':
            os.startfile(folder)
            return True
        import sys, subprocess
        if sys.platform == 'darwin':
            subprocess.Popen(['open', folder])
            return True
        subprocess.Popen(['xdg-open', folder])
        return True
    except Exception:
        return False

def _tab6_export_figure(self, image, idx, dpi=300, title_prefix='ROI'):
    """Create a complete publication-style Tab-6 figure for export.

    The exported figure contains the same processed image, physical nm axes
    when Tab-5 calibration is available, the active Tab-6 colormap, and a
    colorbar tied to the displayed 0..1 intensity scale.
    """
    image = np.asarray(image, dtype=float)
    if image.ndim != 2:
        raise ValueError('Tab-6 export expects a 2-D processed ROI image.')
    h, w = image.shape
    cmap_name = 'RdBu_r'
    try:
        candidate = str(self.colormap_var.get()).strip()
        if candidate:
            cmap_name = candidate
    except Exception:
        pass
    fig = plt.figure(figsize=(8.6, 7.2), dpi=int(dpi), facecolor='white')
    ax = fig.add_axes([0.1, 0.1, 0.76, 0.82])
    cax = fig.add_axes([0.89, 0.1, 0.035, 0.82])
    im = ax.imshow(np.clip(image, 0.0, 1.0), cmap=cmap_name, origin='lower', interpolation='nearest', vmin=0.0, vmax=1.0, aspect='equal')
    physical = False
    try:
        base = self._proc_get_tab5_base_pixel_size()
        if base is not None and np.isfinite(base) and (base > 0):
            ow = int(getattr(self, 'original_roi_w', 0) or w)
            oh = int(getattr(self, 'original_roi_h', 0) or h)
            if ow > 0 and oh > 0:
                pw = float(ow) * float(base)
                ph = float(oh) * float(base)
                im.set_extent((0.0, pw, 0.0, ph))
                ax.set_xlim(0.0, pw)
                ax.set_ylim(0.0, ph)
                ax.set_xlabel('X (nm)')
                ax.set_ylabel('Y (nm)')
                physical = True
    except Exception:
        physical = False
    if not physical:
        im.set_extent((-0.5, float(w) - 0.5, -0.5, float(h) - 0.5))
        ax.set_xlim(-0.5, float(w) - 0.5)
        ax.set_ylim(-0.5, float(h) - 0.5)
        ax.set_xlabel('X (pixel)')
        ax.set_ylabel('Y (pixel)')
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    ax.grid(False)
    field = None
    try:
        if self.real_fields_mT is not None and idx < len(self.real_fields_mT):
            field = float(self.real_fields_mT[idx])
    except Exception:
        field = None
    if field is None:
        ax.set_title(f'{title_prefix} | Frame {idx + 1}', fontsize=12)
    else:
        ax.set_title(f'{title_prefix} | Frame {idx + 1}/{len(self.recons)} | Field = {field:+.2f} mT', fontsize=12)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label('Intensity', rotation=90)
    fig.canvas.draw()
    return fig

def _tab6_save_single_complete(self, extension, fmt):
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Resolution Required', 'Apply SAME CONFIRMED ROI resolution first.', parent=self.root)
        return
    try:
        dpi = int(self.proc_export_dpi_var.get())
    except Exception:
        dpi = 300
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Export DPI', 'Select 150, 300, 600 or 900 DPI.', parent=self.root)
        return
    fig = None
    try:
        folder = os.path.abspath(os.path.expanduser(str(self.output_var.get()).strip() or 'ROI_exports'))
        os.makedirs(folder, exist_ok=True)
        i = max(0, min(int(round(float(self.proc_idx_var.get()))), len(self.recons) - 1))
        field = float(self.real_fields_mT[i])
        path = os.path.join(folder, f'ROI_{i:03d}_{field:+.2f}mT{extension}')
        image = _proc_image(self, i)
        fig = _tab6_export_figure(self, image, i, dpi=dpi, title_prefix='ROI')
        save_kwargs = {'dpi': dpi, 'bbox_inches': 'tight', 'pad_inches': 0.08, 'format': fmt}
        if fmt == 'JPEG':
            save_kwargs['facecolor'] = 'white'
        fig.savefig(path, **save_kwargs)
        self.proc_status_var.set(f'Saved: {path} | {dpi} DPI | axes + colorbar')
        self.log(f'Tab 6 export complete: {path} | format={fmt} | dpi={dpi} | axes+colorbar')
        _tab6_open_output_folder(folder)
    except Exception as exc:
        self.proc_status_var.set('Single-image export failed.')
        self.log(f'Tab 6 single export failed: {exc}')
        messagebox.showerror('Export Error', str(exc), parent=self.root)
    finally:
        if fig is not None:
            try:
                plt.close(fig)
            except Exception:
                pass

def _tab6_batch_complete(self, extension, fmt):
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Resolution Required', 'Apply SAME CONFIRMED ROI resolution first.', parent=self.root)
        return
    try:
        dpi = int(self.proc_export_dpi_var.get())
    except Exception:
        dpi = 300
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Export DPI', 'Select 150, 300, 600 or 900 DPI.', parent=self.root)
        return
    try:
        folder = os.path.abspath(os.path.expanduser(str(self.output_var.get()).strip() or 'ROI_exports'))
        os.makedirs(folder, exist_ok=True)
        total = len(self.recons)
        for i in range(total):
            field = float(self.real_fields_mT[i])
            path = os.path.join(folder, f'ROI_{i:03d}_{field:+.2f}mT{extension}')
            fig = None
            try:
                image = _proc_image(self, i)
                fig = _tab6_export_figure(self, image, i, dpi=dpi, title_prefix='ROI')
                save_kwargs = {'dpi': dpi, 'bbox_inches': 'tight', 'pad_inches': 0.08, 'format': fmt}
                if fmt == 'JPEG':
                    save_kwargs['facecolor'] = 'white'
                fig.savefig(path, **save_kwargs)
            finally:
                if fig is not None:
                    try:
                        plt.close(fig)
                    except Exception:
                        pass
            self.proc_status_var.set(f'Exporting {fmt}: {i + 1}/{total} | {dpi} DPI')
            self.root.update_idletasks()
        self.proc_status_var.set(f'Batch {fmt} export complete | {total} images | axes + colorbar')
        self.log(f'Tab 6 batch export complete: format={fmt} | count={total} | dpi={dpi} | axes+colorbar | folder={folder}')
        _tab6_open_output_folder(folder)
    except Exception as exc:
        self.proc_status_var.set('Batch export failed.')
        self.log(f'Tab 6 batch export failed: {exc}')
        messagebox.showerror('Batch Export Error', str(exc), parent=self.root)

def _tab6_mp4_complete(self, indices, filename):
    if not getattr(self, 'roi_resolution_applied', False):
        messagebox.showwarning('ROI Resolution Required', 'Apply SAME CONFIRMED ROI first.', parent=self.root)
        return
    imageio = _proc_imageio(self)
    if imageio is None:
        return
    try:
        dpi = int(self.proc_export_dpi_var.get())
    except Exception:
        dpi = 300
    if dpi not in (150, 300, 600, 900):
        messagebox.showwarning('Export DPI', 'Select 150, 300, 600 or 900 DPI.', parent=self.root)
        return
    folder = os.path.abspath(os.path.expanduser(str(self.output_var.get()).strip() or 'ROI_exports'))
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, filename)
    indices = list(indices)
    if not indices:
        messagebox.showwarning('MP4 Export', 'No frames selected.', parent=self.root)
        return
    fps = max(1, int(self.fps_var.get()))
    fig = None
    writer = None
    try:
        fig = plt.figure(figsize=(8.6, 7.2), dpi=dpi, facecolor='white')
        ax = fig.add_axes([0.1, 0.1, 0.76, 0.82])
        cax = fig.add_axes([0.89, 0.1, 0.035, 0.82])
        writer = None
        codec_attempts = ('libx264', 'mpeg4', None)
        errors = []
        for codec in codec_attempts:
            try:
                kwargs = {'fps': fps, 'format': 'ffmpeg', 'macro_block_size': None}
                if codec is not None:
                    kwargs['codec'] = codec
                writer = imageio.get_writer(path, **kwargs)
                break
            except Exception as exc:
                errors.append(f"{codec or 'default'}: {exc}")
        if writer is None:
            raise RuntimeError('Could not initialize an MP4 encoder.\n\n' + '\n'.join(errors))
        for count, i in enumerate(indices, start=1):
            image = np.asarray(_proc_image(self, i), dtype=float)
            h, w = image.shape[:2]
            ax.clear()
            cax.clear()
            ax.set_aspect('equal', adjustable='box')
            cmap_name = 'RdBu_r'
            try:
                candidate = str(self.colormap_var.get()).strip()
                if candidate:
                    cmap_name = candidate
            except Exception:
                pass
            im = ax.imshow(np.clip(image, 0.0, 1.0), cmap=cmap_name, origin='lower', interpolation='nearest', vmin=0.0, vmax=1.0, aspect='equal')
            physical = False
            try:
                base = self._proc_get_tab5_base_pixel_size()
                if base is not None and np.isfinite(base) and (base > 0):
                    ow = int(getattr(self, 'original_roi_w', 0) or w)
                    oh = int(getattr(self, 'original_roi_h', 0) or h)
                    if ow > 0 and oh > 0:
                        pw, ph = (float(ow) * float(base), float(oh) * float(base))
                        im.set_extent((0.0, pw, 0.0, ph))
                        ax.set_xlim(0.0, pw)
                        ax.set_ylim(0.0, ph)
                        ax.set_xlabel('X (nm)')
                        ax.set_ylabel('Y (nm)')
                        physical = True
            except Exception:
                physical = False
            if not physical:
                im.set_extent((-0.5, w - 0.5, -0.5, h - 0.5))
                ax.set_xlim(-0.5, w - 0.5)
                ax.set_ylim(-0.5, h - 0.5)
                ax.set_xlabel('X (pixel)')
                ax.set_ylabel('Y (pixel)')
            ax.set_aspect('equal', adjustable='box')
            ax.set_anchor('C')
            try:
                field = float(self.real_fields_mT[i])
                title = f'ROI | Frame {i + 1}/{len(self.recons)} | Field = {field:+.2f} mT'
            except Exception:
                title = f'ROI | Frame {i + 1}/{len(self.recons)}'
            ax.set_title(title, fontsize=12)
            cb = fig.colorbar(im, cax=cax)
            cb.set_label('Intensity', rotation=90)
            fig.canvas.draw()
            frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            fh, fw = frame.shape[:2]
            if fh % 2 or fw % 2:
                frame = np.pad(frame, ((0, fh % 2), (0, fw % 2), (0, 0)), mode='edge')
            writer.append_data(frame)
            self.proc_status_var.set(f'MP4: {count}/{len(indices)} | {fps} FPS | {dpi} DPI | axes + colorbar')
            self.root.update_idletasks()
    except Exception as exc:
        self.proc_status_var.set('MP4 export failed.')
        self.log(f'MP4 export failed: {path} | {exc}')
        messagebox.showerror('MP4 Error', str(exc), parent=self.root)
        return
    finally:
        try:
            if writer is not None:
                writer.close()
        except Exception:
            pass
        try:
            if fig is not None:
                plt.close(fig)
        except Exception:
            pass
    self.proc_status_var.set(f'MP4 saved: {filename} | {fps} FPS | {dpi} DPI | axes + colorbar')
    self.log(f'MP4 saved: {path} | frames={len(indices)} | fps={fps} | dpi={dpi} | axes+colorbar')
    _tab6_open_output_folder(folder)
    messagebox.showinfo('MP4 Export', f'MP4 successfully saved:\n\n{path}\n\nFrames: {len(indices)}\nFPS: {fps}\nDPI: {dpi}\nAxes + colorbar included.', parent=self.root)

def _tab6_single_mp4_complete(self):
    i = int(self.proc_idx_var.get())
    frames = max(1, int(self.fps_var.get()) * 2)
    _tab6_mp4_complete(self, [i] * frames, f'ROI_single_{i:03d}_{self.real_fields_mT[i]:+.2f}mT.mp4')

def _tab6_batch_mp4_complete(self):
    try:
        indices, start, end, fps = _proc_get_batch_mp4_indices(self)
        self.fps_var.set(fps)
        _tab6_mp4_complete(self, indices, f'ROI_frames_{start:04d}-{end:04d}_{fps}fps_axes_colorbar.mp4')
    except Exception as exc:
        messagebox.showerror('Batch MP4 Export', str(exc), parent=self.root)
CDIWorkflowApp.save_single = _tab6_save_single_complete
CDIWorkflowApp.batch_export = _tab6_batch_complete
CDIWorkflowApp.create_mp4 = _tab6_mp4_complete
CDIWorkflowApp.export_single_mp4 = _tab6_single_mp4_complete
CDIWorkflowApp.export_batch_mp4 = _tab6_batch_mp4_complete
CDIWorkflowApp._tab6_open_output_folder = _tab6_open_output_folder

def _build_ui_optimized(self):
    style = ttk.Style()
    try:
        style.theme_use('clam')
    except Exception:
        pass
    top = ttk.Frame(self.root, padding=8)
    top.pack(fill='x')
    ttk.Label(top, text='CDI / Reconstruction / ROI Workflow', font=('Arial', 18, 'bold')).pack(side='left')
    self.status_var = tk.StringVar(value='Ready.')
    ttk.Label(top, textvariable=self.status_var, foreground='darkgreen').pack(side='right')
    self.nb = ttk.Notebook(self.root)
    self.nb.pack(fill='both', expand=True, padx=8, pady=(0, 8))
    self.tab_data = ttk.Frame(self.nb)
    self.tab_recon = ttk.Frame(self.nb)
    self.tab_support = ttk.Frame(self.nb)
    self.tab_roi = ttk.Frame(self.nb)
    self.tab_pixel_calibration = ttk.Frame(self.nb)
    self.tab_proc = ttk.Frame(self.nb)
    for tab, name in [(self.tab_data, '1. Import Data'), (self.tab_recon, '2. Reconstruction'), (self.tab_support, '3. Mask Optimization'), (self.tab_roi, '4. ROI Selection'), (self.tab_pixel_calibration, '5. Pixel Calibration'), (self.tab_proc, '6. Image Processing')]:
        self.nb.add(tab, text=name)
    self._build_data_tab()
    self._build_recon_tab()
    self._build_support_tab()
    self._build_roi_tab()
    self._build_pixel_calibration_tab()
    self._build_proc_tab()
    self.tab_line_profile_analysis = ttk.Frame(self.nb)
    self.tab_line_profile = self.tab_line_profile_analysis
    self.nb.add(self.tab_line_profile_analysis, text='7. Line Profile')
    _build_line_profile_analysis_tab(self)
    try:
        self.nb.bind('<<NotebookTabChanged>>', lambda e: _lpa_tab_selected(self, e), add='+')
        _lpa_fetch_roi2(self, select_tab=False)
    except Exception:
        pass
CDIWorkflowApp._lpa_fetch_roi2 = _lpa_fetch_roi2
CDIWorkflowApp._lpa_render_frame = _lpa_render_frame
CDIWorkflowApp._lpa_slider_changed = _lpa_slider_changed
CDIWorkflowApp._lpa_reset_zoom = _lpa_reset_zoom
CDIWorkflowApp._lpa_start_draw = _lpa_start_draw
CDIWorkflowApp._lpa_clear_lines = _lpa_clear_lines

CDIWorkflowApp._build_ui = _build_ui_optimized
# =====================================================================
# TAB-7 MEASURE / CLEAR-LINES FINAL PATCH
# - CLEAR LINES clears drawn ROI2 lines, profile plots, and measurements.
# - MEASURE starts a two-end-point measurement on the selected line plot.
# - M1/M2 snap to the selected profile curve.
# - Either endpoint can be dragged with LEFT mouse button along that curve.
# - Measured distance is reported in nm and tabulated.
# - The same endpoint positions are highlighted on ROI2.
# =====================================================================

def _t7_measure_pick_curve(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph:
        return None
    if event.xdata is None or event.ydata is None:
        return None
    tx, ty = self.line_profile_ax_graph.transData.transform((float(event.xdata), float(event.ydata)))
    best = None
    for j, line in enumerate(getattr(self, '_t7_lines', []) or []):
        d = np.asarray(line.get('distance_nm', []), dtype=float)
        p = np.asarray(line.get('profile', []), dtype=float)
        if d.size < 2 or p.size != d.size:
            continue
        pts = self.line_profile_ax_graph.transData.transform(np.column_stack((d, p)))
        a = pts[:-1]
        b = pts[1:]
        v = b - a
        vv = np.einsum('ij,ij->i', v, v)
        w = np.array([tx, ty], dtype=float) - a
        t = np.einsum('ij,ij->i', w, v) / np.where(vv > 0, vv, 1.0)
        t = np.clip(t, 0.0, 1.0)
        q = a + v * t[:, None]
        dd = np.hypot(q[:,0] - tx, q[:,1] - ty)
        k = int(np.argmin(dd))
        dist_px = float(dd[k])
        xnm = float(d[k] + t[k] * (d[k+1] - d[k]))
        yv = float(p[k] + t[k] * (p[k+1] - p[k]))
        if best is None or dist_px < best[0]:
            best = (dist_px, j, xnm, yv)
    if best is None or best[0] > 25.0:
        return None
    return best


def _t7_measure_clear_artists(self):
    for attr in ('_t7_measure_graph_artists', '_t7_measure_image_artists'):
        for a in getattr(self, attr, []) or []:
            try:
                a.remove()
            except Exception:
                pass
        setattr(self, attr, [])


def _t7_measure_profile_points(self, m):
    line_idx = int(m['set']) - 1
    if line_idx < 0 or line_idx >= len(getattr(self, '_t7_lines', []) or []):
        return None
    line = self._t7_lines[line_idx]
    d = np.asarray(line.get('distance_nm', []), dtype=float)
    p = np.asarray(line.get('profile', []), dtype=float)
    if d.size < 2 or p.size != d.size:
        return None
    x0 = float(m['m1_nm'])
    x1 = float(m['m2_nm'])
    y0 = float(np.interp(x0, d, p))
    y1 = float(np.interp(x1, d, p))
    return line, d, p, x0, x1, y0, y1


def _t7_measure_redraw(self):
    self._t7_measure_clear_artists()
    self._t7_measure_graph_artists = []
    self._t7_measure_image_artists = []
    ax = self.line_profile_ax_graph
    iax = self.line_profile_ax_image
    sx = float(getattr(self, '_t7_sx_nm_px', 1.0))
    sy = float(getattr(self, '_t7_sy_nm_px', 1.0))

    for mi, m in enumerate(getattr(self, '_t7_measurements', []) or [], 1):
        got = self._t7_measure_profile_points(m)
        if got is None:
            continue
        line, d, p, x0, x1, y0, y1 = got

        seg, = ax.plot([x0, x1], [y0, y1], '-', lw=2.2, color='black', zorder=20)
        self._t7_measure_graph_artists.append(seg)
        for label, x, y in ((f"M{2*mi-1}", x0, y0), (f"M{2*mi}", x1, y1)):
            pt, = ax.plot([x], [y], 'o', ms=8, mfc='yellow', mec='black', mew=1.1, zorder=22)
            self._t7_measure_graph_artists.append(pt)
            tx = ax.annotate(label, (x, y), xytext=(5, 6), textcoords='offset points', fontsize=8, fontweight='bold',
                             bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='black', alpha=0.9), zorder=23)
            self._t7_measure_graph_artists.append(tx)

        ylo, yhi = ax.get_ylim()
        ybar = yhi - (0.10 + 0.08 * (mi - 1)) * max(yhi - ylo, 1e-12)
        bar = ax.annotate('', xy=(x1, ybar), xytext=(x0, ybar),
                          arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
        self._t7_measure_graph_artists.append(bar)
        lab = ax.text((x0+x1)/2.0, ybar, f"ΔL = {abs(x1-x0):.6g} nm",
                      ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='black',
                      bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='black', alpha=0.88), zorder=24)
        self._t7_measure_graph_artists.append(lab)

        # Map the measured x positions back to the original ROI2 drawing line.
        p0 = np.asarray(line['p0'], dtype=float)
        p1 = np.asarray(line['p1'], dtype=float)
        total_nm = float(d[-1]) if d.size else 0.0
        if total_nm > 0:
            q0 = p0 + (x0 / total_nm) * (p1 - p0)
            q1 = p0 + (x1 / total_nm) * (p1 - p0)
        else:
            q0, q1 = p0, p1
        q0 = q0 * np.asarray([sx, sy])
        q1 = q1 * np.asarray([sx, sy])
        sh, = iax.plot([q0[0], q1[0]], [q0[1], q1[1]], '-', lw=7, color='black', alpha=0.75, zorder=40)
        yy, = iax.plot([q0[0], q1[0]], [q0[1], q1[1]], '-', lw=4, color='yellow', zorder=41)
        self._t7_measure_image_artists.extend([sh, yy])
        for q in (q0, q1):
            mk, = iax.plot([q[0]], [q[1]], 'o', ms=7, mfc='yellow', mec='black', mew=1.0, zorder=42)
            self._t7_measure_image_artists.append(mk)

    try:
        self.line_profile_canvas_graph.draw_idle()
        self.line_profile_canvas_image.draw_idle()
    except Exception:
        pass


def _t7_measure_update_table(self):
    tree = getattr(self, 'line_profile_tree', None)
    if tree is None:
        return
    for iid in tree.get_children():
        tree.delete(iid)
    for mi, m in enumerate(getattr(self, '_t7_measurements', []) or [], 1):
        tree.insert('', 'end', values=(
            int(m['set']),
            f"{float(m['m1_nm']):.6g}",
            f"{float(m['m2_nm']):.6g}",
            f"{float(m['delta_nm']):.6g}",
            f"{float(m['m1_intensity']):.6g}",
            f"{float(m['m2_intensity']):.6g}",
        ))


def _t7_measure_start_final(self):
    if not getattr(self, '_t7_lines', None):
        self.line_profile_measure_var.set('Draw at least one profile line before measuring.')
        return
    self._t7_measure_mode = True
    self._t7_measure_first = None
    self.line_profile_measure_var.set('MEASURE: click M1 and M2 on the SAME line profile. Endpoints can be dragged afterwards.')


def _t7_measure_graph_press_final(self, event):
    if event is None or event.inaxes is not self.line_profile_ax_graph or event.button != 1:
        return

    # Existing measurement endpoint gets drag priority when Measure mode is off.
    if not getattr(self, '_t7_measure_mode', False):
        best = None
        for mi, m in enumerate(getattr(self, '_t7_measurements', []) or []):
            got = self._t7_measure_profile_points(m)
            if got is None:
                continue
            _, _, _, x0, x1, y0, y1 = got
            for k, (xn, yn) in enumerate(((x0, y0), (x1, y1))):
                sxp, syp = self.line_profile_ax_graph.transData.transform((xn, yn))
                dd = float(np.hypot(event.x - sxp, event.y - syp))
                if best is None or dd < best[0]:
                    best = (dd, mi, k)
        if best is not None and best[0] <= 16.0:
            self._t7_measure_drag = (best[1], best[2])
            self.line_profile_measure_var.set(f"Dragging M{2*best[1] + best[2] + 1}. Release to set the new endpoint.")
            return

    if not getattr(self, '_t7_measure_mode', False):
        return
    hit = _t7_measure_pick_curve(self, event)
    if hit is None:
        self.line_profile_measure_var.set('Click directly on a profile curve.')
        return
    _dist_px, line_idx, xnm, yv = hit
    if self._t7_measure_first is None:
        self._t7_measure_first = {'set': line_idx + 1, 'x': xnm, 'y': yv}
        self.line_profile_measure_var.set(f"M1 selected on Line {line_idx + 1} at {xnm:.6g} nm. Click M2 on the same line.")
        return

    first = self._t7_measure_first
    if int(first['set']) != int(line_idx + 1):
        self.line_profile_measure_var.set(f"M2 must be on the same Line {first['set']} as M1.")
        return

    line = self._t7_lines[line_idx]
    d = np.asarray(line.get('distance_nm', []), dtype=float)
    p = np.asarray(line.get('profile', []), dtype=float)
    if d.size < 2 or p.size != d.size:
        return
    x1 = float(xnm)
    x0 = float(first['x'])
    m1_nm, m2_nm = (x0, x1)
    i1 = float(np.interp(m1_nm, d, p))
    i2 = float(np.interp(m2_nm, d, p))
    self._t7_measurements.append({
        'set': int(line_idx + 1),
        'm1_nm': m1_nm,
        'm2_nm': m2_nm,
        'delta_nm': abs(m2_nm - m1_nm),
        'm1_intensity': i1,
        'm2_intensity': i2,
    })
    self._t7_measure_first = None
    self._t7_measure_mode = False
    self._t7_measure_update_table()
    self._t7_measure_redraw()
    self.line_profile_measure_var.set(f"Line {line_idx + 1}: M1={m1_nm:.6g} nm, M2={m2_nm:.6g} nm, measured distance ΔL={abs(m2_nm-m1_nm):.6g} nm")


def _t7_measure_graph_motion_final(self, event):
    drag = getattr(self, '_t7_measure_drag', None)
    if drag is None or event is None or event.inaxes is not self.line_profile_ax_graph or event.xdata is None:
        return
    mi, point_idx = drag
    if mi < 0 or mi >= len(getattr(self, '_t7_measurements', []) or []):
        return
    m = self._t7_measurements[mi]
    line_idx = int(m['set']) - 1
    if line_idx < 0 or line_idx >= len(self._t7_lines):
        return
    line = self._t7_lines[line_idx]
    d = np.asarray(line.get('distance_nm', []), dtype=float)
    p = np.asarray(line.get('profile', []), dtype=float)
    if d.size < 2 or p.size != d.size:
        return
    xnm = float(np.clip(float(event.xdata), d[0], d[-1]))
    yv = float(np.interp(xnm, d, p))
    if point_idx == 0:
        m['m1_nm'] = xnm
    else:
        m['m2_nm'] = xnm
    m['m1_nm'], m['m2_nm'] = (float(m['m1_nm']), float(m['m2_nm']))
    m['delta_nm'] = abs(m['m2_nm'] - m['m1_nm'])
    m['m1_intensity'] = float(np.interp(m['m1_nm'], d, p))
    m['m2_intensity'] = float(np.interp(m['m2_nm'], d, p))
    self._t7_measure_update_table()
    self._t7_measure_redraw()
    self.line_profile_measure_var.set(f"Line {line_idx + 1}: measured distance ΔL={m['delta_nm']:.6g} nm. Drag M1 or M2 with left mouse button.")


def _t7_measure_graph_release_final(self, event=None):
    if getattr(self, '_t7_measure_drag', None) is not None:
        self.line_profile_measure_var.set('Measurement endpoint fixed. Press MEASURE for another pair or drag an existing endpoint again.')
    self._t7_measure_drag = None


def _t7_clear_lines_final(self):
    self._t7_draw_mode = False
    self._t7_measure_mode = False
    self._t7_measure_first = None
    self._t7_measure_drag = None
    self._t7_drag_start = None
    self._t7_lines = []
    self._t7_measurements = []
    try:
        self._t7_remove_dynamic_artists()
    except Exception:
        pass
    try:
        _t7_redraw_lines(self)
    except Exception:
        pass
    try:
        _t7_redraw_profiles(self, getattr(self, '_t7_frame', 0))
    except Exception:
        pass
    self._t7_measure_clear_artists()
    self._t7_measure_update_table()
    try:
        self.line_profile_status_var.set('All profile lines, profile plots and measurements cleared.')
        self.line_profile_measure_var.set('')
    except Exception:
        pass


def _t7_rebind_measure_ui(self):
    # Rename the existing measurement button to the requested simple label.
    try:
        tab = self.tab_line_profile
        stack = [tab]
        while stack:
            w = stack.pop()
            try:
                children = list(w.winfo_children())
            except Exception:
                children = []
            for ch in children:
                stack.append(ch)
                if isinstance(ch, (ttk.Button, tk.Button)):
                    try:
                        txt = str(ch.cget('text'))
                        if txt in ('MEASURE M1 / M2', 'DYNAMIC MEASURING SCALE', 'DYNAMIC SCALE'):
                            ch.configure(text='MEASURE', command=self._t7_measure_start_final, width=12)
                    except Exception:
                        pass
    except Exception:
        pass

    # Replace the old graph-click-only measurement binding with press/motion/release.
    try:
        canvas = self.line_profile_canvas_graph
        registry = getattr(canvas.callbacks, 'callbacks', {}).get('button_press_event', {})
        for cid in list(registry.keys()):
            canvas.mpl_disconnect(cid)
        canvas.mpl_connect('button_press_event', self._t7_measure_graph_press_final)
        canvas.mpl_connect('motion_notify_event', self._t7_measure_graph_motion_final)
        canvas.mpl_connect('button_release_event', self._t7_measure_graph_release_final)
    except Exception:
        pass
    self._t7_measure_update_table()
    self._t7_measure_redraw()

_T7_OLD_BUILD_MEASURE = CDIWorkflowApp._build_ui

def _t7_build_measure_final(self, *args, **kwargs):
    result = _T7_OLD_BUILD_MEASURE(self, *args, **kwargs)
    try:
        self._t7_measure_drag = None
        self._t7_rebind_measure_ui()
    except Exception:
        pass
    return result

CDIWorkflowApp._t7_measure_pick_curve = _t7_measure_pick_curve
CDIWorkflowApp._t7_measure_clear_artists = _t7_measure_clear_artists
CDIWorkflowApp._t7_measure_profile_points = _t7_measure_profile_points
CDIWorkflowApp._t7_measure_redraw = _t7_measure_redraw
CDIWorkflowApp._t7_measure_update_table = _t7_measure_update_table
CDIWorkflowApp._t7_measure_start_final = _t7_measure_start_final
CDIWorkflowApp._t7_measure_graph_press_final = _t7_measure_graph_press_final
CDIWorkflowApp._t7_measure_graph_motion_final = _t7_measure_graph_motion_final
CDIWorkflowApp._t7_measure_graph_release_final = _t7_measure_graph_release_final
CDIWorkflowApp._t7_clear_lines_final = _t7_clear_lines_final
CDIWorkflowApp._t7_rebind_measure_ui = _t7_rebind_measure_ui
CDIWorkflowApp._t7_measure_start = _t7_measure_start_final
CDIWorkflowApp._t7_clear_lines = _t7_clear_lines_final
CDIWorkflowApp._build_ui = _t7_build_measure_final



# Final Tab-7 LPA measurement bindings.
CDIWorkflowApp._lpa_measure_start = _lpa_measure_start
CDIWorkflowApp._lpa_measure_graph_press = _lpa_measure_graph_press
CDIWorkflowApp._lpa_measure_graph_motion = _lpa_measure_graph_motion
CDIWorkflowApp._lpa_measure_graph_release = _lpa_measure_graph_release
CDIWorkflowApp._lpa_measure_redraw = _lpa_measure_redraw
CDIWorkflowApp._lpa_measure_update_table = _lpa_measure_update_table


# ============================================================================
# TAB-7 MULTI-LINE MEASUREMENT + EXPORT + ROMAN-I FINAL PATCH
# Requirements implemented:
#   1) Every drawn line has its own pair: Line 1=M1-M2, Line 2=M3-M4, ...
#      Only one two-end-point measurement is allowed per drawn line.
#   2) Measurement values are tabulated in the Tab-7 table.
#   3) TABLE EXPORT saves the complete measurement table as CSV.
#   4) PLOT EXPORT saves both ROI2 (with physical axes, colorbar, scale bar,
#      drawn lines and measurement overlays) and line-profile graph as PNG,
#      300 DPI, then opens the output folder.
#   5) Graph measurement endpoints use a Roman-I style marker instead of
#      yellow circular dots. Endpoints remain draggable with left mouse drag
#      and stay constrained to their corresponding profile curve.
# ============================================================================


def _t7v2_init_measure_state(self):
    if not hasattr(self, '_lpa_measurements'):
        self._lpa_measurements = []
    if not hasattr(self, '_lpa_measure_mode'):
        self._lpa_measure_mode = False
    if not hasattr(self, '_lpa_measure_first'):
        self._lpa_measure_first = None
    if not hasattr(self, '_lpa_measure_drag'):
        self._lpa_measure_drag = None


def _t7v2_line_profile(self, line_idx):
    stack = getattr(self, '_lpa_stack', None)
    lines = getattr(self, '_lpa_lines', []) or []
    if stack is None or len(stack) == 0 or line_idx < 0 or line_idx >= len(lines):
        return None
    try:
        idx = max(0, min(int(self._lpa_frame_var.get()), len(stack) - 1))
    except Exception:
        idx = 0
    image = np.asarray(stack[idx], dtype=float)
    p0, p1 = lines[line_idx]
    d, prof = _lpa_profile_for_line(self, image, p0, p1)
    if d.size < 2 or prof.size != d.size:
        return None
    return d, prof


def _t7v2_point_on_profile(self, line_idx, x_nm):
    got = _t7v2_line_profile(self, line_idx)
    if got is None:
        return None
    d, prof = got
    x = float(np.clip(float(x_nm), float(d[0]), float(d[-1])))
    y = float(np.interp(x, d, prof))
    return x, y, d, prof


def _t7v2_measure_pick_curve(self, event, max_screen=30.0):
    if (event is None or event.inaxes is not self._lpa_plot_ax or
            event.xdata is None or event.ydata is None):
        return None
    target = self._lpa_plot_ax.transData.transform((float(event.xdata), float(event.ydata)))
    best = None
    for j in range(len(getattr(self, '_lpa_lines', []) or [])):
        got = _t7v2_line_profile(self, j)
        if got is None:
            continue
        d, prof = got
        pts = self._lpa_plot_ax.transData.transform(np.column_stack((d, prof)))
        a, b = pts[:-1], pts[1:]
        v = b - a
        vv = np.einsum('ij,ij->i', v, v)
        w = target - a
        t = np.einsum('ij,ij->i', w, v) / np.where(vv > 0, vv, 1.0)
        t = np.clip(t, 0.0, 1.0)
        q = a + v * t[:, None]
        dd = np.hypot(q[:, 0] - target[0], q[:, 1] - target[1])
        k = int(np.argmin(dd))
        if best is None or float(dd[k]) < best[0]:
            x = float(d[k] + t[k] * (d[k + 1] - d[k]))
            y = float(prof[k] + t[k] * (prof[k + 1] - prof[k]))
            best = (float(dd[k]), j, x, y, d, prof)
    return best if best is not None and best[0] <= max_screen else None


def _t7v2_measure_start(self):
    _t7v2_init_measure_state(self)
    lines = getattr(self, '_lpa_lines', []) or []
    if not lines:
        self._lpa_measure_mode = False
        self._lpa_measure_first = None
        self._lpa_status_var.set('Draw at least one line before measuring.')
        return
    measured_lines = {int(m.get('line', -1)) for m in (getattr(self, '_lpa_measurements', []) or [])}
    if len(measured_lines) >= len(lines):
        self._lpa_status_var.set('Every drawn line already has a two-point measurement. Drag M1/M2, or clear lines to restart.')
        return
    self._lpa_measure_mode = True
    self._lpa_measure_first = None
    self._lpa_measure_drag = None
    self._lpa_status_var.set('MEASURE: click the first endpoint on a line, then the second endpoint on the SAME line.')
    _t7v2_measure_redraw(self)


def _t7v2_measure_graph_press(self, event):
    if event is None or event.inaxes is not self._lpa_plot_ax or event.button != 1:
        return
    _t7v2_init_measure_state(self)

    # First priority: drag an existing endpoint with left mouse button.
    best = None
    for mi, m in enumerate(getattr(self, '_lpa_measurements', []) or []):
        line_idx = int(m.get('line', -1))
        for point_idx, x_nm in enumerate((float(m.get('m1_nm', 0.0)), float(m.get('m2_nm', 0.0)))):
            got = _t7v2_point_on_profile(self, line_idx, x_nm)
            if got is None:
                continue
            _, y, _, _ = got
            sxp, syp = self._lpa_plot_ax.transData.transform((x_nm, y))
            dd = float(np.hypot(event.x - sxp, event.y - syp))
            if best is None or dd < best[0]:
                best = (dd, mi, point_idx)
    if best is not None and best[0] <= 18.0:
        self._lpa_measure_drag = (best[1], best[2])
        self._lpa_measure_mode = False
        mi, point_idx = best[1], best[2]
        line_no = int(self._lpa_measurements[mi]['line']) + 1
        m_id = 2 * int(self._lpa_measurements[mi]['line']) + point_idx + 1
        self._lpa_status_var.set(f'Dragging M{m_id} on Line {line_no}. Release to set the new endpoint.')
        return

    if not getattr(self, '_lpa_measure_mode', False):
        return

    hit = _t7v2_measure_pick_curve(self, event)
    if hit is None:
        self._lpa_status_var.set('Click directly on a line-profile curve.')
        return
    _, line_idx, x_nm, yv, d, prof = hit
    measured_lines = {int(m.get('line', -1)) for m in (getattr(self, '_lpa_measurements', []) or [])}
    if int(line_idx) in measured_lines:
        self._lpa_status_var.set(f'Line {line_idx + 1} is already measured. Drag its M{2*line_idx+1}/M{2*line_idx+2} endpoints, or clear lines.')
        return

    if self._lpa_measure_first is None:
        self._lpa_measure_first = {'line': int(line_idx), 'x': float(x_nm)}
        m_id = 2 * int(line_idx) + 1
        self._lpa_status_var.set(f'M{m_id} selected on Line {line_idx + 1} at {x_nm:.6g} nm. Click M{m_id+1} on the SAME line.')
        _t7v2_measure_redraw(self)
        return

    first = self._lpa_measure_first
    if int(first['line']) != int(line_idx):
        m_id = 2 * int(first['line']) + 1
        self._lpa_status_var.set(f'M{m_id+1} must be on the same Line {int(first["line"]) + 1} as M{m_id}.')
        return

    x0 = float(first['x'])
    x1 = float(x_nm)
    i0 = float(np.interp(x0, d, prof))
    i1 = float(np.interp(x1, d, prof))
    self._lpa_measurements.append({
        'line': int(line_idx),
        'm1_nm': x0,
        'm2_nm': x1,
        'delta_nm': abs(x1 - x0),
        'm1_intensity': i0,
        'm2_intensity': i1,
    })
    self._lpa_measure_first = None
    self._lpa_measure_mode = False
    _t7v2_measure_update_table(self)
    _lpa_redraw_lines_and_plot(self)
    _t7v2_measure_redraw(self)
    self._lpa_status_var.set(
        f'Line {line_idx + 1}: M{2*line_idx+1}={x0:.6g} nm, '
        f'M{2*line_idx+2}={x1:.6g} nm, ΔL={abs(x1-x0):.6g} nm'
    )


def _t7v2_measure_graph_motion(self, event):
    drag = getattr(self, '_lpa_measure_drag', None)
    if drag is None or event is None or event.inaxes is not self._lpa_plot_ax or event.xdata is None:
        return
    mi, point_idx = drag
    measurements = getattr(self, '_lpa_measurements', []) or []
    if mi < 0 or mi >= len(measurements):
        return
    m = measurements[mi]
    line_idx = int(m['line'])
    got = _t7v2_point_on_profile(self, line_idx, float(event.xdata))
    if got is None:
        return
    x_nm, y, d, prof = got
    if point_idx == 0:
        m['m1_nm'] = float(x_nm)
        m['m1_intensity'] = float(y)
    else:
        m['m2_nm'] = float(x_nm)
        m['m2_intensity'] = float(y)
    m['delta_nm'] = abs(float(m['m2_nm']) - float(m['m1_nm']))
    _t7v2_measure_update_table(self)
    _lpa_redraw_lines_and_plot(self)
    _t7v2_measure_redraw(self)
    self._lpa_status_var.set(f'Line {line_idx+1}: ΔL={m["delta_nm"]:.6g} nm. Drag M{2*line_idx+1} / M{2*line_idx+2} with left mouse button.')


def _t7v2_measure_graph_release(self, event=None):
    if getattr(self, '_lpa_measure_drag', None) is not None:
        drag = self._lpa_measure_drag
        try:
            m = self._lpa_measurements[drag[0]]
            ln = int(m['line']) + 1
            self._lpa_status_var.set(f'Line {ln} measurement updated. Endpoints remain draggable.')
        except Exception:
            pass
    self._lpa_measure_drag = None


def _t7v2_measure_clear_artists(self):
    for attr in ('_lpa_measure_artists', '_lpa_measure_image_artists'):
        for a in getattr(self, attr, []) or []:
            try:
                a.remove()
            except Exception:
                pass
        setattr(self, attr, [])


def _t7v2_measure_redraw(self):
    _t7v2_init_measure_state(self)
    _t7v2_measure_clear_artists(self)
    ax = self._lpa_plot_ax
    iax = self._lpa_ax
    self._lpa_measure_artists = []
    self._lpa_measure_image_artists = []

    # Completed measurements. Marker is a Roman-I text glyph (no yellow dot).
    for m in getattr(self, '_lpa_measurements', []) or []:
        line_idx = int(m.get('line', -1))
        got1 = _t7v2_point_on_profile(self, line_idx, float(m.get('m1_nm', 0.0)))
        got2 = _t7v2_point_on_profile(self, line_idx, float(m.get('m2_nm', 0.0)))
        if got1 is None or got2 is None:
            continue
        x0, y0, d, p = got1
        x1, y1, _, _ = got2
        m['m1_nm'] = float(x0); m['m2_nm'] = float(x1)
        m['delta_nm'] = abs(float(x1) - float(x0))
        m['m1_intensity'] = float(y0); m['m2_intensity'] = float(y1)

        seg = ax.plot([x0, x1], [y0, y1], '-', lw=2.0, color='black', zorder=20)[0]
        self._lpa_measure_artists.append(seg)
        marker_color = 'black'
        for x, y, mid in ((x0, y0, 2*line_idx+1), (x1, y1, 2*line_idx+2)):
            mi = ax.text(x, y, 'I', ha='center', va='center', fontsize=15,
                         fontweight='bold', color=marker_color, zorder=30,
                         bbox=dict(boxstyle='round,pad=0.08', facecolor='white',
                                   edgecolor='black', alpha=0.95))
            self._lpa_measure_artists.append(mi)
            lab = ax.text(x, y, f' M{mid} ', ha='center', va='bottom', fontsize=8,
                          fontweight='bold', zorder=31,
                          bbox=dict(boxstyle='round,pad=0.12', facecolor='white',
                                    edgecolor='black', alpha=0.9))
            self._lpa_measure_artists.append(lab)

        ymin, ymax = ax.get_ylim()
        ybar = ymax - 0.10 * max(ymax - ymin, 1e-12)
        arrow = ax.annotate('', xy=(x1, ybar), xytext=(x0, ybar),
                            arrowprops=dict(arrowstyle='<->', color='blue', lw=1.8), zorder=24)
        self._lpa_measure_artists.append(arrow)
        midlab = ax.text((x0+x1)/2.0, ybar, f'ΔL = {abs(x1-x0):.6g} nm',
                         ha='center', va='bottom', fontsize=8.5, fontweight='bold',
                         color='black', zorder=25,
                         bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                                   edgecolor='blue', alpha=0.9))
        self._lpa_measure_artists.append(midlab)

        # Matching yellow measurement segment on ROI2, while endpoint marker
        # remains a simple Roman-I style glyph.
        try:
            p0 = np.asarray(self._lpa_lines[line_idx][0], dtype=float)
            p1 = np.asarray(self._lpa_lines[line_idx][1], dtype=float)
            total = float(d[-1])
            if total > 0:
                q0 = p0 + (x0 / total) * (p1 - p0)
                q1 = p0 + (x1 / total) * (p1 - p0)
                image_seg = iax.plot([q0[0], q1[0]], [q0[1], q1[1]], color='yellow', lw=3.0, zorder=70)[0]
                self._lpa_measure_image_artists.append(image_seg)
                for qx, qy, mid in ((q0[0], q0[1], 2*line_idx+1), (q1[0], q1[1], 2*line_idx+2)):
                    ta = iax.text(qx, qy, 'I', color='black', fontsize=14,
                                  fontweight='bold', ha='center', va='center', zorder=71,
                                  bbox=dict(boxstyle='round,pad=0.08', facecolor='white',
                                            edgecolor='black', alpha=0.95))
                    self._lpa_measure_image_artists.append(ta)
                    tl = iax.text(qx, qy, f'M{mid}', color='black', fontsize=8,
                                  fontweight='bold', ha='center', va='bottom', zorder=72,
                                  bbox=dict(boxstyle='round,pad=0.10', facecolor='white',
                                            edgecolor='black', alpha=0.9))
                    self._lpa_measure_image_artists.append(tl)
        except Exception:
            pass

    # Pending first endpoint: also Roman-I, but no distance yet.
    pending = getattr(self, '_lpa_measure_first', None)
    if pending is not None:
        got = _t7v2_point_on_profile(self, int(pending['line']), float(pending['x']))
        if got is not None:
            x, y, _, _ = got
            mid = 2 * int(pending['line']) + 1
            pt = ax.text(x, y, 'I', ha='center', va='center', fontsize=15,
                         fontweight='bold', color='black', zorder=30,
                         bbox=dict(boxstyle='round,pad=0.08', facecolor='white',
                                   edgecolor='black', alpha=0.95))
            self._lpa_measure_artists.append(pt)
            lb = ax.text(x, y, f' M{mid} ', ha='center', va='bottom', fontsize=8,
                         fontweight='bold', zorder=31,
                         bbox=dict(boxstyle='round,pad=0.12', facecolor='white',
                                   edgecolor='black', alpha=0.9))
            self._lpa_measure_artists.append(lb)

    self._lpa_plot_canvas.draw_idle()
    self._lpa_canvas.draw_idle()


def _t7v2_measure_update_table(self):
    tree = getattr(self, '_lpa_measure_tree', None)
    if tree is None:
        return
    for iid in tree.get_children():
        tree.delete(iid)
    for m in sorted(getattr(self, '_lpa_measurements', []) or [], key=lambda x: int(x.get('line', -1))):
        line_idx = int(m.get('line', -1))
        tree.insert('', 'end', values=(
            f'M{2*line_idx+1}-M{2*line_idx+2}',
            line_idx + 1,
            f"{float(m.get('m1_nm', 0.0)):.6g}",
            f"{float(m.get('m2_nm', 0.0)):.6g}",
            f"{float(m.get('delta_nm', 0.0)):.6g}",
            f"{float(m.get('m1_intensity', 0.0)):.6g}",
            f"{float(m.get('m2_intensity', 0.0)):.6g}",
        ))


def _t7v2_table_export(self):
    measurements = getattr(self, '_lpa_measurements', []) or []
    if not measurements:
        messagebox.showwarning('Table Export', 'No measured values are available. Use MEASURE first.', parent=self.root)
        return
    try:
        initial = self._export_initialdir()
    except Exception:
        initial = os.path.expanduser('~')
    fp = filedialog.asksaveasfilename(
        parent=self.root,
        initialdir=initial,
        title='Export Tab-7 Measurement Table',
        defaultextension='.csv',
        filetypes=[('CSV', '*.csv'), ('All files', '*.*')]
    )
    if not fp:
        return
    rows = []
    try:
        frame_idx = int(self._lpa_frame_var.get())
    except Exception:
        frame_idx = 0
    field = ''
    try:
        if self.real_fields_mT is not None and frame_idx < len(self.real_fields_mT):
            field = float(self.real_fields_mT[frame_idx])
    except Exception:
        pass
    for m in sorted(measurements, key=lambda x: int(x.get('line', -1))):
        li = int(m.get('line', -1))
        rows.append({
            'Frame': frame_idx + 1,
            'Field_mT': field,
            'Line': li + 1,
            'Endpoint_1': f'M{2*li+1}',
            'Endpoint_2': f'M{2*li+2}',
            'M1_Position_nm': float(m.get('m1_nm', np.nan)),
            'M2_Position_nm': float(m.get('m2_nm', np.nan)),
            'Distance_nm': float(m.get('delta_nm', np.nan)),
            'M1_Intensity': float(m.get('m1_intensity', np.nan)),
            'M2_Intensity': float(m.get('m2_intensity', np.nan)),
        })
    pd.DataFrame(rows).to_csv(fp, index=False)
    self._lpa_status_var.set(f'Table exported: {fp}')


def _t7v2_nice_scale_length(span):
    span = float(span)
    if not np.isfinite(span) or span <= 0:
        return 1.0
    target = span / 5.0
    power = 10.0 ** np.floor(np.log10(target))
    for m in (1.0, 2.0, 5.0, 10.0):
        v = m * power
        if v >= target:
            return v
    return 10.0 * power


def _t7v2_add_scale_bar(ax, width_nm, height_nm):
    L = _t7v2_nice_scale_length(width_nm)
    x0 = 0.06 * width_nm
    x1 = x0 + L
    y = 0.06 * height_nm
    ax.plot([x0, x1], [y, y], color='white', lw=5, solid_capstyle='butt', zorder=100)
    ax.plot([x0, x1], [y, y], color='black', lw=2, solid_capstyle='butt', zorder=101)
    if L >= 100:
        label = f'{L:g} nm'
    elif L >= 1:
        label = f'{L:g} nm'
    else:
        label = f'{L:.3g} nm'
    ax.text((x0+x1)/2, y + 0.035*height_nm, label,
            color='white', fontsize=10, fontweight='bold', ha='center', va='bottom',
            bbox=dict(boxstyle='round,pad=0.16', facecolor='black', edgecolor='white', alpha=0.75), zorder=102)


def _t7v2_export_plot(self):
    stack = getattr(self, '_lpa_stack', None)
    lines = getattr(self, '_lpa_lines', []) or []
    if stack is None or len(stack) == 0:
        messagebox.showwarning('Plot Export', 'Fetch confirmed ROI2 from Tab 6 first.', parent=self.root)
        return
    if not lines:
        messagebox.showwarning('Plot Export', 'Draw at least one line before exporting.', parent=self.root)
        return
    try:
        idx = max(0, min(int(self._lpa_frame_var.get()), len(stack) - 1))
    except Exception:
        idx = 0
    try:
        initial = self._export_initialdir()
    except Exception:
        initial = os.path.expanduser('~')
    folder = filedialog.askdirectory(parent=self.root, initialdir=initial, title='Select Tab-7 Plot Export Folder')
    if not folder:
        return
    folder = os.path.abspath(folder)
    os.makedirs(folder, exist_ok=True)

    image = np.asarray(stack[idx], dtype=float)
    h, w = image.shape[:2]
    sx, sy = _lpa_scale_nm(self)
    width_nm = float(w * sx)
    height_nm = float(h * sy)
    cmap = 'RdBu_r'
    try:
        c = str(self.colormap_var.get()).strip()
        if c:
            cmap = c
    except Exception:
        pass

    field = None
    try:
        if self.real_fields_mT is not None and idx < len(self.real_fields_mT):
            field = float(self.real_fields_mT[idx])
    except Exception:
        pass
    field_text = f' | Field = {field:+.2f} mT' if field is not None else ''

    # --- ROI2 export ---
    roi_fig = plt.figure(figsize=(8.6, 7.4), dpi=300, facecolor='white')
    roi_ax = roi_fig.add_axes([0.09, 0.10, 0.77, 0.82])
    roi_cax = roi_fig.add_axes([0.89, 0.10, 0.035, 0.82])
    roi_im = roi_ax.imshow(image, origin='lower', interpolation='nearest',
                           cmap=cmap, vmin=0.0, vmax=1.0,
                           extent=(0.0, width_nm, 0.0, height_nm), aspect='equal')
    roi_ax.set_xlabel('X (nm)')
    roi_ax.set_ylabel('Y (nm)')
    roi_ax.set_title(f'Confirmed ROI — Frame {idx+1}/{len(stack)}{field_text}', fontsize=10)
    cb = roi_fig.colorbar(roi_im, cax=roi_cax)
    cb.set_label('Intensity')
    _t7v2_add_scale_bar(roi_ax, width_nm, height_nm)

    colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', ['C0','C1','C2','C3','C4','C5','C6','C7'])
    for j, (p0, p1) in enumerate(lines):
        p0 = np.asarray(p0, dtype=float); p1 = np.asarray(p1, dtype=float)
        roi_ax.plot([p0[0], p1[0]], [p0[1], p1[1]], lw=2.5,
                    color=colors[j % len(colors)], zorder=50, label=f'Line {j+1}')

    for m in getattr(self, '_lpa_measurements', []) or []:
        li = int(m.get('line', -1))
        if li < 0 or li >= len(lines):
            continue
        p0 = np.asarray(lines[li][0], dtype=float); p1 = np.asarray(lines[li][1], dtype=float)
        got = _t7v2_line_profile(self, li)
        if got is None:
            continue
        d, prof = got; total = float(d[-1])
        if total <= 0: continue
        q0 = p0 + (float(m['m1_nm']) / total) * (p1 - p0)
        q1 = p0 + (float(m['m2_nm']) / total) * (p1 - p0)
        roi_ax.plot([q0[0], q1[0]], [q0[1], q1[1]], color='yellow', lw=3.0, zorder=70)
        for q, mid in ((q0, 2*li+1), (q1, 2*li+2)):
            roi_ax.text(q[0], q[1], 'I', color='black', fontsize=15, fontweight='bold',
                        ha='center', va='center', zorder=72,
                        bbox=dict(boxstyle='round,pad=0.08', facecolor='white', edgecolor='black', alpha=0.95))
            roi_ax.text(q[0], q[1], f'M{mid}', color='black', fontsize=8, fontweight='bold',
                        ha='center', va='bottom', zorder=73,
                        bbox=dict(boxstyle='round,pad=0.10', facecolor='white', edgecolor='black', alpha=0.9))
    try:
        roi_ax.legend(loc='best', fontsize=8, framealpha=0.9)
    except Exception:
        pass
    roi_path = os.path.join(folder, f'Tab7_ROI2_Frame_{idx+1:03d}.png')
    roi_fig.savefig(roi_path, dpi=300, bbox_inches='tight', pad_inches=0.08, facecolor='white')
    plt.close(roi_fig)

    # --- line-profile export ---
    plot_fig = plt.figure(figsize=(8.6, 6.8), dpi=300, facecolor='white')
    plot_ax = plot_fig.add_axes([0.10, 0.10, 0.83, 0.82])
    plot_ax.set_xlabel('Line Length (nm)')
    plot_ax.set_ylabel('Intensity')
    plot_ax.grid(True, alpha=0.25)
    dmax = 0.0; ymax = 0.0
    for j, (p0, p1) in enumerate(lines):
        got = _t7v2_line_profile(self, j)
        if got is None: continue
        d, prof = got
        plot_ax.plot(d, prof, lw=2.0, color=colors[j % len(colors)], label=f'Line {j+1}')
        if d.size: dmax = max(dmax, float(d[-1]))
        if prof.size and np.any(np.isfinite(prof)): ymax = max(ymax, float(np.nanmax(prof)))
    if dmax > 0: plot_ax.set_xlim(0.0, max(dmax*1.05, 1e-9))
    if ymax > 0: plot_ax.set_ylim(0.0, max(ymax*1.10, 1e-9))

    for m in getattr(self, '_lpa_measurements', []) or []:
        li = int(m.get('line', -1))
        if li < 0: continue
        x0 = float(m.get('m1_nm', 0.0)); x1 = float(m.get('m2_nm', 0.0))
        got = _t7v2_point_on_profile(self, li, x0); got2 = _t7v2_point_on_profile(self, li, x1)
        if got is None or got2 is None: continue
        _, y0, _, _ = got; _, y1, _, _ = got2
        plot_ax.plot([x0, x1], [y0, y1], '-', color='black', lw=2.0, zorder=20)
        for x, y, mid in ((x0, y0, 2*li+1), (x1, y1, 2*li+2)):
            plot_ax.text(x, y, 'I', color='black', fontsize=15, fontweight='bold',
                         ha='center', va='center', zorder=30,
                         bbox=dict(boxstyle='round,pad=0.08', facecolor='white', edgecolor='black', alpha=0.95))
            plot_ax.text(x, y, f'M{mid}', fontsize=8, fontweight='bold', ha='center', va='bottom', zorder=31,
                         bbox=dict(boxstyle='round,pad=0.12', facecolor='white', edgecolor='black', alpha=0.9))
        ylo, yhi = plot_ax.get_ylim(); ybar = yhi - 0.10*max(yhi-ylo, 1e-12)
        plot_ax.annotate('', xy=(x1, ybar), xytext=(x0, ybar),
                         arrowprops=dict(arrowstyle='<->', color='blue', lw=1.8), zorder=24)
        plot_ax.text((x0+x1)/2, ybar, f'ΔL = {abs(x1-x0):.6g} nm',
                     ha='center', va='bottom', fontsize=8.5, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='blue', alpha=0.9), zorder=25)
    try:
        plot_ax.legend(loc='best', fontsize=8, framealpha=0.9)
    except Exception:
        pass
    plot_ax.set_title(f'Tab-7 Line Profiles — Frame {idx+1}/{len(stack)}{field_text}', fontsize=12)
    plot_path = os.path.join(folder, f'Tab7_LineProfiles_Frame_{idx+1:03d}.png')
    plot_fig.savefig(plot_path, dpi=300, bbox_inches='tight', pad_inches=0.08, facecolor='white')
    plt.close(plot_fig)

    self._lpa_status_var.set(f'Plot export complete: {roi_path} and {plot_path}')
    try:
        _tab6_open_output_folder(folder)
    except Exception:
        try:
            if os.name == 'nt': os.startfile(folder)
            else: subprocess.Popen(['xdg-open' if sys.platform != 'darwin' else 'open', folder])
        except Exception:
            pass


# Replace the old LPA measurement handlers with the final multi-line version.
_lpa_measure_start = _t7v2_measure_start
_lpa_measure_graph_press = _t7v2_measure_graph_press
_lpa_measure_graph_motion = _t7v2_measure_graph_motion
_lpa_measure_graph_release = _t7v2_measure_graph_release
_lpa_measure_redraw = _t7v2_measure_redraw
_lpa_measure_update_table = _t7v2_measure_update_table
CDIWorkflowApp._lpa_measure_start = _t7v2_measure_start
CDIWorkflowApp._lpa_measure_graph_press = _t7v2_measure_graph_press
CDIWorkflowApp._lpa_measure_graph_motion = _t7v2_measure_graph_motion
CDIWorkflowApp._lpa_measure_graph_release = _t7v2_measure_graph_release
CDIWorkflowApp._lpa_measure_redraw = _t7v2_measure_redraw
CDIWorkflowApp._lpa_measure_update_table = _t7v2_measure_update_table
CDIWorkflowApp._lpa_measure_clear_artists = _t7v2_measure_clear_artists
CDIWorkflowApp._tab7_table_export = _t7v2_table_export
CDIWorkflowApp._tab7_plot_export = _t7v2_export_plot

# Make CLEAR LINES also reset the v2 measurement state and table.
_old_lpa_clear_lines_v2 = _lpa_clear_lines

def _lpa_clear_lines_v2(self):
    try:
        _old_lpa_clear_lines_v2(self)
    except Exception:
        self._lpa_lines = []
    self._lpa_measurements = []
    self._lpa_measure_mode = False
    self._lpa_measure_first = None
    self._lpa_measure_drag = None
    try:
        _t7v2_measure_clear_artists(self)
    except Exception:
        pass
    try:
        _t7v2_measure_update_table(self)
    except Exception:
        pass
    try:
        self._lpa_status_var.set('ROI2 loaded. All drawn lines, profile plots and measurements cleared.')
    except Exception:
        pass
_lpa_clear_lines = _lpa_clear_lines_v2
CDIWorkflowApp._lpa_clear_lines = _lpa_clear_lines_v2

# Rebuild the Tab-7 controls after the original builder has created them.
_LPA_BUILD_ORIGINAL_V2 = _build_line_profile_analysis_tab

def _build_line_profile_analysis_tab_v2(self):
    result = _LPA_BUILD_ORIGINAL_V2(self)
    try:
        outer = self.tab_line_profile_analysis.winfo_children()[0]
        # Existing controls row is the second child of the outer frame.
        children = list(outer.winfo_children())
        controls = None
        for w in children:
            if isinstance(w, ttk.Frame) and any(isinstance(c, (ttk.Button, tk.Button)) for c in w.winfo_children()):
                controls = w
                break
        if controls is None:
            controls = ttk.Frame(outer)
            controls.pack(fill='x', pady=(0, 4), before=outer.winfo_children()[-1] if outer.winfo_children() else None)
        ttk.Button(controls, text='TABLE EXPORT', command=lambda: _t7v2_table_export(self), width=13).pack(side='left', padx=5)
        ttk.Button(controls, text='PLOT EXPORT', command=lambda: _t7v2_export_plot(self), width=13).pack(side='left', padx=5)
        self._lpa_measurements = []
        self._lpa_measure_mode = False
        self._lpa_measure_first = None
        self._lpa_measure_drag = None
    except Exception as exc:
        try: self._lpa_status_var.set(f'Tab-7 export controls initialization warning: {exc}')
        except Exception: pass
    return result

_build_line_profile_analysis_tab = _build_line_profile_analysis_tab_v2

# Rebind the v2 measurement mouse events after Tab-7 widgets are created.
_old_ui_builder_v2 = CDIWorkflowApp._build_ui

def _build_ui_v2(self, *args, **kwargs):
    result = _old_ui_builder_v2(self, *args, **kwargs)
    try:
        _t7v2_init_measure_state(self)
        canvas = self._lpa_plot_canvas
        for signal in ('button_press_event', 'motion_notify_event', 'button_release_event'):
            registry = getattr(canvas.callbacks, 'callbacks', {}).get(signal, {})
            for cid in list(registry.keys()):
                try: canvas.mpl_disconnect(cid)
                except Exception: pass
        canvas.mpl_connect('button_press_event', lambda e: _t7v2_measure_graph_press(self, e))
        canvas.mpl_connect('motion_notify_event', lambda e: _t7v2_measure_graph_motion(self, e))
        canvas.mpl_connect('button_release_event', lambda e: _t7v2_measure_graph_release(self, e))
        self._lpa_measure_tree.bind('<<TreeviewSelect>>', lambda e: None)
    except Exception:
        pass
    return result

CDIWorkflowApp._build_ui = _build_ui_v2


# ============================================================================
# FINAL USER-UI FIX — TAB 6 PRIMARY ROI AXES + SECONDARY ROI DRAG + TAB 7 DRAW
# ============================================================================
# This layer intentionally touches only the existing Tab-6/Tab-7 UI hookups.
# It does not replace the processing or ROI data model.

def _final_tab6_apply_nm_primary_axes(self):
    """Force the currently displayed Tab-6 Primary ROI to use physical nm axes."""
    try:
        ax = getattr(self, '_proc_ax', None)
        im = getattr(self, '_proc_image_display', None)
        if ax is None or im is None:
            return
        arr = im.get_array()
        if arr is None:
            return
        h, w = np.asarray(arr).shape[:2]
        base = None
        try:
            base = self._proc_get_tab5_base_pixel_size()
        except Exception:
            base = getattr(self, 'primary_roi_final_nm_per_pixel', None)
        if base is None or not np.isfinite(float(base)) or float(base) <= 0:
            return
        ow = int(getattr(self, 'original_roi_w', 0) or 0)
        oh = int(getattr(self, 'original_roi_h', 0) or 0)
        if ow <= 0:
            ow = int(w)
        if oh <= 0:
            oh = int(h)
        pw = float(ow) * float(base)
        ph = float(oh) * float(base)
        im.set_extent((0.0, pw, 0.0, ph))
        ax.set_axis_on()
        ax.set_xlabel('X (nm)')
        ax.set_ylabel('Y (nm)')
        ax.set_aspect('equal', adjustable='box')
        ax.set_anchor('C')
        ax.set_autoscale_on(False)
        if not getattr(self, '_proc_zoomed', False):
            ax.set_xlim(0.0, pw)
            ax.set_ylim(0.0, ph)
        try:
            cb = getattr(self, '_proc_colorbar', None)
            if cb is not None:
                cb.update_normal(im)
                cb.set_label('Intensity')
        except Exception:
            pass
        self._proc_full_physical_extent = (0.0, pw, 0.0, ph)
    except Exception:
        pass

_OLD_FINAL_PROC_UPDATE_FOR_AXES = CDIWorkflowApp._proc_update

def _final_proc_update_with_nm_axes(self, *args, **kwargs):
    result = _OLD_FINAL_PROC_UPDATE_FOR_AXES(self, *args, **kwargs)
    _final_tab6_apply_nm_primary_axes(self)
    try:
        self._proc_canvas.draw_idle()
    except Exception:
        pass
    return result

CDIWorkflowApp._proc_update = _final_proc_update_with_nm_axes
CDIWorkflowApp.update_processing_view = _final_proc_update_with_nm_axes


def _final_tab6_secondary_press_nm(self, event):
    """Start a Secondary ROI drag in the physical-nm coordinate system."""
    if not getattr(self, '_proc_secondary_roi_selecting', False):
        return
    if event is None or event.inaxes is not getattr(self, '_proc_ax', None):
        return
    if getattr(event, 'button', None) != 1 or event.xdata is None or event.ydata is None:
        return
    self._proc_secondary_press_xy = (float(event.xdata), float(event.ydata))
    if getattr(self, '_proc_secondary_patch', None) is not None:
        try:
            self._proc_secondary_patch.remove()
        except Exception:
            pass
        self._proc_secondary_patch = None
    try:
        self.proc_secondary_status_var.set(
            'DRAWING SECONDARY ROI: release mouse to finish the rectangle.'
        )
    except Exception:
        pass


def _final_disconnect_event(canvas, signal):
    try:
        registry = getattr(canvas.callbacks, 'callbacks', {}).get(signal, {})
        for cid in list(registry.keys()):
            try:
                canvas.mpl_disconnect(cid)
            except Exception:
                pass
    except Exception:
        pass


def _final_rebind_tab6_callbacks(self):
    canvas = getattr(self, '_proc_canvas', None)
    if canvas is None:
        return
    for sig in ('button_press_event', 'motion_notify_event', 'button_release_event'):
        _final_disconnect_event(canvas, sig)
    canvas.mpl_connect('button_press_event', self._final_tab6_secondary_press_nm)
    canvas.mpl_connect('motion_notify_event', lambda e: _proc_secondary_on_move_nm_safe(self, e))
    canvas.mpl_connect('button_release_event', lambda e: _proc_secondary_on_release_nm_safe(self, e))
    # Keep the existing Tab-6 profile-measurement functionality available.
    canvas.mpl_connect('button_press_event', self._proc_profile_on_press)
    canvas.mpl_connect('motion_notify_event', self._proc_profile_on_move)
    canvas.mpl_connect('button_release_event', self._proc_profile_on_release)

CDIWorkflowApp._final_tab6_secondary_press_nm = _final_tab6_secondary_press_nm


def _final_lpa_force_draw_bindings(self):
    """Rebind Tab-7 ROI2 drawing so DRAW LINE always uses the current image axes."""
    canvas = getattr(self, '_lpa_canvas', None)
    ax = getattr(self, '_lpa_ax', None)
    if canvas is None or ax is None:
        return
    for sig in ('button_press_event', 'motion_notify_event', 'button_release_event'):
        _final_disconnect_event(canvas, sig)
    canvas.mpl_connect('button_press_event', lambda e: _lpa_line_press(self, e))
    canvas.mpl_connect('motion_notify_event', lambda e: _lpa_line_motion(self, e))
    canvas.mpl_connect('button_release_event', lambda e: _lpa_line_release(self, e))
    try:
        self._lpa_status_var.set(
            'ROI2 ready. Set No. of lines, click DRAW LINE, then left-drag on ROI2.'
        )
    except Exception:
        pass


def _final_rebind_tab7_measure_bindings(self):
    canvas = getattr(self, '_lpa_plot_canvas', None)
    if canvas is None:
        return
    for sig in ('button_press_event', 'motion_notify_event', 'button_release_event'):
        _final_disconnect_event(canvas, sig)
    canvas.mpl_connect('button_press_event', self._lpa_measure_graph_press)
    canvas.mpl_connect('motion_notify_event', self._lpa_measure_graph_motion)
    canvas.mpl_connect('button_release_event', self._lpa_measure_graph_release)


# ============================================================================


# ============================================================================
# TAB 8 — ROBUST REBUILD (V7)
# Skyrmion / Particle — Outer Particle + Inner Core + Core Symmetry
# ============================================================================
# Scope:
#   * Tabs 1–7 and all later tabs are left untouched.
#   * Tab 8 consumes the exact processed/finalized Primary ROI1 from Tab 6.
#   * Tab-6 X/Y calibration is authoritative; all Tab-8 image axes are in nm.
#   * Frame slider is one-based in the UI, always enabled when source data exist,
#     and responds directly to mouse dragging.
#   * PLAY / PAUSE / FIRST / PREVIOUS / NEXT provide deterministic frame control.
#   * ROI3 is selected on the Primary ROI1 image, then explicitly confirmed.
#   * Detection, manual ADD BLOB, selection, delete, undo, CLEAR MARK and RESET
#     are isolated to Tab 8.
#   * Analysis results are retained by acquisition/frame number and plotted only
#     for frames that were actually analyzed.
# ============================================================================

def _tab8_v7_get_scale(self):
    """Return the exact physical X/Y scale used by Tab 6/Fig. 2.

    Priority is the physical extent actually applied to the Tab-6 Primary ROI
    display. This prevents Tab 8 from falling back to the resized output-pixel
    calibration and therefore showing a compressed ~0-2400 nm FOV when Fig. 2
    represents the full ~0-13500 nm Primary ROI.
    """
    # 1) Authoritative extent generated by Tab 6 physical-axis update.
    try:
        ext = getattr(self, '_proc_full_physical_extent', None)
        arr = getattr(self, '_proc_image_display', None)
        if ext is not None and arr is not None and arr.get_array() is not None:
            a = np.asarray(arr.get_array())
            h, w = a.shape[:2]
            x0, x1, y0, y1 = map(float, ext)
            if w > 0 and h > 0 and x1 > x0 and y1 > y0:
                return ((x1-x0)/float(w), (y1-y0)/float(h))
    except Exception:
        pass

    # 2) Read the actual extent from the Tab-6 image artist if available.
    try:
        im = getattr(self, '_proc_image_display', None)
        if im is not None and im.get_array() is not None:
            a = np.asarray(im.get_array())
            h, w = a.shape[:2]
            x0, x1, y0, y1 = map(float, im.get_extent())
            if w > 0 and h > 0 and x1 > x0 and y1 > y0:
                return ((x1-x0)/float(w), (y1-y0)/float(h))
    except Exception:
        pass

    # 3) Existing calibrated physical ROI dimensions as fallback.
    try:
        base = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
        ow = int(getattr(self, 'original_roi_w', 0) or 0)
        oh = int(getattr(self, 'original_roi_h', 0) or 0)
        rw, rh = getattr(self, 'roi_resolution', (0, 0))
        rw, rh = int(rw), int(rh)
        if np.isfinite(base) and base > 0 and ow > 0 and oh > 0 and rw > 0 and rh > 0:
            return (base*float(ow)/float(rw), base*float(oh)/float(rh))
    except Exception:
        pass

    try:
        v = float(getattr(self, 'primary_roi_final_nm_per_pixel'))
        if np.isfinite(v) and v > 0:
            return (v, v)
    except Exception:
        pass
    return None, None

def _tab8_v7_get_field(self, idx):
    """Return the field value for a frame without modifying any other tab."""
    for name in ('_tab7_v3_field', '_tab7_field_value'):
        try:
            fn = globals().get(name)
            if callable(fn):
                val = float(fn(self, int(idx)))
                if np.isfinite(val):
                    return val
        except Exception:
            pass
    for attr in ('real_fields_mT', 'calib_field_mT'):
        try:
            arr = getattr(self, attr, None)
            if arr is not None and len(arr) > int(idx):
                val = float(arr[int(idx)])
                if np.isfinite(val):
                    return val
        except Exception:
            pass
    try:
        arr = getattr(self, 'raw_fields', None)
        if arr is not None and len(arr) > int(idx):
            return float(arr[int(idx)])
    except Exception:
        pass
    return float('nan')


def _tab8_v7_get_primary_stack(self):
    stack = getattr(self, '_tab8_primary_stack', None)
    try:
        a = np.asarray(stack)
        if a.ndim == 3 and a.shape[0] > 0 and a.shape[1] > 1 and a.shape[2] > 1:
            return a
    except Exception:
        pass
    return None


def _tab8_v7_get_roi3_stack(self):
    stack = getattr(self, '_tab8_roi3_stack', None)
    try:
        a = np.asarray(stack)
        if a.ndim == 3 and a.shape[0] > 0 and a.shape[1] > 1 and a.shape[2] > 1:
            return a
    except Exception:
        pass
    return None


def _tab8_v7_primary_extent(self):
    st = _tab8_v7_get_primary_stack(self)
    sx, sy = getattr(self, '_tab8_fixed_scale', (None, None))
    if st is None or sx is None or sy is None:
        return None
    h, w = st.shape[1:3]
    return (0.0, float(w) * float(sx), 0.0, float(h) * float(sy))


def _tab8_v7_roi3_extent(self):
    st = _tab8_v7_get_roi3_stack(self)
    sx, sy = getattr(self, '_tab8_fixed_scale', (None, None))
    if st is None or sx is None or sy is None:
        return None
    h, w = st.shape[1:3]
    return (0.0, float(w) * float(sx), 0.0, float(h) * float(sy))


def _tab8_v7_equiv_scale(self):
    sx, sy = _tab8_v7_get_scale(self)
    if sx is None or sy is None:
        return None
    return float(np.sqrt(sx * sy))


def _tab8_v7_clear_artists(self, attr):
    for artist in list(getattr(self, attr, []) or []):
        try:
            artist.remove()
        except Exception:
            pass
    setattr(self, attr, [])


def _tab8_v7_stop_playback(self, update_button=True):
    """Cancel only Tab-8's scheduled playback callback."""
    self._tab8_playing = False
    job = getattr(self, '_tab8_play_job', None)
    if job is not None:
        try:
            self.root.after_cancel(job)
        except Exception:
            pass
    self._tab8_play_job = None
    if update_button:
        try:
            self.pd_play_button.configure(text='PLAY')
        except Exception:
            pass


def _tab8_v7_set_frame(self, idx, user_action=False):
    """Set a valid frame index and refresh only Tab-8."""
    st = _tab8_v7_get_primary_stack(self)
    if st is None:
        return False
    n = int(st.shape[0])
    idx = max(0, min(int(idx), n - 1))
    try:
        self._tab8_programmatic = True
        self._tab8_frame_var.set(idx)
        self._tab8_frame_slider.set(idx)
    finally:
        self._tab8_programmatic = False
    _tab8_v7_frame_changed(self, idx)
    return True


def _tab8_v7_frame_changed(self, value=None):
    """Authoritative manual/playback frame callback."""
    st = _tab8_v7_get_primary_stack(self)
    if st is None:
        try:
            self._tab8_frame_label_var.set('Frame --/-- | Field = --')
            self._tab8_frame_slider.configure(from_=0, to=0, state='disabled')
        except Exception:
            pass
        return False

    n = int(st.shape[0])
    try:
        idx = int(round(float(value))) if value is not None else int(self._tab8_frame_var.get())
    except Exception:
        idx = int(getattr(self, 'pd_frame_var', tk.IntVar(value=0)).get())
    idx = max(0, min(idx, n - 1))

    try:
        self._tab8_programmatic = True
        self._tab8_frame_var.set(idx)
    finally:
        self._tab8_programmatic = False

    # Load only the detections belonging to this frame. Do not run analysis
    # merely because the user moved the slider.
    self.pd_current_detections = [
        dict(d) for d in (getattr(self, 'pd_frame_detections', {}) or {}).get(idx, [])
    ]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()

    _tab8_v7_refresh_image(self)

    try:
        field = _tab8_v7_get_field(self, idx)
        ftxt = f'{field:+.2f} mT' if np.isfinite(field) else 'Field = --'
        self._tab8_frame_label_var.set(f'Frame {idx + 1}/{n} | {ftxt}')
    except Exception:
        pass

    return True


def _tab8_v7_refresh_image(self):
    """Refresh the two image panels while preserving fixed physical FOV."""
    primary = _tab8_v7_get_primary_stack(self)
    if primary is None:
        return False

    idx = max(0, min(int(self._tab8_frame_var.get()), primary.shape[0] - 1))
    pimg = np.asarray(primary[idx], dtype=float)

    sx, sy = getattr(self, '_tab8_fixed_scale', (None, None))
    if sx is None or sy is None:
        sx, sy = _tab8_v7_get_scale(self)
    if sx is None or sy is None:
        return False

    h, w = pimg.shape[:2]
    # Match the exact physical FOV of Tab 6 / Fig. 2 whenever available.
    # Do not derive the primary-panel extent from the resized output dimensions.
    pext = None
    try:
        tab6_ext = getattr(self, '_proc_full_physical_extent', None)
        if tab6_ext is not None:
            tx0, tx1, ty0, ty1 = map(float, tab6_ext)
            if tx1 > tx0 and ty1 > ty0:
                pext = (tx0, tx1, ty0, ty1)
    except Exception:
        pext = None
    if pext is None:
        pext = (0.0, w * float(sx), 0.0, h * float(sy))
    self._tab8_primary_xy_extent = pext

    try:
        self.pd_image_artist_gray.set_data(pimg)
        self.pd_image_artist_gray.set_extent(pext)
        try:
            self.pd_image_artist_gray.set_cmap(str(self.colormap_var.get() or 'gray'))
        except Exception:
            pass
        self.pd_ax_gray.set_autoscale_on(False)
        self.pd_ax_gray.set_xlim(*pext[:2])
        self.pd_ax_gray.set_ylim(*pext[2:])
        self.pd_ax_gray.set_aspect('equal', adjustable='box')
        self.pd_ax_gray.set_xlabel('X (nm)')
        self.pd_ax_gray.set_ylabel('Y (nm)')
        self.pd_ax_gray.set_title('Finalized Primary ROI', fontsize=10)
    except Exception:
        pass

    roi3 = _tab8_v7_get_roi3_stack(self)
    if roi3 is not None:
        rimg = np.asarray(roi3[idx], dtype=float)

        # ROI3 is a cropped analysis image. Its pixel coordinates (including
        # detections and Gaussian-fit centers) are LOCAL to the ROI3 crop.
        # Therefore keep ROI3's origin at (0, 0), but use the SAME physical
        # nm/pixel calibration as Tab 6 / Primary ROI1.
        #
        # Do NOT use the ROI3 location inside Primary ROI1 as the ROI3 axis
        # origin; that would make the crop appear at e.g. X=400..1600 nm and
        # would put locally stored particle labels outside the image.
        rext = (
            0.0,
            float(rimg.shape[1]) * float(sx),
            0.0,
            float(rimg.shape[0]) * float(sy),
        )
        self._tab8_roi3_xy_extent = rext
        try:
            self.pd_image_artist_color.set_data(rimg)
            self.pd_image_artist_color.set_extent(rext)
            try:
                self.pd_image_artist_color.set_cmap(str(self.colormap_var.get() or 'viridis'))
            except Exception:
                pass
            self.pd_ax_color.set_autoscale_on(False)
            self.pd_ax_color.set_xlim(rext[0], rext[1])
            self.pd_ax_color.set_ylim(rext[2], rext[3])
            self.pd_ax_color.set_aspect('equal', adjustable='box')
            self.pd_ax_color.set_xlabel('X (nm)')
            self.pd_ax_color.set_ylabel('Y (nm)')
            self.pd_ax_color.set_title('ROI — ANALYSIS', fontsize=10)
            self.pd_colorbar.update_normal(self.pd_image_artist_color)
            self.pd_colorbar.set_label('Intensity', fontsize=8)
        except Exception:
            pass
    else:
        self._tab8_roi3_xy_extent = None
        try:
            self.pd_image_artist_color.set_data(pimg)
            self.pd_image_artist_color.set_extent(pext)
            self.pd_ax_color.set_autoscale_on(False)
            self.pd_ax_color.set_xlim(*pext[:2])
            self.pd_ax_color.set_ylim(*pext[2:])
            self.pd_ax_color.set_aspect('equal', adjustable='box')
            self.pd_ax_color.set_xlabel('X (nm)')
            self.pd_ax_color.set_ylabel('Y (nm)')
            self.pd_ax_color.set_title('Primary ROI — select ROI', fontsize=9)
            self.pd_colorbar.update_normal(self.pd_image_artist_color)
            self.pd_colorbar.set_label('Intensity', fontsize=8)
        except Exception:
            pass

    _tab8_v7_draw_overlays(self)

    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass
    return True


def _tab8_v7_draw_overlays(self):
    """Redraw only Tab-8 overlays in the current physical coordinate system."""
    _tab8_v7_clear_artists(self, 'pd_geom_artists')
    _tab8_v7_clear_artists(self, 'pd_label_marker_artists')

    try:
        patch = getattr(self, '_tab8_roi3_patch', None)
        if patch is not None:
            patch.remove()
    except Exception:
        pass
    self._tab8_roi3_patch = None

    pext = _tab8_v7_primary_extent(self)
    sx, sy = getattr(self, '_tab8_fixed_scale', (None, None))
    if pext is None or sx is None or sy is None:
        return

    # ROI3 rectangle is always shown on the Primary ROI1 image.
    c = getattr(self, '_tab8_roi3_coords', None)
    if isinstance(c, dict):
        try:
            x0 = int(c['xmin']) * sx
            x1 = (int(c['xmax']) + 1) * sx
            y0 = int(c['ymin']) * sy
            y1 = (int(c['ymax']) + 1) * sy
            edge = 'lime' if getattr(self, '_tab8_roi3_confirmed', False) else 'red'
            self._tab8_roi3_patch = plt.Rectangle(
                (x0, y0), max(1e-12, x1 - x0), max(1e-12, y1 - y0),
                fill=False, edgecolor=edge, linewidth=2.4, zorder=30
            )
            self.pd_ax_gray.add_patch(self._tab8_roi3_patch)
        except Exception:
            self._tab8_roi3_patch = None

    roi3 = _tab8_v7_get_roi3_stack(self)
    if roi3 is None or getattr(self, '_tab8_marks_cleared', False):
        try:
            self.pd_canvas.draw_idle()
        except Exception:
            pass
        return

    dets = [
        dict(d) for d in (getattr(self, 'pd_current_detections', []) or [])
        if d.get('accepted', True)
    ]
    selected = set(getattr(self, 'pd_multi_selected', set()) or set())
    theta = np.linspace(0, 2 * np.pi, 180)

    for k, d in enumerate(dets):
        try:
            ox = float(d.get('outer_x', d.get('x', 0.0)))
            oy = float(d.get('outer_y', d.get('y', 0.0)))
            cx = float(d.get('x', ox))
            cy = float(d.get('y', oy))
            od = max(0.0, float(d.get('diameter_px', 0.0)))
            cd = max(0.0, float(d.get('core_diameter_px', 0.0)))
            ar = max(1.0, float(d.get('core_aspect_ratio', 1.0)))
            outer, = self.pd_ax_color.plot(
                (ox + od / 2 * np.cos(theta)) * sx,
                (oy + od / 2 * np.sin(theta)) * sy,
                color='yellow', lw=2.8 if k in selected else 1.8, zorder=20
            )
            inner, = self.pd_ax_color.plot(
                (cx + cd * ar / 2 * np.cos(theta)) * sx,
                (cy + cd / 2 * np.sin(theta)) * sy,
                color='lime', lw=2.2 if k in selected else 1.4, zorder=21
            )
            self.pd_geom_artists.extend([outer, inner])
            if bool(self.pd_show_centers_var.get()):
                center, = self.pd_ax_color.plot(
                    [cx * sx], [cy * sy],
                    marker='o', ms=5.5, mfc='none', mec='lime', mew=1.6, zorder=22
                )
                label = self.pd_ax_color.text(
                    cx * sx, cy * sy, str(k + 1),
                    color='black', ha='center', va='center', fontsize=7,
                    bbox=dict(boxstyle='circle,pad=0.15', fc='yellow', ec='lime', lw=1.0),
                    zorder=23
                )
                self.pd_label_marker_artists.extend([center, label])
        except Exception:
            continue

    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass


def _tab8_v7_normalize(self, image):
    """Stable normalization for Tab-8 detection, using finite pixels only."""
    z = np.asarray(image, dtype=float)
    if z.ndim != 2:
        raise ValueError(f'Expected a 2-D image, got shape {z.shape}.')
    finite = z[np.isfinite(z)]
    if finite.size == 0:
        return np.zeros_like(z, dtype=float)
    lo, hi = np.percentile(finite, [1.0, 99.0])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = float(np.min(finite))
        hi = float(np.max(finite))
    if hi <= lo:
        return np.zeros_like(z, dtype=float)
    return np.clip((z - lo) / (hi - lo), 0.0, 1.0)


def _tab8_v7_run_detection(self, image, frame_idx=None):
    """
    Detect outer particles first, then determine the inner core from each
    accepted outer component. All size thresholds remain in pixels internally;
    reported sizes are converted to nm using the fixed Tab-6 scale.
    """
    from scipy import ndimage as ndi
    from skimage.measure import regionprops
    from skimage.morphology import remove_small_objects, opening, closing, disk

    z = _tab8_v7_normalize(self, image)
    h, w = z.shape

    def gv(name, default):
        try:
            return float(getattr(self, name).get())
        except Exception:
            return float(default)

    threshold = float(np.clip(gv('pd_threshold_var', 0.25), 0.01, 0.99))
    core_level = float(np.clip(gv('pd_core_contrast_var', 0.55), 0.05, 0.99))
    dmin = max(1.0, gv('pd_min_diam_var', 10.0))
    dmax = max(dmin, gv('pd_max_diam_var', 60.0))
    sep = max(0.0, gv('pd_min_sep_var', 25.0))
    amin = max(0.0, gv('pd_min_area_var', 0.0))
    amax = max(amin, gv('pd_max_area_var', 1e6))
    qmin = float(np.clip(gv('pd_min_circularity_var', 0.0), 0.0, 1.0))
    qmax = float(np.clip(gv('pd_max_circularity_var', 1.0), 0.0, 1.0))
    bg_sigma = max(1.0, gv('pd_background_sigma_var', 20.0))
    med_size = max(1, int(round(gv('pd_median_size_var', 3.0))))
    # Median-filter size must be odd for a symmetric neighbourhood.
    if med_size % 2 == 0:
        med_size += 1
    cmin = max(1.0, gv('pd_core_min_diam_var', 4.0))
    cmax = max(cmin, gv('pd_core_max_diam_var', 40.0))
    ar_max = max(1.0, gv('pd_max_aspect_var', 1.8))

    try:
        polarity = str(self.pd_polarity_var.get()).strip().lower()
    except Exception:
        polarity = 'bright'
    if polarity not in ('bright', 'dark', 'both'):
        polarity = 'bright'
    try:
        include_edge = bool(self.pd_include_edge_var.get())
    except Exception:
        include_edge = False

    # Preprocess once; use the same processed image for both polarity paths.
    sm = ndi.gaussian_filter(
        ndi.median_filter(z, size=med_size, mode='nearest'),
        sigma=1.0, mode='nearest'
    )
    bg = ndi.gaussian_filter(sm, sigma=bg_sigma, mode='nearest')
    detail = sm - bg

    candidates = []
    if polarity in ('bright', 'both'):
        candidates.append(('bright', detail))
    if polarity in ('dark', 'both'):
        candidates.append(('dark', -detail))

    all_candidates = []
    for pol, signal in candidates:
        positive = signal[signal > 0]
        scale = float(np.percentile(positive, 99.0)) if positive.size else 1.0
        scale = max(scale, 1e-12)
        q = np.clip(signal / scale, 0.0, 1.0)
        mask = q >= threshold

        try:
            min_obj = max(3, int(np.pi * (dmin / 2.0) ** 2 * 0.05))
            mask = remove_small_objects(mask, min_size=min_obj)
            mask = opening(mask, footprint=disk(1))
            mask = closing(mask, footprint=disk(1))
        except Exception:
            pass

        labels, nlab = ndi.label(mask)
        if nlab <= 0:
            continue

        for rp in regionprops(labels, intensity_image=q):
            try:
                eqd = float(rp.equivalent_diameter_area)
                area = float(rp.area)
                peri = float(rp.perimeter)
                circ = 4 * np.pi * area / max(peri ** 2, 1e-12)
                cy, cx = map(float, rp.centroid)
            except Exception:
                continue

            if not (dmin <= eqd <= dmax):
                continue
            if not (amin <= area <= amax):
                continue
            if not (qmin <= circ <= qmax):
                continue

            # Edge test uses the actual component extent rather than a guessed
            # image-centre radius.
            minr, minc, maxr, maxc = rp.bbox
            edge = minr <= 0 or minc <= 0 or maxr >= h or maxc >= w
            if edge and not include_edge:
                continue

            feret = float(getattr(rp, 'feret_diameter_max', eqd))
            if not np.isfinite(feret):
                feret = eqd

            # Inner-core analysis is localized to the outer component.
            rad = max(6, int(np.ceil(0.75 * eqd)))
            y0 = max(0, int(round(cy)) - rad)
            y1 = min(h, int(round(cy)) + rad + 1)
            x0 = max(0, int(round(cx)) - rad)
            x1 = min(w, int(round(cx)) + rad + 1)
            cand_mask = labels[y0:y1, x0:x1] == rp.label
            patch = q[y0:y1, x0:x1]
            if int(cand_mask.sum()) < 8:
                continue

            outside = patch[~cand_mask]
            inside = patch[cand_mask]
            local_bg = float(np.median(outside)) if outside.size else 0.0
            peak = float(np.percentile(inside, 98.0)) if inside.size else float(np.max(inside))
            denom = peak - local_bg
            if denom <= 1e-9:
                continue

            rel = np.clip((patch - local_bg) / denom, 0.0, 1.0)
            core_mask = cand_mask & (rel >= core_level)
            try:
                core_mask = remove_small_objects(
                    core_mask,
                    min_size=max(3, int(np.pi * (cmin / 2.0) ** 2 * 0.05))
                )
            except Exception:
                pass

            core_props = []
            if core_mask.any():
                clab, _ = ndi.label(core_mask)
                core_props = regionprops(clab, intensity_image=rel)

            if not core_props:
                continue

            # Highest mean intensity core is preferred; area breaks ties.
            cr = max(core_props, key=lambda p: (float(p.intensity_mean), float(p.area)))
            c_eqd = float(cr.equivalent_diameter_area)
            if not (cmin <= c_eqd <= cmax):
                continue

            cmaj = float(getattr(cr, 'axis_major_length', c_eqd))
            cminor = float(getattr(cr, 'axis_minor_length', c_eqd))
            car = cmaj / max(cminor, 1e-12)
            ccy, ccx = map(float, cr.centroid)
            ccx += x0
            ccy += y0
            classification = 'particle' if car <= ar_max else 'worm'

            all_candidates.append({
                'x': float(ccx),
                'y': float(ccy),
                'outer_x': float(cx),
                'outer_y': float(cy),
                'diameter_px': float(eqd),
                'radius_px': float(eqd / 2.0),
                'area_px': float(area),
                'outer_circularity': float(circ),
                'feret_diameter_px': float(feret),
                'core_diameter_px': float(c_eqd),
                'core_area_px': float(cr.area),
                'core_aspect_ratio': float(car),
                'aspect_ratio': float(car),
                'core_contrast': float(core_level),
                'contrast': float(denom),
                'polarity': pol,
                'edge': bool(edge),
                'classification': classification,
                'accepted': bool(classification == 'particle'),
                'manual': False,
            })

    accepted = [d for d in all_candidates if d.get('accepted', True)]
    accepted.sort(key=lambda d: (d['core_contrast'], d['core_area_px']), reverse=True)

    # Suppress overlapping duplicates from bright/dark/both polarity detection.
    kept = []
    for d in accepted:
        if all(
            np.hypot(d['x'] - k['x'], d['y'] - k['y']) >= sep
            for k in kept
        ):
            kept.append(d)

    return kept


def _tab8_v7_refresh_density_row(self, idx):
    st = _tab8_v7_get_roi3_stack(self)
    if st is None:
        return
    sx, sy = getattr(self, '_tab8_fixed_scale', (None, None))
    if sx is None or sy is None:
        return

    h, w = st.shape[1:3]
    area_um2 = (w * float(sx)) * (h * float(sy)) / 1.0e6
    dets = [
        d for d in (getattr(self, 'pd_frame_detections', {}) or {}).get(int(idx), [])
        if d.get('accepted', True) and (
            not d.get('edge', False) or bool(self.pd_include_edge_var.get())
        )
    ]
    count = len(dets)
    density = count / area_um2 if area_um2 > 0 else np.nan

    eq = float(np.sqrt(float(sx) * float(sy)))
    outer = np.asarray([
        d.get('diameter_px', np.nan) for d in dets
    ], dtype=float)
    outer = outer[np.isfinite(outer)]
    core = np.asarray([
        d.get('core_diameter_px', np.nan) for d in dets
    ], dtype=float)
    core = core[np.isfinite(core)]
    ars = np.asarray([
        d.get('core_aspect_ratio', np.nan) for d in dets
    ], dtype=float)
    ars = ars[np.isfinite(ars)]
    boundary = np.asarray([
        d.get('feret_diameter_px', d.get('diameter_px', np.nan))
        for d in dets
    ], dtype=float)
    boundary = boundary[np.isfinite(boundary)]

    mean_outer = float(np.mean(outer) * eq) if outer.size else np.nan
    mean_core = float(np.mean(core) * eq) if core.size else np.nan
    mean_ar = float(np.mean(ars)) if ars.size else np.nan
    mean_boundary = float(np.mean(boundary) * eq) if boundary.size else np.nan
    std_boundary = (
        float(np.std(boundary * eq, ddof=1))
        if boundary.size >= 2 else
        (0.0 if boundary.size == 1 else np.nan)
    )

    self.pd_results[int(idx)] = (
        int(idx) + 1,
        _tab8_v7_get_field(self, int(idx)),
        count,
        area_um2,
        density,
        mean_outer,
        mean_core,
        mean_ar,
        mean_boundary,
        std_boundary,
    )


def _tab8_v7_refresh_table(self):
    tree = getattr(self, 'pd_table', None)
    if tree is None:
        return
    try:
        for item in tree.get_children():
            tree.delete(item)
        for idx in sorted((getattr(self, 'pd_results', {}) or {}).keys()):
            row = self.pd_results[idx]
            if isinstance(row, (tuple, list)) and len(row) >= 10:
                tree.insert('', 'end', values=list(row[:10]))
    except Exception:
        pass


def _tab8_v7_valid_rows(self):
    rows = []
    analyzed = getattr(self, 'pd_frame_detections', {}) or {}
    for idx, row in sorted((getattr(self, 'pd_results', {}) or {}).items()):
        if idx not in analyzed or not isinstance(row, (tuple, list)) or len(row) < 10:
            continue
        try:
            frame = int(row[0])
            field = float(row[1])
            count = float(row[2])
            density = float(row[4])
            size = float(row[8])
            std = float(row[9])
        except Exception:
            continue
        if not all(np.isfinite(v) for v in (field, count, density)):
            continue
        rows.append((int(idx), frame, field, count, density, size, std))
    return rows


def _tab8_v7_plot(self, empty_message='Analyze current or selected frames'):
    """Plot only analyzed frames, preserving acquisition order."""
    canvas = getattr(self, 'pd_canvas', None)
    ax_count = getattr(self, 'pd_count_ax', None)
    ax_size = getattr(self, 'pd_size_ax', None)
    ax_density = getattr(self, 'pd_density_ax', None)
    if canvas is None or ax_count is None or ax_size is None or ax_density is None:
        return

    for attr in ('pd_count_frame_ax', 'pd_size_frame_ax'):
        old = getattr(self, attr, None)
        if old is not None:
            try:
                old.remove()
            except Exception:
                pass
        setattr(self, attr, None)

    ax_count.clear()
    ax_size.clear()
    ax_density.clear()
    ax_density.patch.set_visible(False)

    rows = _tab8_v7_valid_rows(self)
    ax_count.set_xlabel('Field (mT)')
    ax_count.set_ylabel('Particle count')
    ax_density.set_ylabel('Density (skyrmions/µm²)')
    ax_density.yaxis.tick_right()
    ax_density.yaxis.set_label_position('right')
    ax_size.set_xlabel('Field (mT)')
    ax_size.set_ylabel('Average particle size (nm)')
    ax_count.set_title('Particle Count/Density vs Field', fontsize=10)
    ax_size.set_title('Average Particle Size vs Field', fontsize=10)
    ax_count.grid(True, alpha=.2)
    ax_size.grid(True, alpha=.2)

    if not rows:
        ax_count.text(.5, .5, empty_message, ha='center', va='center', transform=ax_count.transAxes, fontsize=9)
        ax_size.text(.5, .5, empty_message, ha='center', va='center', transform=ax_size.transAxes, fontsize=9)
        try:
            canvas.draw_idle()
        except Exception:
            pass
        return

    # Preserve acquisition order. Split a line only when the measured field
    # direction actually reverses.
    segments = []
    current = [rows[0]]
    previous_direction = 0
    for row in rows[1:]:
        delta = float(row[2] - current[-1][2])
        direction = 1 if delta > 0 else (-1 if delta < 0 else previous_direction)
        if previous_direction and direction and direction != previous_direction:
            segments.append(current)
            current = [row]
        else:
            current.append(row)
        if direction:
            previous_direction = direction
    segments.append(current)

    for j, seg in enumerate(segments):
        x = np.asarray([r[2] for r in seg], dtype=float)
        count = np.asarray([r[3] for r in seg], dtype=float)
        density = np.asarray([r[4] for r in seg], dtype=float)
        size = np.asarray([r[5] for r in seg], dtype=float)
        err = np.asarray([r[6] for r in seg], dtype=float)

        ax_count.plot(x, count, 'o-', lw=1.5, ms=4, label='Count' if j == 0 else None)
        ax_density.plot(x, density, 's--', lw=1.3, ms=3, label='Density' if j == 0 else None)

        good = np.isfinite(size)
        if np.any(good):
            e = np.where(np.isfinite(err[good]) & (err[good] >= 0), err[good], 0.0)
            ax_size.errorbar(
                x[good], size[good], yerr=e,
                fmt='o-', lw=1.5, ms=4, capsize=3, capthick=1,
                label='Average boundary/size' if j == 0 else None
            )

    ticks = np.asarray([r[2] for r in rows], dtype=float)
    labels = [str(r[1]) for r in rows]
    if len(ticks) > 20:
        step = int(np.ceil(len(ticks) / 20))
        ticks, labels = ticks[::step], labels[::step]

    ax_count_frame = ax_count.twiny()
    ax_count_frame.set_xlabel('Frame no.')
    ax_count_frame.set_xlim(ax_count.get_xlim())
    ax_count_frame.set_xticks(ticks)
    ax_count_frame.set_xticklabels(labels, rotation=45, ha='left', fontsize=7)
    self.pd_count_frame_ax = ax_count_frame

    ax_size_frame = ax_size.twiny()
    ax_size_frame.set_xlabel('Frame no.')
    ax_size_frame.set_xlim(ax_size.get_xlim())
    ax_size_frame.set_xticks(ticks)
    ax_size_frame.set_xticklabels(labels, rotation=45, ha='left', fontsize=7)
    self.pd_size_frame_ax = ax_size_frame

    ax_count.legend(loc='best', fontsize=8)
    ax_size.legend(loc='best', fontsize=8)
    try:
        canvas.draw_idle()
    except Exception:
        pass


def _tab8_v7_particle_number_size_plot(self):
    """Figure 5: particle number versus outer-particle boundary/size."""
    ax = getattr(self, 'pd_particle_size_ax', None)
    if ax is None:
        return
    ax.clear()

    st = _tab8_v7_get_primary_stack(self)
    if st is None:
        return
    try:
        idx = max(0, min(int(self._tab8_frame_var.get()), st.shape[0] - 1))
    except Exception:
        idx = 0

    dets = [
        dict(d) for d in (getattr(self, 'pd_current_detections', []) or [])
        if d.get('accepted', True)
    ]
    dets.sort(key=lambda d: (
        float(d.get('y', d.get('outer_y', 0))),
        float(d.get('x', d.get('outer_x', 0)))
    ))

    scale = _tab8_v7_equiv_scale(self) or 1.0
    numbers, sizes = [], []
    for number, d in enumerate(dets, 1):
        try:
            dpx = float(d.get('feret_diameter_px', d.get('diameter_px', np.nan)))
            if np.isfinite(dpx):
                numbers.append(number)
                sizes.append(dpx * scale)
        except Exception:
            pass

    row = (getattr(self, 'pd_results', {}) or {}).get(idx)
    try:
        frame_std = float(row[9]) if isinstance(row, (tuple, list)) and len(row) >= 10 else 0.0
        if not np.isfinite(frame_std) or frame_std < 0:
            frame_std = 0.0
    except Exception:
        frame_std = 0.0

    if numbers:
        self.pd_particle_size_ax.errorbar(
            numbers, sizes,
            yerr=np.full(len(sizes), frame_std),
            fmt='o-', lw=1.5, ms=5, capsize=3, capthick=1,
            label='Particle size ± frame STD'
        )
        self.pd_particle_size_ax.set_xlim(.5, max(numbers) + .5)
        if len(numbers) <= 20:
            self.pd_particle_size_ax.set_xticks(numbers)
    else:
        self.pd_particle_size_ax.text(
            .5, .5, 'Analyze current frame to display particle sizes',
            ha='center', va='center', transform=self.pd_particle_size_ax.transAxes,
            fontsize=9
        )

    self.pd_particle_size_ax.set_xlabel('Particle No.')
    self.pd_particle_size_ax.set_ylabel('Particle Size (nm)')
    self.pd_particle_size_ax.set_title(
        f'Particle No. vs Particle Size — Frame {idx + 1}',
        fontsize=12, pad=10
    )
    self.pd_particle_size_ax.grid(True, alpha=.2)
    if numbers:
        self.pd_particle_size_ax.legend(loc='best', fontsize=8)
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass


def _tab8_v7_analyze_current(self):
    st = _tab8_v7_get_roi3_stack(self)
    if st is None:
        messagebox.showwarning(
            'Tab 8 Analysis',
            'Confirm ROI3 in Tab 8 first.',
            parent=self.root
        )
        return False
    idx = max(0, min(int(self._tab8_frame_var.get()), st.shape[0] - 1))
    try:
        det = _tab8_v7_run_detection(self, np.asarray(st[idx], dtype=float), idx)
    except Exception as exc:
        self.pd_status_var.set(f'Analysis failed: {type(exc).__name__}: {exc}')
        return False

    self.pd_current_detections = [dict(d) for d in det]
    self.pd_frame_detections[idx] = [dict(d) for d in det]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._tab8_clear_display = False
    self._tab8_marks_cleared = False

    _tab8_v7_refresh_density_row(self, idx)
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self)
    _tab8_v7_particle_number_size_plot(self)
    _tab8_v7_refresh_image(self)

    self.pd_status_var.set(
        f'Analyzed frame {idx + 1}: {len(det)} accepted particle(s).'
    )
    return True


def _tab8_v7_analyze_selected(self):
    st = _tab8_v7_get_roi3_stack(self)
    if st is None:
        messagebox.showwarning(
            'Tab 8 Analysis',
            'Confirm ROI3 in Tab 8 first.',
            parent=self.root
        )
        return False

    n = int(st.shape[0])
    try:
        a = int(self.pd_range_start_var.get())
        b = int(self.pd_range_end_var.get())
    except Exception:
        self.pd_status_var.set('Enter valid Start and End frame numbers.')
        return False

    if not (1 <= a <= n and 1 <= b <= n):
        self.pd_status_var.set(f'Frame range must be within 1–{n}.')
        return False
    if a > b:
        self.pd_status_var.set(f'Start frame ({a}) must be ≤ End frame ({b}).')
        return False

    old_idx = int(self._tab8_frame_var.get())
    _tab8_v7_stop_playback(self, update_button=True)
    self.pd_status_var.set(f'Analyzing frames {a}–{b}…')
    try:
        self.root.update_idletasks()
    except Exception:
        pass

    analyzed = 0
    for idx in range(a - 1, b):
        try:
            det = _tab8_v7_run_detection(self, np.asarray(st[idx], dtype=float), idx)
            self.pd_frame_detections[idx] = [dict(d) for d in det]
            _tab8_v7_refresh_density_row(self, idx)
            analyzed += 1
            self.pd_status_var.set(f'Analyzing frame {idx + 1}/{n}…')
            try:
                self.root.update_idletasks()
            except Exception:
                pass
        except Exception as exc:
            self.pd_status_var.set(
                f'Analysis stopped at frame {idx + 1}: {type(exc).__name__}: {exc}'
            )
            return False

    _tab8_v7_set_frame(self, max(a - 1, min(old_idx, b - 1)))
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self)
    _tab8_v7_particle_number_size_plot(self)
    self.pd_status_var.set(
        f'Completed analysis for frames {a}–{b}. {analyzed} frame(s) analyzed.'
    )
    return True


def _tab8_v7_select(self, event):
    """Select an existing particle or activate one-click manual addition."""
    if event is None or getattr(event, 'button', None) != 1:
        return
    if event.inaxes is not getattr(self, 'pd_ax_color', None):
        return
    if event.xdata is None or event.ydata is None:
        return

    roi3 = _tab8_v7_get_roi3_stack(self)
    sx, sy = getattr(self, '_tab8_fixed_scale', (None, None))
    if roi3 is None or sx is None or sy is None:
        return

    idx = max(0, min(int(self._tab8_frame_var.get()), roi3.shape[0] - 1))
    xpx = float(event.xdata) / float(sx)
    ypx = float(event.ydata) / float(sy)
    h, w = roi3[idx].shape[:2]
    if not (0 <= xpx < w and 0 <= ypx < h):
        return

    # Manual ADD BLOB mode.
    if bool(getattr(self, 'pd_add_blob_mode', False)):
        dets = [dict(d) for d in (getattr(self, 'pd_current_detections', []) or [])]
        try:
            dmin = float(self.pd_min_diam_var.get())
            dmax = float(self.pd_max_diam_var.get())
            duplicate_radius = max(4.0, 0.25 * max(dmin, dmax))
        except Exception:
            duplicate_radius = 8.0

        for d in dets:
            try:
                if np.hypot(
                    float(d.get('x', d.get('outer_x', 0))) - xpx,
                    float(d.get('y', d.get('outer_y', 0))) - ypx
                ) < duplicate_radius:
                    self.pd_status_var.set('A particle already exists near this location.')
                    _tab8_v7_set_add_mode(self, False)
                    return
            except Exception:
                pass

        try:
            candidate, msg = _tab8_v7_fit_manual_blob(
                self, np.asarray(roi3[idx], dtype=float), xpx, ypx
            )
        except Exception as exc:
            self.pd_status_var.set(f'ADD BLOB failed: {type(exc).__name__}: {exc}')
            return
        if candidate is None:
            self.pd_status_var.set(msg)
            return

        dets.append(dict(candidate))
        self.pd_current_detections = [dict(d) for d in dets]
        self.pd_frame_detections[idx] = [dict(d) for d in dets]
        self.pd_selected_detection = len(dets) - 1
        self.pd_multi_selected = {len(dets) - 1}
        self._tab8_clear_display = False
        self._tab8_marks_cleared = False

        _tab8_v7_refresh_density_row(self, idx)
        _tab8_v7_refresh_table(self)
        _tab8_v7_plot(self)
        _tab8_v7_particle_number_size_plot(self)
        _tab8_v7_refresh_image(self)
        _tab8_v7_set_add_mode(self, False)

        scale = _tab8_v7_equiv_scale(self) or 1.0
        self.pd_status_var.set(
            f'✓ {msg} | Particle #{len(dets)} | '
            f'Outer boundary/size = '
            f'{float(candidate.get("feret_diameter_px", candidate.get("diameter_px", np.nan))) * scale:.2f} nm | '
            f'Core = {float(candidate.get("core_diameter_px", np.nan)) * scale:.2f} nm | '
            f'Core AR = {float(candidate.get("core_aspect_ratio", np.nan)):.3f}'
        )
        return

    # Normal particle selection.
    dets = getattr(self, 'pd_current_detections', []) or []
    if not dets:
        return

    best = None
    for j, d in enumerate(dets):
        try:
            cx = float(d.get('x', d.get('outer_x', 0)))
            cy = float(d.get('y', d.get('outer_y', 0)))
            dist = float(np.hypot(cx - xpx, cy - ypx))
            hit = max(6.0, 0.8 * float(d.get('diameter_px', 10)))
            if dist <= hit and (best is None or dist < best[0]):
                best = (dist, j)
        except Exception:
            pass
    if best is None:
        return

    j = best[1]
    ctrl = str(getattr(event, 'key', '')).lower() in ('control', 'ctrl')
    if ctrl:
        if j in self.pd_multi_selected:
            self.pd_multi_selected.remove(j)
        else:
            self.pd_multi_selected.add(j)
    else:
        self.pd_multi_selected = {j}
    self.pd_selected_detection = j

    d = dets[j]
    scale = _tab8_v7_equiv_scale(self) or 1.0
    try:
        self.pd_particle_status.config(
            text=f'Particle #{j + 1}\n'
                 f'Outer boundary/size = '
                 f'{float(d.get("feret_diameter_px", d.get("diameter_px", np.nan))) * scale:.2f} nm\n'
                 f'Inner core diameter = '
                 f'{float(d.get("core_diameter_px", np.nan)) * scale:.2f} nm\n'
                 f'Core aspect ratio = '
                 f'{float(d.get("core_aspect_ratio", np.nan)):.3f}\n'
                 f'Classification = '
                 f'{str(d.get("classification", "particle")).upper()}'
        )
    except Exception:
        pass
    _tab8_v7_draw_overlays(self)


def _tab8_v7_fit_manual_blob(self, image, xpx, ypx):
    """Localized manual fit using the same controls as automatic detection."""
    from scipy import ndimage as ndi
    from skimage.measure import regionprops

    z = _tab8_v7_normalize(self, image)
    h, w = z.shape

    def gv(name, default):
        try:
            return float(getattr(self, name).get())
        except Exception:
            return float(default)

    threshold = float(np.clip(gv('pd_threshold_var', .25), .01, .99))
    core_level = float(np.clip(gv('pd_core_contrast_var', .55), .05, .99))
    dmin = max(1.0, gv('pd_min_diam_var', 10.0))
    dmax = max(dmin, gv('pd_max_diam_var', 60.0))
    cmin = max(1.0, gv('pd_core_min_diam_var', 4.0))
    cmax = max(cmin, gv('pd_core_max_diam_var', 40.0))
    ar_max = max(1.0, gv('pd_max_aspect_var', 1.8))
    bg_sigma = max(1.0, gv('pd_background_sigma_var', 20.0))
    med_size = max(1, int(round(gv('pd_median_size_var', 3.0))))
    if med_size % 2 == 0:
        med_size += 1

    try:
        polarity = str(self.pd_polarity_var.get()).strip().lower()
    except Exception:
        polarity = 'bright'
    if polarity not in ('bright', 'dark', 'both'):
        polarity = 'bright'

    pad = max(8, int(np.ceil(1.5 * dmax)) + int(np.ceil(bg_sigma * .25)))
    xa = max(0, int(np.floor(xpx - pad)))
    xb = min(w, int(np.ceil(xpx + pad + 1)))
    ya = max(0, int(np.floor(ypx - pad)))
    yb = min(h, int(np.ceil(ypx + pad + 1)))
    patch = z[ya:yb, xa:xb]
    if patch.size < 9:
        return None, 'Manual particle window is too small.'

    sm = ndi.gaussian_filter(
        ndi.median_filter(patch, size=med_size, mode='nearest'),
        sigma=1.0, mode='nearest'
    )
    bg = ndi.gaussian_filter(sm, sigma=bg_sigma, mode='nearest')
    detail = sm - bg

    paths = []
    if polarity in ('bright', 'both'):
        paths.append(('bright', detail))
    if polarity in ('dark', 'both'):
        paths.append(('dark', -detail))
    if not paths:
        paths = [('bright', detail)]

    best = None
    lx, ly = xpx - xa, ypx - ya
    for pol, signal in paths:
        positive = signal[signal > 0]
        scale = float(np.percentile(positive, 99.0)) if positive.size else 1.0
        scale = max(scale, 1e-12)
        q = np.clip(signal / scale, 0, 1)
        mask = q >= threshold
        labels, nlab = ndi.label(mask)
        if nlab <= 0:
            continue

        click_y = int(np.clip(round(ly), 0, patch.shape[0] - 1))
        click_x = int(np.clip(round(lx), 0, patch.shape[1] - 1))
        click_label = int(labels[click_y, click_x])
        props = regionprops(labels, intensity_image=q)
        if click_label:
            matches = [p for p in props if p.label == click_label]
        else:
            matches = sorted(
                props,
                key=lambda p: np.hypot(float(p.centroid[1]) - lx, float(p.centroid[0]) - ly)
            )[:1]
        if not matches:
            continue

        rp = matches[0]
        eqd = float(rp.equivalent_diameter_area)
        if np.isfinite(eqd):
            eqd = float(np.clip(eqd, dmin, dmax))
        else:
            eqd = float((dmin + dmax) / 2.0)

        cy, cx = map(float, rp.centroid)
        cx += xa
        cy += ya

        rad = max(6, int(np.ceil(.75 * eqd)))
        qx0 = max(0, int(round(cx)) - rad)
        qx1 = min(w, int(round(cx)) + rad + 1)
        qy0 = max(0, int(round(cy)) - rad)
        qy1 = min(h, int(round(cy)) + rad + 1)
        local = q[qy0 - ya:qy1 - ya, qx0 - xa:qx1 - xa]
        local_labels = labels[qy0 - ya:qy1 - ya, qx0 - xa:qx1 - xa]
        cand = local_labels == rp.label
        outside = local[~cand]
        inside = local[cand]
        local_bg = float(np.median(outside)) if outside.size else 0.0
        peak = float(np.percentile(inside, 98.0)) if inside.size else float(np.max(local))
        denom = peak - local_bg

        core_props = []
        if denom > 1e-9:
            rel = np.clip((local - local_bg) / denom, 0, 1)
            core = cand & (rel >= core_level)
            clab, _ = ndi.label(core)
            if clab.max() > 0:
                core_props = regionprops(clab, intensity_image=rel)

        if core_props:
            cr = max(core_props, key=lambda p: (float(p.intensity_mean), float(p.area)))
            c_eqd = float(cr.equivalent_diameter_area)
            c_eqd = float(np.clip(c_eqd, cmin, cmax)) if np.isfinite(c_eqd) else float(np.clip(eqd*.5, cmin, cmax))
            cmaj = float(getattr(cr, 'axis_major_length', c_eqd))
            cminor = float(getattr(cr, 'axis_minor_length', c_eqd))
            car = cmaj / max(cminor, 1e-12)
            ccy, ccx = map(float, cr.centroid)
            ccx += qx0
            ccy += qy0
            core_area = float(cr.area)
        else:
            c_eqd = float(np.clip(eqd * .5, cmin, cmax))
            car = 1.0
            ccx, ccy = cx, cy
            core_area = float(np.pi * (c_eqd / 2) ** 2)

        classification = 'particle' if car <= ar_max else 'worm'
        area = float(rp.area)
        peri = float(rp.perimeter)
        circ = 4 * np.pi * area / max(peri ** 2, 1e-12) if peri > 0 else 0.0
        feret = float(getattr(rp, 'feret_diameter_max', eqd))
        if not np.isfinite(feret):
            feret = eqd

        candidate = {
            'x': float(ccx), 'y': float(ccy),
            'outer_x': float(cx), 'outer_y': float(cy),
            'diameter_px': float(eqd), 'radius_px': float(eqd / 2),
            'area_px': area, 'outer_circularity': float(circ),
            'feret_diameter_px': float(feret),
            'core_diameter_px': float(c_eqd), 'core_area_px': core_area,
            'core_aspect_ratio': float(car), 'aspect_ratio': float(car),
            'core_contrast': float(core_level),
            'contrast': float(denom) if np.isfinite(denom) else np.nan,
            'polarity': pol, 'edge': False,
            'classification': classification, 'accepted': True, 'manual': True
        }
        distance = float(np.hypot(cx - xpx, cy - ypx))
        metric = distance - (0.25 if np.isfinite(denom) and denom > 0 else 0)
        if best is None or metric < best[0]:
            best = (metric, candidate)

    if best is None:
        md = float((dmin + dmax) / 2.0)
        cd = float(np.clip(md * .5, cmin, cmax))
        return {
            'x': float(xpx), 'y': float(ypx),
            'outer_x': float(xpx), 'outer_y': float(ypx),
            'diameter_px': md, 'radius_px': md / 2,
            'area_px': float(np.pi * (md / 2) ** 2),
            'outer_circularity': 1.0, 'feret_diameter_px': md,
            'core_diameter_px': cd, 'core_area_px': float(np.pi * (cd / 2) ** 2),
            'core_aspect_ratio': 1.0, 'aspect_ratio': 1.0,
            'core_contrast': core_level, 'contrast': np.nan,
            'polarity': 'manual', 'edge': False,
            'classification': 'particle', 'accepted': True, 'manual': True
        }, 'Local segmentation did not isolate the click; used parameter-based manual fit.'
    return best[1], 'Manual blob fitted using current Tab-8 parameters.'


def _tab8_v7_set_add_mode(self, active=None):
    if active is None:
        active = not bool(getattr(self, 'pd_add_blob_mode', False))
    self.pd_add_blob_mode = bool(active)
    try:
        self.pd_add_blob_button.configure(
            relief='sunken' if self.pd_add_blob_mode else 'raised',
            text='ADD BLOB — CLICK IMAGE' if self.pd_add_blob_mode else 'ADD BLOB'
        )
    except Exception:
        pass
    self.pd_status_var.set(
        'ADD BLOB active — click the ROI3 image at the particle centre.'
        if self.pd_add_blob_mode else 'ADD BLOB mode cancelled.'
    )


def _tab8_v7_delete_selected(self):
    """DELETE SELECTED: remove only the selected particle/data point from Tab-8."""
    dets = [dict(d) for d in (getattr(self, 'pd_current_detections', []) or [])]
    selected = sorted({
        int(j) for j in (getattr(self, 'pd_multi_selected', set()) or set())
        if 0 <= int(j) < len(dets)
    })
    if not selected and getattr(self, 'pd_selected_detection', None) is not None:
        try:
            j = int(self.pd_selected_detection)
            if 0 <= j < len(dets):
                selected = [j]
        except Exception:
            pass
    if not selected:
        self.pd_status_var.set('DELETE SELECTED: select a data point/particle first.')
        return False

    idx = int(self._tab8_frame_var.get())
    selected_set = set(selected)
    before = [dict(d) for d in dets]
    deleted = [(j, dict(dets[j])) for j in selected]
    self.pd_last_deleted = {'frame': idx, 'before': before, 'deleted': deleted}

    # Only the selected particle(s) are removed from the current frame.
    keep = [dict(d) for j, d in enumerate(dets) if j not in selected_set]
    self.pd_current_detections = keep
    self.pd_frame_detections[idx] = [dict(d) for d in keep]
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._tab8_clear_display = False
    self._tab8_marks_cleared = False

    # Recalculate the affected frame and rebuild all derived displays.
    _tab8_v7_refresh_density_row(self, idx)
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self)
    _tab8_v7_particle_number_size_plot(self)
    _tab8_v7_refresh_image(self)
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass
    self.pd_status_var.set(
        f'DELETE SELECTED: deleted {len(deleted)} data point(s) from frame '
        f'{idx + 1}; graphs, table and calculations updated.'
    )
    return True

def _tab8_v7_undo_delete(self):
    rec = getattr(self, 'pd_last_deleted', None)
    if not rec:
        self.pd_status_var.set('Nothing to undo.')
        return False
    idx = int(rec['frame'])
    restored = [dict(d) for d in rec['before']]
    self.pd_frame_detections[idx] = restored
    if int(self._tab8_frame_var.get()) == idx:
        self.pd_current_detections = [dict(d) for d in restored]
    self.pd_last_deleted = None
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._tab8_clear_display = False
    _tab8_v7_refresh_density_row(self, idx)
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self)
    _tab8_v7_particle_number_size_plot(self)
    _tab8_v7_refresh_image(self)
    self.pd_status_var.set(f'Undo: restored deleted particle(s) in frame {idx + 1}.')
    return True


def _tab8_v7_clear_mark(self):
    """CLEAR MARK: clear all marks/labels and analysis data for the selected frame only."""
    idx = int(self._tab8_frame_var.get())

    # CLEAR MARK is frame-wide: no particle selection is required.
    # Other frames remain completely untouched.
    self.pd_current_detections = []
    if isinstance(getattr(self, 'pd_frame_detections', None), dict):
        self.pd_frame_detections[idx] = []

    # Remove this frame's calculated result so its graph point and table row vanish.
    if isinstance(getattr(self, 'pd_results', None), dict):
        self.pd_results.pop(idx, None)

    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._tab8_clear_display = True
    self._tab8_marks_cleared = True

    # Clear frame-indexed analysis caches, if present.
    for name in ('pd_analysis_data', 'pd_graph_data', 'pd_plot_data',
                 'pd_frame_results', 'pd_frame_analysis'):
        obj = getattr(self, name, None)
        if isinstance(obj, dict):
            obj.pop(idx, None)
        elif isinstance(obj, list) and 0 <= idx < len(obj):
            obj[idx] = None

    # Rebuild graphs/table from all remaining frames.
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self)

    # Figure 5 is frame-specific; it must not retain the cleared frame's points.
    ax = getattr(self, 'pd_particle_size_ax', None)
    if ax is not None:
        try:
            ax.clear()
            ax.set_xlabel('Particle No.')
            ax.set_ylabel('Particle Size (nm)')
            ax.set_title(
                f'Particle No. vs Particle Size — Frame {idx + 1}',
                fontsize=10, pad=10
            )
            ax.text(.5, .5, 'All marks cleared for this frame',
                    ha='center', va='center', transform=ax.transAxes, fontsize=9)
            ax.grid(True, alpha=.2)
        except Exception:
            pass

    # Remove all ROI marks/labels for the selected frame.
    _tab8_v7_refresh_image(self)
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass
    self.pd_status_var.set(
        f'CLEAR MARK: all marks/labels and graph/table/calculation data '
        f'cleared for selected frame {idx + 1} only. Other frames retained.'
    )
    return True

def _tab8_v7_reset_analysis(self):
    """RESET: clear all Tab-8 marks, labels, data, graphs and calculations for all frames."""
    # Stop playback/timers without destroying/rebuilding the Tab-8 UI.
    for name in ('pd_playing', '_pd_playing', '_tab8_playing'):
        if hasattr(self, name):
            try:
                setattr(self, name, False)
            except Exception:
                pass
    for name in ('pd_after_id', '_pd_after_id', '_tab8_after_id'):
        aid = getattr(self, name, None)
        if aid is not None:
            try:
                self.root.after_cancel(aid)
            except Exception:
                pass
            try:
                setattr(self, name, None)
            except Exception:
                pass

    # Clear every frame's marks/detections and every accumulated result.
    if isinstance(getattr(self, 'pd_frame_detections', None), dict):
        for k in list(self.pd_frame_detections.keys()):
            self.pd_frame_detections[k] = []
    self.pd_current_detections = []
    if isinstance(getattr(self, 'pd_results', None), dict):
        self.pd_results.clear()

    # Clear all frame-indexed analysis caches.
    for name in ('pd_analysis_data', 'pd_graph_data', 'pd_plot_data',
                 'pd_frame_results', 'pd_frame_analysis'):
        obj = getattr(self, name, None)
        if isinstance(obj, dict):
            obj.clear()
        elif isinstance(obj, list):
            for i in range(len(obj)):
                obj[i] = None

    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._tab8_clear_display = True
    self._tab8_marks_cleared = True

    # Figure 3/4 and the results table become empty.
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self, empty_message='RESET — all frames cleared')

    # Figure 5 must also be completely empty after RESET.
    ax = getattr(self, 'pd_particle_size_ax', None)
    if ax is not None:
        try:
            ax.clear()
            ax.set_xlabel('Particle No.')
            ax.set_ylabel('Particle Size (nm)')
            ax.set_title(
                'Particle No. vs Particle Size — Reset',
                fontsize=9, pad=10
            )
            ax.text(.5, .5, 'RESET — all frames cleared',
                    ha='center', va='center', transform=ax.transAxes, fontsize=9)
            ax.grid(True, alpha=.2)
        except Exception:
            pass

    # Remove all ROI marks/labels from the displayed frame.
    _tab8_v7_refresh_image(self)
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass
    self.pd_status_var.set(
        'RESET: all marks/labels, graph points, table rows and calculations '
        'cleared for ALL frames.'
    )
    return True

def _tab8_v7_clear(self):
    """Clear analysis/results/overlays but keep source Primary ROI1 and ROI3."""
    _tab8_v7_stop_playback(self, update_button=True)
    self.pd_results = {}
    self.pd_frame_detections = {}
    self.pd_current_detections = []
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self.pd_last_deleted = None
    self._tab8_clear_display = True
    self._tab8_marks_cleared = True
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self)
    _tab8_v7_refresh_image(self)
    self.pd_status_var.set(
        'CLEAR: particle labels, centres, outer/core circles, table values and graphs removed.'
    )


def _tab8_v7_restore_default_controls(self):
    defaults = {
        'pd_threshold_var': .25,
        'pd_polarity_var': 'bright',
        'pd_min_diam_var': 10.0,
        'pd_max_diam_var': 60.0,
        'pd_min_sep_var': 25.0,
        'pd_min_area_var': 0.0,
        'pd_max_area_var': 1e6,
        'pd_min_circularity_var': 0.0,
        'pd_max_circularity_var': 1.0,
        'pd_background_sigma_var': 20.0,
        'pd_median_size_var': 3.0,
        'pd_core_contrast_var': .55,
        'pd_core_min_diam_var': 4.0,
        'pd_core_max_diam_var': 40.0,
        'pd_max_aspect_var': 1.8,
        'pd_include_edge_var': False,
        'pd_show_centers_var': True,
    }
    for name, value in defaults.items():
        try:
            getattr(self, name).set(value)
        except Exception:
            pass
    self.pd_status_var.set('DEFAULT: Tab-8 detection controls restored.')


def _tab8_v7_confirm_roi3(self):
    primary = _tab8_v7_get_primary_stack(self)
    c = getattr(self, '_tab8_pending_roi3_coords', None) or getattr(self, '_tab8_roi3_coords', None)
    if primary is None:
        messagebox.showwarning(
            'ROI3',
            'Processed/finalized Primary ROI1 from Tab 6 is not available.',
            parent=self.root
        )
        return False
    if not isinstance(c, dict):
        messagebox.showwarning(
            'ROI3',
            'Click SELECT ROI3 and drag a rectangle on Primary ROI1 first.',
            parent=self.root
        )
        return False

    h, w = primary.shape[1:3]
    try:
        x0, x1 = int(c['xmin']), int(c['xmax'])
        y0, y1 = int(c['ymin']), int(c['ymax'])
    except Exception:
        self.pd_status_var.set('Invalid ROI3 coordinates.')
        return False

    if not (0 <= x0 < x1 < w and 0 <= y0 < y1 < h):
        self.pd_status_var.set('ROI3 must be inside Primary ROI1 and have non-zero area.')
        return False

    self._tab8_roi3_coords = {'xmin': x0, 'xmax': x1, 'ymin': y0, 'ymax': y1}
    self._tab8_pending_roi3_coords = None
    self._tab8_roi3_stack = primary[:, y0:y1 + 1, x0:x1 + 1].astype(np.float32, copy=True)
    self._tab8_roi3_shape = self._tab8_roi3_stack.shape[1:3]
    self._tab8_roi3_xy_extent = (
        0.0, self._tab8_roi3_stack.shape[2] * self._tab8_fixed_scale[0],
        0.0, self._tab8_roi3_stack.shape[1] * self._tab8_fixed_scale[1]
    )
    self._tab8_roi3_confirmed = True

    # A new ROI3 is a new analysis source, so stale detections are invalid.
    self.pd_results = {}
    self.pd_frame_detections = {}
    self.pd_current_detections = []
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._tab8_clear_display = False
    self._tab8_marks_cleared = False
    _tab8_v7_stop_playback(self, update_button=True)

    _tab8_v7_apply_fixed_axes(self)
    _tab8_v7_refresh_image(self)
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self)

    sx, sy = self._tab8_fixed_scale
    rh, rw = self._tab8_roi3_stack.shape[1:3]
    self.pd_status_var.set(
        f'ROI3 CONFIRMED — {rw} × {rh} px | '
        f'{rw * sx:.3f} × {rh * sy:.3f} nm | analysis source locked to ROI3.'
    )
    try:
        self.pd_roi3_select_button.configure(text='SELECT ROI3')
    except Exception:
        pass
    return True


def _tab8_v7_apply_fixed_axes(self):
    primary = _tab8_v7_get_primary_stack(self)
    if primary is None:
        return False
    sx, sy = getattr(self, '_tab8_fixed_scale', (None, None))
    if sx is None or sy is None:
        sx, sy = _tab8_v7_get_scale(self)
    if sx is None or sy is None:
        return False
    self._tab8_fixed_scale = (float(sx), float(sy))

    for ax, artist, extent in (
        (self.pd_ax_gray, self.pd_image_artist_gray, _tab8_v7_primary_extent(self)),
        (self.pd_ax_color, self.pd_image_artist_color, _tab8_v7_roi3_extent(self) or _tab8_v7_primary_extent(self)),
    ):
        if extent is None:
            continue
        artist.set_extent(extent)
        ax.set_autoscale_on(False)
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel('X (nm)')
        ax.set_ylabel('Y (nm)')
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass
    return True


def _tab8_v7_select_roi3_press(self, event):
    if not getattr(self, '_tab8_roi3_select_mode', False):
        return
    if event is None or getattr(event, 'button', None) != 1:
        return
    if event.inaxes is not getattr(self, 'pd_ax_gray', None):
        return
    if event.xdata is None or event.ydata is None:
        return
    self._tab8_roi3_start = (float(event.xdata), float(event.ydata))
    self._tab8_roi3_dragging = True
    try:
        if getattr(self, '_tab8_roi3_patch', None) is not None:
            self._tab8_roi3_patch.remove()
    except Exception:
        pass
    self._tab8_roi3_patch = None


def _tab8_v7_select_roi3_motion(self, event):
    if not getattr(self, '_tab8_roi3_dragging', False):
        return
    if event is None or event.inaxes is not getattr(self, 'pd_ax_gray', None):
        return
    if event.xdata is None or event.ydata is None or getattr(self, '_tab8_roi3_start', None) is None:
        return

    sx, sy = self._tab8_fixed_scale
    pext = _tab8_v7_primary_extent(self)
    if pext is None:
        return
    x0, y0 = self._tab8_roi3_start
    x1, y1 = float(event.xdata), float(event.ydata)
    x0 = max(0.0, min(x0, pext[1]))
    x1 = max(0.0, min(x1, pext[1]))
    y0 = max(0.0, min(y0, pext[3]))
    y1 = max(0.0, min(y1, pext[3]))
    xa, xb = sorted((x0, x1))
    ya, yb = sorted((y0, y1))

    try:
        if getattr(self, '_tab8_roi3_patch', None) is not None:
            self._tab8_roi3_patch.remove()
    except Exception:
        pass

    self._tab8_roi3_patch = plt.Rectangle(
        (xa, ya), max(1e-12, xb - xa), max(1e-12, yb - ya),
        fill=False, edgecolor='red', linewidth=2.4, zorder=30
    )
    self.pd_ax_gray.add_patch(self._tab8_roi3_patch)
    try:
        self.pd_canvas.draw_idle()
    except Exception:
        pass


def _tab8_v7_select_roi3_release(self, event):
    if not getattr(self, '_tab8_roi3_dragging', False):
        return
    self._tab8_roi3_dragging = False

    if event is None or event.inaxes is not getattr(self, 'pd_ax_gray', None):
        self._tab8_roi3_start = None
        return
    if event.xdata is None or event.ydata is None:
        self._tab8_roi3_start = None
        return

    sx, sy = self._tab8_fixed_scale
    pext = _tab8_v7_primary_extent(self)
    if pext is None:
        self._tab8_roi3_start = None
        return

    x0, y0 = self._tab8_roi3_start
    x1, y1 = float(event.xdata), float(event.ydata)
    self._tab8_roi3_start = None

    x0 = max(0.0, min(x0, pext[1]))
    x1 = max(0.0, min(x1, pext[1]))
    y0 = max(0.0, min(y0, pext[3]))
    y1 = max(0.0, min(y1, pext[3]))
    xa, xb = sorted((x0 / sx, x1 / sx))
    ya, yb = sorted((y0 / sy, y1 / sy))

    ph, pw = self._tab8_primary_shape
    x0p = max(0, min(int(np.floor(xa)), pw - 1))
    x1p = max(0, min(int(np.ceil(xb) - 1), pw - 1))
    y0p = max(0, min(int(np.floor(ya)), ph - 1))
    y1p = max(0, min(int(np.ceil(yb) - 1), ph - 1))

    if x1p <= x0p or y1p <= y0p:
        self.pd_status_var.set('ROI3 too small. Drag a larger rectangle.')
        return

    self._tab8_pending_roi3_coords = {
        'xmin': x0p, 'xmax': x1p, 'ymin': y0p, 'ymax': y1p
    }
    self._tab8_roi3_coords = dict(self._tab8_pending_roi3_coords)
    self._tab8_roi3_confirmed = False
    self.pd_status_var.set('ROI3 selected. Click CONFIRM ROI3 to create the analysis stack.')
    _tab8_v7_draw_overlays(self)


def _tab8_v7_clear_roi3(self):
    _tab8_v7_stop_playback(self, update_button=True)
    self._tab8_roi3_coords = None
    self._tab8_pending_roi3_coords = None
    self._tab8_roi3_stack = None
    self._tab8_roi3_shape = None
    self._tab8_roi3_xy_extent = None
    self._tab8_roi3_confirmed = False
    self._tab8_roi3_select_mode = False
    self._tab8_roi3_dragging = False
    self._tab8_roi3_start = None
    self.pd_results = {}
    self.pd_frame_detections = {}
    self.pd_current_detections = []
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self._tab8_clear_display = False
    self._tab8_marks_cleared = False
    try:
        self.pd_particle_status.config(text='No particle selected.')
    except Exception:
        pass
    try:
        self.pd_roi3_select_button.configure(text='SELECT ROI3')
    except Exception:
        pass
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self, empty_message='ROI3 cleared — select and confirm a new ROI3.')
    _tab8_v7_refresh_image(self)
    self.pd_status_var.set('ROI3 cleared. Primary ROI1 remains available for a new selection.')
    return True


def _tab6_build_exact_processed_primary_stack(self):
    """
    Build/read the exact Primary ROI1 stack represented by Tab 6's
    ROI-processing display (_proc_image), not the merely resolution-improved
    roi_stack. The result is cached by the source/processing signature.
    """
    if getattr(self,'recons',None) is None:
        return None

    base=getattr(self,'roi_stack',None)
    local=getattr(self,'local_mask',None)
    if base is None or local is None:
        return None

    try:
        n=len(base)
        shape=tuple(np.asarray(base[0]).shape[:2]) if n else None
    except Exception:
        return None
    if n<=0 or shape is None:
        return None

    def _get(var,default):
        try:
            return var.get()
        except Exception:
            return default

    try:
        kind=str(_get(getattr(self,'filter_var',None),'None'))
    except Exception:
        kind='None'
    try:
        strength=float(_get(getattr(self,'strength_var',None),1.0))
    except Exception:
        strength=1.0
    try:
        contrast=float(_get(getattr(self,'contrast_var',None),1.0))
    except Exception:
        contrast=1.0
    try:
        brightness=float(_get(getattr(self,'brightness_var',None),0.0))
    except Exception:
        brightness=0.0

    signature=(
        id(base),tuple(np.asarray(base).shape),
        id(local),tuple(np.asarray(local).shape),
        kind,round(strength,12),round(contrast,12),round(brightness,12)
    )

    cached=getattr(self,'_tab6_exact_processed_primary_stack_cache',None)
    cached_sig=getattr(self,'_tab6_exact_processed_primary_stack_signature',None)
    if cached is not None and cached_sig==signature:
        return cached

    frames=[]
    for i in range(n):
        arr=np.asarray(_proc_image(self,i),dtype=np.float32)
        if arr.ndim!=2 or arr.shape!=shape:
            raise RuntimeError(
                f'Tab 6 processed Primary ROI1 frame {i+1}/{n} '
                f'has inconsistent shape {arr.shape}, expected {shape}.'
            )
        frames.append(arr)

    processed=np.stack(frames,axis=0).astype(np.float32,copy=True)

    self._tab6_exact_processed_primary_stack_cache=processed
    self._tab6_exact_processed_primary_stack_signature=signature
    return processed


def _tab8_v7_sync_source(self, preserve_current=True):
    """
    Synchronize Tab 8 with the exact processed Primary ROI1 stack from Tab 6.
    Only Tab-8 cache/state is changed here.
    """
    try:
        if getattr(self, 'roi_stack', None) is None or getattr(self, 'recons', None) is None:
            raise RuntimeError('Processed Primary ROI1 is not available in Tab 6.')
        sx, sy = _tab8_v7_get_scale(self)
        if sx is None or sy is None:
            raise RuntimeError('Tab 6 Measurement Scale is unavailable.')
        builder = globals().get('_tab6_build_exact_processed_primary_stack')
        if not callable(builder):
            raise RuntimeError('Tab 6 processed-stack builder is unavailable.')
        processed = builder(self)
        processed = np.asarray(processed, dtype=np.float32)
        if processed.ndim != 3 or processed.shape[0] < 1:
            raise RuntimeError(f'Invalid processed Primary ROI1 stack shape: {processed.shape}.')
    except Exception as exc:
        self._tab8_primary_stack = None
        self._tab8_primary_shape = None
        try:
            self._tab8_frame_slider.configure(from_=0, to=0, state='disabled')
        except Exception:
            pass
        self.pd_status_var.set(
            f'Tab 8 source unavailable: {type(exc).__name__}: {exc}'
        )
        return False

    n, h, w = map(int, processed.shape)
    old_idx = 0
    if preserve_current:
        try:
            old_idx = int(self._tab8_frame_var.get())
        except Exception:
            old_idx = 0
    old_idx = max(0, min(old_idx, n - 1))

    old_sig = getattr(self, '_tab8_source_geometry_signature', None)
    new_sig = (id(processed), n, h, w, round(float(sx), 12), round(float(sy), 12))
    geometry_changed = (
        old_sig is not None and
        (old_sig[2], old_sig[3], old_sig[4], old_sig[5]) !=
        (h, w, round(float(sx), 12), round(float(sy), 12))
    )
    source_changed = old_sig != new_sig

    self._tab8_primary_stack = processed.copy()
    self._tab8_primary_shape = (h, w)
    self._tab8_fixed_shape = (h, w)
    # If Tab 6 has an authoritative physical extent, use its exact scale.
    try:
        tab6_ext = getattr(self, '_proc_full_physical_extent', None)
        if tab6_ext is not None:
            tx0, tx1, ty0, ty1 = map(float, tab6_ext)
            if tx1 > tx0 and ty1 > ty0 and w > 0 and h > 0:
                sx = (tx1 - tx0) / float(w)
                sy = (ty1 - ty0) / float(h)
    except Exception:
        pass
    self._tab8_fixed_scale = (float(sx), float(sy))
    self._tab8_source_geometry_signature = new_sig
    self._tab8_primary_xy_extent = (0.0, w * float(sx), 0.0, h * float(sy))

    # Preserve ROI3 only when its pixel geometry remains valid. If Tab-6 output
    # geometry changes, old ROI3 coordinates cannot safely be reused.
    if geometry_changed:
        self._tab8_roi3_coords = None
        self._tab8_pending_roi3_coords = None
        self._tab8_roi3_stack = None
        self._tab8_roi3_shape = None
        self._tab8_roi3_xy_extent = None
        self._tab8_roi3_confirmed = False
        self.pd_results = {}
        self.pd_frame_detections = {}
        self.pd_current_detections = []
        self.pd_selected_detection = None
        self.pd_multi_selected = set()
    elif source_changed:
        self.pd_results = {}
        self.pd_frame_detections = {}
        self.pd_current_detections = []
        self.pd_selected_detection = None
        self.pd_multi_selected = set()
    elif isinstance(getattr(self, '_tab8_roi3_coords', None), dict):
        c = self._tab8_roi3_coords
        try:
            x0, x1 = int(c['xmin']), int(c['xmax'])
            y0, y1 = int(c['ymin']), int(c['ymax'])
            valid = 0 <= x0 < x1 < w and 0 <= y0 < y1 < h
        except Exception:
            valid = False
        if valid:
            self._tab8_roi3_stack = self._tab8_primary_stack[:, y0:y1 + 1, x0:x1 + 1].copy()
            self._tab8_roi3_shape = self._tab8_roi3_stack.shape[1:3]
            self._tab8_roi3_xy_extent = (
                0.0, (x1 - x0 + 1) * float(sx),
                0.0, (y1 - y0 + 1) * float(sy)
            )
        else:
            self._tab8_roi3_coords = None
            self._tab8_pending_roi3_coords = None
            self._tab8_roi3_stack = None
            self._tab8_roi3_shape = None
            self._tab8_roi3_xy_extent = None
            self._tab8_roi3_confirmed = False
            self.pd_results = {}
            self.pd_frame_detections = {}
            self.pd_current_detections = []

    self._tab8_programmatic = True
    try:
        self._tab8_frame_slider.configure(
            from_=0, to=n - 1, resolution=1,
            variable=self._tab8_frame_var, state='normal'
        )
        self.pd_range_start_spin.configure(from_=1, to=n)
        self.pd_range_end_spin.configure(from_=1, to=n)
        self.pd_range_start_var.set(max(1, min(int(self.pd_range_start_var.get()), n)))
        self.pd_range_end_var.set(max(1, min(int(self.pd_range_end_var.get()), n)))
        self._tab8_frame_var.set(old_idx)
        self._tab8_frame_slider.set(old_idx)
    finally:
        self._tab8_programmatic = False

    if not getattr(self, '_tab8_slider_initialized', False):
        # Start at the middle frame only once; thereafter preserve the user's frame.
        mid = (n - 1) // 2
        self._tab8_programmatic = True
        try:
            self._tab8_frame_var.set(mid)
            self._tab8_frame_slider.set(mid)
            old_idx = mid
        finally:
            self._tab8_programmatic = False
        self._tab8_slider_initialized = True

    _tab8_v7_frame_changed(self, old_idx)
    self.pd_status_var.set(
        f'✓ Exact processed Primary ROI1 from Tab 6 | '
        f'FOV = {w * sx:.6g} × {h * sy:.6g} nm | Frame {old_idx + 1}/{n}'
    )
    return True


def _tab8_v7_play(self):
    st = _tab8_v7_get_primary_stack(self)
    if st is None or st.shape[0] <= 1:
        self.pd_status_var.set('PLAY requires at least two source frames.')
        return False
    if self._tab8_playing:
        return True
    self._tab8_playing = True
    try:
        self.pd_play_button.configure(text='PLAYING…')
        self.pd_pause_button.configure(state='normal')
    except Exception:
        pass
    _tab8_v7_play_step(self)
    return True


def _tab8_v7_play_step(self):
    if not getattr(self, '_tab8_playing', False):
        return
    st = _tab8_v7_get_primary_stack(self)
    if st is None:
        _tab8_v7_stop_playback(self)
        return
    n = int(st.shape[0])
    idx = int(self._tab8_frame_var.get())
    next_idx = idx + 1
    if next_idx >= n:
        next_idx = 0
    _tab8_v7_set_frame(self, next_idx, user_action=True)

    try:
        fps = max(1, min(60, int(self.pd_fps_var.get())))
    except Exception:
        fps = 10
    delay = max(15, int(round(1000.0 / fps)))
    try:
        self._tab8_play_job = self.root.after(delay, lambda: _tab8_v7_play_step(self))
    except Exception:
        self._tab8_play_job = None
        self._tab8_playing = False


def _tab8_v7_pause(self):
    _tab8_v7_stop_playback(self, update_button=True)
    self.pd_status_var.set(f'PAUSED at frame {int(self._tab8_frame_var.get()) + 1}.')


def _tab8_v7_first(self):
    _tab8_v7_stop_playback(self, update_button=True)
    _tab8_v7_set_frame(self, 0)
    self.pd_status_var.set('Moved to first frame.')


def _tab8_v7_previous(self):
    _tab8_v7_stop_playback(self, update_button=True)
    idx = max(0, int(self._tab8_frame_var.get()) - 1)
    _tab8_v7_set_frame(self, idx)


def _tab8_v7_next(self):
    _tab8_v7_stop_playback(self, update_button=True)
    st = _tab8_v7_get_primary_stack(self)
    if st is not None:
        _tab8_v7_set_frame(self, min(st.shape[0] - 1, int(self._tab8_frame_var.get()) + 1))


def _tab8_v7_export_table(self):
    if not getattr(self, 'pd_results', None):
        messagebox.showwarning('Tab 8', 'No analyzed-frame results available.', parent=self.root)
        return False
    fp = filedialog.asksaveasfilename(
        parent=self.root, initialdir=self._export_initialdir(),
        title='Export Tab 8 table', defaultextension='.csv',
        filetypes=[('CSV', '*.csv')]
    )
    if not fp:
        return False
    cols = [
        'Frame', 'Field (mT)', 'Count', 'ROI area (µm²)',
        'Density (skyrmions/µm²)', 'Mean outer d (nm)',
        'Mean core d (nm)', 'Mean core AR',
        'Outer-particle boundary/size (nm)',
        'STD outer boundary/size (nm)'
    ]
    rows = [
        list(self.pd_results[k][:10])
        for k in sorted(self.pd_results)
        if isinstance(self.pd_results[k], (tuple, list))
        and len(self.pd_results[k]) >= 10
    ]
    try:
        pd.DataFrame(rows, columns=cols).to_csv(fp, index=False)
        self.pd_status_var.set(f'Exported table: {os.path.basename(fp)}')
        return True
    except Exception as exc:
        messagebox.showerror(
            'Tab 8 Export', f'Could not export table:\n\n{type(exc).__name__}: {exc}',
            parent=self.root
        )
        return False


def _tab8_v7_export_figure5_csv(self):
    if _tab8_v7_get_primary_stack(self) is None:
        messagebox.showwarning('Tab 8', 'Processed Primary ROI1 is not available.', parent=self.root)
        return False
    idx = int(self._tab8_frame_var.get())
    dets = [
        dict(d) for d in (getattr(self, 'pd_current_detections', []) or [])
        if d.get('accepted', True)
    ]
    dets.sort(key=lambda d: (
        float(d.get('y', d.get('outer_y', 0))),
        float(d.get('x', d.get('outer_x', 0)))
    ))
    scale = _tab8_v7_equiv_scale(self) or 1.0
    row = (getattr(self, 'pd_results', {}) or {}).get(idx)
    try:
        std = float(row[9]) if isinstance(row, (tuple, list)) and len(row) >= 10 else 0.0
    except Exception:
        std = 0.0
    field = _tab8_v7_get_field(self, idx)

    fp = filedialog.asksaveasfilename(
        parent=self.root, initialdir=self._export_initialdir(),
        title='Export Figure 5 data', defaultextension='.csv',
        filetypes=[('CSV', '*.csv')]
    )
    if not fp:
        return False

    records = []
    for k, d in enumerate(dets, 1):
        try:
            dpx = float(d.get('diameter_px', np.nan))
            feret = float(d.get('feret_diameter_px', np.nan))
            if np.isfinite(dpx):
                records.append([
                    k, idx + 1, field, dpx * scale, std,
                    feret * scale if np.isfinite(feret) else np.nan
                ])
        except Exception:
            pass

    try:
        pd.DataFrame(
            records,
            columns=[
                'Particle No.', 'Frame', 'Field (mT)',
                'Outer Particle Size (nm)',
                'Frame STD Outer Size (nm)',
                'Feret Boundary/Size (nm)'
            ]
        ).to_csv(fp, index=False)
        self.pd_status_var.set(f'Exported Figure 5 data: {os.path.basename(fp)}')
        return True
    except Exception as exc:
        messagebox.showerror(
            'Tab 8 Export', f'Could not export Figure 5 data:\n\n{type(exc).__name__}: {exc}',
            parent=self.root
        )
        return False


def _tab8_v7_export_image(self):
    primary = _tab8_v7_get_primary_stack(self)
    if primary is None:
        messagebox.showwarning('Tab 8 Export', 'Processed Primary ROI1 is not available.', parent=self.root)
        return False
    sx, sy = getattr(self, '_tab8_fixed_scale', (None, None))
    if sx is None or sy is None:
        messagebox.showwarning('Tab 8 Export', 'Tab 6 Measurement Scale is unavailable.', parent=self.root)
        return False

    idx = max(0, min(int(self._tab8_frame_var.get()), primary.shape[0] - 1))
    outdir = self._export_initialdir() or os.getcwd()
    os.makedirs(outdir, exist_ok=True)
    field = _tab8_v7_get_field(self, idx)
    field_tag = f'{field:+.2f}mT' if np.isfinite(field) else f'Frame{idx + 1:04d}'
    field_tag = field_tag.replace('+', 'p').replace('-', 'm')

    dets = [
        dict(d) for d in (getattr(self, 'pd_current_detections', []) or [])
        if d.get('accepted', True)
    ]

    def save_png(arr, title, cmap_name, cbar_label, path, markers=False):
        a = np.asarray(arr, dtype=float)
        h, w = a.shape[:2]
        finite = a[np.isfinite(a)]
        lo, hi = (0.0, 1.0)
        if finite.size:
            lo, hi = float(np.min(finite)), float(np.max(finite))
            if hi <= lo:
                hi = lo + 1.0
        ext = (0.0, w * sx, 0.0, h * sy)
        fig = plt.Figure(figsize=(6.2, 5.8), dpi=300, facecolor='white')
        ax = fig.add_axes([.13, .12, .68, .76])
        im = ax.imshow(
            a, origin='lower', extent=ext, interpolation='nearest',
            aspect='equal', cmap=cmap_name, vmin=lo, vmax=hi
        )
        ax.set_xlim(ext[0], ext[1])
        ax.set_ylim(ext[2], ext[3])
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel('X (nm)', fontsize=10)
        ax.set_ylabel('Y (nm)', fontsize=10)
        ax.set_title(title, fontsize=12, pad=6)
        ax.tick_params(labelsize=8)

        if markers and dets:
            theta = np.linspace(0, 2 * np.pi, 240)
            for k, d in enumerate(dets, 1):
                ox = float(d.get('outer_x', d.get('x', 0)))
                oy = float(d.get('outer_y', d.get('y', 0)))
                cx = float(d.get('x', ox))
                cy = float(d.get('y', oy))
                od = max(0.0, float(d.get('diameter_px', d.get('feret_diameter_px', 0))))
                cd = max(0.0, float(d.get('core_diameter_px', 0)))
                ar = max(1.0, float(d.get('core_aspect_ratio', 1.0)))
                ax.plot(
                    (ox + od / 2 * np.cos(theta)) * sx,
                    (oy + od / 2 * np.sin(theta)) * sy,
                    color='yellow', lw=2.2, zorder=20
                )
                ax.plot(
                    (cx + cd * ar / 2 * np.cos(theta)) * sx,
                    (cy + cd / 2 * np.sin(theta)) * sy,
                    color='lime', lw=1.8, zorder=21
                )
                ax.plot([cx * sx], [cy * sy], marker='o', ms=5.5,
                        mfc='none', mec='lime', mew=1.6, zorder=22)
                ax.text(
                    cx * sx, cy * sy, str(k), color='black',
                    ha='center', va='center', fontsize=7.5,
                    bbox=dict(boxstyle='circle,pad=.18', fc='yellow', ec='lime', lw=1.0),
                    zorder=23
                )
        cax = fig.add_axes([.85, .12, .04, .76])
        cb = fig.colorbar(im, cax=cax)
        cb.set_label(cbar_label, fontsize=9)
        cb.ax.tick_params(labelsize=8)
        fig.savefig(path, dpi=300, format='png', bbox_inches='tight', facecolor='white')
        plt.close(fig)

    try:
        cmap = str(self.colormap_var.get() or 'gray')
    except Exception:
        cmap = 'gray'

    ppath = os.path.join(outdir, f'Tab8_ROI1_Frame{idx + 1:04d}_{field_tag}.png')
    save_png(
        primary[idx], 'PROCESSED PRIMARY ROI',
        cmap, 'Intensity', ppath, markers=False
    )

    roi3 = _tab8_v7_get_roi3_stack(self)
    if roi3 is not None:
        rpath = os.path.join(outdir, f'Tab8_ROI3_Frame{idx + 1:04d}_{field_tag}.png')
        save_png(
            roi3[idx], 'ANALYSIS ROI',
            cmap, 'Intensity', rpath, markers=True
        )
        self.pd_status_var.set(
            f'Exported ROIs PNGs at 300 PPI: '
            f'{os.path.basename(ppath)}; {os.path.basename(rpath)}'
        )
    else:
        self.pd_status_var.set(
            f'Exported ROI PNG at 300 PPI: {os.path.basename(ppath)}'
        )

    try:
        os.startfile(outdir)
    except Exception:
        try:
            subprocess.Popen(['xdg-open', outdir])
        except Exception:
            pass
    return True


def _tab8_v7_export_figure(self):
    if not getattr(self, 'pd_results', None):
        messagebox.showwarning('Tab 8', 'No analyzed-frame results available.', parent=self.root)
        return False
    fp = filedialog.asksaveasfilename(
        parent=self.root, initialdir=self._export_initialdir(),
        title='Export Tab 8 analysis graphs',
        defaultextension='.png',
        filetypes=[('PNG', '*.png')]
    )
    if not fp:
        return False
    try:
        self.pd_plot_fig.savefig(fp, dpi=300, bbox_inches='tight')
        self.pd_status_var.set(f'Exported analysis graphs: {os.path.basename(fp)}')
        return True
    except Exception as exc:
        messagebox.showerror(
            'Tab 8 Export', f'Could not export analysis graphs:\n\n{type(exc).__name__}: {exc}',
            parent=self.root
        )
        return False


def _tab8_v7_reset_tab_state(self):
    """Reset Tab 8 in place; do not create or remove notebook tabs."""
    _tab8_v7_stop_playback(self, update_button=True)
    self._tab8_roi3_coords = None
    self._tab8_pending_roi3_coords = None
    self._tab8_roi3_stack = None
    self._tab8_roi3_shape = None
    self._tab8_roi3_xy_extent = None
    self._tab8_roi3_confirmed = False
    self._tab8_roi3_select_mode = False
    self._tab8_roi3_dragging = False
    self._tab8_roi3_start = None
    self._tab8_clear_display = False
    self._tab8_marks_cleared = False
    self.pd_results = {}
    self.pd_frame_detections = {}
    self.pd_current_detections = []
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self.pd_last_deleted = None
    self.pd_add_blob_mode = False
    _tab8_v7_restore_default_controls(self)

    primary = _tab8_v7_get_primary_stack(self)
    if primary is not None:
        self.pd_range_start_var.set(1)
        self.pd_range_end_var.set(int(primary.shape[0]))
        _tab8_v7_set_frame(self, int((primary.shape[0] - 1) // 2))
    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self, empty_message='Tab 8 reset — select ROI3 and analyze frames.')
    _tab8_v7_particle_number_size_plot(self)
    _tab8_v7_refresh_image(self)
    self.pd_status_var.set('TAB 8 RESET complete. Processed Primary ROI1 from Tab 6 retained.')
    return True


def _tab8_v7_build(self):
    """
    Build Tab 8 once. Existing notebook frame is reused; only its children
    are rebuilt, so Tabs 1–7 cannot be modified by this function.
    """
    tab = self.tab_particle_density

    _tab8_v7_stop_playback(self, update_button=False)
    for child in list(tab.winfo_children()):
        try:
            child.destroy()
        except Exception:
            pass

    # -------- State --------
    self._tab8_frame_var = tk.IntVar(value=0)
    self.pd_fps_var = tk.IntVar(value=10)

    self.pd_threshold_var = tk.DoubleVar(value=.25)
    self.pd_polarity_var = tk.StringVar(value='bright')
    self.pd_min_diam_var = tk.DoubleVar(value=10.0)
    self.pd_max_diam_var = tk.DoubleVar(value=60.0)
    self.pd_min_sep_var = tk.DoubleVar(value=25.0)
    self.pd_min_area_var = tk.DoubleVar(value=0.0)
    self.pd_max_area_var = tk.DoubleVar(value=1e6)
    self.pd_min_circularity_var = tk.DoubleVar(value=0.0)
    self.pd_max_circularity_var = tk.DoubleVar(value=1.0)
    self.pd_background_sigma_var = tk.DoubleVar(value=20.0)
    self.pd_median_size_var = tk.DoubleVar(value=3.0)
    self.pd_core_contrast_var = tk.DoubleVar(value=.55)
    self.pd_core_min_diam_var = tk.DoubleVar(value=4.0)
    self.pd_core_max_diam_var = tk.DoubleVar(value=40.0)
    self.pd_max_aspect_var = tk.DoubleVar(value=1.8)
    self.pd_include_edge_var = tk.BooleanVar(value=False)
    self.pd_show_centers_var = tk.BooleanVar(value=True)

    self.pd_range_start_var = tk.IntVar(value=1)
    self.pd_range_end_var = tk.IntVar(value=1)
    self.pd_status_var = tk.StringVar(
        value='Waiting for processed/finalized Primary ROI1 from Tab 6.'
    )
    self.pd_scale_status_var = tk.StringVar(value='Tab 6 Measurement Scale: --')
    self._tab8_frame_label_var = tk.StringVar(value='Frame --/-- | Field = --')

    self.pd_results = {}
    self.pd_frame_detections = {}
    self.pd_current_detections = []
    self.pd_selected_detection = None
    self.pd_multi_selected = set()
    self.pd_last_deleted = None
    self.pd_add_blob_mode = False

    self._tab8_play_job = None
    self._tab8_playing = False
    self._tab8_programmatic = False
    self._tab8_slider_initialized = False

    self._tab8_clear_display = False
    self._tab8_marks_cleared = False
    self._tab8_fixed_scale = None
    self._tab8_primary_stack = None
    self._tab8_primary_shape = None
    self._tab8_fixed_shape = None
    self._tab8_primary_xy_extent = None
    self._tab8_roi3_stack = None
    self._tab8_roi3_shape = None
    self._tab8_roi3_xy_extent = None
    self._tab8_roi3_coords = None
    self._tab8_pending_roi3_coords = None
    self._tab8_roi3_confirmed = False
    self._tab8_roi3_select_mode = False
    self._tab8_roi3_dragging = False
    self._tab8_roi3_start = None
    self._tab8_roi3_patch = None
    self.pd_geom_artists = []
    self.pd_label_marker_artists = []

    # -------- Layout --------
    tab.grid_rowconfigure(0, weight=1)
    tab.grid_columnconfigure(0, weight=1)

    main = ttk.Frame(tab, padding=5)
    main.grid(row=0, column=0, sticky='nsew')
    main.grid_rowconfigure(1, weight=1)
    main.grid_columnconfigure(0, weight=1)

    header = ttk.LabelFrame(main, text='FRAME CONTROL', padding=5)
    header.grid(row=0, column=0, sticky='ew', pady=(0, 4))
    header.grid_columnconfigure(1, weight=1)

    ttk.Label(
        header,
        text='BLOB Analysis— OUTER PARTICLE/INNER CORE/CORE SYMMETRY',
        font=('Arial', 11, 'bold')
    ).grid(row=0, column=0, columnspan=9, sticky='w')

    ttk.Label(
        header, text='Source: Processed / Finalized Primary ROI1 from Tab 6',
        foreground='darkgreen'
    ).grid(row=1, column=0, columnspan=2, sticky='w')

    ttk.Label(
        header, textvariable=self.pd_scale_status_var,
        foreground='navy'
    ).grid(row=1, column=2, columnspan=7, sticky='w', padx=10)

    ttk.Label(header, text='Frame').grid(row=2, column=0, sticky='w', pady=(4, 0))
    self._tab8_frame_slider = tk.Scale(
        header, from_=0, to=0, orient='horizontal',
        variable=self._tab8_frame_var, resolution=1,
        showvalue=True, highlightthickness=0, bd=1,
        length=460, takefocus=True, state='disabled',
        command=lambda v: _tab8_v7_frame_changed(self, v)
    )
    self._tab8_frame_slider.grid(row=2, column=1, columnspan=5, sticky='ew', padx=6, pady=(4, 0))
    # Button-release is an additional safety net for manual mouse dragging.
    self._tab8_frame_slider.bind(
        '<ButtonRelease-1>',
        lambda e: _tab8_v7_frame_changed(self),
        add='+'
    )

    self._tab8_frame_label = ttk.Label(
        header, textvariable=self._tab8_frame_label_var, width=27
    )
    self._tab8_frame_label.grid(row=2, column=6, sticky='w', padx=4)

    self.pd_first_button = ttk.Button(header, text='FIRST', command=lambda: _tab8_v7_first(self))
    self.pd_prev_button = ttk.Button(header, text='PREVIOUS', command=lambda: _tab8_v7_previous(self))
    self.pd_next_button = ttk.Button(header, text='NEXT', command=lambda: _tab8_v7_next(self))
    self.pd_play_button = ttk.Button(header, text='PLAY', command=lambda: _tab8_v7_play(self))
    self.pd_pause_button = ttk.Button(header, text='PAUSE', command=lambda: _tab8_v7_pause(self))
    for col, btn in enumerate((
        self.pd_first_button, self.pd_prev_button, self.pd_next_button,
        self.pd_play_button, self.pd_pause_button
    ), start=7):
        btn.grid(row=2, column=col, sticky='ew', padx=1, pady=(4, 0))

    ttk.Label(header, text='FPS').grid(row=3, column=0, sticky='w', pady=(3, 0))
    self.pd_fps_scale = tk.Scale(
        header, from_=1, to=60, resolution=1, orient='horizontal',
        variable=self.pd_fps_var, showvalue=True, highlightthickness=0,
        bd=1, length=220
    )
    self.pd_fps_scale.grid(row=3, column=1, columnspan=2, sticky='w', padx=6)

    # -------- Main horizontal split --------
    hp = ttk.Panedwindow(main, orient='horizontal')
    hp.grid(row=1, column=0, sticky='nsew')
    left = ttk.Frame(hp)
    right = ttk.Frame(hp)
    hp.add(left, weight=7)
    hp.add(right, weight=3)

    left.grid_rowconfigure(0, weight=1)
    left.grid_columnconfigure(0, weight=1)

    vis = ttk.LabelFrame(left, text='ROI + ANALYSIS GRAPHS', padding=2)
    vis.grid(row=0, column=0, sticky='nsew')
    vis.grid_rowconfigure(0, weight=1)
    vis.grid_columnconfigure(0, weight=1)

    self.pd_plot_fig = plt.Figure(figsize=(10, 15), dpi=100)
    gs = self.pd_plot_fig.add_gridspec(
        3, 2, height_ratios=[1.25, 0.8, 0.8],
        width_ratios=[1.0, 1.0], hspace=0.35, wspace=0.35
    )
    self.pd_ax_gray = self.pd_plot_fig.add_subplot(gs[0, 0])
    self.pd_ax_color = self.pd_plot_fig.add_subplot(gs[0, 1])
    self.pd_count_ax = self.pd_plot_fig.add_subplot(gs[1, 0])
    self.pd_size_ax = self.pd_plot_fig.add_subplot(gs[1, 1])
    self.pd_particle_size_ax = self.pd_plot_fig.add_subplot(gs[2, :])
    self.pd_density_ax = self.pd_count_ax.twinx()

    self.pd_image_artist_gray = self.pd_ax_gray.imshow(
        np.zeros((2, 2)), cmap='gray', origin='lower',
        vmin=0, vmax=1, interpolation='nearest'
    )
    self.pd_image_artist_color = self.pd_ax_color.imshow(
        np.zeros((2, 2)), cmap='viridis', origin='lower',
        vmin=0, vmax=1, interpolation='nearest'
    )
    self.pd_colorbar = self.pd_plot_fig.colorbar(
        self.pd_image_artist_color, ax=self.pd_ax_color,
        fraction=.046, pad=.04
    )
    self.pd_colorbar.set_label('Intensity', fontsize=9)

    for ax in (self.pd_ax_gray, self.pd_ax_color):
        ax.set_xlabel('X (nm)')
        ax.set_ylabel('Y (nm)')
        ax.set_aspect('equal', adjustable='box')

    self.pd_particle_size_ax.set_xlabel('Particle No.')
    self.pd_particle_size_ax.set_ylabel('Particle Size (nm)')
    self.pd_particle_size_ax.set_title('Particle No. vs Particle Size', fontsize=10)
    self.pd_particle_size_ax.grid(True, alpha=.2)
    self.pd_plot_fig.subplots_adjust(left=.075, right=.91, bottom=.06, top=.95, wspace=.28, hspace=.46)

    # The large figure lives in a scrollable Tk canvas; this avoids squashing
    # the three rows of plots into an unreadable area.
    fig_wrap = tk.Frame(vis)
    fig_wrap.grid(row=0, column=0, sticky='nsew')
    fig_wrap.grid_rowconfigure(0, weight=1)
    fig_wrap.grid_columnconfigure(0, weight=1)
    fc = tk.Canvas(fig_wrap, highlightthickness=0)
    fvs = ttk.Scrollbar(fig_wrap, orient='vertical', command=fc.yview)
    fhs = ttk.Scrollbar(fig_wrap, orient='horizontal', command=fc.xview)
    fc.configure(yscrollcommand=fvs.set, xscrollcommand=fhs.set)
    fc.grid(row=0, column=0, sticky='nsew')
    fvs.grid(row=0, column=1, sticky='ns')
    fhs.grid(row=1, column=0, sticky='ew')
    fh = tk.Frame(fc, width=1120, height=1450)
    fw = fc.create_window((0, 0), window=fh, anchor='nw')
    fh.bind('<Configure>', lambda e: fc.configure(scrollregion=fc.bbox('all')))
    fc.bind('<Configure>', lambda e: fc.itemconfigure(fw, width=max(e.width, 1120)))

    self.pd_canvas = FigureCanvasTkAgg(self.pd_plot_fig, master=fh)
    self.pd_canvas.draw()
    self.pd_canvas.get_tk_widget().pack(fill='both', expand=True)

    # Matplotlib event handlers are attached once per build and disconnected
    # when this tab is rebuilt. ROI3 drag occurs only on the gray Primary ROI1.
    self._tab8_roi3_press_cid = self.pd_canvas.mpl_connect(
        'button_press_event', lambda e: _tab8_v7_select_roi3_press(self, e)
    )
    self._tab8_roi3_motion_cid = self.pd_canvas.mpl_connect(
        'motion_notify_event', lambda e: _tab8_v7_select_roi3_motion(self, e)
    )
    self._tab8_roi3_release_cid = self.pd_canvas.mpl_connect(
        'button_release_event', lambda e: _tab8_v7_select_roi3_release(self, e)
    )
    self._tab8_select_cid = self.pd_canvas.mpl_connect(
        'button_press_event', lambda e: _tab8_v7_select(self, e)
    )

    # -------- Right side: actions + parameters + results --------
    right.grid_rowconfigure(0, weight=1)
    right.grid_columnconfigure(0, weight=1)
    vp = ttk.Panedwindow(right, orient='vertical')
    vp.grid(row=0, column=0, sticky='nsew')
    upper = ttk.Frame(vp)
    lower = ttk.Frame(vp)
    vp.add(upper, weight=3)
    vp.add(lower, weight=2)
    upper.grid_rowconfigure(1, weight=1)
    upper.grid_columnconfigure(0, weight=1)

    actions = ttk.LabelFrame(upper, text='ACTION PANEL', padding=5)
    actions.grid(row=0, column=0, sticky='ew', pady=(0, 4))
    for col in range(3):
        actions.grid_columnconfigure(col, weight=1, uniform='tab8action')

    def ab(row, col, text, cmd, span=1):
        b = ttk.Button(actions, text=text, command=cmd)
        b.grid(row=row, column=col, columnspan=span, sticky='ew', padx=2, pady=2, ipady=2)
        return b

    ab(0, 0, 'PRIMARY ROI FROM TAB 6',
       lambda: _tab8_v7_sync_source(self, preserve_current=True), 3)
    self.pd_roi3_select_button = ab(1, 0, 'SELECT ROI',
       lambda: _tab8_v7_toggle_roi3_mode(self))
    self.pd_confirm_roi3_button = ab(1, 1, 'CONFIRM ROI',
       lambda: _tab8_v7_confirm_roi3(self))
    self.pd_clear_roi3_button = ab(1, 2, 'CLEAR ROI',
       lambda: _tab8_v7_clear_roi3(self))

    ab(2, 1, 'DELETE SELECTED', lambda: _tab8_v7_delete_selected(self))
    ab(2, 2, 'UNDO DELETE', lambda: _tab8_v7_undo_delete(self))
    ab(4, 0, 'CLEAR MARK', lambda: _tab8_v7_clear_mark(self))

    ab(3, 0, 'ANALYZE CURRENT FRAME', lambda: _tab8_v7_analyze_current(self))
    ab(3, 1, 'ANALYZE SELECTED FRAMES', lambda: _tab8_v7_analyze_selected(self))
    ab(3, 2, 'RESET', lambda: _tab8_v7_reset_analysis(self))

    self.pd_add_blob_button = ab(2, 0, 'ADD BLOB', lambda: _tab8_v7_set_add_mode(self))
    ab(4, 1, 'EXPORT IMAGE', lambda: _tab8_v7_export_image(self))
    ab(4, 2, 'EXPORT TABLE — CSV', lambda: _tab8_v7_export_table(self))

    ab(5, 0, 'EXPORT ANALYSIS GRAPHS', lambda: _tab8_v7_export_figure(self))
    ab(5, 1, 'EXPORT FIGURE 5 — CSV', lambda: _tab8_v7_export_figure5_csv(self))
    ab(5, 2, 'DEFAULT', lambda: _tab8_v7_restore_default_controls(self))

    range_row = ttk.Frame(actions)
    range_row.grid(row=6, column=0, columnspan=3, sticky='ew', pady=(5, 0))
    ttk.Label(range_row, text='START FRAME').pack(side='left')
    self.pd_range_start_spin = tk.Spinbox(
        range_row, from_=1, to=1, width=7, textvariable=self.pd_range_start_var
    )
    self.pd_range_start_spin.pack(side='left', padx=(4, 12))
    ttk.Label(range_row, text='END FRAME').pack(side='left')
    self.pd_range_end_spin = tk.Spinbox(
        range_row, from_=1, to=1, width=7, textvariable=self.pd_range_end_var
    )
    self.pd_range_end_spin.pack(side='left', padx=4)

    # Scrollable detection controls.
    sh = ttk.Frame(upper)
    sh.grid(row=1, column=0, sticky='nsew')
    sh.grid_rowconfigure(0, weight=1)
    sh.grid_columnconfigure(0, weight=1)
    cv = tk.Canvas(sh, highlightthickness=0)
    sb = ttk.Scrollbar(sh, orient='vertical', command=cv.yview)
    inner = ttk.Frame(cv, padding=(4, 2, 10, 8))
    win = cv.create_window((0, 0), window=inner, anchor='nw')
    cv.configure(yscrollcommand=sb.set)
    cv.grid(row=0, column=0, sticky='nsew')
    sb.grid(row=0, column=1, sticky='ns')
    inner.bind('<Configure>', lambda e: cv.configure(scrollregion=cv.bbox('all')))
    cv.bind('<Configure>', lambda e: cv.itemconfigure(win, width=e.width))

    def wheel(e):
        delta = getattr(e, 'delta', 0)
        num = getattr(e, 'num', None)
        step = -3 if (num == 4 or delta > 0) else 3
        try:
            cv.yview_scroll(step, 'units')
        except Exception:
            pass
        return 'break'
    for wdg in (cv, inner):
        wdg.bind('<MouseWheel>', wheel, add='+')
        wdg.bind('<Button-4>', wheel, add='+')
        wdg.bind('<Button-5>', wheel, add='+')

    detbox = ttk.LabelFrame(inner, text='DETECTION / ANALYSIS PARAMETERS', padding=6)
    detbox.pack(fill='x', pady=(0, 5))

    def scl(label, var, lo, hi, res):
        ttk.Label(detbox, text=label).pack(anchor='w')
        s = tk.Scale(
            detbox, from_=lo, to=hi, resolution=res,
            orient='horizontal', variable=var, length=255,
            showvalue=True, highlightthickness=0, bd=1, takefocus=True
        )
        s.pack(fill='x')
        return s

    ttk.Label(detbox, text='OUTER PARTICLE', font=('Arial', 10, 'bold')).pack(anchor='w')
    scl('Detection threshold', self.pd_threshold_var, .01, .9, .01)
    ttk.Label(detbox, text='Polarity').pack(anchor='w')
    ttk.Combobox(
        detbox, textvariable=self.pd_polarity_var,
        values=('bright', 'dark', 'both'), state='readonly', width=10
    ).pack(anchor='w', pady=(0, 2))
    scl('Minimum outer diameter (px)', self.pd_min_diam_var, 3, 100, 1)
    scl('Maximum outer diameter (px)', self.pd_max_diam_var, 5, 180, 1)
    scl('Minimum separation (px)', self.pd_min_sep_var, 1, 150, 1)

    ttk.Separator(detbox).pack(fill='x', pady=4)
    ttk.Label(detbox, text='SHAPE / FILTER GATES', font=('Arial', 10, 'bold')).pack(anchor='w')
    scl('Minimum outer area (px²)', self.pd_min_area_var, 0, 5000, 10)
    scl('Maximum outer area (px²)', self.pd_max_area_var, 100, 20000, 100)
    scl('Minimum outer circularity', self.pd_min_circularity_var, 0, 1, .01)
    scl('Maximum outer circularity', self.pd_max_circularity_var, 0, 1, .01)
    scl('Background smoothing σ (px)', self.pd_background_sigma_var, 3, 80, 1)
    scl('Median filter size (px)', self.pd_median_size_var, 1, 9, 2)

    ttk.Separator(detbox).pack(fill='x', pady=4)
    ttk.Label(detbox, text='INNER CORE + SYMMETRY', font=('Arial', 10, 'bold')).pack(anchor='w')
    scl('Core contrast level (relative)', self.pd_core_contrast_var, .10, .95, .01)
    scl('Core minimum diameter (px)', self.pd_core_min_diam_var, 1, 40, 1)
    scl('Core maximum diameter (px)', self.pd_core_max_diam_var, 5, 80, 1)
    scl('Maximum core aspect ratio', self.pd_max_aspect_var, 1, 3, .01)
    ttk.Checkbutton(detbox, text='Include edge detections',
                    variable=self.pd_include_edge_var).pack(anchor='w')
    ttk.Checkbutton(
        detbox, text='Show outer/core overlays + particle numbers',
        variable=self.pd_show_centers_var,
        command=lambda: _tab8_v7_refresh_image(self)
    ).pack(anchor='w')

    self.pd_particle_status = ttk.Label(
        detbox, text='No particle selected.', justify='left', wraplength=285
    )
    self.pd_particle_status.pack(anchor='w', pady=(4, 2))
    ttk.Label(
        detbox, textvariable=self.pd_status_var,
        justify='left', wraplength=285
    ).pack(anchor='w', pady=2)

    # Results table.
    tablebox = ttk.LabelFrame(lower, text='DENSITY / PARTICLE RESULTS', padding=3)
    tablebox.pack(fill='both', expand=True)
    tf = ttk.Frame(tablebox)
    tf.pack(fill='both', expand=True)
    tf.grid_rowconfigure(0, weight=1)
    tf.grid_columnconfigure(0, weight=1)

    cols = (
        'Frame', 'Field (mT)', 'Count', 'ROI area (µm²)',
        'Density (skyrmions/µm²)', 'Mean outer d (nm)',
        'Mean core d (nm)', 'Mean core AR',
        'Outer-particle boundary/size (nm)',
        'STD outer boundary/size (nm)'
    )
    self.pd_table = ttk.Treeview(tf, columns=cols, show='headings')
    widths = [55, 88, 55, 105, 130, 110, 110, 90, 170, 155]
    for col, width in zip(cols, widths):
        self.pd_table.heading(col, text=col)
        self.pd_table.column(col, width=width, minwidth=55, anchor='center')
    ysb = ttk.Scrollbar(tf, orient='vertical', command=self.pd_table.yview)
    xsb = ttk.Scrollbar(tablebox, orient='horizontal', command=self.pd_table.xview)
    self.pd_table.configure(yscrollcommand=ysb.set, xscrollcommand=xsb.set)
    self.pd_table.grid(row=0, column=0, sticky='nsew')
    ysb.grid(row=0, column=1, sticky='ns')
    xsb.pack(fill='x')

    # Source fetch occurs after every Tab-8 widget exists.
    _tab8_v7_sync_source(self, preserve_current=True)
    if _tab8_v7_get_primary_stack(self) is None:
        self._tab8_frame_slider.configure(state='disabled')
    else:
        st = _tab8_v7_get_primary_stack(self)
        self._tab8_frame_slider.configure(from_=0, to=st.shape[0] - 1, state='normal')

    _tab8_v7_refresh_table(self)
    _tab8_v7_plot(self)
    _tab8_v7_particle_number_size_plot(self)
    _tab8_v7_refresh_image(self)
    return True


def _tab8_v7_toggle_roi3_mode(self):
    active = not bool(getattr(self, '_tab8_roi3_select_mode', False))
    self._tab8_roi3_select_mode = active
    self._tab8_roi3_dragging = False
    self._tab8_roi3_start = None
    try:
        self.pd_roi3_select_button.configure(
            text='CANCEL ROI3' if active else 'SELECT ROI3'
        )
    except Exception:
        pass
    self.pd_status_var.set(
        'ROI3 selection active — drag a rectangle on Primary ROI1.'
        if active else
        'ROI3 selection cancelled.'
    )


# Compatibility names used by the existing TAB RESET mechanism and any
# previous internal Tab-8 callbacks.
_tab8_v6_scales = _tab8_v7_get_scale
_tab8_v6_stack = _tab8_v7_get_roi3_stack
_tab8_v6_field = _tab8_v7_get_field
_tab8_v6_equiv_scale = _tab8_v7_equiv_scale
_tab8_v6_set_axes_fixed = _tab8_v7_apply_fixed_axes
_tab8_v6_clear_artists = _tab8_v7_clear_artists
_tab8_v6_draw_overlays = _tab8_v7_draw_overlays
_tab8_v6_refresh_image = _tab8_v7_refresh_image
_tab8_v6_frame_changed = _tab8_v7_frame_changed
_tab8_v6_select = _tab8_v7_select
_tab8_v6_delete_selected = _tab8_v7_delete_selected
_tab8_v6_undo_delete = _tab8_v7_undo_delete
_tab8_v6_run_detection = _tab8_v7_run_detection
_tab8_v6_refresh_density_row = _tab8_v7_refresh_density_row
_tab8_v6_refresh_table = _tab8_v7_refresh_table
_tab8_v6_plot = _tab8_v7_plot
_tab8_v6_particle_number_size_plot = _tab8_v7_particle_number_size_plot
_tab8_v6_analyze_current = _tab8_v7_analyze_current
_tab8_v6_analyze_selected = _tab8_v7_analyze_selected
_tab8_v6_clear = _tab8_v7_clear
_tab8_v6_clear_mark = _tab8_v7_clear_mark
_tab8_v6_reset_analysis = _tab8_v7_reset_analysis
_tab8_v6_export_image = _tab8_v7_export_image
_tab8_v6_export_table = _tab8_v7_export_table
_tab8_v6_export_figure5_csv = _tab8_v7_export_figure5_csv
_tab8_v6_export_figure = _tab8_v7_export_figure
_tab8_v6_restore_default_controls = _tab8_v7_restore_default_controls
_tab8_v6_confirm_roi3 = _tab8_v7_confirm_roi3
_tab8_v6_clear_roi3 = _tab8_v7_clear_roi3
_tab8_v6_toggle_roi3_mode = _tab8_v7_toggle_roi3_mode
_tab8_v6_reset_tab_state = _tab8_v7_reset_tab_state
_tab8_v6_toggle_add_blob = _tab8_v7_set_add_mode
_tab8_v6_build = _tab8_v7_build

def _tab8_v7_notebook_changed(self, event=None):
    try:
        selected = self.nb.select()
        if selected == str(self.tab_particle_density):
            self.root.after_idle(lambda: _tab8_v7_ensure_ready(self))
    except Exception:
        pass


def _tab8_v7_ensure_ready(self):
    """Refresh source/slider when entering Tab 8, without rebuilding other tabs."""
    try:
        st = _tab8_v7_get_primary_stack(self)
        if st is None:
            return bool(_tab8_v7_sync_source(self, preserve_current=True))
        n = st.shape[0]
        idx = max(0, min(int(self._tab8_frame_var.get()), n - 1))
        self._tab8_frame_slider.configure(from_=0, to=n - 1, resolution=1, state='normal')
        self._tab8_frame_var.set(idx)
        _tab8_v7_refresh_image(self)
        return True
    except Exception as exc:
        try:
            self.pd_status_var.set(f'Tab 8 refresh failed: {type(exc).__name__}: {exc}')
        except Exception:
            pass
        return False


# Bind the authoritative methods expected by the existing application.
for _name, _fn in {
    '_tab8_scales': _tab8_v7_get_scale,
    '_tab8_stack': _tab8_v7_get_roi3_stack,
    '_tab8_field': _tab8_v7_get_field,
    '_tab8_equiv_scale': _tab8_v7_equiv_scale,
    '_tab8_apply_fixed_axes': _tab8_v7_apply_fixed_axes,
    '_tab8_remove_artists': _tab8_v7_clear_artists,
    '_tab8_draw_overlay': _tab8_v7_draw_overlays,
    '_tab8_refresh_image': _tab8_v7_refresh_image,
    '_tab8_frame_changed': _tab8_v7_frame_changed,
    '_tab8_select': _tab8_v7_select,
    '_tab8_delete_selected': _tab8_v7_delete_selected,
    '_tab8_undo_delete': _tab8_v7_undo_delete,
    '_tab8_run_detection': _tab8_v7_run_detection,
    '_tab8_refresh_density_row': _tab8_v7_refresh_density_row,
    '_tab8_refresh_table': _tab8_v7_refresh_table,
    '_tab8_plot': _tab8_v7_plot,
    '_tab8_analyze_current': _tab8_v7_analyze_current,
    '_tab8_analyze_selected': _tab8_v7_analyze_selected,
    '_tab8_clear': _tab8_v7_clear,
    '_tab8_clear_mark': _tab8_v7_clear_mark,
    '_tab8_reset_analysis': _tab8_v7_reset_analysis,
    '_tab8_export_image': _tab8_v7_export_image,
    '_tab8_export_table': _tab8_v7_export_table,
    '_tab8_export_figure5_csv': _tab8_v7_export_figure5_csv,
    '_tab8_export_figure': _tab8_v7_export_figure,
    '_tab8_restore_default_controls': _tab8_v7_restore_default_controls,
    '_tab8_confirm_roi3': _tab8_v7_confirm_roi3,
    '_tab8_clear_roi3': _tab8_v7_clear_roi3,
    '_tab8_roi3_select': _tab8_v7_toggle_roi3_mode,
    '_tab8_add_blob': _tab8_v7_set_add_mode,
    '_tab8_play': _tab8_v7_play,
    '_tab8_pause': _tab8_v7_pause,
    '_tab8_first': _tab8_v7_first,
    '_tab8_previous': _tab8_v7_previous,
    '_tab8_next': _tab8_v7_next,
}.items():
    setattr(CDIWorkflowApp, _name, _fn)

CDIWorkflowApp._tab8_roi3_press_handler = _tab8_v7_select_roi3_press
CDIWorkflowApp._tab8_roi3_motion_handler = _tab8_v7_select_roi3_motion
CDIWorkflowApp._tab8_roi3_release_handler = _tab8_v7_select_roi3_release
CDIWorkflowApp._tab8_sync_source = _tab8_v7_sync_source
CDIWorkflowApp._tab8_cleanup_duplicate_tabs = lambda self: self.tab_particle_density
CDIWorkflowApp._tab8_ensure_ready_on_enter = _tab8_v7_ensure_ready


# Replace only the final Tab-8 UI wrapper. Tabs 1–7 are built by the existing
# base builder and are not modified here.
_BASE_BUILD_UI_TAB8_REBUILT = CDIWorkflowApp._build_ui

def _build_ui_tab8_rebuilt(self, *args, **kwargs):
    result = _BASE_BUILD_UI_TAB8_REBUILT(self, *args, **kwargs)
    try:
        # The base UI creates Tabs 1–7 only in this version. Create Tab 8
        # explicitly before building its widgets. Add it exactly once at index 7.
        if not hasattr(self, 'tab_particle_density') or str(self.tab_particle_density) not in self.nb.tabs():
            self.tab_particle_density = ttk.Frame(self.nb)
            self.nb.insert(7, self.tab_particle_density, text='8. Blob Analysis')
        else:
            self.nb.tab(self.tab_particle_density, text='8. Blob Analysis')
        _tab8_v7_build(self)

        # Reinstall the Tab-8 notebook-enter hook once.
        bind_id = getattr(self, '_tab8_rebuilt_notebook_bind_id', None)
        if bind_id is None:
            self._tab8_rebuilt_notebook_bind_id = self.nb.bind(
                '<<NotebookTabChanged>>',
                lambda e: _tab8_v7_notebook_changed(self, e),
                add='+'
            )

        self.root.after_idle(lambda: _tab8_v7_ensure_ready(self))
    except Exception as exc:
        try:
            self.log(f'Tab 8 initialization warning: {type(exc).__name__}: {exc}')
        except Exception:
            pass
    return result

CDIWorkflowApp._build_ui = _build_ui_tab8_rebuilt


# Keep Tab 6 processing intact. After Tab 6 applies its finalized Primary ROI
# resolution, refresh only Tab 8's source cache/slider.
try:
    _TAB6_APPLY_PRIMARY_BEFORE_TAB8_REBUILT = CDIWorkflowApp.apply_same_confirmed_roi_resolution

    def _tab6_apply_primary_refresh_tab8_rebuilt(self, *args, **kwargs):
        result = _TAB6_APPLY_PRIMARY_BEFORE_TAB8_REBUILT(self, *args, **kwargs)
        try:
            if hasattr(self, 'pd_frame_slider'):
                _tab8_v7_sync_source(self, preserve_current=True)
        except Exception as exc:
            try:
                self.pd_status_var.set(
                    f'Tab 8 refresh from Tab 6 failed: {type(exc).__name__}: {exc}'
                )
            except Exception:
                pass
        return result

    CDIWorkflowApp.apply_same_confirmed_roi_resolution = _tab6_apply_primary_refresh_tab8_rebuilt
except Exception:
    pass


# ---------------------------------------------------------------------------
# TAB 9 — PRIMARY ROI1 -> ROI4
# ---------------------------------------------------------------------------
# Tab 9 is independent of the old Tab-6 Secondary ROI/ROI2 workflow.
# It opens the processed Primary ROI1, lets the user draw/confirm/clear ROI4,
# displays ROI1 and ROI4 side-by-side, keeps X/Y axes in physical nm, and
# exports ROI4 only.

def _tab9_roi4_scale_nm(self):
    """Return the current processed Primary ROI output scale in nm/pixel."""
    try:
        fn = getattr(self, '_tab6_effective_nm_per_output_px', None)
        if callable(fn):
            v = fn()
            if v is not None:
                sx, sy = float(v[0]), float(v[1])
                if np.isfinite(sx) and np.isfinite(sy) and sx > 0 and sy > 0:
                    return sx, sy
    except Exception:
        pass

    # Fallback to the final Primary ROI calibration from Tab 5.
    try:
        v = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
        if np.isfinite(v) and v > 0:
            return v, v
    except Exception:
        pass
    return None, None




def _tab9_roi4_fetch_from_tab6(self, preserve_frame=True):
    """
    Fetch the exact Tab-6 ROI-processed Primary ROI1 stack. Keep the
    Tab9->Tab10 handoff attributes stable.
    """
    try:
        if getattr(self,'recons',None) is None or getattr(self,'roi_stack',None) is None:
            raise RuntimeError('Tab 6 ROI stack is not available.')
        sx,sy=_tab6_effective_nm_per_output_px(self)
        sx,sy=float(sx),float(sy)
        if not (np.isfinite(sx) and np.isfinite(sy) and sx>0 and sy>0):
            raise RuntimeError('Invalid Tab 6 output scale.')
        processed=_tab6_build_exact_processed_primary_stack(self)
    except Exception as exc:
        self._tab9_roi1_ready=False
        try:
            self.tab9_roi4_status_var.set(
                f'Could not fetch exact processed Primary ROI1 from Tab 6: '
                f'{type(exc).__name__}: {exc}'
            )
            self.tab9_roi4_slider.configure(
                from_=0,to=0,state='disabled'
            )
            self.tab9_roi4_play_button.configure(
                text='PLAY',state='disabled'
            )
            self.tab9_roi4_first_button.configure(state='disabled')
        except Exception:
            pass
        return False

    if processed is None or processed.ndim!=3 or processed.shape[0]<1:
        self._tab9_roi1_ready=False
        self.tab9_roi4_status_var.set(
            'Exact processed Primary ROI1 stack from Tab 6 is invalid.'
        )
        return False

    n,h,w=map(int,processed.shape)
    try:
        old_idx=int(self.tab9_roi4_frame_var.get()) if preserve_frame else 0
    except Exception:
        old_idx=0
    old_idx=max(0,min(old_idx,n-1))

    # Immutable fixed geometry.
    self._tab9_roi1_stack=processed.copy()
    self._tab9_roi1_stack_ref=self._tab9_roi1_stack
    self._tab9_roi1_shape=(h,w)
    self._tab9_roi1_extent=(0.0,w*sx,0.0,h*sy)
    self._tab9_roi4_scale=(sx,sy)
    self._tab9_roi1_ready=True
    self._tab9_roi1_cmap=str(
        getattr(self,'colormap_var',tk.StringVar(value='gray')).get()
        or 'gray'
    )
    self._tab9_roi1_clim=(0.0,1.0)

    # IMPORTANT: the confirmed ROI4 coordinate system stays tied to this
    # fixed h×w geometry; frame changes cannot resize it.
    c=getattr(self,'_tab9_roi4_coords',None)
    if isinstance(c,dict):
        try:
            valid=(
                0<=int(c['xmin'])<=int(c['xmax'])<w
                and 0<=int(c['ymin'])<=int(c['ymax'])<h
            )
        except Exception:
            valid=False
        if not valid:
            self._tab9_roi4_coords=None
            self._tab9_roi4_preview_coords=None
            self._tab9_roi4_confirmed=False
            self._tab9_roi4_stack=None
            self._tab9_roi4_stack_coords=None
            self._tab9_roi4_stack_shape=None

    self._tab9_roi4_programmatic=True
    try:
        self.tab9_roi4_slider.configure(
            from_=0,to=n-1,resolution=1,state='normal'
        )
        self.tab9_roi4_start_spin.configure(from_=1, to=n, increment=1)
        self.tab9_roi4_end_spin.configure(from_=1, to=n, increment=1)
        self.tab9_roi4_start_var.set(1)
        self.tab9_roi4_end_var.set(n)
        try:
            # Export controls and slider use the exact same loaded-frame count.
            self.tab9_roi4_slider.configure(
                from_=0, to=max(0,n-1), resolution=1, state='normal'
            )
        except Exception:
            pass
        self.tab9_roi4_frame_var.set(old_idx)
        self.tab9_roi4_slider.set(old_idx)
    finally:
        self._tab9_roi4_programmatic=False

    if not getattr(self,'_tab9_slider_initialized',False):
        mid=(n-1)//2
        self._tab9_roi4_programmatic=True
        try:
            self.tab9_roi4_frame_var.set(mid)
            self.tab9_roi4_slider.set(mid)
            old_idx=mid
        finally:
            self._tab9_roi4_programmatic=False
        self._tab9_slider_initialized=True

    _tab9_roi4_show_frame(self,old_idx,preserve_zoom=True)

    try:
        self.tab9_roi4_play_button.configure(
            text='PLAY',state='normal'
        )
        self.tab9_roi4_first_button.configure(state='normal')
    except Exception:
        pass

    self.tab9_roi4_status_var.set(
        f'✓ Exact processed Primary ROI1 fetched from Tab 6 | '
        f'FOV = {w*sx:.6g} × {h*sy:.6g} nm | '
        f'Frame {old_idx+1}/{n}'
    )
    return True




def _tab9_roi4_refresh(self, preserve_frame=True):
    return _tab9_roi4_fetch_from_tab6(
        self, preserve_frame=preserve_frame
    )


def _tab9_roi4_get_current_processed(self, idx):
    if not getattr(self, '_tab9_roi1_ready', False):
        if not _tab9_roi4_fetch_from_tab6(
            self, preserve_frame=True
        ):
            return None
    try:
        stack = np.asarray(
            self._tab9_roi1_stack, dtype=np.float32
        )
        ii = max(0, min(int(idx), stack.shape[0] - 1))
        return stack[ii]
    except Exception as exc:
        try:
            self.tab9_roi4_status_var.set(
                f'Unable to read Tab 6 processed Primary ROI1: {exc}'
            )
        except Exception:
            pass
        return None


def _tab9_roi4_draw_rect(self, confirmed=False, redraw=True):
    # Always remove the previous ROI rectangle before drawing a new one.
    try:
        old = getattr(self, '_tab9_roi4_rect', None)
        if old is not None:
            old.remove()
    except Exception:
        pass
    self._tab9_roi4_rect = None

    c = getattr(self, '_tab9_roi4_coords', None)
    if not isinstance(c, dict):
        if redraw:
            self.tab9_roi4_canvas.draw_idle()
        return

    try:
        sx, sy = self._tab9_roi4_scale
        xmin = int(c['xmin']); xmax = int(c['xmax'])
        ymin = int(c['ymin']); ymax = int(c['ymax'])
        rect = plt.Rectangle(
            (xmin*sx, ymin*sy),
            (xmax-xmin+1)*sx,
            (ymax-ymin+1)*sy,
            fill=False,
            edgecolor='lime' if confirmed else 'red',
            linewidth=2.5,
            zorder=30
        )
        self._tab9_roi4_rect = rect
        self.tab9_roi4_ax1.add_patch(rect)
    except Exception:
        self._tab9_roi4_rect = None

    if redraw:
        self.tab9_roi4_canvas.draw_idle()


def _tab9_roi4_start_selection(self):
    """Enter a fresh drag-selection state without destroying the source."""
    if not getattr(self, '_tab9_roi1_ready', False):
        if not _tab9_roi4_fetch_from_tab6(self, preserve_frame=True):
            return False

    self._tab9_roi4_selecting = True
    self._tab9_roi4_press_xy = None
    self._tab9_roi4_preview_coords = None
    self._tab9_roi4_coords = None
    self._tab9_roi4_confirmed = False
    self._tab9_roi4_stack = None
    self._tab9_roi4_stack_coords = None
    self._tab9_roi4_stack_shape = None

    try:
        self._tab9_roi4_rect.remove()
    except Exception:
        pass
    self._tab9_roi4_rect = None

    # Clear only ROI4 display; never touch Primary ROI1.
    self.tab9_roi4_image2.set_data(
        np.zeros((10,10), dtype=float)
    )
    self.tab9_roi4_image2.set_extent((0,10,0,10))
    self.tab9_roi4_ax2.set_autoscale_on(False)
    self.tab9_roi4_ax2.set_xlim(0,10)
    self.tab9_roi4_ax2.set_ylim(0,10)
    self.tab9_roi4_ax2.set_aspect('equal', adjustable='box')
    self.tab9_roi4_ax2.set_title(
        'ROI — SELECTED AREA PENDING CONFIRMATION',
        fontsize=11
    )
    self.tab9_roi4_cb2.update_normal(self.tab9_roi4_image2)

    self.tab9_roi4_status_var.set(
        'ROI4 selection active — drag a rectangle on Processed Primary ROI1, then CONFIRM ROI.'
    )
    try:
        self.tab9_roi4_select_button.configure(
            text='DRAG ROI', state='normal'
        )
        self.tab9_roi4_confirm_button.configure(
            text='✓ CONFIRM ROI', state='normal'
        )
        self.tab9_roi4_canvas.get_tk_widget().configure(
            cursor='crosshair'
        )
        self.tab9_roi4_canvas.get_tk_widget().focus_set()
    except Exception:
        pass
    self.tab9_roi4_canvas.draw_idle()
    return True


def _tab9_roi4_on_press(self, event):
    if not getattr(self, '_tab9_roi4_selecting', False):
        return
    if (
        event.inaxes is not self.tab9_roi4_ax1
        or event.xdata is None
        or event.ydata is None
    ):
        return
    self._tab9_roi4_press_xy = (
        float(event.xdata), float(event.ydata)
    )


def _tab9_roi4_on_move(self, event):
    if (
        not getattr(self, '_tab9_roi4_selecting', False)
        or getattr(self, '_tab9_roi4_press_xy', None) is None
        or event.inaxes is not self.tab9_roi4_ax1
        or event.xdata is None
        or event.ydata is None
    ):
        return

    x0, y0 = self._tab9_roi4_press_xy
    x1, y1 = float(event.xdata), float(event.ydata)

    sx, sy = self._tab9_roi4_scale
    h, w = self._tab9_roi1_shape

    xmin_nm, xmax_nm = sorted((x0,x1))
    ymin_nm, ymax_nm = sorted((y0,y1))

    xmin = int(np.floor(xmin_nm/sx))
    xmax = int(np.ceil(xmax_nm/sx))-1
    ymin = int(np.floor(ymin_nm/sy))
    ymax = int(np.ceil(ymax_nm/sy))-1

    xmin=max(0,min(xmin,w-1)); xmax=max(0,min(xmax,w-1))
    ymin=max(0,min(ymin,h-1)); ymax=max(0,min(ymax,h-1))

    if xmax <= xmin or ymax <= ymin:
        return

    self._tab9_roi4_preview_coords = {
        'xmin':xmin,'xmax':xmax,
        'ymin':ymin,'ymax':ymax
    }

    # Preview rectangle only. Final confirmation creates the ROI4 stack.
    try:
        if getattr(self,'_tab9_roi4_rect',None) is not None:
            self._tab9_roi4_rect.remove()
    except Exception:
        pass

    self._tab9_roi4_rect=plt.Rectangle(
        (xmin*sx,ymin*sy),
        (xmax-xmin+1)*sx,
        (ymax-ymin+1)*sy,
        fill=False, edgecolor='red',
        linewidth=2.5,zorder=30
    )
    self.tab9_roi4_ax1.add_patch(self._tab9_roi4_rect)
    self.tab9_roi4_canvas.draw_idle()


def _tab9_roi4_on_release(self, event):
    if not getattr(self,'_tab9_roi4_selecting',False):
        return

    self._tab9_roi4_selecting=False
    self._tab9_roi4_press_xy=None

    try:
        self.tab9_roi4_canvas.get_tk_widget().configure(cursor='')
    except Exception:
        pass

    c=getattr(self,'_tab9_roi4_preview_coords',None)
    if not isinstance(c,dict):
        self.tab9_roi4_status_var.set(
            'ROI4 selection cancelled. Drag a valid rectangle and try again.'
        )
        return

    self._tab9_roi4_coords=dict(c)
    self._tab9_roi4_preview_coords=None
    self._tab9_roi4_confirmed=False

    try:
        self.tab9_roi4_select_button.configure(
            text='SELECT ROI',state='normal'
        )
        self.tab9_roi4_confirm_button.configure(
            text='✓ CONFIRM ROI',state='normal'
        )
    except Exception:
        pass

    # Show the currently selected crop immediately for visual verification.
    idx=int(self.tab9_roi4_frame_var.get())
    _tab9_roi4_show_frame(
        self, idx, preserve_zoom=True
    )
    try:
        sx,sy=self._tab9_roi4_scale
        rw=(c['xmax']-c['xmin']+1)*sx
        rh=(c['ymax']-c['ymin']+1)*sy
        self.tab9_roi4_ax2.set_title(
            f'ROI — SELECTED | {rw:.6g} × {rh:.6g} nm',
            fontsize=11
        )
        self.tab9_roi4_canvas.draw_idle()
    except Exception:
        pass

    self.tab9_roi4_status_var.set(
        f'ROI4 selected: X={c["xmin"]}:{c["xmax"]}, '
        f'Y={c["ymin"]}:{c["ymax"]}. '
        f'Click CONFIRM ROI.'
    )


def _tab9_roi4_confirm(self):
    c=getattr(self,'_tab9_roi4_coords',None)
    if not isinstance(c,dict):
        messagebox.showwarning(
            'ROI4',
            'Select ROI4 and drag a rectangle on Processed Primary ROI1 first.',
            parent=self.root
        )
        return False

    if not getattr(self,'_tab9_roi1_ready',False):
        if not _tab9_roi4_fetch_from_tab6(self,preserve_frame=True):
            return False

    h,w=self._tab9_roi1_shape
    try:
        xmin=int(c['xmin']); xmax=int(c['xmax'])
        ymin=int(c['ymin']); ymax=int(c['ymax'])
    except Exception:
        messagebox.showwarning(
            'ROI4',
            'ROI4 coordinates are invalid.',
            parent=self.root
        )
        return False

    if not (
        0 <= xmin <= xmax < w
        and 0 <= ymin <= ymax < h
        and xmax > xmin and ymax > ymin
    ):
        messagebox.showwarning(
            'ROI4',
            'ROI4 must be inside the processed Primary ROI1 and have non-zero area.',
            parent=self.root
        )
        return False

    try:
        stack=np.asarray(
            self._tab9_roi1_stack,dtype=np.float32
        )
        if stack.ndim!=3:
            raise ValueError(
                f'Primary ROI1 stack shape is {stack.shape}; expected [frame,y,x].'
            )

        self._tab9_roi4_stack=stack[
            :, ymin:ymax+1, xmin:xmax+1
        ].copy()
        self._tab9_roi4_stack_coords=dict(c)
        self._tab9_roi4_stack_shape=tuple(
            self._tab9_roi4_stack.shape
        )
    except Exception as exc:
        self._tab9_roi4_confirmed=False
        messagebox.showerror(
            'ROI4',
            f'Could not create the confirmed ROI4 stack:\n'
            f'{type(exc).__name__}: {exc}',
            parent=self.root
        )
        return False

    self._tab9_roi4_confirmed=True

    # Lock only the current confirmed data, but allow re-selection through
    # SELECT ROI4 and CLEAR ROI4 at any time.
    try:
        self.tab9_roi4_select_button.configure(
            text='SELECT ROI', state='normal'
        )
        self.tab9_roi4_confirm_button.configure(
            text='✓ ROI CONFIRMED', state='normal'
        )
    except Exception:
        pass

    idx=int(self.tab9_roi4_frame_var.get())
    _tab9_roi4_show_frame(
        self,idx,preserve_zoom=True
    )

    sx,sy=self._tab9_roi4_scale
    self.tab9_roi4_status_var.set(
        f'✓ ROI CONFIRMED | '
        f'X={xmin*sx:.6g}–{(xmax+1)*sx:.6g} nm | '
        f'Y={ymin*sy:.6g}–{(ymax+1)*sy:.6g} nm | '
        f'FOV={(xmax-xmin+1)*sx:.6g} × '
        f'{(ymax-ymin+1)*sy:.6g} nm'
    )
    self.tab9_roi4_canvas.draw_idle()
    return True


def _tab9_roi4_clear(self):
    _tab9_roi4_stop(self,update_status=False)

    self._tab9_roi4_coords=None
    self._tab9_roi4_preview_coords=None
    self._tab9_roi4_confirmed=False
    self._tab9_roi4_selecting=False
    self._tab9_roi4_press_xy=None
    self._tab9_roi4_stack=None
    self._tab9_roi4_stack_coords=None
    self._tab9_roi4_stack_shape=None

    try:
        if getattr(self,'_tab9_roi4_rect',None) is not None:
            self._tab9_roi4_rect.remove()
    except Exception:
        pass
    self._tab9_roi4_rect=None

    try:
        self.tab9_roi4_ax2.clear()
        self.tab9_roi4_image2=self.tab9_roi4_ax2.imshow(
            np.zeros((10,10),dtype=float),
            cmap=self._tab9_roi1_cmap,
            origin='lower',
            extent=(0,10,0,10),
            vmin=0,vmax=1,
            interpolation='nearest',
            aspect='equal'
        )
        self.tab9_roi4_ax2.set_title(
            'ROI — NOT SELECTED',fontsize=11
        )
        self.tab9_roi4_ax2.set_xlabel('X (nm)')
        self.tab9_roi4_ax2.set_ylabel('Y (nm)')
        self.tab9_roi4_ax2.set_aspect(
            'equal',adjustable='box'
        )
        # Rebuild the colorbar for the new image artist.
        try:
            self.tab9_roi4_cb2.remove()
        except Exception:
            pass
        cb2_ax=self.tab9_roi4_fig.add_axes(
            [0.955,0.11,0.018,0.78]
        )
        self.tab9_roi4_cb2=self.tab9_roi4_fig.colorbar(
            self.tab9_roi4_image2,cax=cb2_ax
        )
        self.tab9_roi4_cb2.set_label('Intensity')
    except Exception:
        pass

    try:
        self.tab9_roi4_select_button.configure(
            text='SELECT ROI',state='normal'
        )
        self.tab9_roi4_confirm_button.configure(
            text='✓ CONFIRM ROI',state='normal'
        )
        self.tab9_roi4_canvas.get_tk_widget().configure(
            cursor=''
        )
    except Exception:
        pass

    self.tab9_roi4_status_var.set(
        'ROI4 cleared. Click SELECT ROI4 and drag a new rectangle on Processed Primary ROI1.'
    )
    try:
        self.tab9_roi4_canvas.draw_idle()
    except Exception:
        pass
    return True


def _tab9_roi4_show_frame(self,idx,preserve_zoom=True):
    """
    Render the current Processed Primary ROI1 and the CURRENT ROI4 selection.
    ROI4 display comes directly from the frozen confirmed stack whenever
    confirmed; otherwise from the currently selected coordinates as preview.
    """
    if not getattr(self,'_tab9_roi1_ready',False):
        return

    stack=np.asarray(self._tab9_roi1_stack,dtype=np.float32)
    if stack.ndim!=3 or stack.shape[0]==0:
        return

    n,h,w=map(int,stack.shape)
    idx=max(0,min(int(idx),n-1))
    sx,sy=self._tab9_roi4_scale
    ph,pw=self._tab9_roi1_shape
    # Fixed geometry: slider position changes image data only, never FOV.
    self._tab9_roi1_extent=(0.0,pw*sx,0.0,ph*sy)
    image=stack[idx]

    self.tab9_roi4_frame_var.set(idx)

    # ---- Primary ROI1 ----
    self.tab9_roi4_image1.set_data(image)
    self.tab9_roi4_image1.set_extent(
        (0.0,w*sx,0.0,h*sy)
    )
    self.tab9_roi4_image1.set_cmap(
        self._tab9_roi1_cmap
    )
    self.tab9_roi4_image1.set_clim(
        *self._tab9_roi1_clim
    )
    self.tab9_roi4_ax1.set_autoscale_on(False)
    self.tab9_roi4_ax1.set_xlim(0.0,w*sx)
    self.tab9_roi4_ax1.set_ylim(0.0,h*sy)
    self.tab9_roi4_ax1.set_aspect(
        'equal',adjustable='box'
    )
    self.tab9_roi4_ax1.set_xlabel('X (nm)')
    self.tab9_roi4_ax1.set_ylabel('Y (nm)')
    field=_tab9_roi4_field(self,idx)
    self.tab9_roi4_ax1.set_title(
        f'PROCESSED ROI | Frame {idx+1}/{n} | '
        f'Field = {field:+.2f} mT',
        fontsize=11
    )
    self.tab9_roi4_cb1.update_normal(
        self.tab9_roi4_image1
    )
    self.tab9_roi4_cb1.set_label(
        'Intensity'
    )

    # ---- Current ROI4 ----
    c=getattr(self,'_tab9_roi4_coords',None)
    confirmed=bool(
        getattr(self,'_tab9_roi4_confirmed',False)
    )
    roi4_data=None

    if isinstance(c,dict):
        # Confirmed ROI4: use the exact frozen stack.
        if confirmed:
            cs=getattr(self,'_tab9_roi4_stack',None)
            if cs is not None:
                cs=np.asarray(cs,dtype=np.float32)
                if cs.ndim==3 and cs.shape[0]==n:
                    roi4_data=cs[idx]

        # Before confirmation, show the currently selected crop.
        if roi4_data is None:
            try:
                xmin=int(c['xmin']); xmax=int(c['xmax'])
                ymin=int(c['ymin']); ymax=int(c['ymax'])
                if (
                    0<=xmin<=xmax<w
                    and 0<=ymin<=ymax<h
                ):
                    roi4_data=image[
                        ymin:ymax+1,xmin:xmax+1
                    ]
            except Exception:
                roi4_data=None

    if roi4_data is not None:
        try:
            if isinstance(c,dict):
                xmin=int(c['xmin']); xmax=int(c['xmax'])
                ymin=int(c['ymin']); ymax=int(c['ymax'])
                rw=(xmax-xmin+1)*sx
                rh=(ymax-ymin+1)*sy
            else:
                rw=roi4_data.shape[1]*sx
                rh=roi4_data.shape[0]*sy

            self.tab9_roi4_image2.set_data(
                roi4_data
            )
            self.tab9_roi4_image2.set_extent(
                (0.0,rw,0.0,rh)
            )
            self.tab9_roi4_image2.set_cmap(
                self._tab9_roi1_cmap
            )
            self.tab9_roi4_image2.set_clim(
                *self._tab9_roi1_clim
            )
            self.tab9_roi4_ax2.set_autoscale_on(False)
            self.tab9_roi4_ax2.set_xlim(
                0.0,rw
            )
            self.tab9_roi4_ax2.set_ylim(
                0.0,rh
            )
            self.tab9_roi4_ax2.set_aspect(
                'equal',adjustable='box'
            )
            self.tab9_roi4_ax2.set_xlabel(
                'X (nm)'
            )
            self.tab9_roi4_ax2.set_ylabel(
                'Y (nm)'
            )
            self.tab9_roi4_ax2.set_title(
                (
                    f'ROI — CONFIRMED | {rw:.6g} × {rh:.6g} nm'
                    if confirmed else
                    f'ROI — SELECTED | {rw:.6g} × {rh:.6g} nm'
                ),
                fontsize=11
            )
            self.tab9_roi4_cb2.update_normal(
                self.tab9_roi4_image2
            )
            self.tab9_roi4_cb2.set_label(
                'Intensity'
            )
        except Exception:
            pass
    else:
        try:
            self.tab9_roi4_image2.set_data(
                np.zeros((10,10),dtype=float)
            )
            self.tab9_roi4_image2.set_extent(
                (0,10,0,10)
            )
            self.tab9_roi4_ax2.set_xlim(0,10)
            self.tab9_roi4_ax2.set_ylim(0,10)
            self.tab9_roi4_ax2.set_aspect(
                'equal',adjustable='box'
            )
            self.tab9_roi4_ax2.set_title(
                'ROI — NOT SELECTED',
                fontsize=11
            )
            self.tab9_roi4_cb2.update_normal(
                self.tab9_roi4_image2
            )
        except Exception:
            pass

    _tab9_roi4_draw_rect(
        self,
        confirmed=confirmed,
        redraw=False
    )
    self.tab9_roi4_frame_label.configure(
        text=(
            f'Frame {idx+1}/{n} | Field = {field:+.2f} mT | '
            f'Processed ROI1 FOV = {w*sx:.6g} × {h*sy:.6g} nm'
        )
    )
    self.tab9_roi4_canvas.draw_idle()


def _tab9_roi4_frame_changed(self,value=None):
    if getattr(self,'_tab9_roi4_programmatic',False):
        return
    if not getattr(self,'_tab9_roi1_ready',False):
        if not _tab9_roi4_fetch_from_tab6(
            self,preserve_frame=True
        ):
            return

    stack=np.asarray(self._tab9_roi1_stack)
    n=stack.shape[0]
    try:
        idx=int(round(float(value))) if value is not None else int(
            self.tab9_roi4_frame_var.get()
        )
    except Exception:
        idx=int(self.tab9_roi4_frame_var.get())

    idx=max(0,min(idx,n-1))
    self._tab9_roi4_programmatic=True
    try:
        self.tab9_roi4_frame_var.set(idx)
    finally:
        self._tab9_roi4_programmatic=False

    _tab9_roi4_show_frame(
        self,idx,preserve_zoom=True
    )


def _tab9_roi4_stop(self,update_status=True):
    self._tab9_roi4_playing=False
    self._tab9_roi4_play_token=getattr(
        self,'_tab9_roi4_play_token',0
    )+1

    job=getattr(self,'_tab9_roi4_play_job',None)
    self._tab9_roi4_play_job=None
    if job is not None:
        try:
            self.root.after_cancel(job)
        except Exception:
            pass

    try:
        self.tab9_roi4_play_button.configure(
            text='PLAY',state='normal'
        )
    except Exception:
        pass

    if (
        update_status
        and getattr(self,'_tab9_roi1_ready',False)
    ):
        try:
            idx=int(self.tab9_roi4_frame_var.get())
            n=len(self._tab9_roi1_stack)
            self.tab9_roi4_status_var.set(
                f'Playback paused | Frame {idx+1}/{n}'
            )
        except Exception:
            pass


def _tab9_roi4_toggle_play(self):
    if getattr(self,'_tab9_roi4_playing',False):
        _tab9_roi4_stop(self,update_status=True)
        return

    if not getattr(self,'_tab9_roi1_ready',False):
        if not _tab9_roi4_fetch_from_tab6(
            self,preserve_frame=True
        ):
            return

    n=len(self._tab9_roi1_stack)
    if n<=0:
        return

    try:
        idx=int(self.tab9_roi4_frame_var.get())
    except Exception:
        idx=0
    idx=max(0,min(idx,n-1))
    if idx>=n-1:
        idx=0

    _tab9_roi4_stop(self,update_status=False)
    self._tab9_roi4_playing=True
    self._tab9_roi4_play_token=getattr(
        self,'_tab9_roi4_play_token',0
    )+1
    token=self._tab9_roi4_play_token

    self._tab9_roi4_programmatic=True
    try:
        self.tab9_roi4_frame_var.set(idx)
        self.tab9_roi4_slider.set(idx)
    finally:
        self._tab9_roi4_programmatic=False

    _tab9_roi4_show_frame(
        self,idx,preserve_zoom=True
    )
    try:
        self.tab9_roi4_play_button.configure(
            text='PAUSE',state='normal'
        )
    except Exception:
        pass
    _tab9_roi4_schedule_next(self,token)


def _tab9_roi4_first_frame(self):
    _tab9_roi4_stop(self,update_status=False)
    if not getattr(self,'_tab9_roi1_ready',False):
        if not _tab9_roi4_fetch_from_tab6(
            self,preserve_frame=False
        ):
            return

    self._tab9_roi4_programmatic=True
    try:
        self.tab9_roi4_frame_var.set(0)
        self.tab9_roi4_slider.set(0)
    finally:
        self._tab9_roi4_programmatic=False

    _tab9_roi4_show_frame(
        self,0,preserve_zoom=False
    )
    self.tab9_roi4_status_var.set(
        f'Ready | frame 1/{len(self._tab9_roi1_stack)}'
    )


def _tab9_roi4_build(self):
    """Complete, single-pass Tab 9 builder. Safe to call repeatedly."""
    tab=self.tab_primary_roi9

    # Stop old playback and destroy old children when rebuilding in-place.
    try:
        _tab9_roi4_stop(self,update_status=False)
    except Exception:
        pass
    for child in list(tab.winfo_children()):
        try:
            child.destroy()
        except Exception:
            pass

    # State
    self.tab9_roi4_frame_var=tk.IntVar(value=0)
    self.tab9_roi4_start_var=tk.IntVar(value=1)
    self.tab9_roi4_end_var=tk.IntVar(value=1)
    self.tab9_roi4_fps_var=tk.IntVar(value=10)
    self.tab9_roi4_dpi_var=tk.IntVar(value=300)
    self.tab9_roi4_format_var=tk.StringVar(value='PNG')
    self.tab9_roi4_status_var=tk.StringVar(
        value='Fetching processed Primary ROI1 from Tab 6...'
    )
    self.tab9_roi4_export_status_var=tk.StringVar(value='')

    self._tab9_roi1_stack=None
    self._tab9_roi1_stack_ref=None
    self._tab9_roi1_ready=False
    self._tab9_roi1_shape=(1,1)
    self._tab9_roi1_extent=None
    self._tab9_roi1_cmap='gray'
    self._tab9_roi1_clim=(0.0,1.0)

    # IMPORTANT: Do not destroy/rewrite the current Tab9->Tab10 handoff
    # until a new ROI4 is actually confirmed.
    old_c=getattr(self,'_tab9_roi4_coords',None)
    old_confirmed=bool(
        getattr(self,'_tab9_roi4_confirmed',False)
    )
    old_stack=getattr(self,'_tab9_roi4_stack',None)
    old_stack_coords=getattr(
        self,'_tab9_roi4_stack_coords',None
    )
    old_stack_shape=getattr(
        self,'_tab9_roi4_stack_shape',None
    )

    self._tab9_roi4_coords=old_c if old_confirmed else None
    self._tab9_roi4_preview_coords=None
    self._tab9_roi4_confirmed=old_confirmed
    self._tab9_roi4_stack=old_stack if old_confirmed else None
    self._tab9_roi4_stack_coords=(
        old_stack_coords if old_confirmed else None
    )
    self._tab9_roi4_stack_shape=(
        old_stack_shape if old_confirmed else None
    )
    self._tab9_roi4_rect=None
    self._tab9_roi4_selecting=False
    self._tab9_roi4_press_xy=None
    self._tab9_roi4_playing=False
    self._tab9_roi4_play_job=None
    self._tab9_roi4_play_token=0
    self._tab9_roi4_programmatic=False
    self._tab6_exact_processed_primary_stack_cache=None
    self._tab6_exact_processed_primary_stack_signature=None
    self._tab9_slider_initialized=bool(
        getattr(self,'_tab9_slider_initialized',False)
    )

    # ---- Layout ----
    tab.grid_rowconfigure(3,weight=1)
    tab.grid_columnconfigure(0,weight=1)

    hdr=ttk.Frame(tab,padding=(8,6,8,4))
    hdr.grid(row=0,column=0,sticky='ew')
    hdr.grid_columnconfigure(1,weight=1)

    ttk.Label(
        hdr,
        text='PROCESSED PRIMARY ROI',
        font=('Arial',12,'bold')
    ).grid(row=0,column=0,sticky='w')

    self.tab9_roi4_frame_label=ttk.Label(
        hdr,text='Frame = -- | Field = --'
    )
    self.tab9_roi4_frame_label.grid(
        row=0,column=1,padx=14,sticky='w'
    )

    ttk.Label(
        hdr,textvariable=self.tab9_roi4_status_var,
        wraplength=1500,justify='left'
    ).grid(
        row=1,column=0,columnspan=2,sticky='w',
        pady=(2,0)
    )

    ctrl=ttk.LabelFrame(
        tab,text='PROCESSED PRIMARY ROI / ROI-SELECTION',
        padding=6
    )
    ctrl.grid(
        row=1,column=0,sticky='ew',
        padx=8,pady=(2,5)
    )
    for col in range(4):
        ctrl.grid_columnconfigure(
            col,weight=1,uniform='tab9_roi4_btn'
        )

    self.tab9_roi1_from_tab6_button=ttk.Button(
        ctrl,text='Primary ROI from Tab 6',
        command=lambda:_tab9_roi4_fetch_from_tab6(
            self,preserve_frame=True
        )
    )
    self.tab9_roi1_from_tab6_button.grid(
        row=0,column=0,padx=3,pady=3,sticky='ew'
    )

    self.tab9_roi4_select_button=ttk.Button(
        ctrl,text='SELECT ROI',
        command=lambda:_tab9_roi4_start_selection(self)
    )
    self.tab9_roi4_select_button.grid(
        row=0,column=1,padx=3,pady=3,sticky='ew'
    )

    self.tab9_roi4_confirm_button=ttk.Button(
        ctrl,text='✓ CONFIRM ROI',
        command=lambda:_tab9_roi4_confirm(self)
    )
    self.tab9_roi4_confirm_button.grid(
        row=0,column=2,padx=3,pady=3,sticky='ew'
    )

    self.tab9_roi4_clear_button=ttk.Button(
        ctrl,text='CLEAR ROI',
        command=lambda:_tab9_roi4_clear(self)
    )
    self.tab9_roi4_clear_button.grid(
        row=0,column=3,padx=3,pady=3,sticky='ew'
    )

    play=ttk.LabelFrame(
        tab,text='FRAME / PLAYBACK',padding=6
    )
    play.grid(
        row=2,column=0,sticky='ew',
        padx=8,pady=(0,5)
    )
    play.grid_columnconfigure(1,weight=1)

    ttk.Label(play,text='FRAME SLIDER').grid(
        row=0,column=0,padx=(2,6),sticky='w'
    )
    self.tab9_roi4_slider=tk.Scale(
        play,from_=0,to=0,
        orient='horizontal',
        variable=self.tab9_roi4_frame_var,
        resolution=1,showvalue=True,
        highlightthickness=0,bd=1,length=550,
        command=lambda v:_tab9_roi4_frame_changed(self,v)
    )
    self.tab9_roi4_slider.grid(
        row=0,column=1,sticky='ew',padx=4
    )

    ttk.Label(play,text='FPS').grid(
        row=0,column=2,padx=(10,3),sticky='e'
    )
    tk.Spinbox(
        play,from_=1,to=120,
        textvariable=self.tab9_roi4_fps_var,
        width=5
    ).grid(row=0,column=3,padx=3)

    self.tab9_roi4_play_button=ttk.Button(
        play,text='PLAY',
        command=lambda:_tab9_roi4_toggle_play(self)
    )
    self.tab9_roi4_play_button.grid(
        row=0,column=4,padx=(10,3)
    )

    self.tab9_roi4_first_button=ttk.Button(
        play,text='FIRST FRAME',
        command=lambda:_tab9_roi4_first_frame(self)
    )
    self.tab9_roi4_first_button.grid(
        row=0,column=5,padx=(3,2)
    )

    exp=ttk.LabelFrame(
        tab,text='ROI4 EXPORT ONLY',padding=6
    )
    exp.grid(
        row=4,column=0,sticky='ew',
        padx=8,pady=(0,5)
    )

    ttk.Label(exp,text='START FRAME').grid(
        row=0,column=0,padx=3,sticky='w'
    )
    self.tab9_roi4_start_spin = tk.Spinbox(
        exp,from_=1,to=1,increment=1,
        textvariable=self.tab9_roi4_start_var,
        width=7
    )
    self.tab9_roi4_start_spin.grid(row=0,column=1,padx=3)

    ttk.Label(exp,text='END FRAME').grid(
        row=0,column=2,padx=(10,3),sticky='w'
    )
    self.tab9_roi4_end_spin = tk.Spinbox(
        exp,from_=1,to=1,increment=1,
        textvariable=self.tab9_roi4_end_var,
        width=7
    )
    self.tab9_roi4_end_spin.grid(row=0,column=3,padx=3)

    ttk.Label(exp,text='DPI').grid(
        row=0,column=4,padx=(10,3),sticky='w'
    )
    ttk.Combobox(
        exp,textvariable=self.tab9_roi4_dpi_var,
        values=(150,300,600,900),
        state='readonly',width=7
    ).grid(row=0,column=5,padx=3)

    ttk.Label(exp,text='FORMAT').grid(
        row=0,column=6,padx=(10,3),sticky='w'
    )
    ttk.Combobox(
        exp,textvariable=self.tab9_roi4_format_var,
        values=('PNG','JPG','TIFF'),
        state='readonly',width=8
    ).grid(row=0,column=7,padx=3)

    ttk.Button(
        exp,text='EXPORT IMAGES',
        command=lambda:_tab9_roi4_batch_export_images(self)
    ).grid(row=0,column=8,padx=(14,4))

    ttk.Button(
        exp,text='EXPORT MP4',
        command=lambda:_tab9_roi4_batch_export_mp4(self)
    ).grid(row=0,column=9,padx=4)

    ttk.Button(
        exp,text='USE ALL FRAMES',
        command=lambda:(
            self.tab9_roi4_start_var.set(1),
            self.tab9_roi4_end_var.set(
                _tab9_roi4_frame_count(self) or 1
            )
        )
    ).grid(row=0,column=10,padx=4)

    ttk.Label(
        exp,textvariable=self.tab9_roi4_export_status_var,
        wraplength=1500,justify='left'
    ).grid(
        row=1,column=0,columnspan=11,
        sticky='w',padx=3,pady=(4,0)
    )

    image_box=ttk.LabelFrame(
        tab,text='PROCESSED PRIMARY ROI1 + ROI4',padding=2
    )
    image_box.grid(
        row=3,column=0,sticky='nsew',
        padx=8,pady=(0,8)
    )
    image_box.grid_rowconfigure(0,weight=1)
    image_box.grid_columnconfigure(0,weight=1)

    self.tab9_roi4_fig=plt.Figure(
        figsize=(13.0,6.5),dpi=100,facecolor='white'
    )
    self.tab9_roi4_ax1=self.tab9_roi4_fig.add_axes(
        [0.06,0.11,0.40,0.78]
    )
    self.tab9_roi4_ax2=self.tab9_roi4_fig.add_axes(
        [0.54,0.11,0.40,0.78]
    )

    dummy=np.zeros((10,10),dtype=float)
    self.tab9_roi4_image1=self.tab9_roi4_ax1.imshow(
        dummy,cmap='gray',origin='lower',
        extent=(0,10,0,10),vmin=0,vmax=1,
        interpolation='nearest',aspect='equal'
    )
    self.tab9_roi4_image2=self.tab9_roi4_ax2.imshow(
        dummy,cmap='gray',origin='lower',
        extent=(0,10,0,10),vmin=0,vmax=1,
        interpolation='nearest',aspect='equal'
    )

    self.tab9_roi4_ax1.set_title(
        'PROCESSED PRIMARY ROI',fontsize=11
    )
    self.tab9_roi4_ax1.set_xlabel('X (nm)')
    self.tab9_roi4_ax1.set_ylabel('Y (nm)')
    self.tab9_roi4_ax1.set_aspect(
        'equal',adjustable='box'
    )

    self.tab9_roi4_ax2.set_title(
        'ROI — NOT SELECTED',fontsize=11
    )
    self.tab9_roi4_ax2.set_xlabel('X (nm)')
    self.tab9_roi4_ax2.set_ylabel('Y (nm)')
    self.tab9_roi4_ax2.set_aspect(
        'equal',adjustable='box'
    )

    cb1_ax=self.tab9_roi4_fig.add_axes(
        [0.40,0.11,0.02,0.78]
    )
    cb2_ax=self.tab9_roi4_fig.add_axes(
        [0.95,0.11,0.02,0.78]
    )
    self.tab9_roi4_cb1=self.tab9_roi4_fig.colorbar(
        self.tab9_roi4_image1,cax=cb1_ax
    )
    self.tab9_roi4_cb2=self.tab9_roi4_fig.colorbar(
        self.tab9_roi4_image2,cax=cb2_ax
    )
    self.tab9_roi4_cb1.set_label(
        'Intensity'
    )
    self.tab9_roi4_cb2.set_label(
        'intensity'
    )

    self.tab9_roi4_canvas=FigureCanvasTkAgg(
        self.tab9_roi4_fig,master=image_box
    )
    self.tab9_roi4_canvas.draw()
    self.tab9_roi4_canvas.get_tk_widget().grid(
        row=0,column=0,sticky='nsew'
    )

    self.tab9_roi4_canvas.mpl_connect(
        'button_press_event',
        lambda e:_tab9_roi4_on_press(self,e)
    )
    self.tab9_roi4_canvas.mpl_connect(
        'motion_notify_event',
        lambda e:_tab9_roi4_on_move(self,e)
    )
    self.tab9_roi4_canvas.mpl_connect(
        'button_release_event',
        lambda e:_tab9_roi4_on_release(self,e)
    )
    self.tab9_roi4_canvas.mpl_connect(
        'scroll_event',
        lambda e:_tab9_roi4_zoom(self,e)
    )

    # Initial source fetch if Tab 6 is already ready.
    try:
        if (
            getattr(self,'roi_stack',None) is not None
            and getattr(self,'roi_resolution_applied',False)
        ):
            _tab9_roi4_fetch_from_tab6(
                self,preserve_frame=True
            )
        else:
            self.tab9_roi4_slider.configure(
                state='disabled'
            )
            self.tab9_roi4_play_button.configure(
                state='disabled'
            )
            self.tab9_roi4_first_button.configure(
                state='disabled'
            )
            self.tab9_roi4_status_var.set(
                'Tab 9 ready. Apply processed Primary ROI1 in Tab 6, then click ROI 1 from Tab 6.'
            )
    except Exception as exc:
        self.tab9_roi4_status_var.set(
            f'Tab 9 initialization warning: {type(exc).__name__}: {exc}'
        )


def _tab9_roi4_field(self, idx):
    try:
        if self.real_fields_mT is not None and idx < len(self.real_fields_mT):
            return float(self.real_fields_mT[idx])
    except Exception:
        pass
    return float(idx)


def _set_tab9_slider_default_if_needed(self,n):
    """Set Tab 9 slider to midpoint only on first source initialization."""
    n=int(n)
    if n<=0:
        return
    if getattr(self,'_tab9_slider_initialized',False):
        return
    mid=(n-1)//2
    self.tab9_roi4_frame_var.set(mid)
    self.tab9_roi4_slider.set(mid)
    self._tab9_slider_initialized=True

def _tab9_roi4_schedule_next(self, token):
    """Schedule the next frame without recursive callback chains."""
    if not getattr(self, '_tab9_roi4_playing', False):
        return
    if token != getattr(self, '_tab9_roi4_play_token', -1):
        return

    try:
        fps = max(1, min(120, int(self.tab9_roi4_fps_var.get())))
    except Exception:
        fps = 10

    delay = max(10, int(round(1000.0 / fps)))
    self._tab9_roi4_play_job = self.root.after(
        delay, lambda: _tab9_roi4_play_next(self, token)
    )


def _tab9_roi4_play_next(self, token=None):
    """Advance exactly one frame, then schedule the next one."""
    self._tab9_roi4_play_job = None

    if not getattr(self, '_tab9_roi4_playing', False):
        return
    if token is not None and token != getattr(self, '_tab9_roi4_play_token', -1):
        return

    n = len(self.recons)
    if n <= 0:
        _tab9_roi4_stop(self, update_status=False)
        return

    try:
        idx = int(round(float(self.tab9_roi4_frame_var.get())))
    except Exception:
        idx = 0

    if idx >= n - 1:
        _tab9_roi4_stop(self, update_status=False)
        self.tab9_roi4_status_var.set(f'Playback finished | frame {n}/{n}')
        return

    idx += 1

    self._tab9_roi4_programmatic = True
    try:
        self.tab9_roi4_frame_var.set(idx)
        self.tab9_roi4_slider.set(idx)
    finally:
        self._tab9_roi4_programmatic = False

    _tab9_roi4_show_frame(self, idx, preserve_zoom=True)
    _tab9_roi4_schedule_next(self, token)




def _tab9_roi4_reset_zoom(self):
    if not getattr(self, '_tab9_roi1_ready', False):
        return
    ext = self._tab9_roi1_extent
    self.tab9_roi4_ax1.set_xlim(ext[0], ext[1])
    self.tab9_roi4_ax1.set_ylim(ext[2], ext[3])
    self.tab9_roi4_ax1.set_aspect('equal', adjustable='box')
    self.tab9_roi4_canvas.draw_idle()


def _tab9_roi4_zoom(self, event):
    """Mouse-wheel zoom on Processed ROI1 while keeping the axes in nm."""
    if event.inaxes is not self.tab9_roi4_ax1 or event.xdata is None or event.ydata is None:
        return
    ext = self._tab9_roi1_extent
    if ext is None:
        return

    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        step = 1.0 if getattr(event, 'num', None) == 4 else -1.0

    factor = 1.2 ** (-step)
    x0, x1 = self.tab9_roi4_ax1.get_xlim()
    y0, y1 = self.tab9_roi4_ax1.get_ylim()
    cx, cy = float(event.xdata), float(event.ydata)

    rx = (cx - x0) / (x1 - x0) if x1 != x0 else 0.5
    ry = (cy - y0) / (y1 - y0) if y1 != y0 else 0.5
    nx0 = cx - rx * (x1 - x0) * factor
    nx1 = cx + (1 - rx) * (x1 - x0) * factor
    ny0 = cy - ry * (y1 - y0) * factor
    ny1 = cy + (1 - ry) * (y1 - y0) * factor

    fx0, fx1, fy0, fy1 = ext
    nx0 = max(fx0, min(nx0, fx1))
    nx1 = min(fx1, max(nx1, fx0))
    ny0 = max(fy0, min(ny0, fy1))
    ny1 = min(fy1, max(ny1, fy0))

    if nx1 <= nx0 or ny1 <= ny0:
        return

    self.tab9_roi4_ax1.set_xlim(nx0, nx1)
    self.tab9_roi4_ax1.set_ylim(ny0, ny1)
    self.tab9_roi4_ax1.set_aspect('equal', adjustable='box')
    self.tab9_roi4_canvas.draw_idle()


def _tab9_roi4_export_figure(self, idx, dpi, fmt='PNG'):
    if not getattr(self, '_tab9_roi4_confirmed', False) or getattr(self, '_tab9_roi4_coords', None) is None:
        raise RuntimeError('Confirm ROI before exporting.')

    image = _tab9_roi4_get_current_processed(self, idx)
    if image is None:
        raise RuntimeError('Processed Primary ROI1 is unavailable.')

    c = self._tab9_roi4_coords
    sx, sy = self._tab9_roi4_scale
    crop = np.asarray(
        image[c['ymin']:c['ymax'] + 1, c['xmin']:c['xmax'] + 1],
        dtype=float
    )
    roi4_w_nm = (c['xmax'] - c['xmin'] + 1) * sx
    roi4_h_nm = (c['ymax'] - c['ymin'] + 1) * sy
    # Export ROI4 with local coordinates starting at 0 nm.
    ext = (0.0, roi4_w_nm, 0.0, roi4_h_nm)
    fig = plt.figure(figsize=(7.2, 6.5), dpi=int(dpi), facecolor='white')
    ax = fig.add_axes([0.11, 0.11, 0.72, 0.78])
    cax = fig.add_axes([0.86, 0.11, 0.04, 0.78])

    # IMPORTANT: TAB 9 export uses one fixed intensity scale for ALL frames.
    # Do NOT calculate vmin/vmax from each frame, because that would
    # re-normalize the contrast and hide the real field-dependent change.
    # The scientific intensity scale is always exactly 0 to 1 for both
    # exported still images and MP4 frames.
    export_vmin = 0.0
    export_vmax = 1.0

    im = ax.imshow(
        crop, origin='lower', extent=ext, interpolation='nearest',
        cmap=str(getattr(self, 'colormap_var').get() or 'gray'),
        vmin=export_vmin, vmax=export_vmax, aspect='equal'
    )
    ax.set_autoscale_on(False)
    ax.set_xlim(ext[0], ext[1])
    ax.set_ylim(ext[2], ext[3])
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel(f'X (nm)')
    ax.set_ylabel(f'Y (nm)')
    field = _tab9_roi4_field(self, idx)
    ax.set_title(
        f'ROI — Frame {idx + 1}/{_tab9_roi4_frame_count(self)} | Field = {field:+.2f} mT',
        fontsize=12
    )
    cb = fig.colorbar(im, cax=cax)
    cb.set_label('Intensity')
    return fig


def _tab9_roi4_frame_count(self):
    """Return the exact number of frames available to TAB 9.

    TAB 9 displays the processed Primary ROI1 stack.  Therefore its slider
    and ROI4 export range use this same stack length.  This corresponds to
    the number of successfully loaded HDF5 reconstructions containing the
    required p_pc/n_pc datasets, not merely the number of files discovered.
    """
    stack=getattr(self,'_tab9_roi1_stack',None)
    if stack is not None:
        try:
            a=np.asarray(stack)
            if a.ndim==3 and a.shape[0]>0:
                return int(a.shape[0])
        except Exception:
            pass

    recons=getattr(self,'recons',None)
    if recons is not None:
        try:
            a=np.asarray(recons)
            if a.ndim>=1 and a.shape[0]>0:
                return int(a.shape[0])
        except Exception:
            pass
    return 0


def _tab9_roi4_get_export_range(self):
    if not getattr(self,'_tab9_roi4_confirmed',False):
        raise RuntimeError('Confirm ROI before exporting.')

    n=_tab9_roi4_frame_count(self)
    if n<=0:
        raise RuntimeError('No processed HDF5 frames are available in Tab 9.')

    try:
        start=int(self.tab9_roi4_start_var.get())
        end=int(self.tab9_roi4_end_var.get())
    except Exception:
        raise ValueError('Start and End frame must be integers.')

    if not (1<=start<=end<=n):
        raise ValueError(
            f'Frame range must be within 1–{n} '
            f'(Tab 9 has {n} loaded HDF5 frame(s)).'
        )

    try:
        dpi=int(self.tab9_roi4_dpi_var.get())
    except Exception:
        dpi=300

    if dpi not in (150,300,600,900):
        raise ValueError('DPI must be 150, 300, 600 or 900.')

    return start,end,dpi



def _tab9_roi4_batch_export_images(self):
    try:
        start, end, dpi = _tab9_roi4_get_export_range(self)
        fmt = str(self.tab9_roi4_format_var.get()).upper()
        fmt_map = {
            'PNG': ('png', '.png'),
            'JPG': ('jpg', '.jpg'),
            'TIFF': ('tiff', '.tiff')
        }
        if fmt not in fmt_map:
            raise ValueError('Image format must be PNG, JPG or TIFF.')

        mpl_fmt, extension = fmt_map[fmt]
        folder = os.path.abspath(
            os.path.expanduser(
                str(self.output_var.get()).strip() or
                self._export_initialdir() or 'ROI4_exports'
            )
        )
        os.makedirs(folder, exist_ok=True)

        for i in range(start - 1, end):
            field = _tab9_roi4_field(self, i)
            path = os.path.join(
                folder,
                f'ROI4_Frame{i + 1:03d}_{field:+.2f}mT{extension}'
            )
            fig = None
            try:
                fig = _tab9_roi4_export_figure(self, i, dpi, fmt)
                fig.savefig(
                    path, dpi=dpi, bbox_inches='tight',
                    facecolor='white', format=mpl_fmt
                )
            finally:
                if fig is not None:
                    plt.close(fig)

        self.tab9_roi4_export_status_var.set(
            f'✓ Exported ROI4 only | {end - start + 1} {fmt} image(s) | '
            f'frames {start}–{end} | {dpi} DPI'
        )
        _tab6_open_output_folder(folder)

    except Exception as exc:
        self.tab9_roi4_export_status_var.set(f'✗ ROI4 export failed: {exc}')
        messagebox.showerror('ROI4 Export', str(exc), parent=self.root)


def _tab9_roi4_batch_export_mp4(self):
    """
    Export exactly the selected TAB 9 ROI4 frame range at the selected FPS.

    MP4/H.264 is required to accept arbitrary matplotlib raster sizes.
    If a rendered figure has an odd width or height, the image content is
    preserved and only a one-pixel edge pad is added before encoding.
    Thus dimensions such as 1080x975 are exported as 1080x976 instead of
    failing with FFmpeg Broken pipe.  No resizing/cropping is performed.
    """
    writer = None
    try:
        start, end, dpi = _tab9_roi4_get_export_range(self)
        fps = max(1, min(60, int(self.tab9_roi4_fps_var.get())))

        folder = os.path.abspath(
            os.path.expanduser(
                str(self.output_var.get()).strip() or
                self._export_initialdir() or 'ROI4_exports'
            )
        )
        os.makedirs(folder, exist_ok=True)

        path = os.path.join(
            folder,
            f'ROI4_Frames{start:03d}-{end:03d}_{fps}FPS_{dpi}DPI.mp4'
        )

        import imageio.v2 as imageio

        # Explicitly disable ImageIO's macro-block resizing.  We handle
        # encoder-safe dimensions ourselves so the scientific figure is
        # never silently resized.
        errors = []
        for codec in ('libx264', 'mpeg4', None):
            try:
                kwargs = {
                    'fps': fps,
                    'format': 'ffmpeg',
                    'macro_block_size': None
                }
                if codec is not None:
                    kwargs['codec'] = codec
                writer = imageio.get_writer(path, **kwargs)
                break
            except Exception as exc:
                errors.append(f'{codec or "default"}: {exc}')

        if writer is None:
            raise RuntimeError(
                'Could not initialize an MP4 encoder.\n\n' +
                '\n'.join(errors)
            )

        expected = end - start + 1
        written = 0

        try:
            for i in range(start - 1, end):
                fig = None
                try:
                    fig = _tab9_roi4_export_figure(self, i, dpi, 'PNG')
                    fig.canvas.draw()

                    frame = np.asarray(
                        fig.canvas.buffer_rgba(), dtype=np.uint8
                    )[..., :3].copy()

                    if frame.ndim != 3 or frame.shape[2] != 3:
                        raise RuntimeError(
                            f'Invalid rendered RGB frame shape: {frame.shape}'
                        )

                    h, w = frame.shape[:2]

                    # H.264/yuv420p requires even dimensions.  Preserve
                    # every original pixel; only duplicate the final edge
                    # row/column when necessary.  No interpolation, crop,
                    # or scientific-data alteration occurs.
                    pad_h = h % 2
                    pad_w = w % 2
                    if pad_h or pad_w:
                        frame = np.pad(
                            frame,
                            ((0, pad_h), (0, pad_w), (0, 0)),
                            mode='edge'
                        )

                    writer.append_data(frame)
                    written += 1

                finally:
                    if fig is not None:
                        plt.close(fig)

                self.tab9_roi4_export_status_var.set(
                    f'⏳ Exporting ROI4 MP4 | frame {i + 1}/{end} | '
                    f'{written}/{expected} | {fps} FPS'
                )
                self.root.update_idletasks()

        finally:
            writer.close()
            writer = None

        if written != expected:
            raise RuntimeError(
                f'MP4 export wrote {written} frame(s), '
                f'but {expected} were requested.'
            )

        self.tab9_roi4_export_status_var.set(
            f'✓ Exported ROI4 only as MP4 | frames {start}–{end} | '
            f'{written} frames | {fps} FPS | {dpi} DPI'
        )
        _tab6_open_output_folder(folder)

    except Exception as exc:
        if writer is not None:
            try:
                writer.close()
            except Exception:
                pass
        self.tab9_roi4_export_status_var.set(
            f'✗ ROI4 MP4 export failed: {exc}'
        )
        messagebox.showerror(
            'ROI MP4 Export', str(exc), parent=self.root
        )





    # Primary ROI1 is fetched explicitly with the 'ROI 1 from Tab 6' button.


def _tab9_roi4_open(self):
    """Open Tab 9 and refresh from the current processed Primary ROI1."""
    try:
        self.nb.select(self.tab_primary_roi9)
    except Exception:
        pass
    _tab9_roi4_fetch_from_tab6(self, preserve_frame=True)


# Install Tab 9.
CDIWorkflowApp._tab9_roi4_open = _tab9_roi4_open
CDIWorkflowApp._tab9_roi4_fetch_from_tab6 = _tab9_roi4_fetch_from_tab6
CDIWorkflowApp._tab9_roi4_clear = _tab9_roi4_clear

_TAB9_BASE_BUILD_UI_NEW = CDIWorkflowApp._build_ui

def _build_ui_with_tab9_primary_roi4(self, *args, **kwargs):
    result = _TAB9_BASE_BUILD_UI_NEW(self, *args, **kwargs)
    try:
        if not hasattr(self, 'tab_primary_roi9'):
            self.tab_primary_roi9 = ttk.Frame(self.nb)
            self.nb.add(self.tab_primary_roi9, text='9. Primary ROI')
        _tab9_roi4_build(self)
    except Exception as exc:
        try:
            self.log(
                f'Tab 9 Primary ROI1/ROI4 initialization warning: '
                f'{type(exc).__name__}: {exc}'
            )
        except Exception:
            pass
    return result

CDIWorkflowApp._build_ui = _build_ui_with_tab9_primary_roi4


# ---------------------------------------------------------------------------
# TAB 10 — SINGLE-ROI 1D GAUSSIAN FIT (CONFIRMED ROI4 FROM TAB 9)
# ---------------------------------------------------------------------------

def _tab10_roi4_available(self):
    # ROI4 availability is determined by the CONFIRMED ROI4 state in Tab 9.
    # The processed ROI1 stack is fetched on demand if necessary.
    return (
        bool(getattr(self, '_tab9_roi4_confirmed', False))
        and getattr(self, '_tab9_roi4_coords', None) is not None
    )


def _tab10_sync_roi4_from_tab9(self, preserve_frame=True):
    """Fetch the CURRENT confirmed ROI from Tab 9 exactly."""
    try:
        self.tab10_status_var.set('Fetching CURRENT confirmed ROI from Tab 9...')
        self.root.update_idletasks()
    except Exception:
        pass

    if not bool(getattr(self,'_tab9_roi4_confirmed',False)):
        self.tab10_status_var.set(
            'ROI4 is not confirmed in Tab 9. Confirm ROI in Tab 9 first.'
        )
        self.tab10_slider.configure(state='disabled')
        return False

    c=getattr(self,'_tab9_roi4_coords',None)
    if not isinstance(c,dict):
        self.tab10_status_var.set(
            'Tab 9 reports ROI confirmed, but the current ROI4 coordinates are missing.'
        )
        self.tab10_slider.configure(state='disabled')
        return False

    source=getattr(self,'_tab9_roi1_stack',None)
    if source is None or not bool(getattr(self,'_tab9_roi1_ready',False)):
        if not _tab9_roi4_fetch_from_tab6(self,preserve_frame=True):
            self.tab10_status_var.set(
                'Could not fetch processed Primary ROI1 from Tab 6.'
            )
            self.tab10_slider.configure(state='disabled')
            return False
        source=getattr(self,'_tab9_roi1_stack',None)

    source=np.asarray(source,dtype=np.float32)
    if source.ndim!=3 or source.shape[0]<1:
        self.tab10_status_var.set(
            f'Invalid Tab 9 processed Primary ROI1 stack: {source.shape}'
        )
        self.tab10_slider.configure(state='disabled')
        return False

    h,w=source.shape[1:3]
    xmin=int(c['xmin']); xmax=int(c['xmax'])
    ymin=int(c['ymin']); ymax=int(c['ymax'])
    if not (0<=xmin<=xmax<w and 0<=ymin<=ymax<h):
        self.tab10_status_var.set(
            f'Confirmed ROI4 is outside Tab 9 Primary ROI1: '
            f'X={xmin}:{xmax}, Y={ymin}:{ymax}, source={w}×{h}px'
        )
        self.tab10_slider.configure(state='disabled')
        return False

    # Prefer the frozen stack only when it matches the CURRENT coordinates and shape.
    frozen=np.asarray(getattr(self,'_tab9_roi4_stack',None),dtype=np.float32) \
        if getattr(self,'_tab9_roi4_stack',None) is not None else None
    fc=getattr(self,'_tab9_roi4_stack_coords',None)
    expected_shape=(source.shape[0],ymax-ymin+1,xmax-xmin+1)

    if (
        frozen is not None
        and frozen.ndim==3
        and isinstance(fc,dict)
        and fc==c
        and tuple(frozen.shape)==tuple(expected_shape)
    ):
        roi_stack=frozen.copy()
    else:
        roi_stack=source[:,ymin:ymax+1,xmin:xmax+1].copy()
        try:
            self._tab9_roi4_stack=roi_stack.copy()
            self._tab9_roi4_stack_coords=dict(c)
            self._tab9_roi4_stack_shape=tuple(roi_stack.shape)
        except Exception:
            pass

    # Transfer exactly Tab9 scale.
    try:
        sx,sy=map(float,self._tab9_roi4_scale)
    except Exception:
        sx,sy=map(float,_tab9_roi4_scale_nm(self))
    self._tab10_roi4_scale=(sx,sy)

    self._tab10_roi4_stack=roi_stack.copy()
    self._tab10_roi4_global_coords=dict(c)
    self._tab10_roi4_coords={
        'xmin':0,'xmax':roi_stack.shape[2]-1,
        'ymin':0,'ymax':roi_stack.shape[1]-1
    }
    self._tab10_roi4_confirmed=True

    # Match Tab9 display color/limits.
    try:
        art=self.tab9_roi4_image2
        self._tab10_roi4_display_cmap=str(art.get_cmap().name)
        self._tab10_roi4_display_clim=tuple(float(v) for v in art.get_clim())
    except Exception:
        self._tab10_roi4_display_cmap=str(
            getattr(self,'_tab9_roi1_cmap','gray')
        )
        self._tab10_roi4_display_clim=tuple(
            getattr(self,'_tab9_roi1_clim',(0.0,1.0))
        )

    n=int(roi_stack.shape[0])

    # IMPORTANT: when preserve_frame=True, use Tab9's CURRENT frame.
    # This eliminates Tab10 opening at frame 1 while Tab9 is on frame 80.
    if preserve_frame:
        try:
            old=int(self.tab9_roi4_frame_var.get())
        except Exception:
            old=0
    else:
        try:
            old=int(self.tab10_frame_var.get())
        except Exception:
            old=0
    old=max(0,min(old,n-1))

    self._tab10_programmatic=True
    try:
        self.tab10_frame_var.set(old)
        self.tab10_slider.configure(
            from_=0,to=n-1,resolution=1,state='normal'
        )
        self.tab10_slider.set(old)
        _set_frame_slider_midpoint_once(self,self.tab10_slider,n,self.tab10_frame_var,zero_based=True,token='tab10')
        self.tab10_start_var.set(0)
        self.tab10_end_var.set(n-1)
        self.tab10_start_display_var.set('1')
        self.tab10_end_display_var.set(str(n))
        try:
            self.tab10_frame_label_var.set(
                f'Frame No. {old + 1}/{n} | Equivalent Field: '
                f'{float(_tab9_roi4_field(self, old)):+.6g} mT'
            )
            self.tab10_frame_info_var.set(
                f'Frame No. {old + 1}/{n} | Equivalent Field: '
                f'{float(_tab9_roi4_field(self, old)):+.6g} mT'
            )
            self.tab10_start_spin.configure(from_=1, to=n, increment=1)
            self.tab10_end_spin.configure(from_=1, to=n, increment=1)
        except Exception:
            pass
    finally:
        self._tab10_programmatic=False

    self.tab10_status_var.set(
        f'✓ CURRENT confirmed ROI4 fetched from Tab 9 | '
        f'FOV={roi_stack.shape[2]*sx:.6g} × {roi_stack.shape[1]*sy:.6g} nm | '
        f'Frame {old+1}/{n}'
    )
    _tab10_render_current(self,old)
    return True






def _tab10_get_crop(self, frame_idx):
    """Return the already-cropped confirmed ROI4 for a frame."""
    if getattr(self, '_tab10_roi4_stack', None) is None:
        if not _tab10_sync_roi4_from_tab9(self, preserve_frame=True):
            return None
    stack = np.asarray(self._tab10_roi4_stack, dtype=float)
    if stack.ndim != 3 or stack.shape[0] < 1:
        raise ValueError(f'ROI4 stack is invalid: {stack.shape}')
    idx = max(0, min(int(frame_idx), stack.shape[0] - 1))
    crop = np.asarray(stack[idx], dtype=float).copy()
    if crop.size == 0 or crop.ndim != 2:
        raise ValueError('ROI4 crop is empty or not a 2D image.')
    return crop



def _tab10_center(self, crop):
    """Same strongest-signed contrast-center algorithm, using Tab-10 threshold 0.50."""
    image = np.asarray(crop, dtype=float)
    finite = np.isfinite(image)
    if not np.any(finite):
        raise ValueError('No finite pixels in ROI4.')
    z = image.copy()
    z[~finite] = np.nanmedian(z[finite])
    h, w = z.shape
    b = max(1, int(round(0.1 * min(h, w))))
    border = np.concatenate([z[:b, :].ravel(), z[-b:, :].ravel(), z[:, :b].ravel(), z[:, -b:].ravel()])
    background = float(np.nanmedian(border))
    contrast = z - background
    max_pos = float(np.nanmax(contrast))
    max_neg = float(np.nanmin(contrast))
    if abs(max_pos) >= abs(max_neg):
        signed = contrast; sign = 1.0; peak = max_pos
    else:
        signed = -contrast; sign = -1.0; peak = -max_neg
    peak = max(peak, 1e-12)
    try:
        frac = float(np.clip(self.tab10_contrast_fraction_var.get(), 0.0, 0.95))
    except Exception:
        frac = 0.50
    threshold = frac * peak
    weights = np.clip(signed - threshold, 0.0, None)
    if np.sum(weights) <= 0:
        weights = np.clip(signed, 0.0, None)
    yy, xx = np.indices(z.shape, dtype=float)
    ws = float(np.sum(weights))
    if ws <= 0:
        y0, x0 = np.unravel_index(np.argmax(np.abs(contrast)), z.shape)
        x0, y0 = float(x0), float(y0)
    else:
        x0 = float(np.sum(xx * weights) / ws)
        y0 = float(np.sum(yy * weights) / ws)
    return {'x0': x0, 'y0': y0, 'background': background, 'sign': sign, 'peak': peak, 'threshold': threshold}



def _tab10_get_reference_angles(self):
    return {
        '1': [0.0],
        '2': [0.0, 90.0],
        '4': [0.0, 45.0, 90.0, 135.0],
        '6': [0.0, 30.0, 60.0, 90.0, 120.0, 150.0]
    }.get(str(self.tab10_ref_mode_var.get()), [0.0, 90.0])


def _tab10_update_reference_label(self, event=None):
    angles = _tab10_get_reference_angles(self)
    self.tab10_ref_label.configure(
        text=f'{len(angles)} lines: ' + ' / '.join(f'{a:g}°' for a in angles)
    )
    _tab10_render_current(self)


def _tab10_extract_profiles(self, crop, cx, cy, angles):
    # Reuse the existing line-profile extraction algorithm.
    return self._extract_multi_line_profiles(
        crop, float(cx), float(cy), list(angles), nsamples=800
    )



def _tab10_detect_particle_for_tracking(self, crop):
    """
    Robust particle-presence gate for TAB 10 TRACK-SELECTED FRAMES.

    The detector is used only when "Zero if Undetected" is enabled.
    It is intentionally sequential: a valid particle detected in one frame
    establishes a local tracking reference for the next frame.  This prevents
    a transient low-contrast frame from being incorrectly converted to zero,
    while compactness/SNR/prominence tests still reject background noise.

    FIT CURRENT ROI and _tab10_center() are not modified.
    """
    try:
        a = np.asarray(crop, dtype=float)
        if a.ndim != 2 or a.size < 49:
            return False, None

        finite = np.isfinite(a)
        if not np.any(finite):
            return False, None
        med = float(np.nanmedian(a[finite]))
        if not np.isfinite(med):
            return False, None
        a = np.where(finite, a, med)

        h, w = a.shape
        if min(h, w) < 9:
            return False, None

        fixed = getattr(self, '_tab10_tracking_fixed_center', None)
        if not isinstance(fixed, dict):
            fixed = dict(_tab10_center(self, crop))

        fx = float(fixed.get('x0', np.nan))
        fy = float(fixed.get('y0', np.nan))
        if not (np.isfinite(fx) and np.isfinite(fy)):
            return False, None

        # Use the previous accepted core when available.  This is the key
        # sequential-tracking improvement: frame 76 is allowed to remain
        # attached to the real particle even if its contrast temporarily
        # falls below the global threshold.
        prev = getattr(self, '_tab10_last_detected_core', None)
        if isinstance(prev, dict):
            px = float(prev.get('x0', fx))
            py = float(prev.get('y0', fy))
            if np.isfinite(px) and np.isfinite(py):
                refx, refy = px, py
            else:
                refx, refy = fx, fy
        else:
            refx, refy = fx, fy

        # The search remains anchored to the fixed ROI center.  A previous
        # accepted core may tighten the search locally but cannot move the
        # tracking reference outside the ROI.
        base_radius = max(5.0, 0.32 * min(h, w))
        prev_radius = max(4.0, 0.20 * min(h, w))
        yy, xx = np.indices((h, w), dtype=float)

        # Background removal for detection only.
        bg_sigma = max(3.0, 0.18 * min(h, w))
        bg = gaussian_filter(a, sigma=bg_sigma, mode='nearest')
        residual = a - bg

        # Robust residual noise.
        b = max(2, int(round(0.10 * min(h, w))))
        border = np.concatenate([
            residual[:b, :].ravel(), residual[-b:, :].ravel(),
            residual[:, :b].ravel(), residual[:, -b:].ravel()
        ])
        border = border[np.isfinite(border)]
        if border.size < 20:
            border = residual.ravel()

        noise = 1.4826 * float(np.median(
            np.abs(border - np.median(border))
        ))
        if not np.isfinite(noise) or noise <= 0:
            noise = float(np.std(border))
        if not np.isfinite(noise) or noise <= 0:
            noise = max(np.finfo(float).eps, float(np.std(residual)) * 0.1)

        scales = (
            max(1.5, 0.025 * min(h, w)),
            max(2.0, 0.035 * min(h, w)),
            max(2.5, 0.050 * min(h, w)),
            max(3.0, 0.070 * min(h, w)),
            max(4.0, 0.095 * min(h, w)),
            max(5.0, 0.125 * min(h, w)),
        )

        # User's 0.60 threshold remains part of validation.
        frac = 0.60
        try:
            frac = float(np.clip(
                self.tab10_contrast_fraction_var.get(), 0.0, 0.95
            ))
        except Exception:
            pass

        best = None

        for sig in scales:
            smooth = gaussian_filter(residual, sigma=sig, mode='nearest')

            rb = np.concatenate([
                smooth[:b, :].ravel(), smooth[-b:, :].ravel(),
                smooth[:, :b].ravel(), smooth[:, -b:].ravel()
            ])
            rb = rb[np.isfinite(rb)]
            rmad = 1.4826 * float(np.median(
                np.abs(rb - np.median(rb))
            ))
            if not np.isfinite(rmad) or rmad <= 0:
                rmad = float(np.std(rb))
            if not np.isfinite(rmad) or rmad <= 0:
                rmad = noise / max(1.0, np.sqrt(2*np.pi)*sig)

            for polarity in (1.0, -1.0):
                response = polarity * smooth

                # Primary search: around fixed center.  If a previous core
                # exists, include its local neighborhood as an equally valid
                # search area, provided it remains close to fixed center.
                search = (
                    ((xx - fx)**2 + (yy - fy)**2 <= base_radius**2) |
                    (((xx - refx)**2 + (yy - refy)**2 <= prev_radius**2) &
                     ((xx - fx)**2 + (yy - fy)**2 <= base_radius**2))
                )

                candidate = np.where(search, response, -np.inf)
                flat = int(np.argmax(candidate))
                cy, cx = np.unravel_index(flat, candidate.shape)
                peak = float(response[cy, cx])
                if not np.isfinite(peak):
                    continue

                snr = peak / max(rmad, np.finfo(float).eps)

                inner_r = max(1.5, 0.65 * sig)
                outer_r = max(inner_r + 1.0, 1.75 * sig)
                dist = np.hypot(xx - cx, yy - cy)
                inner = response[dist <= inner_r]
                annulus = response[
                    (dist >= inner_r) & (dist <= outer_r)
                ]
                if inner.size < 3 or annulus.size < 8:
                    continue

                center_level = float(np.median(inner))
                annulus_level = float(np.median(annulus))
                prominence = center_level - annulus_level

                # Two operating levels:
                # A) normal frames: the original robust gate;
                # B) sequential-recovery frames: slightly relaxed only when
                # the candidate is close to the previous accepted core.
                near_previous = (
                    isinstance(prev, dict) and
                    np.hypot(cx - refx, cy - refy) <= prev_radius
                )

                if near_previous:
                    snr_min = 3.0
                    prom_abs = 1.8 * rmad
                    prom_rel = 0.07 * peak
                else:
                    snr_min = 4.0
                    prom_abs = 2.2 * rmad
                    prom_rel = 0.10 * peak

                if snr < snr_min:
                    continue
                if prominence < max(prom_abs, prom_rel):
                    continue

                # Edge rejection.
                margin = max(2.0, 1.35 * sig)
                if (
                    cx < margin or cx > w - 1 - margin or
                    cy < margin or cy > h - 1 - margin
                ):
                    continue

                # 0.60 contrast + compactness.  For a dim real particle the
                # mask may be smaller, but it must remain localized.
                patch = response[
                    max(0, int(cy - outer_r)):min(h, int(cy + outer_r) + 1),
                    max(0, int(cx - outer_r)):min(w, int(cx + outer_r) + 1)
                ]
                if patch.size == 0:
                    continue

                threshold = frac * peak
                area = int(np.count_nonzero(patch >= threshold))
                expected = np.pi * max(sig, 1.0)**2

                min_area = max(2, int(0.10 * expected))
                max_area = max(12, int(8.5 * expected))
                if area < min_area or area > max_area:
                    continue

                # Central signal must exceed the noise floor.
                center_floor = 1.8 if near_previous else 2.3
                if center_level < center_floor * rmad:
                    continue

                # Candidate must remain close to the fixed center.  Previous
                # continuity never permits an unrelated distant structure.
                if np.hypot(cx - fx, cy - fy) > base_radius:
                    continue

                score = (
                    snr
                    + 0.45 * prominence / max(rmad, 1e-12)
                    + (1.5 if near_previous else 0.0)
                )

                if best is None or score > best['score']:
                    best = {
                        'x0': float(cx),
                        'y0': float(cy),
                        'background': float(np.nanmedian(bg)),
                        'sign': float(polarity),
                        'peak': float(peak),
                        'threshold': float(frac * peak),
                        'snr': float(snr),
                        'prominence': float(prominence),
                        'scale': float(sig),
                        'score': float(score),
                        'near_previous': bool(near_previous),
                    }

        if best is None:
            return False, None

        return True, best

    except Exception:
        return False, None



def _tab10_fit_roi4(self, frame_idx):
    crop = _tab10_get_crop(self, frame_idx)
    if crop is None:
        return None

    center = _tab10_center(self, crop)
    cx = float(center['x0'])
    cy = float(center['y0'])
    angles = _tab10_get_reference_angles(self)

    t_px, profiles, avg = _tab10_extract_profiles(
        self, crop, cx, cy, angles
    )

    sx, sy = self._tab10_roi4_scale
    # Existing profile sampling is in image pixels. Convert only the profile
    # coordinate to nm for physical-axis display/fitting; the Gaussian model
    # itself remains the existing fixed-center model.
    sline = float(np.sqrt(float(sx) * float(sy))) if sx > 0 and sy > 0 else 1.0
    t_nm = np.asarray(t_px, dtype=float) * sline

    fit = self._fit_fixed_center_1d_gaussian(t_nm, avg)

    stack = np.vstack([profiles[float(a)] for a in angles])
    per_line_rmse = np.sqrt(
        np.nanmean((stack - avg[None, :]) ** 2, axis=1)
    )
    line_rmse = float(np.nanmean(per_line_rmse))
    line_cv = float(
        line_rmse / max(float(np.nanmax(np.abs(avg))), 1e-12)
    )

    field = _tab9_roi4_field(self, int(frame_idx))
    c = self._tab10_roi4_coords

    return {
        'roi_index': 4,
        'frame': int(frame_idx),
        'field_mT': float(field),
        'center_x_px': float(c['xmin'] + cx),
        'center_y_px': float(c['ymin'] + cy),
        'local_center_x_px': float(cx),
        'local_center_y_px': float(cy),
        'contrast_background': float(center['background']),
        'contrast_threshold': float(center['threshold']),
        'contrast_sign': float(center['sign']),
        'contrast_peak': float(center['peak']),
        'angles': list(angles),
        't_px': np.asarray(t_px, dtype=float),
        't_nm': t_nm,
        'profiles': profiles,
        'average_profile': np.asarray(avg, dtype=float),
        'fit_profile': np.asarray(fit['fit_profile'], dtype=float),
        'residual_profile': np.asarray(fit['residual_profile'], dtype=float),
        'background': float(fit['background']),
        'amplitude': float(fit['amplitude']),
        'amplitude_err': float(fit.get('amplitude_err', np.nan)),
        'sigma': float(fit['sigma']),
        'sigma_err': float(fit.get('sigma_err', np.nan)),
        'fwhm': float(fit['fwhm']),
        'fwhm_err': float(fit.get('fwhm_err', np.nan)),
        'r2': float(fit.get('r2', np.nan)),
        'rmse': float(fit.get('rmse', np.nan)),
        'line_rmse': line_rmse,
        'line_cv': line_cv,
        'per_line_rmse': per_line_rmse,
        'fov_width_nm': float((c['xmax'] - c['xmin'] + 1) * sx),
        'fov_height_nm': float((c['ymax'] - c['ymin'] + 1) * sy)
    }


def _tab10_render_current(self, idx=None):
    """Refresh all four Tab-10 panels for the selected confirmed ROI4 frame."""
    if getattr(self, '_tab10_roi4_stack', None) is None:
        return
    n = len(self._tab10_roi4_stack)
    if n <= 0:
        return
    if idx is None:
        idx = int(self.tab10_frame_var.get())
    idx = max(0, min(int(idx), n - 1))
    crop = _tab10_get_crop(self, idx)
    if crop is None:
        return

    sx, sy = self._tab10_roi4_scale
    center = _tab10_center(self, crop)
    cx, cy = float(center['x0']), float(center['y0'])
    angles = _tab10_get_reference_angles(self)
    field = _tab9_roi4_field(self, idx)

    # Panel 1: exact ROI4 crop from Tab 9, rebased to local 0..FOV nm.
    # Remove the previous colorbar before clearing the image axis so repeated
    # frame changes/FIT/TRACK operations never accumulate colorbar axes.
    if getattr(self, '_tab10_cbar_image', None) is not None:
        try: self._tab10_cbar_image.remove()
        except Exception: pass
        self._tab10_cbar_image = None
    self.tab10_ax_image.clear()
    cmap = getattr(self, '_tab10_roi4_display_cmap', getattr(self, '_tab9_roi1_cmap', 'gray'))
    clim = getattr(self, '_tab10_roi4_display_clim', getattr(self, '_tab9_roi1_clim', (0.0, 1.0)))
    self.tab10_ax_image.imshow(
        crop, cmap=cmap, origin='lower',
        extent=(0.0, crop.shape[1] * sx, 0.0, crop.shape[0] * sy),
        vmin=clim[0], vmax=clim[1], interpolation='nearest', aspect='equal'
    )
    self.tab10_ax_image.plot(cx * sx, cy * sy, marker='+', markersize=16,
                             markeredgewidth=2.5, linestyle='None', label='Fixed center')
    rmax = 2.0 * max(1.0, min(cx, crop.shape[1] - 1 - cx, cy, crop.shape[0] - 1 - cy))
    tline = np.array([-rmax, rmax], dtype=float)
    for angle in angles:
        th = np.deg2rad(float(angle))
        self.tab10_ax_image.plot((cx + tline*np.cos(th))*sx,
                                 (cy + tline*np.sin(th))*sy,
                                 linewidth=1.5, label=f'{angle:g}°')
    self.tab10_ax_image.set_title(f'ROI — Frame {idx + 1}/{n} | Field = {field:+.2f} mT')
    self.tab10_ax_image.set_xlabel('X (nm)'); self.tab10_ax_image.set_ylabel('Y (nm)')
    self.tab10_ax_image.set_xlim(0, crop.shape[1]*sx); self.tab10_ax_image.set_ylim(0, crop.shape[0]*sy)
    self.tab10_ax_image.set_aspect('equal', adjustable='box'); self.tab10_ax_image.grid(alpha=0.15)
    try:
        im = self.tab10_ax_image.images[-1]
        self._tab10_cbar_image = self.tab10_fig.colorbar(im, ax=self.tab10_ax_image, fraction=0.046, pad=0.04)
        self._tab10_cbar_image.set_label('Intensity')
    except Exception:
        self._tab10_cbar_image = None

    # Panel 2: individual full profiles + BLACK average.
    t_px, profiles, avg = _tab10_extract_profiles(self, crop, cx, cy, angles)
    sline = float(np.sqrt(sx*sy)) if sx > 0 and sy > 0 else 1.0
    t_nm = np.asarray(t_px, dtype=float) * sline
    self.tab10_ax_profiles.clear()
    for angle in angles:
        self.tab10_ax_profiles.plot(t_nm, profiles[float(angle)], linewidth=1.2, label=f'{angle:g}°')
    self.tab10_ax_profiles.plot(t_nm, avg, color='black', linewidth=2.8, label='Average')
    self.tab10_ax_profiles.axvline(0.0, linestyle='--', linewidth=1.0)
    self.tab10_ax_profiles.set_title(f'Selected Line Profiles')
    self.tab10_ax_profiles.set_xlabel('Position from fixed center (nm)'); self.tab10_ax_profiles.set_ylabel('Intensity')
    self.tab10_ax_profiles.legend(fontsize=8, loc='best'); self.tab10_ax_profiles.grid(alpha=0.2)

    # Keep Panel 3 clean until a fit is requested.
    if getattr(self, '_tab10_last_fit', None) is None:
        self.tab10_ax_fit.clear()
        self.tab10_ax_fit.set_title('Average full profile + fixed-center 1D Gaussian')
        self.tab10_ax_fit.set_xlabel('Position from fixed center (nm)'); self.tab10_ax_fit.set_ylabel('Intensity')
        self.tab10_ax_fit.grid(alpha=0.2)

    # Panel 4 is intentionally refreshed only by FIT/TRACK so that a frame
    # change does not destroy an already completed tracking plot.
    self.tab10_frame_var.set(idx)
    try:
        n = len(self._tab10_roi4_stack)
        field = float(_tab9_roi4_field(self, idx))
        self.tab10_frame_label_var.set(
            f'Frame No. {idx + 1}/{n} | Equivalent Field: {field:+.2f} mT'
        )
        self.tab10_frame_info_var.set(
            f'Frame No. {idx + 1}/{n} | Equivalent Field: {field:+.2f} mT'
        )
    except Exception:
        pass
    _tab10_update_fit_parameters(self, center=center, message='Center calculated; Gaussian fit not performed.')
    self.tab10_canvas.draw_idle()




def _tab10_sync_frame_range_controls(self, n=None):
    """Keep TAB 10 frame/range controls 1-based while tracking stays 0-based internally."""
    try:
        if n is None:
            n = len(self._tab10_roi4_stack)
        n = max(1, int(n))
    except Exception:
        n = 1

    try:
        start0 = int(self.tab10_start_var.get())
    except Exception:
        start0 = 0
    try:
        end0 = int(self.tab10_end_var.get())
    except Exception:
        end0 = n - 1

    start0 = max(0, min(start0, n - 1))
    end0 = max(0, min(end0, n - 1))
    if start0 > end0:
        start0, end0 = end0, start0

    self.tab10_start_var.set(start0)
    self.tab10_end_var.set(end0)

    try:
        self.tab10_start_display_var.set(str(start0 + 1))
        self.tab10_end_display_var.set(str(end0 + 1))
        self.tab10_start_spin.configure(from_=1, to=n, increment=1)
        self.tab10_end_spin.configure(from_=1, to=n, increment=1)
    except Exception:
        pass


def _tab10_range_spinbox_changed(self, which=None, event=None):
    """Apply a manually entered or arrow-changed 1-based frame range."""
    try:
        n = len(self._tab10_roi4_stack)
    except Exception:
        n = 0
    if n <= 0:
        return

    def parse(var, fallback):
        try:
            return int(round(float(var.get())))
        except Exception:
            return fallback

    start = parse(self.tab10_start_display_var, int(self.tab10_start_var.get()) + 1)
    end = parse(self.tab10_end_display_var, int(self.tab10_end_var.get()) + 1)

    start = max(1, min(start, n))
    end = max(1, min(end, n))

    if start > end:
        if which == 'start':
            start = end
        elif which == 'end':
            end = start
        else:
            start, end = end, start

    self.tab10_start_display_var.set(str(start))
    self.tab10_end_display_var.set(str(end))
    self.tab10_start_var.set(start - 1)
    self.tab10_end_var.set(end - 1)


def _tab10_frame_changed(self, value=None):
    try:
        idx = int(round(float(value))) if value is not None else int(self.tab10_frame_var.get())
    except Exception:
        idx = int(self.tab10_frame_var.get())
    if getattr(self, '_tab10_programmatic', False):
        return
    self._tab10_programmatic = True
    try:
        if getattr(self, '_tab10_roi4_stack', None) is None:
            _tab10_sync_roi4_from_tab9(self, preserve_frame=True)
        if getattr(self, '_tab10_roi4_stack', None) is not None:
            n = len(self._tab10_roi4_stack)
            idx = max(0, min(idx, n - 1))
            self.tab10_frame_var.set(idx)
            try:
                self.tab10_frame_label_var.set(
                    f'Frame No. {idx + 1}/{n} | Equivalent Field: '
                    f'{float(_tab9_roi4_field(self, idx)):+.6g} mT'
                )
                self.tab10_frame_info_var.set(
                    f'Frame No. {idx + 1}/{n} | Equivalent Field: '
                    f'{float(_tab9_roi4_field(self, idx)):+.6g} mT'
                )
            except Exception:
                pass
            self._tab10_last_fit = None
            _tab10_render_current(self, idx)
            _tab10_update_fit_parameters(
                self, message='Frame changed — fit current ROI4 when ready.'
            )
            self.tab10_fit_status_var.set(
                'Frame changed — fit current ROI4 when ready.'
            )
    finally:
        self._tab10_programmatic = False



def _tab10_recalculate_center(self):
    if getattr(self, '_tab10_roi4_stack', None) is None:
        if not _tab10_sync_roi4_from_tab9(self, preserve_frame=True):
            return
    try:
        frame=int(self.tab10_frame_var.get())
        crop=_tab10_get_crop(self,frame)
        center=_tab10_center(self,crop)
        self._tab10_center_cache=center
        _tab10_update_fit_parameters(self, center=center, message='Center recalculated; Gaussian fit not performed.')
        self._tab10_last_fit=None
        _tab10_render_current(self,frame)
        self.tab10_fit_status_var.set(f'✓ Center recalculated | ({center["x0"]:.3f}, {center["y0"]:.3f}) px local')
    except Exception as exc:
        messagebox.showerror('Tab 10 Center', str(exc), parent=self.root)



def _tab10_fit_current(self):
    """Fit the current confirmed ROI4 and refresh the four panels cleanly."""
    if getattr(self, '_tab10_roi4_stack', None) is None:
        if not _tab10_sync_roi4_from_tab9(self, preserve_frame=True):
            return
    try:
        frame = max(0, min(int(self.tab10_frame_var.get()), len(self._tab10_roi4_stack) - 1))
        result = _tab10_fit_roi4(self, frame)
        self._tab10_last_fit = result

        # Panel 1/2 refresh from the fitted current ROI4.
        _tab10_render_current(self, frame)

        # Panel 3: cleanly rebuild average profile + Gaussian + residual.
        if getattr(self, '_tab10_fit_residual_ax', None) is not None:
            try: self._tab10_fit_residual_ax.remove()
            except Exception: pass
            self._tab10_fit_residual_ax = None
        self.tab10_ax_fit.clear()
        self.tab10_ax_fit.plot(result['t_nm'], result['average_profile'], color='black', linewidth=1.7, label='Average full profile')
        self.tab10_ax_fit.plot(result['t_nm'], result['fit_profile'], color='blue', linewidth=2.3, linestyle='--', label='Fixed-center Gaussian fit')
        residual_ax = self.tab10_ax_fit.twinx(); self._tab10_fit_residual_ax = residual_ax
        residual_ax.plot(result['t_nm'], result['residual_profile'], linestyle=':', linewidth=1.2, label='Residual')
        residual_ax.axhline(0.0, linestyle='--', linewidth=0.8)
        residual_ax.set_ylabel('Residual')
        self.tab10_ax_fit.axvline(0.0, linestyle=':', linewidth=1.0)
        self.tab10_ax_fit.set_title(f'Average of {len(result["angles"])} Lines - Gaussian Fit')
        self.tab10_ax_fit.set_xlabel('Position from fixed center (nm)'); self.tab10_ax_fit.set_ylabel('Intensity')
        h1,l1=self.tab10_ax_fit.get_legend_handles_labels(); h2,l2=residual_ax.get_legend_handles_labels()
        self.tab10_ax_fit.legend(h1+h2,l1+l2,fontsize=8,loc='best'); self.tab10_ax_fit.grid(alpha=0.2)

        # Panel 4: one fitted point, X1=field and X2=frame number.
        if getattr(self, '_tab10_ax_tracking2', None) is not None:
            try: self._tab10_ax_tracking2.remove()
            except Exception: pass
            self._tab10_ax_tracking2=None
        if getattr(self, '_tab10_tracking_secx', None) is not None:
            try: self._tab10_tracking_secx.remove()
            except Exception: pass
            self._tab10_tracking_secx=None
        self.tab10_ax_tracking.clear()
        ax2=self.tab10_ax_tracking.twinx(); self._tab10_ax_tracking2=ax2
        field=float(result['field_mT'])
        self.tab10_ax_tracking.errorbar([field],[result['fwhm']],yerr=[result['fwhm_err']],marker='^',linewidth=1.5,capsize=2,color='black',label='FWHM')
        ax2.errorbar([field],[result['amplitude']],yerr=[result['amplitude_err']],marker='o',linestyle='--',linewidth=1.5,capsize=2,color='blue',label='Amplitude')
        self.tab10_ax_tracking.set_title('ROI — FWHM + Gaussian Amplitude')
        self.tab10_ax_tracking.set_xlabel('Equivalent Field (mT)'); self.tab10_ax_tracking.set_ylabel('FWHM (nm)'); ax2.set_ylabel('Gaussian Amplitude')
        self.tab10_ax_tracking.grid(alpha=0.2)
        top=self.tab10_ax_tracking.twiny(); top.set_xlim(self.tab10_ax_tracking.get_xlim()); top.set_xticks([field]); top.set_xticklabels([str(frame+1)]); top.set_xlabel('Frame number', labelpad=6); self._tab10_tracking_secx=top
        h1,l1=self.tab10_ax_tracking.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels(); self.tab10_ax_tracking.legend(h1+h2,l1+l2,fontsize=8,loc='best')

        _tab10_update_fit_parameters(self, result=result)
        self.tab10_fit_status_var.set(f'✓ ROI4 fitted | Frame {frame+1}/{len(self._tab10_roi4_stack)} | FWHM={result["fwhm"]:.6g} nm | Sigma={result["sigma"]:.6g} nm | Amplitude={result["amplitude"]:.6g} | R²={result["r2"]:.6g} | RMSE={result["rmse"]:.6g} | Line RMSE={result["line_rmse"]:.6g} | Line CV={result["line_cv"]:.6g}')
        _tab10_refresh_layout(self)
    except Exception as exc:
        self.tab10_fit_status_var.set(f'✗ Gaussian fit failed: {type(exc).__name__}: {exc}')
        try: _tab10_update_fit_parameters(self, message=f'Gaussian fit failed: {type(exc).__name__}: {exc}')
        except Exception: pass
        messagebox.showerror('Tab 10 Gaussian Fit', str(exc), parent=self.root)



def _tab10_use_all_frames(self):
    if not _tab10_roi4_available(self):
        messagebox.showwarning(
            'Tab 10',
            'Confirm ROI in Tab 9 first.',
            parent=self.root
        )
        return
    n = len(self._tab9_roi1_stack)
    self.tab10_start_var.set(0)
    self.tab10_end_var.set(max(0, n - 1))
    try:
        self.tab10_start_display_var.set('1')
        self.tab10_end_display_var.set(str(n))
        self.tab10_start_spin.configure(from_=1, to=n, increment=1)
        self.tab10_end_spin.configure(from_=1, to=n, increment=1)
    except Exception:
        pass
    self.tab10_frame_var.set(0)
    self.tab10_slider.set(0)
    try:
        field0 = float(_tab9_roi4_field(self, 0))
        self.tab10_frame_label_var.set(
            f'Frame No. 1/{n} | Equivalent Field: {field0:+.6g} mT'
        )
        self.tab10_frame_info_var.set(
            f'Frame No. 1/{n} | Equivalent Field: {field0:+.6g} mT'
        )
    except Exception:
        try: self.tab10_frame_label_var.set(f'Frame No. 1/{n} | Equivalent Field: —')
        except Exception: pass
    _tab10_render_current(self, 0)
    self.tab10_fit_status_var.set(f'Frame range set to all frames: 0–{n - 1}.')


def _tab10_plot_tracking(self, df, start, end, draw_now=True):
    """Render Figure 4 from the current tracking dataframe immediately."""
    if getattr(self, '_tab10_ax_tracking2', None) is not None:
        try: self._tab10_ax_tracking2.remove()
        except Exception: pass
    if getattr(self, '_tab10_tracking_secx', None) is not None:
        try: self._tab10_tracking_secx.remove()
        except Exception: pass
    self._tab10_ax_tracking2 = None
    self._tab10_tracking_secx = None
    self.tab10_ax_tracking.clear()

    ax2 = self.tab10_ax_tracking.twinx()
    self._tab10_ax_tracking2 = ax2
    if df is None or df.empty:
        self.tab10_ax_tracking.set_title('FWHM + Gaussian Amplitude')
        self.tab10_ax_tracking.set_xlabel('Equivalent Field (mT)', labelpad=8)
        self.tab10_ax_tracking.set_ylabel('FWHM (nm)', labelpad=8)
        ax2.set_ylabel('Gaussian Amplitude', labelpad=12)
    else:
        x = df['field_mT'].to_numpy(float)
        f = df['fwhm'].to_numpy(float)
        a = df['amplitude'].to_numpy(float)
        fe = df['fwhm_err'].to_numpy(float)
        ae = df['amplitude_err'].to_numpy(float)
        gf = np.isfinite(x) & np.isfinite(f)
        ga = np.isfinite(x) & np.isfinite(a)
        if np.any(gf):
            self.tab10_ax_tracking.errorbar(
                x[gf], f[gf], yerr=fe[gf], marker='^', linestyle='-',color='black',
                linewidth=1.5, capsize=2, label='FWHM'
            )
        if np.any(ga):
            ax2.errorbar(
                x[ga], a[ga], yerr=ae[ga], marker='o', linestyle='-',color='blue',
                linewidth=1.5, capsize=2, label='Amplitude'
            )

        # Keep the exact field/frame mapping for hysteresis or non-monotonic sweeps.
        top = self.tab10_ax_tracking.twiny()
        top.set_xlim(self.tab10_ax_tracking.get_xlim())
        valid = np.isfinite(x)
        inds = np.flatnonzero(valid)
        if inds.size:
            max_ticks = 12
            take = inds if inds.size <= max_ticks else inds[np.linspace(0, inds.size - 1, max_ticks, dtype=int)]
            top.set_xticks(x[take])
            top.set_xticklabels([str(int(df['frame'].iloc[i]) + 1) for i in take])
        top.set_xlabel('Frame number', labelpad=7)
        top.xaxis.set_label_position('top')
        top.xaxis.tick_top()
        self._tab10_tracking_secx = top

        self.tab10_ax_tracking.set_title(
            f'FWHM + Gaussian Amplitude | frames {start + 1}–{end + 1}', pad=5
        )
        self.tab10_ax_tracking.set_xlabel('Equivalent Field (mT)', labelpad=8)
        self.tab10_ax_tracking.set_ylabel('FWHM (nm)', labelpad=8)
        ax2.set_ylabel('Gaussian Amplitude', labelpad=12)
        ax2.yaxis.set_label_position('right')
        ax2.yaxis.tick_right()
        self.tab10_ax_tracking.grid(alpha=0.2)
        h1, l1 = self.tab10_ax_tracking.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        if h1 or h2:
            self.tab10_ax_tracking.legend(h1 + h2, l1 + l2, fontsize=8, loc='best')

    _tab10_refresh_layout(self)
    if draw_now:
        try:
            self.tab10_canvas.draw()
            self.root.update_idletasks()
        except Exception:
            pass


def _tab10_track_all(self):
    """
    TAB 10 TRACK-SELECTED FRAMES.

    Tracking sequence intentionally mirrors TAB 11 TRACK ALL ROIs:
        frame -> crop -> core detection -> fit only when detected.

    Only TAB 10 tracking is changed here. FIT CURRENT ROI, TAB 11,
    TAB 9, and all other functions/cells remain untouched.
    """
    if getattr(self, '_tab10_roi4_stack', None) is None:
        if not _tab10_sync_roi4_from_tab9(self, preserve_frame=True):
            return

    nframes = len(self._tab10_roi4_stack)
    if nframes <= 0:
        return

    start = max(0, min(int(self.tab10_start_var.get()), nframes - 1))
    end = max(0, min(int(self.tab10_end_var.get()), nframes - 1))
    if start > end:
        start, end = end, start

    # Read the option once for the complete tracking run, exactly as TAB 11.
    zero_mode = bool(
        getattr(self, 'tab10_zero_if_undetected_var', None) is not None
        and self.tab10_zero_if_undetected_var.get()
    )

    # Keep one fixed center for the complete tracking run.
    # _tab10_center() itself is NOT modified.
    self._tab10_tracking_fixed_center = None
    # Sequential detector state: reset for each tracking run.
    self._tab10_last_detected_core = None
    if zero_mode:
        try:
            cache = getattr(self, '_tab10_center_cache', None)
            if isinstance(cache, dict):
                self._tab10_tracking_fixed_center = dict(cache)
            else:
                first_crop = _tab10_get_crop(self, start)
                self._tab10_tracking_fixed_center = dict(
                    _tab10_center(self, first_crop)
                )
        except Exception:
            self._tab10_tracking_fixed_center = None

    rows = []
    self._tab10_tracking_df = pd.DataFrame()

    # Same tracking-panel initialization philosophy as TAB 11:
    # clear only the tracking result; do not alter FIT CURRENT ROI.
    _tab10_plot_tracking(
        self,
        self._tab10_tracking_df,
        start,
        end,
        draw_now=True
    )

    self.tab10_fit_status_var.set(
        f'⏳ Tracking ROI4 | frames {start + 1}–{end + 1} | 0/{end - start + 1}'
    )
    self.root.update_idletasks()

    total = end - start + 1
    done = 0
    detected_count = 0

    # ---------------------------------------------------------------
    # Same fundamental order as TAB 11 TRACK ALL ROIs:
    # frame -> crop -> detect -> fit only if valid core exists.
    # ---------------------------------------------------------------
    for frame in range(start, end + 1):
        try:
            crop = _tab10_get_crop(self, frame)

            if zero_mode:
                detected, det = _tab10_detect_particle_for_tracking(
                    self, crop
                )
            else:
                # Zero mode OFF: preserve the original TAB 10 fitting path.
                detected, det = True, None

            if not detected:
                # Exactly equivalent to TAB 11's undetected branch:
                # NO Gaussian fit is attempted.
                rows.append({
                    'roi_index': 4,
                    'frame': frame,
                    'field_mT': float(_tab9_roi4_field(self, frame)),
                    'fwhm': 0.0,
                    'fwhm_err': 0.0,
                    'amplitude': 0.0,
                    'amplitude_err': 0.0,
                    'sigma': 0.0,
                    'sigma_err': 0.0,
                    'r2': 0.0,
                    'rmse': 0.0,
                    'line_rmse': 0.0,
                    'line_cv': 0.0,
                    'core_detected': False,
                })
            else:
                # Same detector-first / fitter-second sequence as TAB 11.
                r = _tab10_fit_roi4(self, frame)

                if r is None:
                    raise RuntimeError(
                        'Particle detected but fitting returned no result.'
                    )

                rows.append({
                    k: v for k, v in r.items()
                    if k not in (
                        't_px', 't_nm', 'profiles', 'average_profile',
                        'fit_profile', 'residual_profile',
                        'per_line_rmse', 'angles'
                    )
                })
                rows[-1]['core_detected'] = True
                detected_count += 1
                if zero_mode and isinstance(det, dict):
                    # Carry only a validated core into the next frame.
                    self._tab10_last_detected_core = {
                        'x0': float(det.get('x0', np.nan)),
                        'y0': float(det.get('y0', np.nan)),
                    }

        except Exception as exc:
            # Match TAB 11 exception semantics:
            # Zero mode converts an unsuccessful detection/fitting attempt
            # to a zero measurement; normal mode preserves the old NaN path.
            rows.append({
                'roi_index': 4,
                'frame': frame,
                'field_mT': float(_tab9_roi4_field(self, frame)),
                'fwhm': 0.0 if zero_mode else np.nan,
                'fwhm_err': 0.0 if zero_mode else np.nan,
                'amplitude': 0.0 if zero_mode else np.nan,
                'amplitude_err': 0.0 if zero_mode else np.nan,
                'sigma': 0.0 if zero_mode else np.nan,
                'sigma_err': 0.0 if zero_mode else np.nan,
                'r2': 0.0 if zero_mode else np.nan,
                'rmse': 0.0 if zero_mode else np.nan,
                'line_rmse': 0.0 if zero_mode else np.nan,
                'line_cv': 0.0 if zero_mode else np.nan,
                'core_detected': False,
                'error': f'{type(exc).__name__}: {exc}',
            })

        done += 1

        # Keep the existing TAB 10 live tracking plot behavior.
        self._tab10_tracking_df = pd.DataFrame(rows)
        _tab10_plot_tracking(
            self,
            self._tab10_tracking_df,
            start,
            end,
            draw_now=True
        )

        self.tab10_fit_status_var.set(
            f'⏳ ROI4 tracking | frame {frame + 1}/{nframes} | '
            f'{done}/{total} | cores detected: {detected_count}'
        )
        self.root.update_idletasks()

    # Final tracking result.
    self._tab10_tracking_df = pd.DataFrame(rows)
    _tab10_plot_tracking(
        self,
        self._tab10_tracking_df,
        start,
        end,
        draw_now=True
    )

    self.tab10_fit_status_var.set(
        f'✓ ROI4 tracking complete | frames {start + 1}–{end + 1} | '
        f'{len(rows)} measurements | {detected_count} particle cores detected'
    )

    self._tab10_tracking_fixed_center = None



def _tab10_export_current_profile_csv(self):
    r = getattr(self, '_tab10_last_fit', None)
    if not isinstance(r, dict):
        messagebox.showwarning(
            'Tab 10 Export',
            'Run FIT CURRENT SELECTED ROI first.',
            parent=self.root
        )
        return
    path = filedialog.asksaveasfilename(
        parent=self.root,
        title='Export ROI4 line-profile CSV',
        defaultextension='.csv',
        filetypes=[('CSV', '*.csv'), ('All files', '*.*')],
        initialfile=f'ROI4_Frame{r["frame"] + 1:03d}_LineProfiles.csv'
    )
    if not path:
        return
    data = {'Position_nm': r['t_nm']}
    for angle in r['angles']:
        data[f'Profile_{angle:g}deg'] = r['profiles'][float(angle)]
    data['Average_Profile'] = r['average_profile']
    data['Gaussian_Fit'] = r['fit_profile']
    data['Residual'] = r['residual_profile']
    pd.DataFrame(data).to_csv(path, index=False)
    self.tab10_export_status_var.set(
        f'✓ Exported current ROI4 line-profile CSV: {Path(path).name}'
    )


def _tab10_export_tracking_csv(self):
    df = getattr(self, '_tab10_tracking_df', None)
    if df is None or df.empty:
        messagebox.showwarning(
            'Tab 10 Export',
            'Run TRACK ALL ROIs — FWHM + AMPLITUDE vs FIELD first.',
            parent=self.root
        )
        return
    path = filedialog.asksaveasfilename(
        parent=self.root,
        title='Export Figure-4 ROI4 tracking CSV',
        defaultextension='.csv',
        filetypes=[('CSV', '*.csv'), ('All files', '*.*')],
        initialfile='FWHM_Amp_ROI4.csv'
    )
    if not path:
        return
    df.to_csv(path, index=False)
    self.tab10_export_status_var.set(
        f'✓ Exported Figure-4 ROI4 tracking CSV: {Path(path).name}'
    )


def _tab10_tab_changed(self,event=None):
    try:
        if self.nb.select()==str(self.tab_single_roi10):
            _tab10_sync_roi4_from_tab9(self,preserve_frame=True)
    except Exception as exc:
        try:
            self.tab10_status_var.set(
                f'Could not fetch ROI from Tab 9: '
                f'{type(exc).__name__}: {exc}'
            )
        except Exception:
            pass





CDIWorkflowApp._tab10_tab_changed = _tab10_tab_changed


def _tab10_update_fit_parameters(self, result=None, center=None, message=None):
    """Refresh the scrollable Tab-10 parameter panel without changing analysis."""
    vars_ = getattr(self, 'tab10_param_vars', {})
    if not vars_:
        return
    vals = {k: '—' for k in vars_}
    if center is not None:
        vals.update({
            'center': f"({float(center.get('x0', np.nan)):.4f}, {float(center.get('y0', np.nan)):.4f}) px",
            'background': f"{float(center.get('background', np.nan)):.6g}",
            'threshold': f"{float(center.get('threshold', np.nan)):.6g}",
            'sign': f"{float(center.get('sign', np.nan)):.0f}",
        })
    if result is not None:
        vals.update({
            'center': f"({result['local_center_x_px']:.4f}, {result['local_center_y_px']:.4f}) px",
            'background': f"{result['background']:.6g}",
            'threshold': f"{result['contrast_threshold']:.6g}",
            'sign': f"{result['contrast_sign']:.0f}",
            'amplitude': f"{result['amplitude']:.6g} ± {result['amplitude_err']:.3g}",
            'sigma': f"{result['sigma']:.6g} ± {result['sigma_err']:.3g} nm",
            'fwhm': f"{result['fwhm']:.6g} ± {result['fwhm_err']:.3g} nm",
            'r2': f"{result['r2']:.6g}",
            'rmse': f"{result['rmse']:.6g}",
            'line_rmse': f"{result['line_rmse']:.6g}",
            'line_cv': f"{result['line_cv']:.6g}",
            'fov': f"{result['fov_width_nm']:.6g} × {result['fov_height_nm']:.6g} nm",
            'frame': f"{result['frame'] + 1} / {len(self._tab10_roi4_stack)}",
            'field': f"{result['field_mT']:+.6g} mT",
        })
    if message is not None:
        vals['status'] = str(message)
    else:
        vals['status'] = 'Gaussian fit not performed.' if result is None else 'Gaussian fit complete.'
    for k, v in vals.items():
        if k in vars_:
            vars_[k].set(v)


def _tab10_refresh_layout(self):
    """Keep the Tab-6-style 2x2 layout stable and fully responsive."""
    try:
        fig = self.tab10_fig
        # Do NOT call tight_layout here: twin Y/X axes used by Figure 4 can
        # be moved/clipped by repeated tight_layout calls.  Fixed margins
        # provide predictable space for all titles, labels and legends.
        fig.subplots_adjust(
            left=0.075, right=0.905, bottom=0.095, top=0.905,
            wspace=0.30, hspace=0.48
        )

        # Figure 4: reserve explicit space for the right Y2 label and top X2.
        ax = getattr(self, 'tab10_ax_tracking', None)
        ax2 = getattr(self, '_tab10_ax_tracking2', None)
        secx = getattr(self, '_tab10_tracking_secx', None)
        if ax is not None:
            ax.set_xlabel('Equivalent field (mT)', labelpad=8)
            ax.set_ylabel('FWHM (nm)', labelpad=8)
            ax.title.set_pad(5)
        if ax2 is not None:
            ax2.set_ylabel('Gaussian Amplitude', labelpad=12)
            ax2.yaxis.set_label_position('right')
            ax2.yaxis.tick_right()
        if secx is not None:
            secx.set_xlabel('Frame number', labelpad=7)
            secx.xaxis.set_label_position('top')
            secx.xaxis.tick_top()

        # Keep titles/labels separated between the top and bottom panels.
        for a in (
            getattr(self, 'tab10_ax_image', None),
            getattr(self, 'tab10_ax_profiles', None),
            getattr(self, 'tab10_ax_fit', None),
        ):
            if a is not None:
                a.title.set_pad(5)

        self.tab10_canvas.draw_idle()
    except Exception as exc:
        try:
            self.tab10_fit_status_var.set(
                f'Layout adjustment warning: {type(exc).__name__}: {exc}'
            )
        except Exception:
            pass


def _tab10_export_figures_png(self):
    """Export the four visible Tab-10 panels as separate 300-ppi PNG files."""
    if getattr(self, 'tab10_fig', None) is None:
        messagebox.showwarning('Tab 10 Export', 'Tab 10 figure is not available.', parent=self.root)
        return
    folder = filedialog.askdirectory(parent=self.root, title='Select folder for Tab 10 figure export')
    if not folder:
        return
    fig = self.tab10_fig
    all_axes = list(fig.axes)
    frame = int(getattr(self, 'tab10_frame_var', tk.IntVar(value=0)).get()) + 1
    field = np.nan
    try:
        field = float(_tab9_roi4_field(self, frame - 1))
    except Exception:
        pass
    field_txt = f'{field:+.2f}mT' if np.isfinite(field) else 'FieldNA'
    groups = [
        ('Figure1_ROI4_Image', [self.tab10_ax_image] + ([self._tab10_cbar_image.ax] if getattr(self, '_tab10_cbar_image', None) is not None else [])),
        ('Figure2_LineProfiles', [self.tab10_ax_profiles]),
        ('Figure3_GaussianFit', [self.tab10_ax_fit] + ([self._tab10_fit_residual_ax] if getattr(self, '_tab10_fit_residual_ax', None) is not None else [])),
        ('Figure4_FWHM_Amplitude_vs_Field', [self.tab10_ax_tracking] + ([self._tab10_ax_tracking2] if getattr(self, '_tab10_ax_tracking2', None) is not None else []) + ([self._tab10_tracking_secx] if getattr(self, '_tab10_tracking_secx', None) is not None else [])),
    ]
    visible_state = {ax: ax.get_visible() for ax in all_axes}
    written = []
    try:
        for stem, keep_axes in groups:
            keep = {ax for ax in keep_axes if ax is not None and ax in all_axes}
            for ax in all_axes:
                ax.set_visible(ax in keep)
            fig.canvas.draw()
            path = os.path.join(folder, f'Tab10_ROI4_{stem}_Frame{frame:03d}_{field_txt}.png')
            fig.savefig(path, dpi=300, bbox_inches='tight', pad_inches=0.15, facecolor='white')
            written.append(path)
    finally:
        for ax, state in visible_state.items():
            try:
                ax.set_visible(state)
            except Exception:
                pass
        fig.canvas.draw_idle()
    self.tab10_export_status_var.set(f'✓ Exported {len(written)} separate Tab 10 figures at 300 ppi.')
    messagebox.showinfo('Tab 10 Export', f'Exported {len(written)} separate PNG figures (300 ppi) to:\n{folder}', parent=self.root)


def _tab10_build(self):
    """Build Tab 10 as a full-client-area, responsive 2x2 plot + scrollable controls layout."""
    if not hasattr(self, 'tab_single_roi10'):
        self.tab_single_roi10 = ttk.Frame(self.nb)
        self.nb.add(self.tab_single_roi10, text='10. Single-ROI Fit')

    tab = self.tab_single_roi10
    # Let Tab 10 occupy the entire notebook client area; do not constrain the
    # plotting/controls pane to a fixed height or width.
    tab.grid_rowconfigure(0, weight=1)
    tab.grid_columnconfigure(0, weight=1)
    tab.pack_propagate(False)
    for child in list(tab.winfo_children()):
        try:
            child.destroy()
        except Exception:
            pass

    self.tab10_contrast_fraction_var = tk.DoubleVar(value=0.50)
    self.tab10_ref_mode_var = tk.StringVar(value='2')
    self.tab10_frame_var = tk.IntVar(value=0)
    self.tab10_start_var = tk.IntVar(value=0)
    self.tab10_end_var = tk.IntVar(value=0)
    self.tab10_status_var = tk.StringVar(value='Use the confirmed ROI from Tab 9.')
    self.tab10_fit_status_var = tk.StringVar(value='No Gaussian fit performed.')
    self.tab10_export_status_var = tk.StringVar(value='')
    self.tab10_zero_if_undetected_var = tk.BooleanVar(value=False)
    self.tab10_frame_info_var = tk.StringVar(value='Frame No. 1/1 | Equivalent Field: — mT')
    self.tab10_start_display_var = tk.StringVar(value='1')
    self.tab10_end_display_var = tk.StringVar(value='1')
    self._tab10_roi4_coords = None
    self._tab10_roi4_stack = None
    self._tab10_roi4_scale = (1.0, 1.0)
    self._tab10_roi4_confirmed = False
    self._tab10_last_fit = None
    self._tab10_tracking_df = None
    self._tab10_programmatic = False
    self._tab10_fit_residual_ax = None
    self._tab10_ax_tracking2 = None
    self._tab10_tracking_secx = None
    self._tab10_cbar_image = None

    # Adjustable horizontal divider: drag the separator between plots and controls.
    paned = ttk.Panedwindow(tab, orient='horizontal')
    paned.grid(row=0, column=0, sticky='nsew', padx=0, pady=0)
    paned.grid_rowconfigure(0, weight=1)
    paned.grid_columnconfigure(0, weight=1)
    self.tab10_paned = paned

    # LEFT: four-panel plotting area.
    plotbox = ttk.LabelFrame(paned, text='SINGLE ROI — 1D GAUSSIAN / FWHM', padding=2)
    plotbox.grid_rowconfigure(0, weight=1)
    plotbox.grid_columnconfigure(0, weight=1)

    # Tab-6-style four-panel figure: stable margins so titles/axis labels/twin axes
    # remain inside the plotting canvas at all window sizes.
    self.tab10_fig = plt.Figure(figsize=(12.0, 8.0), dpi=100, facecolor='white')
    gs = self.tab10_fig.add_gridspec(
        2, 2,
        left=0.07, right=0.92, bottom=0.095, top=0.905,
        wspace=0.40, hspace=0.48
    )
    self.tab10_ax_image = self.tab10_fig.add_subplot(gs[0, 0])
    self.tab10_ax_profiles = self.tab10_fig.add_subplot(gs[0, 1])
    self.tab10_ax_fit = self.tab10_fig.add_subplot(gs[1, 0])
    self.tab10_ax_tracking = self.tab10_fig.add_subplot(gs[1, 1])

    dummy = np.zeros((10, 10), dtype=float)
    self.tab10_ax_image.imshow(dummy, cmap='gray', origin='lower', extent=(0, 10, 0, 10),
                               vmin=0, vmax=1, interpolation='nearest', aspect='equal')
    self.tab10_ax_image.set_xlabel('X (nm)')
    self.tab10_ax_image.set_ylabel('Y (nm)')
    self.tab10_ax_profiles.set_xlabel('Position from fixed center (nm)')
    self.tab10_ax_profiles.set_ylabel('Intensity')
    self.tab10_ax_fit.set_xlabel('Position from fixed center (nm)')
    self.tab10_ax_fit.set_ylabel('Intensity')
    self.tab10_ax_tracking.set_xlabel('Equivalent field (mT)')
    self.tab10_ax_tracking.set_ylabel('FWHM (nm)')

    self.tab10_canvas = FigureCanvasTkAgg(self.tab10_fig, master=plotbox)
    self.tab10_canvas.draw()
    self.tab10_canvas.get_tk_widget().grid(row=0, column=0, sticky='nsew')

    # RIGHT: scrollable controls + fitting parameters + exports.
    side = ttk.Frame(paned)
    side.grid_rowconfigure(0, weight=1)
    side.grid_columnconfigure(0, weight=1)
    side.grid_columnconfigure(1, weight=0)
    paned.add(plotbox, weight=7)
    paned.add(side, weight=2)

    side_canvas = tk.Canvas(side, highlightthickness=0, borderwidth=0, width=355)
    side_scroll = ttk.Scrollbar(side, orient='vertical', command=side_canvas.yview)
    side_canvas.configure(yscrollcommand=side_scroll.set)
    side_canvas.grid(row=0, column=0, sticky='nsew')
    side_scroll.grid(row=0, column=1, sticky='ns')

    side_inner = ttk.Frame(side_canvas, padding=7)
    side_window = side_canvas.create_window((0, 0), window=side_inner, anchor='nw')
    def _side_sync_width(event=None):
        try:
            side_canvas.itemconfigure(side_window, width=side_canvas.winfo_width())
        except Exception:
            pass
    def _side_scroll_region(event=None):
        try:
            side_canvas.configure(scrollregion=side_canvas.bbox('all'))
        except Exception:
            pass
    side_canvas.bind('<Configure>', _side_sync_width)
    side_inner.bind('<Configure>', _side_scroll_region)
    side_canvas.bind('<MouseWheel>', lambda e: side_canvas.yview_scroll(int(-e.delta / 120), 'units'))
    side_inner.bind('<MouseWheel>', lambda e: side_canvas.yview_scroll(int(-e.delta / 120), 'units'))

    # Fetch/fit controls.
    fetch_box = ttk.LabelFrame(side_inner, text='ROI4 SOURCE', padding=6)
    fetch_box.pack(fill='x', pady=(0, 7))
    ttk.Button(fetch_box, text='ROI from Tab 9', command=lambda: _tab10_sync_roi4_from_tab9(self, preserve_frame=True)).pack(fill='x', pady=2)
    ttk.Label(fetch_box, textvariable=self.tab10_status_var, wraplength=320, justify='left').pack(fill='x', pady=(4, 0))

    settings = ttk.LabelFrame(side_inner, text='ANALYSIS SETTINGS', padding=6)
    settings.pack(fill='x', pady=7)
    ttk.Label(settings, text='REFERENCE LINES').grid(row=0, column=0, sticky='w', pady=3)
    combo = ttk.Combobox(settings, textvariable=self.tab10_ref_mode_var, values=('1', '2', '4', '6'), state='readonly', width=7)
    combo.grid(row=0, column=1, sticky='e', pady=3)
    combo.bind('<<ComboboxSelected>>', lambda e: _tab10_update_reference_label(self, e))
    self.tab10_ref_label = ttk.Label(settings, text='2 lines: 0° / 90°')
    self.tab10_ref_label.grid(row=1, column=0, columnspan=2, sticky='w', pady=(0, 5))
    ttk.Label(settings, text='CONTRAST THRESHOLD').grid(row=2, column=0, sticky='w', pady=3)
    tk.Spinbox(settings, from_=0.0, to=0.95, increment=0.05, textvariable=self.tab10_contrast_fraction_var, width=8).grid(row=2, column=1, sticky='e', pady=3)

    action = ttk.LabelFrame(side_inner, text='FIT / CENTER / TRACKING', padding=6)
    action.pack(fill='x', pady=7)
    ttk.Button(action, text='RECALCULATE CENTER', command=lambda: _tab10_recalculate_center(self)).pack(fill='x', pady=2)
    ttk.Button(action, text='FIT CURRENT ROI', command=lambda: _tab10_fit_current(self)).pack(fill='x', pady=2)
    ttk.Button(action, text='TRACK-SELECTED FRAMES', command=lambda: _tab10_track_all(self)).pack(fill='x', pady=2)
    ttk.Checkbutton(action, text='Zero if Undetected', variable=self.tab10_zero_if_undetected_var).pack(anchor='w', pady=(0, 2))
    ttk.Button(action, text='USE ALL FRAMES', command=lambda: _tab10_use_all_frames(self)).pack(fill='x', pady=2)

    frame_box = ttk.LabelFrame(side_inner, text='FRAME / RANGE', padding=6)
    frame_box.pack(fill='x', pady=7)
    frame_box.columnconfigure(0, weight=1)
    frame_box.columnconfigure(1, weight=0)

    self.tab10_frame_label_var = tk.StringVar(
        value='Frame No. 1/1 | Equivalent Field: — mT'
    )
    ttk.Label(
        frame_box, textvariable=self.tab10_frame_label_var
    ).grid(row=0, column=0, columnspan=2, sticky='w', pady=(0, 3))

    self.tab10_slider = tk.Scale(
        frame_box, from_=0, to=0,
        orient='horizontal',
        variable=self.tab10_frame_var,
        resolution=1,
        showvalue=False,
        highlightthickness=0, bd=1, length=1,
        command=lambda v: _tab10_frame_changed(self, v)
    )
    self.tab10_slider.grid(
        row=1, column=0, columnspan=2, sticky='ew', pady=3
    )

    ttk.Label(frame_box, text='START').grid(
        row=2, column=0, sticky='w', pady=3
    )
    self.tab10_start_spin = tk.Spinbox(
        frame_box, from_=1, to=1, increment=1,
        textvariable=self.tab10_start_display_var,
        width=8,
        command=lambda: _tab10_range_spinbox_changed(self, 'start')
    )
    self.tab10_start_spin.grid(row=2, column=1, sticky='e')
    self.tab10_start_spin.bind(
        '<Return>', lambda e: _tab10_range_spinbox_changed(self, 'start', e)
    )
    self.tab10_start_spin.bind(
        '<FocusOut>', lambda e: _tab10_range_spinbox_changed(self, 'start', e)
    )

    ttk.Label(frame_box, text='END').grid(
        row=3, column=0, sticky='w', pady=3
    )
    self.tab10_end_spin = tk.Spinbox(
        frame_box, from_=1, to=1, increment=1,
        textvariable=self.tab10_end_display_var,
        width=8,
        command=lambda: _tab10_range_spinbox_changed(self, 'end')
    )
    self.tab10_end_spin.grid(row=3, column=1, sticky='e')
    self.tab10_end_spin.bind(
        '<Return>', lambda e: _tab10_range_spinbox_changed(self, 'end', e)
    )
    self.tab10_end_spin.bind(
        '<FocusOut>', lambda e: _tab10_range_spinbox_changed(self, 'end', e)
    )


    param_box = ttk.LabelFrame(side_inner, text='FITTING PARAMETERS', padding=6)
    param_box.pack(fill='x', pady=7)
    param_keys = [
        ('center', 'Fixed center'), ('background', 'Gaussian background'), ('threshold', 'Contrast threshold'),
        ('sign', 'Contrast sign'), ('amplitude', 'Amplitude'), ('sigma', 'Sigma'), ('fwhm', 'FWHM'),
        ('r2', 'R²'), ('rmse', 'RMSE'), ('line_rmse', 'Line RMSE'), ('line_cv', 'Line CV'),
        ('fov', 'ROI4 FOV'), ('frame', 'Frame'), ('field', 'Equivalent field')
    ]
    self.tab10_param_vars = {}
    for r, (key, label) in enumerate(param_keys):
        var = tk.StringVar(value='—')
        self.tab10_param_vars[key] = var
        ttk.Label(param_box, text=label + ':').grid(row=r, column=0, sticky='w', padx=(0, 6), pady=2)
        ttk.Label(param_box, textvariable=var, justify='right').grid(row=r, column=1, sticky='e', pady=2)
    self.tab10_param_vars['status'] = tk.StringVar(value='Gaussian fit not performed.')

    export_box = ttk.LabelFrame(side_inner, text='EXPORT', padding=6)
    export_box.pack(fill='x', pady=7)
    ttk.Button(export_box, text='EXPORT FIGURES', command=lambda: _tab10_export_figures_png(self)).pack(fill='x', pady=2)
    ttk.Button(export_box, text='EXPORT CURRENT ROI LINE PROFILE', command=lambda: _tab10_export_current_profile_csv(self)).pack(fill='x', pady=2)
    ttk.Button(export_box, text='EXPORT FIGURE-4 TRACKING', command=lambda: _tab10_export_tracking_csv(self)).pack(fill='x', pady=2)
    ttk.Label(export_box, textvariable=self.tab10_export_status_var, wraplength=320, justify='left').pack(fill='x', pady=(4, 0))

    ttk.Label(side_inner, text='Fit status:', font=('TkDefaultFont', 9, 'bold')).pack(anchor='w', pady=(7, 2))
    ttk.Label(side_inner, textvariable=self.tab10_fit_status_var, wraplength=320, justify='left').pack(fill='x')

    self.tab10_slider.configure(state='disabled')
    self.nb.bind('<<NotebookTabChanged>>', self._tab10_tab_changed, add='+')

    # Expose stable instance callbacks for any later UI/event code.
    self._tab10_render_current = lambda idx=None: _tab10_render_current(self, idx)
    self._tab10_frame_changed = lambda value=None: _tab10_frame_changed(self, value)
    self._tab10_update_reference_label = lambda event=None: _tab10_update_reference_label(self, event)
    self._tab10_refresh_layout = lambda: _tab10_refresh_layout(self)

    if _tab10_roi4_available(self):
        _tab10_sync_roi4_from_tab9(self, preserve_frame=True)

    # Initial clean layout. The divider controls the plot/control split.
    _tab10_refresh_layout(self)


# Install Tab 10 after Tab 9 has been installed.
CDIWorkflowApp._tab10_build = _tab10_build
CDIWorkflowApp._tab10_refresh_layout = _tab10_refresh_layout
CDIWorkflowApp._tab10_sync_roi4_from_tab9 = _tab10_sync_roi4_from_tab9
CDIWorkflowApp._tab10_fetch_roi4_button = _tab10_sync_roi4_from_tab9
CDIWorkflowApp._tab10_render_current = _tab10_render_current
CDIWorkflowApp._tab10_frame_changed = _tab10_frame_changed
CDIWorkflowApp._tab10_update_reference_label = _tab10_update_reference_label
CDIWorkflowApp._tab10_fit_current = _tab10_fit_current
CDIWorkflowApp._tab10_recalculate_center = _tab10_recalculate_center
CDIWorkflowApp._tab10_use_all_frames = _tab10_use_all_frames
CDIWorkflowApp._tab10_track_all = _tab10_track_all
CDIWorkflowApp._tab10_export_current_profile_csv = _tab10_export_current_profile_csv
CDIWorkflowApp._tab10_export_tracking_csv = _tab10_export_tracking_csv
CDIWorkflowApp._tab10_export_figures_png = _tab10_export_figures_png

_TAB10_BASE_BUILD_UI = CDIWorkflowApp._build_ui

def _build_ui_with_tab10(self, *args, **kwargs):
    result = _TAB10_BASE_BUILD_UI(self, *args, **kwargs)
    try:
        _tab10_build(self)
    except Exception as exc:
        try:
            self.log(
                f'Tab 10 Single-ROI Gaussian initialization warning: '
                f'{type(exc).__name__}: {exc}'
            )
        except Exception:
            pass
    return result

CDIWorkflowApp._build_ui = _build_ui_with_tab10

# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# TAB 11 — MULTI-ROI 1D GAUSSIAN / FWHM
# ---------------------------------------------------------------------------
# Layout requested:
#   3 rows x 2 columns
#   Row 1: [Processed Primary ROI1 from Tab 6] [Selected ROI]
#   Row 2: [Current ROI average profile + Gaussian + residual]
#          [Current ROI FWHM + amplitude vs field/frame]
#   Row 3: [Average FWHM + amplitude of ALL selected ROIs vs field/frame]
#
# Both the figure panel and the right control panel support scrolling.
# The divider between them is adjustable through a horizontal Panedwindow.
# ---------------------------------------------------------------------------

def _tab11_field_mT(self, idx):
    try:
        if getattr(self, 'real_fields_mT', None) is not None and len(self.real_fields_mT) > idx:
            return float(self.real_fields_mT[idx])
    except Exception:
        pass
    try:
        return float(_tab9_roi4_field(self, idx))
    except Exception:
        return float('nan')


def _tab11_scale_nm(self):
    try:
        v = _tab6_effective_nm_per_output_px(self)
        if v is not None:
            sx, sy = float(v[0]), float(v[1])
            if np.isfinite(sx) and np.isfinite(sy) and sx > 0 and sy > 0:
                return sx, sy
    except Exception:
        pass
    try:
        v = float(self.primary_roi_final_nm_per_pixel)
        if np.isfinite(v) and v > 0:
            return v, v
    except Exception:
        pass
    return None, None


def _tab11_primary_processed_image(self, idx):
    """Fetch the EXACT image displayed by Tab 6 ROI Processing for this frame.

    Pipeline is intentionally identical to Tab 6:
        _proc_image()  -> selected Filter/Brightness/Contrast processing
        -> current View (Image / FFT / IFFT)
        -> local support mask handling
        -> normalized 0..1 display data

    The Tab-6 colormap is read dynamically at fetch/render time so Tab 11
    always follows the current Tab-6 color selection.
    """
    if getattr(self, 'recons', None) is None:
        return None
    if not getattr(self, 'roi_resolution_applied', False):
        return None

    try:
        image = np.asarray(_proc_image(self, int(idx)), dtype=np.float32)
        view = str(getattr(self, 'view_var').get() or 'Image')

        if view == 'Image':
            display = image

        elif view == 'FFT':
            _, display = _proc_fft(self, image)

        else:
            # Exact Tab-6 IFFT path.
            F, _ = _proc_fft(self, image)
            Fm, _ = _proc_fft_mask(self, F, self.fft_mask_var.get())
            r = np.abs(np.fft.ifft2(np.fft.ifftshift(Fm)))
            mn = float(np.nanmin(r))
            mx = float(np.nanmax(r))
            display = (r - mn) / (mx - mn + 1e-12)
            display = np.where(
                np.asarray(self.local_mask, dtype=bool),
                display,
                0.0
            )

        return np.asarray(display, dtype=np.float32)

    except Exception as exc:
        try:
            self.tab11_status_var.set(
                f'Cannot fetch Tab-6 processed/display image: '
                f'{type(exc).__name__}: {exc}'
            )
        except Exception:
            pass
        return None


def _tab11_fetch_primary(self, preserve_frame=True):
    """Fetch the processed Primary ROI1 exactly as displayed in Tab 6."""
    if getattr(self, 'recons', None) is None:
        self.tab11_status_var.set('No reconstruction data loaded.')
        return False
    if not getattr(self, 'roi_resolution_applied', False):
        self.tab11_status_var.set(
            'Tab 6 ROI Processing is not ready. Confirm Primary ROI and apply resolution first.'
        )
        return False

    sx, sy = _tab11_scale_nm(self)
    if sx is None or sy is None:
        self.tab11_status_var.set('Primary ROI physical calibration is unavailable.')
        return False

    try:
        old_idx = int(self.tab11_frame_var.get()) if preserve_frame else 0
    except Exception:
        old_idx = 0

    n = len(self.recons)
    old_idx = max(0, min(old_idx, n - 1))

    frames = []
    for i in range(n):
        arr = _tab11_primary_processed_image(self, i)
        if arr is None:
            self.tab11_status_var.set(
                f'Failed to fetch processed Primary ROI frame {i + 1}/{n}.'
            )
            return False
        frames.append(arr)

    self._tab11_primary_stack = np.asarray(frames, dtype=np.float32)
    self._tab11_scale_nm = (float(sx), float(sy))
    self._tab11_primary_ready = True

    # Always follow the live Tab-6 visualization settings.
    try:
        self._tab11_cmap = str(self.colormap_var.get() or 'gray')
    except Exception:
        self._tab11_cmap = str(getattr(self, '_tab11_cmap', 'gray') or 'gray')
    self._tab11_view = str(getattr(self, 'view_var').get() or 'Image')

    h, w = self._tab11_primary_stack[0].shape[:2]
    self._tab11_primary_extent = (0.0, w * sx, 0.0, h * sy)

    finite = np.isfinite(self._tab11_primary_stack[old_idx])
    if np.any(finite):
        self._tab11_clim = (
            float(np.nanmin(self._tab11_primary_stack[old_idx][finite])),
            float(np.nanmax(self._tab11_primary_stack[old_idx][finite]))
        )
        if self._tab11_clim[1] <= self._tab11_clim[0]:
            self._tab11_clim = (0.0, 1.0)
    else:
        self._tab11_clim = (0.0, 1.0)

    self._tab11_programmatic = True
    try:
        self.tab11_frame_var.set(old_idx)
        self.tab11_slider.configure(
            from_=0, to=max(0, n - 1), resolution=1, state='normal'
        )
        self.tab11_slider.set(old_idx)
        self.tab11_start_spin.configure(from_=0, to=max(0, n - 1), increment=1)
        self.tab11_end_spin.configure(from_=0, to=max(0, n - 1), increment=1)
        self.tab11_start_var.set(0)
        self.tab11_end_var.set(n - 1)
    finally:
        self._tab11_programmatic = False

    self.tab11_status_var.set(
        f'✓ Tab-6 {getattr(self, "_tab11_view", "Image")} processed Primary ROI1 fetched | '
        f'FOV = {w*sx:.6g} × {h*sy:.6g} nm | Frame {old_idx + 1}/{n}'
    )
    self._tab11_render( old_idx)
    return True


def _tab11_selected_roi_index(self):
    rois = getattr(self, '_tab11_rois', [])
    if not rois:
        return 0
    text = str(self.tab11_roi_select_var.get())
    m = re.search(r'(\d+)', text)
    try:
        idx = int(m.group(1)) - 1 if m else int(text)
    except Exception:
        idx = 0
    return max(0, min(idx, len(rois) - 1))


def _tab11_update_roi_selector(self, select_idx=None):
    rois = getattr(self, '_tab11_rois', [])
    values = [f'ROI {i + 1}' for i in range(len(rois))]
    self.tab11_roi_select_combo.configure(values=values)
    if not values:
        self.tab11_roi_select_var.set('')
        return
    if select_idx is None:
        select_idx = self._tab11_selected_roi_index(self)
    select_idx = max(0, min(int(select_idx), len(values) - 1))
    self.tab11_roi_select_var.set(values[select_idx])


def _tab11_mouse_press(self, event):
    if not getattr(self, '_tab11_select_mode', False):
        return
    if event.inaxes is not self.tab11_ax_primary or event.xdata is None or event.ydata is None:
        return
    self._tab11_press_xy = (float(event.xdata), float(event.ydata))


def _tab11_mouse_move(self, event):
    if (
        not getattr(self, '_tab11_select_mode', False)
        or getattr(self, '_tab11_press_xy', None) is None
        or event.inaxes is not self.tab11_ax_primary
        or event.xdata is None or event.ydata is None
    ):
        return

    x0, y0 = self._tab11_press_xy
    x1, y1 = float(event.xdata), float(event.ydata)
    sx, sy = self._tab11_scale_nm
    xmin_nm, xmax_nm = sorted((x0, x1))
    ymin_nm, ymax_nm = sorted((y0, y1))

    xmin = int(np.floor(xmin_nm / sx))
    xmax = int(np.ceil(xmax_nm / sx)) - 1
    ymin = int(np.floor(ymin_nm / sy))
    ymax = int(np.ceil(ymax_nm / sy)) - 1

    h, w = self._tab11_primary_stack.shape[1:3]
    xmin = max(0, min(xmin, w - 1)); xmax = max(0, min(xmax, w - 1))
    ymin = max(0, min(ymin, h - 1)); ymax = max(0, min(ymax, h - 1))
    if xmax <= xmin or ymax <= ymin:
        return

    if getattr(self, '_tab11_preview_rect', None) is not None:
        try:
            self._tab11_preview_rect.remove()
        except Exception:
            pass

    self._tab11_preview_rect = plt.Rectangle(
        (xmin * sx, ymin * sy),
        (xmax - xmin + 1) * sx,
        (ymax - ymin + 1) * sy,
        fill=False, edgecolor='yellow', linewidth=2.0, linestyle='--', zorder=50
    )
    self.tab11_ax_primary.add_patch(self._tab11_preview_rect)
    self.tab11_canvas.draw_idle()


def _tab11_mouse_release(self, event):
    if not getattr(self, '_tab11_select_mode', False):
        return

    self._tab11_select_mode = False
    try:
        self.tab11_canvas.get_tk_widget().configure(cursor='')
    except Exception:
        pass

    c = getattr(self, '_tab11_preview_rect', None)
    if c is not None:
        try:
            c.remove()
        except Exception:
            pass
        self._tab11_preview_rect = None

    press = getattr(self, '_tab11_press_xy', None)
    self._tab11_press_xy = None
    if press is None or event.inaxes is not self.tab11_ax_primary or event.xdata is None or event.ydata is None:
        self.tab11_status_var.set('ROI selection cancelled.')
        self.tab11_canvas.draw_idle()
        return

    sx, sy = self._tab11_scale_nm
    x0_nm, y0_nm = press
    x1_nm, y1_nm = float(event.xdata), float(event.ydata)
    xmin_nm, xmax_nm = sorted((x0_nm, x1_nm))
    ymin_nm, ymax_nm = sorted((y0_nm, y1_nm))

    xmin = int(np.floor(xmin_nm / sx))
    xmax = int(np.ceil(xmax_nm / sx)) - 1
    ymin = int(np.floor(ymin_nm / sy))
    ymax = int(np.ceil(ymax_nm / sy)) - 1

    h, w = self._tab11_primary_stack.shape[1:3]
    xmin = max(0, min(xmin, w - 1)); xmax = max(0, min(xmax, w - 1))
    ymin = max(0, min(ymin, h - 1)); ymax = max(0, min(ymax, h - 1))

    if xmax <= xmin or ymax <= ymin:
        self.tab11_status_var.set('ROI is too small. Draw a larger rectangle.')
        return

    self._tab11_rois.append({
        'xmin': xmin, 'xmax': xmax, 'ymin': ymin, 'ymax': ymax
    })
    self._tab11_center_fixed = False
    self._tab11_auto_frame_fit = False
    self._tab11_dynamic_centers = {}
    self._tab11_centers.pop(len(self._tab11_rois) - 1, None)
    idx = len(self._tab11_rois) - 1
    _tab11_update_roi_selector(self, idx)

    self.tab11_status_var.set(
        f'✓ ROI {idx + 1} added | '
        f'FOV = {(xmax - xmin + 1)*sx:.6g} × {(ymax - ymin + 1)*sy:.6g} nm'
    )
    self._tab11_render( int(self.tab11_frame_var.get()))


def _tab11_start_selection(self):
    if not getattr(self, '_tab11_primary_ready', False):
        if not _tab11_fetch_primary(self, preserve_frame=True):
            return
    self._tab11_select_mode = True
    self._tab11_press_xy = None
    self.tab11_status_var.set('SELECT ROI: drag a rectangle on the processed Primary ROI.')
    try:
        self.tab11_canvas.get_tk_widget().configure(cursor='crosshair')
    except Exception:
        pass


def _tab11_clear_selected(self):
    if not getattr(self, '_tab11_rois', []):
        return
    idx = _tab11_selected_roi_index(self)
    self._tab11_rois.pop(idx)

    old = self._tab11_centers
    self._tab11_centers = {}
    for old_idx, value in old.items():
        if old_idx == idx:
            continue
        self._tab11_centers[old_idx if old_idx < idx else old_idx - 1] = value

    self._tab11_center_fixed = False
    self._tab11_auto_frame_fit = False
    self._tab11_dynamic_centers = {}
    self._tab11_last_fit = None
    _tab11_update_roi_selector(self, min(idx, len(self._tab11_rois) - 1) if self._tab11_rois else None)
    self.tab11_status_var.set('Selected ROI cleared.')
    self._tab11_render( int(self.tab11_frame_var.get()))


def _tab11_clear_all(self):
    """CLEAR ALL: remove every Tab-11 ROI/data point and clear all Tab-11 result graphs/tables."""
    # ------------------------------------------------------------------
    # 1. Clear ALL ROI selections, centers, marks and labels.
    # ------------------------------------------------------------------
    self._tab11_rois = []
    self._tab11_centers = {}
    self._tab11_center_fixed = False
    self._tab11_last_fit = None
    self._tab11_current_tracking_df = None
    self._tab11_tracking_df = None

    self._tab11_select_mode = False
    self._tab11_press_xy = None
    self._tab11_preview_rect = None

    # ------------------------------------------------------------------
    # 2. Clear the actual data sources used by the Tab-11 graphs.
    # ------------------------------------------------------------------
    # Remove any fit/tracking result objects that may contain plotted points.
    for name in (
        '_tab11_results',
        '_tab11_fit_results',
        '_tab11_current_results',
        '_tab11_all_results',
        '_tab11_tracking_results',
    ):
        obj = getattr(self, name, None)
        if isinstance(obj, dict):
            obj.clear()
        elif isinstance(obj, list):
            obj.clear()
        else:
            try:
                setattr(self, name, None)
            except Exception:
                pass

    # ------------------------------------------------------------------
    # 3. Remove twin axes FIRST, then clear every result/graph base axis.
    #    This guarantees that no old FWHM/amplitude/frame data points
    #    survive on hidden Matplotlib twin axes.
    # ------------------------------------------------------------------
    for attr in (
        '_tab11_fit_residual_ax',
        '_tab11_current_amp_ax',
        '_tab11_current_frame_ax',
        '_tab11_all_amp_ax',
        '_tab11_all_frame_ax',
    ):
        ax = getattr(self, attr, None)
        if ax is not None:
            try:
                ax.remove()
            except Exception:
                pass
        try:
            setattr(self, attr, None)
        except Exception:
            pass

    # Figure 3 — fit profile / Gaussian / residual.
    ax = getattr(self, 'tab11_ax_fit', None)
    if ax is not None:
        try:
            ax.clear()
            ax.set_title('FIT CURRENT ROI', pad=9)
            ax.set_xlabel('Position from fixed center (nm)')
            ax.set_ylabel('Intensity')
            ax.grid(alpha=0.2)
        except Exception:
            pass

    # Figure 4 — current ROI tracking.
    ax = getattr(self, 'tab11_ax_current_tracking', None)
    if ax is not None:
        try:
            ax.clear()
            ax.set_title('CURRENT ROI', pad=9)
            ax.set_xlabel('Equivalent field (mT)')
            ax.set_ylabel('FWHM (nm)')
            ax.grid(alpha=0.2)
        except Exception:
            pass

    # Figure 5 — ALL selected ROI tracking.
    ax = getattr(self, 'tab11_ax_all_tracking', None)
    if ax is not None:
        try:
            ax.clear()
            ax.set_title('ALL SELECTED ROIs', pad=9)
            ax.set_xlabel('Equivalent field (mT)')
            ax.set_ylabel('Avg. FWHM (nm)')
            ax.grid(alpha=0.2)
        except Exception:
            pass

    # ------------------------------------------------------------------
    # 4. Clear ANY Treeview table actually belonging to Tab 11.
    #    The current build may not expose the table under one fixed name,
    #    so walk only the Tab-11 widget tree and clear Treeview rows.
    # ------------------------------------------------------------------
    def _clear_tab11_treeviews(widget):
        try:
            if isinstance(widget, ttk.Treeview):
                for item in widget.get_children():
                    widget.delete(item)
        except Exception:
            pass
        try:
            for child in widget.winfo_children():
                _clear_tab11_treeviews(child)
        except Exception:
            pass

    tab11 = getattr(self, 'tab_multi_roi11', None)
    if tab11 is not None:
        _clear_tab11_treeviews(tab11)

    # Also clear explicitly named Tab-11 result tables if present.
    for name in (
        'tab11_results_tree',
        'tab11_table',
        'tab11_results_table',
        '_tab11_results_tree',
        '_tab11_table',
    ):
        tree = getattr(self, name, None)
        if tree is not None:
            try:
                for item in tree.get_children():
                    tree.delete(item)
            except Exception:
                pass

    # ------------------------------------------------------------------
    # 5. Update the ROI selector without rebuilding the Tab-11 UI.
    # ------------------------------------------------------------------
    try:
        _tab11_update_roi_selector(self)
    except Exception:
        pass

    # Render the source image/selected-ROI panels with no ROI overlays.
    # This does NOT recreate the result data points in Figures 3–5.
    try:
        if getattr(self, '_tab11_primary_ready', False):
            _tab11_render(self, int(self.tab11_frame_var.get()))
    except Exception:
        pass

    # ------------------------------------------------------------------
    # 6. Reset status text and redraw the complete figure.
    # ------------------------------------------------------------------
    try:
        self.tab11_fit_status_var.set('No fit/tracking performed. All data cleared.')
    except Exception:
        pass

    try:
        self.tab11_export_status_var.set('')
    except Exception:
        pass

    try:
        self.tab11_status_var.set(
            'CLEAR ALL: all ROI marks/labels, graph data points and table data cleared.'
        )
    except Exception:
        pass

    try:
        self.tab11_canvas.draw_idle()
    except Exception:
        pass

def _tab11_zoom(self, event):
    if not getattr(self, '_tab11_primary_ready', False):
        return
    if event.inaxes is not self.tab11_ax_primary or event.xdata is None or event.ydata is None:
        return
    ext = self._tab11_primary_extent
    if ext is None:
        return

    step = float(getattr(event, 'step', 0) or 0)
    if step == 0:
        step = 1.0 if getattr(event, 'num', None) == 4 else -1.0

    factor = 1.25 ** (-step)
    x0, x1 = self.tab11_ax_primary.get_xlim()
    y0, y1 = self.tab11_ax_primary.get_ylim()
    cx, cy = float(event.xdata), float(event.ydata)

    rx = (cx - x0) / max(x1 - x0, 1e-12)
    ry = (cy - y0) / max(y1 - y0, 1e-12)
    nx0 = cx - rx * (x1 - x0) * factor
    nx1 = cx + (1 - rx) * (x1 - x0) * factor
    ny0 = cy - ry * (y1 - y0) * factor
    ny1 = cy + (1 - ry) * (y1 - y0) * factor

    fx0, fx1, fy0, fy1 = ext
    nx0 = max(fx0, min(nx0, fx1)); nx1 = min(fx1, max(nx1, fx0))
    ny0 = max(fy0, min(ny0, fy1)); ny1 = min(fy1, max(ny1, fy0))
    if nx1 <= nx0 or ny1 <= ny0:
        return

    self.tab11_ax_primary.set_xlim(nx0, nx1)
    self.tab11_ax_primary.set_ylim(ny0, ny1)
    self.tab11_ax_primary.set_aspect('equal', adjustable='box')
    self.tab11_canvas.draw_idle()


def _tab11_reset_zoom(self):
    if not getattr(self, '_tab11_primary_ready', False):
        return
    ext = self._tab11_primary_extent
    self.tab11_ax_primary.set_xlim(ext[0], ext[1])
    self.tab11_ax_primary.set_ylim(ext[2], ext[3])
    self.tab11_ax_primary.set_aspect('equal', adjustable='box')
    self.tab11_canvas.draw_idle()


def _tab11_center(self, crop):
    """Strongest signed contrast center, same mathematics as Tab 10."""
    image = np.asarray(crop, dtype=float)
    finite = np.isfinite(image)
    if not np.any(finite):
        raise ValueError('No finite pixels in selected ROI.')
    z = image.copy()
    z[~finite] = np.nanmedian(z[finite])
    h, w = z.shape

    b = max(1, int(round(0.10 * min(h, w))))
    border = np.concatenate([
        z[:b, :].ravel(), z[-b:, :].ravel(),
        z[:, :b].ravel(), z[:, -b:].ravel()
    ])
    background = float(np.nanmedian(border))
    contrast = z - background
    max_pos = float(np.nanmax(contrast))
    max_neg = float(np.nanmin(contrast))

    if abs(max_pos) >= abs(max_neg):
        signed, sign, peak = contrast, 1.0, max_pos
    else:
        signed, sign, peak = -contrast, -1.0, -max_neg

    peak = max(peak, 1e-12)
    try:
        frac = float(np.clip(self.tab11_contrast_fraction_var.get(), 0.0, 0.95))
    except Exception:
        frac = 0.50

    threshold = frac * peak
    weights = np.clip(signed - threshold, 0.0, None)
    if np.sum(weights) <= 0:
        weights = np.clip(signed, 0.0, None)

    yy, xx = np.indices(z.shape, dtype=float)
    ws = float(np.sum(weights))
    if ws <= 0:
        y0, x0 = np.unravel_index(np.argmax(np.abs(contrast)), z.shape)
        x0, y0 = float(x0), float(y0)
    else:
        x0 = float(np.sum(xx * weights) / ws)
        y0 = float(np.sum(yy * weights) / ws)

    return {
        'x0': x0, 'y0': y0,
        'background': background,
        'sign': sign, 'peak': peak,
        'threshold': threshold
    }


def _tab11_roi_crop(self, roi_idx, frame_idx):
    rois = getattr(self, '_tab11_rois', [])
    if not rois or not getattr(self, '_tab11_primary_ready', False):
        return None, None
    roi_idx = int(roi_idx)
    if roi_idx < 0 or roi_idx >= len(rois):
        return None, None

    c = rois[roi_idx]
    image = np.asarray(self._tab11_primary_stack[int(frame_idx)], dtype=float)
    crop = image[c['ymin']:c['ymax'] + 1, c['xmin']:c['xmax'] + 1].copy()
    return crop, c


def _tab11_get_angles(self):
    return {
        '1': [0.0],
        '2': [0.0, 90.0],
        '4': [0.0, 45.0, 90.0, 135.0],
        '6': [0.0, 30.0, 60.0, 90.0, 120.0, 150.0]
    }.get(str(self.tab11_ref_mode_var.get()), [0.0, 90.0])


def _tab11_fit_roi(self, roi_idx, frame_idx, store=True, dynamic_center=False):
    crop, bounds = _tab11_roi_crop(self, roi_idx, frame_idx)
    if crop is None:
        return None

    # Manual fitting/tracking retains the existing fixed-center behavior.
    # Automatic frame-slider analysis requests a fresh core detection for
    # the current frame while keeping the ROI rectangle fixed.
    if dynamic_center:
        center = _tab11_center(self, crop)
        self._tab11_dynamic_centers[int(roi_idx)] = center
    elif getattr(self, '_tab11_center_fixed', False) and roi_idx in self._tab11_centers:
        center = self._tab11_centers[roi_idx]
    else:
        center = _tab11_center(self, crop)
        self._tab11_centers[roi_idx] = center

    angles = _tab11_get_angles(self)
    t_px, profiles, avg = self._extract_multi_line_profiles(
        crop, float(center['x0']), float(center['y0']),
        angles, nsamples=800
    )

    sx, sy = self._tab11_scale_nm
    scale_line = float(np.sqrt(sx * sy)) if sx > 0 and sy > 0 else 1.0
    t_nm = np.asarray(t_px, dtype=float) * scale_line

    # Existing fixed-center Gaussian function is retained.
    fit = self._fit_fixed_center_1d_gaussian(t_nm, avg)

    stack = np.vstack([profiles[float(a)] for a in angles])
    per_line_rmse = np.sqrt(np.nanmean((stack - avg[None, :]) ** 2, axis=1))
    line_rmse = float(np.nanmean(per_line_rmse))
    line_cv = float(
        line_rmse / max(float(np.nanmax(np.abs(avg))), 1e-12)
    )

    result = {
        'roi_index': int(roi_idx + 1),
        'frame': int(frame_idx),
        'field_mT': _tab11_field_mT(self, int(frame_idx)),
        'center_x_px': float(bounds['xmin'] + center['x0']),
        'center_y_px': float(bounds['ymin'] + center['y0']),
        'local_center_x_px': float(center['x0']),
        'local_center_y_px': float(center['y0']),
        'contrast_background': float(center['background']),
        'contrast_threshold': float(center['threshold']),
        'contrast_sign': float(center['sign']),
        'contrast_peak': float(center['peak']),
        'angles': list(angles),
        't_px': np.asarray(t_px, dtype=float),
        't_nm': t_nm,
        'profiles': profiles,
        'average_profile': np.asarray(avg, dtype=float),
        'fit_profile': np.asarray(fit['fit_profile'], dtype=float),
        'residual_profile': np.asarray(fit['residual_profile'], dtype=float),
        'background': float(fit['background']),
        'amplitude': float(fit['amplitude']),
        'amplitude_err': float(fit.get('amplitude_err', np.nan)),
        'sigma': float(fit['sigma']),
        'sigma_err': float(fit.get('sigma_err', np.nan)),
        'fwhm': float(fit['fwhm']),
        'fwhm_err': float(fit.get('fwhm_err', np.nan)),
        'r2': float(fit.get('r2', np.nan)),
        'rmse': float(fit.get('rmse', np.nan)),
        'line_rmse': line_rmse,
        'line_cv': line_cv,
        'per_line_rmse': per_line_rmse,
        'fov_width_nm': float((bounds['xmax'] - bounds['xmin'] + 1) * sx),
        'fov_height_nm': float((bounds['ymax'] - bounds['ymin'] + 1) * sy),
    }
    if store:
        self._tab11_last_fit = result
    return result



def _tab11_auto_analyze_current_frame(self, frame_idx=None, draw=True):
    """
    After FIX CENTERS — ALL ROIs, automatically process the current slider
    frame. ROI bounds remain fixed; the core is freshly detected in each ROI
    for the current frame, followed by profile extraction, Gaussian fitting,
    FWHM and intensity (Gaussian amplitude).
    """
    if not getattr(self, '_tab11_primary_ready', False):
        return []
    rois = getattr(self, '_tab11_rois', [])
    if not rois:
        return []
    stack = getattr(self, '_tab11_primary_stack', None)
    if stack is None or len(stack) == 0:
        return []

    n = len(stack)
    if frame_idx is None:
        frame_idx = int(self.tab11_frame_var.get())
    frame_idx = max(0, min(int(frame_idx), n - 1))

    if not isinstance(getattr(self, '_tab11_dynamic_centers', None), dict):
        self._tab11_dynamic_centers = {}

    results=[]
    errors=[]
    for ridx in range(len(rois)):
        try:
            r=_tab11_fit_roi(self,ridx,frame_idx,store=False,dynamic_center=True)
            if r is not None:
                results.append(r)
        except Exception as exc:
            errors.append(f"ROI {ridx+1}: {type(exc).__name__}: {exc}")

    if not results:
        self.tab11_fit_status_var.set(
            f'Automatic analysis failed at frame {frame_idx+1}.')
        return []

    selected_idx=_tab11_selected_roi_index(self)
    selected=next(
        (r for r in results if int(r['roi_index'])==selected_idx+1),
        results[0])

    self._tab11_last_fit=selected

    current_row={
        'roi_index':selected['roi_index'],'frame':selected['frame'],
        'field_mT':selected['field_mT'],'fwhm':selected['fwhm'],
        'fwhm_err':selected['fwhm_err'],'amplitude':selected['amplitude'],
        'amplitude_err':selected['amplitude_err']}
    old=getattr(self,'_tab11_current_tracking_df',None)
    rows=old.to_dict('records') if isinstance(old,pd.DataFrame) else []
    rows=[x for x in rows if not (
        int(x.get('roi_index',-999))==int(selected['roi_index']) and
        int(x.get('frame',-999))==int(selected['frame']))]
    rows.append(current_row)
    self._tab11_current_tracking_df=pd.DataFrame(rows).sort_values(
        ['frame','roi_index']).reset_index(drop=True)

    # Figure 5 tracking store: replace only same ROI/frame, retain other
    # frames visited through the slider.
    old=getattr(self,'_tab11_tracking_df',None)
    rows=old.to_dict('records') if isinstance(old,pd.DataFrame) else []
    keys={(int(r['roi_index']),int(r['frame'])) for r in results}
    rows=[x for x in rows if (int(x.get('roi_index',-999)),int(x.get('frame',-999))) not in keys]
    for r in results:
        rows.append({k:r[k] for k in (
            'roi_index','frame','field_mT','fwhm','fwhm_err',
            'amplitude','amplitude_err','sigma','sigma_err',
            'r2','rmse','line_rmse','line_cv')})
    self._tab11_tracking_df=pd.DataFrame(rows).sort_values(
        ['frame','roi_index']).reset_index(drop=True)

    # Display the selected ROI exactly as a current-frame fit.
    _tab11_render(self,frame_idx)
    _tab11_draw_current_fit_views(self,selected)
    _tab11_plot_tracking_panel(
        self,self._tab11_tracking_df,self.tab11_ax_all_tracking,
        mode='all',title='ALL SELECTED ROIs',
        draw=False)
    _tab11_update_params(
        self,result=selected,
        status='✓ Automatic core detection + Gaussian fit complete.')
    self.tab11_fit_status_var.set(
        f'✓ Frame {frame_idx+1}/{n} | {len(results)} ROIs | '
        f'ROI {selected["roi_index"]}: FWHM = {selected["fwhm"]:.6g} nm | '
        f'Intensity = {selected["amplitude"]:.6g}')
    if errors:
        self.tab11_status_var.set(
            f'Auto analysis: {len(results)}/{len(rois)} ROIs fitted at '
            f'frame {frame_idx+1}; ' + '; '.join(errors[:2]))
    else:
        self.tab11_status_var.set(
            f'✓ Frame {frame_idx+1}: automatically detected cores and '
            f'calculated FWHM + intensity for all {len(results)} ROIs.')
    if draw:
        self.tab11_canvas.draw_idle()
    return results

def _tab11_update_params(self, result=None, status=None):
    vars_ = getattr(self, 'tab11_param_vars', {})
    for key, var in vars_.items():
        var.set('—')

    if result is not None:
        vals = {
            'roi': f"ROI {result['roi_index']}",
            'center': f"({result['local_center_x_px']:.3f}, {result['local_center_y_px']:.3f})",
            'background': f"{result['background']:.6g}",
            'threshold': f"{result['contrast_threshold']:.6g}",
            'sign': f"{result['contrast_sign']:.0f}",
            'amplitude': f"{result['amplitude']:.6g} ± {result['amplitude_err']:.3g}",
            'sigma': f"{result['sigma']:.6g} ± {result['sigma_err']:.3g} nm",
            'fwhm': f"{result['fwhm']:.6g} ± {result['fwhm_err']:.3g} nm",
            'r2': f"{result['r2']:.6g}",
            'rmse': f"{result['rmse']:.6g}",
            'line_rmse': f"{result['line_rmse']:.6g}",
            'line_cv': f"{result['line_cv']:.6g}",
            'fov': f"{result['fov_width_nm']:.6g} × {result['fov_height_nm']:.6g} nm",
            'frame': f"{result['frame'] + 1} / {len(self._tab11_primary_stack)}",
            'field': (
                f"{result['field_mT']:+.6g} mT"
                if np.isfinite(result['field_mT']) else '—'
            )
        }
        for key, value in vals.items():
            if key in vars_:
                vars_[key].set(value)

    if 'status' in vars_:
        vars_['status'].set(
            status if status is not None
            else ('Gaussian fit complete.' if result is not None else 'No fit performed.')
        )


def _tab11_clear_twin_axes(self, base_ax_name):
    base = getattr(self, base_ax_name, None)
    if base is None:
        return
    tracked_names = {
        '_tab11_current_amp_ax': 'current',
        '_tab11_current_frame_ax': 'current',
        '_tab11_all_amp_ax': 'all',
        '_tab11_all_frame_ax': 'all',
        '_tab11_fit_residual_ax': 'fit',
    }
    for attr in tracked_names:
        ax = getattr(self, attr, None)
        if ax is not None:
            try:
                ax.remove()
            except Exception:
                pass
            setattr(self, attr, None)


def _tab11_plot_tracking_panel(self, df, ax_base, mode='current', title='', draw=True):
    """
    mode='current': one selected ROI.
    mode='all': show EVERY selected ROI plus an overall mean curve.
    """
    amp_attr = '_tab11_current_amp_ax' if mode == 'current' else '_tab11_all_amp_ax'
    frame_attr = '_tab11_current_frame_ax' if mode == 'current' else '_tab11_all_frame_ax'

    for attr in (amp_attr, frame_attr):
        old_ax = getattr(self, attr, None)
        if old_ax is not None:
            try:
                old_ax.remove()
            except Exception:
                pass
            setattr(self, attr, None)

    ax_base.clear()

    if df is None or df.empty:
        ax_base.set_title(title, pad=10)
        ax_base.set_xlabel('Equivalent field (mT)')
        ax_base.set_ylabel('FWHM (nm)')
        ax_base.grid(alpha=0.2)
        if draw:
            self.tab11_canvas.draw_idle()
        return

    data = df.copy()
    for col in ('field_mT', 'frame', 'roi_index', 'fwhm', 'fwhm_err', 'amplitude', 'amplitude_err'):
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors='coerce')
    data = data.dropna(subset=['field_mT', 'frame'])
    if data.empty:
        ax_base.set_title(title, pad=10)
        ax_base.set_xlabel('Equivalent field (mT)')
        ax_base.set_ylabel('FWHM (nm)')
        ax_base.grid(alpha=0.2)
        if draw:
            self.tab11_canvas.draw_idle()
        return

    # For current ROI, plot its own tracking curve.
    # For all ROIs, keep each ROI separate and add a mean curve.
    plot_groups = []
    if mode == 'current':
        ridx = _tab11_selected_roi_index(self)
        plot_groups = [(ridx + 1, data[data['roi_index'] == ridx + 1].sort_values('frame'))]
    else:
        for roi_id in sorted(data['roi_index'].dropna().astype(int).unique()):
            g = data[data['roi_index'].astype(int) == int(roi_id)].sort_values('frame')
            if not g.empty:
                plot_groups.append((int(roi_id), g))

    # Use one FWHM axis and one amplitude axis.
    ax_amp = ax_base.twinx()
    setattr(self, amp_attr, ax_amp)

    # Each ROI has a dedicated legend entry for FWHM and amplitude.
    line_count = len(plot_groups)
    for pos, (roi_id, g) in enumerate(plot_groups):
        xf = g['field_mT'].to_numpy(float)
        fwhm = g['fwhm'].to_numpy(float)
        amp = g['amplitude'].to_numpy(float)
        fwhm_err = pd.to_numeric(
            g.get('fwhm_err', pd.Series(np.zeros(len(g)))),
            errors='coerce'
        ).fillna(0).to_numpy(float)
        amp_err = pd.to_numeric(
            g.get('amplitude_err', pd.Series(np.zeros(len(g)))),
            errors='coerce'
        ).fillna(0).to_numpy(float)

        ax_base.errorbar(
            xf, fwhm, yerr=fwhm_err,
            marker='^', linestyle='-',linewidth=1.5, capsize=2,
            label=f'ROI {roi_id} — FWHM'
        )
        ax_amp.errorbar(
            xf, amp, yerr=amp_err,
            marker='o', linestyle=':', linewidth=1.5,
            capsize=2, label=f'ROI {roi_id} — Amplitude'
        )

    if mode == 'all' and len(plot_groups) > 1:
        # Add overall mean with separate, visually dominant line styles.
        agg = data.groupby(
            ['frame', 'field_mT'], as_index=False
        ).agg(
            fwhm=('fwhm', 'mean'),
            fwhm_err=('fwhm_err', 'mean'),
            amplitude=('amplitude', 'mean'),
            amplitude_err=('amplitude_err', 'mean')
        ).sort_values('frame')

        xf = agg['field_mT'].to_numpy(float)
        fwhm = agg['fwhm'].to_numpy(float)
        amp = agg['amplitude'].to_numpy(float)
        fwhm_err = pd.to_numeric(
            agg['fwhm_err'], errors='coerce'
        ).fillna(0).to_numpy(float)
        amp_err = pd.to_numeric(
            agg['amplitude_err'], errors='coerce'
        ).fillna(0).to_numpy(float)

        ax_base.errorbar(
            xf, fwhm, yerr=fwhm_err,
            marker='^', linestyle='-',linewidth=0.7, capsize=1.0,color='black',
            label='AVERAGE — FWHM'
        )
        ax_amp.errorbar(
            xf, amp, yerr=amp_err,
            marker='*', linestyle='--', linewidth=0.7,color='black',
            capsize=1.0, label='AVERAGE — Amplitude'
        )

    # Secondary X2: frame number, aligned to field positions.
    secx = ax_base.twiny()
    setattr(self, frame_attr, secx)
    secx.set_xlim(ax_base.get_xlim())

    # Use all unique frame-field pairs available in the plotted data.
    frame_field = data[['frame', 'field_mT']].drop_duplicates().sort_values('frame')
    max_ticks = min(8, len(frame_field))
    if max_ticks:
        sel = np.linspace(0, len(frame_field) - 1, max_ticks).round().astype(int)
        x_ticks = frame_field.iloc[sel]['field_mT'].to_numpy(float)
        f_ticks = frame_field.iloc[sel]['frame'].to_numpy(float) + 1
        secx.set_xticks(x_ticks)
        secx.set_xticklabels([str(int(v)) for v in f_ticks])

    ax_base.set_xlabel('Equivalent field (mT)', labelpad=8)
    ax_base.set_ylabel('FWHM (nm)', labelpad=8)
    ax_base.set_title(title, pad=12)
    ax_base.grid(alpha=0.2)

    ax_amp.set_ylabel('Gaussian amplitude', labelpad=10)
    ax_amp.yaxis.set_label_position('right')
    ax_amp.yaxis.tick_right()

    secx.set_xlabel('Frame number', labelpad=8)
    secx.xaxis.set_label_position('top')
    secx.xaxis.tick_top()

    # Combined legend outside the data region so it does not cover curves.
    h1, l1 = ax_base.get_legend_handles_labels()
    h2, l2 = ax_amp.get_legend_handles_labels()
    ax_base.legend(
        h1 + h2, l1 + l2,
        fontsize=7.5,
        loc='upper left',
        bbox_to_anchor=(0.01, 0.99),
        frameon=True
    )

    if draw:
        self.tab11_fig.subplots_adjust(
            left=0.07, right=0.94, bottom=0.07, top=0.94,
            wspace=0.32, hspace=0.48
        )
        self.tab11_canvas.draw_idle()



def _tab11_clear_current_fit_views(self):
    """Clear Fig.3/Fig.4 only; Fig.5/all-ROI tracking is independent."""
    # Remove residual twin axis from current fit.
    axr = getattr(self, '_tab11_fit_residual_ax', None)
    if axr is not None:
        try:
            axr.remove()
        except Exception:
            pass
        self._tab11_fit_residual_ax = None

    # Remove current ROI Figure-4 twin axes.
    for attr in ('_tab11_current_amp_ax', '_tab11_current_frame_ax'):
        ax = getattr(self, attr, None)
        if ax is not None:
            try:
                ax.remove()
            except Exception:
                pass
            setattr(self, attr, None)

    # Clear Fig.3.
    ax = getattr(self, 'tab11_ax_fit', None)
    if ax is not None:
        ax.clear()
        ax.set_title(
            'Average Line Profile — Gaussian Fit',
            pad=9
        )
        ax.set_xlabel('Position From Fixed Center (nm)')
        ax.set_ylabel('Intensity')
        ax.grid(alpha=0.2)

    # Clear Fig.4/current-ROI tracking.
    ax = getattr(self, 'tab11_ax_current_tracking', None)
    if ax is not None:
        ax.clear()
        ax.set_title('CURRENT ROI', pad=9)
        ax.set_xlabel('Equivalent Field (mT)')
        ax.set_ylabel('FWHM (nm)')
        ax.grid(alpha=0.2)

    self._tab11_last_fit = None
    self._tab11_current_tracking_df = None



def _tab11_export_multi_fit_info(self):
    """Export line profiles + average + Gaussian fit for ALL selected ROIs."""
    try:
        if not getattr(self, '_tab11_primary_ready', False):
            if not _tab11_fetch_primary(self, preserve_frame=True):
                return

        rois = getattr(self, '_tab11_rois', [])
        if not rois:
            messagebox.showwarning(
                'Tab 11 Export',
                'Add at least one ROI first.',
                parent=self.root
            )
            return

        frame = int(self.tab11_frame_var.get())
        results = []
        for ridx in range(len(rois)):
            result = _tab11_fit_roi(self, ridx, frame, store=False)
            if result is not None:
                results.append(result)

        if not results:
            raise RuntimeError('No selected ROI fitting results were generated.')

        path = filedialog.asksaveasfilename(
            parent=self.root,
            title='Export ALL ROI Profiles + Average + Gaussian Fit',
            defaultextension='.csv',
            filetypes=[('CSV', '*.csv'), ('All files', '*.*')],
            initialfile=(
                f'Tab11_ALL_ROIs_Frame{frame + 1:03d}_'
                f'LineProfiles_Avg_Gaussian.csv'
            )
        )
        if not path:
            return

        rows = []
        for r in results:
            t = np.asarray(r['t_nm'], dtype=float)
            avg = np.asarray(r['average_profile'], dtype=float)
            gf = np.asarray(r['fit_profile'], dtype=float)
            n = len(t)

            profiles = {
                float(a): np.asarray(r['profiles'][float(a)], dtype=float)
                for a in r['angles']
            }

            for j in range(n):
                row = {
                    'ROI': int(r['roi_index']),
                    'Frame': int(r['frame']) + 1,
                    'Field_mT': r['field_mT'],
                    'Position_nm': t[j],
                    'Average_Profile': avg[j],
                    'Gaussian_Fit': gf[j],
                }
                for angle, arr in profiles.items():
                    row[f'Profile_{angle:g}deg'] = (
                        arr[j] if j < len(arr) else np.nan
                    )
                rows.append(row)

        pd.DataFrame(rows).to_csv(path, index=False)

        self.tab11_export_status_var.set(
            f'✓ Exported all {len(results)} ROIs: {Path(path).name}'
        )

    except Exception as exc:
        self.tab11_export_status_var.set(
            f'✗ ALL ROI export failed: {type(exc).__name__}: {exc}'
        )
        messagebox.showerror(
            'Tab 11 Export — ALL ROI',
            f'{type(exc).__name__}: {exc}',
            parent=self.root
        )


def _tab11_render(self, idx=None):
    if not getattr(self, '_tab11_primary_ready', False):
        return

    n = len(self._tab11_primary_stack)
    idx = int(self.tab11_frame_var.get()) if idx is None else int(idx)
    idx = max(0, min(idx, n - 1))
    self.tab11_frame_var.set(idx)

    sx, sy = self._tab11_scale_nm
    image = np.asarray(self._tab11_primary_stack[idx], dtype=float)
    h, w = image.shape[:2]
    ext = (0.0, w*sx, 0.0, h*sy)
    self._tab11_primary_extent = ext

    # ---------- Row 1 / Col 1: processed Primary ROI1 ----------
    self.tab11_ax_primary.clear()
    # Pull the live Tab-6 colormap each time the frame is rendered.
    try:
        self._tab11_cmap = str(self.colormap_var.get() or self._tab11_cmap)
    except Exception:
        pass

    im = self.tab11_ax_primary.imshow(
        image, cmap=self._tab11_cmap, origin='lower',
        extent=ext, vmin=0.0, vmax=1.0,
        interpolation='nearest', aspect='equal'
    )
    self.tab11_ax_primary.set_title(
        f'{getattr(self, "_tab11_view", "")}| PROCESSED ROI | '
        f'Frame {idx + 1}/{n} | Field = {_tab11_field_mT(self, idx):+.2f} mT',
        pad=9
    )
    self.tab11_ax_primary.set_xlabel('X (nm)')
    self.tab11_ax_primary.set_ylabel('Y (nm)')
    self.tab11_ax_primary.set_xlim(ext[0], ext[1])
    self.tab11_ax_primary.set_ylim(ext[2], ext[3])
    self.tab11_ax_primary.set_aspect('equal', adjustable='box')
    self.tab11_ax_primary.grid(alpha=0.10)

    try:
        if getattr(self, '_tab11_cbar', None) is None or self._tab11_cbar.ax not in self.tab11_fig.axes:
            self._tab11_cbar = self.tab11_fig.colorbar(
                im, ax=self.tab11_ax_primary, fraction=0.046, pad=0.04
            )
        else:
            self._tab11_cbar.update_normal(im)
        self._tab11_cbar.set_label('Intensity')
    except Exception:
        pass

    palette = ['red', 'lime', 'cyan', 'magenta', 'orange', 'yellow', 'white', 'purple']
    for ridx, c in enumerate(getattr(self, '_tab11_rois', [])):
        col = palette[ridx % len(palette)]
        self.tab11_ax_primary.add_patch(plt.Rectangle(
            (c['xmin']*sx, c['ymin']*sy),
            (c['xmax'] - c['xmin'] + 1)*sx,
            (c['ymax'] - c['ymin'] + 1)*sy,
            fill=False, edgecolor=col, linewidth=2.0, zorder=20
        ))
        self.tab11_ax_primary.text(
            c['xmin']*sx + 4*sx,
            c['ymax']*sy - 6*sy,
            f'ROI {ridx + 1}',
            color=col, fontsize=9, weight='bold',
            bbox=dict(facecolor='black', alpha=0.26, edgecolor='none', pad=1.4),
            zorder=21
        )

    # ---------- Row 1 / Col 2: selected ROI ----------
    self.tab11_ax_selected.clear()
    if getattr(self, '_tab11_rois', []):
        ridx = _tab11_selected_roi_index(self)
        crop, bounds = _tab11_roi_crop(self, ridx, idx)
    else:
        ridx, crop, bounds = 0, None, None

    if crop is None:
        self.tab11_ax_selected.imshow(
            np.zeros((10, 10)), cmap=self._tab11_cmap, origin='lower',
            extent=(0, 10, 0, 10), vmin=self._tab11_clim[0], vmax=self._tab11_clim[1],
            interpolation='nearest', aspect='equal'
        )
        self.tab11_ax_selected.set_title('SELECTED ROI — none', pad=9)
        self.tab11_ax_selected.set_xlabel('X (nm)')
        self.tab11_ax_selected.set_ylabel('Y (nm)')
    else:
        roi_h, roi_w = crop.shape
        self.tab11_ax_selected.imshow(
            crop, cmap=self._tab11_cmap, origin='lower',
            extent=(0, roi_w*sx, 0, roi_h*sy),
            vmin=0.0, vmax=1.0,
            interpolation='nearest', aspect='equal'
        )
        self.tab11_ax_selected.set_title(
            f'Selected ROI—{ridx + 1}  | Frame {idx + 1}/{n}',
            pad=9
        )
        self.tab11_ax_selected.set_xlabel('X (nm)')
        self.tab11_ax_selected.set_ylabel('Y (nm)')
        self.tab11_ax_selected.set_xlim(0, roi_w*sx)
        self.tab11_ax_selected.set_ylim(0, roi_h*sy)
        self.tab11_ax_selected.set_aspect('equal', adjustable='box')

        center = (
            self._tab11_dynamic_centers.get(ridx)
            if getattr(self, '_tab11_auto_frame_fit', False)
            else self._tab11_centers.get(ridx)
        )
        if center is not None:
            self.tab11_ax_selected.plot(
                center['x0']*sx, center['y0']*sy,
                marker='+', markersize=14, markeredgewidth=2.4,
                color='white', linestyle='None', label='Fixed center'
            )
            angles = _tab11_get_angles(self)
            rmax = 2.0 * max(1.0, min(
                center['x0'], roi_w - 1 - center['x0'],
                center['y0'], roi_h - 1 - center['y0']
            ))
            tline = np.array([-rmax, rmax], dtype=float)
            for ai, angle in enumerate(angles):
                th = np.deg2rad(float(angle))
                self.tab11_ax_selected.plot(
                    (center['x0'] + tline*np.cos(th))*sx,
                    (center['y0'] + tline*np.sin(th))*sy,
                    linewidth=1.5, color=palette[ai % len(palette)],
                    label=f'{angle:g}°'
                )
            self.tab11_ax_selected.legend(fontsize=7, loc='best')

    # ---------- Row 2 / Col 1: fit detail ----------
    if getattr(self, '_tab11_last_fit', None) is None:
        self.tab11_ax_fit.clear()
        self.tab11_ax_fit.set_title('FIT CURRENT ROI | Average Line Profile - Gaussian Fit', pad=9)
        self.tab11_ax_fit.set_xlabel('Position from fixed center (nm)')
        self.tab11_ax_fit.set_ylabel('Intensity')
        self.tab11_ax_fit.grid(alpha=0.2)

    try:
        self.tab11_frame_label_var.set(
            f'Frame {idx + 1}/{n} | Field = {_tab11_field_mT(self, idx):+.2f} mT'
        )
    except Exception:
        pass

    self.tab11_fig.subplots_adjust(
        left=0.065, right=0.90, bottom=0.06, top=0.95,
        wspace=0.30, hspace=0.52
    )
    self.tab11_canvas.draw_idle()



def _tab11_draw_current_fit_views(self, result):
    """Draw Figure 3 and Figure 4 for the current selected ROI only."""
    # Remove stale twin axes.
    for attr in (
        '_tab11_fit_residual_ax',
        '_tab11_current_amp_ax',
        '_tab11_current_frame_ax',
    ):
        ax_old = getattr(self, attr, None)
        if ax_old is not None:
            try:
                ax_old.remove()
            except Exception:
                pass
            setattr(self, attr, None)

    # ---------------- Figure 3: average profile + Gaussian + residual ----------------
    ax = self.tab11_ax_fit
    ax.clear()

    t = __import__('numpy').asarray(result['t_nm'], dtype=float)
    avg = __import__('numpy').asarray(result['average_profile'], dtype=float)
    fit = __import__('numpy').asarray(result['fit_profile'], dtype=float)
    residual = __import__('numpy').asarray(result['residual_profile'], dtype=float)

    ax.plot(t, avg, color='black', linewidth=2.0,
            label='Average full profile', zorder=3)
    ax.plot(t, fit, linestyle='--', linewidth=2.0,color='blue',
            label='Fixed-center Gaussian fit', zorder=2)
    ax.axvline(0.0, linestyle=':', linewidth=1.0,
                label='Fixed center')

    axr = ax.twinx()
    self._tab11_fit_residual_ax = axr
    axr.plot(t, residual, linestyle=':', linewidth=1.5,
             label='Residual')
    axr.axhline(0.0, linestyle='--', linewidth=0.8)
    axr.set_ylabel('Residual')
    axr.yaxis.set_label_position('right')
    axr.yaxis.tick_right()

    ax.set_title(
        f'ROI—{result["roi_index"]} Average Line Profile — Gaussian Fit',
        pad=10
    )
    ax.set_xlabel('Position from fixed center (nm)', labelpad=8)
    ax.set_ylabel('Intensity', labelpad=8)
    ax.grid(alpha=0.20)

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = axr.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=8, loc='best', frameon=True)

    # ---------------- Figure 4: current ROI FWHM + amplitude ----------------
    current_df = getattr(self, '_tab11_current_tracking_df', None)
    _tab11_plot_tracking_panel(
        self,
        current_df,
        self.tab11_ax_current_tracking,
        mode='current',
        title=(
            f'ROI—{result["roi_index"]}'
            ' FWHM/Amplitude vs Field'
        ),
        draw=False
    )

    self.tab11_fig.subplots_adjust(
        left=0.065, right=0.90, bottom=0.06, top=0.95,
        wspace=0.30, hspace=0.52
    )
    self.tab11_canvas.draw_idle()


def _tab11_fit_current(self):
    if not getattr(self, '_tab11_primary_ready', False):
        if not _tab11_fetch_primary(self, preserve_frame=True):
            return
    if not getattr(self, '_tab11_rois', []):
        messagebox.showwarning(
            'Tab 11',
            'Add at least one ROI first.',
            parent=self.root
        )
        return

    ridx = _tab11_selected_roi_index(self)
    frame = int(self.tab11_frame_var.get())

    try:
        result = _tab11_fit_roi(self, ridx, frame, store=True)
        if result is None:
            raise RuntimeError('No fitting result returned for the selected ROI.')

        self._tab11_last_fit = result
        self._tab11_current_tracking_df = pd.DataFrame([{
            'roi_index': result['roi_index'],
            'frame': result['frame'],
            'field_mT': result['field_mT'],
            'fwhm': result['fwhm'],
            'fwhm_err': result['fwhm_err'],
            'amplitude': result['amplitude'],
            'amplitude_err': result['amplitude_err']
        }])

        # Render source/selected ROI without touching Fig.3/Fig.4.
        self._tab11_render( frame)

        # Deterministically redraw only Fig.3 and Fig.4.
        _tab11_draw_current_fit_views(self, result)

        _tab11_update_params(
            self,
            result=result,
            status='✓ Gaussian fit complete.'
        )
        self.tab11_fit_status_var.set(
            f'✓ ROI {result["roi_index"]} fit complete | '
            f'FWHM = {result["fwhm"]:.6g} nm | '
            f'Amplitude = {result["amplitude"]:.6g}'
        )

    except Exception as exc:
        self._tab11_last_fit = None
        self._tab11_current_tracking_df = None
        _tab11_clear_current_fit_views(self)
        self.tab11_fit_status_var.set(
            f'✗ FIT CURRENT ROI failed: {type(exc).__name__}: {exc}'
        )
        messagebox.showerror(
            'Tab 11 — FIT CURRENT ROI',
            f'{type(exc).__name__}: {exc}',
            parent=self.root
        )



def _tab11_recalculate_center_selected(self):
    if not getattr(self, '_tab11_primary_ready', False):
        if not _tab11_fetch_primary(self, preserve_frame=True):
            return
    if not getattr(self, '_tab11_rois', []):
        messagebox.showwarning('Tab 11', 'Add at least one ROI first.', parent=self.root)
        return

    ridx = _tab11_selected_roi_index(self)
    frame = int(self.tab11_frame_var.get())
    crop, _ = _tab11_roi_crop(self, ridx, frame)
    if crop is None:
        return

    center = _tab11_center(self, crop)
    self._tab11_centers[ridx] = center
    self._tab11_center_fixed = False
    self._tab11_last_fit = None

    self.tab11_status_var.set(
        f'✓ ROI {ridx + 1} center recalculated | '
        f'center = ({center["x0"]:.3f}, {center["y0"]:.3f})'
    )
    _tab11_update_params(self, status='Center recalculated. Fit the selected ROI.')
    self._tab11_render( frame)


def _tab11_fix_all_centers(self):
    if not getattr(self, '_tab11_primary_ready', False):
        if not _tab11_fetch_primary(self, preserve_frame=True):
            return
    if not getattr(self, '_tab11_rois', []):
        messagebox.showwarning('Tab 11', 'Add at least one ROI first.', parent=self.root)
        return

    frame=int(self.tab11_frame_var.get())
    centers={}
    for ridx in range(len(self._tab11_rois)):
        crop,_=_tab11_roi_crop(self,ridx,frame)
        if crop is not None:
            centers[ridx]=_tab11_center(self,crop)

    self._tab11_centers=centers
    self._tab11_dynamic_centers=dict(centers)
    self._tab11_center_fixed=True
    self._tab11_auto_frame_fit=True

    self.tab11_status_var.set(
        f'✓ Fixed centers for all {len(centers)} selected ROIs at frame {frame+1}. '
        'Frame Slider AUTO: detect core + calculate FWHM + intensity.')
    _tab11_auto_analyze_current_frame(self,frame,draw=True)
def _tab11_ref_changed(self, event=None):
    _tab11_clear_current_fit_views(self)
    if getattr(self, '_tab11_primary_ready', False):
        self._tab11_render( int(self.tab11_frame_var.get()))


def _tab11_roi_selection_changed(self, event=None):
    _tab11_clear_current_fit_views(self)
    if getattr(self, '_tab11_primary_ready', False):
        self._tab11_render( int(self.tab11_frame_var.get()))


def _tab11_frame_changed(self, value=None):
    if getattr(self, '_tab11_programmatic', False):
        return
    if not getattr(self, '_tab11_primary_ready', False):
        if not _tab11_fetch_primary(self, preserve_frame=True):
            return

    try:
        idx=int(round(float(value))) if value is not None else int(self.tab11_frame_var.get())
    except Exception:
        idx=int(self.tab11_frame_var.get())

    idx=max(0,min(idx,len(self._tab11_primary_stack)-1))
    self._tab11_programmatic=True
    try:
        self.tab11_frame_var.set(idx)
        self.tab11_slider.set(idx)
    finally:
        self._tab11_programmatic=False

    # Once FIX CENTERS — ALL ROIs has been clicked, the slider is the
    # trigger for automatic per-frame core detection + FWHM/intensity fit.
    if (getattr(self,'_tab11_auto_frame_fit',False) and
        getattr(self,'_tab11_center_fixed',False) and
        getattr(self,'_tab11_rois',[])):
        try:
            _tab11_auto_analyze_current_frame(self,idx,draw=True)
            return
        except Exception as exc:
            self.tab11_status_var.set(
                f'Automatic frame analysis failed: {type(exc).__name__}: {exc}')

    # Existing behavior before FIX CENTERS — ALL ROIs.
    _tab11_clear_current_fit_views(self)
    self._tab11_render(idx)

def _tab11_clear_figure5(self):
    """Clear ONLY Figure 5 curves/data and its ALL-ROI tracking dataframe.

    Important: do not use _tab11_clear_twin_axes() here because that helper
    also removes the twin axes belonging to Figure 3 (current ROI) and
    Figure 4 (fit residual/current tracking).  Figure 5 has its own twin
    axes, so remove only those Figure-5-specific axes.
    """
    # Remove ONLY Figure-5 twin axes:
    #   _tab11_all_amp_ax   -> Gaussian amplitude axis for Figure 5
    #   _tab11_all_frame_ax -> frame-number axis for Figure 5
    for attr in ('_tab11_all_amp_ax', '_tab11_all_frame_ax'):
        ax5 = getattr(self, attr, None)
        if ax5 is not None:
            try:
                ax5.remove()
            except Exception:
                pass
            setattr(self, attr, None)

    ax = getattr(self, 'tab11_ax_all_tracking', None)
    if ax is not None:
        try:
            ax.clear()
            ax.set_title('ALL SELECTED ROIs', pad=10)
            ax.set_xlabel('Equivalent field (mT)')
            ax.set_ylabel('Avg. FWHM (nm)')
            ax.grid(alpha=0.2)
        except Exception:
            pass

    # Clear only Figure-5 tracking data. Do not touch ROI/current-fit state.
    self._tab11_tracking_df = pd.DataFrame()

    try:
        self.tab11_fit_status_var.set(
            '✓ Figure 5 cleared. ROI definitions, fixed centers and current-frame analysis preserved.'
        )
    except Exception:
        pass
    try:
        self.tab11_status_var.set('✓ Figure 5 cleared.')
    except Exception:
        pass
    try:
        self.tab11_canvas.draw_idle()
    except Exception:
        pass


def _tab11_detect_particle_for_tracking(self, crop, roi_idx):
    """
    Particle-presence detector used ONLY by TRACK ALL ROIs-SELECTED FRAMES.

    The detector is deliberately independent of _tab11_center() and
    _tab11_fit_roi().  It uses the fixed ROI center as a search reference,
    removes slow background variation, searches several particle-size
    scales, and requires a localized, high-SNR core with measurable
    center-vs-annulus contrast.  A noise maximum therefore does not get
    passed to the Gaussian fitter.

    Returns
    -------
    detected : bool
    center : dict or None
        Detection center in ROI-local pixels, plus diagnostic quantities.
    """
    try:
        a = np.asarray(crop, dtype=float)
        if a.ndim != 2 or a.size < 49:
            return False, None

        finite = np.isfinite(a)
        if not np.any(finite):
            return False, None

        med_all = float(np.nanmedian(a[finite]))
        if not np.isfinite(med_all):
            return False, None
        a = np.where(finite, a, med_all)

        h, w = a.shape
        if min(h, w) < 9:
            return False, None

        # Fixed center established by FIX CENTERS — ALL ROIs.
        fixed = getattr(self, '_tab11_centers', {}).get(int(roi_idx), None)
        if not isinstance(fixed, dict):
            return False, None

        fx = float(fixed.get('x0', np.nan))
        fy = float(fixed.get('y0', np.nan))
        if not (np.isfinite(fx) and np.isfinite(fy)):
            return False, None

        # The particle may move within the selected ROI, but do not allow a
        # distant noise feature to become the tracked particle.
        search_radius = max(5.0, 0.28 * min(h, w))

        # Remove illumination/background gradients.  This is only for
        # detection; the original image is still sent to the existing fitter.
        bg_sigma = max(3.0, 0.18 * min(h, w))
        bg = gaussian_filter(a, sigma=bg_sigma, mode='nearest')
        residual = a - bg

        # Robust global noise estimate from the residual.
        b = max(2, int(round(0.10 * min(h, w))))
        border = np.concatenate([
            residual[:b, :].ravel(), residual[-b:, :].ravel(),
            residual[:, :b].ravel(), residual[:, -b:].ravel()
        ])
        border = border[np.isfinite(border)]
        if border.size < 20:
            border = residual.ravel()

        noise = 1.4826 * float(np.median(
            np.abs(border - np.median(border))
        ))
        if not np.isfinite(noise) or noise <= 0:
            noise = float(np.std(border))
        if not np.isfinite(noise) or noise <= 0:
            noise = max(float(np.finfo(float).eps), np.std(residual) * 0.1)

        # Multi-scale matched-filter-like search.  The Gaussian filtering
        # suppresses isolated pixel noise while retaining particle-sized cores.
        scales = (
            max(1.5, 0.025 * min(h, w)),
            max(2.0, 0.035 * min(h, w)),
            max(2.5, 0.050 * min(h, w)),
            max(3.0, 0.070 * min(h, w)),
            max(4.0, 0.095 * min(h, w)),
            max(5.0, 0.125 * min(h, w)),
        )

        yy, xx = np.indices((h, w), dtype=float)
        search = (xx - fx) ** 2 + (yy - fy) ** 2 <= search_radius ** 2

        best = None
        for sig in scales:
            smooth = gaussian_filter(residual, sigma=sig, mode='nearest')

            # Estimate the noise in this smoothed response from the border.
            rb = np.concatenate([
                smooth[:b, :].ravel(), smooth[-b:, :].ravel(),
                smooth[:, :b].ravel(), smooth[:, -b:].ravel()
            ])
            rb = rb[np.isfinite(rb)]
            rmad = 1.4826 * float(np.median(
                np.abs(rb - np.median(rb))
            ))
            if not np.isfinite(rmad) or rmad <= 0:
                rmad = float(np.std(rb))
            if not np.isfinite(rmad) or rmad <= 0:
                rmad = noise / max(1.0, np.sqrt(2.0 * np.pi) * sig)

            # Evaluate both contrast polarities.
            for polarity in (1.0, -1.0):
                response = polarity * smooth
                candidate = np.where(search, response, -np.inf)
                flat = int(np.argmax(candidate))
                cy, cx = np.unravel_index(flat, candidate.shape)
                peak = float(response[cy, cx])

                if not np.isfinite(peak):
                    continue

                snr = peak / max(rmad, np.finfo(float).eps)

                # First gate: a particle must be clearly above filtered noise.
                if snr < 4.5:
                    continue

                # Local prominence against an annulus around the candidate.
                rr = max(3.0, 2.2 * sig)
                inner_r = max(1.5, 0.65 * sig)
                outer_r = max(inner_r + 1.0, 1.75 * sig)

                dy = yy - cy
                dx = xx - cx
                dist = np.sqrt(dx * dx + dy * dy)

                inner = response[dist <= inner_r]
                annulus = response[
                    (dist >= inner_r) & (dist <= outer_r)
                ]

                if inner.size < 3 or annulus.size < 8:
                    continue

                center_level = float(np.median(inner))
                annulus_level = float(np.median(annulus))
                prominence = center_level - annulus_level

                # Both absolute and relative prominence gates are required.
                if prominence < max(2.5 * rmad, 0.12 * peak):
                    continue

                # The candidate must be away from the ROI edge.
                margin = max(2.0, 1.5 * sig)
                if (
                    cx < margin or cx > w - 1 - margin or
                    cy < margin or cy > h - 1 - margin
                ):
                    continue

                # Compactness / localization test using the existing 0.60
                # contrast threshold.  This prevents a broad gradient from
                # being accepted just because it has a maximum.
                frac = 0.60
                try:
                    frac = float(np.clip(
                        self.tab11_contrast_fraction_var.get(), 0.0, 0.95
                    ))
                except Exception:
                    pass

                local_patch = response[
                    max(0, int(cy - outer_r)):min(h, int(cy + outer_r) + 1),
                    max(0, int(cx - outer_r)):min(w, int(cx + outer_r) + 1)
                ]
                if local_patch.size == 0:
                    continue

                threshold = frac * peak
                core_mask = local_patch >= threshold
                area = int(np.count_nonzero(core_mask))

                expected_area = np.pi * max(sig, 1.0) ** 2
                if area < max(2, int(0.18 * expected_area)):
                    continue
                if area > max(12, int(7.0 * expected_area)):
                    continue

                # Require the central region itself to carry signal.
                if center_level < max(2.5 * rmad, 0.08 * peak):
                    continue

                score = snr + 0.5 * (prominence / max(rmad, 1e-12))
                if best is None or score > best['score']:
                    best = {
                        'x0': float(cx),
                        'y0': float(cy),
                        'background': float(np.nanmedian(bg)),
                        'sign': float(polarity),
                        'peak': float(peak),
                        'threshold': float(frac * peak),
                        'snr': float(snr),
                        'prominence': float(prominence),
                        'scale': float(sig),
                        'score': float(score),
                    }

        if best is None:
            return False, None

        # Final fixed-center proximity gate. This is intentionally applied
        # after the multi-scale search so real shifted particles can still be
        # found, while unrelated structures/noise cannot hijack tracking.
        if np.hypot(best['x0'] - fx, best['y0'] - fy) > search_radius:
            return False, None

        return True, best

    except Exception:
        return False, None


def _tab11_track_all(self):
    if not getattr(self, '_tab11_primary_ready', False):
        if not _tab11_fetch_primary(self, preserve_frame=True):
            return
    rois = getattr(self, '_tab11_rois', [])
    if not rois:
        messagebox.showwarning('Tab 11', 'Add at least one ROI first.', parent=self.root)
        return

    start = max(0, min(int(self.tab11_start_var.get()), len(self._tab11_primary_stack) - 1))
    end = max(0, min(int(self.tab11_end_var.get()), len(self._tab11_primary_stack) - 1))
    if start > end:
        start, end = end, start

    if not getattr(self, '_tab11_center_fixed', False):
        _tab11_fix_all_centers(self)

    rows = []
    # IMPORTANT: FIT CURRENT ROI Fig.3/Fig.4 are intentionally untouched here.
    # Only the third-row ALL-SELECTED-ROIs tracking panel is refreshed.
    self._tab11_tracking_df = pd.DataFrame()
    self.tab11_fig.subplots_adjust(
        left=0.065, right=0.90, bottom=0.06, top=0.95,
        wspace=0.30, hspace=0.52
    )
    self.tab11_canvas.draw_idle()
    self.root.update_idletasks()

    total = (end - start + 1) * len(rois)
    done = 0
    detected_count = 0

    self.tab11_fit_status_var.set(
        f'⏳ Tracking {len(rois)} ROIs | frames {start + 1}–{end + 1} | 0/{total}'
    )
    self.root.update_idletasks()

    for frame in range(start, end + 1):
        for ridx in range(len(rois)):
            try:
                crop, bounds = _tab11_roi_crop(self, ridx, frame)

                # Per-ROI, per-frame particle-presence test.
                # The existing _tab11_center() and _tab11_fit_roi() are not
                # modified and are used only after a real core is detected.
                zero_mode = bool(
                    getattr(self, 'tab11_zero_if_undetected_var', None) is not None
                    and self.tab11_zero_if_undetected_var.get()
                )

                if zero_mode:
                    detected, det = _tab11_detect_particle_for_tracking(
                        self, crop, ridx
                    )
                else:
                    # Existing TAB 11 fitting/tracking path is retained.
                    detected, det = True, None

                if not detected:
                    # IMPORTANT: absence/noise is a valid zero measurement.
                    # Do NOT call the Gaussian fitter on this frame.
                    rows.append({
                        'roi_index': ridx + 1,
                        'frame': frame,
                        'field_mT': _tab11_field_mT(self, frame),
                        'fwhm': 0.0,
                        'fwhm_err': 0.0,
                        'amplitude': 0.0,
                        'amplitude_err': 0.0,
                        'sigma': 0.0,
                        'sigma_err': 0.0,
                        'r2': 0.0,
                        'rmse': 0.0,
                        'line_rmse': 0.0,
                        'line_cv': 0.0,
                        'core_detected': False,
                    })
                else:
                    r = _tab11_fit_roi(self, ridx, frame, store=False)
                    if r is None:
                        raise RuntimeError('Particle detected but fitting returned no result.')

                    rows.append({
                        'roi_index': r['roi_index'],
                        'frame': r['frame'],
                        'field_mT': r['field_mT'],
                        'fwhm': r['fwhm'],
                        'fwhm_err': r['fwhm_err'],
                        'amplitude': r['amplitude'],
                        'amplitude_err': r['amplitude_err'],
                        'sigma': r['sigma'],
                        'sigma_err': r['sigma_err'],
                        'r2': r['r2'],
                        'rmse': r['rmse'],
                        'line_rmse': r['line_rmse'],
                        'line_cv': r['line_cv'],
                        'core_detected': True,
                    })
                    detected_count += 1

            except Exception as exc:
                rows.append({
                    'roi_index': ridx + 1,
                    'frame': frame,
                    'field_mT': _tab11_field_mT(self, frame),
                    'fwhm': 0.0 if zero_mode else np.nan,
                    'fwhm_err': 0.0 if zero_mode else np.nan,
                    'amplitude': 0.0 if zero_mode else np.nan,
                    'amplitude_err': 0.0 if zero_mode else np.nan,
                    'sigma': 0.0 if zero_mode else np.nan,
                    'sigma_err': 0.0 if zero_mode else np.nan,
                    'r2': 0.0 if zero_mode else np.nan,
                    'rmse': 0.0 if zero_mode else np.nan,
                    'line_rmse': 0.0 if zero_mode else np.nan,
                    'line_cv': 0.0 if zero_mode else np.nan,
                    'core_detected': False,
                    'error': f'{type(exc).__name__}: {exc}',
                })

            done += 1

            # Keep the existing live TRACK ALL progress display.
            self._tab11_tracking_df = pd.DataFrame(rows)
            _tab11_plot_tracking_panel(
                self, self._tab11_tracking_df,
                self.tab11_ax_all_tracking, mode='all',
                title='ALL SELECTED ROIs',
                draw=False
            )
            self.tab11_fig.subplots_adjust(
                left=0.065, right=0.90, bottom=0.06, top=0.95,
                wspace=0.30, hspace=0.52
            )
            self.tab11_canvas.draw_idle()
            self.root.update_idletasks()

            self.tab11_fit_status_var.set(
                f'⏳ Tracking {len(rois)} ROIs | '
                f'frame {frame + 1}/{len(self._tab11_primary_stack)} | '
                f'{done}/{total} | cores detected: {detected_count}'
            )

    self._tab11_tracking_df = pd.DataFrame(rows)

    _tab11_plot_tracking_panel(
        self, self._tab11_tracking_df,
        self.tab11_ax_all_tracking, mode='all',
        title='ALL SELECTED ROIs',
        draw=False
    )
    self.tab11_fig.subplots_adjust(
        left=0.065, right=0.90, bottom=0.06, top=0.95,
        wspace=0.30, hspace=0.52
    )
    self.tab11_canvas.draw_idle()

    self.tab11_fit_status_var.set(
        f'✓ Tracking complete | {len(rois)} ROIs | '
        f'frames {start + 1}–{end + 1} | {len(rows)} measurements | '
        f'{detected_count} particle cores detected'
    )


def _tab11_export_current_csv(self):
    """Export only the currently selected ROI profile/average/Gaussian fit."""
    result = getattr(self, '_tab11_last_fit', None)
    if not isinstance(result, dict):
        messagebox.showwarning(
            'Tab 11 Export',
            'Run FIT CURRENT ROI first.',
            parent=self.root
        )
        return

    try:
        roi_id = int(result['roi_index'])
        frame_id = int(result['frame']) + 1
        field = result.get('field_mT', np.nan)
        field_txt = f'{field:+.2f}mT' if np.isfinite(field) else 'FieldNA'

        path = filedialog.asksaveasfilename(
            parent=self.root,
            title='Export Current Selected ROI',
            defaultextension='.csv',
            filetypes=[('CSV', '*.csv'), ('All files', '*.*')],
            initialfile=(
                f'Tab11_ROI{roi_id}_Current_Frame{frame_id:03d}_'
                f'LineProfile_Avg_Gaussian.csv'
            )
        )
        if not path:
            return

        data = {
            'ROI': np.full(len(result['t_nm']), roi_id, dtype=int),
            'Frame': np.full(len(result['t_nm']), frame_id, dtype=int),
            'Field_mT': np.full(len(result['t_nm']), field, dtype=float),
            'Position_nm': np.asarray(result['t_nm'], dtype=float),
        }

        for angle in result['angles']:
            data[f'Profile_{float(angle):g}deg'] = np.asarray(
                result['profiles'][float(angle)], dtype=float
            )

        data['Average_Profile'] = np.asarray(
            result['average_profile'], dtype=float
        )
        data['Gaussian_Fit'] = np.asarray(
            result['fit_profile'], dtype=float
        )

        pd.DataFrame(data).to_csv(path, index=False)

        self.tab11_export_status_var.set(
            f'✓ Exported current ROI {roi_id}: {Path(path).name}'
        )

    except Exception as exc:
        self.tab11_export_status_var.set(
            f'✗ Current ROI export failed: {type(exc).__name__}: {exc}'
        )
        messagebox.showerror(
            'Tab 11 Export — Current ROI',
            f'{type(exc).__name__}: {exc}',
            parent=self.root
        )



def _tab11_export_tracking_csv(self):
    """Export all data points represented by Figure 5."""
    df = getattr(self, '_tab11_tracking_df', None)
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        messagebox.showwarning(
            'Tab 11 Export',
            'Figure 5 has no tracking data. Run FIT AND TRACK ALL SELECTED ROIs first.',
            parent=self.root
        )
        return

    try:
        path = filedialog.asksaveasfilename(
            parent=self.root,
            title='Export Track Data — Figure 5',
            defaultextension='.csv',
            filetypes=[('CSV', '*.csv'), ('All files', '*.*')],
            initialfile='Tab11_Figure5_All_Selected_ROI_Track_Data.csv'
        )
        if not path:
            return

        export_df = df.copy()
        if 'frame' in export_df.columns:
            export_df['Frame'] = pd.to_numeric(
                export_df['frame'], errors='coerce'
            ) + 1
        if 'field_mT' in export_df.columns:
            export_df['Equivalent_Field_mT'] = pd.to_numeric(
                export_df['field_mT'], errors='coerce'
            )
        if 'fwhm' in export_df.columns:
            export_df['FWHM_nm'] = pd.to_numeric(
                export_df['fwhm'], errors='coerce'
            )
        if 'fwhm_err' in export_df.columns:
            export_df['FWHM_Error_nm'] = pd.to_numeric(
                export_df['fwhm_err'], errors='coerce'
            )
        if 'amplitude' in export_df.columns:
            export_df['Gaussian_Amplitude'] = pd.to_numeric(
                export_df['amplitude'], errors='coerce'
            )
        if 'amplitude_err' in export_df.columns:
            export_df['Amplitude_Error'] = pd.to_numeric(
                export_df['amplitude_err'], errors='coerce'
            )

        export_df.to_csv(path, index=False)

        self.tab11_export_status_var.set(
            f'✓ Exported Figure 5 track data: {Path(path).name}'
        )

    except Exception as exc:
        self.tab11_export_status_var.set(
            f'✗ Track data export failed: {type(exc).__name__}: {exc}'
        )
        messagebox.showerror(
            'Tab 11 Export — Track Data',
            f'{type(exc).__name__}: {exc}',
            parent=self.root
        )


def _tab11_export_figures(self):
    """Export each Tab-11 figure panel separately as 300-PPI PNG."""
    if getattr(self, 'tab11_fig', None) is None:
        return

    folder = filedialog.askdirectory(
        parent=self.root,
        title='Select folder for Tab 11 figure export'
    )
    if not folder:
        return

    try:
        self.tab11_canvas.draw()
        fig = self.tab11_fig
        renderer = fig.canvas.get_renderer()

        frame0 = int(self.tab11_frame_var.get())
        field = _tab11_field_mT(self, frame0)
        field_txt = f'{field:+.2f}mT' if np.isfinite(field) else 'FieldNA'

        panels = [
            ('Figure1_ProcessedPrimaryROI1_MultiROI',
             [getattr(self, 'tab11_ax_primary', None)]),
            ('Figure2_SelectedROI_ReferenceLines',
             [getattr(self, 'tab11_ax_selected', None)]),
            ('Figure3_CurrentROI_AvgProfile_Gaussian_Residual',
             [getattr(self, 'tab11_ax_fit', None),
              getattr(self, '_tab11_fit_residual_ax', None)]),
            ('Figure4_CurrentROI_FWHM_Amplitude',
             [getattr(self, 'tab11_ax_current_tracking', None),
              getattr(self, '_tab11_current_amp_ax', None),
              getattr(self, '_tab11_current_frame_ax', None)]),
            ('Figure5_AllSelectedROIs_AvgFWHM_Amplitude',
             [getattr(self, 'tab11_ax_all_tracking', None),
              getattr(self, '_tab11_all_amp_ax', None),
              getattr(self, '_tab11_all_frame_ax', None)]),
        ]

        written=[]
        for stem, axes in panels:
            axes=[a for a in axes if a is not None and a in fig.axes]
            if not axes:
                continue

            # Matplotlib Bbox objects are not sequences; never use len(bbox).
            boxes=[]
            for ax in axes:
                try:
                    b=ax.get_tightbbox(renderer)
                    if b is not None and np.isfinite(
                        [b.x0,b.y0,b.x1,b.y1]
                    ).all():
                        boxes.append(b)
                except Exception:
                    pass

            if boxes:
                # Union the Bboxes explicitly.
                x0=min(b.x0 for b in boxes)
                y0=min(b.y0 for b in boxes)
                x1=max(b.x1 for b in boxes)
                y1=max(b.y1 for b in boxes)
                from matplotlib.transforms import Bbox
                box=Bbox.from_extents(x0,y0,x1,y1)
                bbox=box.transformed(
                    fig.dpi_scale_trans.inverted()
                ).expanded(1.06,1.10)
            else:
                bbox='tight'

            filename=(
                f'Tab11_{stem}_Frame{frame0+1:03d}_{field_txt}.png'
            )
            path=os.path.join(folder,filename)

            fig.savefig(
                path,
                dpi=300,
                bbox_inches=bbox,
                facecolor='white',
                edgecolor='white',
                format='png'
            )
            written.append(path)

        self.tab11_export_status_var.set(
            f'✓ Exported {len(written)} separate Tab 11 figures at 300 PPI.'
        )
        messagebox.showinfo(
            'Tab 11 Export Figures',
            f'Exported {len(written)} separate PNG figures at 300 PPI to:\n{folder}',
            parent=self.root
        )

    except Exception as exc:
        self.tab11_export_status_var.set(
            f'✗ Figure export failed: {type(exc).__name__}: {exc}'
        )
        messagebox.showerror(
            'Tab 11 Export Figures',
            f'{type(exc).__name__}: {exc}',
            parent=self.root
        )



def _tab11_export_selected_roi_data(self):
    """Export all stored data points currently represented by Figure 4."""
    df = getattr(self, '_tab11_current_tracking_df', None)
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        messagebox.showwarning(
            'Tab 11 Export',
            'Figure 4 has no current-ROI tracking data. Run FIT CURRENT ROI first.',
            parent=self.root
        )
        return

    try:
        roi_id = _tab11_selected_roi_index(self) + 1
        path = filedialog.asksaveasfilename(
            parent=self.root,
            title='Export Selected ROI Data — Figure 4',
            defaultextension='.csv',
            filetypes=[('CSV', '*.csv'), ('All files', '*.*')],
            initialfile=f'Tab11_Selected_ROI{roi_id}_Figure4_Data.csv'
        )
        if not path:
            return

        export_df = df.copy()
        if 'frame' in export_df.columns:
            export_df['Frame'] = pd.to_numeric(
                export_df['frame'], errors='coerce'
            ) + 1
        if 'field_mT' in export_df.columns:
            export_df['Equivalent_Field_mT'] = pd.to_numeric(
                export_df['field_mT'], errors='coerce'
            )
        if 'fwhm' in export_df.columns:
            export_df['FWHM_nm'] = pd.to_numeric(
                export_df['fwhm'], errors='coerce'
            )
        if 'fwhm_err' in export_df.columns:
            export_df['FWHM_Error_nm'] = pd.to_numeric(
                export_df['fwhm_err'], errors='coerce'
            )
        if 'amplitude' in export_df.columns:
            export_df['Gaussian_Amplitude'] = pd.to_numeric(
                export_df['amplitude'], errors='coerce'
            )
        if 'amplitude_err' in export_df.columns:
            export_df['Amplitude_Error'] = pd.to_numeric(
                export_df['amplitude_err'], errors='coerce'
            )

        # Keep the exported version compact and meaningful.
        preferred = [
            'roi_index', 'Frame', 'Equivalent_Field_mT',
            'FWHM_nm', 'FWHM_Error_nm',
            'Gaussian_Amplitude', 'Amplitude_Error'
        ]
        cols = [c for c in preferred if c in export_df.columns]
        export_df[cols if cols else export_df.columns].to_csv(
            path, index=False
        )

        self.tab11_export_status_var.set(
            f'✓ Exported Figure 4 data: {Path(path).name}'
        )

    except Exception as exc:
        self.tab11_export_status_var.set(
            f'✗ Selected ROI data export failed: {type(exc).__name__}: {exc}'
        )
        messagebox.showerror(
            'Tab 11 Export — Selected ROI Data',
            f'{type(exc).__name__}: {exc}',
            parent=self.root
        )

def _tab11_build(self):
    if not hasattr(self, 'tab_multi_roi11'):
        self.tab_multi_roi11 = ttk.Frame(self.nb)
        self.nb.add(self.tab_multi_roi11, text='11. Multi-ROI Fit')

    tab = self.tab_multi_roi11
    for child in list(tab.winfo_children()):
        try:
            child.destroy()
        except Exception:
            pass

    self.tab11_contrast_fraction_var = tk.DoubleVar(value=0.50)
    self.tab11_ref_mode_var = tk.StringVar(value='2')
    self.tab11_frame_var = tk.IntVar(value=0)
    self.tab11_start_var = tk.IntVar(value=0)
    self.tab11_end_var = tk.IntVar(value=0)
    self.tab11_frame_label_var = tk.StringVar(value='Frame = 1')
    self.tab11_roi_select_var = tk.StringVar(value='')
    self.tab11_status_var = tk.StringVar(value='Click FETCH PRIMARY ROI FROM TAB 6 — uses current Filter / Brightness / Contrast / View / Color.')
    self.tab11_fit_status_var = tk.StringVar(value='No fit/tracking performed.')
    self.tab11_export_status_var = tk.StringVar(value='')
    self.tab11_zero_if_undetected_var = tk.BooleanVar(value=False)

    self._tab11_primary_ready = False
    self._tab11_primary_stack = None
    self._tab11_scale_nm = (1.0, 1.0)
    self._tab11_primary_extent = None
    self._tab11_cmap = str(getattr(self, 'colormap_var', tk.StringVar(value='gray')).get() or 'gray')
    self._tab11_view = str(getattr(self, 'view_var', tk.StringVar(value='Image')).get() or 'Image')
    self._tab11_clim = (0.0, 1.0)
    self._tab11_rois = []
    self._tab11_centers = {}
    self._tab11_center_fixed = False
    self._tab11_select_mode = False
    self._tab11_press_xy = None
    self._tab11_preview_rect = None
    self._tab11_last_fit = None
    self._tab11_current_tracking_df = None
    self._tab11_tracking_df = None
    self._tab11_programmatic = False
    self._tab11_cbar = None
    self._tab11_fit_residual_ax = None
    self._tab11_current_amp_ax = None
    self._tab11_current_frame_ax = None
    self._tab11_all_amp_ax = None
    self._tab11_all_frame_ax = None

    # ---------- Adjustable divider: plot area | scrollable controls ----------
    outer = ttk.Panedwindow(tab, orient='horizontal')
    outer.grid(row=0, column=0, sticky='nsew')
    tab.grid_rowconfigure(0, weight=1)
    tab.grid_columnconfigure(0, weight=1)

    # ---------- LEFT: scrollable figure panel ----------
    plot_outer = ttk.Frame(outer)
    plot_outer.grid_rowconfigure(0, weight=1)
    plot_outer.grid_columnconfigure(0, weight=1)

    plot_scroll = tk.Canvas(
        plot_outer, highlightthickness=0, borderwidth=0
    )
    plot_vbar = ttk.Scrollbar(
        plot_outer, orient='vertical', command=plot_scroll.yview
    )
    plot_hbar = ttk.Scrollbar(
        plot_outer, orient='horizontal', command=plot_scroll.xview
    )
    plot_scroll.configure(
        yscrollcommand=plot_vbar.set,
        xscrollcommand=plot_hbar.set
    )
    plot_scroll.grid(row=0, column=0, sticky='nsew')
    plot_vbar.grid(row=0, column=1, sticky='ns')
    plot_hbar.grid(row=1, column=0, sticky='ew')

    plot_host = ttk.Frame(plot_scroll, width=1180, height=1080)
    plot_window = plot_scroll.create_window((0, 0), window=plot_host, anchor='nw')

    def _plot_sync(_event=None):
        plot_scroll.configure(scrollregion=plot_scroll.bbox('all'))
    plot_host.bind('<Configure>', _plot_sync)

    outer.add(plot_outer, weight=7)

    self.tab11_fig = plt.Figure(
        figsize=(15, 11), dpi=100, facecolor='white'
    )
    gs = self.tab11_fig.add_gridspec(
        3, 2,
        left=0.08, right=0.92,
        bottom=0.05, top=0.96,
        wspace=0.42, hspace=0.40,
        width_ratios=[1.0, 1.0],
        height_ratios=[1.50, 0.9, 0.9]
    )
    self.tab11_ax_primary = self.tab11_fig.add_subplot(gs[0, 0])
    self.tab11_ax_selected = self.tab11_fig.add_subplot(gs[0, 1])
    self.tab11_ax_fit = self.tab11_fig.add_subplot(gs[1, 0])
    self.tab11_ax_current_tracking = self.tab11_fig.add_subplot(gs[1, 1])
    self.tab11_ax_all_tracking = self.tab11_fig.add_subplot(gs[2, :])

    dummy = np.zeros((10, 10))
    for ax, title in (
        (self.tab11_ax_primary, 'PROCESSED PRIMARY ROI'),
        (self.tab11_ax_selected, 'SELECTED ROI — None')
    ):
        ax.imshow(
            dummy, cmap=self._tab11_cmap, origin='lower',
            extent=(0, 10, 0, 10), vmin=0, vmax=1,
            interpolation='nearest', aspect='equal'
        )
        ax.set_xlabel('X (nm)')
        ax.set_ylabel('Y (nm)')
        ax.set_title(title, pad=9)

    self.tab11_ax_fit.set_title(
        'FIT CURRENT ROI',
        pad=9
    )
    self.tab11_ax_fit.set_xlabel('Position from fixed center (nm)')
    self.tab11_ax_fit.set_ylabel('Intensity')
    self.tab11_ax_fit.grid(alpha=0.2)

    self.tab11_ax_current_tracking.set_title(
        'CURRENT ROI',
        pad=9
    )
    self.tab11_ax_current_tracking.set_xlabel('Equivalent field (mT)')
    self.tab11_ax_current_tracking.set_ylabel('FWHM (nm)')
    self.tab11_ax_current_tracking.grid(alpha=0.2)

    self.tab11_ax_all_tracking.set_title(
        'ALL SELECTED ROIs',
        pad=9
    )
    self.tab11_ax_all_tracking.set_xlabel('Equivalent field (mT)')
    self.tab11_ax_all_tracking.set_ylabel('Avg. FWHM (nm)')
    self.tab11_ax_all_tracking.grid(alpha=0.2)

    self.tab11_canvas = FigureCanvasTkAgg(self.tab11_fig, master=plot_host)
    self.tab11_canvas.draw()
    widget = self.tab11_canvas.get_tk_widget()
    widget.configure(width=1200, height=1500)
    widget.grid(row=0, column=0, sticky='nsew')

    self.tab11_canvas.mpl_connect(
        'button_press_event', lambda e: _tab11_mouse_press(self, e)
    )
    self.tab11_canvas.mpl_connect(
        'motion_notify_event', lambda e: _tab11_mouse_move(self, e)
    )
    self.tab11_canvas.mpl_connect(
        'button_release_event', lambda e: _tab11_mouse_release(self, e)
    )
    self.tab11_canvas.mpl_connect(
        'scroll_event', lambda e: _tab11_zoom(self, e)
    )

    # Mouse-wheel scrolling only when pointer is outside the Matplotlib axes.
    def _figure_wheel(event):
        if getattr(event, 'inaxes', None) is None:
            return
        try:
            plot_scroll.yview_scroll(int(-1 * (event.step or 0)), 'units')
        except Exception:
            pass
    self.tab11_canvas.mpl_connect('scroll_event', _figure_wheel)

    # ---------- RIGHT: scrollable control panel ----------
    side_outer = ttk.Frame(outer, width=370)
    side_outer.grid_rowconfigure(0, weight=1)
    side_outer.grid_columnconfigure(0, weight=1)
    outer.add(side_outer, weight=3)

    side_canvas = tk.Canvas(
        side_outer, highlightthickness=0, borderwidth=0, width=360
    )
    side_vbar = ttk.Scrollbar(
        side_outer, orient='vertical', command=side_canvas.yview
    )
    side_canvas.configure(yscrollcommand=side_vbar.set)
    side_canvas.grid(row=0, column=0, sticky='nsew')
    side_vbar.grid(row=0, column=1, sticky='ns')

    inner = ttk.Frame(side_canvas, padding=6)
    side_window = side_canvas.create_window((0, 0), window=inner, anchor='nw')

    def _side_sync(_event=None):
        side_canvas.configure(scrollregion=side_canvas.bbox('all'))
        try:
            side_canvas.itemconfigure(side_window, width=max(1, side_canvas.winfo_width() - 4))
        except Exception:
            pass
    inner.bind('<Configure>', _side_sync)
    side_canvas.bind('<Configure>', _side_sync)

    def _side_wheel(event):
        try:
            side_canvas.yview_scroll(int(-1 * (event.delta / 120)), 'units')
        except Exception:
            pass
    side_canvas.bind('<MouseWheel>', _side_wheel)
    inner.bind('<MouseWheel>', _side_wheel)

    # Source and ROI selection.
    src_box = ttk.LabelFrame(inner, text='PRIMARY ROI SOURCE / ROI SELECTION', padding=6)
    src_box.pack(fill='x', pady=(0, 6))

    ttk.Button(
        src_box, text='FETCH ROI FROM TAB 6',
        command=lambda: _tab11_fetch_primary(self, preserve_frame=True)
    ).pack(fill='x', pady=2)

    ttk.Label(
        src_box,
        text='Draw multiple ROIs directly on the fetched processed Primary ROI1.'
    ).pack(anchor='w', pady=(2, 4))

    row = ttk.Frame(src_box)
    row.pack(fill='x', pady=2)
    ttk.Label(row, text='ROI selection').pack(side='left')
    self.tab11_roi_select_combo = ttk.Combobox(
        row, textvariable=self.tab11_roi_select_var,
        values=[], state='readonly', width=12
    )
    self.tab11_roi_select_combo.pack(
        side='right', fill='x', expand=True, padx=(8, 0)
    )
    self.tab11_roi_select_combo.bind(
        '<<ComboboxSelected>>',
        lambda e: _tab11_roi_selection_changed(self, e)
    )

    btnrow = ttk.Frame(src_box)
    btnrow.pack(fill='x', pady=2)
    ttk.Button(
        btnrow, text='SELECT / ADD ROI',
        command=lambda: _tab11_start_selection(self)
    ).pack(side='left', fill='x', expand=True, padx=1)
    ttk.Button(
        btnrow, text='CLEAR SELECTED',
        command=lambda: _tab11_clear_selected(self)
    ).pack(side='left', fill='x', expand=True, padx=1)
    ttk.Button(
        btnrow, text='CLEAR ALL',
        command=lambda: _tab11_clear_all(self)
    ).pack(side='left', fill='x', expand=True, padx=1)

    ttk.Button(
        src_box, text='RESET ZOOM',
        command=lambda: _tab11_reset_zoom(self)
    ).pack(fill='x', pady=(3, 0))

    # Analysis settings.
    anal_box = ttk.LabelFrame(inner, text='ANALYSIS SETTINGS', padding=6)
    anal_box.pack(fill='x', pady=6)

    rr = ttk.Frame(anal_box)
    rr.pack(fill='x', pady=2)
    ttk.Label(rr, text='REFERENCE LINES').pack(side='left')
    ttk.Combobox(
        rr, textvariable=self.tab11_ref_mode_var,
        values=('1', '2', '4', '6'), state='readonly', width=6
    ).pack(side='right')
    self.tab11_ref_mode_var.trace_add('write', lambda *_: _tab11_ref_changed(self))

    tr = ttk.Frame(anal_box)
    tr.pack(fill='x', pady=2)
    ttk.Label(tr, text='CONTRAST THRESHOLD').pack(side='left')
    tk.Spinbox(
        tr, from_=0.00, to=0.95, increment=0.05,
        textvariable=self.tab11_contrast_fraction_var,
        width=7, format='%.2f'
    ).pack(side='right')

    self.tab11_ref_label = ttk.Label(
        anal_box, text='2 lines: 0° / 90°'
    )
    self.tab11_ref_label.pack(anchor='w', pady=(2, 0))

    # Center control.
    center_box = ttk.LabelFrame(inner, text='CENTER CONTROL', padding=6)
    center_box.pack(fill='x', pady=6)

    ttk.Button(
        center_box, text='RECALCULATE CENTER — SELECTED ROI',
        command=lambda: _tab11_recalculate_center_selected(self)
    ).pack(fill='x', pady=2)

    ttk.Button(
        center_box, text='FIX CENTERS',
        command=lambda: _tab11_fix_all_centers(self)
    ).pack(fill='x', pady=2)

    # Fit/tracking.
    fit_box = ttk.LabelFrame(inner, text='FIT / TRACKING', padding=6)
    fit_box.pack(fill='x', pady=6)

    ttk.Button(
        fit_box, text='FIT CURRENT ROI',
        command=lambda: _tab11_fit_current(self)
    ).pack(fill='x', pady=2)

    ttk.Button(
        fit_box, text='TRACK ALL ROIs-SELECTED FRAMES',
        command=lambda: _tab11_track_all(self)
    ).pack(fill='x', pady=2)
    ttk.Checkbutton(
        fit_box, text='Zero if Undetected',
        variable=self.tab11_zero_if_undetected_var
    ).pack(anchor='w', pady=(0, 2))

    ttk.Button(
        fit_box, text='CLEAR FIGURE 5',
        command=lambda: _tab11_clear_figure5(self)
    ).pack(fill='x', pady=2)

    # Frame/range.
    fr_box = ttk.LabelFrame(inner, text='FRAME / TRACKING RANGE', padding=6)
    fr_box.pack(fill='x', pady=6)

    ttk.Label(
        fr_box, textvariable=self.tab11_frame_label_var
    ).pack(anchor='w', pady=(0, 2))

    self.tab11_slider = tk.Scale(
        fr_box, from_=0, to=0,
        orient='horizontal',
        variable=self.tab11_frame_var,
        resolution=1,
        showvalue=True,
        highlightthickness=0, bd=1,
        command=lambda v: _tab11_frame_changed(self, v)
    )
    self.tab11_slider.pack(fill='x', pady=2)
    try:
        _n11 = len(getattr(self,'_tab11_primary_stack',None)) if getattr(self,'_tab11_primary_stack',None) is not None else 0
        if _n11:
            self.tab11_slider.configure(from_=0,to=_n11-1,state='normal')
            _set_frame_slider_midpoint_once(self,self.tab11_slider,_n11,self.tab11_frame_var,zero_based=True,token='tab11')
    except Exception:
        pass

    rg = ttk.Frame(fr_box)
    rg.pack(fill='x', pady=2)
    ttk.Label(rg, text='START').pack(side='left')
    self.tab11_start_spin = tk.Spinbox(
        rg, textvariable=self.tab11_start_var,
        from_=0, to=0, width=7, increment=1
    )
    self.tab11_start_spin.pack(side='left', padx=(5, 14))
    ttk.Label(rg, text='END').pack(side='left')
    self.tab11_end_spin = tk.Spinbox(
        rg, textvariable=self.tab11_end_var,
        from_=0, to=0, width=7, increment=1
    )
    self.tab11_end_spin.pack(side='left', padx=(5, 0))

    ttk.Button(
        fr_box, text='USE ALL FRAMES',
        command=lambda: (
            self.tab11_start_var.set(0),
            self.tab11_end_var.set(max(0, len(self._tab11_primary_stack)-1))
            if getattr(self, '_tab11_primary_stack', None) is not None else None
        )
    ).pack(fill='x', pady=3)

    ttk.Label(
        fr_box, textvariable=self.tab11_status_var,
        wraplength=330, justify='left'
    ).pack(fill='x', pady=(3, 0))

    # Fitting parameters.
    pbox = ttk.LabelFrame(inner, text='FITTING PARAMETERS', padding=6)
    pbox.pack(fill='x', pady=6)

    param_keys = [
        ('roi', 'Selected ROI'), ('center', 'Fixed center'),
        ('background', 'Gaussian background'), ('threshold', 'Contrast threshold'),
        ('sign', 'Contrast sign'), ('amplitude', 'Amplitude'),
        ('sigma', 'Sigma'), ('fwhm', 'FWHM'), ('r2', 'R²'),
        ('rmse', 'RMSE'), ('line_rmse', 'Line RMSE'), ('line_cv', 'Line CV'),
        ('fov', 'ROI FOV'), ('frame', 'Frame'), ('field', 'Equivalent field'),
        ('status', 'Status')
    ]
    self.tab11_param_vars = {}
    for key, label in param_keys:
        var = tk.StringVar(value='—')
        self.tab11_param_vars[key] = var
        row = ttk.Frame(pbox)
        row.pack(fill='x', pady=1)
        ttk.Label(row, text=label + ':').pack(side='left')
        ttk.Label(
            row, textvariable=var, justify='right', anchor='e'
        ).pack(side='right', fill='x', expand=True)

    self.tab11_param_vars['status'].set('No fit performed.')

    # Export options.
    exp = ttk.LabelFrame(inner, text='EXPORT OPTIONS', padding=6)
    exp.pack(fill='x', pady=6)

    ttk.Button(
        exp, text='EXPORT FIGURES — PNG',
        command=lambda: _tab11_export_figures(self)
    ).pack(fill='x', pady=2)

    ttk.Button(
        exp, text='EXPORT CURRENT ROI',
        command=lambda: _tab11_export_current_csv(self)
    ).pack(fill='x', pady=2)

    ttk.Button(
        exp, text='EXPORT ALL ROI',
        command=lambda: _tab11_export_multi_fit_info(self)
    ).pack(fill='x', pady=2)

    ttk.Button(
        exp, text='EXPORT ROI DATA — FIGURE 4',
        command=lambda: _tab11_export_selected_roi_data(self)
    ).pack(fill='x', pady=2)

    ttk.Button(
        exp, text='EXPORT TRACK DATA — FIGURE 5',
        command=lambda: _tab11_export_tracking_csv(self)
    ).pack(fill='x', pady=2)

    ttk.Label(
        exp, textvariable=self.tab11_export_status_var,
        wraplength=330, justify='left'
    ).pack(fill='x', pady=(3, 0))

    ttk.Label(
        inner, textvariable=self.tab11_fit_status_var,
        wraplength=330, justify='left'
    ).pack(fill='x', pady=(6, 10))

    # Bind instance callbacks.
    self._tab11_render = lambda idx=None: _tab11_render(self, idx)
    self._tab11_frame_changed = lambda value=None: _tab11_frame_changed(self, value)
    self._tab11_fit_current = lambda: _tab11_fit_current(self)
    self._tab11_track_all = lambda: _tab11_track_all(self)

    _tab11_update_roi_selector(self)


# Replace old Tab 11 section with the corrected 3x2/scrollable implementation.
_TAB11_BASE_BUILD_UI = CDIWorkflowApp._build_ui

def _build_ui_with_tab11(self, *args, **kwargs):
    result = _TAB11_BASE_BUILD_UI(self, *args, **kwargs)
    try:
        _tab11_build(self)
    except Exception as exc:
        try:
            self.log(
                f'Tab 11 Multi-ROI Gaussian initialization warning: '
                f'{type(exc).__name__}: {exc}'
            )
        except Exception:
            pass
    return result

CDIWorkflowApp._build_ui = _build_ui_with_tab11

# ---------------------------------------------------------------------------


# ============================================================================
# TAB 12 — HDF5/FRC FILE EXPLORER
# ============================================================================
# Read-only, integrated HDF5 hierarchy explorer + generic dataset plotter +
# FRC analysis. Uses the same loaded HDF5 source for both explorer and FRC.
# Existing Tabs 1–11 are left untouched.
# ============================================================================

def _tab12_safe_value(v):
    if isinstance(v, np.ndarray):
        if v.size <= 1000:
            return v.tolist()
        return {
            "array": True,
            "shape": list(v.shape),
            "dtype": str(v.dtype),
            "size": int(v.size),
        }
    if isinstance(v, np.generic):
        return _tab12_safe_value(v.item())
    if isinstance(v, bytes):
        return v.decode("utf-8", errors="replace")
    if isinstance(v, (list, tuple)):
        return [_tab12_safe_value(x) for x in v]
    if isinstance(v, dict):
        return {str(k): _tab12_safe_value(x) for k, x in v.items()}
    if isinstance(v, (str, int, float, bool)) or v is None:
        return v
    return str(v)


def _tab12_read_attrs(obj):
    out = {}
    try:
        keys = list(obj.attrs.keys())
    except Exception:
        keys = []
    for key in keys:
        try:
            out[str(key)] = _tab12_safe_value(obj.attrs[key])
        except Exception as exc:
            out[str(key)] = f"<ERROR: {type(exc).__name__}: {exc}>"
    return out


def _tab12_get_unit(obj):
    preferred = (
        "units", "unit", "Units", "Unit",
        "physical_unit", "physical_units",
        "value_unit", "dimension_unit"
    )
    try:
        for key in preferred:
            if key not in obj.attrs:
                continue
            value = obj.attrs[key]
            if isinstance(value, np.ndarray) and value.size == 1:
                value = value.reshape(-1)[0]
            if isinstance(value, bytes):
                value = value.decode("utf-8", errors="replace")
            text = str(value).strip()
            if text and text.lower() not in {"none", "n/a", "na"}:
                return text
    except Exception:
        pass
    return None


def _tab12_axis_unit_for_dataset(obj):
    candidates = (
        "axis_unit", "axis_units",
        "pixel_unit", "pixel_units",
        "xy_unit", "xy_units",
        "spatial_unit", "spatial_units",
        "coordinate_unit", "coordinate_units"
    )
    try:
        for key in candidates:
            if key not in obj.attrs:
                continue
            value = obj.attrs[key]
            if isinstance(value, np.ndarray) and value.size == 1:
                value = value.reshape(-1)[0]
            if isinstance(value, bytes):
                value = value.decode("utf-8", errors="replace")
            text = str(value).strip()
            if text:
                return text
    except Exception:
        pass
    return None


def _tab12_format_unit_label(label, unit):
    return f"{label} ({unit})" if unit else label


def _tab12_numeric_stats(arr):
    a = np.asarray(arr)
    result = {
        "shape": list(a.shape),
        "dtype": str(a.dtype),
        "size": int(a.size),
        "ndim": int(a.ndim),
    }
    if a.size == 0:
        return result
    try:
        if np.iscomplexobj(a):
            channels = {
                "real": np.real(a),
                "imaginary": np.imag(a),
                "amplitude": np.abs(a),
                "phase_rad": np.angle(a),
            }
            for name, x in channels.items():
                x = np.asarray(x)
                f = x[np.isfinite(x)]
                if f.size:
                    result[name] = {
                        "min": float(np.min(f)),
                        "max": float(np.max(f)),
                        "mean": float(np.mean(f)),
                        "std": float(np.std(f)),
                        "median": float(np.median(f)),
                    }
        elif a.dtype == bool or np.issubdtype(a.dtype, np.number):
            x = a.astype(float, copy=False)
            f = x[np.isfinite(x)]
            if f.size:
                result["numeric"] = {
                    "min": float(np.min(f)),
                    "max": float(np.max(f)),
                    "mean": float(np.mean(f)),
                    "std": float(np.std(f)),
                    "median": float(np.median(f)),
                }
    except Exception as exc:
        result["statistics_error"] = f"{type(exc).__name__}: {exc}"
    return result


def _tab12_bounded_preview(ds, max_side=50, max_elements=2500):
    shape = ds.shape
    size = int(ds.size)
    if size <= max_elements:
        return {
            "mode": "full",
            "shape": list(shape),
            "data": _tab12_safe_value(ds[()])
        }
    if ds.ndim == 2:
        r, c = ds.shape
        ri = np.linspace(0, max(r - 1, 0), min(r, max_side), dtype=int)
        ci = np.linspace(0, max(c - 1, 0), min(c, max_side), dtype=int)
        arr = ds[np.ix_(ri, ci)]
        return {
            "mode": "2d_sample",
            "original_shape": list(shape),
            "sample_shape": list(arr.shape),
            "data": _tab12_safe_value(arr),
        }
    slices = tuple(slice(0, min(int(n), max_side)) for n in shape)
    arr = ds[slices]
    return {
        "mode": "nd_sample",
        "original_shape": list(shape),
        "sample_shape": list(arr.shape),
        "data": _tab12_safe_value(arr),
    }


def _tab12_component_data(self, arr):
    component = str(self.tab12_component_var.get() or "Auto")
    a = np.asarray(arr)
    if component == "Raw":
        return a, "Raw"
    if component == "Real":
        return np.real(a), "Real"
    if component == "Imaginary":
        return np.imag(a), "Imaginary"
    if component == "Amplitude":
        return np.abs(a), "Amplitude"
    if component == "Phase":
        return np.angle(a), "Phase (rad)"
    if np.iscomplexobj(a):
        return np.abs(a), "Amplitude"
    return a, "Raw"


def _tab12_slice_for_display(self, data):
    a = np.asarray(data)
    if a.ndim <= 2:
        return a, "full"
    try:
        axis = int(self.tab12_slice_axis_var.get())
    except Exception:
        axis = 0
    axis = max(0, min(axis, a.ndim - 1))
    max_idx = max(0, a.shape[axis] - 1)
    try:
        idx = int(self.tab12_slice_index_var.get())
    except Exception:
        idx = 0
    idx = max(0, min(idx, max_idx))
    sl = [slice(None)] * a.ndim
    sl[axis] = idx
    out = a[tuple(sl)]
    while out.ndim > 2:
        out = out[0]
    return np.asarray(out), f"slice axis={axis}, index={idx}"


def _tab12_close_figure(self):
    old = getattr(self, "tab12_h5_fig", None)
    if old is not None:
        try:
            plt.close(old)
        except Exception:
            pass
    self.tab12_h5_fig = None
    self.tab12_h5_canvas = None


def _tab12_clear_plot(self):
    _tab12_close_figure(self)
    if getattr(self, "tab12_plot_host", None) is not None:
        for child in self.tab12_plot_host.winfo_children():
            try:
                child.destroy()
            except Exception:
                pass


def _tab12_show_figure(self, fig):
    _tab12_close_figure(self)
    self.tab12_h5_fig = fig
    self.tab12_h5_canvas = FigureCanvasTkAgg(fig, master=self.tab12_plot_host)
    self.tab12_h5_canvas.draw()
    self.tab12_h5_canvas.get_tk_widget().pack(fill="both", expand=True)


def _tab12_load_array(self, path):
    if not self.tab12_file_path:
        return None
    with h5py.File(self.tab12_file_path, "r") as f:
        ds = f[path]
        if not isinstance(ds, h5py.Dataset):
            return None
        # Bound memory usage for very large datasets.
        if ds.size > 10_000_000:
            self.tab12_status_var.set(
                f"Large dataset ({ds.size:,} elements): use 2D/slice preview."
            )
            return np.asarray(_tab12_bounded_preview(ds)["data"])
        return ds[()]


def _tab12_populate_tree(self):
    self.tab12_tree.delete(*self.tab12_tree.get_children())
    self.tab12_tree_map = {}
    if not self.tab12_file_path:
        return
    try:
        with h5py.File(self.tab12_file_path, "r") as f:
            root_id = self.tab12_tree.insert(
                "", "end", text="/",
                values=("GROUP", f"{len(f)} items", ""),
                open=True
            )
            self.tab12_tree_map[root_id] = "/"
            nodes = {"/": root_id}

            def ensure_group(path):
                if path in nodes:
                    return nodes[path]
                parts = [p for p in path.strip("/").split("/") if p]
                cur = "/"
                parent = nodes["/"]
                for part in parts:
                    nxt = "/" + part if cur == "/" else cur + "/" + part
                    if nxt not in nodes:
                        item = self.tab12_tree.insert(
                            parent, "end", text=part,
                            values=("GROUP", "", ""), open=False
                        )
                        nodes[nxt] = item
                        self.tab12_tree_map[item] = nxt
                    parent = nodes[nxt]
                    cur = nxt
                return parent

            def visitor(name, obj):
                path = "/" + name.strip("/")
                parent_path = "/" + "/".join(name.strip("/").split("/")[:-1])
                if parent_path == "":
                    parent_path = "/"
                parent = ensure_group(parent_path)
                item_name = name.strip("/").split("/")[-1]
                if isinstance(obj, h5py.Group):
                    item = ensure_group(path)
                    try:
                        count = len(obj)
                    except Exception:
                        count = 0
                    self.tab12_tree.item(
                        item, text=item_name,
                        values=("GROUP", f"{count} items", "")
                    )
                elif isinstance(obj, h5py.Dataset):
                    item = self.tab12_tree.insert(
                        parent, "end", text=item_name,
                        values=("DATASET", str(obj.shape), str(obj.dtype))
                    )
                    self.tab12_tree_map[item] = path

            f.visititems(visitor)
        self.tab12_status_var.set(
            f"Loaded HDF5: {self.tab12_file_path.name}"
        )
    except Exception as exc:
        self.tab12_status_var.set(
            f"HDF5 tree error: {type(exc).__name__}: {exc}"
        )


def _tab12_show_object(self, path):
    if not self.tab12_file_path:
        return
    self.tab12_current_path = path
    self.tab12_meta_text.delete("1.0", "end")
    self.tab12_preview_text.delete("1.0", "end")

    try:
        with h5py.File(self.tab12_file_path, "r") as f:
            obj = f[path]
            attrs = _tab12_read_attrs(obj)

            if isinstance(obj, h5py.Group):
                self.tab12_path_var.set(f"GROUP: {path}")
                lines = [f"GROUP: {path}", "", "ATTRIBUTES:"]
                if attrs:
                    lines += [f"  {k}: {v}" for k,v in attrs.items()]
                else:
                    lines.append("  (none)")
                lines.append("")
                lines.append("CHILDREN:")
                for name in obj.keys():
                    ch = obj[name]
                    if isinstance(ch, h5py.Dataset):
                        lines.append(
                            f"  DATASET {name} | shape={ch.shape} | dtype={ch.dtype}"
                        )
                    else:
                        lines.append(f"  GROUP   {name}")
                self.tab12_meta_text.insert("end", "\n".join(lines))
                self.tab12_preview_text.insert(
                    "end", "Select a DATASET to inspect and plot its contents."
                )
                _tab12_clear_plot(self)
                return

            if isinstance(obj, h5py.Dataset):
                self.tab12_path_var.set(f"DATASET: {path}")
                self.tab12_current_unit = _tab12_get_unit(obj)
                self.tab12_current_axis_unit = _tab12_axis_unit_for_dataset(obj)
                lines = [
                    f"DATASET: {path}",
                    f"Shape       : {obj.shape}",
                    f"Dtype       : {obj.dtype}",
                    f"Dimensions  : {obj.ndim}",
                    f"Elements    : {obj.size:,}",
                    f"Chunks      : {obj.chunks}",
                    f"Compression : {obj.compression}",
                    f"Comp. opts  : {obj.compression_opts}",
                    f"Shuffle     : {obj.shuffle}",
                    f"Fletcher32  : {obj.fletcher32}",
                    f"Scaleoffset : {obj.scaleoffset}",
                    f"Value unit  : {_tab12_get_unit(obj)}",
                    f"Axis unit   : {_tab12_axis_unit_for_dataset(obj)}",
                    "",
                    "ATTRIBUTES:"
                ]
                lines += (
                    [f"  {k}: {v}" for k,v in attrs.items()]
                    if attrs else ["  (none)"]
                )

                try:
                    if obj.size <= 2_000_000:
                        stats = _tab12_numeric_stats(obj[()])
                    else:
                        stats = {
                            "shape": list(obj.shape),
                            "dtype": str(obj.dtype),
                            "size": int(obj.size),
                            "note": "Large dataset; bounded preview used."
                        }
                except Exception as exc:
                    stats = {"error": f"{type(exc).__name__}: {exc}"}

                lines += ["", "STATISTICS:", json.dumps(stats, indent=2, default=str)]
                self.tab12_meta_text.insert("end", "\n".join(lines))

                try:
                    preview = _tab12_bounded_preview(obj)
                    self.tab12_preview_text.insert(
                        "end", json.dumps(preview, indent=2, default=str)
                    )
                except Exception as exc:
                    self.tab12_preview_text.insert(
                        "end", f"Preview error: {type(exc).__name__}: {exc}"
                    )

                # Configure slice controls for 3D+ datasets.
                self.tab12_slice_axis_var.set(0)
                self.tab12_slice_index_var.set(0)
                self.tab12_slice_axis_spin.configure(
                    from_=0, to=max(0, obj.ndim - 1)
                )
                self.tab12_slice_index_spin.configure(
                    from_=0,
                    to=max(0, obj.shape[0] - 1) if obj.ndim else 0
                )
                return
    except Exception as exc:
        self.tab12_meta_text.insert(
            "end", f"ERROR: {type(exc).__name__}: {exc}"
        )


def _tab12_tree_select(self,event=None):
    selection=self.tab12_tree.selection()
    if not selection:return
    path=self.tab12_tree_map.get(selection[0])
    if not path:return
    _tab12_show_object(self,path)
    if getattr(self,"tab12_auto_plot_var",None) is not None:
        try:
            if self.tab12_auto_plot_var.get():
                with h5py.File(self.tab12_file_path,"r") as f:
                    if isinstance(f[path],h5py.Dataset):
                        _tab12_plot_selected(self)
        except Exception as exc:
            self.tab12_status_var.set(
                f"Auto-plot warning: {type(exc).__name__}: {exc}"
            )



def _tab12_open_file(self):
    fp = filedialog.askopenfilename(
        parent=self.root,
        title="Select ONE HDF5 file",
        filetypes=[("HDF5 files", "*.h5 *.hdf5"), ("All files", "*.*")]
    )
    if not fp:
        return
    self.tab12_load_hdf5(Path(fp))


def _tab12_load_hdf5(self, path):
    path = Path(path)
    try:
        with h5py.File(path, "r") as f:
            _ = list(f.keys())
        self.tab12_file_path = path
        self.tab12_current_path = None
        self.tab12_frc_entries = []
        _tab12_populate_tree(self)
        _tab12_scan_frc(self)
        _tab12_render_frc(self)
        self.tab12_status_var.set(
            f"Loaded: {path.name} | FRC entries: {len(self.tab12_frc_entries)}"
        )
        try:
            self.log(f"Tab 12 HDF5 loaded: {path}")
        except Exception:
            pass
    except Exception as exc:
        self.tab12_status_var.set(
            f"Could not open HDF5: {type(exc).__name__}: {exc}"
        )
        messagebox.showerror(
            "HDF5 Error",
            f"Could not open HDF5 file:\n\n{type(exc).__name__}: {exc}",
            parent=self.root
        )


def _tab12_refresh(self):
    if self.tab12_file_path:
        _tab12_load_hdf5(self, self.tab12_file_path)


def _tab12_plot_selected(self):
    path = getattr(self, "tab12_current_path", None)
    if not path or not self.tab12_file_path:
        return
    try:
        arr = _tab12_load_array(self, path)
        if arr is None:
            return
        data, component = _tab12_component_data(self, arr)
        data, slice_note = _tab12_slice_for_display(self, data)

        fig = plt.Figure(figsize=(8.5, 6.8), dpi=100, facecolor="white")
        if np.asarray(data).ndim == 0:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, str(np.asarray(data).item()),
                    ha="center", va="center", fontsize=14)
            ax.set_axis_off()
        elif np.asarray(data).ndim == 1:
            ax = fig.add_subplot(111)
            ax.plot(np.arange(data.size), data)
            ax.set_xlabel(_tab12_format_unit_label("Index", self.tab12_current_axis_unit))
            ax.set_ylabel(_tab12_format_unit_label(component, self.tab12_current_unit))
            ax.set_title(f"{path} | {component} | {slice_note}")
            ax.grid(True, alpha=0.2)
        elif np.asarray(data).ndim == 2:
            ax = fig.add_subplot(111)
            im = ax.imshow(data, origin="lower", aspect="auto")
            ax.set_xlabel(_tab12_format_unit_label("X", self.tab12_current_axis_unit))
            ax.set_ylabel(_tab12_format_unit_label("Y", self.tab12_current_axis_unit))
            ax.set_title(f"{path} | {component} | {slice_note}")
            fig.colorbar(im, ax=ax, pad=0.02)
        else:
            ax = fig.add_subplot(111)
            ax.plot(np.asarray(data).reshape(-1))
            ax.set_title(f"{path} | {component} | flattened")
        _tab12_show_figure(self, fig)
    except Exception as exc:
        messagebox.showerror(
            "Dataset Plot Error",
            f"{type(exc).__name__}: {exc}",
            parent=self.root
        )


def _tab12_plot_histogram(self):
    path = getattr(self, "tab12_current_path", None)
    if not path or not self.tab12_file_path:
        return
    try:
        arr = np.asarray(_tab12_load_array(self, path))
        data, component = _tab12_component_data(self, arr)
        vals = np.asarray(data).reshape(-1)
        if np.iscomplexobj(vals):
            vals = np.abs(vals)
        if vals.dtype == bool:
            vals = vals.astype(float)
        if vals.dtype.kind not in "uif":
            messagebox.showinfo(
                "Histogram",
                "Histogram is available for numeric datasets.",
                parent=self.root
            )
            return
        if vals.size > 5_000_000:
            vals = vals[:5_000_000]
        vals = vals[np.isfinite(vals)]
        fig = plt.Figure(figsize=(8.5, 6.8), dpi=100, facecolor="white")
        ax = fig.add_subplot(111)
        if vals.size:
            ax.hist(vals, bins=100)
        ax.set_title(f"Histogram — {path}")
        ax.set_xlabel(_tab12_format_unit_label("Value", self.tab12_current_unit))
        ax.set_ylabel("Count")
        ax.grid(True, alpha=0.2)
        _tab12_show_figure(self, fig)
    except Exception as exc:
        messagebox.showerror(
            "Histogram Error", f"{type(exc).__name__}: {exc}",
            parent=self.root
        )


def _tab12_scan_frc(self):
    self.tab12_frc_entries = []
    if not self.tab12_file_path:
        return

    recognized = {
        "frc_smooth", "frc_raw", "frc",
        "FRC_smooth", "FRC_raw", "FRC"
    }

    try:
        with h5py.File(self.tab12_file_path, "r") as f:
            def visitor(name, obj):
                base = name.rsplit("/", 1)[-1]
                if isinstance(obj, h5py.Dataset) and base in recognized:
                    try:
                        raw = np.asarray(obj[()])
                    except Exception:
                        return
                    entry = _tab12_extract_frc_entry(self, f, name, obj, raw)
                    if entry is not None:
                        self.tab12_frc_entries.append(entry)
            f.visititems(visitor)

        # Also support cases where FRC metadata lives on the parent group.
        self.tab12_frc_entries.sort(
            key=lambda e: (
                np.nan_to_num(e.get("magnet_B", np.nan), nan=1e30),
                e.get("path", "")
            )
        )
    except Exception as exc:
        self.tab12_status_var.set(
            f"FRC scan error: {type(exc).__name__}: {exc}"
        )


def _tab12_attr_numeric(obj, keys):
    for key in keys:
        try:
            if key in obj.attrs:
                v = obj.attrs[key]
                if isinstance(v, np.ndarray):
                    if v.size != 1:
                        continue
                    v = v.reshape(-1)[0]
                if isinstance(v, bytes):
                    v = v.decode("utf-8", errors="replace")
                return float(v)
        except Exception:
            pass
    return np.nan


def _tab12_extract_frc_entry(self, f, name, ds, raw):
    if raw.size == 0:
        return None
    # Accept standard Nx2 and 1D FRC formats. Preserve exact values.
    arr = np.asarray(raw)
    if arr.ndim == 1:
        q = np.arange(arr.size, dtype=float)
        frc = np.asarray(arr)
    elif arr.ndim == 2 and arr.shape[1] >= 2:
        q = np.asarray(arr[:, 0], dtype=float)
        frc = np.asarray(arr[:, 1])
    elif arr.ndim == 2 and arr.shape[0] >= 2:
        q = np.asarray(arr[0, :], dtype=float)
        frc = np.asarray(arr[1, :])
    else:
        flat = arr.reshape(-1)
        q = np.arange(flat.size, dtype=float)
        frc = flat

    parent = f[name.rsplit("/", 1)[0]] if "/" in name else f["/"]

    magnet_B = _tab12_attr_numeric(
        ds, ("magnet_B", "magnetic_field", "field", "B", "Magnet_B")
    )
    if not np.isfinite(magnet_B):
        magnet_B = _tab12_attr_numeric(
            parent, ("magnet_B", "magnetic_field", "field", "B", "Magnet_B")
        )

    resolution = _tab12_attr_numeric(
        ds, ("resolution", "Resolution", "resolution_nm", "resolution_nm_px")
    )
    if not np.isfinite(resolution):
        resolution = _tab12_attr_numeric(
            parent, ("resolution", "Resolution", "resolution_nm", "resolution_nm_px")
        )

    q_cut = _tab12_attr_numeric(ds, ("q_cut", "qcut", "Q_cut"))
    if not np.isfinite(q_cut):
        q_cut = _tab12_attr_numeric(parent, ("q_cut", "qcut", "Q_cut"))

    fft_cut = _tab12_attr_numeric(
        ds, ("FFT_pixel_cut", "fft_pixel_cut", "FFT_pixel", "fft_cut")
    )
    if not np.isfinite(fft_cut):
        fft_cut = _tab12_attr_numeric(
            parent, ("FFT_pixel_cut", "fft_pixel_cut", "FFT_pixel", "fft_cut")
        )

    return {
        "name": name.rsplit("/",1)[-1],
        "path": "/" + name.strip("/"),
        "shape": list(arr.shape),
        "dtype": str(arr.dtype),
        "q": np.asarray(q),
        "frc": np.asarray(frc),
        "magnet_B": magnet_B,
        "resolution": resolution,
        "q_cut": q_cut,
        "FFT_pixel_cut": fft_cut,
        "dataset_attrs": _tab12_read_attrs(ds),
        "group_attrs": _tab12_read_attrs(parent),
    }


def _tab12_render_frc(self):
    fig = getattr(self, "tab12_frc_fig", None)
    if fig is None:
        return

    self.tab12_frc_ax1.clear()
    self.tab12_frc_ax2.clear()

    entries = getattr(self, "tab12_frc_entries", []) or []
    for e in entries:
        q = np.asarray(e["q"], dtype=float)
        frc = np.asarray(e["frc"])
        if np.iscomplexobj(frc):
            frc = np.real(frc)
        try:
            self.tab12_frc_ax1.plot(
                q, np.asarray(frc, dtype=float),
                linewidth=1.5,
                label=e["name"] + " | " + e["path"]
            )
        except Exception:
            pass

    self.tab12_frc_ax1.set_xlabel("q / Pixel")
    self.tab12_frc_ax1.set_ylabel("FRC")
    self.tab12_frc_ax1.set_title("FRC Curves")
    self.tab12_frc_ax1.grid(True, alpha=0.22)
    if entries:
        self.tab12_frc_ax1.legend(fontsize=7, loc="best")

    bx = []
    by = []
    labels = []
    for e in entries:
        if np.isfinite(e["magnet_B"]) and np.isfinite(e["resolution"]):
            bx.append(e["magnet_B"])
            by.append(e["resolution"])
            labels.append(e["name"])

    if bx:
        order = np.argsort(np.asarray(bx))
        bx = np.asarray(bx)[order]
        by = np.asarray(by)[order]
        self.tab12_frc_ax2.plot(
            bx, by, marker="o", linewidth=1.6
        )
        for x, y, lab in zip(bx, by, np.asarray(labels, dtype=object)[order]):
            self.tab12_frc_ax2.annotate(
                lab, (x, y),
                xytext=(4, 4), textcoords="offset points", fontsize=7
            )

    self.tab12_frc_ax2.set_xlabel("Magnetic Field (mT)")
    self.tab12_frc_ax2.set_ylabel("Resolution")
    self.tab12_frc_ax2.set_title("Resolution vs Magnetic Field")
    self.tab12_frc_ax2.grid(True, alpha=0.22)
    self.tab12_frc_canvas.draw_idle()

    # Refresh summary table.
    for iid in self.tab12_frc_tree.get_children():
        self.tab12_frc_tree.delete(iid)
    for e in entries:
        fmt = lambda v: "" if not np.isfinite(v) else f"{v:.6g}"
        self.tab12_frc_tree.insert(
            "", "end",
            values=(
                e["name"], e["path"], str(tuple(e["shape"])),
                fmt(e["magnet_B"]), fmt(e["resolution"]),
                fmt(e["q_cut"]), fmt(e["FFT_pixel_cut"])
            )
        )


def _tab12_open_complete_frc_info(self):
    if not self.tab12_file_path:
        messagebox.showwarning(
            "FRC", "Open an HDF5 file first.", parent=self.root
        )
        return

    win = tk.Toplevel(self.root)
    win.title("Complete HDF5 / FRC Information")
    win.geometry("1200x800")

    text = tk.Text(win, wrap="none")
    sy = ttk.Scrollbar(win, orient="vertical", command=text.yview)
    sx = ttk.Scrollbar(win, orient="horizontal", command=text.xview)
    text.configure(yscrollcommand=sy.set, xscrollcommand=sx.set)
    text.grid(row=0, column=0, sticky="nsew")
    sy.grid(row=0, column=1, sticky="ns")
    sx.grid(row=1, column=0, sticky="ew")
    win.grid_rowconfigure(0, weight=1)
    win.grid_columnconfigure(0, weight=1)

    lines = []
    try:
        with h5py.File(self.tab12_file_path, "r") as f:
            lines += ["FILE:", str(self.tab12_file_path), "", "ROOT ATTRIBUTES:"]
            for k,v in _tab12_read_attrs(f).items():
                lines.append(f"  {k}: {v}")
            lines += ["", "HDF5 HIERARCHY:"]
            def visitor(name, obj):
                prefix = "GROUP" if isinstance(obj, h5py.Group) else "DATASET"
                lines.append(f"{prefix}: /{name}")
                if isinstance(obj, h5py.Dataset):
                    lines.append(f"  shape={obj.shape} dtype={obj.dtype} size={obj.size}")
                    lines.append(f"  value_unit={_tab12_get_unit(obj)} axis_unit={_tab12_axis_unit_for_dataset(obj)}")
                attrs = _tab12_read_attrs(obj)
                for k,v in attrs.items():
                    lines.append(f"  attr[{k}] = {v}")
            f.visititems(visitor)

            lines += ["", "RECOGNIZED FRC ENTRIES:"]
            for e in self.tab12_frc_entries:
                lines.append(
                    f"{e['name']} | {e['path']} | shape={tuple(e['shape'])} | "
                    f"magnet_B={e['magnet_B']} | resolution={e['resolution']} | "
                    f"q_cut={e['q_cut']} | FFT_pixel_cut={e['FFT_pixel_cut']}"
                )
                lines.append(
                    f"  dataset_attrs={json.dumps(e['dataset_attrs'], default=str)}"
                )
                lines.append(
                    f"  group_attrs={json.dumps(e['group_attrs'], default=str)}"
                )

    except Exception as exc:
        lines.append(f"ERROR: {type(exc).__name__}: {exc}")

    text.insert("end", "\n".join(lines))
    text.see("1.0")


def _tab12_export_report(self):
    if not self.tab12_file_path:
        messagebox.showwarning(
            "Export", "Open an HDF5 file first.", parent=self.root
        )
        return

    path = filedialog.asksaveasfilename(
        parent=self.root,
        title="Export complete HDF5/FRC report",
        defaultextension=".json",
        filetypes=[("JSON", "*.json"), ("All files", "*.*")],
        initialfile="HDF5_FRC_Complete_Report.json"
    )
    if not path:
        return

    report = {
        "file": {
            "path": str(self.tab12_file_path.resolve()),
            "name": self.tab12_file_path.name,
            "size_bytes": self.tab12_file_path.stat().st_size,
        },
        "hdf5_library": h5py.version.hdf5_version,
        "file_attributes": {},
        "groups": [],
        "datasets": [],
        "frc_entries": [],
    }

    try:
        with h5py.File(self.tab12_file_path, "r") as f:
            report["file_attributes"] = _tab12_read_attrs(f)

            def visitor(name, obj):
                if isinstance(obj, h5py.Group):
                    report["groups"].append({
                        "path": obj.name,
                        "attributes": _tab12_read_attrs(obj)
                    })
                elif isinstance(obj, h5py.Dataset):
                    d = {
                        "path": obj.name,
                        "shape": list(obj.shape),
                        "dtype": str(obj.dtype),
                        "ndim": int(obj.ndim),
                        "size": int(obj.size),
                        "chunks": obj.chunks,
                        "compression": obj.compression,
                        "compression_opts": obj.compression_opts,
                        "shuffle": obj.shuffle,
                        "fletcher32": obj.fletcher32,
                        "scaleoffset": obj.scaleoffset,
                        "value_unit": _tab12_get_unit(obj),
                        "axis_unit": _tab12_axis_unit_for_dataset(obj),
                        "attributes": _tab12_read_attrs(obj),
                    }
                    try:
                        if obj.size <= 2_000_000:
                            d["statistics"] = _tab12_numeric_stats(obj[()])
                        else:
                            d["statistics"] = {
                                "note": "Large dataset; bounded preview used."
                            }
                    except Exception as exc:
                        d["statistics_error"] = f"{type(exc).__name__}: {exc}"
                    report["datasets"].append(d)
            f.visititems(visitor)

        for e in self.tab12_frc_entries:
            report["frc_entries"].append({
                "name": e["name"],
                "path": e["path"],
                "shape": e["shape"],
                "dtype": e["dtype"],
                "magnet_B": None if not np.isfinite(e["magnet_B"]) else float(e["magnet_B"]),
                "resolution": None if not np.isfinite(e["resolution"]) else float(e["resolution"]),
                "q_cut": None if not np.isfinite(e["q_cut"]) else float(e["q_cut"]),
                "FFT_pixel_cut": None if not np.isfinite(e["FFT_pixel_cut"]) else float(e["FFT_pixel_cut"]),
                "dataset_attrs": e["dataset_attrs"],
                "group_attrs": e["group_attrs"],
            })

        with open(path, "w", encoding="utf-8") as fh:
            json.dump(report, fh, indent=2, default=str)

        self.tab12_status_var.set(f"✓ Exported complete report: {Path(path).name}")
        self.log(f"Tab 12 complete HDF5/FRC report exported: {path}")
    except Exception as exc:
        messagebox.showerror(
            "Export Error", f"{type(exc).__name__}: {exc}",
            parent=self.root
        )


def _tab12_export_inventory_csv(self):
    if not self.tab12_file_path:
        messagebox.showwarning("Export", "Open an HDF5 file first.", parent=self.root)
        return

    path = filedialog.asksaveasfilename(
        parent=self.root,
        title="Export HDF5 dataset inventory",
        defaultextension=".csv",
        filetypes=[("CSV", "*.csv"), ("All files", "*.*")],
        initialfile="HDF5_Dataset_Inventory.csv"
    )
    if not path:
        return

    rows = []
    try:
        with h5py.File(self.tab12_file_path, "r") as f:
            def visitor(name, obj):
                if isinstance(obj, h5py.Dataset):
                    st = {}
                    try:
                        if obj.size <= 2_000_000:
                            st = _tab12_numeric_stats(obj[()])
                    except Exception:
                        pass
                    n = st.get("numeric", {})
                    r = st.get("real", {})
                    rows.append({
                        "path": obj.name,
                        "shape": str(tuple(obj.shape)),
                        "dtype": str(obj.dtype),
                        "ndim": int(obj.ndim),
                        "elements": int(obj.size),
                        "attributes": len(obj.attrs),
                        "compression": obj.compression,
                        "value_unit": _tab12_get_unit(obj),
                        "axis_unit": _tab12_axis_unit_for_dataset(obj),
                        "min": n.get("min", r.get("min")),
                        "max": n.get("max", r.get("max")),
                        "mean": n.get("mean", r.get("mean")),
                        "std": n.get("std", r.get("std")),
                    })
            f.visititems(visitor)

        pd.DataFrame(rows).to_csv(path, index=False)
        self.tab12_status_var.set(f"✓ Exported dataset inventory: {Path(path).name}")
    except Exception as exc:
        messagebox.showerror(
            "Inventory Export",
            f"{type(exc).__name__}: {exc}",
            parent=self.root
        )




# ============================================================================
# TAB 12 — HDF5 FILE EXPLORER
# ============================================================================

# TAB 12: definitive HDF5-only callbacks. These deliberately do not touch FRC.
def _tab12_open_file(self):
    fp=filedialog.askopenfilename(
        parent=self.root,title="Open HDF5 file",
        filetypes=[("HDF5 files","*.h5 *.hdf5"),("All files","*.*")]
    )
    if fp:_tab12_load_hdf5(self,Path(fp))

def _tab12_load_hdf5(self,path):
    path=Path(path)
    try:
        with h5py.File(path,"r") as f:_=list(f.keys())
        self.tab12_file_path=path
        self.tab12_current_path=None
        self.tab12_current_unit=None
        self.tab12_current_axis_unit=None
        _tab12_populate_tree(self)
        _tab12_clear_plot(self)
        self.tab12_status_var.set(
            f"HDF5 loaded: {path.name} | select a group or dataset."
        )
    except Exception as exc:
        self.tab12_status_var.set(
            f"HDF5 error: {type(exc).__name__}: {exc}"
        )
        messagebox.showerror(
            "HDF5 Error",f"{type(exc).__name__}: {exc}",parent=self.root
        )

def _tab12_refresh(self):
    path=getattr(self,"tab12_file_path",None)
    if path:_tab12_load_hdf5(self,path)
    else:self.tab12_status_var.set("No HDF5 file loaded.")

def _tab12_populate_tree(self):
    self.tab12_tree.delete(*self.tab12_tree.get_children())
    self.tab12_tree_map={}
    if not self.tab12_file_path:return
    try:
        with h5py.File(self.tab12_file_path,"r") as f:
            rid=self.tab12_tree.insert(
                "","end",text="/",
                values=("GROUP",f"{len(f)} items",""),open=True
            )
            self.tab12_tree_map[rid]="/";nodes={"/":rid}

            def ensure_group(path):
                if path in nodes:return nodes[path]
                parts=[x for x in path.strip("/").split("/") if x]
                cur="/";parent=nodes["/"]
                for part in parts:
                    nxt="/"+part if cur=="/" else cur+"/"+part
                    if nxt not in nodes:
                        item=self.tab12_tree.insert(
                            parent,"end",text=part,
                            values=("GROUP","","")
                        )
                        nodes[nxt]=item
                        self.tab12_tree_map[item]=nxt
                    parent=nodes[nxt];cur=nxt
                return parent

            def visitor(name,obj):
                parts=[x for x in name.strip("/").split("/") if x]
                if not parts:return
                path="/"+name.strip("/")
                pp="/"+"/".join(parts[:-1]) if len(parts)>1 else "/"
                parent=ensure_group(pp);leaf=parts[-1]
                if isinstance(obj,h5py.Group):
                    item=ensure_group(path)
                    self.tab12_tree.item(
                        item,text=leaf,
                        values=("GROUP",f"{len(obj)} items","")
                    )
                elif isinstance(obj,h5py.Dataset):
                    item=self.tab12_tree.insert(
                        parent,"end",text=leaf,
                        values=("DATASET",str(obj.shape),str(obj.dtype))
                    )
                    self.tab12_tree_map[item]=path
            f.visititems(visitor)
        self.tab12_status_var.set(
            f"HDF5 tree refreshed: {self.tab12_file_path.name}"
        )
    except Exception as exc:
        self.tab12_status_var.set(
            f"Tree error: {type(exc).__name__}: {exc}"
        )

def _tab12_tree_select(self,event=None):
    """Show metadata/preview and immediately plot any selected DATASET."""
    try:
        selection=self.tab12_tree.selection()
        if not selection:
            return
        path=self.tab12_tree_map.get(selection[0])
        if not path:
            return
        _tab12_show_object(self,path)
        with h5py.File(self.tab12_file_path,"r") as f:
            obj=f[path]
            if isinstance(obj,h5py.Dataset):
                _tab12_plot_selected(self)
                self.tab12_status_var.set(
                    f"Selected dataset plotted: {path}"
                )
            else:
                self.tab12_status_var.set(f"Selected group: {path}")
    except Exception as exc:
        self.tab12_status_var.set(
            f"Dataset selection/plot error: {type(exc).__name__}: {exc}"
        )


def _tab12_load_array(self,path):
    if not self.tab12_file_path:return None
    with h5py.File(self.tab12_file_path,"r") as f:
        obj=f[path]
        if not isinstance(obj,h5py.Dataset):return None
        if obj.size>10_000_000:
            return np.asarray(
                _tab12_bounded_preview(
                    obj,max_side=512,max_elements=250000
                )["data"]
            )
        return obj[()]

def _tab12_clear_plot(self):
    fig=getattr(self,"tab12_h5_fig",None)
    if fig is not None:
        try:plt.close(fig)
        except Exception:pass
    self.tab12_h5_fig=None
    self.tab12_h5_canvas=None
    host=getattr(self,"tab12_plot_host",None)
    if host is not None:
        for w in host.winfo_children():
            try:w.destroy()
            except Exception:pass

def _tab12_show_figure(self,fig):
    _tab12_clear_plot(self)
    self.tab12_h5_fig=fig
    self.tab12_h5_canvas=FigureCanvasTkAgg(fig,master=self.tab12_plot_host)
    self.tab12_h5_canvas.get_tk_widget().pack(fill="both",expand=True)
    self.tab12_h5_canvas.draw()

def _tab12_show_object(self,path):
    if not self.tab12_file_path or not path:return
    self.tab12_current_path=path
    self.tab12_meta_text.delete("1.0","end")
    self.tab12_preview_text.delete("1.0","end")
    try:
        with h5py.File(self.tab12_file_path,"r") as f:
            obj=f[path];attrs=_tab12_read_attrs(obj)
            if isinstance(obj,h5py.Group):
                self.tab12_path_var.set(f"GROUP: {path}")
                lines=[f"GROUP: {path}","","ATTRIBUTES:"]
                lines += [f"  {k}: {v}" for k,v in attrs.items()] or ["  (none)"]
                lines += ["","CHILDREN:"]
                for name in obj.keys():
                    ch=obj[name]
                    lines.append(
                        f"  {'DATASET' if isinstance(ch,h5py.Dataset) else 'GROUP'} "
                        f"{name}" +
                        (f" | shape={ch.shape} | dtype={ch.dtype}"
                         if isinstance(ch,h5py.Dataset) else "")
                    )
                self.tab12_meta_text.insert("end","\n".join(lines))
                self.tab12_preview_text.insert(
                    "end","Select a DATASET to inspect and plot its contents."
                )
                _tab12_clear_plot(self);return

            if isinstance(obj,h5py.Dataset):
                self.tab12_path_var.set(f"DATASET: {path}")
                self.tab12_current_unit=_tab12_get_unit(obj)
                self.tab12_current_axis_unit=_tab12_axis_unit_for_dataset(obj)
                lines=[
                    f"DATASET: {path}",
                    f"Shape       : {obj.shape}",
                    f"Dtype       : {obj.dtype}",
                    f"Dimensions  : {obj.ndim}",
                    f"Elements    : {obj.size:,}",
                    f"Chunks      : {obj.chunks}",
                    f"Compression : {obj.compression}",
                    f"Value unit  : {self.tab12_current_unit or '(not stored)'}",
                    f"Axis unit   : {self.tab12_current_axis_unit or '(not stored)'}",
                    "","ATTRIBUTES:"
                ]
                lines += [f"  {k}: {v}" for k,v in attrs.items()] or ["  (none)"]
                try:
                    stats=_tab12_numeric_stats(obj[()]) if obj.size<=2_000_000 else {
                        "shape":list(obj.shape),"dtype":str(obj.dtype),
                        "size":int(obj.size),"note":"large dataset"
                    }
                except Exception as exc:
                    stats={"error":f"{type(exc).__name__}: {exc}"}
                lines += ["","STATISTICS:",json.dumps(stats,indent=2,default=str)]
                self.tab12_meta_text.insert("end","\n".join(lines))
                try:
                    self.tab12_preview_text.insert(
                        "end",json.dumps(
                            _tab12_bounded_preview(obj),
                            indent=2,default=str
                        )
                    )
                except Exception as exc:
                    self.tab12_preview_text.insert(
                        "end",f"Preview error: {type(exc).__name__}: {exc}"
                    )
                nd=int(obj.ndim)
                self.tab12_slice_axis_spin.configure(from_=0,to=max(0,nd-1))
                if nd:
                    axis=max(0,min(int(self.tab12_slice_axis_var.get()),nd-1))
                    self.tab12_slice_axis_var.set(axis)
                    self.tab12_slice_index_spin.configure(
                        from_=0,to=max(0,int(obj.shape[axis])-1)
                    )
                    self.tab12_slice_index_var.set(
                        max(0,min(
                            int(self.tab12_slice_index_var.get()),
                            max(0,int(obj.shape[axis])-1)
                        ))
                    )
    except Exception as exc:
        self.tab12_meta_text.insert(
            "end",f"ERROR: {type(exc).__name__}: {exc}"
        )

def _tab12_plot_selected(self):
    path=getattr(self,"tab12_current_path",None)
    if not path or not self.tab12_file_path:
        self.tab12_status_var.set("Select a dataset first.");return
    try:
        arr=_tab12_load_array(self,path)
        if arr is None:return
        data,component=_tab12_component_data(self,arr)
        data,slice_note=_tab12_slice_for_display(self,data)
        data=np.asarray(data)
        fig=plt.Figure(figsize=(10,7),dpi=100,facecolor="white")

        if data.ndim==0:
            ax=fig.add_subplot(111)
            ax.text(.5,.5,str(data.item()),ha="center",va="center")
            ax.set_axis_off()
        elif data.ndim==1:
            ax=fig.add_subplot(111)
            ax.plot(np.arange(data.size),data)
            ax.set_xlabel(
                _tab12_format_unit_label("Index",self.tab12_current_axis_unit)
            )
            ax.set_ylabel(
                _tab12_format_unit_label(component,self.tab12_current_unit)
            )
            ax.set_title(f"{path} | {component} | {slice_note}",pad=10)
            ax.grid(True,alpha=.2)
        elif data.ndim==2:
            ax=fig.add_subplot(111)
            im=ax.imshow(
                data,origin="lower",aspect="auto",interpolation="nearest"
            )
            ax.set_xlabel(
                _tab12_format_unit_label("X",self.tab12_current_axis_unit)
            )
            ax.set_ylabel(
                _tab12_format_unit_label("Y",self.tab12_current_axis_unit)
            )
            ax.set_title(f"{path} | {component} | {slice_note}",pad=10)
            fig.colorbar(im,ax=ax,pad=.02)
        else:
            ax=fig.add_subplot(111)
            ax.plot(np.asarray(data).reshape(-1))
            ax.set_xlabel("Flattened index")
            ax.set_ylabel(component)
            ax.set_title(f"{path} | {component} | flattened",pad=10)
        fig.tight_layout()
        _tab12_show_figure(self,fig)
        self.tab12_status_var.set(
            f"Plotted: {path} | {component} | {slice_note}"
        )
    except Exception as exc:
        self.tab12_status_var.set(
            f"Plot error: {type(exc).__name__}: {exc}"
        )
        messagebox.showerror(
            "Dataset Plot Error",f"{type(exc).__name__}: {exc}",
            parent=self.root
        )

def _tab12_plot_histogram(self):
    path=getattr(self,"tab12_current_path",None)
    if not path or not self.tab12_file_path:
        self.tab12_status_var.set("Select a dataset first.");return
    try:
        arr=np.asarray(_tab12_load_array(self,path))
        data,component=_tab12_component_data(self,arr)
        vals=np.asarray(data).reshape(-1)
        if np.iscomplexobj(vals):vals=np.abs(vals)
        if vals.dtype==bool:vals=vals.astype(float)
        if vals.dtype.kind not in "uif":
            messagebox.showinfo(
                "Histogram","Histogram is available for numeric datasets.",
                parent=self.root
            );return
        vals=vals[:5_000_000]
        vals=vals[np.isfinite(vals)]
        fig=plt.Figure(figsize=(10,7),dpi=100,facecolor="white")
        ax=fig.add_subplot(111)
        if vals.size:ax.hist(vals,bins=100)
        ax.set_title(f"Histogram — {path}",pad=10)
        ax.set_xlabel(
            _tab12_format_unit_label("Value",self.tab12_current_unit)
        )
        ax.set_ylabel("Count");ax.grid(True,alpha=.2)
        fig.tight_layout();_tab12_show_figure(self,fig)
    except Exception as exc:
        self.tab12_status_var.set(
            f"Histogram error: {type(exc).__name__}: {exc}"
        )



def _tab12_build(self):
    if hasattr(self,"tab_hdf5_12"):
        return

    self.tab_hdf5_12=ttk.Frame(self.nb)
    self.nb.add(self.tab_hdf5_12,text="12. HDF5")

    self.tab12_file_path=None
    self.tab12_tree_map={}
    self.tab12_current_path=None
    self.tab12_current_unit=None
    self.tab12_current_axis_unit=None
    self.tab12_h5_fig=None
    self.tab12_h5_canvas=None
    self.tab12_component_var=tk.StringVar(value="Auto")
    self.tab12_slice_axis_var=tk.IntVar(value=0)
    self.tab12_slice_index_var=tk.IntVar(value=0)
    self.tab12_status_var=tk.StringVar(
        value="Ready — OPEN / BROWSE HDF5."
    )
    self.tab12_path_var=tk.StringVar(value="")

    self.tab_hdf5_12.grid_rowconfigure(1,weight=1)
    self.tab_hdf5_12.grid_columnconfigure(0,weight=1)

    toolbar=ttk.LabelFrame(
        self.tab_hdf5_12,text="HDF5 FILE CONTROLS",padding=5
    )
    toolbar.grid(row=0,column=0,sticky="ew",padx=6,pady=(5,3))
    toolbar.grid_columnconfigure(0,weight=1)
    toolbar.grid_columnconfigure(1,weight=1)

    ttk.Button(
        toolbar,text="OPEN / BROWSE HDF5",
        command=lambda:_tab12_open_file(self)
    ).grid(row=0,column=0,sticky="ew",padx=2)
    ttk.Button(
        toolbar,text="REFRESH HDF5",
        command=lambda:_tab12_refresh(self)
    ).grid(row=0,column=1,sticky="ew",padx=2)
    ttk.Label(
        toolbar,textvariable=self.tab12_status_var,anchor="w"
    ).grid(row=1,column=0,columnspan=2,sticky="ew",padx=3,pady=(3,0))

    pane=ttk.Panedwindow(self.tab_hdf5_12,orient="horizontal")
    pane.grid(row=1,column=0,sticky="nsew",padx=6,pady=(2,6))

    left=ttk.LabelFrame(pane,text="HDF5 HIERARCHY",padding=5)
    pane.add(left,weight=3)
    left.grid_rowconfigure(0,weight=1);left.grid_columnconfigure(0,weight=1)

    self.tab12_tree=ttk.Treeview(
        left,columns=("type","shape","dtype"),show="tree headings"
    )
    for col,title in (("#0","Path / Name"),("type","Type"),("shape","Shape"),("dtype","Dtype")):
        self.tab12_tree.heading(col,text=title)
    self.tab12_tree.column("#0",width=300,minwidth=200)
    self.tab12_tree.column("type",width=75,minwidth=55)
    self.tab12_tree.column("shape",width=150,minwidth=100)
    self.tab12_tree.column("dtype",width=120,minwidth=90)
    ty=ttk.Scrollbar(left,orient="vertical",command=self.tab12_tree.yview)
    tx=ttk.Scrollbar(left,orient="horizontal",command=self.tab12_tree.xview)
    self.tab12_tree.configure(yscrollcommand=ty.set,xscrollcommand=tx.set)
    self.tab12_tree.grid(row=0,column=0,sticky="nsew")
    ty.grid(row=0,column=1,sticky="ns");tx.grid(row=1,column=0,sticky="ew")
    self.tab12_tree.bind(
        "<<TreeviewSelect>>",lambda e:_tab12_tree_select(self,e)
    )

    mid=ttk.LabelFrame(pane,text="METADATA / PREVIEW",padding=5)
    pane.add(mid,weight=3)
    mid.grid_rowconfigure(1,weight=1);mid.grid_rowconfigure(4,weight=1);mid.grid_columnconfigure(0,weight=1)

    ttk.Label(mid,textvariable=self.tab12_path_var,wraplength=420,anchor="w").grid(
        row=0,column=0,sticky="ew",pady=(0,4)
    )
    mf=ttk.Frame(mid);mf.grid(row=1,column=0,sticky="nsew")
    mf.grid_rowconfigure(0,weight=1);mf.grid_columnconfigure(0,weight=1)
    self.tab12_meta_text=tk.Text(mf,wrap="word")
    my=ttk.Scrollbar(mf,orient="vertical",command=self.tab12_meta_text.yview)
    self.tab12_meta_text.configure(yscrollcommand=my.set)
    self.tab12_meta_text.grid(row=0,column=0,sticky="nsew");my.grid(row=0,column=1,sticky="ns")

    ttk.Separator(mid).grid(row=2,column=0,sticky="ew",pady=5)
    ttk.Label(mid,text="DATA PREVIEW",font=("Arial",10,"bold")).grid(row=3,column=0,sticky="nw")

    pf=ttk.Frame(mid);pf.grid(row=4,column=0,sticky="nsew")
    pf.grid_rowconfigure(0,weight=1);pf.grid_columnconfigure(0,weight=1)
    self.tab12_preview_text=tk.Text(pf,wrap="none")
    py=ttk.Scrollbar(pf,orient="vertical",command=self.tab12_preview_text.yview)
    px=ttk.Scrollbar(pf,orient="horizontal",command=self.tab12_preview_text.xview)
    self.tab12_preview_text.configure(yscrollcommand=py.set,xscrollcommand=px.set)
    self.tab12_preview_text.grid(row=0,column=0,sticky="nsew");py.grid(row=0,column=1,sticky="ns");px.grid(row=1,column=0,sticky="ew")

    right=ttk.LabelFrame(pane,text="DATASET PLOTTER",padding=5)
    pane.add(right,weight=5)
    right.grid_rowconfigure(2,weight=1);right.grid_columnconfigure(0,weight=1)

    # Two compact rows; no Auto Plot checkbox. Selecting a dataset in the
    # hierarchy is the auto-plot trigger.
    r1=ttk.Frame(right);r1.grid(row=0,column=0,sticky="ew",pady=(0,3))
    ttk.Label(r1,text="COMPONENT").pack(side="left",padx=3)
    self.tab12_component_cb=ttk.Combobox(
        r1,textvariable=self.tab12_component_var,
        values=("Auto","Raw","Real","Imaginary","Amplitude","Phase"),
        state="readonly",width=12
    )
    self.tab12_component_cb.pack(side="left",padx=3)
    self.tab12_component_cb.bind(
        "<<ComboboxSelected>>",lambda e:_tab12_plot_selected(self)
    )
    ttk.Label(r1,text="Slice axis").pack(side="left",padx=(10,2))
    self.tab12_slice_axis_spin=tk.Spinbox(
        r1,textvariable=self.tab12_slice_axis_var,
        from_=0,to=0,width=5
    )
    self.tab12_slice_axis_spin.pack(side="left",padx=2)
    ttk.Label(r1,text="Slice index").pack(side="left",padx=(10,2))
    self.tab12_slice_index_spin=tk.Spinbox(
        r1,textvariable=self.tab12_slice_index_var,
        from_=0,to=0,width=7
    )
    self.tab12_slice_index_spin.pack(side="left",padx=2)

    r2=ttk.Frame(right);r2.grid(row=1,column=0,sticky="ew",pady=(0,5))
    actions=[
        ("PLOT",lambda:_tab12_plot_selected(self)),
        ("HISTOGRAM",lambda:_tab12_plot_histogram(self)),
        ("CLEAR",lambda:_tab12_clear_plot(self)),
        ("OPEN HDF5 INFO",lambda:_tab12_open_complete_frc_info(self)),
        ("EXPORT REPORT JSON",lambda:_tab12_export_report(self)),
        ("EXPORT DATASET CSV",lambda:_tab12_export_inventory_csv(self)),
    ]
    for j,(txt,cmd) in enumerate(actions):
        r2.grid_columnconfigure(j,weight=1)
        ttk.Button(r2,text=txt,command=cmd).grid(
            row=0,column=j,sticky="ew",padx=2
        )

    wrap=tk.Frame(right)
    wrap.grid(row=2,column=0,sticky="nsew")
    wrap.grid_rowconfigure(0,weight=1);wrap.grid_columnconfigure(0,weight=1)
    pc=tk.Canvas(wrap,highlightthickness=0,borderwidth=0)
    pv=ttk.Scrollbar(wrap,orient="vertical",command=pc.yview)
    ph=ttk.Scrollbar(wrap,orient="horizontal",command=pc.xview)
    pc.configure(yscrollcommand=pv.set,xscrollcommand=ph.set)
    pc.grid(row=0,column=0,sticky="nsew");pv.grid(row=0,column=1,sticky="ns");ph.grid(row=1,column=0,sticky="ew")
    host=ttk.Frame(pc,width=1100,height=760)
    pc.create_window((0,0),window=host,anchor="nw")
    host.bind("<Configure>",lambda e:pc.configure(scrollregion=pc.bbox("all")))
    self.tab12_plot_host=host


# ============================================================================
# TAB 13 — FRC CALLBACKS (verified complete chain)
# ============================================================================
def _tab13_open_frc_file(self):
    fp=filedialog.askopenfilename(
        parent=self.root,
        title="Open FRC HDF5 file",
        filetypes=[("HDF5 files","*.h5 *.hdf5"),("All files","*.*")]
    )
    if fp:
        _tab13_load_frc(self,Path(fp))



def _tab13_load_frc(self,path):
    path=Path(path)
    try:
        if not path.exists():
            raise FileNotFoundError(str(path))
        with h5py.File(path,"r") as f:
            _=list(f.keys())
        self.tab13_frc_file_path=path
        _tab13_scan_frc_entries(self)
        _tab13_render_frc(self)
        count=len(getattr(self,"tab13_frc_entries",[]) or [])
        self.tab13_status_var.set(
            f"FRC loaded: {path.name} | {count} recognized entries"
        )
        try:self.log(f"Tab 13 FRC loaded: {path}")
        except Exception:pass
    except Exception as exc:
        self.tab13_status_var.set(
            f"FRC open error: {type(exc).__name__}: {exc}"
        )
        messagebox.showerror(
            "FRC HDF5 Error",
            f"Could not open FRC HDF5 file:\n\n{type(exc).__name__}: {exc}",
            parent=self.root
        )



def _tab13_refresh(self):
    path=getattr(self,"tab13_frc_file_path",None)
    if path is None:
        self.tab13_status_var.set("No FRC HDF5 file loaded.")
        return
    _tab13_load_frc(self,Path(path))



def _tab13_attr_numeric(obj,keys):
    for key in keys:
        try:
            if key not in obj.attrs:continue
            value=obj.attrs[key]
            if isinstance(value,np.ndarray):
                if value.size!=1:continue
                value=value.reshape(-1)[0]
            if isinstance(value,bytes):
                value=value.decode("utf-8",errors="replace")
            return float(value)
        except Exception:
            pass
    return np.nan


def _tab13_scan_frc_entries(self):
    self.tab13_frc_entries=[]
    path=getattr(self,"tab13_frc_file_path",None)
    if not path:return

    recognized={
        "frc_smooth","frc_raw","frc",
        "FRC_smooth","FRC_raw","FRC"
    }
    try:
        with h5py.File(path,"r") as f:
            def visitor(name,obj):
                base=name.rsplit("/",1)[-1]
                if not isinstance(obj,h5py.Dataset) or base not in recognized:
                    return
                try:raw=np.asarray(obj[()])
                except Exception:return

                arr=np.asarray(raw)
                if arr.size==0:return
                if arr.ndim==1:
                    q=np.arange(arr.size,dtype=float)
                    frc=arr
                elif arr.ndim==2 and arr.shape[1]>=2:
                    q=np.asarray(arr[:,0],dtype=float)
                    frc=arr[:,1]
                elif arr.ndim==2 and arr.shape[0]>=2:
                    q=np.asarray(arr[0,:],dtype=float)
                    frc=arr[1,:]
                else:
                    flat=arr.reshape(-1)
                    q=np.arange(flat.size,dtype=float)
                    frc=flat

                parent_path=name.rsplit("/",1)[0] if "/" in name else "/"
                parent=f[parent_path]

                fb=("magnet_B","Magnet_B","magnetic_field","field","B")
                rk=("resolution","Resolution","resolution_nm","resolution_nm_px")
                qk=("q_cut","qcut","Q_cut")
                fk=("FFT_pixel_cut","fft_pixel_cut","FFT_pixel","fft_cut")

                B=_tab13_attr_numeric(obj,fb)
                if not np.isfinite(B):B=_tab13_attr_numeric(parent,fb)
                R=_tab13_attr_numeric(obj,rk)
                if not np.isfinite(R):R=_tab13_attr_numeric(parent,rk)
                QC=_tab13_attr_numeric(obj,qk)
                if not np.isfinite(QC):QC=_tab13_attr_numeric(parent,qk)
                FC=_tab13_attr_numeric(obj,fk)
                if not np.isfinite(FC):FC=_tab13_attr_numeric(parent,fk)

                self.tab13_frc_entries.append({
                    "name":base,
                    "path":"/"+name.strip("/"),
                    "shape":list(arr.shape),
                    "dtype":str(arr.dtype),
                    "q":np.asarray(q),
                    "frc":np.asarray(frc),
                    "magnet_B":B,
                    "resolution":R,
                    "q_cut":QC,
                    "FFT_pixel_cut":FC,
                    "dataset_attrs":_tab12_read_attrs(obj),
                    "group_attrs":_tab12_read_attrs(parent)
                })
            f.visititems(visitor)

        self.tab13_frc_entries.sort(
            key=lambda e:(
                np.nan_to_num(e["magnet_B"],nan=1e30),
                e["path"]
            )
        )
    except Exception as exc:
        self.tab13_status_var.set(
            f"FRC scan error: {type(exc).__name__}: {exc}"
        )


def _tab13_render_frc(self):
    if getattr(self,"tab13_frc_fig",None) is None:return
    self.tab13_frc_ax1.clear()
    self.tab13_frc_ax2.clear()

    entries=getattr(self,"tab13_frc_entries",[]) or []
    for e in entries:
        try:
            y=np.asarray(e["frc"])
            if np.iscomplexobj(y):y=np.real(y)
            self.tab13_frc_ax1.plot(
                np.asarray(e["q"],dtype=float),
                np.asarray(y,dtype=float),
                linewidth=1.3,
                label=e["name"]+" | "+e["path"]
            )
        except Exception:
            pass

    self.tab13_frc_ax1.set_title("FRC Curves",pad=12)
    self.tab13_frc_ax1.set_xlabel("q / Pixel",labelpad=9)
    self.tab13_frc_ax1.set_ylabel("FRC",labelpad=9)
    self.tab13_frc_ax1.grid(True,alpha=.2)
    if entries:
        self.tab13_frc_ax1.legend(fontsize=6,loc="best")

    bx=[];by=[];labels=[]
    for e in entries:
        if np.isfinite(e["magnet_B"]) and np.isfinite(e["resolution"]):
            bx.append(e["magnet_B"])
            by.append(e["resolution"])
            labels.append(e["name"])
    if bx:
        order=np.argsort(np.asarray(bx))
        bx=np.asarray(bx)[order]
        by=np.asarray(by)[order]
        labels=np.asarray(labels,dtype=object)[order]
        self.tab13_frc_ax2.plot(
            bx,by,marker="o",linewidth=1.5
        )
        for x,y,lab in zip(bx,by,labels):
            self.tab13_frc_ax2.annotate(
                str(lab),(x,y),
                xytext=(3,3),
                textcoords="offset points",
                fontsize=6
            )

    self.tab13_frc_ax2.set_title(
        "Resolution vs Magnetic Field",pad=12
    )
    self.tab13_frc_ax2.set_xlabel(
        "Magnetic Field (mT)",labelpad=9
    )
    self.tab13_frc_ax2.set_ylabel("Resolution",labelpad=9)
    self.tab13_frc_ax2.grid(True,alpha=.2)

    self.tab13_frc_canvas.draw_idle()

    for iid in self.tab13_frc_tree.get_children():
        self.tab13_frc_tree.delete(iid)

    for e in entries:
        fmt=lambda v:"" if not np.isfinite(v) else f"{v:.6g}"
        self.tab13_frc_tree.insert(
            "","end",
            values=(
                e["name"],e["path"],
                str(tuple(e["shape"])),
                fmt(e["magnet_B"]),
                fmt(e["resolution"]),
                fmt(e["q_cut"]),
                fmt(e["FFT_pixel_cut"])
            )
        )


def _tab13_open_complete_frc_info(self):
    """Open and display the complete loaded FRC HDF5 information."""
    path = getattr(self, "tab13_frc_file_path", None)
    if not path:
        messagebox.showwarning(
            "FRC",
            "Open an FRC HDF5 file first.",
            parent=self.root
        )
        return

    path = Path(path)

    # Validate/access the actual source file first.
    try:
        with h5py.File(path, "r") as f:
            _ = list(f.keys())
    except Exception as exc:
        messagebox.showerror(
            "FRC HDF5 Error",
            f"Cannot open the loaded FRC file:\n\n"
            f"{type(exc).__name__}: {exc}",
            parent=self.root
        )
        return

    win = tk.Toplevel(self.root)
    win.title(f"Complete FRC Info — {path.name}")
    win.geometry("1200x820")
    win.minsize(800, 550)

    outer = ttk.Frame(win, padding=6)
    outer.pack(fill="both", expand=True)

    top = ttk.Frame(outer)
    top.pack(fill="x", pady=(0, 5))

    ttk.Label(
        top,
        text=f"FRC FILE: {path}",
        anchor="w"
    ).pack(side="left", fill="x", expand=True)

    ttk.Button(
        top,
        text="REFRESH INFO",
        command=lambda: _tab13_open_complete_frc_info(self)
    ).pack(side="right", padx=(5, 0))

    text_frame = ttk.Frame(outer)
    text_frame.pack(fill="both", expand=True)

    # Wrap text so long HDF5 paths/attributes stay inside the window.
    text = tk.Text(
        text_frame,
        wrap="word",
        font=("Consolas", 10)
    )
    sy = ttk.Scrollbar(
        text_frame,
        orient="vertical",
        command=text.yview
    )
    text.configure(yscrollcommand=sy.set)

    text.grid(row=0, column=0, sticky="nsew")
    sy.grid(row=0, column=1, sticky="ns")

    text_frame.grid_rowconfigure(0, weight=1)
    text_frame.grid_columnconfigure(0, weight=1)

    lines = [
        "============================================================",
        "COMPLETE FRC / HDF5 INFORMATION",
        "============================================================",
        f"FILE: {path}",
        f"FILE NAME: {path.name}",
        f"FILE SIZE (bytes): {path.stat().st_size}",
        "",
        "ROOT ATTRIBUTES:",
    ]

    try:
        with h5py.File(path, "r") as f:
            root_attrs = _tab12_read_attrs(f)
            if root_attrs:
                lines.extend(
                    [f"  {k}: {v}" for k, v in root_attrs.items()]
                )
            else:
                lines.append("  (none)")

            lines += [
                "",
                "FULL HDF5 HIERARCHY:",
            ]

            def visitor(name, obj):
                kind = "GROUP" if isinstance(obj, h5py.Group) else "DATASET"
                lines.append(f"{kind}: /{name}")

                if isinstance(obj, h5py.Dataset):
                    lines.append(
                        f"  shape={obj.shape} | dtype={obj.dtype} | "
                        f"ndim={obj.ndim} | size={obj.size}"
                    )
                    try:
                        lines.append(
                            f"  unit={_tab12_get_unit(obj)} | "
                            f"axis_unit={_tab12_axis_unit_for_dataset(obj)}"
                        )
                    except Exception:
                        pass

                attrs = _tab12_read_attrs(obj)
                if attrs:
                    for k, v in attrs.items():
                        lines.append(f"  attr[{k}] = {v}")
                else:
                    lines.append("  attr: (none)")

            f.visititems(visitor)

            lines += [
                "",
                "RECOGNIZED FRC ENTRIES:",
            ]

            entries = getattr(self, "tab13_frc_entries", []) or []
            if entries:
                for e in entries:
                    lines.append(
                        f"  {e.get('name','')} | {e.get('path','')} | "
                        f"shape={tuple(e.get('shape',()))} | "
                        f"magnet_B={e.get('magnet_B')} | "
                        f"resolution={e.get('resolution')} | "
                        f"q_cut={e.get('q_cut')} | "
                        f"FFT_pixel_cut={e.get('FFT_pixel_cut')}"
                    )
            else:
                lines.append("  (none recognized)")

    except Exception as exc:
        lines.extend([
            "",
            f"ERROR: {type(exc).__name__}: {exc}"
        ])

    text.insert("1.0", "\n".join(lines))
    text.see("1.0")


def _tab13_export_report(self):
    path=getattr(self,'tab13_frc_file_path',None)
    if not path:
        messagebox.showwarning('Export','Open an FRC HDF5 file first.',parent=self.root);return
    outpath=filedialog.asksaveasfilename(
        parent=self.root,title='Export complete FRC report',
        defaultextension='.json',filetypes=[('JSON','*.json'),('All files','*.*')],
        initialfile='FRC_Complete_Report.json')
    if not outpath:return
    report={'file':str(Path(path).resolve()),'root_attributes':{},'groups':[],'datasets':[],'frc_entries':[]}
    try:
        with h5py.File(path,'r') as f:
            report['root_attributes']=_tab12_read_attrs(f)
            def visitor(name,obj):
                if isinstance(obj,h5py.Group):
                    report['groups'].append({'path':obj.name,'attributes':_tab12_read_attrs(obj)})
                elif isinstance(obj,h5py.Dataset):
                    report['datasets'].append({
                        'path':obj.name,'shape':list(obj.shape),'dtype':str(obj.dtype),
                        'ndim':int(obj.ndim),'size':int(obj.size),
                        'attributes':_tab12_read_attrs(obj),
                        'value_unit':_tab12_get_unit(obj),
                        'axis_unit':_tab12_axis_unit_for_dataset(obj)})
            f.visititems(visitor)
        for e in getattr(self,'tab13_frc_entries',[]) or []:
            report['frc_entries'].append({
                'name':e['name'],'path':e['path'],'shape':e['shape'],'dtype':e['dtype'],
                'magnet_B':None if not np.isfinite(e['magnet_B']) else float(e['magnet_B']),
                'resolution':None if not np.isfinite(e['resolution']) else float(e['resolution']),
                'q_cut':None if not np.isfinite(e['q_cut']) else float(e['q_cut']),
                'FFT_pixel_cut':None if not np.isfinite(e['FFT_pixel_cut']) else float(e['FFT_pixel_cut']),
                'dataset_attrs':e['dataset_attrs'],'group_attrs':e['group_attrs']})
        with open(outpath,'w',encoding='utf-8') as fh: json.dump(report,fh,indent=2,default=str)
        self.tab13_status_var.set(f'✓ Exported FRC report: {Path(outpath).name}')
    except Exception as exc:
        messagebox.showerror('FRC Export Error',f'{type(exc).__name__}: {exc}',parent=self.root)


def _tab13_export_summary_csv(self):
    entries=getattr(self,'tab13_frc_entries',[]) or []
    if not entries:
        messagebox.showwarning('Export','No recognized FRC entries to export.',parent=self.root);return
    outpath=filedialog.asksaveasfilename(
        parent=self.root,title='Export FRC summary CSV',
        defaultextension='.csv',filetypes=[('CSV','*.csv'),('All files','*.*')],
        initialfile='FRC_Summary.csv')
    if not outpath:return
    rows=[{
        'Entry':e['name'],'Path':e['path'],'FRC_Shape':str(tuple(e['shape'])),
        'Magnet_B':e['magnet_B'],'Resolution':e['resolution'],
        'q_cut':e['q_cut'],'FFT_pixel_cut':e['FFT_pixel_cut']
    } for e in entries]
    pd.DataFrame(rows).to_csv(outpath,index=False)
    self.tab13_status_var.set(f'✓ Exported FRC summary: {Path(outpath).name}')

def _tab13_export_plot(self):
    """Export the plotted FRC curves and resolution-vs-field data as CSV."""
    entries = getattr(self, "tab13_frc_entries", []) or []
    if not entries:
        messagebox.showwarning(
            "Export Plot",
            "No FRC data are loaded.",
            parent=self.root
        )
        return

    outpath = filedialog.asksaveasfilename(
        parent=self.root,
        title="Export both FRC plots to CSV",
        defaultextension=".csv",
        filetypes=[
            ("CSV files", "*.csv"),
            ("All files", "*.*")
        ],
        initialfile="FRC_Plots_All_Data.csv"
    )
    if not outpath:
        return

    rows = []

    # Plot 1: complete FRC curve values.
    for e in entries:
        q = np.asarray(e.get("q", []))
        frc = np.asarray(e.get("frc", []))
        n = min(q.size, frc.size)

        for i in range(n):
            qv = q.reshape(-1)[i]
            fv = frc.reshape(-1)[i]
            if np.iscomplexobj(fv):
                fv = np.real(fv)

            rows.append({
                "Plot": "FRC Curves",
                "Entry": e.get("name", ""),
                "Path": e.get("path", ""),
                "Point": i,
                "q / Pixel": qv,
                "FRC": fv,
                "Magnetic Field (mT)": e.get("magnet_B", np.nan),
                "Resolution": e.get("resolution", np.nan),
                "q cut": e.get("q_cut", np.nan),
                "FFT pixel cut": e.get("FFT_pixel_cut", np.nan),
            })

    # Plot 2: resolution-vs-field points.
    for i, e in enumerate(entries):
        B = e.get("magnet_B", np.nan)
        R = e.get("resolution", np.nan)
        if np.isfinite(B) and np.isfinite(R):
            rows.append({
                "Plot": "Resolution vs Magnetic Field",
                "Entry": e.get("name", ""),
                "Path": e.get("path", ""),
                "Point": i,
                "q / Pixel": np.nan,
                "FRC": np.nan,
                "Magnetic Field (mT)": B,
                "Resolution": R,
                "q cut": e.get("q_cut", np.nan),
                "FFT pixel cut": e.get("FFT_pixel_cut", np.nan),
            })

    try:
        df = pd.DataFrame(rows)
        df.to_csv(outpath, index=False)

        self.tab13_status_var.set(
            f"✓ Both FRC plots exported to CSV: "
            f"{Path(outpath).name}"
        )
        try:
            self.log(f"Tab 13 FRC plots exported: {outpath}")
        except Exception:
            pass

        messagebox.showinfo(
            "Export Plot",
            f"Exported both FRC plots to:\n\n{outpath}",
            parent=self.root
        )

    except Exception as exc:
        messagebox.showerror(
            "Export Plot Error",
            f"{type(exc).__name__}: {exc}",
            parent=self.root
        )

def _tab13_build(self):
    if hasattr(self,"tab_frc_13"):
        return

    self.tab_frc_13=ttk.Frame(self.nb)
    self.nb.add(self.tab_frc_13,text="13. FRC")

    self.tab13_frc_file_path=None
    self.tab13_frc_entries=[]
    self.tab13_status_var=tk.StringVar(
        value="Ready — open an FRC HDF5 file."
    )

    self.tab_frc_13.grid_rowconfigure(1,weight=1)
    self.tab_frc_13.grid_columnconfigure(0,weight=1)

    toolbar=ttk.LabelFrame(
        self.tab_frc_13,
        text="FRC FILE / ANALYSIS CONTROLS",
        padding=5
    )
    toolbar.grid(
        row=0,column=0,sticky="ew",
        padx=6,pady=(5,3)
    )

    for j in range(6):
        toolbar.grid_columnconfigure(j,weight=1)

    buttons=[
        ("OPEN / BROWSE FRC",
         lambda:_tab13_open_frc_file(self)),
        ("REFRESH FRC",
         lambda:_tab13_refresh(self)),
        ("OPEN FRC INFO",
         lambda:_tab13_open_complete_frc_info(self)),
        ("EXPORT COMPLETE REPORT (JSON)",
         lambda:_tab13_export_report(self)),
        ("EXPORT SUMMARY",
         lambda:_tab13_export_summary_csv(self)),
        ("EXPORT PLOT",
         lambda:_tab13_export_plot(self)),
    ]

    for j,(txt,cmd) in enumerate(buttons):
        ttk.Button(
            toolbar,text=txt,command=cmd
        ).grid(
            row=0,column=j,
            sticky="ew",
            padx=2
        )

    ttk.Label(
        toolbar,
        textvariable=self.tab13_status_var,
        anchor="w"
    ).grid(
        row=1,column=0,columnspan=6,
        sticky="ew",
        padx=3,pady=(3,0)
    )

    # Complete FRC workspace with vertical/horizontal scrolling.
    scroll=tk.Canvas(
        self.tab_frc_13,
        highlightthickness=0,
        borderwidth=0
    )
    sy=ttk.Scrollbar(
        self.tab_frc_13,
        orient="vertical",
        command=scroll.yview
    )
    sx=ttk.Scrollbar(
        self.tab_frc_13,
        orient="horizontal",
        command=scroll.xview
    )
    scroll.configure(
        yscrollcommand=sy.set,
        xscrollcommand=sx.set
    )

    scroll.grid(
        row=1,column=0,
        sticky="nsew",
        padx=6,pady=(2,6)
    )
    sy.grid(row=1,column=1,sticky="ns")
    sx.grid(row=2,column=0,sticky="ew")

    host=ttk.Frame(
        scroll,
        width=1500,
        height=900
    )
    scroll.create_window(
        (0,0),
        window=host,
        anchor="nw"
    )
    host.bind(
        "<Configure>",
        lambda e:scroll.configure(
            scrollregion=scroll.bbox("all")
        )
    )

    figbox=ttk.LabelFrame(
        host,
        text="FRC FIGURES — 1 × 2",
        padding=5
    )
    figbox.pack(
        fill="x",
        pady=(0,6)
    )

    self.tab13_frc_fig=plt.Figure(
        figsize=(14.5,6.0),
        dpi=100,
        facecolor="white"
    )
    self.tab13_frc_ax1=self.tab13_frc_fig.add_subplot(121)
    self.tab13_frc_ax2=self.tab13_frc_fig.add_subplot(122)

    self.tab13_frc_ax1.set_title(
        "FRC Curves",
        pad=12
    )
    self.tab13_frc_ax1.set_xlabel(
        "q / Pixel",
        labelpad=9
    )
    self.tab13_frc_ax1.set_ylabel(
        "FRC",
        labelpad=9
    )
    self.tab13_frc_ax1.grid(
        True,
        alpha=.2
    )

    self.tab13_frc_ax2.set_title(
        "Resolution vs Magnetic Field",
        pad=12
    )
    self.tab13_frc_ax2.set_xlabel(
        "Magnetic Field (mT)",
        labelpad=9
    )
    self.tab13_frc_ax2.set_ylabel(
        "Resolution",
        labelpad=9
    )
    self.tab13_frc_ax2.grid(
        True,
        alpha=.2
    )

    self.tab13_frc_fig.subplots_adjust(
        left=.065,
        right=.985,
        bottom=.10,
        top=.90,
        wspace=.28
    )

    self.tab13_frc_canvas=FigureCanvasTkAgg(
        self.tab13_frc_fig,
        master=figbox
    )
    self.tab13_frc_canvas.draw()
    self.tab13_frc_canvas.get_tk_widget().pack(
        fill="both",
        expand=False
    )

    tablebox=ttk.LabelFrame(
        host,
        text="FRC SUMMARY",
        padding=3
    )
    tablebox.pack(
        fill="x",
        pady=5
    )

    cols=(
        "Entry",
        "Path",
        "FRC Shape",
        "Magnet B",
        "Resolution",
        "q cut",
        "FFT pixel cut"
    )

    self.tab13_frc_tree=ttk.Treeview(
        tablebox,
        columns=cols,
        show="headings",
        height=14
    )

    for col in cols:
        self.tab13_frc_tree.heading(
            col,
            text=col
        )
        self.tab13_frc_tree.column(
            col,
            width=150 if col!="Path" else 430,
            minwidth=90,
            anchor="w"
        )

    vy=ttk.Scrollbar(
        tablebox,
        orient="vertical",
        command=self.tab13_frc_tree.yview
    )
    hx=ttk.Scrollbar(
        tablebox,
        orient="horizontal",
        command=self.tab13_frc_tree.xview
    )
    self.tab13_frc_tree.configure(
        yscrollcommand=vy.set,
        xscrollcommand=hx.set
    )

    self.tab13_frc_tree.grid(
        row=0,column=0,
        sticky="ew"
    )
    vy.grid(row=0,column=1,sticky="ns")
    hx.grid(row=1,column=0,sticky="ew")
    tablebox.grid_columnconfigure(0,weight=1)



def _tab14_refresh(self):
    if not hasattr(self,"tab14_log_text"): return
    try:
        self.tab14_log_text.delete("1.0","end")
        self.tab14_log_text.insert(
            "end","\n".join(
                str(x) for x in (getattr(self,"_log_messages",[]) or [])
            )
        )
        self.tab14_log_text.see("end")
    except Exception:
        pass


def _tab14_clear(self):
    try:self._log_messages.clear()
    except Exception:pass
    _tab14_refresh(self)


def _tab14_export(self):
    path=filedialog.asksaveasfilename(
        parent=self.root,title="Export LOG",
        defaultextension=".txt",
        filetypes=[("Text file","*.txt"),("All files","*.*")],
        initialfile="CDI_Workflow_LOG.txt"
    )
    if not path:return
    try:
        with open(path,"w",encoding="utf-8") as fh:
            fh.write("\n".join(
                str(x) for x in (getattr(self,"_log_messages",[]) or [])
            ))
        try:self.status_var.set(f"Log exported: {Path(path).name}")
        except Exception:pass
    except Exception as exc:
        messagebox.showerror(
            "LOG Export",f"{type(exc).__name__}: {exc}",parent=self.root
        )



# ============================================================================
# TAB 14 — LOG
# ============================================================================
def _tab14_build(self):
    """Create a dedicated, scrollable LOG tab with refresh/export controls."""
    # Prevent duplicate construction if the UI is rebuilt.
    if getattr(self, "_tab14_built", False):
        return

    self.tab14_log = ttk.Frame(self.nb)
    self.nb.add(self.tab14_log, text="14. LOG")
    self._tab14_built = True

    self.tab14_log.grid_rowconfigure(1, weight=1)
    self.tab14_log.grid_columnconfigure(0, weight=1)

    # Top controls.
    controls = ttk.LabelFrame(
        self.tab14_log,
        text="LOG CONTROLS",
        padding=5
    )
    controls.grid(
        row=0,
        column=0,
        sticky="ew",
        padx=6,
        pady=(5, 3)
    )
    controls.grid_columnconfigure(0, weight=1)
    controls.grid_columnconfigure(1, weight=1)

    ttk.Button(
        controls,
        text="REFRESH",
        command=lambda: _tab14_refresh(self)
    ).grid(
        row=0,
        column=0,
        sticky="ew",
        padx=2
    )

    ttk.Button(
        controls,
        text="EXPORT REPORT",
        command=lambda: _tab14_export(self)
    ).grid(
        row=0,
        column=1,
        sticky="ew",
        padx=2
    )

    # Scrollable text area.
    body = ttk.Frame(self.tab14_log, padding=6)
    body.grid(
        row=1,
        column=0,
        sticky="nsew"
    )
    body.grid_rowconfigure(0, weight=1)
    body.grid_columnconfigure(0, weight=1)

    text = tk.Text(
        body,
        wrap="word",
        font=("Consolas", 10)
    )
    scrollbar = ttk.Scrollbar(
        body,
        orient="vertical",
        command=text.yview
    )
    text.configure(
        yscrollcommand=scrollbar.set
    )

    text.grid(
        row=0,
        column=0,
        sticky="nsew"
    )
    scrollbar.grid(
        row=0,
        column=1,
        sticky="ns"
    )

    self.tab14_log_text = text

    # The existing self.log() method writes to self.log_text, so point it
    # to the new Tab-14 widget without changing the logging algorithm.
    self.log_text = self.tab14_log_text

    _tab14_refresh(self)


def _tab14_refresh(self):
    """Refresh the Tab-14 view from the existing in-memory log messages."""
    widget = getattr(self, "tab14_log_text", None)
    if widget is None:
        return

    try:
        widget.delete("1.0", "end")
        messages = getattr(self, "_log_messages", []) or []
        if messages:
            widget.insert("end", "\n".join(str(x) for x in messages))
            widget.insert("end", "\n")
        widget.see("end")
    except tk.TclError:
        self.tab14_log_text = None
        if getattr(self, "log_text", None) is widget:
            self.log_text = None


def _tab14_export(self):
    """Export the current complete log as a TXT report."""
    path = filedialog.asksaveasfilename(
        parent=self.root,
        title="Export LOG report",
        defaultextension=".txt",
        filetypes=[
            ("Text report", "*.txt"),
            ("All files", "*.*")
        ],
        initialfile="CDI_Workflow_LOG_Report.txt"
    )
    if not path:
        return

    try:
        messages = getattr(self, "_log_messages", []) or []
        report = "\n".join(str(x) for x in messages)

        with open(path, "w", encoding="utf-8") as fh:
            fh.write(report)
            if report:
                fh.write("\n")

        # Do not route this through self.log(), otherwise the export action
        # itself would alter the report just written.
        try:
            self.status_var.set(
                f"LOG report exported: {Path(path).name}"
            )
        except Exception:
            pass

    except Exception as exc:
        messagebox.showerror(
            "LOG Export Error",
            f"Could not export LOG report:\n\n"
            f"{type(exc).__name__}: {exc}",
            parent=self.root
        )


# ============================================================================
# BUILD / APPEND TABS 12-14
# Existing Tabs 1–11 _build_ui code is not modified.
# ============================================================================
_BASE_BUILD_UI_12_14 = CDIWorkflowApp._build_ui

def _build_ui_with_12_14(self,*args,**kwargs):
    result = _BASE_BUILD_UI_12_14(self,*args,**kwargs)
    try:
        _tab12_build(self)
    except Exception as exc:
        try:self.log(f"Tab 12 init warning: {type(exc).__name__}: {exc}")
        except Exception:pass
    try:
        _tab13_build(self)
    except Exception as exc:
        try:self.log(f"Tab 13 init warning: {type(exc).__name__}: {exc}")
        except Exception:pass
    try:
        _tab14_build(self)
    except Exception as exc:
        try:self.log(f"Tab 14 init warning: {type(exc).__name__}: {exc}")
        except Exception:pass
    try:
        self._install_tab_reset_buttons()
    except Exception as exc:
        try:self.log(f"TAB RESET button initialization warning: {type(exc).__name__}: {exc}")
        except Exception:pass
    return result



# Explicit Tab-13 method bindings; toolbar buttons call these stable methods.
CDIWorkflowApp._tab13_open_frc_file=_tab13_open_frc_file
CDIWorkflowApp._tab13_load_frc=_tab13_load_frc
CDIWorkflowApp._tab13_refresh=_tab13_refresh
CDIWorkflowApp._tab13_scan_frc_entries=_tab13_scan_frc_entries
CDIWorkflowApp._tab13_render_frc=_tab13_render_frc
CDIWorkflowApp._tab13_open_complete_frc_info=_tab13_open_complete_frc_info
CDIWorkflowApp._tab13_export_report=_tab13_export_report
CDIWorkflowApp._tab13_export_summary_csv=_tab13_export_summary_csv

# FINAL TAB-8 VISIBILITY GUARANTEE
# This is deliberately the LAST _build_ui wrapper.  It only guarantees that
# the existing Tab-8 frame is present at Notebook index 7.  It does not touch
# Tab 9 or any later tab, and it never rebuilds an already-populated Tab 8.
_BASE_BUILD_UI_FINAL_ALL_TABS = _build_ui_with_12_14

def _build_ui_final_tab8_visible(self, *args, **kwargs):
    result = _BASE_BUILD_UI_FINAL_ALL_TABS(self, *args, **kwargs)
    try:
        nb = self.nb
        tab8 = getattr(self, 'tab_particle_density', None)
        if tab8 is None or str(tab8) not in nb.tabs():
            tab8 = ttk.Frame(nb)
            self.tab_particle_density = tab8
            nb.insert(7, tab8, text='8. Blob Analysis')
        else:
            # Ensure the existing frame is at exactly position 8 (index 7).
            current = list(nb.tabs()).index(str(tab8))
            if current != 7:
                nb.forget(tab8)
                nb.insert(7, tab8, text='8. Blob Analysis')
            else:
                nb.tab(tab8, text='8. Blob Analysis')

        # Build Tab 8 only when its frame is empty.
        if len(tab8.winfo_children()) == 0:
            _tab8_v7_build(self)

        # Refresh source after all tabs have been constructed, without altering Tab 9.
        try:
            if callable(globals().get('_tab8_v7_ensure_ready')):
                self.root.after_idle(lambda: _tab8_v7_ensure_ready(self))
        except Exception:
            pass
    except Exception as exc:
        try:
            self.log(f'Tab 8 visibility/build warning: {type(exc).__name__}: {exc}')
        except Exception:
            pass
    return result

CDIWorkflowApp._build_ui = _build_ui_final_tab8_visible


# ============================================================================
# TAB 6 ONLY — ROBUST NATIVE TK MOUSE-WHEEL ZOOM
# The existing Matplotlib scroll callback can be swallowed by some TkAgg
# configurations.  Use the native Tk canvas event as a reliable fallback.
# This is intentionally scoped to Tab 6 Primary ROI only.
# ============================================================================

def _tab6_native_tk_mouse_zoom(self, event):
    """Handle Primary ROI mouse-wheel zoom directly from the Tk canvas."""
    try:
        canvas = getattr(self, '_proc_canvas', None)
        ax = getattr(self, '_proc_ax', None)
        image = getattr(self, '_proc_image_display', None)
        if canvas is None or ax is None or image is None:
            return "break"
        if image.get_array() is None:
            return "break"

        # Windows/macOS Tk: positive delta = wheel up.
        # Linux Tk: Button-4 = up, Button-5 = down.
        delta = getattr(event, 'delta', 0) or 0
        if delta:
            step = float(delta) / 120.0
        else:
            num = getattr(event, 'num', None)
            if num == 4:
                step = 1.0
            elif num == 5:
                step = -1.0
            else:
                return "break"

        if step == 0:
            return "break"

        # Convert native Tk canvas coordinates (origin at top-left) to
        # Matplotlib display coordinates (origin at bottom-left).
        widget = canvas.get_tk_widget()
        try:
            cw = max(1, int(widget.winfo_width()))
            ch = max(1, int(widget.winfo_height()))
        except Exception:
            cw = ch = 1

        x_disp = float(event.x)
        y_disp = float(ch - float(event.y))

        # Convert the cursor position to the current data coordinates.
        try:
            renderer = self._proc_fig.canvas.get_renderer()
            bbox = ax.get_window_extent(renderer=renderer)
            if not (bbox.x0 <= x_disp <= bbox.x1 and bbox.y0 <= y_disp <= bbox.y1):
                return "break"
            cx, cy = ax.transData.inverted().transform((x_disp, y_disp))
        except Exception:
            return "break"

        # Reuse the established Tab-6 zoom implementation so the existing
        # physical nm extent, zoom state and reset behavior remain unchanged.
        class _NativeWheelEvent:
            pass

        ev = _NativeWheelEvent()
        ev.inaxes = ax
        ev.xdata = float(cx)
        ev.ydata = float(cy)
        ev.step = float(step)

        self._tab6_wheel_zoom_handler(ev)
        return "break"

    except Exception as exc:
        try:
            self.log(f"Tab 6 native mouse-wheel zoom warning: {type(exc).__name__}: {exc}")
        except Exception:
            pass
        return "break"


def _tab6_install_native_tk_mouse_zoom(self):
    """Install the Tab-6-only native Tk wheel binding."""
    try:
        canvas = getattr(self, '_proc_canvas', None)
        if canvas is None:
            return
        tk_widget = canvas.get_tk_widget()

        # Prevent the Matplotlib scroll callback from applying a second zoom.
        try:
            registry = getattr(canvas.callbacks, 'callbacks', {}).get('scroll_event', {})
            for cid in list(registry.keys()):
                try:
                    canvas.mpl_disconnect(cid)
                except Exception:
                    pass
        except Exception:
            pass

        # Native Tk bindings are independent of Tabs 1–5 and 7–14.
        tk_widget.bind('<MouseWheel>', lambda e: self._tab6_native_tk_mouse_zoom(e), add='+')
        tk_widget.bind('<Button-4>', lambda e: self._tab6_native_tk_mouse_zoom(e), add='+')
        tk_widget.bind('<Button-5>', lambda e: self._tab6_native_tk_mouse_zoom(e), add='+')

    except Exception as exc:
        try:
            self.log(f"Tab 6 native mouse-wheel hookup warning: {type(exc).__name__}: {exc}")
        except Exception:
            pass


CDIWorkflowApp._tab6_native_tk_mouse_zoom = _tab6_native_tk_mouse_zoom
CDIWorkflowApp._tab6_install_native_tk_mouse_zoom = _tab6_install_native_tk_mouse_zoom

# Install after the complete UI has been constructed, but before mainloop().
# No Tab-6 processing, ROI data, sliders, playback, or other tabs are changed.
_previous_build_ui_for_native_zoom = CDIWorkflowApp._build_ui

def _build_ui_with_tab6_native_zoom(self, *args, **kwargs):
    result = _previous_build_ui_for_native_zoom(self, *args, **kwargs)
    try:
        self._tab6_install_native_tk_mouse_zoom()
    except Exception:
        pass
    return result

CDIWorkflowApp._build_ui = _build_ui_with_tab6_native_zoom

root=tk.Tk()
app=CDIWorkflowApp(root)
root.mainloop()



# ============================================================================
# FINAL TAB-7 RESET PATCH
# RESET must restore Tab 7 state IN PLACE.
# It must NOT rebuild the Line Profile widget tree, create a new canvas,
# or create another notebook/Tk window. This is deliberately scoped to Tab 7.
# ============================================================================

_TAB7_RESET_SAFE_ORIGINAL = CDIWorkflowApp._reset_current_tab

def _reset_current_tab_tab7_in_place(self):
    try:
        selected = self.nb.select()
        if not selected:
            return

        tab_index = self.nb.index(selected)

        # Tab 7 = notebook index 6.
        # Reset its data/analysis state only; preserve the existing widgets,
        # Matplotlib canvases, notebook frame and window.
        if tab_index == 6:
            try:
                # Stop any active drawing/measurement interaction.
                self._lpa_draw_mode = False
                self._lpa_draw_target = 1
                self._lpa_drag_start = None
                self._lpa_temp_line = None
                self._lpa_measure_mode = False
                self._lpa_measure_first = None
                self._lpa_measure_drag = None
            except Exception:
                pass

            try:
                # Restore the default number of lines without rebuilding UI.
                if hasattr(self, "_lpa_nlines_var"):
                    self._lpa_nlines_var.set(1)
            except Exception:
                pass

            try:
                # Remove all existing line/profile/measurement state.
                _lpa_clear_lines(self)
            except Exception:
                try:
                    self._lpa_lines = []
                    self._lpa_measurements = []
                except Exception:
                    pass

            try:
                # Restore full ROI2 view.
                self._lpa_zoomed = False
                self._lpa_zoom_limits = None
                _lpa_reset_zoom(self)
            except Exception:
                pass

            try:
                # Return to the first frame and refresh the existing canvas.
                if getattr(self, "_lpa_stack", None) is not None:
                    n = len(self._lpa_stack)
                    if n:
                        self._lpa_frame_var.set(0)
                        self._lpa_slider.configure(
                            from_=0, to=max(0, n - 1), state="normal"
                        )
                        _lpa_render_frame(self, 0, preserve_zoom=False)
                else:
                    _lpa_fetch_roi2(self, select_tab=False)
            except Exception:
                pass

            try:
                self._lpa_status_var.set(
                    "Tab 7 reset complete — existing Line Profile window restored in place."
                )
            except Exception:
                pass

            try:
                self.status_var.set("TAB RESET complete — 7. Line Profile Analysis")
            except Exception:
                pass

            return

    except Exception:
        # If the Tab-7-specific reset encounters an unexpected state,
        # fall through to the original handler rather than breaking the app.
        pass

    return _TAB7_RESET_SAFE_ORIGINAL(self)


CDIWorkflowApp._reset_current_tab = _reset_current_tab_tab7_in_place



# ============================================================================
# FINAL TAB 9 PATCH — RELIABLE FETCH OF PROCESSED PRIMARY ROI1 FROM TAB 6
# Isolated: does not modify Tabs 1–8 or Tabs 10–14.
# ============================================================================

def _tab9_final_fetch_processed_roi_from_tab6(self, preserve_frame=True):
    """Fetch the complete processed Primary ROI1 stack from Tab 6.

    Tab 6 remains the source of truth: confirmed Primary ROI -> finalized
    resolution -> Tab-6 processing controls -> _proc_image(frame).
    Tab 9 receives a private copy and never edits Tab-6 data/state.
    """
    def status(msg):
        try:
            self.tab9_roi4_status_var.set(msg)
            self.root.update_idletasks()
        except Exception:
            pass

    def disable_controls():
        for name in ('tab9_roi4_slider', 'tab9_roi4_play_button', 'tab9_roi4_first_button'):
            try:
                getattr(self, name).configure(state='disabled')
            except Exception:
                pass

    # 1) Validate the actual Tab-6 source.
    recons = getattr(self, 'recons', None)
    roi_stack = getattr(self, 'roi_stack', None)
    if recons is None or roi_stack is None:
        self._tab9_roi1_ready = False
        disable_controls()
        status('TAB 9: Tab 6 processed Primary ROI1 is not available. Confirm Primary ROI1 in Tab 6 first.')
        return False

    try:
        base = np.asarray(roi_stack)
        recon = np.asarray(recons)
        if base.ndim != 3 or base.shape[0] < 1:
            raise RuntimeError(f'Invalid Tab-6 ROI stack shape: {base.shape}')
        n, h, w = map(int, base.shape)
        if recon.ndim < 3 or len(recon) != n:
            raise RuntimeError(f'Tab-6 reconstruction/source frame count mismatch: recons={recon.shape}, roi_stack={base.shape}')
    except Exception as exc:
        self._tab9_roi1_ready = False
        disable_controls()
        status(f'TAB 9: invalid Tab-6 ROI source — {type(exc).__name__}: {exc}')
        return False

    # 2) Require the same finalized resolution that Tab 6 uses for processing.
    if not bool(getattr(self, 'roi_resolution_applied', False)):
        self._tab9_roi1_ready = False
        disable_controls()
        status('TAB 9: Apply SAME CONFIRMED ROI resolution in Tab 6 first.')
        return False

    # 3) Resolve the processing mask. Normally Tab 6 supplies local_mask;
    # recover it only locally if an old session lost that attribute.
    local_mask = getattr(self, 'local_mask', None)
    try:
        if local_mask is None:
            local_mask = np.isfinite(base[0])
        local_mask = np.asarray(local_mask, dtype=bool)
        if local_mask.shape != (h, w) or not np.any(local_mask):
            raise RuntimeError(f'Invalid Tab-6 local mask shape/content: {local_mask.shape}')
    except Exception as exc:
        self._tab9_roi1_ready = False
        disable_controls()
        status(f'TAB 9: Tab-6 processing mask is unavailable — {exc}')
        return False

    # 4) Use Tab 6's own processing function for EVERY frame. This preserves
    # filtering, strength, contrast and brightness exactly as shown in Tab 6.
    proc_fn = globals().get('_proc_image')
    if proc_fn is None:
        self._tab9_roi1_ready = False
        disable_controls()
        status('TAB 9: Tab-6 processing function _proc_image is unavailable.')
        return False

    try:
        frames = []
        for i in range(n):
            arr = np.asarray(proc_fn(self, i), dtype=np.float32)
            if arr.ndim != 2 or arr.shape != (h, w):
                raise RuntimeError(f'Processed frame {i+1}/{n}: got {arr.shape}, expected {(h, w)}')
            frames.append(arr)
        processed = np.stack(frames, axis=0).astype(np.float32, copy=True)
    except Exception as exc:
        self._tab9_roi1_ready = False
        disable_controls()
        status(f'TAB 9: failed to fetch processed ROI1 from Tab 6 — {type(exc).__name__}: {exc}')
        return False

    # 5) Get the authoritative physical calibration from Tab 6/Tab 5.
    try:
        scales = _tab6_effective_nm_per_output_px(self)
    except Exception:
        scales = None
    if scales is None:
        try:
            v = float(getattr(self, 'primary_roi_final_nm_per_pixel', np.nan))
            if np.isfinite(v) and v > 0:
                scales = (v, v)
        except Exception:
            scales = None
    if scales is None:
        self._tab9_roi1_ready = False
        disable_controls()
        status('TAB 9: Tab-6 processed ROI scale is unavailable. Check Primary ROI calibration.')
        return False

    sx, sy = map(float, scales)
    if not (np.isfinite(sx) and np.isfinite(sy) and sx > 0 and sy > 0):
        self._tab9_roi1_ready = False
        disable_controls()
        status(f'TAB 9: invalid physical scale X={sx}, Y={sy} nm/output-px.')
        return False

    # 6) Preserve the current Tab-9 frame if possible.
    try:
        old_idx = int(round(float(self.tab9_roi4_frame_var.get()))) if preserve_frame else 0
    except Exception:
        old_idx = 0
    old_idx = max(0, min(old_idx, n - 1))

    # 7) Private immutable Tab-9 copy. Tab 6 is never modified.
    self._tab9_roi1_stack = processed
    self._tab9_roi1_stack_ref = processed
    self._tab9_roi1_shape = (h, w)
    self._tab9_roi1_extent = (0.0, float(w) * sx, 0.0, float(h) * sy)
    self._tab9_roi4_scale = (sx, sy)
    self._tab9_roi1_ready = True
    try:
        self._tab9_roi1_cmap = str(self.colormap_var.get() or 'gray')
    except Exception:
        self._tab9_roi1_cmap = 'gray'
    self._tab9_roi1_clim = (0.0, 1.0)

    # Keep a confirmed ROI4 only if it still belongs to the same source geometry.
    c = getattr(self, '_tab9_roi4_coords', None)
    if isinstance(c, dict):
        try:
            valid = (0 <= int(c['xmin']) <= int(c['xmax']) < w and
                     0 <= int(c['ymin']) <= int(c['ymax']) < h)
        except Exception:
            valid = False
        if not valid:
            self._tab9_roi4_coords = None
            self._tab9_roi4_preview_coords = None
            self._tab9_roi4_confirmed = False
            self._tab9_roi4_stack = None
            self._tab9_roi4_stack_coords = None
            self._tab9_roi4_stack_shape = None

    self._tab9_roi4_programmatic = True
    try:
        self.tab9_roi4_slider.configure(from_=0, to=n-1, resolution=1, state='normal')
        self.tab9_roi4_start_var.set(1)
        self.tab9_roi4_end_var.set(n)
        self.tab9_roi4_frame_var.set(old_idx)
        self.tab9_roi4_slider.set(old_idx)
    finally:
        self._tab9_roi4_programmatic = False

    try:
        _tab9_roi4_show_frame(self, old_idx, preserve_zoom=True)
    except Exception as exc:
        status(f'TAB 9: processed ROI1 fetched, but display update failed — {type(exc).__name__}: {exc}')
        return False

    try:
        self.tab9_roi4_play_button.configure(text='PLAY', state='normal')
        self.tab9_roi4_first_button.configure(state='normal')
    except Exception:
        pass

    status(f'✓ PROCESSED PRIMARY ROI1 FETCHED FROM TAB 6 | Frames={n} | Size={w} × {h} px | FOV={w*sx:.6g} × {h*sy:.6g} nm | Frame={old_idx+1}/{n}')
    return True

# Only replace Tab-9's fetch entry point and its visible button callback.
CDIWorkflowApp._tab9_roi4_fetch_from_tab6 = _tab9_final_fetch_processed_roi_from_tab6
CDIWorkflowApp._tab9_roi4_refresh = lambda self, preserve_frame=True: _tab9_final_fetch_processed_roi_from_tab6(self, preserve_frame=preserve_frame)

def _tab9_rebind_processed_fetch_button(self):
    try:
        self.tab9_roi1_from_tab6_button.configure(
            command=lambda: _tab9_final_fetch_processed_roi_from_tab6(self, preserve_frame=True)
        )
    except Exception:
        pass

# Re-fetch automatically when the user enters Tab 9, and after Tab 6 applies
# the finalized ROI resolution. No other tab's source/state is changed.
try:
    _TAB6_APPLY_FOR_TAB9_FINAL = CDIWorkflowApp.apply_same_confirmed_roi_resolution
except Exception:
    _TAB6_APPLY_FOR_TAB9_FINAL = None

if _TAB6_APPLY_FOR_TAB9_FINAL is not None:
    def _tab6_apply_for_tab9_final(self, *args, **kwargs):
        result = _TAB6_APPLY_FOR_TAB9_FINAL(self, *args, **kwargs)
        try:
            if hasattr(self, 'tab9_roi1_from_tab6_button'):
                _tab9_final_fetch_processed_roi_from_tab6(self, preserve_frame=True)
        except Exception as exc:
            try:
                self.tab9_roi4_status_var.set(f'TAB 9 refresh after Tab 6 processing failed: {type(exc).__name__}: {exc}')
            except Exception:
                pass
        return result
    CDIWorkflowApp.apply_same_confirmed_roi_resolution = _tab6_apply_for_tab9_final

try:
    _TAB9_PREV_BUILD_FOR_FINAL_FETCH = CDIWorkflowApp._build_ui
    def _tab9_build_ui_final_fetch(self, *args, **kwargs):
        result = _TAB9_PREV_BUILD_FOR_FINAL_FETCH(self, *args, **kwargs)
        try:
            _tab9_rebind_processed_fetch_button(self)
        except Exception:
            pass
        try:
            if hasattr(self, 'nb') and hasattr(self, 'tab_primary_roi9'):
                def _on_tab9_selected(event=None):
                    try:
                        if event is not None and event.widget.select() != str(self.tab_primary_roi9):
                            return
                        _tab9_final_fetch_processed_roi_from_tab6(self, preserve_frame=True)
                    except Exception:
                        pass
                self.nb.bind('<<NotebookTabChanged>>', _on_tab9_selected, add='+')
        except Exception:
            pass
        return result
    CDIWorkflowApp._build_ui = _tab9_build_ui_final_fetch
except Exception:
    pass





Tab 8 initialization warning: TclError: Slave index 7 out of bounds
Found 102 HDF5 files; identified 101 CDI reconstruction files.
Calibration loaded: 28 points; field -485.50 to 485.00 mT
Loading 101 CDI reconstructions in Ramp Down (+ to -) order.
Loaded 1/101: Recon_ImId_2924_RefId_2923_cdi_field=1.0000_diff_wittrocs_crop=400_photonlevel=-1000.0_0_-1.png.hdf5 | raw field=+1.000000 | shape=(1248, 1248)
Loaded 2/101: Recon_ImId_2924_RefId_2923_cdi_field=0.9836_diff_wittrocs_crop=400_photonlevel=-1000.0_0_-1.png.hdf5 | raw field=+0.983600 | shape=(1248, 1248)
Loaded 3/101: Recon_ImId_2924_RefId_2923_cdi_field=0.9626_diff_wittrocs_crop=400_photonlevel=-1000.0_0_-1.png.hdf5 | raw field=+0.962600 | shape=(1248, 1248)
Loaded 4/101: Recon_ImId_2924_RefId_2923_cdi_field=0.9435_diff_wittrocs_crop=400_photonlevel=-1000.0_0_-1.png.hdf5 | raw field=+0.943500 | shape=(1248, 1248)
Loaded 5/101: Recon_ImId_2924_RefId_2923_cdi_field=0.9222_diff_wittrocs_crop=400_photonlevel=-1000.0_0_-1.png.hdf5 | r

C:\Users\seela\AppData\Local\Temp\ipykernel_26712\3272865667.py:24572: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = remove_small_objects(mask, min_size=min_obj)
C:\Users\seela\AppData\Local\Temp\ipykernel_26712\3272865667.py:24632: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  core_mask = remove_small_obj